# 실험 내용

리트리버 검증을 위해 임의의 쿼리 셋(Query Set)을 구축하여 대조군 비교를 진행했습니다.

점수 기준:
- 3점:query에 대한 answer이 retriever에 모두 들어있음
- 2점:query에 대한 answer이 retriever에 일부 들어있음
- 1점:query 문장의 핵심 키워드가 retriever에 들어있음
- 0점:질문과 연관 없음

Query A list 만 점: 18점 (6개 질문)

Query B list 만 점: 18점 (6개 질문)

Query C list 만 점: 30점 (10개 질문)

---

**텍스트 분할 방식과 정보량에 따른 성능 비교**

실험에서는 **‘텍스트의 양’**이 성능에 어떤 영향을 미치는지 확인하는 것을 목표로 했습니다.

가장 기본적인 검색 방식인 ‘유사도 검색(sim_results)’을 기준으로 고정한 뒤, 두 가지 핵심 요소를 변경하며 성능을 측정했습니다. 

텍스트를 단순히 글자 수로 나누는 **‘Naive 방식’**과 문장의 의미적 연관성을 고려하여 분할하는 **‘Semantic 방식’**의 결과를 비교했습니다. 

검색 결과로 가져오는 텍스트의 최소 글자 수를 ‘500자’, ‘2000자’, ‘5000자’로 다르게 설정하여, 답변 생성에 사용되는 정보의 양이 많을수록 성능이 향상되는지 관찰했습니다.

### Query A list
sim_results

| chunk방식        | 500자 이상 | 2000자 이상 | 5000자 이상 |
|----------------|-----------|-------------|-------------|
| naive_results  | 12        | 14          | 14          |
| semantic_results | 12      | 14          | 14          |

---

### Query B list
sim_results

| chunk방식        | 500자 이상 | 2000자 이상 | 5000자 이상 |
|----------------|-----------|-------------|-------------|
| naive_results  | 14        | 17          | 18          |
| semantic_results | 18      | 18          | 18          |

---

**다양한 검색 알고리즘 성능 비교**

두 번째 실험에서는 1단계 결과를 바탕으로, 더 정교하고 발전된 **‘검색 방식(Search Method)’**들이 성능 향상에 얼마나 기여하는지 평가했습니다.

검색 결과의 최소 글자 수를 1000자로 고정한 후, 여러 검색 알고리즘을 적용하여 그 성능을 비교했습니다. 

기본이 되는 ‘유사도 검색(sim_results)’ 외에도, ‘MMR(Maximal Marginal Relevance)’, ‘하이브리드 검색(Hybrid Search)’, **‘다중 쿼리 검색(Multi-query Search)’**을 각각 테스트했습니다.

### Query A list
1000자 (불러온 모든 청크 텍스트 최소 글자 수)

| chunk방식        | sim_results | mmr_results | hybrid_results | multi_query_search |
|----------------|-------------|-------------|----------------|--------------------|
| naive_results  | 14          | 14          | 14             | 14                 |
| semantic_results | 13        | 13          | 13             | 13                 |

---

### Query B list
1000자 (불러온 모든 청크 텍스트 최소 글자 수)

| chunk방식        | sim_results | mmr_results | hybrid_results | multi_query_search |
|----------------|-------------|-------------|----------------|--------------------|
| naive_results  | 16          | 16          | 16             | 16                 |
| semantic_results | 18        | 18          | 18             | 18                 |

---

3단계: 청크 크기 및 중복 영역 최적화

마지막 실험에서는 텍스트 분할 과정의 가장 중요한 세부 매개변수인 **‘청크 크기(Chunk Size)’**와 **‘중복 크기(Overlap)’**를 조정하며 실험을 진행했습니다.


다시 검색 방식을 ‘유사도 검색’으로 고정하고, 800자 크기 / 200자 중복 설정과 1200자 크기 / 300자 중복 설정을 비교했습니다. 

이 테스트는 청크를 얼마나 큰 단위로 나눌지, 그리고 청크 간의 문맥적 연결을 위해 얼마나 많은 내용을 중복시킬지가 성능에 미치는 영향을 구체적으로 확인하기 위해 진행되었습니다.

### Query A list
sim_results  
5000자 (불러온 모든 청크 텍스트 최소 글자 수)

| chunk방식        | 800/200 | 1200/300 |
|----------------|---------|----------|
| naive_results  | 14      | 18       |
| semantic_results | 14    | 14       |

---

### Query B list
sim_results  
5000자 (불러온 모든 청크 텍스트 최소 글자 수)

| chunk방식        | 800/200 | 1200/300 |
|----------------|---------|----------|
| naive_results  | 18      | 16       |
| semantic_results | 18    | 18       |

---

### Query C list
sim_results  
5000자 (불러온 모든 청크 텍스트 최소 글자 수)

| chunk방식        | 800/200 | 1200/300 |
|----------------|---------|----------|
| naive_results  | 23      | 20       |
| semantic_results | 20    | 22       |

---

# 종합 결론

**Semantic 분할 방식이 조금 우위**

naive_results score: (총 합) 318

semantic_results score: (총 합) 324

문맥을 고려한 'Semantic' 텍스트 분할 방식이 'Naive' 방식보다 전반적으로 높거나 안정적인 성능을 보였습니다.

이는 관련성 높은 정보를 효과적으로 그룹화하는 능력 덕분으로 분석됩니다.

**정보량과 청크 설정의 상호작용**

검색 성능은 단순히 정보량을 늘리거나 특정 청크 크기를 고수하는 것만으로 최적화되지 않는거 같습니다. 

데이터의 내용과 구조에 따라 적절한 청크 크기와 중복 영역을 설정하고, 충분하지만 과하지 않은 최적의 정보량을 제공하는 것이 중요한거 같습니다.

# 실험 코드

In [2]:
import os
import sys
import warnings

In [3]:
from src.document_loader import DocumentLoader

In [4]:
from dotenv import load_dotenv

In [5]:
# 프로젝트 루트 설정
PROJECT_ROOT = os.path.dirname(os.path.abspath("__file__"))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

In [6]:
load_dotenv(os.path.join(PROJECT_ROOT, ".env"))

# API 키 확인
api_key = os.getenv("OPENAI_API_KEY", "")
if not api_key or api_key == "your_api_key_here":
    print("!! .env 파일에 OPENAI_API_KEY를 설정해 주세요 !!")
    print(f"   경로: {os.path.join(PROJECT_ROOT, '.env')}")
else:
    print(f"API 키 로드 완료 (끝 4자리: ...{api_key[-4:]})")

API 키 로드 완료 (끝 4자리: ...d34A)


In [7]:
from configs.config import Config

dir_config = Config(
    scenario="B",
    openai_embedding_model="text-embedding-3-small",
    openai_chat_model="gpt-5-mini",
    chunk_size=800,
    chunk_overlap=200,
    chunking_method="naive",
    retrieval_top_k=5,
    retrieval_method="similarity",
    temperature=0.1,
    vectordb_type="chroma",
)

In [8]:
# 메타데이터 확인
import pandas as pd

metadata_path = os.path.join(PROJECT_ROOT, dir_config.metadata_csv)
if os.path.exists(metadata_path):
    metadata_df = pd.read_csv(metadata_path)
    print(f"메타데이터: {len(metadata_df)}건")
    print(f"컬럼: {list(metadata_df.columns)}")
    #display(metadata_df.head())
else:
    metadata_df = None
    print(f"메타데이터 파일이 없습니다: {metadata_path}")
    print("data/data_list.csv 파일을 준비해 주세요.")

메타데이터: 100건
컬럼: ['공고 번호', '공고 차수', '사업명', '사업 금액', '발주 기관', '공개 일자', '입찰 참여 시작일', '입찰 참여 마감일', '사업 요약', '파일형식', '파일명', '텍스트']


---

**문서 로드**

In [11]:
# 문서 로드 (PDF, HWP 모두 지원)
loader = DocumentLoader(
    documents_dir=os.path.join(PROJECT_ROOT, dir_config.documents_dir),
    metadata_csv=metadata_path if metadata_df is not None else None,
)

print("문서 로드 중...")
documents = loader.load_all()

# 로드된 문서 통계
if documents:
    total_chars = sum(len(d["text"]) for d in documents)
    avg_chars = total_chars / len(documents)
    print(f"\n총 문자 수: {total_chars:,}")
    print(f"평균 문자 수: {avg_chars:,.0f}")
    
    # 샘플 확인
    print(f"\n첫 번째 문서 미리보기 (처음 500자):")
    print(documents[0]["text"][:500])
    print(f"\n메타데이터: {documents[0]['metadata']}")
else:
    print("로드된 문서가 없습니다. data/documents/ 디렉토리에 PDF/HWP 파일을 넣어주세요.")

문서 로드 중...
  ✓ 로드 완료: (사)벤처기업협회_2024년 벤처확인종합관리시스템 기능 고도화 용역사업 .hwp (102283 chars)
  ✓ 로드 완료: (사)부산국제영화제_2024년 BIFF & ACFM 온라인서비스 재개발 및 행사지원시.hwp (50352 chars)
  ✓ 로드 완료: (사）한국대학스포츠협의회_KUSF 체육특기자 경기기록 관리시스템 개발.hwp (73204 chars)
  ✓ 로드 완료: (재)예술경영지원센터_통합 정보시스템 구축 사전 컨설팅.hwp (41818 chars)
  ✓ 로드 완료: 2025 구미 아시아육상경기선수권대회 조직위원회_2025 구미아시아육상경.hwp (46166 chars)
  ✓ 로드 완료: BioIN_의료기기산업 종합정보시스템(정보관리기관) 기능개선 사업(2차).hwp (69365 chars)
  ✓ 로드 완료: KOICA 전자조달_[긴급] [지문] [국제] 우즈베키스탄 열린 의정활동 상하원 .hwp (127455 chars)
  ✓ 로드 완료: 경기도 안양시_호계체육관 배드민턴장 및 탁구장 예약시스템 구축 용역.hwp (61744 chars)
  ✓ 로드 완료: 경기도 평택시_2024년도 평택시 버스정보시스템(BIS) 구축사업.hwp (79922 chars)
  ✓ 로드 완료: 경기도사ᄒ

In [11]:
documents[0]['text'][:100]

'2024년 ｢벤처확인종합관리시스템 기능 고도화｣ 용역사업\n(복수의결권주식, 스톡옵션, 성과조건부주식) -\n제안요청서\n2024. 03.\n목 차\n1. 추진개요 螨ȃ 3\n2. 추진방안 '

In [12]:
documents[0]['metadata']

{'filename': '(사)벤처기업협회_2024년 벤처확인종합관리시스템 기능 고도화 용역사업 .hwp',
 'file_path': '/home/qkrtntjd/sprint_ai_07_middle/data/documents/(사)벤처기업협회_2024년 벤처확인종합관리시스템 기능 고도화 용역사업 .hwp'}

---

In [9]:
import pickle
file_path = "data/documents_list.pkl"

In [17]:
with open(file_path, "wb") as f:
    pickle.dump(documents, f)
print(f"저장 완료: {file_path}")

저장 완료: data/documents_list.pkl


In [10]:
with open(file_path, "rb") as f:
    documents = pickle.load(f)
print(f"불러오기 완료: {len(documents)}개의 문서")

불러오기 완료: 100개의 문서


---

---

---

## 백터 DB 생성

In [13]:
from src.chunker import chunk_documents

In [14]:
import gc
import shutil
from src.embedder import EmbeddingModel, VectorStore
import pickle

In [15]:
from src.retriever import Retriever

In [16]:
# 임베딩 모델
embedding_model = EmbeddingModel(dir_config)

---

In [17]:
def create_vectorDB( embedding_model,
                     vectorDB_name,
                     documents,
                     chunking_method: str="naive",
                     retrieval_method: str="similarity",
                     chunk_size: int=800,
                     chunk_overlap: int=200,
                   ):

    chunk_size = chunk_size
    chunk_overlap = chunk_overlap
    
    #config 설정
    data_config = Config(
        scenario="B",
        openai_embedding_model="text-embedding-3-small",
        openai_chat_model="gpt-5-mini",
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        chunking_method=chunking_method,
        retrieval_top_k=5,
        retrieval_method=retrieval_method,
        temperature=0.1,
        vectordb_type="chroma",
        vectordb_dir = f"data/{vectorDB_name}"
    )
    
    # 청킹
    print("청킹을 시작합니다...")
    chunks = chunk_documents(
        documents,
        method=chunking_method,
        chunk_size=data_config.chunk_size,
        chunk_overlap=data_config.chunk_overlap,
    )

    print("-"*40)
    print("벡터 DB를 생성합니다...")
    vector_store = VectorStore(data_config, embedding_model)
    vector_store.initialize(collection_name="rfp_documents")

    vector_store.add_documents(chunks, show_progress=False)
    print(f"\n naive 최종 저장 문서 수: {vector_store.get_collection_count()}")

    retriever = Retriever(data_config, vector_store, embedding_model)

    return (retriever, vector_store, data_config)



def del_vectorDB(dataDB_tuple):
    retriever, vector_store, data_config = dataDB_tuple
    shutil.rmtree(data_config.vectordb_dir, ignore_errors=True)
    del retriever
    del vector_store
    gc.collect()

In [21]:
def get_invoke(query_list, 
               retriever, 
               retrieval_top_k=5, 
               print_mode=False, 
               save_mode=False, 
               save_name="results"):
    
    r_list = []

    if type(query_list) == dict:
        query_list = [k for k,v in query_list.items()]

    for query in query_list:
        if print_mode:
            print("="*60)
            print(f"query: {query}")
    
        r_list.append({
            'query' : query,
            'sim_results' : retriever.similarity_search(query, top_k=retrieval_top_k),
            'mmr_results' : retriever.mmr_search(query, top_k=retrieval_top_k),
            'hybrid_results' : retriever.hybrid_search(query, top_k=retrieval_top_k),
            'multi_query_search' : retriever.multi_query_search(query, top_k=retrieval_top_k),
            'hybrid_search' : retriever.hybrid_search(query, top_k=retrieval_top_k),
        })

    if save_mode:
        save_path = os.path.join(results_save_path, save_name)
        with open(save_path, "wb") as f:
            pickle.dump(r_list, f)
        if print_mode:
            print(f"[{i}] 저장 완료: {save_path}")
    return r_list

In [19]:
def result2txt(results):
    return "\n".join([res['text'] for res in results])

def result2txt_s(results, text_size=500):
    res_plus = ""
    for res in results:
        res_plus += (res['text'] + '\n')
        if len(res_plus) > text_size:
            break
    return res_plus

In [22]:
PROMPT_TEMPLATE_SYSTEM = """아래 제공된 query와 실제 answer과 연관된 정보를 찾았는지 retriever를 판별해야 합니다.
각 retriever A B 에 대해 점수를 매겨야 합니다. 간결하고 명확하게 판별해야 합니다.

점수 기준:
3점:query에 대한 answer이 retriever에 모두 들어있음
2점:query에 대한 answer이 retriever에 일부 들어있음
1점:query 문장의 핵심 키워드가 retriever에 들어있음
0점:질문과 연관 없음

출력 양식(예시):
retriever A 점수: 1점
(이유 설명)
retriever B 점수: 2점
(이유 설명)
"""


PROMPT_TEMPLATE = """# query:
{query}

# answer:
{answer}

---

# retriever A:
{retriever_A}

---

# retriever B:
{retriever_B}

---
"""
import openai

client = openai.OpenAI(api_key=api_key)

def openai_eval(r_list_A, 
                r_list_B,
                answer_dict={},
                search = "sim_results",
                text_size=0,
                PROMPT_TEMPLATE_SYSTEM=PROMPT_TEMPLATE_SYSTEM, 
                PROMPT_TEMPLATE=PROMPT_TEMPLATE,
                print_mode=True):

    search = "sim_results"

    answer_list = []
    rag_prompt_list = []
    
    for nrl, srl in zip(r_list_A, r_list_B):
        searchA = nrl[search]
        searchB = srl[search]
    
        if nrl['query'] != srl['query']:
            print("비교 결과의 query가 다릅니다.")
            pass
    
        print("="*60)
        print(f"query: {nrl['query']}")
        print(f"answer: {answer_dict.get(nrl['query'], 'answer없음.(문맥으로 평가)')}")

        if text_size == 0:
            print(f"A text size: {len(result2txt_s(searchA))}")
            print(f"B text size: {len(result2txt_s(searchB))}")
            rag_prompt = PROMPT_TEMPLATE.format(
                query=nrl['query'], 
                retriever_A=result2txt(searchA), 
                retriever_B=result2txt(searchB),
                answer=answer_dict.get(nrl['query'], "answer없음.(문맥으로 평가)")
            )
        elif text_size > 0:
            print(f"A text size: {len(result2txt_s(searchA, text_size))}")
            print(f"B text size: {len(result2txt_s(searchB, text_size))}")
            rag_prompt = PROMPT_TEMPLATE.format(
                query=nrl['query'], 
                retriever_A=result2txt_s(searchA, text_size), 
                retriever_B=result2txt_s(searchB, text_size),
                answer=answer_dict.get(nrl['query'], "answer없음.(문맥으로 평가)")
            )
        
        messages=[
            {"role": "system", "content":PROMPT_TEMPLATE_SYSTEM},
            {"role": "user", "content": rag_prompt}
        ]
        
        answer_text = client.chat.completions.create(
            model="gpt-5",
            messages=messages
        ).choices[0].message.content

        if print_mode:
            print(answer_text)

        rag_prompt_list.append(rag_prompt)
        answer_list.append(answer_text)

    return rag_prompt_list, answer_list

---

---

---

---

In [32]:
del_vectorDB(naive_r_v_p)
del_vectorDB(semantic_r_v_p)

In [33]:
naive_r_v_p = create_vectorDB(embedding_model=embedding_model,
                        vectorDB_name="naive_vectordb3",         #항상 바꿔줘야함(커널 재시작 없이 실험 시)
                        documents=documents,
                        chunking_method="naive",        # naive, semantic
                        retrieval_method="similarity")

semantic_r_v_p = create_vectorDB(embedding_model=embedding_model,
                        vectorDB_name="semantic_vectordb3",      #항상 바꿔줘야함(커널 재시작 없이 실험 시)
                        documents=documents,
                        chunking_method="semantic",     # naive, semantic
                        retrieval_method="similarity")

청킹을 시작합니다...
----------------------------------------
벡터 DB를 생성합니다...

 naive 최종 저장 문서 수: 11567
청킹을 시작합니다...
----------------------------------------
벡터 DB를 생성합니다...

 naive 최종 저장 문서 수: 50902


In [34]:
query_dict = {
    "아이돌봄 인적성 검사 시스템 운영 사업의 주요 목적은 무엇인가?":              "인적성 검사 수행을 지원하고, 돌봄활동과 연계한 사후관리 및 인력 검증을 강화하는 것이다.",
    "아이돌봄 시스템에서 개인정보 보호를 위해 요구되는 조치는 무엇인가?":              "접속기록 관리, 내부관리계획 수립 등 기술적·관리적 보호조치를 수행해야 한다.",
    "2025 구미 아시아육상경기선수권대회의 개최 기간은 언제인가?":                 "2025년 5월 27일부터 5월 31일까지 총 5일간 개최된다.",
    "구미 아시아육상경기선수권대회 홈페이지에서 제공되는 주요 콘텐츠 항목에는 무엇이 포함되는가?":     "대회소개, 경기일정 및 결과, 메달순위, 참가선수 정보, 커뮤니티 등이 포함된다.",
    "원자력 선량평가시스템의 성능 요구사항 중 질의응답 시간 기준은 어떻게 정의되어 있는가?":                  "일반 조회 요청은 3초 이내 처리되어야 하며, 10초 이상 소요되는 작업은 사용자에게 팝업 메시지를 제공해야 한다.",
    "원자력 선량평가시스템의 포트란 모듈 개선 요구사항에서 추가로 고려해야 하는 평가 요소는 무엇인가?":                        " ICRP60 기준에 따른 23개 장기를 고려한 평가장기 추가가 요구된다.",
}

In [38]:
query_dict = {
    "본 과업지시서에서 '극저온시스템'은 무엇을 포함하는 것으로 정의하고 있습니까?": "'극저온시스템'은 SCL3/SCL2 극저온플랜트 및 부대설비, SCL3/SCL2 분배시스템 및 부대설비, SCL3/SCL2 초전도가속관 저온유지모듈, 기체헬륨 탱크, 액체질소 시스템 및 극저온시스템용 헬륨회수시스템을 모두 포함하는 것으로 정의됩니다.", 
              "극저온시스템 과업 추진 일정에 따르면, 4분기에 SCL3 구간에 대해 예정된 주요 업무는 무엇이며, 이때의 업무 기간은 '가동기간'과 '휴지기간' 중 어느 것에 해당합니까?": "2025년 4분기에 SCL3 구간에 예정된 주요 업무는 '유지보수'이며, 이 기간은 '휴지기간'에 해당합니다.",
             "극저온시스템 '가동기간' 동안 계약자가 24시간 수행해야 할 극저온시스템 모니터링 업무에는 어떤 것들이 있습니까? (3가지 이상)": """가동기간 동안 수행해야 할 모니터링 업무는 다음과 같습니다.
알람 메시지 확인
주요 센서(온도, 압력, 유량 등)의 트렌드 확인
정상 운전 상황의 주요 센서 지시값 파악 및 숙지
비정상 운전 상황 감지
제어로직 정상 작동 여부 확인""",
             "중이온가속기 극저온시스템의 '가동기간'과 '휴지기간'의 운전 업무 형태(근무 방식)는 어떻게 다릅니까?": "'가동기간'에는 시스템의 24시간 안정적인 운영을 위해 교대근무(24시간 운전)가 이루어집니다. 반면, '휴지기간'에는 주로 시스템 유지보수 등을 위해 주간 운전을 기본으로 합니다.",
             "극저온시스템 과업지시서에 따르면, 가동기간 중 교대근무 인력이 개별적으로 얻은 운전 경험을 통합하여 정리하기 어려운 이유는 무엇이며, 이 업무는 주로 언제 수행되어야 합니까?": "교대근무로 인해 각 운전 인력마다 경험하는 운전 상황이 서로 달라, 모든 인원이 함께 모여 운전 경험을 통합하고 정리할 여건이 형성되기 어렵기 때문입니다. 이러한 운전 경험 통합 및 정리(문서화) 업무는 주로 주간 근무가 이루어지는 '휴지기간'에 집중적으로 수행됩니다.",
             "계약업체가 매월 운전 현황을 정리해서 연구원에 보고해야 하는 공식적인 문서는 무엇이며, 제출 시점은 언제입니까?": "매월 운전 상황을 요약하여 제출해야 하는 문서는 '월간보고서'이며, 매월 기성 비용을 신청할 때 제출해야 합니다.",
}


In [62]:
query_dict = {
    "학업성취도 다차원 종단분석 통합시스템에서 소프트웨어 개발보안 가이드 준수를 위해 제안사가 반드시 제시해야 하는 항목은 무엇인가?": "표준 코딩스타일 정의, 개발절차 및 방법론, 교육계획 등의 준수방안과 함께 소스코드 보안취약성을 자체 진단하고 제거하기 위한 방안(진단도구, 전문인력, 진단환경, 진단회수, 조치방안 등)을 제시해야 한다.",
    "학업성취도 다차원 종단분석 통합시스템의 구축시스템 테스트 요구사항에서 제시해야 하는 테스트 관련 요소는 무엇인가?": "단위 테스트, 통합 테스트, 시스템 테스트, 성능 테스트 등 테스트 유형에 따른 테스트 환경, 방법, 절차 등을 제시해야 한다.",
    "학업성취도 다차원 종단분석 통합시스템의 제안서 작성 시 페이지 번호는 어떻게 부여해야 하는가?": "각 페이지 하단 중앙에 일련번호를 부여하며, 각 장별로 번호를 부여해야 한다.",
    "학업성취도 다차원 종단분석 통합시스템의 제안서 작성 시 사용해서는 안 되는 표현 방식은 무엇인가?": "‘사용가능하다’, ‘할 수 있다’, ‘고려하고 있다’와 같은 모호한 표현은 사용하면 안 되며, 평가 시 불가능한 것으로 간주된다.",
    "학업성취도 다차원 종단분석 통합시스템에서 중소기업이 입찰에 참여하기 위해 반드시 제출해야 하는 증명서는 무엇인가?": "직접생산확인증명서(정보시스템개발서비스, 8111159901)를 제출해야 한다.",
    "학업성취도 다차원 종단분석 통합시스템 사업에서 대기업 및 상호출자제한기업집단의 참여가 제한되는 이유는 무엇인가?": "본 사업이 20억 미만 사업으로서 소프트웨어 진흥법에 따라 중소 소프트웨어사업자의 참여 보호를 위해 대기업 및 상호출자제한기업집단의 참여가 제한된다.",
    "학업성취도 다차원 종단분석 통합시스템의 하도급 계약 시 허용되는 최대 비율은 얼마이며, 재하도급은 어떻게 처리되는가?": "하도급 비율은 총 사업금액의 50%를 초과할 수 없으며, 재하도급은 원칙적으로 금지된다.",
    "학업성취도 다차원 종단분석 통합시스템의 제안서 본문의 권장 분량 및 제출 형식은 어떻게 되는가?": "제안서 본문은 단면 기준 200장 이내로 작성하며, PDF 전자문서 형태로 제출해야 하고 용량은 300MB 이내여야 한다.",
    "학업성취도 다차원 종단분석 통합시스템의 보안 요구사항에서 준수해야 하는 대표적인 보안 지침에는 어떤 것들이 포함되는가?": "국가 정보보안 기본지침, 개인정보보호 지침, 소프트웨어 개발보안 가이드, 보안약점 진단가이드 등 다양한 공공 보안 지침을 준수해야 한다.",
    "학업성취도 다차원 종단분석 통합시스템 제안서에 포함된 내용과 이후 수정된 내용의 법적 효력은 어떻게 되는가?": "제안서 내용과 이후 수정·보완된 내용은 계약서에 명시되지 않더라도 계약서와 동일한 효력을 가지며, 계약서에 명시된 경우에는 계약서가 우선한다.",
}

In [63]:
invoke1 = get_invoke(query_list=       query_dict, 
                     retriever=        naive_r_v_p[0],
                     retrieval_top_k=  40,
                     print_mode=       False, 
                     save_mode=        False
                    )

invoke2 = get_invoke(query_list=       query_dict, 
                     retriever=        semantic_r_v_p[0],
                     retrieval_top_k=  40,
                     print_mode=       False, 
                     save_mode=        False
                    )

In [58]:
r_a = openai_eval(r_list_A=invoke1,
            r_list_B=invoke2,
            answer_dict=query_dict,
            search = "hybrid_search",
            text_size=1000)
                                        # "sim_results" 
                                        # "mmr_results"
                                        # "hybrid_results"
                                        # "multi_query_search"
                                        # "hybrid_search"

query: 학업성취도 다차원 종단분석 통합시스템에서 소프트웨어 개발보안 가이드 준수를 위해 제안사가 반드시 제시해야 하는 항목은 무엇인가?
answer: 표준 코딩스타일 정의, 개발절차 및 방법론, 교육계획 등의 준수방안과 함께 소스코드 보안취약성을 자체 진단하고 제거하기 위한 방안(진단도구, 전문인력, 진단환경, 진단회수, 조치방안 등)을 제시해야 한다.
A text size: 5256
B text size: 5066
retriever A 점수: 3점
질문에 대한 정답의 모든 항목(표준 코딩스타일 정의, 개발절차·방법론, 교육계획 제시 및 소스코드 보안취약성 자체 진단·제거 방안: 진단도구, 전문인력, 진단환경, 진단회수, 조치방안)이 그대로 포함되어 있음.

retriever B 점수: 3점
정답에서 요구한 동일 항목들이 명시되어 있음(표준 코딩스타일·개발절차·방법론·교육계획 제시와 보안취약성 자체 진단·제거 방안: 진단도구, 전문인력, 진단환경, 진단회수, 조치방안).
query: 학업성취도 다차원 종단분석 통합시스템의 구축시스템 테스트 요구사항에서 제시해야 하는 테스트 관련 요소는 무엇인가?
answer: 단위 테스트, 통합 테스트, 시스템 테스트, 성능 테스트 등 테스트 유형에 따른 테스트 환경, 방법, 절차 등을 제시해야 한다.
A text size: 5292
B text size: 5106
retriever A 점수: 3점
단위·통합·성능 테스트 등 테스트 유형과 함께 테스트 환경, 방법, 절차(계획서/시나리오/데이터/일정 등) 제시 요구가 명확히 포함되어 있어 답변의 요소를 모두 충족함.

retriever B 점수: 3점
“단위/통합/시스템/성능 테스트 등 유형별 테스트 환경·방법·절차 제시”를 직접적으로 명시하고 있어 정답과 완전히 일치함.
query: 학업성취도 다차원 종단분석 통합시스템의 제안서 작성 시 페이지 번호는 어떻게 부여해야 하는가?
answer: 각 페이지 하단 중앙에 일련번호를 부여하며, 각 장별로 번호를 부여해야 한다.
A

In [41]:
for r, a in zip(r_a[0], r_a[1]):
    print("="*60)
    print(r.replace("#", "###"))
    print("### openai answer")
    print(a, "\n")

### query:
본 과업지시서에서 '극저온시스템'은 무엇을 포함하는 것으로 정의하고 있습니까?

### answer:
'극저온시스템'은 SCL3/SCL2 극저온플랜트 및 부대설비, SCL3/SCL2 분배시스템 및 부대설비, SCL3/SCL2 초전도가속관 저온유지모듈, 기체헬륨 탱크, 액체질소 시스템 및 극저온시스템용 헬륨회수시스템을 모두 포함하는 것으로 정의됩니다.

---

### retriever A:
7. 안전관리 .............................................................................................................. 19
8. 붙임 자료 ........................................................................................................... 19
문서번호 : -
2025년도 중이온가속기용 극저온시스템
개정번호 : 0
운전 용역 과업지시서 발 행 일 : 2024. 10. 30
페 이 지 : 5/19
1. 서 론
1.1 목 적
기초과학연구원 중이온가속기연구소(이하 “연구원”)에서 운영중인 중이온가속기용 극저
온시스템(이하 “극저온시스템”)은 고도의 공정 시스템으로, 안전하고 안정적인 중이온가속
기의 운영을 위해 상시 모니터링이 필요하다. 이에 본 과업지시서는 극저온시스템의 운전
용역에 대한 상세 과업 내용을 기술한다.
1.2 용어 설명
본 과업지시서에서 사용하는 용어에 대한 상세 내용은 표 1.과 같다. 계약자는 반드시 해당
용어의 의미를 충분히 이해하고 숙지해야 하며, 용어 미숙지로 인한 책임은 계약자에게 있
다.
표 1. 본 과업지시서에 사용되는 용어 설명
용어 설명
연구원 중이온가속기연구소
계약자 본 과업을 수행할 계약상대자
SCL3/SCL2 극저온플랜트 및 부대설비,
SCL3/SCL2 분배시스템 및 부대설비,
SCL3/SCL2 초전도가속관 저온유지모듈,
극저온시스템
기

---

---

---

---

In [78]:
del_vectorDB(naive_r_v_p)
del_vectorDB(semantic_r_v_p)

In [44]:
naive1200_r_v_p = create_vectorDB(embedding_model=embedding_model,
                        vectorDB_name="naive_vectordb_chunk_size_test2",         #항상 바꿔줘야함(커널 재시작 없이 실험 시)
                        documents=documents,
                        chunking_method="naive",        # naive, semantic
                        retrieval_method="similarity",
                        chunk_size=1200,
                        chunk_overlap=300,)

semantic1200_r_v_p = create_vectorDB(embedding_model=embedding_model,
                        vectorDB_name="semantic_vectordb_chunk_size_test2",      #항상 바꿔줘야함(커널 재시작 없이 실험 시)
                        documents=documents,
                        chunking_method="semantic",     # naive, semantic
                        retrieval_method="similarity",
                        chunk_size=1200,
                        chunk_overlap=300,)

청킹을 시작합니다...
----------------------------------------
벡터 DB를 생성합니다...

 naive 최종 저장 문서 수: 7720
청킹을 시작합니다...
----------------------------------------
벡터 DB를 생성합니다...

 naive 최종 저장 문서 수: 49260


In [45]:
query_dict = {
    "아이돌봄 인적성 검사 시스템 운영 사업의 주요 목적은 무엇인가?":              "인적성 검사 수행을 지원하고, 돌봄활동과 연계한 사후관리 및 인력 검증을 강화하는 것이다.",
    "아이돌봄 시스템에서 개인정보 보호를 위해 요구되는 조치는 무엇인가?":              "접속기록 관리, 내부관리계획 수립 등 기술적·관리적 보호조치를 수행해야 한다.",
    "2025 구미 아시아육상경기선수권대회의 개최 기간은 언제인가?":                 "2025년 5월 27일부터 5월 31일까지 총 5일간 개최된다.",
    "구미 아시아육상경기선수권대회 홈페이지에서 제공되는 주요 콘텐츠 항목에는 무엇이 포함되는가?":     "대회소개, 경기일정 및 결과, 메달순위, 참가선수 정보, 커뮤니티 등이 포함된다.",
    "원자력 선량평가시스템의 성능 요구사항 중 질의응답 시간 기준은 어떻게 정의되어 있는가?":                  "일반 조회 요청은 3초 이내 처리되어야 하며, 10초 이상 소요되는 작업은 사용자에게 팝업 메시지를 제공해야 한다.",
    "원자력 선량평가시스템의 포트란 모듈 개선 요구사항에서 추가로 고려해야 하는 평가 요소는 무엇인가?":                        " ICRP60 기준에 따른 23개 장기를 고려한 평가장기 추가가 요구된다.",
}

In [59]:
query_dict = {
    "본 과업지시서에서 '극저온시스템'은 무엇을 포함하는 것으로 정의하고 있습니까?": "'극저온시스템'은 SCL3/SCL2 극저온플랜트 및 부대설비, SCL3/SCL2 분배시스템 및 부대설비, SCL3/SCL2 초전도가속관 저온유지모듈, 기체헬륨 탱크, 액체질소 시스템 및 극저온시스템용 헬륨회수시스템을 모두 포함하는 것으로 정의됩니다.", 
              "극저온시스템 과업 추진 일정에 따르면, 4분기에 SCL3 구간에 대해 예정된 주요 업무는 무엇이며, 이때의 업무 기간은 '가동기간'과 '휴지기간' 중 어느 것에 해당합니까?": "2025년 4분기에 SCL3 구간에 예정된 주요 업무는 '유지보수'이며, 이 기간은 '휴지기간'에 해당합니다.",
             "극저온시스템 '가동기간' 동안 계약자가 24시간 수행해야 할 극저온시스템 모니터링 업무에는 어떤 것들이 있습니까? (3가지 이상)": """가동기간 동안 수행해야 할 모니터링 업무는 다음과 같습니다.
알람 메시지 확인
주요 센서(온도, 압력, 유량 등)의 트렌드 확인
정상 운전 상황의 주요 센서 지시값 파악 및 숙지
비정상 운전 상황 감지
제어로직 정상 작동 여부 확인""",
             "중이온가속기 극저온시스템의 '가동기간'과 '휴지기간'의 운전 업무 형태(근무 방식)는 어떻게 다릅니까?": "'가동기간'에는 시스템의 24시간 안정적인 운영을 위해 교대근무(24시간 운전)가 이루어집니다. 반면, '휴지기간'에는 주로 시스템 유지보수 등을 위해 주간 운전을 기본으로 합니다.",
             "극저온시스템 과업지시서에 따르면, 가동기간 중 교대근무 인력이 개별적으로 얻은 운전 경험을 통합하여 정리하기 어려운 이유는 무엇이며, 이 업무는 주로 언제 수행되어야 합니까?": "교대근무로 인해 각 운전 인력마다 경험하는 운전 상황이 서로 달라, 모든 인원이 함께 모여 운전 경험을 통합하고 정리할 여건이 형성되기 어렵기 때문입니다. 이러한 운전 경험 통합 및 정리(문서화) 업무는 주로 주간 근무가 이루어지는 '휴지기간'에 집중적으로 수행됩니다.",
             "계약업체가 매월 운전 현황을 정리해서 연구원에 보고해야 하는 공식적인 문서는 무엇이며, 제출 시점은 언제입니까?": "매월 운전 상황을 요약하여 제출해야 하는 문서는 '월간보고서'이며, 매월 기성 비용을 신청할 때 제출해야 합니다.",
}


In [53]:
query_dict = {
    "학업성취도 다차원 종단분석 통합시스템에서 소프트웨어 개발보안 가이드 준수를 위해 제안사가 반드시 제시해야 하는 항목은 무엇인가?": "표준 코딩스타일 정의, 개발절차 및 방법론, 교육계획 등의 준수방안과 함께 소스코드 보안취약성을 자체 진단하고 제거하기 위한 방안(진단도구, 전문인력, 진단환경, 진단회수, 조치방안 등)을 제시해야 한다.",
    "학업성취도 다차원 종단분석 통합시스템의 구축시스템 테스트 요구사항에서 제시해야 하는 테스트 관련 요소는 무엇인가?": "단위 테스트, 통합 테스트, 시스템 테스트, 성능 테스트 등 테스트 유형에 따른 테스트 환경, 방법, 절차 등을 제시해야 한다.",
    "학업성취도 다차원 종단분석 통합시스템의 제안서 작성 시 페이지 번호는 어떻게 부여해야 하는가?": "각 페이지 하단 중앙에 일련번호를 부여하며, 각 장별로 번호를 부여해야 한다.",
    "학업성취도 다차원 종단분석 통합시스템의 제안서 작성 시 사용해서는 안 되는 표현 방식은 무엇인가?": "‘사용가능하다’, ‘할 수 있다’, ‘고려하고 있다’와 같은 모호한 표현은 사용하면 안 되며, 평가 시 불가능한 것으로 간주된다.",
    "학업성취도 다차원 종단분석 통합시스템에서 중소기업이 입찰에 참여하기 위해 반드시 제출해야 하는 증명서는 무엇인가?": "직접생산확인증명서(정보시스템개발서비스, 8111159901)를 제출해야 한다.",
    "학업성취도 다차원 종단분석 통합시스템 사업에서 대기업 및 상호출자제한기업집단의 참여가 제한되는 이유는 무엇인가?": "본 사업이 20억 미만 사업으로서 소프트웨어 진흥법에 따라 중소 소프트웨어사업자의 참여 보호를 위해 대기업 및 상호출자제한기업집단의 참여가 제한된다.",
    "학업성취도 다차원 종단분석 통합시스템의 하도급 계약 시 허용되는 최대 비율은 얼마이며, 재하도급은 어떻게 처리되는가?": "하도급 비율은 총 사업금액의 50%를 초과할 수 없으며, 재하도급은 원칙적으로 금지된다.",
    "학업성취도 다차원 종단분석 통합시스템의 제안서 본문의 권장 분량 및 제출 형식은 어떻게 되는가?": "제안서 본문은 단면 기준 200장 이내로 작성하며, PDF 전자문서 형태로 제출해야 하고 용량은 300MB 이내여야 한다.",
    "학업성취도 다차원 종단분석 통합시스템의 보안 요구사항에서 준수해야 하는 대표적인 보안 지침에는 어떤 것들이 포함되는가?": "국가 정보보안 기본지침, 개인정보보호 지침, 소프트웨어 개발보안 가이드, 보안약점 진단가이드 등 다양한 공공 보안 지침을 준수해야 한다.",
    "학업성취도 다차원 종단분석 통합시스템 제안서에 포함된 내용과 이후 수정된 내용의 법적 효력은 어떻게 되는가?": "제안서 내용과 이후 수정·보완된 내용은 계약서에 명시되지 않더라도 계약서와 동일한 효력을 가지며, 계약서에 명시된 경우에는 계약서가 우선한다.",
}

In [60]:
invoke1 = get_invoke(query_list=       query_dict, 
                     retriever=        naive1200_r_v_p[0],
                     retrieval_top_k=  100,
                     print_mode=       False, 
                     save_mode=        False
                    )

invoke2 = get_invoke(query_list=       query_dict, 
                     retriever=        semantic1200_r_v_p[0],
                     retrieval_top_k=  100,
                     print_mode=       False, 
                     save_mode=        False
                    )

In [61]:
r_a = openai_eval(r_list_A=invoke1,
            r_list_B=invoke2,
            answer_dict=query_dict,
            search = "sim_results",
            text_size=5000)
                                        # "sim_results" 
                                        # "mmr_results"
                                        # "hybrid_results"
                                        # "multi_query_search"

query: 본 과업지시서에서 '극저온시스템'은 무엇을 포함하는 것으로 정의하고 있습니까?
answer: '극저온시스템'은 SCL3/SCL2 극저온플랜트 및 부대설비, SCL3/SCL2 분배시스템 및 부대설비, SCL3/SCL2 초전도가속관 저온유지모듈, 기체헬륨 탱크, 액체질소 시스템 및 극저온시스템용 헬륨회수시스템을 모두 포함하는 것으로 정의됩니다.
A text size: 5854
B text size: 5227
retriever A 점수: 3점
질문에 대한 정의 문구가 표 1 용어 설명에 그대로 포함되어 있으며, 답변의 모든 구성요소(SCL3/SCL2 극저온플랜트·분배시스템·초전도가속관 저온유지모듈, 기체헬륨 탱크, 액체질소 시스템, 헬륨회수시스템)가 모두 명시됨.

retriever B 점수: 3점
표 1 용어 설명에 ‘극저온시스템’ 정의가 완전하게 수록되어 있어, 답변의 모든 항목을 동일하게 포함함.
query: 극저온시스템 과업 추진 일정에 따르면, 4분기에 SCL3 구간에 대해 예정된 주요 업무는 무엇이며, 이때의 업무 기간은 '가동기간'과 '휴지기간' 중 어느 것에 해당합니까?
answer: 2025년 4분기에 SCL3 구간에 예정된 주요 업무는 '유지보수'이며, 이 기간은 '휴지기간'에 해당합니다.
A text size: 5892
B text size: 5001
retriever A 점수: 3점
표 2의 과업 추진 일정에 2025년 4분기 SCL3 구간 ‘유지보수(휴지기간)’가 명시되어 있어 주요 업무와 기간 구분을 모두 확인할 수 있습니다.

retriever B 점수: 3점
표 2의 과업 추진 일정에 2025년 4분기 SCL3 구간 ‘유지보수(휴지기간)’가 명시되어 있어 주요 업무와 기간 구분을 모두 포함합니다.
query: 극저온시스템 '가동기간' 동안 계약자가 24시간 수행해야 할 극저온시스템 모니터링 업무에는 어떤 것들이 있습니까? (3가지 이상)
answer: 가동기간 동안 수행해야 할 모니터링 업무는 다음과 같습니다.
알람 메시지 

In [41]:
for r, a in zip(r_a[0], r_a[1]):
    print("="*60)
    print(r.replace("#", "###"))
    print("### openai answer")
    print(a, "\n")

### query:
본 과업지시서에서 '극저온시스템'은 무엇을 포함하는 것으로 정의하고 있습니까?

### answer:
'극저온시스템'은 SCL3/SCL2 극저온플랜트 및 부대설비, SCL3/SCL2 분배시스템 및 부대설비, SCL3/SCL2 초전도가속관 저온유지모듈, 기체헬륨 탱크, 액체질소 시스템 및 극저온시스템용 헬륨회수시스템을 모두 포함하는 것으로 정의됩니다.

---

### retriever A:
7. 안전관리 .............................................................................................................. 19
8. 붙임 자료 ........................................................................................................... 19
문서번호 : -
2025년도 중이온가속기용 극저온시스템
개정번호 : 0
운전 용역 과업지시서 발 행 일 : 2024. 10. 30
페 이 지 : 5/19
1. 서 론
1.1 목 적
기초과학연구원 중이온가속기연구소(이하 “연구원”)에서 운영중인 중이온가속기용 극저
온시스템(이하 “극저온시스템”)은 고도의 공정 시스템으로, 안전하고 안정적인 중이온가속
기의 운영을 위해 상시 모니터링이 필요하다. 이에 본 과업지시서는 극저온시스템의 운전
용역에 대한 상세 과업 내용을 기술한다.
1.2 용어 설명
본 과업지시서에서 사용하는 용어에 대한 상세 내용은 표 1.과 같다. 계약자는 반드시 해당
용어의 의미를 충분히 이해하고 숙지해야 하며, 용어 미숙지로 인한 책임은 계약자에게 있
다.
표 1. 본 과업지시서에 사용되는 용어 설명
용어 설명
연구원 중이온가속기연구소
계약자 본 과업을 수행할 계약상대자
SCL3/SCL2 극저온플랜트 및 부대설비,
SCL3/SCL2 분배시스템 및 부대설비,
SCL3/SCL2 초전도가속관 저온유지모듈,
극저온시스템
기

---

# Query list

**Query A list**

============================================================

Q: 아이돌봄 인적성 검사 시스템 운영 사업의 주요 목적은 무엇인가?

A: 인적성 검사 수행을 지원하고, 돌봄활동과 연계한 사후관리 및 인력 검증을 강화하는 것이다.

============================================================

Q: 아이돌봄 시스템에서 개인정보 보호를 위해 요구되는 조치는 무엇인가?

A: 접속기록 관리, 내부관리계획 수립 등 기술적·관리적 보호조치를 수행해야 한다.

============================================================

Q: 2025 구미 아시아육상경기선수권대회의 개최 기간은 언제인가?

A: 2025년 5월 27일부터 5월 31일까지 총 5일간 개최된다.

============================================================

Q: 구미 아시아육상경기선수권대회 홈페이지에서 제공되는 주요 콘텐츠 항목에는 무엇이 포함되는가?

A: 대회소개, 경기일정 및 결과, 메달순위, 참가선수 정보, 커뮤니티 등이 포함된다.

============================================================

Q: 원자력 선량평가시스템의 성능 요구사항 중 질의응답 시간 기준은 어떻게 정의되어 있는가?

A: 일반 조회 요청은 3초 이내 처리되어야 하며, 10초 이상 소요되는 작업은 사용자에게 팝업 메시지를 제공해야 한다.

============================================================

Q: 원자력 선량평가시스템의 포트란 모듈 개선 요구사항에서 추가로 고려해야 하는 평가 요소는 무엇인가?

A:  ICRP60 기준에 따른 23개 장기를 고려한 평가장기 추가가 요구된다.

---

**Query B list**

============================================================

Q: 본 과업지시서에서 '극저온시스템'은 무엇을 포함하는 것으로 정의하고 있습니까?

A: '극저온시스템'은 SCL3/SCL2 극저온플랜트 및 부대설비, SCL3/SCL2 분배시스템 및 부대설비, SCL3/SCL2 초전도가속관 저온유지모듈, 기체헬륨 탱크, 액체질소 시스템 및 극저온시스템용 헬륨회수시스템을 모두 포함하는 것으로 정의됩니다.

============================================================

Q: 극저온시스템 과업 추진 일정에 따르면, 4분기에 SCL3 구간에 대해 예정된 주요 업무는 무엇이며, 이때의 업무 기간은 '가동기간'과 '휴지기간' 중 어느 것에 해당합니까?

A: 2025년 4분기에 SCL3 구간에 예정된 주요 업무는 '유지보수'이며, 이 기간은 '휴지기간'에 해당합니다.

============================================================

Q: 극저온시스템 '가동기간' 동안 계약자가 24시간 수행해야 할 극저온시스템 모니터링 업무에는 어떤 것들이 있습니까? (3가지 이상)

A: 가동기간 동안 수행해야 할 모니터링 업무는 다음과 같습니다.
알람 메시지 확인
주요 센서(온도, 압력, 유량 등)의 트렌드 확인
정상 운전 상황의 주요 센서 지시값 파악 및 숙지
비정상 운전 상황 감지
제어로직 정상 작동 여부 확인

============================================================

Q: 중이온가속기 극저온시스템의 '가동기간'과 '휴지기간'의 운전 업무 형태(근무 방식)는 어떻게 다릅니까?

A: '가동기간'에는 시스템의 24시간 안정적인 운영을 위해 교대근무(24시간 운전)가 이루어집니다. 반면, '휴지기간'에는 주로 시스템 유지보수 등을 위해 주간 운전을 기본으로 합니다.

============================================================

Q: 극저온시스템 과업지시서에 따르면, 가동기간 중 교대근무 인력이 개별적으로 얻은 운전 경험을 통합하여 정리하기 어려운 이유는 무엇이며, 이 업무는 주로 언제 수행되어야 합니까?

A: 교대근무로 인해 각 운전 인력마다 경험하는 운전 상황이 서로 달라, 모든 인원이 함께 모여 운전 경험을 통합하고 정리할 여건이 형성되기 어렵기 때문입니다. 이러한 운전 경험 통합 및 정리(문서화) 업무는 주로 주간 근무가 이루어지는 '휴지기간'에 집중적으로 수행됩니다.

============================================================

Q: 계약업체가 매월 운전 현황을 정리해서 연구원에 보고해야 하는 공식적인 문서는 무엇이며, 제출 시점은 언제입니까?

A: 매월 운전 상황을 요약하여 제출해야 하는 문서는 '월간보고서'이며, 매월 기성 비용을 신청할 때 제출해야 합니다.

---

**Query C list**

============================================================

Q: 학업성취도 다차원 종단분석 통합시스템에서 소프트웨어 개발보안 가이드 준수를 위해 제안사가 반드시 제시해야 하는 항목은 무엇인가?

A: 표준 코딩스타일 정의, 개발절차 및 방법론, 교육계획 등의 준수방안과 함께 소스코드 보안취약성을 자체 진단하고 제거하기 위한 방안(진단도구, 전문인력, 진단환경, 진단회수, 조치방안 등)을 제시해야 한다.

============================================================

Q: 학업성취도 다차원 종단분석 통합시스템의 구축시스템 테스트 요구사항에서 제시해야 하는 테스트 관련 요소는 무엇인가?

A: 단위 테스트, 통합 테스트, 시스템 테스트, 성능 테스트 등 테스트 유형에 따른 테스트 환경, 방법, 절차 등을 제시해야 한다.

============================================================

Q: 학업성취도 다차원 종단분석 통합시스템의 제안서 작성 시 페이지 번호는 어떻게 부여해야 하는가?

A: 각 페이지 하단 중앙에 일련번호를 부여하며, 각 장별로 번호를 부여해야 한다.

============================================================

Q: 학업성취도 다차원 종단분석 통합시스템의 제안서 작성 시 사용해서는 안 되는 표현 방식은 무엇인가?

A: ‘사용가능하다’, ‘할 수 있다’, ‘고려하고 있다’와 같은 모호한 표현은 사용하면 안 되며, 평가 시 불가능한 것으로 간주된다.

============================================================

Q: 학업성취도 다차원 종단분석 통합시스템에서 중소기업이 입찰에 참여하기 위해 반드시 제출해야 하는 증명서는 무엇인가?

A: 직접생산확인증명서(정보시스템개발서비스, 8111159901)를 제출해야 한다.

============================================================

Q: 학업성취도 다차원 종단분석 통합시스템 사업에서 대기업 및 상호출자제한기업집단의 참여가 제한되는 이유는 무엇인가?

A: 본 사업이 20억 미만 사업으로서 소프트웨어 진흥법에 따라 중소 소프트웨어사업자의 참여 보호를 위해 대기업 및 상호출자제한기업집단의 참여가 제한된다.

============================================================

Q: 학업성취도 다차원 종단분석 통합시스템의 하도급 계약 시 허용되는 최대 비율은 얼마이며, 재하도급은 어떻게 처리되는가?

A: 하도급 비율은 총 사업금액의 50%를 초과할 수 없으며, 재하도급은 원칙적으로 금지된다.

============================================================

Q: 학업성취도 다차원 종단분석 통합시스템의 제안서 본문의 권장 분량 및 제출 형식은 어떻게 되는가?

A: 제안서 본문은 단면 기준 200장 이내로 작성하며, PDF 전자문서 형태로 제출해야 하고 용량은 300MB 이내여야 한다.

============================================================

Q: 학업성취도 다차원 종단분석 통합시스템의 보안 요구사항에서 준수해야 하는 대표적인 보안 지침에는 어떤 것들이 포함되는가?

A: 국가 정보보안 기본지침, 개인정보보호 지침, 소프트웨어 개발보안 가이드, 보안약점 진단가이드 등 다양한 공공 보안 지침을 준수해야 한다.

============================================================

Q: 학업성취도 다차원 종단분석 통합시스템 제안서에 포함된 내용과 이후 수정된 내용의 법적 효력은 어떻게 되는가?

A: 제안서 내용과 이후 수정·보완된 내용은 계약서에 명시되지 않더라도 계약서와 동일한 효력을 가지며, 계약서에 명시된 경우에는 계약서가 우선한다.

---

# eval_by_text_length results

## Query 1

## size 500

============================================================
### query:
아이돌봄 인적성 검사 시스템 운영 사업의 주요 목적은 무엇인가?

### answer:
인적성 검사 수행을 지원하고, 돌봄활동과 연계한 사후관리 및 인력 검증을 강화하는 것이다.

---

### retriever A:
◦원할한 인적성 검사 수행을 위한 정보시스템 지원 필요
◦돌봄활동과 연계한 사후관리 기반을 강화하여 이용자 만족도 향상
◦아이돌봄인력 인·적성에 기초한 돌봄인력 관리 강화 필요
◦정보시스템 개인정보보호법 준수를 위한 보호조치
2
사업 개요
□ 사 업 명: 아이돌봄인력 인적성 검사 정보시스템 운영
- 2024. 정보시스템 구축에 따른 유지보수 사업
□ 사업기간: 계약일 ~ 2025. 12. 31.
□ 소요예산: 172,700,000원(부가세 포함)
□ 추진방식: 제한경쟁입찰(협상에 의한 계약)
3
사업 범위
□ 인·적성 프로그램 기능수정 및 안정화 기술지원
◦인·적성 시스템의 안정적인 정보화 지원을 위한 상시 유지관리
체계 지원
◦인·적성 검사 변경에 따른 프로그램 수정 및 결과 반영
◦웹 접근성, 기술적 보안 등 전자정부 가이드라인 준수
□ 개인정보보호 및 정보보안 강화
◦개인정보보호 및 정보보호 강화 조치(방안)
◦정보시스템 운영을 위한 물리적, 기술적 보안 대책 제시
□ 안정적인 클라우드 환경 운영 및 제공
◦클라우드 무중단 환경 제공
◦클라우드 환경 운영을 위한 기술적 유지관리 지원
◦시스템 용을 위한 SW 및 외부 서비스 연계
4
기대 효과
□ (안전한 돌봄 환경 조성) 아이돌봄인력 선별 및 검증 강화 통한 안심하고 맡길 수 있는 돌봄서비스 제공
□ (효율성 제고) 아이돌봄인력 대상 인적성 검사를 편리하게 실시하여 돌봄인력 검증 절차 효율성 제고
□ (정책지원 및 제도개선 지원) 체계적인 검사 결과를 축적하고 정책 수립지원을 위한 기초자료를 제공, 지속 가능한 아이돌봄서비스 제도개선에 활용


---

### retriever B:
기대 효과
□ (안전한 돌봄 환경 조성) 아이돌봄인력 선별 및 검증 강화 통한 안심하고 맡길 수 있는 돌봄서비스 제공
□ (효율성 제고) 아이돌봄인력 대상 인적성 검사를 편리하게 실시하여 돌봄인력 검증 절차 효율성 제고
□ (정책지원 및 제도개선 지원) 체계적인 검사 결과를 축적하고 정책 수립지원을 위한 기초자료를 제공, 지속 가능한 아이돌봄서비스 제도개선에 활용
□ (돌봄인력 관리) 데이터를 기반으로 하는 아이돌봄인력에 대한 효율적 관리·감독 가능
Ⅱ. 사업 현황
1
업무현황
□ 인·적성검사 실시 정보시스템 지원
ㅇ 인·적성검사 및 실시를 위한 정보 시스템 제공
ㅇ 정보시스템 안정화 및 운영지원
□ 인·적성검사 운영
ㅇ 인적성 검사 실시 및 수행 방법 안내
ㅇ 인적성 검사 수행 모니터링 및 결과 분석
ㅇ 검사 실시 운영자 권한 부여 및 인증 관리
2
정보화 현황
□ 시스템 구성도
□ 정보시스템 구성도 요청 시 별도 공개
※ 행정기관 및 공공기관 정보시스템 구축 및 운영 지침(행정안전부 고시 제 2024-66호, 2024.8.28.) 제 17조(제안요청서 보안사항 등)에 따라, 시스템 구성도 및 상세자료 등은 공개하지 않으며, 입찰에 참가하고자 하는 사업자의 요청이 있는 경우 보안서약서를 제출 후 담당자 입회하에 제한적으로 열람 가능
□ 시스템(장비)·소프트웨어 운영현황
AS-IS
TO-BE (2025년)
ㅇ인·적성 검사 관리기능 및 사용자화면 개발
※ 클라우드 환경 기반 개발
ㅇ인·적성 검사 기능 최적화 및 운영 기술지원
-인·적성 검사 고도화에 따른 수정 반영
(검사도구 4종)


---

### openai answer
retriever A 점수: 3점
(인적성 검사 수행 지원, 돌봄활동 연계 사후관리 강화, 인력 검증 강화 등 답의 핵심 목적이 모두 명시되어 있음)

retriever B 점수: 2점
(인적성 검사 수행 지원과 인력 검증 강화는 포함되어 있으나, 돌봄활동과 연계한 사후관리 내용이 명시적으로 없음) 

============================================================
### query:
아이돌봄 시스템에서 개인정보 보호를 위해 요구되는 조치는 무엇인가?

### answer:
접속기록 관리, 내부관리계획 수립 등 기술적·관리적 보호조치를 수행해야 한다.

---

### retriever A:
- 개인정보의 기술절·관리적 보호조치 기준(개인정보위)
- 표준 개인정보 보호 지침(개인정보위)
가명정보의 결합 및 반출 등에 관한 고시(개인정보위)
개인정보 영향평가에 관한 고시(개인정보위)
※ 고시에 따라 아래 기준 이상의 개인정보 처리 시 발주처에서 개인정보 영향평가를 발주하며 영향평가결과를 반영하여 시스템을 구축해야 함
내 용
1
구축‧운용 또는 변경하려는 개인정보파일로서 5만명 이상의 정보주체에 관한 민감정보 또는 고유식별정보의 처리가 수반되는 개인정보파일
2
구축‧운용하고 있는 개인정보파일을 해당 기관 내부 또는 외부에서 구축‧운용하고 있는 다른 개인정보파일과 연계하려는 경우로서 연계결과 50만명 이상의 정보주체에 관한 개인정보가 포함되는 개인정보파일
3
구축‧운용 또는 변경하려는 개인정보파일로서 100만명 이상의 정보주체에 관한 개인정보파일
4
개인정보 영향평가를 받은 후에 개인정보 검색체계 등 개인정보파일의 운용체계를 변경하려는 경우 그 개인정보파일
○ 개인정보 안전성 확보조치 기준 이행·반영
- 개인정보 열람 시 접속기록(최소 2년 이상) 보관·관리
- 개인정보 접근권한 부여·변경·말소 이력(최소 3년 이상) 보관·관리
- 고유식별번호 및 비밀번호 생체인식정보는 안전한 암호 알고리즘을 사용 하여 저장
○ 데이터의 보안 강화를 위해 SSL서비스 도입
○ 개인정보보호를 위한 DB 암호화하여 자료 유출방지
- 인증서 기반(GPKI) 로그인 방식 적용 후 ID/PW 로그인
- 정보보안 강화를 위한 시스템에 각종 패치, 환경설정 지원 등


---

### retriever B:
6. 그 밖에 개인정보의 보호를 위하여 필요한 사항
-295-
1. 개인정보의 안전성 확보조치 철저(「개인정보보호법」제29조)
❍ 개인정보처리자는 개인정보가 분실·도난·유출·변조 또는 훼손되지 않도록 다음 각 호의 안전성 확보조치를 하여야한다.
구 분
주 요 내 용
내부관리계획(제3조)
◦내부관리계획 포함사항 명시 ※개인정보 보호책임자 지정, 개인정보취급자 교육 등
◦개인정보 처리 업무 위탁 시 관리·감독 사항
접근권한 관리(제4조)
◦개인정보처리시스템의 접근권한 부여․변경․말소 기록 최소 3년간 보관
◦안전한 비밀번호 작성규칙 수립․적용
접근통제(제5조)
◦접근통제 시스템 설치․운영 의무화 ※접근통제 시스템 : 침입차단(Firewall), 침입탐지(IDS) 기능 포함
◦주민번호 처리 시 추가 정보 확인, 모바일 기기 보호조치 조항, 홈페이지에서 고유식별정보를 처리할 경우 년 1회 취약점 점검 조치
개인정보 암호화
(제6조)
◦암호화 대상 : 고유식별정보(주민등록번호, 여권번호, 운전면허번호, 외국인등록번호), 비밀번호, 바이오정보
◦암호화 기준
구 분
암 호 화 기 준
송․수신시
모든 정보 암호화 전송
저장시
비밀번호,
바이오정보
비밀번호는 단방향 암호화,
바이오정보는 양방향 암호화
고유식별정보
ⓛ 인터넷 구간,공통(DMZ) 구간 저장시 : 의무적 암호화


---

### openai answer
retriever A 점수: 2점
- 접속기록 보관·관리 등 기술적·관리적 보호조치 내용은 포함되어 있으나, 내부관리계획 수립에 대한 명시가 없음.

retriever B 점수: 2점
- 내부관리계획 수립 등 안전성 확보조치는 명시되어 있으나, 접속기록 관리(접속기록 보관·관리)에 대한 직접적인 언급이 없음. 

============================================================
### query:
2025 구미 아시아육상경기선수권대회의 개최 기간은 언제인가?

### answer:
2025년 5월 27일부터 5월 31일까지 총 5일간 개최된다.

---

### retriever A:
종합정보시스템 및 홈페이지 등 구축 용역 제안요청서
1
대회개요
아시아의 꿈, 구미에서 세계로
 The dream of ASIA, From GUMI to the World 
비전
ㅇ (대 회 명) 2025 구미아시아육상경기선수권대회
ㅇ (기 간) 2025. 5. 27.(화) ~ 5. 31.(토) 5일간
ㅇ (장 소) 구미시민운동장 일원
ㅇ (규 모) 아시아 45개국 1,200여명 / 선수단 및 임원
ㅇ (주 최) 아시아육상연맹(Asian Athletics, AA)
ㅇ (주 관) 2025 구미아시아육상경기선수권대회 조직위원회
ㅇ (경기종목) 45개 종목
구 분
종 목 명
남자
(22)
100m, 200m, 400m, 800m, 1,500m, 5,000m,
10,000m, 110mH, 400mH, 3,000m장애물달리기,
20km경보, 4×100mR, 4×400mR, 높이뛰기,
장대높이뛰기, 멀리뛰기, 세단뛰기, 포환던지기,
원반던지기, 해머던지기, 창던지기, 10종 경기
여자
(22)
100m, 200m, 400m, 800m, 1,500m, 5,000m,
10,000m, 100mH, 400mH, 3,000m장애물달리기,
20km경보, 4×100mR, 4×400mR, 높이뛰기,
장대높이뛰기, 멀리뛰기, 세단뛰기, 포환던지기,
원반던지기, 해머던지기, 창던지기, 7종 경기
남·여
(1)
4×400m 믹스릴레이
요
2
과업개요
ㅇ (사 업 명) 2025 구미아시아육상경기선수권대회 종합정보시스템 및 홈페이지 등 구축 용역
ㅇ (사업기간) 계약일 ~ 2025. 6. 6.(금)
ㅇ (사업위치) 시민운동장 일원


---

### retriever B:
종합정보시스템 및 홈페이지 등 구축 용역 제안요청서
1
대회개요
아시아의 꿈, 구미에서 세계로
 The dream of ASIA, From GUMI to the World 
비전
ㅇ (대 회 명) 2025 구미아시아육상경기선수권대회
ㅇ (기 간) 2025. 5. 27.(화) ~ 5. 31.(토) 5일간
ㅇ (장 소) 구미시민운동장 일원
ㅇ (규 모) 아시아 45개국 1,200여명 / 선수단 및 임원
ㅇ (주 최) 아시아육상연맹(Asian Athletics, AA)
ㅇ (주 관) 2025 구미아시아육상경기선수권대회 조직위원회
ㅇ (경기종목) 45개 종목
구 분
종 목 명
남자
(22)
100m, 200m, 400m, 800m, 1,500m, 5,000m,
10,000m, 110mH, 400mH, 3,000m장애물달리기,
20km경보, 4×100mR, 4×400mR, 높이뛰기,
장대높이뛰기, 멀리뛰기, 세단뛰기, 포환던지기,
원반던지기, 해머던지기, 창던지기, 10종 경기
여자
(22)
100m, 200m, 400m, 800m, 1,500m, 5,000m,
10,000m, 100mH, 400mH, 3,000m장애물달리기,
20km경보, 4×100mR, 4×400mR, 높이뛰기,
장대높이뛰기, 멀리뛰기, 세단뛰기, 포환던지기,
원반던지기, 해머던지기, 창던지기, 7종 경기
남·여
(1)
4×400m 믹스릴레이
요
2
과업개요
ㅇ (사 업 명) 2025 구미아시아육상경기선수권대회 종합정보시스템 및 홈페이지 등 구축 용역
ㅇ (사업기간) 계약일 ~ 2025. 6. 6.(금)
ㅇ (사업위치) 시민운동장 일원


---

### openai answer
retriever A 점수: 3점
질문한 개최 기간이 “2025. 5. 27.(화) ~ 5. 31.(토) 5일간”으로 정확히 명시되어 있어 정답과 완전히 일치함.

retriever B 점수: 3점
A와 동일하게 개최 기간 “2025. 5. 27.(화) ~ 5. 31.(토) 5일간”이 명시되어 있어 정답과 완전히 일치함. 

============================================================
### query:
구미 아시아육상경기선수권대회 홈페이지에서 제공되는 주요 콘텐츠 항목에는 무엇이 포함되는가?

### answer:
대회소개, 경기일정 및 결과, 메달순위, 참가선수 정보, 커뮤니티 등이 포함된다.

---

### retriever A:
-경기참가자(Start List), 경기결과(Official Result, Summary) PDF 파일 제공
<다음 정보를 홈페이지에 전송>
Level1
Level2
기능 설명
경기
일정
전체 경기 일정
대회기간 전체 일정 제공
일자별 경기 일정
일자별 세부 경기 일정 제공
세부 종목별 경기 일정
세부 종목별 경기 일정 제공
선수별 경기 일정
선수별 경기 일정 제공
메달
국가/선수별 메달 현황
국가/선수별 메달 현황 제공
종목별 메달 현황
종목별 메달현황 및 메달수상자 정보 제공
경기
정보
스타트 리스트
세부종목별 편성 조 및 출전선수 목록 제공
대진 정보
세부 종목별 대진정보 조회 및 출력
(경기진행 상태에 따른 결과 반영 포함)
경기 결과
각 조별, 종목별 경기 결과 조회 및 출력
선수
정보
국가별 선수 정보
국가별 참가선수 목록 및 정보 조회
세부 종목별 선수 정보
세부 종목별 참가선수 목록 및 정보 조회
순위 정보
세부종목별 참가선수 순위 정보 목록 조회
과업내용
명칭
대분류
대회 종합정보시스템 S/W 개발
소분류
보고서관리시스템 개발
과업내용
데이터 출력 기능
<PDF 및 Excel, 증서 페이지 등 출력>
-참가선수 신상정보
-국가별/세부 종목별 참가선수 리스트
-심판/임원 신상정보
-국가별/세부 종목별 심판/임원 리스트
-세부 종목별, 예결선 Start List 및 조 편성
-세부 종목별 경기 결과 (Live Result, Official Result, Summary)
-세부 종목별 메달 현황 및 신기록 현황
-참가증서
ː참가선수(종목별 1위~6위 증서 및 참가증서 필수)


---

### retriever B:
 운영 매뉴얼 또는 지침서 제공 및 시스템 운영 관계자 교육
ㅇ (현장IT팀 상주 및 시스템 운영)
 사전 등록, 경기 진행 및 결과 출력 시스템 등 운영
 대회시작 4일전~대회 종료일까지 경기장 상주
 경기장/비경기장별 운영 인원 배치
ㅇ (홈페이지 구축)
 메뉴 구성도(안)
아시아육상경기선수권대회 메인
대회소개
소식지
경기일정/결과
메달순위
육상경기
커뮤니티
환영인사
공지사항
경기일정
국가별순위
경기종목
안내사항
대회개요
보도자료
행사일정
선수별순위
참가선수
아카이브
대회상징물
사진자료
시설안내
유실물센터
조직위안내
자주묻는질문
홍보영상
자원봉사자
찾아오시는 길
구미시 안내
메뉴 상세설명(안)
메뉴명
부메뉴
주요 내용
메인 화면
◦2025 구미 아시아육상경기선수권대회를 대표할 수 있는 디자인 (공식 상징물 활용 등)
◦활용빈도가 높은 메뉴 배치 및 팝업배너
대회 소개
환영인사
◦대회 개최에 대한 인사 말씀
대회개요
◦대회명칭, 대회기간, 대회장소, 참가규모, 참가국, 참가인원, 종목, 주최, 주관 등
대회상징물
◦대회마크, 마스코트, 포스터, 슬로건
조직위안내
◦조직위원회·아시아육상추진단 조직도
홍보영상
◦홍보영상, 유튜브 쇼츠 영상, 축하 메시지 등
찾아오시는 길
◦경기장 위치, 교통수단별 길 안내
소식지
공지사항
◦대회 관련 공지사항 등
보도자료
◦대회 관련기사 스크랩 자료
사진자료
◦개·폐회식 사진, 각 종목별 경기 이미지 게시
경기일정
/
결과
경기일정
◦경기장 및 개최 일자, 시간, 라운드, 진행상태
◦각 종목별 경기결과, 종합순위
행사일정
◦개·폐회식 정보 및 식순 안내


---

### openai answer
retriever A 점수: 2점
- 경기일정/결과, 메달 현황, 참가선수 정보 등은 있으나 ‘대회소개’, ‘커뮤니티’ 항목이 명시되지 않아 일부만 포함됨.

retriever B 점수: 3점
- 홈페이지 메뉴에 대회소개, 경기일정/결과, 메달순위, 참가선수, 커뮤니티가 모두 명시되어 있어 정답 구성요소를 완전히 포함함. 

============================================================
### query:
원자력 선량평가시스템의 성능 요구사항 중 질의응답 시간 기준은 어떻게 정의되어 있는가?

### answer:
일반 조회 요청은 3초 이내 처리되어야 하며, 10초 이상 소요되는 작업은 사용자에게 팝업 메시지를 제공해야 한다.

---

### retriever A:
질의·응답시간
요구사항
상세설명
정의
질의·응답시간
세부
내용
의는 사용자가 요청을 하는 시간으로부터 3000ms 내에 그 결과를 보여줘야 함
❍ 임의의 선택 기준이 허용되는 대량의 데이터에 대한 질의 및 고용량의 멀티미디어 파일을 가지고 있는 프로그램에서는 적절한 처리 상태를 표시하여야 함
산출정보
성능측정 결과 내역서
관련 요구사항
요구사항 고유번호
PER-002
요구사항 명
오류 응답시간
요구사항
상세설명
정의
오류 응답시간
세부
내용
시스템에 데이터 입력 등의 사용 중에 발생한 모든 오류는 원인이 발생한지 3000ms 이하로 사용자에게 오류 메시지를 보여줘야 함
산출정보
성능측정 결과 내역서
관련 요구사항
요구사항 고유번호
PER-003
요구사항 명
통계 처리 사전 경고
요구사항
상세설명
정의
통계 처리 사전 경고
세부
내용
리 시 5,000ms 이상 지연되는 경우 연산 작업을 시작하는 시점에 예상되는 결과에 대해 사용자에게 메시지로 알려야 함
산출정보
성능측정 결과 내역서
관련 요구사항
요구사항 고유번호
PER-004
요구사항 명
부하증가 대비 성능유지
요구사항
상세설명
정의
부하증가 대비 성능유지
세부
내용
시스템의 경우 동시 접속자(사용자)는 500명 이상 접속가능 하여야 하며, 이에 따른 처리속도 등 성능이 저하되지 않아야 함
❍ 포털 시스템의 경우 동시 접속자(사용자)는 3000명 이상 접속가능 하여야 하며, 이에 따른 처리속도 등 성능이 저하되지 않아야 함
산출정보
성능측정 결과 내역서
관련 요구사항
요구사항 고유번호
PER-005
요구사항 명
오프라인 배치성 업무 응답시간
요구사항


---

### retriever B:
성능 요구사항
상세
설명
정의
정보시스템 업무 응답시간 목표
세부
내용
□ 요청(질의) 응답시간
o 모든 질의는 사용자가 요청을 하는 시간으로부터 3초 내에 최초의 결과 값을 보여줘야 함
□ 이 요구사항은 임의의 선택 기준이 허용되는 대량의 데이터(2GB 이상)에 대한 질의에는 적용되지 않음
산출정보
o SW개발에 따른 산출물
성능 요구사항
상세
설명
정의
정보시스템 업무 응답시간 목표
세부
내용
□ 요청(질의) 응답시간
o 모든 질의는 사용자가 요청을 하는 시간으로부터 3초 내에 최초의 결과 값을 보여줘야 함
□ 이 요구사항은 임의의 선택 기준이 허용되는 대량의 데이터(2GB 이상)에 대한 질의에는 적용되지 않음
산출정보
o SW개발에 따른 산출물
성능 요구사항
응낙수준
필수
요구
사항
상세
설명
정의
사용자 요구에 대한 웹페이지 디스플레이 반응 시간
세부
내용
○ 시스템은 정상 상태에서 사용자의 질의 요청에 대한 결과 페이지를 화면에 출력할 때 3초 이내 보장
○ 평균 응답시간은 시스템을 사용하는 동시 사용자가 최대 사용자 수의 90%를 초과하는 경우에는 미적용
산출정보
관련요구사항


---

### openai answer
retriever A 점수: 2점
- 3초 이내 응답 기준은 포함되어 있으나, 10초 이상 소요 시 팝업 메시지 제공이라는 구체 기준은 없음(장시간 처리 시 상태표시/메시지 언급과 5초 경고만 존재).

retriever B 점수: 2점
- 모든 질의 3초 이내 응답 기준은 명확히 포함되어 있으나, 10초 이상 작업 시 팝업 안내 기준은 없음. 

============================================================
### query:
원자력 선량평가시스템의 포트란 모듈 개선 요구사항에서 추가로 고려해야 하는 평가 요소는 무엇인가?

### answer:
 ICRP60 기준에 따른 23개 장기를 고려한 평가장기 추가가 요구된다.

---

### retriever A:
가 방법
평가부문
평가항목
평가기준
배점
한도
정량
평가
(10)
경영상태
경영상태(신용상태등급) 평가
10
전략 및 방법론
(22)
사업
이해도
사업의 특성 및 목표에 부합하는 사업전략을 제시하고 있는지를 평가한다.
7
추진전략
개발업무 수행 시 일정 및 위험요소를 고려하여 얼마나 타당한 추진 전략을 수립하였는지 평가한다.
5
적용기술
사업에서 적용하고자 하는 기술이 향후 확장성을 고려하여 현실적으로 실현 가능하게 제시되어 있는지를 평가한다.
4
표준프레임 워크 적용
제안사는 사업에 네팔 정보통신기술부 프레임워크 기반의 공통 컴포넌트의 유형 및 사용계획을 기술하고 사용성 향상을 위한 추가 컴포넌트에 대해 적용방안을 제시하여야한다.
3
개발
방법론
사업에 적정한 개발 방법론이 제시되었는지 평가하고, 실제 적용 사례 및 경험을 바탕으로 단계별 산출물이 제시되었는지 여부를 평가한다.
3
기술 및 기능
(26)
시스템 장비 구성 요구사항
요구 규격을 충족하는 장비 제안 여부와 현 시스템과의 인터페이스 및 확장 가능성 여부를 평가한다. 또한 도입 장비의 설치 및 공급 계획, 유지관리에 대한 방안이 구체적으로 기술되어 있는가를 평가한다.
7
기능 요구사항
기능 요구사항·기대사항·제약사항 등을 파악한 후 구현 방안이 구체적으로 기술되어 있는지를 평가한다. 또한 제안한 방안 및 기술이 적용 가능한지를 평가한다.
7
데이터
요구사항
데이터 요구사항 분석을 바탕으로 데이터 전환계획 및 검증 방법, 데이터 오류 발생 시 처리 방안이 구체적으로 제시되어 있는지를 평가한다.
4
보안
요구사항


---

### retriever B:
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항


---

### openai answer
retriever A 점수: 0점
(원자력/선량평가/포트란/ICRP60/23개 장기 등 핵심 키워드나 답의 내용과 관련된 정보가 전혀 없음)

retriever B 점수: 0점
(“기능 요구사항” 반복뿐이며 원자력 선량평가, ICRP60, 23개 장기 등 답에 필요한 정보가 없음) 

## size 2000

============================================================
### query:
아이돌봄 인적성 검사 시스템 운영 사업의 주요 목적은 무엇인가?

### answer:
인적성 검사 수행을 지원하고, 돌봄활동과 연계한 사후관리 및 인력 검증을 강화하는 것이다.

---

### retriever A:
◦원할한 인적성 검사 수행을 위한 정보시스템 지원 필요
◦돌봄활동과 연계한 사후관리 기반을 강화하여 이용자 만족도 향상
◦아이돌봄인력 인·적성에 기초한 돌봄인력 관리 강화 필요
◦정보시스템 개인정보보호법 준수를 위한 보호조치
2
사업 개요
□ 사 업 명: 아이돌봄인력 인적성 검사 정보시스템 운영
- 2024. 정보시스템 구축에 따른 유지보수 사업
□ 사업기간: 계약일 ~ 2025. 12. 31.
□ 소요예산: 172,700,000원(부가세 포함)
□ 추진방식: 제한경쟁입찰(협상에 의한 계약)
3
사업 범위
□ 인·적성 프로그램 기능수정 및 안정화 기술지원
◦인·적성 시스템의 안정적인 정보화 지원을 위한 상시 유지관리
체계 지원
◦인·적성 검사 변경에 따른 프로그램 수정 및 결과 반영
◦웹 접근성, 기술적 보안 등 전자정부 가이드라인 준수
□ 개인정보보호 및 정보보안 강화
◦개인정보보호 및 정보보호 강화 조치(방안)
◦정보시스템 운영을 위한 물리적, 기술적 보안 대책 제시
□ 안정적인 클라우드 환경 운영 및 제공
◦클라우드 무중단 환경 제공
◦클라우드 환경 운영을 위한 기술적 유지관리 지원
◦시스템 용을 위한 SW 및 외부 서비스 연계
4
기대 효과
□ (안전한 돌봄 환경 조성) 아이돌봄인력 선별 및 검증 강화 통한 안심하고 맡길 수 있는 돌봄서비스 제공
□ (효율성 제고) 아이돌봄인력 대상 인적성 검사를 편리하게 실시하여 돌봄인력 검증 절차 효율성 제고
□ (정책지원 및 제도개선 지원) 체계적인 검사 결과를 축적하고 정책 수립지원을 위한 기초자료를 제공, 지속 가능한 아이돌봄서비스 제도개선에 활용
□ (안전한 돌봄 환경 조성) 아이돌봄인력 선별 및 검증 강화 통한 안심하고 맡길 수 있는 돌봄서비스 제공
□ (효율성 제고) 아이돌봄인력 대상 인적성 검사를 편리하게 실시하여 돌봄인력 검증 절차 효율성 제고
□ (정책지원 및 제도개선 지원) 체계적인 검사 결과를 축적하고 정책 수립지원을 위한 기초자료를 제공, 지속 가능한 아이돌봄서비스 제도개선에 활용
□ (돌봄인력 관리) 데이터를 기반으로 하는 아이돌봄인력에 대한 효율적 관리·감독 가능
Ⅱ. 사업 현황
1
업무현황
□ 인·적성검사 실시 정보시스템 지원
ㅇ 인·적성검사 및 실시를 위한 정보 시스템 제공
ㅇ 정보시스템 안정화 및 운영지원
□ 인·적성검사 운영
ㅇ 인적성 검사 실시 및 수행 방법 안내
ㅇ 인적성 검사 수행 모니터링 및 결과 분석
ㅇ 검사 실시 운영자 권한 부여 및 인증 관리
2
정보화 현황
□ 시스템 구성도
□ 정보시스템 구성도 요청 시 별도 공개
※ 행정기관 및 공공기관 정보시스템 구축 및 운영 지침(행정안전부 고시 제 2024-66호, 2024.8.28.) 제 17조(제안요청서 보안사항 등)에 따라, 시스템 구성도 및 상세자료 등은 공개하지 않으며, 입찰에 참가하고자 하는 사업자의 요청이 있는 경우 보안서약서를 제출 후 담당자 입회하에 제한적으로 열람 가능
□ 시스템(장비)·소프트웨어 운영현황
AS-IS
TO-BE (2025년)
ㅇ인·적성 검사 관리기능 및 사용자화면 개발
※ 클라우드 환경 기반 개발
ㅇ인·적성 검사 기능 최적화 및 운영 기술지원
-인·적성 검사 고도화에 따른 수정 반영
(검사도구 4종)
※사업 운영에 따라 변경될수 있음
8. 제안서 보상 穠ȃ 44
9. 기타사항 舨ȃ 44
Ⅶ. 별첨
[별첨] 입찰서류 및 제안서 제출안내(온라인제출) 㜜ȃ 46
[별첨] 사업자 보안위규 처리기준 墄ȃ 71
[별첨] 보안 위약금 부과 기준 後ȃ 73
[별첨] 누출금지 대상정보 棬ȃ 74
[별첨] 기술적용계획표 濴ȃ 75
楴䵴
Ⅰ.사업 개요
1
추진배경
□ 추진배경
◦ 국정과제(46:촘촘한 아동돌봄체계 마련) 실현을 위한 돌봄인력 검증 강화
아이돌봄지원법 제7조(아이돌보미의 자격) 6의 ②항 아이돌보미가 되려는 사람은 여성가족부장관이 실시하는 적성ㆍ인성 검사를 받아야 한다.
◦ 아이돌봄인력 대상으로 온라인 정서·심리검사를 실시하고 결과를 반영하여 아동학대 예방 등 돌봄 활동 연계 등에 활용
◦아이돌봄인력의 직무적합도에 대한 서비스 이용자 요구 증가
◦개인정보보호법 제29조(안전조치의무), 시행령 제30조(개인정보 안전성 확보조치)
개인정보처리자는 개인정보가 분실〮도난〮유출〮위조〮변조 또는 훼손되지 아니하도록 내부 관리계획 수립, 접속기록 보관 등
대통령령으로 정하는 바에 따라 안전성 확보에 필요한 기술적〮관리적 및 물리적 조치를 하여야 한다.
□ 목적 및 필요성
◦원할한 인적성 검사 수행을 위한 정보시스템 지원 필요
◦돌봄활동과 연계한 사후관리 기반을 강화하여 이용자 만족도 향상
◦아이돌봄인력 인·적성에 기초한 돌봄인력 관리 강화 필요
◦정보시스템 개인정보보호법 준수를 위한 보호조치
2
사업 개요
□ 사 업 명: 아이돌봄인력 인적성 검사 정보시스템 운영
- 2024. 정보시스템 구축에 따른 유지보수 사업


---

### retriever B:
기대 효과
□ (안전한 돌봄 환경 조성) 아이돌봄인력 선별 및 검증 강화 통한 안심하고 맡길 수 있는 돌봄서비스 제공
□ (효율성 제고) 아이돌봄인력 대상 인적성 검사를 편리하게 실시하여 돌봄인력 검증 절차 효율성 제고
□ (정책지원 및 제도개선 지원) 체계적인 검사 결과를 축적하고 정책 수립지원을 위한 기초자료를 제공, 지속 가능한 아이돌봄서비스 제도개선에 활용
□ (돌봄인력 관리) 데이터를 기반으로 하는 아이돌봄인력에 대한 효율적 관리·감독 가능
Ⅱ. 사업 현황
1
업무현황
□ 인·적성검사 실시 정보시스템 지원
ㅇ 인·적성검사 및 실시를 위한 정보 시스템 제공
ㅇ 정보시스템 안정화 및 운영지원
□ 인·적성검사 운영
ㅇ 인적성 검사 실시 및 수행 방법 안내
ㅇ 인적성 검사 수행 모니터링 및 결과 분석
ㅇ 검사 실시 운영자 권한 부여 및 인증 관리
2
정보화 현황
□ 시스템 구성도
□ 정보시스템 구성도 요청 시 별도 공개
※ 행정기관 및 공공기관 정보시스템 구축 및 운영 지침(행정안전부 고시 제 2024-66호, 2024.8.28.) 제 17조(제안요청서 보안사항 등)에 따라, 시스템 구성도 및 상세자료 등은 공개하지 않으며, 입찰에 참가하고자 하는 사업자의 요청이 있는 경우 보안서약서를 제출 후 담당자 입회하에 제한적으로 열람 가능
□ 시스템(장비)·소프트웨어 운영현황
AS-IS
TO-BE (2025년)
ㅇ인·적성 검사 관리기능 및 사용자화면 개발
※ 클라우드 환경 기반 개발
ㅇ인·적성 검사 기능 최적화 및 운영 기술지원
-인·적성 검사 고도화에 따른 수정 반영
(검사도구 4종)
□ 시스템(장비)·소프트웨어 운영현황
AS-IS
TO-BE (2025년)
ㅇ인·적성 검사 관리기능 및 사용자화면 개발
※ 클라우드 환경 기반 개발
ㅇ인·적성 검사 기능 최적화 및 운영 기술지원
-인·적성 검사 고도화에 따른 수정 반영
(검사도구 4종)
※사업 운영에 따라 변경될수 있음
ㅇ무중단 클라우드 기반 제공
ㅇ클라우드 안정화 운영기반 및 기술지원
SW 도입 현황
SW 명칭
용도
비고
리포팅 툴
OZ Report
인·적성 결과지 출력
발주처 제공
□ 시스템 장비 현황 요청 시 별도 공개
※ 행정기관 및 공공기관 정보시스템 구축 및 운영 지침(행정안전부 고시 제 2024-66호, 2024.8.28.) 제 17조(제안요청서 보안사항 등)에 따라, 시스템 구성도 및 상세자료 등은 공개하지 않으며, 입찰에 참가하고자 하는 사업자의 요청이 있는 경우 보안서약서를 제출 후 담당자 입회하에 제한적으로 열람 가능
Ⅲ. 사업 추진방안
1
추진 목표
□ `24년 개발된 인·적성 검사 운영을 지원하고, 데이터 수집 및 분석을 위한 정보체계 관리
□ 인·적성 검사 정보시스템 안정적 기반 지원 및 보안 강화
2
추진체계
□ 추진체계도
발주기관
<한국건강가정진흥원>
협조부서
<경영지원부/
소통협력실>
주관부서
<돌봄지원부>
관련부처
<여성가족부>
주관사업자
<사업수행업체>
□ 추진 주체별 역할
구 분
주요 업무 내용
주관부서
ㅇ 사업총괄 관리 및 시행
ㅇ 사업계획서 수립 및 제안요청서 작성
ㅇ 개발내역 업무정의 및 과업내용 제시
ㅇ 개발관련 자료제공 및 시스템 인수
ㅇ 사업수행 관리 및 검사, 결과 평가
협조부서
2. 과업 목적
ㅇ 수동 중심 검사방식을 디지털·자동화 품질검사가 가능한 도구개발을 통하여 품질 검사시간 단축 및 업무효율성 제고
ㅇ 항해용 간행물 종류 및 품질관리 항목별 오류 정보의 체계적인 관리를 통하여 신속한 품질관리 업무지원
2. 추진 배경 및 필요성
□ 기관생명윤리위원회(IRB) 업무프로세스 구축
ㅇ 기관 생명윤리위원회 규정 및 지침에 의한 위원회 업무 통합 관리 시스템 구축 및 절차 확립
ㅇ 임상연구 심의, 심의결과 통보, 변경신청관리, 결과보고서 관리
□ 동물실험계획 업무 프로세스 구축
ㅇ 기관 동물실험운영규정에 위한 동물실험 업무 통합 관리 시스템 구축 및 절차 확립
ㅇ동물실험계획심의, 승인, 실험계획변경, 실험동물반입, 종료보고서 관리, 사체처리결과서
□ 국가연구개발사업 연구개발비 사용기준 및 회계기준 준수
ㅇ 사전 회의 개최(안) 신청 및 지급신청 연계
ㅇ 구매전용카드 관리, 전자세금계산서 발급신청
□ 기존 시스템 업무 프로세스 개선 및 기능 고도화
ㅇ 연구계획서 기술완성도 목표 관리
ㅇ 연구노트 대장 관리 기능 개선


---

### openai answer
retriever A 점수: 3점
(원활한 인적성 검사 수행 지원, 돌봄활동 연계 사후관리 강화, 인력 검증/관리 강화 등 답안의 핵심 목적이 모두 명시됨)

retriever B 점수: 2점
(인적성 검사 수행 지원과 인력 검증 강화 내용은 있으나, 돌봄활동과 연계한 사후관리 명시는 부족함) 

============================================================
### query:
아이돌봄 시스템에서 개인정보 보호를 위해 요구되는 조치는 무엇인가?

### answer:
접속기록 관리, 내부관리계획 수립 등 기술적·관리적 보호조치를 수행해야 한다.

---

### retriever A:
- 개인정보의 기술절·관리적 보호조치 기준(개인정보위)
- 표준 개인정보 보호 지침(개인정보위)
가명정보의 결합 및 반출 등에 관한 고시(개인정보위)
개인정보 영향평가에 관한 고시(개인정보위)
※ 고시에 따라 아래 기준 이상의 개인정보 처리 시 발주처에서 개인정보 영향평가를 발주하며 영향평가결과를 반영하여 시스템을 구축해야 함
내 용
1
구축‧운용 또는 변경하려는 개인정보파일로서 5만명 이상의 정보주체에 관한 민감정보 또는 고유식별정보의 처리가 수반되는 개인정보파일
2
구축‧운용하고 있는 개인정보파일을 해당 기관 내부 또는 외부에서 구축‧운용하고 있는 다른 개인정보파일과 연계하려는 경우로서 연계결과 50만명 이상의 정보주체에 관한 개인정보가 포함되는 개인정보파일
3
구축‧운용 또는 변경하려는 개인정보파일로서 100만명 이상의 정보주체에 관한 개인정보파일
4
개인정보 영향평가를 받은 후에 개인정보 검색체계 등 개인정보파일의 운용체계를 변경하려는 경우 그 개인정보파일
○ 개인정보 안전성 확보조치 기준 이행·반영
- 개인정보 열람 시 접속기록(최소 2년 이상) 보관·관리
- 개인정보 접근권한 부여·변경·말소 이력(최소 3년 이상) 보관·관리
- 고유식별번호 및 비밀번호 생체인식정보는 안전한 암호 알고리즘을 사용 하여 저장
○ 데이터의 보안 강화를 위해 SSL서비스 도입
○ 개인정보보호를 위한 DB 암호화하여 자료 유출방지
- 인증서 기반(GPKI) 로그인 방식 적용 후 ID/PW 로그인
- 정보보안 강화를 위한 시스템에 각종 패치, 환경설정 지원 등
◦ 보안 요구사항
요구사항 고유번호
SER-001
요구사항 명칭
개인정보 처리
요구사항 분류
보안 요구사항
요구사항
상세설명
정의
개인정보 처리 및 암호화
세부
내용
ㅇ 개인정보를 보유할 경우에는 관련 개인정보를 암호화하여야 함
ㅇ 서비스의 보안성 강화를 위하여 SSL 보안 통신을 실시하여야 함
ㅇ 사업의 암호화 대상을 식별하고 암호화 도구를 활용하여 암복호화 방안을 마련해야 함
- 고유식별정보 : 주민번호, 여권번호, 운전면허정보 등
ㅇ개인을 식별할 수 있는 정보(사용자 인증 정보, 패스워드 등)를 운영DB 또는 개발DB에 저장할 경우, 암호화하여 저장하며, 소스코드에 직접 하드코딩하지 않음(패스워드는 단방향 암호화 처리, SHA-256 이상 암호화 알고리즘 사용)
ㅇ 개인정보 취급자가 개인정보처리 시스템에 접속하는 경우 접속 로그를 시스템에 최소 1년 이상 기록 관리해야 함(5만명 이상의 정보주체에 관하여 개인정보를 처리하거나 고유식별정보 또는 민감정보를 처리하는 개인정보처리시스템의 경우에는 2년 이상 보관)
◈ 필수 관리 항목
• 개인정보취급자 식별정보 : ID ୴Ā • 날짜 및 시간 : 접속일시
• 접속자 IP 주소 : 접속자 정보
• 처리한 정보주체 정보 : 정보주체 식별정보(계정, 회원번호 등)
• 수행업무 : 열람, 수정, 삭제, 인쇄, 입력, 다운로드 등
ㅇ 개인정보 취급자가 개인정보를 처리할 경우 해당 접근 로그를 시스템에 기록할 것
ㅇ 개인정보처리방침에 따라 개인정보를 처리/파기해야 하며 백업데이터 생성 금지
개인정보 암호화
(제6조)
◦암호화 대상 : 고유식별정보(주민등록번호, 여권번호, 운전면허번호, 외국인등록번호), 비밀번호, 바이오정보
◦암호화 기준
구 분
암 호 화 기 준
송․수신시
모든 정보 암호화 전송
저장시
비밀번호,
바이오정보
비밀번호는 단방향 암호화,
바이오정보는 양방향 암호화
고유식별정보
ⓛ 인터넷 구간,공통(DMZ) 구간 저장시 : 의무적 암호화
② 내부망 저장시 : 위험도 분석결과에 따라 암호화 적용범위 결정
(※ 영향평가 대상 공공기관은 영향평가 결과에 따름)
* 고유식별정보가 유출・변조・훼손되지 않도록 연1회 이상 취약점 점검
주민등록번호
암호화 알고리즘 적용 권고(’16.1.1, 의무시행)
◦암호화 방법 : 공공기관은 국가 암호화 알고리즘, 그 외는 자율
◦암호화 적용 : 고유식별정보 암호화 시 지침 시행일로부터 3개월 이내에 암호화 계획 수립, 계획 수립일로부터 12개월 이내에 암호화 적용
접속기록 보관·점검(제7조)
◦개인정보처리시스템 접속기록은 6개월 이상 보관
※ DBMS의 접속기록을 반기별 1회 이상 점검
◦접속기록 반기별 점검 의무화
악성프로그램 등 방지(제8조)
◦보안 프로그램(백신 S/W 등) 설치 및 업데이트 ※ 자동 업데이트 또는 일 1회 이상
물리적 접근 방지
(제9조)
◦별도의 개인정보 물리적 보관 장소에 대한 출입통제 등
개인정보의 파기
(제10조)
◦복원이 불가능한 개인정보의 파기방법 구체화
부칙
◦전산센터․클라우드컴퓨팅센터 등의 운영환경에서의 안전조치
2. 개인정보처리시스템 위탁 관리 철저
❍ 위탁목적 등 문서화(「개인정보보호법」제26조)


---

### retriever B:
6. 그 밖에 개인정보의 보호를 위하여 필요한 사항
-295-
1. 개인정보의 안전성 확보조치 철저(「개인정보보호법」제29조)
❍ 개인정보처리자는 개인정보가 분실·도난·유출·변조 또는 훼손되지 않도록 다음 각 호의 안전성 확보조치를 하여야한다.
구 분
주 요 내 용
내부관리계획(제3조)
◦내부관리계획 포함사항 명시 ※개인정보 보호책임자 지정, 개인정보취급자 교육 등
◦개인정보 처리 업무 위탁 시 관리·감독 사항
접근권한 관리(제4조)
◦개인정보처리시스템의 접근권한 부여․변경․말소 기록 최소 3년간 보관
◦안전한 비밀번호 작성규칙 수립․적용
접근통제(제5조)
◦접근통제 시스템 설치․운영 의무화 ※접근통제 시스템 : 침입차단(Firewall), 침입탐지(IDS) 기능 포함
◦주민번호 처리 시 추가 정보 확인, 모바일 기기 보호조치 조항, 홈페이지에서 고유식별정보를 처리할 경우 년 1회 취약점 점검 조치
개인정보 암호화
(제6조)
◦암호화 대상 : 고유식별정보(주민등록번호, 여권번호, 운전면허번호, 외국인등록번호), 비밀번호, 바이오정보
◦암호화 기준
구 분
암 호 화 기 준
송․수신시
모든 정보 암호화 전송
저장시
비밀번호,
바이오정보
비밀번호는 단방향 암호화,
바이오정보는 양방향 암호화
고유식별정보
ⓛ 인터넷 구간,공통(DMZ) 구간 저장시 : 의무적 암호화
보안 요구사항
상세
설명
정의
개인정보의 안전성 확보를 위한 기술적 조치사항
세부
내용
 개인정보를 처리함에 있어 『개인정보 보호법』, 『개인정보의 안전성 확보조치 기준』 등 관련 법규 및 규정을 준수하여야 함
 접근 권한 관리
- 개인정보처리시스템에 접속할 수 있는 사용자계정을 발급하는 경우, 개인정보취급자 별로 사용자계정을 발급하여야 하며, 취급자의 역할에 따라 차등부여할 수 있어야 함
- 개인정보 처리내역에 대한 책임 추적성을 확보하기 위하여 접근권한 부여, 변경, 말소에 대한 내역을 이력 관리 또는 조회가 가능하여야 하며, 해당 기록을 최소 3년간 보관할 수 있도록 구현해야 함
예) 신청자 정보, 신청일시, 승인자 및 발급자 정보, 신청 및 발급 사유 등
- 개인정보처리시스템에 접근하여 개인정보를 생성수정삭제열람 등을 하고자 하는 경우 다음 사항을 포함하는 정보이용 내역을 기록하고 남길 수 있도록 시스템을 구현하여야 함.
- 개인정보의 안전성 확보조치 기준(개인정보보호위원회고시) 준용
❑ 각 기능에 명시된 개인정보는 반드시 암호화하여 DB에 저장하여야 함
❑ 관리자인 경우에도 화면에 해당 정보를 그대로 노출시키지 않아야 함
❑ 암호화는 재단에서 제공하는 DB암호화 솔루션을 적용
산출정보
서, 개인정보처리위탁계약서, 수탁사 개인정보 안전성확보조치 관련 관리 감독 보고서
4. 개인정보처리 준수사항
□ 개인정보의 안전성 확보조치 철저(개인정보보호법 제29조)
❍ 개인정보처리자는 개인정보가 분실ㆍ도난ㆍ유출ㆍ변조 또는 훼손되지 않도록 다음 각 호의 안전성 확보조치를 하여야함.
구 분
주 요 내 용
내부관리계획(제4조)
▪내부관리계획 포함사항 명시(※ 개인정보 보호책임자 지정, 개인정보취급자 교육 등)
접근권한 관리(제5조)
▪개인정보처리시스템의 접근권한 부여ㆍ변경ㆍ말소 기록 최소 3년간 보관
접근통제 시스템 설치ㆍ운영(제6조)
▪접근통제 시스템 설치ㆍ운영 의무화
※ 접근통제 시스템 : 침입차단(Firewall), 침입탐지(IDS) 기능 포함
개인정보 암호화
(제7조)
▪암호화 대상 : 고유식별정보(주민등록번호, 여권번호, 운전면허번호, 외국인등록번호), 비밀번호, 바이오정보
▪암호화 기준
구 분
암 호 화 기 준
송ㆍ수신시
모든 정보 암호화 전송
저장시
비밀번호,
바이오정보
▪비밀번호는 단방향 암호화,
▪바이오정보는 양방향 암호화
고유식별
정보
ⓛ 인터넷 구간, 공통(DMZ) 구간 저장시 : 의무적 암호화
4. 데이터 취합 시 개인정보에 대한 보안 방안을 제시하여야 함
⑤ 개인정보에 대한 접근 제한 등 안전성 확보 조치에 관한 사항
❍ 개인정보의 안전성 확보조치 기준(개인정보보호위원회 고시 제2021-2호) 준수
❍ 표준 개인정보 보호지침(개인정보보호위원회 고시 제2020-1호) 준수
❍ 개인정보 수집제공 동의서 작성 가이드라인(2018)
❍ 주민등록번호 수집 금지 제도 가이드라인(2014)
❍ 개인정보 수집 최소화 가이드라인(2020.12)
❍ 홈페이지 개인정보 노출방지 안내서(2020.12)
❍ 웹 취약점 점검 결과와 취약점 조치 완료 보고서 제출
❍ 그 밖에 명시되지 않은 관련법규 및 가이드라인은 본 과업을 수행함에 있어 표준을 준수
❍ 설계단계 완료 이전에 위 표준(지침)이 변경되거나 새 표준이 발표될 때는 변경된 내용 및 새 표준을 반영하여야 함
❍ 보안관리자를 지정하고, 프로그램 소스코드 및 관련 S/W의 유출을 방지하기 위해 디스크, CD, 출력물은 별도 관리하여야 함.
❍ 불의의 사고로 인하여 시스템 및 파일이 피해를 입더라도 최근에 백업한 시점의 내용으로 복구할 수 있는 백업정책 수립 및 구현
❍ 긴급 재해·재난 발생 시 데이터를 보호하며 시스템 Off와 전원을 차단 할 수 있는 시스템 구현
2. DB보안 및 개인정보보호법 준수
❍ 사용자 인증 정보, 주민등록번호, 비밀번호, 계좌번호 등 개인을 식별할 수 있는 정보를 운영DB 또는 개발DB에 저장할 경우, 암호화하여 저장하며 소스코드에 직접 하드코딩하지 않아야 함
❍ 사용자 정보 등 개인정보 취급 시 반드시 개인정보보호법과 관련 지침을 준수함
❍ DB암호화 솔루션을 통하여 저장되는 콘텐츠 중 개인정보 및 보안을 요구하는 데이터들에 대한 암호화 및 관리 기능 적용


---

### openai answer
retriever A 점수: 2점
(접속기록 보관·점검 등 로그 관리와 기술적·관리적 보호조치 기준 준수는 포함되어 있으나, 내부관리계획 수립에 대한 명시적 언급이 없음)

retriever B 점수: 3점
(내부관리계획 수립(제3조) 명시, 접근·이용 내역 기록 등 로그 관리 요구, 기술적·관리적 보호조치 전반이 포함되어 정답 요소를 모두 충족) 

============================================================
### query:
2025 구미 아시아육상경기선수권대회의 개최 기간은 언제인가?

### answer:
2025년 5월 27일부터 5월 31일까지 총 5일간 개최된다.

---

### retriever A:
종합정보시스템 및 홈페이지 등 구축 용역 제안요청서
1
대회개요
아시아의 꿈, 구미에서 세계로
 The dream of ASIA, From GUMI to the World 
비전
ㅇ (대 회 명) 2025 구미아시아육상경기선수권대회
ㅇ (기 간) 2025. 5. 27.(화) ~ 5. 31.(토) 5일간
ㅇ (장 소) 구미시민운동장 일원
ㅇ (규 모) 아시아 45개국 1,200여명 / 선수단 및 임원
ㅇ (주 최) 아시아육상연맹(Asian Athletics, AA)
ㅇ (주 관) 2025 구미아시아육상경기선수권대회 조직위원회
ㅇ (경기종목) 45개 종목
구 분
종 목 명
남자
(22)
100m, 200m, 400m, 800m, 1,500m, 5,000m,
10,000m, 110mH, 400mH, 3,000m장애물달리기,
20km경보, 4×100mR, 4×400mR, 높이뛰기,
장대높이뛰기, 멀리뛰기, 세단뛰기, 포환던지기,
원반던지기, 해머던지기, 창던지기, 10종 경기
여자
(22)
100m, 200m, 400m, 800m, 1,500m, 5,000m,
10,000m, 100mH, 400mH, 3,000m장애물달리기,
20km경보, 4×100mR, 4×400mR, 높이뛰기,
장대높이뛰기, 멀리뛰기, 세단뛰기, 포환던지기,
원반던지기, 해머던지기, 창던지기, 7종 경기
남·여
(1)
4×400m 믹스릴레이
요
2
과업개요
ㅇ (사 업 명) 2025 구미아시아육상경기선수권대회 종합정보시스템 및 홈페이지 등 구축 용역
ㅇ (사업기간) 계약일 ~ 2025. 6. 6.(금)
ㅇ (사업위치) 시민운동장 일원
2024. . .
대표자 : (인)
대리인 : (인)
2025 구미아시아육상경기선수권대회 조직위원장 귀하
※ 첨부서류 : 재직증명서, 신분증
※ 위임장에 사용하는 도장은 대표자 인감과 같아야 함.
※ 대리 접수로 인하여 발생하는 모든 문제는 대표자 본인이 책임짐
【 서식 3 】
재 직 증 명 서
성 명
생 년 월 일
부서/직위
소 속
주 소
재직기간
위와 같이 당사에 재직 중임을 증명합니다.
2024. . .
ྠĀ ྠĀ ྠĀ ྠĀ ྠĀ ྠĀ 업 체 명 :
ྠĀ ྠĀ ྠĀ ྠĀ ྠĀ ྠĀ 주 소 :
ྠĀ ྠĀ ྠĀ ྠĀ ྠĀ ྠĀ 대 표 자 : (인)
2025 구미아시아육상경기선수권대회 조직위원장 귀하
【 서식 4 】
사 용 인 감 계
사용인감
는(은) 2025 구미아시아육상경기선수권대회 조직위원회와 거래함에 있어 입찰, 계약 및 이에 수반되는 모든 행위에 대하여 위의 인감을 사용하고자 하며, 동 인감을 사용함으로써 민․형사상의 문제가 발생할 경우 이(가) 책임을 질 것을 확인합니다.
붙임 : 인감(법인)증명서 1부. 끝.
2024년 월 일
ྠĀ ྠĀ ྠĀ ྠĀ ྠĀ ྠĀ 주 소 :
ྠĀ ྠĀ ྠĀ ྠĀ ྠĀ ྠĀ 상 호 :
ྠĀ ྠĀ ྠĀ ྠĀ ྠĀ ྠĀ 대표자 : (인)
2025 구미아시아육상경기선수권대회 조직위원장 귀하
【 서식 5 】
서 약 서
당사는 2025 구미아시아육상경기선수권대회 종합정보시스템 및 홈페이지 등 구축 용역 참가신청과 관련하여 모든 서류와 증빙자료를 신의와 성실의 원칙에 입각하여 작성 제출하며, 아래 내용을 준수할 것을 확약하고 이에 서약서를 제출합니다.
2025 구미아시아육상경기선수권대회 조직위원장 귀하
【 서식 5 】
서 약 서
당사는 2025 구미아시아육상경기선수권대회 종합정보시스템 및 홈페이지 등 구축 용역 참가신청과 관련하여 모든 서류와 증빙자료를 신의와 성실의 원칙에 입각하여 작성 제출하며, 아래 내용을 준수할 것을 확약하고 이에 서약서를 제출합니다.
1. 당사는 제안요청서에 기술된 내용 및 절차를 인정하고, 관련 지침을 성실히 준수하겠으며, 본 지침을 위해할 시 어떠한 불이익도 감수 하겠습니다.
2. 본 용역과 관련한 사업비는 변동될 수 있으므로 변동금액에 대하여 일체의 이의를 제기하지 않겠습니다.
3. 제안서 평가를 위해 구성된 평가위원회의 심사방법 및 심사 기준, 평가결과에 어떠한 이의제기도 하지 않겠습니다.
2024년 월 일
신청자
업 체 명
:
대표자
:
(인)
2025 구미아시아육상경기선수권대회 조직위원장 귀하
【 서식 6 】
보 안 서 약 서
본인은 2025 구미아시아육상경기선수권대회 종합정보시스템 및 홈페이지 등 구축 용역을 수행함에 있어 다음 사항을 준수할 것을 엄숙히 서약합니다.
1. 본인은 위 용역과 관련하여 귀 조직위원회에서 수령한 자료 및 정보를 관리함에 철저를 기하겠으며, 무단으로 복제하거나 유출하지 아니 할 것을 서약합니다.
2. 본인은 용역 기간은 물론 이후에도 일체 타인에게 누설하지 아니할 것을 서약하며, 용역을 성실히 수행하겠습니다.
3. 본인은 위에서 서약한 사항을 위반함으로써 부과되는 민․형사상의 책임을 감수하겠음을 서약합니다.
2024년 월 일
서약자 주 소 :
업체명 :
대표자 : (인)


---

### retriever B:
종합정보시스템 및 홈페이지 등 구축 용역 제안요청서
1
대회개요
아시아의 꿈, 구미에서 세계로
 The dream of ASIA, From GUMI to the World 
비전
ㅇ (대 회 명) 2025 구미아시아육상경기선수권대회
ㅇ (기 간) 2025. 5. 27.(화) ~ 5. 31.(토) 5일간
ㅇ (장 소) 구미시민운동장 일원
ㅇ (규 모) 아시아 45개국 1,200여명 / 선수단 및 임원
ㅇ (주 최) 아시아육상연맹(Asian Athletics, AA)
ㅇ (주 관) 2025 구미아시아육상경기선수권대회 조직위원회
ㅇ (경기종목) 45개 종목
구 분
종 목 명
남자
(22)
100m, 200m, 400m, 800m, 1,500m, 5,000m,
10,000m, 110mH, 400mH, 3,000m장애물달리기,
20km경보, 4×100mR, 4×400mR, 높이뛰기,
장대높이뛰기, 멀리뛰기, 세단뛰기, 포환던지기,
원반던지기, 해머던지기, 창던지기, 10종 경기
여자
(22)
100m, 200m, 400m, 800m, 1,500m, 5,000m,
10,000m, 100mH, 400mH, 3,000m장애물달리기,
20km경보, 4×100mR, 4×400mR, 높이뛰기,
장대높이뛰기, 멀리뛰기, 세단뛰기, 포환던지기,
원반던지기, 해머던지기, 창던지기, 7종 경기
남·여
(1)
4×400m 믹스릴레이
요
2
과업개요
ㅇ (사 업 명) 2025 구미아시아육상경기선수권대회 종합정보시스템 및 홈페이지 등 구축 용역
ㅇ (사업기간) 계약일 ~ 2025. 6. 6.(금)
ㅇ (사업위치) 시민운동장 일원
1. 기본정보
사업명
2025 구미아시아육상경기선수권대회 종합정보시스템 및 홈페이지 등 구축 용역
영향평가단계
□ 예산편성
□ 그 외 필요시
■ 사업발주
□ 재평가
주요 내용
대회 기본 자료 관리 및 대회 계측기 연동 등
사업기간
(또는 개발기간)
계약일 ~ 2025년 6월
구분
붙임 : 산출내역서 각 1부
2024. . .
제안업체 인(사용인감)
2025 구미아시아육상경기선수권대회 조직위원장 귀하
※ 제안금액은 반드시 한글로 띄어쓰기 없이 붙여 쓰고 아라비아 숫자는 괄호 안에 기재할 것.
【 서식 9-1 】
산 출 내 역 서
구 분
항 목
금 액
구 성 비
비 고
순
용
역
원
가
소 계
소 계
소 계
소 계
일반관리비 ( )%
이 윤 ( )%
총 원 가
부 가 가 치 세
총 용 역 비
※ 작성내용에 따라 줄 수 증감가능
【 서식 10 】
접수번호
2025 구미아시아육상경기선수권대회 종합정보시스템 및 홈페이지 등 구축 용역
정량적 평가 제안서
2024년 월 일
업 체 명
※ 주의사항
3. 제안서 평가를 위해 구성된 평가위원회의 심사방법 및 심사 기준, 평가결과에 어떠한 이의제기도 하지 않겠습니다.
2024년 월 일
신청자
업 체 명
:
대표자
:
(인)
2025 구미아시아육상경기선수권대회 조직위원장 귀하
【 서식 6 】
보 안 서 약 서
본인은 2025 구미아시아육상경기선수권대회 종합정보시스템 및 홈페이지 등 구축 용역을 수행함에 있어 다음 사항을 준수할 것을 엄숙히 서약합니다.
2024. . .
대표자 : (인)
대리인 : (인)
2025 구미아시아육상경기선수권대회 조직위원장 귀하
※ 첨부서류 : 재직증명서, 신분증
※ 위임장에 사용하는 도장은 대표자 인감과 같아야 함.
※ 대리 접수로 인하여 발생하는 모든 문제는 대표자 본인이 책임짐
【 서식 3 】
재 직 증 명 서
성 명
생 년 월 일
부서/직위
소 속
주 소
재직기간
위와 같이 당사에 재직 중임을 증명합니다.
2024. . .
ྠĀ ྠĀ ྠĀ ྠĀ ྠĀ ྠĀ 업 체 명 :
ྠĀ ྠĀ ྠĀ ྠĀ ྠĀ ྠĀ 주 소 :
ྠĀ ྠĀ ྠĀ ྠĀ ྠĀ ྠĀ 대 표 자 : (인)
2025 구미아시아육상경기선수권대회 조직위원장 귀하
【 서식 4 】
사 용 인 감 계
사용인감
는(은) 2025 구미아시아육상경기선수권대회 조직위원회와 거래함에 있어 입찰, 계약 및 이에 수반되는 모든 행위에 대하여 위의 인감을 사용하고자 하며, 동 인감을 사용함으로써 민․형사상의 문제가 발생할 경우 이(가) 책임을 질 것을 확인합니다.
붙임 : 인감(법인)증명서 1부. 끝.
2024년 월 일
ྠĀ ྠĀ ྠĀ ྠĀ ྠĀ ྠĀ 주 소 :
ྠĀ ྠĀ ྠĀ ྠĀ ྠĀ ྠĀ 상 호 :
ྠĀ ྠĀ ྠĀ ྠĀ ྠĀ ྠĀ 대표자 : (인)
2025 구미아시아육상경기선수권대회 조직위원장 귀하
【 서식 5 】
서 약 서
당사는 2025 구미아시아육상경기선수권대회 종합정보시스템 및 홈페이지 등 구축 용역 참가신청과 관련하여 모든 서류와 증빙자료를 신의와 성실의 원칙에 입각하여 작성 제출하며, 아래 내용을 준수할 것을 확약하고 이에 서약서를 제출합니다.


---

### openai answer
retriever A 점수: 3점
(대회개요에 ‘(기 간) 2025. 5. 27.(화) ~ 5. 31.(토) 5일간’이 명시되어 있어 답변의 날짜와 기간이 모두 포함됨)

retriever B 점수: 3점
(대회개요에 ‘(기 간) 2025. 5. 27.(화) ~ 5. 31.(토) 5일간’이 명시되어 있어 답변의 날짜와 기간이 모두 포함됨) 

============================================================
### query:
구미 아시아육상경기선수권대회 홈페이지에서 제공되는 주요 콘텐츠 항목에는 무엇이 포함되는가?

### answer:
대회소개, 경기일정 및 결과, 메달순위, 참가선수 정보, 커뮤니티 등이 포함된다.

---

### retriever A:
-경기참가자(Start List), 경기결과(Official Result, Summary) PDF 파일 제공
<다음 정보를 홈페이지에 전송>
Level1
Level2
기능 설명
경기
일정
전체 경기 일정
대회기간 전체 일정 제공
일자별 경기 일정
일자별 세부 경기 일정 제공
세부 종목별 경기 일정
세부 종목별 경기 일정 제공
선수별 경기 일정
선수별 경기 일정 제공
메달
국가/선수별 메달 현황
국가/선수별 메달 현황 제공
종목별 메달 현황
종목별 메달현황 및 메달수상자 정보 제공
경기
정보
스타트 리스트
세부종목별 편성 조 및 출전선수 목록 제공
대진 정보
세부 종목별 대진정보 조회 및 출력
(경기진행 상태에 따른 결과 반영 포함)
경기 결과
각 조별, 종목별 경기 결과 조회 및 출력
선수
정보
국가별 선수 정보
국가별 참가선수 목록 및 정보 조회
세부 종목별 선수 정보
세부 종목별 참가선수 목록 및 정보 조회
순위 정보
세부종목별 참가선수 순위 정보 목록 조회
과업내용
명칭
대분류
대회 종합정보시스템 S/W 개발
소분류
보고서관리시스템 개발
과업내용
데이터 출력 기능
<PDF 및 Excel, 증서 페이지 등 출력>
-참가선수 신상정보
-국가별/세부 종목별 참가선수 리스트
-심판/임원 신상정보
-국가별/세부 종목별 심판/임원 리스트
-세부 종목별, 예결선 Start List 및 조 편성
-세부 종목별 경기 결과 (Live Result, Official Result, Summary)
-세부 종목별 메달 현황 및 신기록 현황
-참가증서
ː참가선수(종목별 1위~6위 증서 및 참가증서 필수)
 운영 매뉴얼 또는 지침서 제공 및 시스템 운영 관계자 교육
ㅇ (현장IT팀 상주 및 시스템 운영)
 사전 등록, 경기 진행 및 결과 출력 시스템 등 운영
 대회시작 4일전~대회 종료일까지 경기장 상주
 경기장/비경기장별 운영 인원 배치
ㅇ (홈페이지 구축)
 메뉴 구성도(안)
아시아육상경기선수권대회 메인
대회소개
소식지
경기일정/결과
메달순위
육상경기
커뮤니티
환영인사
공지사항
경기일정
국가별순위
경기종목
안내사항
대회개요
보도자료
행사일정
선수별순위
참가선수
아카이브
대회상징물
사진자료
시설안내
유실물센터
조직위안내
자주묻는질문
홍보영상
자원봉사자
찾아오시는 길
구미시 안내
메뉴 상세설명(안)
메뉴명
부메뉴
주요 내용
메인 화면
◦2025 구미 아시아육상경기선수권대회를 대표할 수 있는 디자인 (공식 상징물 활용 등)
◦활용빈도가 높은 메뉴 배치 및 팝업배너
대회 소개
환영인사
◦대회 개최에 대한 인사 말씀
대회개요
◦대회명칭, 대회기간, 대회장소, 참가규모, 참가국, 참가인원, 종목, 주최, 주관 등
대회상징물
◦대회마크, 마스코트, 포스터, 슬로건
조직위안내
◦조직위원회·아시아육상추진단 조직도
홍보영상
◦홍보영상, 유튜브 쇼츠 영상, 축하 메시지 등
찾아오시는 길
◦경기장 위치, 교통수단별 길 안내
소식지
공지사항
◦대회 관련 공지사항 등
보도자료
◦대회 관련기사 스크랩 자료
사진자료
◦개·폐회식 사진, 각 종목별 경기 이미지 게시
경기일정
/
결과
경기일정
◦경기장 및 개최 일자, 시간, 라운드, 진행상태
◦각 종목별 경기결과, 종합순위
행사일정
◦개·폐회식 정보 및 식순 안내
찾아오시는 길
◦경기장 위치, 교통수단별 길 안내
소식지
공지사항
◦대회 관련 공지사항 등
보도자료
◦대회 관련기사 스크랩 자료
사진자료
◦개·폐회식 사진, 각 종목별 경기 이미지 게시
경기일정
/
결과
경기일정
◦경기장 및 개최 일자, 시간, 라운드, 진행상태
◦각 종목별 경기결과, 종합순위
행사일정
◦개·폐회식 정보 및 식순 안내
◦행사장, 공연, 전시 등 안내
◦메달 수여식 안내
메달 순위
국가별 순위
◦국가별 메달 집계 및 랭킹 표시
선수별 순위
◦선수별 메달 집계 및 랭킹 표시
육상경기
경기종목
◦마스코트 이미지 또는 픽토그램으로 제시
◦경기 규칙, 유래 등 간단한 설명 자료 제공
◦종목별 주의사항, 해당 경기일정표 안내
참가선수
◦대회 참가 선수의 국가, 선수이름 안내
시설안내
◦종목별 경기장소, 선수촌, 훈련장 안내
커뮤니티
안내사항
◦대회 관련 안내사항 등
아카이브
◦대회 관련 자료 선별 보관
유실물 센터
◦대회 기간 중 유실물 현황
자주묻는질문
◦대회 관련 자주묻는질문 리스트
자원봉사자
◦자원봉사 소개, 신청안내
구미시 안내
◦구미시 문화관광 홈페이지 연동하여 관광 안내
◦지도 및 주요 고속도로, 국도 및 대중교통 안내
◦구미시 숙박시설 안내
◦구미시 모범음식점 등 안내
일정)
구분
작업일정
M
M+1
M+3
M+5
M+6
M+7
프로젝트 착수 및 보고
홈페이지
제작
업무 분석 및 설계
개발 및 디자인 작업
테스트 및 시범 운영
홈페이지 서비스
대회종합정보시스템
요구 분석
상세 설계
구축개발, 단위 테스트
초기데이터 구축 , 검증
구축완료, 통합테스트
유지보수
대회 종료시까지


---

### retriever B:
 운영 매뉴얼 또는 지침서 제공 및 시스템 운영 관계자 교육
ㅇ (현장IT팀 상주 및 시스템 운영)
 사전 등록, 경기 진행 및 결과 출력 시스템 등 운영
 대회시작 4일전~대회 종료일까지 경기장 상주
 경기장/비경기장별 운영 인원 배치
ㅇ (홈페이지 구축)
 메뉴 구성도(안)
아시아육상경기선수권대회 메인
대회소개
소식지
경기일정/결과
메달순위
육상경기
커뮤니티
환영인사
공지사항
경기일정
국가별순위
경기종목
안내사항
대회개요
보도자료
행사일정
선수별순위
참가선수
아카이브
대회상징물
사진자료
시설안내
유실물센터
조직위안내
자주묻는질문
홍보영상
자원봉사자
찾아오시는 길
구미시 안내
메뉴 상세설명(안)
메뉴명
부메뉴
주요 내용
메인 화면
◦2025 구미 아시아육상경기선수권대회를 대표할 수 있는 디자인 (공식 상징물 활용 등)
◦활용빈도가 높은 메뉴 배치 및 팝업배너
대회 소개
환영인사
◦대회 개최에 대한 인사 말씀
대회개요
◦대회명칭, 대회기간, 대회장소, 참가규모, 참가국, 참가인원, 종목, 주최, 주관 등
대회상징물
◦대회마크, 마스코트, 포스터, 슬로건
조직위안내
◦조직위원회·아시아육상추진단 조직도
홍보영상
◦홍보영상, 유튜브 쇼츠 영상, 축하 메시지 등
찾아오시는 길
◦경기장 위치, 교통수단별 길 안내
소식지
공지사항
◦대회 관련 공지사항 등
보도자료
◦대회 관련기사 스크랩 자료
사진자료
◦개·폐회식 사진, 각 종목별 경기 이미지 게시
경기일정
/
결과
경기일정
◦경기장 및 개최 일자, 시간, 라운드, 진행상태
◦각 종목별 경기결과, 종합순위
행사일정
◦개·폐회식 정보 및 식순 안내
종합정보시스템 및 홈페이지 등 구축 용역 제안요청서
1
대회개요
아시아의 꿈, 구미에서 세계로
 The dream of ASIA, From GUMI to the World 
비전
ㅇ (대 회 명) 2025 구미아시아육상경기선수권대회
ㅇ (기 간) 2025. 5. 27.(화) ~ 5. 31.(토) 5일간
ㅇ (장 소) 구미시민운동장 일원
ㅇ (규 모) 아시아 45개국 1,200여명 / 선수단 및 임원
ㅇ (주 최) 아시아육상연맹(Asian Athletics, AA)
ㅇ (주 관) 2025 구미아시아육상경기선수권대회 조직위원회
ㅇ (경기종목) 45개 종목
구 분
종 목 명
남자
(22)
100m, 200m, 400m, 800m, 1,500m, 5,000m,
10,000m, 110mH, 400mH, 3,000m장애물달리기,
20km경보, 4×100mR, 4×400mR, 높이뛰기,
장대높이뛰기, 멀리뛰기, 세단뛰기, 포환던지기,
원반던지기, 해머던지기, 창던지기, 10종 경기
여자
(22)
100m, 200m, 400m, 800m, 1,500m, 5,000m,
10,000m, 100mH, 400mH, 3,000m장애물달리기,
20km경보, 4×100mR, 4×400mR, 높이뛰기,
장대높이뛰기, 멀리뛰기, 세단뛰기, 포환던지기,
원반던지기, 해머던지기, 창던지기, 7종 경기
남·여
(1)
4×400m 믹스릴레이
요
2
과업개요
ㅇ (사 업 명) 2025 구미아시아육상경기선수권대회 종합정보시스템 및 홈페이지 등 구축 용역
ㅇ (사업기간) 계약일 ~ 2025. 6. 6.(금)
ㅇ (사업위치) 시민운동장 일원
찾아오시는 길
◦경기장 위치, 교통수단별 길 안내
소식지
공지사항
◦대회 관련 공지사항 등
보도자료
◦대회 관련기사 스크랩 자료
사진자료
◦개·폐회식 사진, 각 종목별 경기 이미지 게시
경기일정
/
결과
경기일정
◦경기장 및 개최 일자, 시간, 라운드, 진행상태
◦각 종목별 경기결과, 종합순위
행사일정
◦개·폐회식 정보 및 식순 안내
◦행사장, 공연, 전시 등 안내
◦메달 수여식 안내
메달 순위
국가별 순위
◦국가별 메달 집계 및 랭킹 표시
선수별 순위
◦선수별 메달 집계 및 랭킹 표시
육상경기
경기종목
◦마스코트 이미지 또는 픽토그램으로 제시
◦경기 규칙, 유래 등 간단한 설명 자료 제공
◦종목별 주의사항, 해당 경기일정표 안내
참가선수
◦대회 참가 선수의 국가, 선수이름 안내
시설안내
◦종목별 경기장소, 선수촌, 훈련장 안내
커뮤니티
안내사항
◦대회 관련 안내사항 등
아카이브
◦대회 관련 자료 선별 보관
유실물 센터
◦대회 기간 중 유실물 현황
자주묻는질문
◦대회 관련 자주묻는질문 리스트
자원봉사자
◦자원봉사 소개, 신청안내
구미시 안내
◦구미시 문화관광 홈페이지 연동하여 관광 안내
◦지도 및 주요 고속도로, 국도 및 대중교통 안내
◦구미시 숙박시설 안내
◦구미시 모범음식점 등 안내


---

### openai answer
retriever A 점수: 3점
(메뉴 구성도에 대회소개, 경기일정/결과, 메달순위, 참가선수, 커뮤니티 등이 명시되어 있어 답변 요소가 모두 포함됨)

retriever B 점수: 3점
(메뉴 구성도에 대회소개, 경기일정/결과, 메달순위, 참가선수, 커뮤니티 항목이 포함되어 있어 답변의 모든 요소를 충족함) 

============================================================
### query:
원자력 선량평가시스템의 성능 요구사항 중 질의응답 시간 기준은 어떻게 정의되어 있는가?

### answer:
일반 조회 요청은 3초 이내 처리되어야 하며, 10초 이상 소요되는 작업은 사용자에게 팝업 메시지를 제공해야 한다.

---

### retriever A:
질의·응답시간
요구사항
상세설명
정의
질의·응답시간
세부
내용
의는 사용자가 요청을 하는 시간으로부터 3000ms 내에 그 결과를 보여줘야 함
❍ 임의의 선택 기준이 허용되는 대량의 데이터에 대한 질의 및 고용량의 멀티미디어 파일을 가지고 있는 프로그램에서는 적절한 처리 상태를 표시하여야 함
산출정보
성능측정 결과 내역서
관련 요구사항
요구사항 고유번호
PER-002
요구사항 명
오류 응답시간
요구사항
상세설명
정의
오류 응답시간
세부
내용
시스템에 데이터 입력 등의 사용 중에 발생한 모든 오류는 원인이 발생한지 3000ms 이하로 사용자에게 오류 메시지를 보여줘야 함
산출정보
성능측정 결과 내역서
관련 요구사항
요구사항 고유번호
PER-003
요구사항 명
통계 처리 사전 경고
요구사항
상세설명
정의
통계 처리 사전 경고
세부
내용
리 시 5,000ms 이상 지연되는 경우 연산 작업을 시작하는 시점에 예상되는 결과에 대해 사용자에게 메시지로 알려야 함
산출정보
성능측정 결과 내역서
관련 요구사항
요구사항 고유번호
PER-004
요구사항 명
부하증가 대비 성능유지
요구사항
상세설명
정의
부하증가 대비 성능유지
세부
내용
시스템의 경우 동시 접속자(사용자)는 500명 이상 접속가능 하여야 하며, 이에 따른 처리속도 등 성능이 저하되지 않아야 함
❍ 포털 시스템의 경우 동시 접속자(사용자)는 3000명 이상 접속가능 하여야 하며, 이에 따른 처리속도 등 성능이 저하되지 않아야 함
산출정보
성능측정 결과 내역서
관련 요구사항
요구사항 고유번호
PER-005
요구사항 명
오프라인 배치성 업무 응답시간
요구사항
나. 성능 요구사항 (Performance Requirement)
요구사항분류
PER-001
요구사항명
응답시간
요구사항 내용
정의
개선 대상 웹사이트 및 콘텐츠 관리 시스템(CMS) 정보 조회에 대한 응답시간 성능 보장
세부 내용
○ 정보 처리량 및 확장성 등을 고려하여 웹페이지 디스플레이 시간 기능요청에 대한 응답시간이 최소화 되도록 함
- 단순 웹페이지의 경우, 사용자가 요청을 하는 시간으로부터 3초 내에 그 결과를 보여줘야 함
- 시스템이 만드는 각 페이지의 경우, 요청한 시간으로부터 5초 내에 디스플레이 되어야 함
- 이 요구사항은 시스템을 사용하는 동시 사용자가 최대 사용자 수의 90%를 초과하는 질의에는 적용되지 않음
○ 과도한 데이터베이스 접속을 자제하며 응답속도 개선(상영시간표 등)
- 이 요구사항은 임의의 선택 기준이 허용되는 대량의 데이터(데이터량 2GB 이상)에 대한 질의에는 적용되지 않음
- 이 요구사항은 임의의 선택 기준이 허용되는 대량의 데이터에 대한 요청 및 플래시가 있을 경우에는 적용되지 않으며, 한 개 이상의 큰 이미지(이미지 500KB 이상) 및 동영상을 가지고 있는 페이지에는 적용되지 않음
- 동영상의 경우 트래픽 관리를 위해 외부 플랫폼(예: 유튜브, 기타 CDN 서비스)과 연계하여 기능 구현
○ 기능 구현에 있어 10초 이상 걸릴 수 있는 작업에 대해서는 시간지연에 대한 정보를 사용자에게 팝업 등의 방법으로 사용자에게 알려야 함
요구사항분류
PER-002
요구사항명
데이터 형식 오류 응답 시간
요구사항 내용
정의
사용자 입력 오류 응답 시간 제시
세부 내용
요구사항 명칭 성능요건
정의 성능요건 일반사항
❍ 성능요건 일반
- 기존 인프라 장비, 솔루션과의 호환성과 성능 보장
- 기 구축되어 서비스 중인 모든 응용 프로그램의 안정성과 성능을 보장해야
함
- 시스템에 대한 성능 분석을 실시하고 결과 및 개선방안을 제시하여야 함
- 시스템의 안정적 운영을 위하여 서비스 대고객 오픈 전/후와 안정화 기간에
튜닝 등 시스템 환경을 최적화하고 운영 어플리케이션 및 각종 솔루션의
요구 업그레이드 및 시스템 환경을 최적화하고 업그레이드 및 장애이력을
사항 관리하여야 함
상세 세부
설명 내용 ❍ 개발시스템의 성능 관리
- 사업대상 시스템의 성능을 고려한 개발 방안을 제시
- 대상시스템에 대하여 안정적 운영지원 및 사용자 지원 방안을 제시
- 시스템 개발 중 로그 또는 툴(도구)을 이용하여 시스템 성능상태를
모니터링이 가능하도록 환경을 구성하여, 성능상 문제를 미리 파악하여
조치한 후 시스템을 오픈
- WAS 및 DBMS의 부하 감소, 성능 개선, 장애 요인 감소, 시스템 안정화를
위한 튜닝 실시
- 수강신청, 성적조회 등 트래픽이 집중되는 시기의 안정적인 서비스 방안을
제시하여야 함
산출정보 • 성능시험결과서
-175-
요구사항 분류 성능 요구사항
요구사항 번호 PER-002
요구사항 명칭 온라인성 업무응답시간
정의 온라인성 업무응답시간 목표
❍ 업무응답시간 요구사항
- 정보 요청에 따라 결과가 조회되어 화면에 표시되기까지의 소요 시간을
의미
- 모든 질의는 사용자가 요청을 하는 시간으로부터 3초 내에 그 결과를 처리


---

### retriever B:
성능 요구사항
상세
설명
정의
정보시스템 업무 응답시간 목표
세부
내용
□ 요청(질의) 응답시간
o 모든 질의는 사용자가 요청을 하는 시간으로부터 3초 내에 최초의 결과 값을 보여줘야 함
□ 이 요구사항은 임의의 선택 기준이 허용되는 대량의 데이터(2GB 이상)에 대한 질의에는 적용되지 않음
산출정보
o SW개발에 따른 산출물
성능 요구사항
상세
설명
정의
정보시스템 업무 응답시간 목표
세부
내용
□ 요청(질의) 응답시간
o 모든 질의는 사용자가 요청을 하는 시간으로부터 3초 내에 최초의 결과 값을 보여줘야 함
□ 이 요구사항은 임의의 선택 기준이 허용되는 대량의 데이터(2GB 이상)에 대한 질의에는 적용되지 않음
산출정보
o SW개발에 따른 산출물
성능 요구사항
응낙수준
필수
요구
사항
상세
설명
정의
사용자 요구에 대한 웹페이지 디스플레이 반응 시간
세부
내용
○ 시스템은 정상 상태에서 사용자의 질의 요청에 대한 결과 페이지를 화면에 출력할 때 3초 이내 보장
○ 평균 응답시간은 시스템을 사용하는 동시 사용자가 최대 사용자 수의 90%를 초과하는 경우에는 미적용
산출정보
관련요구사항
요구사항 내용
○ 모든 질의는 사용자가 요청하는 시간으로부터 3초 이내에 그 결과값을 보여줘야 함
○ 평균 응답 시간은 시스템을 사용하는 동시 사용자가 최대 사용자 수의 90%를 초과하는 경우 미적용함
○ 한계 인원 이상 접속 시 해결 방안 제시해야 함
② 성능 요구사항
PER-001
평균응답시간 및 성능 최적화
요구사항
상세설명
정의
업무응답시간에 목표 대한 기준
세부내용
○ 모든 질의는 평균 응답시간으로 사용자가 요청한 시간으로부터 3초 내에
그 결과를 보여줘야 함
○ 대량의 데이터 로딩(통계, 검색, 이미지 등), 사용자 숫자가 동시 사용자 용량의 90%를 초과하는 경우에는 적용되지 않음
산출 정보
개발 산출물, 테스트 결과서
관련 요구사항
성능 요구사항
상세
설명
정의
평균 응답시간
세부
내용
❍ 대상시스템은 정상 상태에서 사용자의 질의 요청에 대한 결과
페이지를 화면에 출력할 때 3초 이내에 완전히 출력되어야 함
요구사항
상세설명
정의
평균 응답시간 최적화
세부
내용
○ 시스템은 정상 상태에서 사용자의 질의요청에 대한 결과 페이지를 화면에 출력할 때 3초 이내 보장
○ 사용자의 정상적인 요청에 대한 응답시간 지연을 최소화하여야 함
○ 평균 응답시간은 임의의 선택기준이 허용되는 대량의 데이터에 대한 질의가 있을 경우에는 적용되지 않으며 시스템을 사용하는 동시 사용자가 최대 사용자 수의 90%를 초과하는 경우 미적용함
○ 요청 처리 과정에서 오류가 발생할 경우, 사용자에게 오류 메시지 표시
- 대량 데이터 처리 등 처리시간을 협의하여 결정할 수 있으나 반드시 사용자가 기대할 수 있는 진행상태 등을 인식하도록 구현
4) 인터페이스 요구사항
요구사항 명칭 온라인성 업무응답시간
정의 온라인성 업무응답시간 목표
❍ 업무응답시간 요구사항
- 정보 요청에 따라 결과가 조회되어 화면에 표시되기까지의 소요 시간을
의미
- 모든 질의는 사용자가 요청을 하는 시간으로부터 3초 내에 그 결과를 처리
요구 - 포탈/학사 외에도 레거시 시스템(일반행정, 연구포탈 등)의 업무응답시간도
사항 3초 내에 그 결과를 처리해야 함
상세 세부
설명 내용 ❍ 업무응답시간 예외사항
- 요구사항은 임의의 선택 기준이 허용되는 대량의 데이터(기준 5,000건
이상)에 대한 질의 한 개 이상의 대형 이미지(기준 500KB 이상), 영상이
포함된 페이지에는 미적용하나 대량데이터의 처리시간 기준을 제시하고
적용하여야 함
- 시스템을 사용하는 사용자 숫자가 동시 사용자 용량의 90%를 초과하는
경우에는 미적용
산출정보 • 성능시험결과서(모든 시스템 화면 포함)
요구사항 명칭
질의 응답시간
요구사항 명칭
시스템 응답시간
요구사항 명칭
평균 응답시간
요구
사항
상세
트 응답속도 향상을 보장하여야 하며, 시스템은 정상 상태에서 사용자의 모든 질의 요청에 대한 처리를 3초 이내에 화면에 보여주어야 함
○ 만약 회원관리시스템이 WEB 방식 또는 C/S방식으로 개발되는 경우 모두 회원 관리시스템의 응답속도는 3초 이내가 되어야 하며 연동 장비간(POS장비 등) 지연 또는 오류가 발생되지 않도록 반드시 주의해야 한다.
○ 대량의 데이터 로딩(통계, 검색, 이미지 등), 동시접속자가 사용자 용량의 90%를 초과하는 경우에는 적용되지 않음
○ 웹페이지 용량 최소화, 연결 최소화, 페이지 표시(랜더링) 순서 최적화 등 다양한 최적화 기법을 적용하여 웹사이트의 성능을 최적화해야 함
○ 서비스 개시 전(테스트 단계) 최적화 점검을 하고, 최적화 점검결과서에 따라 반드시 문제점 등을 보완해야 함
○ 사용자 정보 요청 시 또는 사용자가 입력한 데이터 형식의 모든 오류에 대하여 3초 이내의 적절한 응답 또는 오류 메시지를 제공해야 함
○ 대량 데이터 처리 등 10초 이상 걸리는 서비스는 사용자에게 팝업 메시지를 제공해야 함
○ 오류 메시지에 중요 정보(서버 설정 정보, 세션정보, SQL정보 등)가 노출되지 않도록 적절한 에러 처리 로직 및 에러 화면을 구성하여 제공해야 함
※ 단, 평균 응답시간을 향상시키기 위한 추가시스템(WAS, 웹가속시스템 등)이 필요한 경우 발주기관과 사전 협의해야 하고, 이에 소요되는 제반비용은 수주자가 부담하여야 한다.


---

### openai answer
retriever A 점수: 3점
(사용자 요청 3초 이내 결과 제공과 10초 이상 소요 작업 시 팝업 안내 요구사항이 모두 명시됨)

retriever B 점수: 3점
(모든 질의 3초 이내 결과 제공 및 10초 이상 걸리는 서비스에 대한 팝업 메시지 제공이 명시되어 정답의 두 조건을 모두 포함) 

============================================================
### query:
원자력 선량평가시스템의 포트란 모듈 개선 요구사항에서 추가로 고려해야 하는 평가 요소는 무엇인가?

### answer:
 ICRP60 기준에 따른 23개 장기를 고려한 평가장기 추가가 요구된다.

---

### retriever A:
가 방법
평가부문
평가항목
평가기준
배점
한도
정량
평가
(10)
경영상태
경영상태(신용상태등급) 평가
10
전략 및 방법론
(22)
사업
이해도
사업의 특성 및 목표에 부합하는 사업전략을 제시하고 있는지를 평가한다.
7
추진전략
개발업무 수행 시 일정 및 위험요소를 고려하여 얼마나 타당한 추진 전략을 수립하였는지 평가한다.
5
적용기술
사업에서 적용하고자 하는 기술이 향후 확장성을 고려하여 현실적으로 실현 가능하게 제시되어 있는지를 평가한다.
4
표준프레임 워크 적용
제안사는 사업에 네팔 정보통신기술부 프레임워크 기반의 공통 컴포넌트의 유형 및 사용계획을 기술하고 사용성 향상을 위한 추가 컴포넌트에 대해 적용방안을 제시하여야한다.
3
개발
방법론
사업에 적정한 개발 방법론이 제시되었는지 평가하고, 실제 적용 사례 및 경험을 바탕으로 단계별 산출물이 제시되었는지 여부를 평가한다.
3
기술 및 기능
(26)
시스템 장비 구성 요구사항
요구 규격을 충족하는 장비 제안 여부와 현 시스템과의 인터페이스 및 확장 가능성 여부를 평가한다. 또한 도입 장비의 설치 및 공급 계획, 유지관리에 대한 방안이 구체적으로 기술되어 있는가를 평가한다.
7
기능 요구사항
기능 요구사항·기대사항·제약사항 등을 파악한 후 구현 방안이 구체적으로 기술되어 있는지를 평가한다. 또한 제안한 방안 및 기술이 적용 가능한지를 평가한다.
7
데이터
요구사항
데이터 요구사항 분석을 바탕으로 데이터 전환계획 및 검증 방법, 데이터 오류 발생 시 처리 방안이 구체적으로 제시되어 있는지를 평가한다.
4
보안
요구사항
기능 보안
체적이고 적절한지 평가
(30점) 요구사항
- 구현방안이 설계단계부터 구현 및 검증단계까지 고려하고 있는지 평가
- 데이터 전환 계획 및 검증 방법, 에러 데이터 처리 방법에 대해 구체적
데이터
인 내용을 제시하고 있는지 평가 15
요구사항
- 데이터 전환을 위해 책임 조직이 투입되는지 평가
- 기능 및 품질 등 요구사항을 구현 시 관련 제약사항을 충족시키며 구현
제약사항
방안 및 테스트 방안을 구체적으로 기술하였는지 평가
-237-
평가
평가항목 평가기준 배점
부문
- 구현하고자 하는 기능에 요구 성능이 충족되도록 방법론 및 분석 도구
성능 를 통하여 내용을 구체적으로 분석하였는지, 구현 및 테스트 방안을 구
요구사항 체적으로 기술하였는지 평가
- 제안한 방안 및 기술을 통해 요구 성능을 충족시킬 수 있는지 평가
- 분석 도구 및 구현 방안, 테스트 방안을 구체적으로 기술하였는지, 분석
·설계 등 각 단계별 품질 요구사항의 점검 및 검토 방안을 구체적으로
품질
계획하였는지 평가
요구사항
- 각 단계마다 품질 요구사항을 점검하는 별도의 전문 인력이 투입되는지
성능 및 평가
품질 - (시스템 인터페이스) 타 시스템과의 연계 방안들에 대한 장·단점의 분석 10
(10점) 을 통해 가장 적합한 방안을 구체적으로 기술했는지, 구현 경험이 있는
인터페이
인터페이스 담당자가 투입되는지 평가
스
- (사용자 인터페이스) 사용자 편의성을 고려하여 요구사항을 제공하기 위
요구사항
한 분석 및 설계, 구현 방안과 검토 계획을 구체적으로 기술하였는지,
구현 경험이 있는 사용자 인터페이스 담당자가 투입되는지 평가
적용 기술
사업에서 적용하고자 하는 기술이 향후 확장성을 고려하였는지, 현실적으로 실현 가능한지 여부를 평가한다.
표준 프레임
워크 적용
표준 프레임워크 적용 여부와 적용 시 예상되는 문제점을 구체적으로 기술하고 실현 가능한 대응 방안을 제시하였는지 평가한다.
개발 방법론
사업에 적정한 개발 방법론이 제시되었는지 평가하고, 실제 적용 사례 및 경험을 바탕으로 단계별 산출물이 제시되었는지 여부를 평가한다.
기술 및 기능
시스템 요구
사항
요구 규격을 충족하는 장비 제안 여부와 현 시스템과의 인터페이스 및 확장 가능성 여부를 평가한다. 또한 도입 장비의 설치 및 공급 계획, 유지관리에 대한 방안이 구체적으로 기술되어 있는가를 평가한다.
25
기능 요구
사항
기능 요구사항ㆍ기대사항ㆍ제약사항 등을 파악한 후 구현 방안이 구체적으로 기술되어 있는지를 평가한다. 또한 제안한 방안 및 기술이 적용 가능한지를 평가한다.
보안 요구
사항
요구사항 및 시스템 분석을 바탕으로 보안 요구사항의 적용 방안이 구체적으로 기술되었는지 평가한다. 또한 제안한 방안과 기술의 설계단계 반영 여부 및 구현단계까지의 적용 가능성을 평가한다.
데이터 요구
사항
데이터 요구사항 분석을 바탕으로 데이터 전환 계획 및 검증 방법, 데이터 오류 발생 시 처리 방안이 구체적으로 제시되어 있는지를 평가한다.
시스템운영
요구 사항
시스템 운영 요구 사항에 맞는 운영 절차 및 방법을 제시하였는지 평가한다. 또한 운영 중 비상사태 발생 시 대응방안이 구체적으로 제시되고 있는지를 평가한다.
제약 사항


---

### retriever B:
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항


---

### openai answer
retriever A 점수: 0점
원자력 선량평가, 포트란 모듈, ICRP60, 23개 장기 등 핵심 내용이 전혀 없고 일반적인 평가항목/요구사항만 있음.

retriever B 점수: 0점
“기능 요구사항” 반복 외에 ICRP60, 23개 장기, 원자력 선량평가, 포트란 등 답변에 필요한 정보가 전혀 없음. 

## size 5000

============================================================
### query:
아이돌봄 인적성 검사 시스템 운영 사업의 주요 목적은 무엇인가?

### answer:
인적성 검사 수행을 지원하고, 돌봄활동과 연계한 사후관리 및 인력 검증을 강화하는 것이다.

---

### retriever A:
◦원할한 인적성 검사 수행을 위한 정보시스템 지원 필요
◦돌봄활동과 연계한 사후관리 기반을 강화하여 이용자 만족도 향상
◦아이돌봄인력 인·적성에 기초한 돌봄인력 관리 강화 필요
◦정보시스템 개인정보보호법 준수를 위한 보호조치
2
사업 개요
□ 사 업 명: 아이돌봄인력 인적성 검사 정보시스템 운영
- 2024. 정보시스템 구축에 따른 유지보수 사업
□ 사업기간: 계약일 ~ 2025. 12. 31.
□ 소요예산: 172,700,000원(부가세 포함)
□ 추진방식: 제한경쟁입찰(협상에 의한 계약)
3
사업 범위
□ 인·적성 프로그램 기능수정 및 안정화 기술지원
◦인·적성 시스템의 안정적인 정보화 지원을 위한 상시 유지관리
체계 지원
◦인·적성 검사 변경에 따른 프로그램 수정 및 결과 반영
◦웹 접근성, 기술적 보안 등 전자정부 가이드라인 준수
□ 개인정보보호 및 정보보안 강화
◦개인정보보호 및 정보보호 강화 조치(방안)
◦정보시스템 운영을 위한 물리적, 기술적 보안 대책 제시
□ 안정적인 클라우드 환경 운영 및 제공
◦클라우드 무중단 환경 제공
◦클라우드 환경 운영을 위한 기술적 유지관리 지원
◦시스템 용을 위한 SW 및 외부 서비스 연계
4
기대 효과
□ (안전한 돌봄 환경 조성) 아이돌봄인력 선별 및 검증 강화 통한 안심하고 맡길 수 있는 돌봄서비스 제공
□ (효율성 제고) 아이돌봄인력 대상 인적성 검사를 편리하게 실시하여 돌봄인력 검증 절차 효율성 제고
□ (정책지원 및 제도개선 지원) 체계적인 검사 결과를 축적하고 정책 수립지원을 위한 기초자료를 제공, 지속 가능한 아이돌봄서비스 제도개선에 활용
□ (안전한 돌봄 환경 조성) 아이돌봄인력 선별 및 검증 강화 통한 안심하고 맡길 수 있는 돌봄서비스 제공
□ (효율성 제고) 아이돌봄인력 대상 인적성 검사를 편리하게 실시하여 돌봄인력 검증 절차 효율성 제고
□ (정책지원 및 제도개선 지원) 체계적인 검사 결과를 축적하고 정책 수립지원을 위한 기초자료를 제공, 지속 가능한 아이돌봄서비스 제도개선에 활용
□ (돌봄인력 관리) 데이터를 기반으로 하는 아이돌봄인력에 대한 효율적 관리·감독 가능
Ⅱ. 사업 현황
1
업무현황
□ 인·적성검사 실시 정보시스템 지원
ㅇ 인·적성검사 및 실시를 위한 정보 시스템 제공
ㅇ 정보시스템 안정화 및 운영지원
□ 인·적성검사 운영
ㅇ 인적성 검사 실시 및 수행 방법 안내
ㅇ 인적성 검사 수행 모니터링 및 결과 분석
ㅇ 검사 실시 운영자 권한 부여 및 인증 관리
2
정보화 현황
□ 시스템 구성도
□ 정보시스템 구성도 요청 시 별도 공개
※ 행정기관 및 공공기관 정보시스템 구축 및 운영 지침(행정안전부 고시 제 2024-66호, 2024.8.28.) 제 17조(제안요청서 보안사항 등)에 따라, 시스템 구성도 및 상세자료 등은 공개하지 않으며, 입찰에 참가하고자 하는 사업자의 요청이 있는 경우 보안서약서를 제출 후 담당자 입회하에 제한적으로 열람 가능
□ 시스템(장비)·소프트웨어 운영현황
AS-IS
TO-BE (2025년)
ㅇ인·적성 검사 관리기능 및 사용자화면 개발
※ 클라우드 환경 기반 개발
ㅇ인·적성 검사 기능 최적화 및 운영 기술지원
-인·적성 검사 고도화에 따른 수정 반영
(검사도구 4종)
※사업 운영에 따라 변경될수 있음
8. 제안서 보상 穠ȃ 44
9. 기타사항 舨ȃ 44
Ⅶ. 별첨
[별첨] 입찰서류 및 제안서 제출안내(온라인제출) 㜜ȃ 46
[별첨] 사업자 보안위규 처리기준 墄ȃ 71
[별첨] 보안 위약금 부과 기준 後ȃ 73
[별첨] 누출금지 대상정보 棬ȃ 74
[별첨] 기술적용계획표 濴ȃ 75
楴䵴
Ⅰ.사업 개요
1
추진배경
□ 추진배경
◦ 국정과제(46:촘촘한 아동돌봄체계 마련) 실현을 위한 돌봄인력 검증 강화
아이돌봄지원법 제7조(아이돌보미의 자격) 6의 ②항 아이돌보미가 되려는 사람은 여성가족부장관이 실시하는 적성ㆍ인성 검사를 받아야 한다.
◦ 아이돌봄인력 대상으로 온라인 정서·심리검사를 실시하고 결과를 반영하여 아동학대 예방 등 돌봄 활동 연계 등에 활용
◦아이돌봄인력의 직무적합도에 대한 서비스 이용자 요구 증가
◦개인정보보호법 제29조(안전조치의무), 시행령 제30조(개인정보 안전성 확보조치)
개인정보처리자는 개인정보가 분실〮도난〮유출〮위조〮변조 또는 훼손되지 아니하도록 내부 관리계획 수립, 접속기록 보관 등
대통령령으로 정하는 바에 따라 안전성 확보에 필요한 기술적〮관리적 및 물리적 조치를 하여야 한다.
□ 목적 및 필요성
◦원할한 인적성 검사 수행을 위한 정보시스템 지원 필요
◦돌봄활동과 연계한 사후관리 기반을 강화하여 이용자 만족도 향상
◦아이돌봄인력 인·적성에 기초한 돌봄인력 관리 강화 필요
◦정보시스템 개인정보보호법 준수를 위한 보호조치
2
사업 개요
□ 사 업 명: 아이돌봄인력 인적성 검사 정보시스템 운영
- 2024. 정보시스템 구축에 따른 유지보수 사업
[별표 1] 누출금지 대상정보 墰ȃ 75
[별표 2] 보안위규 처리규정 墰ȃ 76
[별표 3] 보안 위약금 부고 기분 ䷀ȃ 78
[별표 4] 정보시스템 관리 대장 偼ȃ 79
[별표 5] 휴대용 저장매체 관리 대장 䋐ȃ 80
[별표 6] 휴대용 저장매체 점검 대장 䋐ȃ 81
[별표 7] 휴대용 저장매체(전산장비 포함) 반출ㆍ입 대장 Ⴔȃ 82
[별표 8] 휴대용 저장매체 불용처리 확인서 ㉨ȃ 83
[별표 9] 자료 인계인수 대장 嗴ȃ 84
[별표 10] 접속단자 봉인 대장 匼ȃ 85
[별표 11] 용역사업 보안관리실태 점검 체크리스트 Ὀȃ 86
[별표 12] 월별 보안점검 양식 匼ȃ 88
[별표 13] 안전사고예방 및 보건상태 점검표 ⾰ȃ 89
[별표 14] 안전·보건 사고 보고서 䮨ȃ 90
Ⅰ
과업 개요
1. 과업 일반
ㅇ 과 업 명 : 2024년 항해용 간행물 품질관리 업무보조 시스템 구축
ㅇ 과업기간 : 계약일로부터 2024년 12월 20일까지
ㅇ 과업예산 : 200백만원(부가가치세 포함)
ㅇ 계약방법 : 제한입찰(협상에 의한 계약)
2. 과업 목적
ㅇ 수동 중심 검사방식을 디지털·자동화 품질검사가 가능한 도구개발을 통하여 품질 검사시간 단축 및 업무효율성 제고
ㅇ 항해용 간행물 종류 및 품질관리 항목별 오류 정보의 체계적인 관리를 통하여 신속한 품질관리 업무지원
3. 과업 범위
1) 디지털 품질검사 요소 분석
가. 종이해도 파일(PCC) 구조 분석 및 디지털 검사방안 도출
나. 바다내비용 전자해도의 디지털 오류검사 방안 도출
다. 항행통보 반영 정보의 품질검사 방안 도출
□ 시스템(장비)·소프트웨어 운영현황
AS-IS
TO-BE (2025년)
ㅇ인·적성 검사 관리기능 및 사용자화면 개발
※ 클라우드 환경 기반 개발
ㅇ인·적성 검사 기능 최적화 및 운영 기술지원
-인·적성 검사 고도화에 따른 수정 반영
(검사도구 4종)
※사업 운영에 따라 변경될수 있음
ㅇ무중단 클라우드 기반 제공
ㅇ클라우드 안정화 운영기반 및 기술지원
SW 도입 현황
SW 명칭
용도
비고
리포팅 툴
OZ Report
인·적성 결과지 출력
발주처 제공
□ 시스템 장비 현황 요청 시 별도 공개
※ 행정기관 및 공공기관 정보시스템 구축 및 운영 지침(행정안전부 고시 제 2024-66호, 2024.8.28.) 제 17조(제안요청서 보안사항 등)에 따라, 시스템 구성도 및 상세자료 등은 공개하지 않으며, 입찰에 참가하고자 하는 사업자의 요청이 있는 경우 보안서약서를 제출 후 담당자 입회하에 제한적으로 열람 가능
Ⅲ. 사업 추진방안
1
추진 목표
□ `24년 개발된 인·적성 검사 운영을 지원하고, 데이터 수집 및 분석을 위한 정보체계 관리
□ 인·적성 검사 정보시스템 안정적 기반 지원 및 보안 강화
2
추진체계
□ 추진체계도
발주기관
<한국건강가정진흥원>
협조부서
<경영지원부/
소통협력실>
주관부서
<돌봄지원부>
관련부처
<여성가족부>
주관사업자
<사업수행업체>
□ 추진 주체별 역할
구 분
주요 업무 내용
주관부서
ㅇ 사업총괄 관리 및 시행
ㅇ 사업계획서 수립 및 제안요청서 작성
ㅇ 개발내역 업무정의 및 과업내용 제시
ㅇ 개발관련 자료제공 및 시스템 인수
ㅇ 사업수행 관리 및 검사, 결과 평가
협조부서
제 안 요 청 서
사 업 명
2025년 아이돌봄인력 인적성 검사 정보시스템 운영
주관기관
한국건강가정진흥원
2024. 12.
담 당
성 명
소 속
전화번호
(팩스번호)
e-mail
진용수
가족친화사업본부 돌봄지원부
02-3479-7701
02-3479-7717
ysjin@kihf.or.kr
- 목 차 -
Ⅰ. 사업 개요
1. 추진배경 蓀ȃ 1
2. 사업개요 蓀ȃ 1
3. 사업범위 舨ȃ 2
4. 기대효과 舨ȃ 2
Ⅱ. 사업 현황
1. 업무현황 蓀ȃ 3
2. 정보화현황 羐ȃ 3
Ⅲ. 사업 추진방안
1. 추진목표 蓀ȃ 5
2. 추진체계 蓀ȃ 5
3. 추진일정 蓀ȃ 6
4. 추진방안 蓀ȃ 6
Ⅳ. 제안요청 내용
1. 목표시스템 구성도 浨ȃ 7
2. 사업내용 蓀ȃ 7
3. 상세 요구사항 矈ȃ 9
Ⅴ. 제안서 작성요령
1. 제안서의 효력 田ȃ 33
2. 제안서 작성지침 및 유의사항 僠ȃ 33
3. 제안서 목차 穠ȃ 35
4. 세부 작성지침 犘ȃ 36
Ⅵ. 제안 안내 사항
1. 입찰방식 舨ȃ 37
2. 제안 평가관련사항 櫐ȃ 39
3. 제출서류 舨ȃ 42
4. 제안서 제출 일정 및 방법 墨ȃ 42
5. 제안요청 설명회 瀀ȃ 42
6. 제안서 평가 穠ȃ 42
7. 입찰시 유의사항 瀀ȃ 43
8. 제안서 보상 穠ȃ 44
9. 기타사항 舨ȃ 44
Ⅶ. 별첨
[별첨] 입찰서류 및 제안서 제출안내(온라인제출) 㜜ȃ 46
[별첨] 사업자 보안위규 처리기준 墄ȃ 71
[별첨] 보안 위약금 부과 기준 後ȃ 73
[별첨] 누출금지 대상정보 棬ȃ 74
[별첨] 기술적용계획표 濴ȃ 75
楴䵴
Ⅰ.사업 개요
1
추진배경
□ 추진배경
Ⅰ. 사업개요
1
사업일반
□ 사 업 명 : 국립민속박물관 민속아카이브 자료관리시스템 기능개선
□ 사업대상 : 국립민속박물관 민속아카이브 자료관리시스템
□ 사업기간 : 계약 후 2024.12.20.(금)까지
□ 소요예산 : 금77,000,000원(금칠천칠백만원, 부가세 포함)
□ 사업추진방식 : 제한경쟁입찰(협상에 의한 계약)
2
추진배경 및 필요성
□ 민속아카이브 자료관리시스템 업무 프로세스 강화 필요
□ 민속아카이브 자료관리시스템 상태 모니터링 기능 강화 필요
□ 민속아카이브 자료관리시스템 자료 관리 기능 강화 필요
□ 민속아카이브 자료관리시스템 사용자 편의성 개선
3
사업내용
□ 민속아카이브 자료관리시스템 사용자 기능 개선
❍ 가등록 프로세스 내 세부 프로세스 추가
❍ 검색 및 통계 기능 세분화 및 오류 추적 개선
□ 민속아카이브 자료관리시스템 모니터링 기능 강화
❍ 운영관리자 기능 신설, 등록(국가귀속) 이후부터 데이터 로그 추적
❍ 계정관리 그룹의 세분화를 통해 보안성 강화
□ 민속아카이브 자료관리시스템 자료 관리 기능 강화
❍ 수장고별 자료 격납 상황 및 이동사항 추적 기능 개발
❍ 매체별 자료 격납 사항 및 이동사항 추적 기능 개발
□ 기타 추가기능 개발 및 요구사항 분석 등
❍ 민속아카이브 자료관리시스템 연계 기설치 API 재개발
❍ 기타 요구사항 수정 및 개발
4
기대효과
□ 효율적인 업무 프로세스 구축을 통한 업무 사용자의 사용성 개선
□ 자료 현황 추적 기능을 통한 안전하고 정확한 자료 관리 체계 구축
□ 사용자 로그 추적 기능 강화를 통한 자료관리시스템의 보안성 제고
Ⅱ. 시스템 현황


---

### retriever B:
기대 효과
□ (안전한 돌봄 환경 조성) 아이돌봄인력 선별 및 검증 강화 통한 안심하고 맡길 수 있는 돌봄서비스 제공
□ (효율성 제고) 아이돌봄인력 대상 인적성 검사를 편리하게 실시하여 돌봄인력 검증 절차 효율성 제고
□ (정책지원 및 제도개선 지원) 체계적인 검사 결과를 축적하고 정책 수립지원을 위한 기초자료를 제공, 지속 가능한 아이돌봄서비스 제도개선에 활용
□ (돌봄인력 관리) 데이터를 기반으로 하는 아이돌봄인력에 대한 효율적 관리·감독 가능
Ⅱ. 사업 현황
1
업무현황
□ 인·적성검사 실시 정보시스템 지원
ㅇ 인·적성검사 및 실시를 위한 정보 시스템 제공
ㅇ 정보시스템 안정화 및 운영지원
□ 인·적성검사 운영
ㅇ 인적성 검사 실시 및 수행 방법 안내
ㅇ 인적성 검사 수행 모니터링 및 결과 분석
ㅇ 검사 실시 운영자 권한 부여 및 인증 관리
2
정보화 현황
□ 시스템 구성도
□ 정보시스템 구성도 요청 시 별도 공개
※ 행정기관 및 공공기관 정보시스템 구축 및 운영 지침(행정안전부 고시 제 2024-66호, 2024.8.28.) 제 17조(제안요청서 보안사항 등)에 따라, 시스템 구성도 및 상세자료 등은 공개하지 않으며, 입찰에 참가하고자 하는 사업자의 요청이 있는 경우 보안서약서를 제출 후 담당자 입회하에 제한적으로 열람 가능
□ 시스템(장비)·소프트웨어 운영현황
AS-IS
TO-BE (2025년)
ㅇ인·적성 검사 관리기능 및 사용자화면 개발
※ 클라우드 환경 기반 개발
ㅇ인·적성 검사 기능 최적화 및 운영 기술지원
-인·적성 검사 고도화에 따른 수정 반영
(검사도구 4종)
□ 시스템(장비)·소프트웨어 운영현황
AS-IS
TO-BE (2025년)
ㅇ인·적성 검사 관리기능 및 사용자화면 개발
※ 클라우드 환경 기반 개발
ㅇ인·적성 검사 기능 최적화 및 운영 기술지원
-인·적성 검사 고도화에 따른 수정 반영
(검사도구 4종)
※사업 운영에 따라 변경될수 있음
ㅇ무중단 클라우드 기반 제공
ㅇ클라우드 안정화 운영기반 및 기술지원
SW 도입 현황
SW 명칭
용도
비고
리포팅 툴
OZ Report
인·적성 결과지 출력
발주처 제공
□ 시스템 장비 현황 요청 시 별도 공개
※ 행정기관 및 공공기관 정보시스템 구축 및 운영 지침(행정안전부 고시 제 2024-66호, 2024.8.28.) 제 17조(제안요청서 보안사항 등)에 따라, 시스템 구성도 및 상세자료 등은 공개하지 않으며, 입찰에 참가하고자 하는 사업자의 요청이 있는 경우 보안서약서를 제출 후 담당자 입회하에 제한적으로 열람 가능
Ⅲ. 사업 추진방안
1
추진 목표
□ `24년 개발된 인·적성 검사 운영을 지원하고, 데이터 수집 및 분석을 위한 정보체계 관리
□ 인·적성 검사 정보시스템 안정적 기반 지원 및 보안 강화
2
추진체계
□ 추진체계도
발주기관
<한국건강가정진흥원>
협조부서
<경영지원부/
소통협력실>
주관부서
<돌봄지원부>
관련부처
<여성가족부>
주관사업자
<사업수행업체>
□ 추진 주체별 역할
구 분
주요 업무 내용
주관부서
ㅇ 사업총괄 관리 및 시행
ㅇ 사업계획서 수립 및 제안요청서 작성
ㅇ 개발내역 업무정의 및 과업내용 제시
ㅇ 개발관련 자료제공 및 시스템 인수
ㅇ 사업수행 관리 및 검사, 결과 평가
협조부서
2. 과업 목적
ㅇ 수동 중심 검사방식을 디지털·자동화 품질검사가 가능한 도구개발을 통하여 품질 검사시간 단축 및 업무효율성 제고
ㅇ 항해용 간행물 종류 및 품질관리 항목별 오류 정보의 체계적인 관리를 통하여 신속한 품질관리 업무지원
2. 추진 배경 및 필요성
□ 기관생명윤리위원회(IRB) 업무프로세스 구축
ㅇ 기관 생명윤리위원회 규정 및 지침에 의한 위원회 업무 통합 관리 시스템 구축 및 절차 확립
ㅇ 임상연구 심의, 심의결과 통보, 변경신청관리, 결과보고서 관리
□ 동물실험계획 업무 프로세스 구축
ㅇ 기관 동물실험운영규정에 위한 동물실험 업무 통합 관리 시스템 구축 및 절차 확립
ㅇ동물실험계획심의, 승인, 실험계획변경, 실험동물반입, 종료보고서 관리, 사체처리결과서
□ 국가연구개발사업 연구개발비 사용기준 및 회계기준 준수
ㅇ 사전 회의 개최(안) 신청 및 지급신청 연계
ㅇ 구매전용카드 관리, 전자세금계산서 발급신청
□ 기존 시스템 업무 프로세스 개선 및 기능 고도화
ㅇ 연구계획서 기술완성도 목표 관리
ㅇ 연구노트 대장 관리 기능 개선
2. 추진 배경 및 필요성
□ 기관생명윤리위원회(IRB) 업무프로세스 구축
ㅇ 기관 생명윤리위원회 규정 및 지침에 의한 위원회 업무 통합 관리 시스템 구축 및 절차 확립
ㅇ 임상연구 심의, 심의결과 통보, 변경신청관리, 결과보고서 관리
□ 동물실험계획 업무 프로세스 구축
ㅇ 기관 동물실험운영규정에 위한 동물실험 업무 통합 관리 시스템 구축 및 절차 확립
ㅇ동물실험계획심의, 승인, 실험계획변경, 실험동물반입, 종료보고서 관리, 사체처리결과서
□ 국가연구개발사업 연구개발비 사용기준 및 회계기준 준수
ㅇ 사전 회의 개최(안) 신청 및 지급신청 연계
ㅇ 구매전용카드 관리, 전자세금계산서 발급신청
□ 기존 시스템 업무 프로세스 개선 및 기능 고도화
ㅇ 연구계획서 기술완성도 목표 관리
ㅇ 연구노트 대장 관리 기능 개선
2. 시험 운영
· 시스템의 운영지원범위를 기술하여 테스트 등 관리운영에 관한 전반적인 방안을 제시
③ 시스템 운영효율을 높이고, 안정적인 연구지원 서비스 제공을 위해 인프라 현황 분석
ㆍ 관리적 측면
-> 실제 운영에서 발생할 수 있는 데이터 오류, SQL 성능문제 등을 사전검증을 위하여 운영 시스템과 유사한 개발환경 내에서 데이터를 구축하고 이관하는 방안을 제시해야 함.
※ DB 구조의 최적화 및 튜닝 작업을 통해 DB 성능 개선 및 안정성을 보장할 수 있어야 함.
산출정보
DB설계 산출물 일체
관련 요구사항
해당사항 없음
-> 실제 운영에서 발생할 수 있는 데이터 오류, SQL 성능문제 등을 사전검증을 위하여 운영 시스템과 유사한 개발환경 내에서 데이터를 구축하고 이관하는 방안을 제시해야 함.
※ DB 구조의 최적화 및 튜닝 작업을 통해 DB 성능 개선 및 안정성을 보장할 수 있어야 함.
산출정보
DB설계 산출물 일체
관련 요구사항
해당사항 없음
○ 성능 측정에 필요한 소프트웨어 및 프로그램 수정 등 일체는 본 사업에 포함됨
○ 시스템의 특성을 고려하여 목표 성능에 대한 시스템 테스트 계획서를 보험개발원과 협의․확정하고 목표 성능을 보장할 수 있도록 구축
산출정보
성능시험결과서, 성능측정계획서
관련요구사항
1. 시험운영
대상 업무별 단위시험, 통합시험 등에 대한 전반적인 방안을 제시하여야 하고, 개발완료 후의 시스템의 이용 및 관리운영에 관한 전반적인 방안을 제시한다.
1. 시험운영
대상 업무별 단위시험, 통합시험 등에 대한 전반적인 방안을 제시하여야 하고, 개발완료 후의 시스템의 이용 및 관리운영에 관한 전반적인 방안을 제시한다.
요구사항
상세설명
정의
안정적 운영을 위한 시범운영 시행
세부내용
○ 검사 완료 후 시범운영(안정화) 수행
○ 시범운영 및 안정화 기간에 대한 구체적인 계획 제시
○ 일정, 목표, 참여 대상 및 범위, 시스템 구성방안 등
- 일정, 목표, 참여 대상 및 범위, 시스템 구성방안 등
- 시범운영 기간 중 성능저하 프로그램에 대한 튜닝수행방안
산출 정보
테스트 시나리오, 테스트 결과서
관련 요구사항
1. 시험운영
• 대상 업무별 단위시험, 통합시험 등에 대한 전반적인 방안을 제시하여야 하고, 개발완료 후의 시스템의 이용 및 관리운영에 관한 전반적인 방안을 제시한다.
요구사항 명칭
성능 일반
상세설명
정의
성능 일반 요구 목적 정의
세부내용
❍ 기존 네팔 DWRI에서 운영 중인 NIMIS 시스템의 성능을 고려한 개발 방안을 제시하여야 함
❍ 안정적 운영지원 및 사용자 지원 방안을 제시하여야 함
❍ 고도화된 기능은 상시 서비스(연중 하루 24시간) 제공을 위한 가용성을 확보하는 것을 원칙으로 함
- 서비스 가용성 수준은 발주기관과 협의
❍ 서비스가 중단되지 않도록 일정조율 등 사전계획 수립이 필요함
❍ 시스템 개발 중 로그 또는 툴(도구)을 이용하여 시스템 성능 상태를 모니터링 하도록 하여 성능 상 문제를 사전에 파악하고 조치한 후 시스템을 오픈하여야 함
- 시스템 성능 모니터링 또는 성능 시험에 대한 상세한 계획을 제안서에 명시해야 함
❍ 시스템 운영 안정성을 침해하지 않도록 개발
❍ 서비스 응답속도를 보장하기 위해 서버, 응용프로그램, DB 쿼리문 등의 성능을 점검하고 개선 작업을 수행
❍ HW, SW 유지관리 인력과 공동작업 등 시스템 장애 발생 시 신속한 복구가 가능해야 함
❍ 시스템 특성을 고려하여 목표성능에 대한 시스템 테스트 계획서를 발주기관과 협의하여 확정하고 목표성능을 보장할 수 있도록 구축해야 함
❍ 서비스 응답 속도를 보장하기 위해 서버, 응용프로그램, DB 쿼리문 등의 성능을 점검하고 개선 작업을 수행해야 함
산출정보
사업수행 계획서, 사업수행 결과보고서
요구사항 상세설명
정의
비상대책 및 장애대응
세부내용
○ 정보의 유출 및 해킹, 변조 방지 및 정보처리의 무결성을 보장하여야 하며, 중요 문제 발생 시 이를 해결하기 위한 전사적인 역량을 동원하여 대응하여야 함
○ 비상대책 및 장애대응 방안
- 서비스 중단 사태 발생에 대한 긴급조치 등 비상대책을 제시하여야 함
○ 각종 장애발생시 신속한 대처 및 즉각적인 원인분석 및 복구 등 위한 체계적 지원방안, 이를 위한 전문 인력 확보 방안 등을 제시하여야 함
- 프로그램 적용 시 오류발생 등 중요한 장애는 발생즉시 보고/대응하는 방안을 제시하여야 함
※ 응용S/W 사전 점검 방안, 서비스 유지보수 당직제 운영, 신규서비스 적용 및 업무 집중시기에 대한 비상 대응 조직 구성 및 운영 방안 등
· 프로그램 오류에 따른 시스템 장애 등에 대한 대처방안
· 자료의 파손, 변질, 분실 등에 따른 자료의 복구 지원 방안
- 발생된 오류에 대하여 동일한 유형의 오류 재발 방지대책을 제시하여야 함
- 장애조치 완료 후에 처리사항에 대한 철저한 기록 관리를 수행하여야 함
○ 각종 모의 훈련, 시스템 점검에 따른 개선 등을 체계적으로 지원하여야 함
산출정보
사업 수행계획서, 사업 완료보고서
PMR-010
2. 시험운영
대상 업무별 단위시험, 통합시험 등에 대한 전반적인 방안을 제시하고, 개발완료 후, 시스템의 이용 및 관리운영에 관한 전반적인 방안을 제시하여야 함


---

### openai answer
retriever A 점수: 3점
정답의 핵심 요소인 인적성 검사 수행 지원, 돌봄활동 연계 사후관리 강화, 인력(선별·검증) 강화가 모두 명시되어 있음.

retriever B 점수: 2점
인적성 검사 수행 지원과 인력 선별·검증 강화 내용은 있으나, 돌봄활동과 연계한 사후관리 강화를 명시적으로 포함하지 않음. 

============================================================
### query:
아이돌봄 시스템에서 개인정보 보호를 위해 요구되는 조치는 무엇인가?

### answer:
접속기록 관리, 내부관리계획 수립 등 기술적·관리적 보호조치를 수행해야 한다.

---

### retriever A:
- 개인정보의 기술절·관리적 보호조치 기준(개인정보위)
- 표준 개인정보 보호 지침(개인정보위)
가명정보의 결합 및 반출 등에 관한 고시(개인정보위)
개인정보 영향평가에 관한 고시(개인정보위)
※ 고시에 따라 아래 기준 이상의 개인정보 처리 시 발주처에서 개인정보 영향평가를 발주하며 영향평가결과를 반영하여 시스템을 구축해야 함
내 용
1
구축‧운용 또는 변경하려는 개인정보파일로서 5만명 이상의 정보주체에 관한 민감정보 또는 고유식별정보의 처리가 수반되는 개인정보파일
2
구축‧운용하고 있는 개인정보파일을 해당 기관 내부 또는 외부에서 구축‧운용하고 있는 다른 개인정보파일과 연계하려는 경우로서 연계결과 50만명 이상의 정보주체에 관한 개인정보가 포함되는 개인정보파일
3
구축‧운용 또는 변경하려는 개인정보파일로서 100만명 이상의 정보주체에 관한 개인정보파일
4
개인정보 영향평가를 받은 후에 개인정보 검색체계 등 개인정보파일의 운용체계를 변경하려는 경우 그 개인정보파일
○ 개인정보 안전성 확보조치 기준 이행·반영
- 개인정보 열람 시 접속기록(최소 2년 이상) 보관·관리
- 개인정보 접근권한 부여·변경·말소 이력(최소 3년 이상) 보관·관리
- 고유식별번호 및 비밀번호 생체인식정보는 안전한 암호 알고리즘을 사용 하여 저장
○ 데이터의 보안 강화를 위해 SSL서비스 도입
○ 개인정보보호를 위한 DB 암호화하여 자료 유출방지
- 인증서 기반(GPKI) 로그인 방식 적용 후 ID/PW 로그인
- 정보보안 강화를 위한 시스템에 각종 패치, 환경설정 지원 등
◦ 보안 요구사항
요구사항 고유번호
SER-001
요구사항 명칭
개인정보 처리
요구사항 분류
보안 요구사항
요구사항
상세설명
정의
개인정보 처리 및 암호화
세부
내용
ㅇ 개인정보를 보유할 경우에는 관련 개인정보를 암호화하여야 함
ㅇ 서비스의 보안성 강화를 위하여 SSL 보안 통신을 실시하여야 함
ㅇ 사업의 암호화 대상을 식별하고 암호화 도구를 활용하여 암복호화 방안을 마련해야 함
- 고유식별정보 : 주민번호, 여권번호, 운전면허정보 등
ㅇ개인을 식별할 수 있는 정보(사용자 인증 정보, 패스워드 등)를 운영DB 또는 개발DB에 저장할 경우, 암호화하여 저장하며, 소스코드에 직접 하드코딩하지 않음(패스워드는 단방향 암호화 처리, SHA-256 이상 암호화 알고리즘 사용)
ㅇ 개인정보 취급자가 개인정보처리 시스템에 접속하는 경우 접속 로그를 시스템에 최소 1년 이상 기록 관리해야 함(5만명 이상의 정보주체에 관하여 개인정보를 처리하거나 고유식별정보 또는 민감정보를 처리하는 개인정보처리시스템의 경우에는 2년 이상 보관)
◈ 필수 관리 항목
• 개인정보취급자 식별정보 : ID ୴Ā • 날짜 및 시간 : 접속일시
• 접속자 IP 주소 : 접속자 정보
• 처리한 정보주체 정보 : 정보주체 식별정보(계정, 회원번호 등)
• 수행업무 : 열람, 수정, 삭제, 인쇄, 입력, 다운로드 등
ㅇ 개인정보 취급자가 개인정보를 처리할 경우 해당 접근 로그를 시스템에 기록할 것
ㅇ 개인정보처리방침에 따라 개인정보를 처리/파기해야 하며 백업데이터 생성 금지
개인정보 암호화
(제6조)
◦암호화 대상 : 고유식별정보(주민등록번호, 여권번호, 운전면허번호, 외국인등록번호), 비밀번호, 바이오정보
◦암호화 기준
구 분
암 호 화 기 준
송․수신시
모든 정보 암호화 전송
저장시
비밀번호,
바이오정보
비밀번호는 단방향 암호화,
바이오정보는 양방향 암호화
고유식별정보
ⓛ 인터넷 구간,공통(DMZ) 구간 저장시 : 의무적 암호화
② 내부망 저장시 : 위험도 분석결과에 따라 암호화 적용범위 결정
(※ 영향평가 대상 공공기관은 영향평가 결과에 따름)
* 고유식별정보가 유출・변조・훼손되지 않도록 연1회 이상 취약점 점검
주민등록번호
암호화 알고리즘 적용 권고(’16.1.1, 의무시행)
◦암호화 방법 : 공공기관은 국가 암호화 알고리즘, 그 외는 자율
◦암호화 적용 : 고유식별정보 암호화 시 지침 시행일로부터 3개월 이내에 암호화 계획 수립, 계획 수립일로부터 12개월 이내에 암호화 적용
접속기록 보관·점검(제7조)
◦개인정보처리시스템 접속기록은 6개월 이상 보관
※ DBMS의 접속기록을 반기별 1회 이상 점검
◦접속기록 반기별 점검 의무화
악성프로그램 등 방지(제8조)
◦보안 프로그램(백신 S/W 등) 설치 및 업데이트 ※ 자동 업데이트 또는 일 1회 이상
물리적 접근 방지
(제9조)
◦별도의 개인정보 물리적 보관 장소에 대한 출입통제 등
개인정보의 파기
(제10조)
◦복원이 불가능한 개인정보의 파기방법 구체화
부칙
◦전산센터․클라우드컴퓨팅센터 등의 운영환경에서의 안전조치
2. 개인정보처리시스템 위탁 관리 철저
❍ 위탁목적 등 문서화(「개인정보보호법」제26조)
- 시스템 및 DBMS에 주요 개인정보를 저장할 경우 암호화 및 접근제어 적용
- 정보통신망을 통하여 개인정보를 송·수신하는 경우 통신 구간에 대한 암호화 적용
- 워크넷 및 외부 채용 플랫폼 정보 연계 등 api 이용 외부 시스템과의 연동 시, 구간 암호화 및 전송 내용 암호화 등 보안 조치
- 이용자 대상 만족도 조사 시, 계약 상대자는 만족도 조사 업무를 필수로 포함한 개인정보 위·수탁 계약서를 체결해야 하며, 이용자의 개인정보 수집·이용 및 제공 동의를 진행 후 실시하여야 함
- 쿠키 등과 같이 개인정보를 자동으로 수집하는 장치를 설치 및 운영할 경우, 설치·운영 및 그 거부에 관한 사항을 명시하여야 하며, 개인정보 처리방침에는 쿠키 정보에 대한 기본적인 설명과 사용하는 이유 및 방법 그리고 목적을 설명하여야 함
※ 브라우저 쿠키정보 활용 시, 서드 파티 쿠키가 아닌 퍼스트 파티 쿠키 수집을 이용
○ 개인정보보호법에서는 개인정보를 분실, 도난, 유출, 제3자에게 제공하는 경우 2년 이하의 징역 또는 2천만원 이하의 벌금에 처하고, 그 행위자뿐 아니라 사업자도 양벌규정을 명시하고 있으므로 이를 유념하여 사업자는 개인정보 업무에 철저를 기하도록 자체 교육을 실시해야 함
○ 개인정보 처리업무 위탁시 의무 조치사항과 관련하여, 수행업체는 계약체결 후 10일 내 ‘개인정보처리위탁 계약서’를 발주기관에 제출해야 함
○ 자동화된 결정에 대한 정보주체의 권리 등(개인정보보호법 제37조의 2, 2024.3.15. 시행)
개인정보파일의 등록 및 공개(제32조)
개인정보의 열람ㆍ정정ㆍ삭제요청에 대한 준수(제35조, 제36조, 제37조)
의 안전조치 의무기준
구분
주요 내용
내부관리계획
(제3조)
o 내부관리계획 포함사항 명시 ※ 개인정보 보호책임자 지정, 개인정보취급자 교육 등
접근권한 관리
(제4조)
o 개인정보처리시스템의 접근권한 부여․변경․말소 기록 최소 3년간 보관
비밀번호 관리
(제5조)
o 안전한 비밀번호 작성규칙 수립․적용
접근통제 시스템 설치․운영(제6조)
o 접근통제 시스템 설치․운영 의무화
※ 접근통제 시스템 : 침입차단(Firewall), 침입탐지(IDS) 기능 포함
개인정보 암호화
(제7조)
o 암호화 대상 : 고유식별정보(주민등록번호, 여권번호, 운전면허번호, 외국인등록번호), 비밀번호, 바이오정보
o 암호화 기준
구 분
암 호 화 기 준
송․수신시
모든 정보 암호화 전송
저
장
시
비밀번호,바이오정보
(비밀번호)단방향 암호화, (바이오정보)양방향 암호화
고유식별정보
ⓛ 인터넷 구간,공통(DMZ) 구간 저장시 : 의무적 암호화
② 내부망 저장시 : 위험도 분석결과에 따라 암호화 적용범위 결정(※ 영향평가 대상 공공기관은 영향평가 결과에 따름)
o 암호화 방법 : 공공기관은 국가 암호화 알고리즘, 그 외는 자율
o 암호화 적용 : 고유식별정보 암호화시 지침 시행일로부터 3개월 이내에 암호화 계획 수립, 계획 수립일로부터 12개월 이내에 암호화 적용
접속기록 보관(제8조)
o 개인정보처리시스템 접속기록은 6개월 이상 보관
보안프로그램
(제9조)
※ 비밀번호를 저장하는 경우에는 일방향 암호화하여 저장(SHA-256 비트 이상)
- 고유식별정보를 인터넷 및 DMZ 구간에 저장하는 경우, 암호화하여야 함
 접속기록의 보관 및 점검
- 접속기록에는 개인정보취급자가 시스템에 접속하여 처리한 업무내역을 알 수 있도록 아래의 항목들을 기록하도록 개발(구축)하여야 함
· 계정 : 개인정보처리시스템에서 접속자를 식별할 수 있도록 부여된 정보(ID등)
· 접속일시 : 접속한 시간 또는 업무를 수행한 시간(년-월-일, 시:분:초)
· 처리한 정보주체 정보 : 개인정보취급자가 누구의 개인정보를 처리하였는지 알 수 있는 식별정보(ID, 고객번호, 학번, 사번 등)
- 해당 접속기록의 이력 관리 또는 조회가 가능하여야 하며, 해당 기록을 1년 이상 또는 2년 이상 보관·관리하도록 구현
· 5만 명 미만의 정보주체에 관하여 개인정보를 처리하는 경우 : 1년 이상 보관
· 5만 명 이상의 정보주체에 관하여 개인정보를 처리하는 경우 : 2년 이상 보관
- 개인정보를 다운로드할 경우, 그 사유를 확인할 수 있어야 함
산출정보
관련요구사항
요구사항번호
SER-003
요구사항 명
개인정보의 처리단계별 보호조치
요구사항 분류
보안 요구사항
상세
설명
정의
개인정보의 안전성 확보를 위한 처리단계별 조치사항
세부
내용
 개인정보 수집 및 이용 대한 동의 : 『개인정보 보호법』 제15조~제18조
- 백시소프트웨어 등 보안프로그램 설치·운영 그러나 동의를 거부할 경우 원활한 ○○○ 서비스 제공에 일부 제한을 받을 수 있습니다.
- 관리용 단말기 안전조치 방법
☞ 위와 같이 개인정보를 수집·이용하는데 동의하십니까? 동의 미동의
- 개인정보 송·수신 시 암호화 등 안전조치 방법
- 재해·재난 대비 개인정보보호 방법
- 개인정보 침해사고 대응 조치 방법
□ 고유식별정보 수집‧이용 내역(수집 시 필수)
- 개인정보 파기 기준 및 방법
- 개인정보의 안전한 보관을 위한 물리적 안전조치 방법 항목 보유기간
수집․이용 목적 수집 근거(필수)
※“등”표시하면안됩니다. ※산정기준[별표 2] 참조
○ 업무 위탁에 따른 개인정보의 처리제한에 관한 사항
주민등록번호,운전면허번호, 「OOO에관한
- 위탁업무 범위를 초과하여 개인정보 이용 및 제공 금지 OOO서비스관리 3년
외국인등록번호,여권번호 법률」제OO조
- 재위탁 제한에 관한 사항
※ 위의 개인정보 수집‧이용에 대한 동의를 거부할 권리가 있습니다.
기타사항 그러나 동의를 거부할 경우 원활한 ○○○ 서비스 제공에 일부 제한을 받을 수 있습니다.
☞ 위와 같이 개인정보를 수집·이용하는데 동의하십니까? 동의 미동의
- 112 - - 113 -
○○ 사이트 회원 가입을 위한 개인정보 수집‧이용 및 제공 동의서 ○○ 사이트 회원 가입을 위한 개인정보 수집‧이용 및 제공 동의서
□ 민감정보 처리 내역(수집 시 필수)
년 월 일
항목 보유기간
수집․이용 목적 수집 근거(선택)


---

### retriever B:
6. 그 밖에 개인정보의 보호를 위하여 필요한 사항
-295-
1. 개인정보의 안전성 확보조치 철저(「개인정보보호법」제29조)
❍ 개인정보처리자는 개인정보가 분실·도난·유출·변조 또는 훼손되지 않도록 다음 각 호의 안전성 확보조치를 하여야한다.
구 분
주 요 내 용
내부관리계획(제3조)
◦내부관리계획 포함사항 명시 ※개인정보 보호책임자 지정, 개인정보취급자 교육 등
◦개인정보 처리 업무 위탁 시 관리·감독 사항
접근권한 관리(제4조)
◦개인정보처리시스템의 접근권한 부여․변경․말소 기록 최소 3년간 보관
◦안전한 비밀번호 작성규칙 수립․적용
접근통제(제5조)
◦접근통제 시스템 설치․운영 의무화 ※접근통제 시스템 : 침입차단(Firewall), 침입탐지(IDS) 기능 포함
◦주민번호 처리 시 추가 정보 확인, 모바일 기기 보호조치 조항, 홈페이지에서 고유식별정보를 처리할 경우 년 1회 취약점 점검 조치
개인정보 암호화
(제6조)
◦암호화 대상 : 고유식별정보(주민등록번호, 여권번호, 운전면허번호, 외국인등록번호), 비밀번호, 바이오정보
◦암호화 기준
구 분
암 호 화 기 준
송․수신시
모든 정보 암호화 전송
저장시
비밀번호,
바이오정보
비밀번호는 단방향 암호화,
바이오정보는 양방향 암호화
고유식별정보
ⓛ 인터넷 구간,공통(DMZ) 구간 저장시 : 의무적 암호화
보안 요구사항
상세
설명
정의
개인정보의 안전성 확보를 위한 기술적 조치사항
세부
내용
 개인정보를 처리함에 있어 『개인정보 보호법』, 『개인정보의 안전성 확보조치 기준』 등 관련 법규 및 규정을 준수하여야 함
 접근 권한 관리
- 개인정보처리시스템에 접속할 수 있는 사용자계정을 발급하는 경우, 개인정보취급자 별로 사용자계정을 발급하여야 하며, 취급자의 역할에 따라 차등부여할 수 있어야 함
- 개인정보 처리내역에 대한 책임 추적성을 확보하기 위하여 접근권한 부여, 변경, 말소에 대한 내역을 이력 관리 또는 조회가 가능하여야 하며, 해당 기록을 최소 3년간 보관할 수 있도록 구현해야 함
예) 신청자 정보, 신청일시, 승인자 및 발급자 정보, 신청 및 발급 사유 등
- 개인정보처리시스템에 접근하여 개인정보를 생성수정삭제열람 등을 하고자 하는 경우 다음 사항을 포함하는 정보이용 내역을 기록하고 남길 수 있도록 시스템을 구현하여야 함.
- 개인정보의 안전성 확보조치 기준(개인정보보호위원회고시) 준용
❑ 각 기능에 명시된 개인정보는 반드시 암호화하여 DB에 저장하여야 함
❑ 관리자인 경우에도 화면에 해당 정보를 그대로 노출시키지 않아야 함
❑ 암호화는 재단에서 제공하는 DB암호화 솔루션을 적용
산출정보
서, 개인정보처리위탁계약서, 수탁사 개인정보 안전성확보조치 관련 관리 감독 보고서
4. 개인정보처리 준수사항
□ 개인정보의 안전성 확보조치 철저(개인정보보호법 제29조)
❍ 개인정보처리자는 개인정보가 분실ㆍ도난ㆍ유출ㆍ변조 또는 훼손되지 않도록 다음 각 호의 안전성 확보조치를 하여야함.
구 분
주 요 내 용
내부관리계획(제4조)
▪내부관리계획 포함사항 명시(※ 개인정보 보호책임자 지정, 개인정보취급자 교육 등)
접근권한 관리(제5조)
▪개인정보처리시스템의 접근권한 부여ㆍ변경ㆍ말소 기록 최소 3년간 보관
접근통제 시스템 설치ㆍ운영(제6조)
▪접근통제 시스템 설치ㆍ운영 의무화
※ 접근통제 시스템 : 침입차단(Firewall), 침입탐지(IDS) 기능 포함
개인정보 암호화
(제7조)
▪암호화 대상 : 고유식별정보(주민등록번호, 여권번호, 운전면허번호, 외국인등록번호), 비밀번호, 바이오정보
▪암호화 기준
구 분
암 호 화 기 준
송ㆍ수신시
모든 정보 암호화 전송
저장시
비밀번호,
바이오정보
▪비밀번호는 단방향 암호화,
▪바이오정보는 양방향 암호화
고유식별
정보
ⓛ 인터넷 구간, 공통(DMZ) 구간 저장시 : 의무적 암호화
4. 데이터 취합 시 개인정보에 대한 보안 방안을 제시하여야 함
⑤ 개인정보에 대한 접근 제한 등 안전성 확보 조치에 관한 사항
❍ 개인정보의 안전성 확보조치 기준(개인정보보호위원회 고시 제2021-2호) 준수
❍ 표준 개인정보 보호지침(개인정보보호위원회 고시 제2020-1호) 준수
❍ 개인정보 수집제공 동의서 작성 가이드라인(2018)
❍ 주민등록번호 수집 금지 제도 가이드라인(2014)
❍ 개인정보 수집 최소화 가이드라인(2020.12)
❍ 홈페이지 개인정보 노출방지 안내서(2020.12)
❍ 웹 취약점 점검 결과와 취약점 조치 완료 보고서 제출
❍ 그 밖에 명시되지 않은 관련법규 및 가이드라인은 본 과업을 수행함에 있어 표준을 준수
❍ 설계단계 완료 이전에 위 표준(지침)이 변경되거나 새 표준이 발표될 때는 변경된 내용 및 새 표준을 반영하여야 함
❍ 보안관리자를 지정하고, 프로그램 소스코드 및 관련 S/W의 유출을 방지하기 위해 디스크, CD, 출력물은 별도 관리하여야 함.
❍ 불의의 사고로 인하여 시스템 및 파일이 피해를 입더라도 최근에 백업한 시점의 내용으로 복구할 수 있는 백업정책 수립 및 구현
❍ 긴급 재해·재난 발생 시 데이터를 보호하며 시스템 Off와 전원을 차단 할 수 있는 시스템 구현
2. DB보안 및 개인정보보호법 준수
❍ 사용자 인증 정보, 주민등록번호, 비밀번호, 계좌번호 등 개인을 식별할 수 있는 정보를 운영DB 또는 개발DB에 저장할 경우, 암호화하여 저장하며 소스코드에 직접 하드코딩하지 않아야 함
❍ 사용자 정보 등 개인정보 취급 시 반드시 개인정보보호법과 관련 지침을 준수함
❍ DB암호화 솔루션을 통하여 저장되는 콘텐츠 중 개인정보 및 보안을 요구하는 데이터들에 대한 암호화 및 관리 기능 적용
3. 보안 요구사항
․ 시스템 보안성 확보방안 및 구체적 개인정보보호 방안 제시
개인정보파일의 등록 및 공개(제32조)
개인정보의 열람ㆍ정정ㆍ삭제요청에 대한 준수(제35조, 제36조, 제37조)
의 안전조치 의무기준
구분
주요 내용
내부관리계획
(제3조)
o 내부관리계획 포함사항 명시 ※ 개인정보 보호책임자 지정, 개인정보취급자 교육 등
접근권한 관리
(제4조)
o 개인정보처리시스템의 접근권한 부여․변경․말소 기록 최소 3년간 보관
비밀번호 관리
(제5조)
o 안전한 비밀번호 작성규칙 수립․적용
접근통제 시스템 설치․운영(제6조)
o 접근통제 시스템 설치․운영 의무화
※ 접근통제 시스템 : 침입차단(Firewall), 침입탐지(IDS) 기능 포함
개인정보 암호화
(제7조)
o 암호화 대상 : 고유식별정보(주민등록번호, 여권번호, 운전면허번호, 외국인등록번호), 비밀번호, 바이오정보
o 암호화 기준
구 분
암 호 화 기 준
송․수신시
모든 정보 암호화 전송
저
장
시
비밀번호,바이오정보
(비밀번호)단방향 암호화, (바이오정보)양방향 암호화
고유식별정보
ⓛ 인터넷 구간,공통(DMZ) 구간 저장시 : 의무적 암호화
4. 개인정보의 기술적·관리적 보호조치에 관한 사항
※ 개인정보에 대한 접근 제한 등 안전성 확보 조치에 관한 사항
4. 개인정보처리 준수사항
□ 개인정보의 안전성 확보조치 철저(개인정보보호법 제29조)
❍ 개인정보처리자는 개인정보가 분실ㆍ도난ㆍ유출ㆍ변조 또는 훼손되지 않도록 다음 각 호의 안전성 확보조치를 하여야함.
구 분
주 요 내 용
내부 관리계획(제4조)
▪내부관리계획 포함사항 명시
※ 개인정보 보호책임자 지정, 개인정보취급자 교육 등
접근권한(제5조)
▪개인정보처리시스템의 접근권한 부여ㆍ변경ㆍ말소 기록 최소 3년간 보관
접근통제(제6조)
▪접근통제 시스템 설치ㆍ운영 의무화
※ 접근통제 시스템 : 침입차단(Firewall), 침입탐지(IDS) 기능 포함
암호화(제7조)
▪암호화 대상 : 고유식별정보(주민등록번호, 여권번호, 운전면허번호, 외국인등록번호), 비밀번호, 생체인식정보
▪암호화 기준
구 분
암 호 화 기 준
송ㆍ수신시
모든 정보 암호화 전송
저장시
비밀번호,
생체인식정보
▪비밀번호는 단방향 암호화,
▪생체인식정보는 양방향 암호화
고유식별정보
ⓛ 인터넷 구간, 공통(DMZ) 구간 저장시 : 의무적 암호화
6. 그 밖에 개인정보의 보호를 위하여 필요한 사항
6. 그 밖에 개인정보의 보호를 위하여 필요한 사항
6. 그 밖에 개인정보의 보호를 위하여 필요한 사항
6. 그 밖에 개인정보의 보호를 위하여 필요한 사항
6. 그 밖에 개인정보의 보호를 위하여 필요한 사항
6. 그 밖에 개인정보의 보호를 위하여 필요한 사항
6. 그 밖에 개인정보의 보호를 위하여 필요한 사항
6. 그 밖에 개인정보의 보호를 위하여 필요한 사항
6. 그 밖에 개인정보의 보호를 위하여 필요한 사항
6. 그 밖에 개인정보의 보호를 위하여 필요한 사항
6. 그 밖에 개인정보의 보호를 위하여 필요한 사항
6. 그 밖에 개인정보의 보호를 위하여 필요한 사항
6. 그 밖에 개인정보의 보호를 위하여 필요한 사항
6. 그 밖에 개인정보의 보호를 위하여 필요한 사항
6. 그 밖에 개인정보의 보호를 위하여 필요한 사항
6. 그 밖에 개인정보의 보호를 위하여 필요한 사항
6. 그 밖에 개인정보의 보호를 위하여 필요한 사항
6. 그 밖에 개인정보의 보호를 위하여 필요한 사항
6. 그 밖에 개인정보의 보호를 위하여 필요한 사항
6. 그 밖에 개인정보의 보호를 위하여 필요한 사항
6. 그 밖에 개인정보의 보호를 위하여 필요한 사항
6. 그 밖에 개인정보의 보호를 위하여 필요한 사항
6. 그 밖에 개인정보의 보호를 위하여 필요한 사항
6. 그 밖에 개인정보의 보호를 위하여 필요한 사항
6. 그 밖에 개인정보의 보호를 위하여 필요한 사항
6. 그 밖에 개인정보의 보호를 위하여 필요한 사항
6. 그 밖에 개인정보의 보호를 위하여 필요한 사항
6. 그 밖에 개인정보의 보호를 위하여 필요한 사항


---

### openai answer
retriever A 점수: 3점
- 접속기록 보관·점검(제7조)과 내부관리계획(제3조)이 명시되어 있고, 기술적·관리적 보호조치 전반(접근권한 관리, 암호화 등)이 구체적으로 포함되어 있어 정답의 내용이 모두 충족됨.

retriever B 점수: 2점
- 내부관리계획 등 기술적·관리적 보호조치는 제시되어 있으나, 접속기록 관리에 대한 명시가 부족하여 정답 요소를 부분적으로만 충족함. 

============================================================
### query:
2025 구미 아시아육상경기선수권대회의 개최 기간은 언제인가?

### answer:
2025년 5월 27일부터 5월 31일까지 총 5일간 개최된다.

---

### retriever A:
종합정보시스템 및 홈페이지 등 구축 용역 제안요청서
1
대회개요
아시아의 꿈, 구미에서 세계로
 The dream of ASIA, From GUMI to the World 
비전
ㅇ (대 회 명) 2025 구미아시아육상경기선수권대회
ㅇ (기 간) 2025. 5. 27.(화) ~ 5. 31.(토) 5일간
ㅇ (장 소) 구미시민운동장 일원
ㅇ (규 모) 아시아 45개국 1,200여명 / 선수단 및 임원
ㅇ (주 최) 아시아육상연맹(Asian Athletics, AA)
ㅇ (주 관) 2025 구미아시아육상경기선수권대회 조직위원회
ㅇ (경기종목) 45개 종목
구 분
종 목 명
남자
(22)
100m, 200m, 400m, 800m, 1,500m, 5,000m,
10,000m, 110mH, 400mH, 3,000m장애물달리기,
20km경보, 4×100mR, 4×400mR, 높이뛰기,
장대높이뛰기, 멀리뛰기, 세단뛰기, 포환던지기,
원반던지기, 해머던지기, 창던지기, 10종 경기
여자
(22)
100m, 200m, 400m, 800m, 1,500m, 5,000m,
10,000m, 100mH, 400mH, 3,000m장애물달리기,
20km경보, 4×100mR, 4×400mR, 높이뛰기,
장대높이뛰기, 멀리뛰기, 세단뛰기, 포환던지기,
원반던지기, 해머던지기, 창던지기, 7종 경기
남·여
(1)
4×400m 믹스릴레이
요
2
과업개요
ㅇ (사 업 명) 2025 구미아시아육상경기선수권대회 종합정보시스템 및 홈페이지 등 구축 용역
ㅇ (사업기간) 계약일 ~ 2025. 6. 6.(금)
ㅇ (사업위치) 시민운동장 일원
2024. . .
대표자 : (인)
대리인 : (인)
2025 구미아시아육상경기선수권대회 조직위원장 귀하
※ 첨부서류 : 재직증명서, 신분증
※ 위임장에 사용하는 도장은 대표자 인감과 같아야 함.
※ 대리 접수로 인하여 발생하는 모든 문제는 대표자 본인이 책임짐
【 서식 3 】
재 직 증 명 서
성 명
생 년 월 일
부서/직위
소 속
주 소
재직기간
위와 같이 당사에 재직 중임을 증명합니다.
2024. . .
ྠĀ ྠĀ ྠĀ ྠĀ ྠĀ ྠĀ 업 체 명 :
ྠĀ ྠĀ ྠĀ ྠĀ ྠĀ ྠĀ 주 소 :
ྠĀ ྠĀ ྠĀ ྠĀ ྠĀ ྠĀ 대 표 자 : (인)
2025 구미아시아육상경기선수권대회 조직위원장 귀하
【 서식 4 】
사 용 인 감 계
사용인감
는(은) 2025 구미아시아육상경기선수권대회 조직위원회와 거래함에 있어 입찰, 계약 및 이에 수반되는 모든 행위에 대하여 위의 인감을 사용하고자 하며, 동 인감을 사용함으로써 민․형사상의 문제가 발생할 경우 이(가) 책임을 질 것을 확인합니다.
붙임 : 인감(법인)증명서 1부. 끝.
2024년 월 일
ྠĀ ྠĀ ྠĀ ྠĀ ྠĀ ྠĀ 주 소 :
ྠĀ ྠĀ ྠĀ ྠĀ ྠĀ ྠĀ 상 호 :
ྠĀ ྠĀ ྠĀ ྠĀ ྠĀ ྠĀ 대표자 : (인)
2025 구미아시아육상경기선수권대회 조직위원장 귀하
【 서식 5 】
서 약 서
당사는 2025 구미아시아육상경기선수권대회 종합정보시스템 및 홈페이지 등 구축 용역 참가신청과 관련하여 모든 서류와 증빙자료를 신의와 성실의 원칙에 입각하여 작성 제출하며, 아래 내용을 준수할 것을 확약하고 이에 서약서를 제출합니다.
2025 구미아시아육상경기선수권대회 조직위원장 귀하
【 서식 5 】
서 약 서
당사는 2025 구미아시아육상경기선수권대회 종합정보시스템 및 홈페이지 등 구축 용역 참가신청과 관련하여 모든 서류와 증빙자료를 신의와 성실의 원칙에 입각하여 작성 제출하며, 아래 내용을 준수할 것을 확약하고 이에 서약서를 제출합니다.
1. 당사는 제안요청서에 기술된 내용 및 절차를 인정하고, 관련 지침을 성실히 준수하겠으며, 본 지침을 위해할 시 어떠한 불이익도 감수 하겠습니다.
2. 본 용역과 관련한 사업비는 변동될 수 있으므로 변동금액에 대하여 일체의 이의를 제기하지 않겠습니다.
3. 제안서 평가를 위해 구성된 평가위원회의 심사방법 및 심사 기준, 평가결과에 어떠한 이의제기도 하지 않겠습니다.
2024년 월 일
신청자
업 체 명
:
대표자
:
(인)
2025 구미아시아육상경기선수권대회 조직위원장 귀하
【 서식 6 】
보 안 서 약 서
본인은 2025 구미아시아육상경기선수권대회 종합정보시스템 및 홈페이지 등 구축 용역을 수행함에 있어 다음 사항을 준수할 것을 엄숙히 서약합니다.
1. 본인은 위 용역과 관련하여 귀 조직위원회에서 수령한 자료 및 정보를 관리함에 철저를 기하겠으며, 무단으로 복제하거나 유출하지 아니 할 것을 서약합니다.
2. 본인은 용역 기간은 물론 이후에도 일체 타인에게 누설하지 아니할 것을 서약하며, 용역을 성실히 수행하겠습니다.
3. 본인은 위에서 서약한 사항을 위반함으로써 부과되는 민․형사상의 책임을 감수하겠음을 서약합니다.
2024년 월 일
서약자 주 소 :
업체명 :
대표자 : (인)
 운영 매뉴얼 또는 지침서 제공 및 시스템 운영 관계자 교육
ㅇ (현장IT팀 상주 및 시스템 운영)
 사전 등록, 경기 진행 및 결과 출력 시스템 등 운영
 대회시작 4일전~대회 종료일까지 경기장 상주
 경기장/비경기장별 운영 인원 배치
ㅇ (홈페이지 구축)
 메뉴 구성도(안)
아시아육상경기선수권대회 메인
대회소개
소식지
경기일정/결과
메달순위
육상경기
커뮤니티
환영인사
공지사항
경기일정
국가별순위
경기종목
안내사항
대회개요
보도자료
행사일정
선수별순위
참가선수
아카이브
대회상징물
사진자료
시설안내
유실물센터
조직위안내
자주묻는질문
홍보영상
자원봉사자
찾아오시는 길
구미시 안내
메뉴 상세설명(안)
메뉴명
부메뉴
주요 내용
메인 화면
◦2025 구미 아시아육상경기선수권대회를 대표할 수 있는 디자인 (공식 상징물 활용 등)
◦활용빈도가 높은 메뉴 배치 및 팝업배너
대회 소개
환영인사
◦대회 개최에 대한 인사 말씀
대회개요
◦대회명칭, 대회기간, 대회장소, 참가규모, 참가국, 참가인원, 종목, 주최, 주관 등
대회상징물
◦대회마크, 마스코트, 포스터, 슬로건
조직위안내
◦조직위원회·아시아육상추진단 조직도
홍보영상
◦홍보영상, 유튜브 쇼츠 영상, 축하 메시지 등
찾아오시는 길
◦경기장 위치, 교통수단별 길 안내
소식지
공지사항
◦대회 관련 공지사항 등
보도자료
◦대회 관련기사 스크랩 자료
사진자료
◦개·폐회식 사진, 각 종목별 경기 이미지 게시
경기일정
/
결과
경기일정
◦경기장 및 개최 일자, 시간, 라운드, 진행상태
◦각 종목별 경기결과, 종합순위
행사일정
◦개·폐회식 정보 및 식순 안내
장대높이뛰기, 멀리뛰기, 세단뛰기, 포환던지기,
원반던지기, 해머던지기, 창던지기, 7종 경기
남·여
(1)
4×400m 믹스릴레이
요
2
과업개요
ㅇ (사 업 명) 2025 구미아시아육상경기선수권대회 종합정보시스템 및 홈페이지 등 구축 용역
ㅇ (사업기간) 계약일 ~ 2025. 6. 6.(금)
ㅇ (사업위치) 시민운동장 일원
ㅇ (사 업 비) 260,000,000원 ※ 부가가치세 포함
ㅇ (사업내용)
 아시아육상대회를 위한 종합정보시스템 구축 및 운영
 종합정보시스템 운영을 위한 대회 정보화 시설 구축
 경기 결과와 계측기, 전광판, 방송 장비, 인터넷 정보시스템의 연동
 종합정보시스템을 연계한 아시아육상대회 홈페이지 구축·관리
2. 추진배경 및 필요성
ㅇ 성공적인 대회 운영과 개최를 위하여 대회를 널리 알리고자 대회정보와 행사정보 등을 제공하는 등 대회 전반에 대해 홍보하고자 함
ㅇ 계측장비, 운동장 전광판(표출기기), 음향·방송기기 등과 연동하여 경기 일정, 결과, 기록, 인적자료(선수) 등의 각종 데이터를 신속하게 전달하고 데이터화 하기 위한 통합 시스템 필요
ㅇ IT 기술과 관련 소프트웨어 기술을 바탕으로 대회 종합정보시스템을 운영하여 대회 관계자와 선수, 관객에게 편의제공은 물론 대회 운영 및 관리에도 효율성을 높여 2025 구미아시아육상경기선수권대회의 성공개최를 도모하고자 함
3. 사업내용
ㅇ (종합정보시스템 구축을 위한 요구분석 및 상세설계)
 종합정보시스템 업무에 대한 요구사항 분석
 대회 기본자료, 경기운영, 인터페이스, 보고서 관리 등 시스템별 상세설계
❍ 입찰, 계약체결 및 계약이행과 관련하여 관계 직원에게 금품, 향응 등을 제공한 사실이 드러날 경우에는 귀 조직위원회에서 발주하는 입찰에 입찰참가 자격제한 처분을 받은 날로부터 6개월 동안 참가하지 않겠습니다.
3. 입찰, 계약체결 및 계약이행과 관련하여 관계 직원에게 금품, 향응 등을 제공한 사실이 드러날 경우에는 계약체결 이전 일 때 낙찰자 결정취소, 공사 착공 전에는 계약취소, 공사착공 이후에는 발주처에서 전체 또는 일부계약을 해지하여도 감수하고 민․형사상 이의를 제기하지 않겠습니다.
4. 회사 임․직원이 관계 직원에게 금품, 향응 등을 제공하거나 담합 등 불공정 행위를 하지 않도록 하는 회사 윤리강령과 내부비리 제보자에 대해서도 일체의 불이익처분을 하지 않는 사규를 제정토록 노력하겠습니다.
위 청렴계약 서약은 상호 신뢰를 바탕으로 한 약속으로서 반드시 지킬 것이며, 낙찰자로 결정될시 본 서약 내용을 그대로 계약 특수조건으로 계약하여 이행하고, 입찰참가 자격제한, 계약해지 등 귀 조직위원회의 조치와 관련하여 당사자가 귀 조직위원회를 상대로 손해배상을 청구하거나 당사를 배제 하는 입찰에 관하여 민․형사상 이의를 제기하지 않을 것을 서약합니다.
2024. . .
서약자 : (인)
2025 구미아시아육상경기선수권대회 조직위원장 귀하
【 서식 9 】
가 격 제 안 서
입
찰
내
용
공고번호
제 호
입찰일자
2024. . .
사 업 명
2025 구미아시아육상경기선수권대회 종합정보시스템 및 홈페이지 등 구축 용역
제안금액
금 원(금 원)
※ 부가가치세 포함한 금액으로 기재.
제
안
자
상 호
또는 법인명칭
전화번호
사업자 번호
제 출 처
2025 구미아시아육상경기
선수권대회 조직위원회
증명서 용도
2025 구미아시아육상경기선수권대회 종합정보시스템 및
홈페이지 등 구축 용역 제출
용역 범위
(금액포함)
용역이행
실적내용
용역명
구
분
용역
개요
계약일자
계약기간
계약금액
이행실적
비고
비율(%)
실적(금액)
증명서
발급기관
위 사실을 증명함
년 월 일
기관명: (인) (전화번호: )
주 소: (팩스번호: )
발급부서:
담당자:
※ 민간거래실적은 세금계산서, 계약서 등 증빙자료를 첨부하여야 함
※ 용역이행실적은 입찰공고시 제시한 용역범위에 부합되는 조건에 한하며, 공동계약으로 이행하였을 경우 비율과 이행실적을 기재하여야 함
【 서식 10-4 】
재정상태 건실도
- 기업신용평가등급 확인서 첨부
【 서식 10-5 】
제안업체 신인도
구 분
처분사유
(관련법령)
처분일시
처분기간
비 고
입찰참가제한
2024. . .
대표자 : (인)
※ 해당사항이 없으면 “해당없음” 표기
※ 최근 3년간 회사(수급자)가 관계법령에 의해 입찰참가 제한 처분된 사실기재
※ 대표자 직인 반드시 날인하여야 함.
※ 사실과 다르게 기재한 경우로 인하여 발생하는 모든 문제는 업체가 모든 책임을 짐.
1 】
◦◦◦◦◦◦◦◦◦◦◦◦◦◦◦◦◦◦◦◦◦(가로 상철)◦◦◦◦◦◦◦◦◦◦◦◦◦◦◦◦◦◦◦◦◦
평가번호
(심사기호)
2025 구미아시아육상경기선수권대회 종합정보시스템 및 홈페이지 등 구축 용역
정성적 평가 제안서
(발 표 용 PPT 출 력 물)
업 체 명
주) 1. 표지 : A4, 표지의 지질은 백색아트지(무코팅), 내지(백상지) 사용


---

### retriever B:
종합정보시스템 및 홈페이지 등 구축 용역 제안요청서
1
대회개요
아시아의 꿈, 구미에서 세계로
 The dream of ASIA, From GUMI to the World 
비전
ㅇ (대 회 명) 2025 구미아시아육상경기선수권대회
ㅇ (기 간) 2025. 5. 27.(화) ~ 5. 31.(토) 5일간
ㅇ (장 소) 구미시민운동장 일원
ㅇ (규 모) 아시아 45개국 1,200여명 / 선수단 및 임원
ㅇ (주 최) 아시아육상연맹(Asian Athletics, AA)
ㅇ (주 관) 2025 구미아시아육상경기선수권대회 조직위원회
ㅇ (경기종목) 45개 종목
구 분
종 목 명
남자
(22)
100m, 200m, 400m, 800m, 1,500m, 5,000m,
10,000m, 110mH, 400mH, 3,000m장애물달리기,
20km경보, 4×100mR, 4×400mR, 높이뛰기,
장대높이뛰기, 멀리뛰기, 세단뛰기, 포환던지기,
원반던지기, 해머던지기, 창던지기, 10종 경기
여자
(22)
100m, 200m, 400m, 800m, 1,500m, 5,000m,
10,000m, 100mH, 400mH, 3,000m장애물달리기,
20km경보, 4×100mR, 4×400mR, 높이뛰기,
장대높이뛰기, 멀리뛰기, 세단뛰기, 포환던지기,
원반던지기, 해머던지기, 창던지기, 7종 경기
남·여
(1)
4×400m 믹스릴레이
요
2
과업개요
ㅇ (사 업 명) 2025 구미아시아육상경기선수권대회 종합정보시스템 및 홈페이지 등 구축 용역
ㅇ (사업기간) 계약일 ~ 2025. 6. 6.(금)
ㅇ (사업위치) 시민운동장 일원
1. 기본정보
사업명
2025 구미아시아육상경기선수권대회 종합정보시스템 및 홈페이지 등 구축 용역
영향평가단계
□ 예산편성
□ 그 외 필요시
■ 사업발주
□ 재평가
주요 내용
대회 기본 자료 관리 및 대회 계측기 연동 등
사업기간
(또는 개발기간)
계약일 ~ 2025년 6월
구분
붙임 : 산출내역서 각 1부
2024. . .
제안업체 인(사용인감)
2025 구미아시아육상경기선수권대회 조직위원장 귀하
※ 제안금액은 반드시 한글로 띄어쓰기 없이 붙여 쓰고 아라비아 숫자는 괄호 안에 기재할 것.
【 서식 9-1 】
산 출 내 역 서
구 분
항 목
금 액
구 성 비
비 고
순
용
역
원
가
소 계
소 계
소 계
소 계
일반관리비 ( )%
이 윤 ( )%
총 원 가
부 가 가 치 세
총 용 역 비
※ 작성내용에 따라 줄 수 증감가능
【 서식 10 】
접수번호
2025 구미아시아육상경기선수권대회 종합정보시스템 및 홈페이지 등 구축 용역
정량적 평가 제안서
2024년 월 일
업 체 명
※ 주의사항
3. 제안서 평가를 위해 구성된 평가위원회의 심사방법 및 심사 기준, 평가결과에 어떠한 이의제기도 하지 않겠습니다.
2024년 월 일
신청자
업 체 명
:
대표자
:
(인)
2025 구미아시아육상경기선수권대회 조직위원장 귀하
【 서식 6 】
보 안 서 약 서
본인은 2025 구미아시아육상경기선수권대회 종합정보시스템 및 홈페이지 등 구축 용역을 수행함에 있어 다음 사항을 준수할 것을 엄숙히 서약합니다.
2024. . .
대표자 : (인)
대리인 : (인)
2025 구미아시아육상경기선수권대회 조직위원장 귀하
※ 첨부서류 : 재직증명서, 신분증
※ 위임장에 사용하는 도장은 대표자 인감과 같아야 함.
※ 대리 접수로 인하여 발생하는 모든 문제는 대표자 본인이 책임짐
【 서식 3 】
재 직 증 명 서
성 명
생 년 월 일
부서/직위
소 속
주 소
재직기간
위와 같이 당사에 재직 중임을 증명합니다.
2024. . .
ྠĀ ྠĀ ྠĀ ྠĀ ྠĀ ྠĀ 업 체 명 :
ྠĀ ྠĀ ྠĀ ྠĀ ྠĀ ྠĀ 주 소 :
ྠĀ ྠĀ ྠĀ ྠĀ ྠĀ ྠĀ 대 표 자 : (인)
2025 구미아시아육상경기선수권대회 조직위원장 귀하
【 서식 4 】
사 용 인 감 계
사용인감
는(은) 2025 구미아시아육상경기선수권대회 조직위원회와 거래함에 있어 입찰, 계약 및 이에 수반되는 모든 행위에 대하여 위의 인감을 사용하고자 하며, 동 인감을 사용함으로써 민․형사상의 문제가 발생할 경우 이(가) 책임을 질 것을 확인합니다.
붙임 : 인감(법인)증명서 1부. 끝.
2024년 월 일
ྠĀ ྠĀ ྠĀ ྠĀ ྠĀ ྠĀ 주 소 :
ྠĀ ྠĀ ྠĀ ྠĀ ྠĀ ྠĀ 상 호 :
ྠĀ ྠĀ ྠĀ ྠĀ ྠĀ ྠĀ 대표자 : (인)
2025 구미아시아육상경기선수권대회 조직위원장 귀하
【 서식 5 】
서 약 서
당사는 2025 구미아시아육상경기선수권대회 종합정보시스템 및 홈페이지 등 구축 용역 참가신청과 관련하여 모든 서류와 증빙자료를 신의와 성실의 원칙에 입각하여 작성 제출하며, 아래 내용을 준수할 것을 확약하고 이에 서약서를 제출합니다.
2. 운영계획
운영기관
□ 단일 기관 : 2025 구미아시아육상경기선수권대회 조직위
□ 다수 기관 (예상 : 개 기관)
사용자
(복수선택 가능)
구분
예상 사용자수
□ 내부 직원
10명
□ 타 기관 직원
명
□ 일반 국민 또는 기업
명
 운영 매뉴얼 또는 지침서 제공 및 시스템 운영 관계자 교육
ㅇ (현장IT팀 상주 및 시스템 운영)
 사전 등록, 경기 진행 및 결과 출력 시스템 등 운영
 대회시작 4일전~대회 종료일까지 경기장 상주
 경기장/비경기장별 운영 인원 배치
ㅇ (홈페이지 구축)
 메뉴 구성도(안)
아시아육상경기선수권대회 메인
대회소개
소식지
경기일정/결과
메달순위
육상경기
커뮤니티
환영인사
공지사항
경기일정
국가별순위
경기종목
안내사항
대회개요
보도자료
행사일정
선수별순위
참가선수
아카이브
대회상징물
사진자료
시설안내
유실물센터
조직위안내
자주묻는질문
홍보영상
자원봉사자
찾아오시는 길
구미시 안내
메뉴 상세설명(안)
메뉴명
부메뉴
주요 내용
메인 화면
◦2025 구미 아시아육상경기선수권대회를 대표할 수 있는 디자인 (공식 상징물 활용 등)
◦활용빈도가 높은 메뉴 배치 및 팝업배너
대회 소개
환영인사
◦대회 개최에 대한 인사 말씀
대회개요
◦대회명칭, 대회기간, 대회장소, 참가규모, 참가국, 참가인원, 종목, 주최, 주관 등
대회상징물
◦대회마크, 마스코트, 포스터, 슬로건
조직위안내
◦조직위원회·아시아육상추진단 조직도
홍보영상
◦홍보영상, 유튜브 쇼츠 영상, 축하 메시지 등
찾아오시는 길
◦경기장 위치, 교통수단별 길 안내
소식지
공지사항
◦대회 관련 공지사항 등
보도자료
◦대회 관련기사 스크랩 자료
사진자료
◦개·폐회식 사진, 각 종목별 경기 이미지 게시
경기일정
/
결과
경기일정
◦경기장 및 개최 일자, 시간, 라운드, 진행상태
◦각 종목별 경기결과, 종합순위
행사일정
◦개·폐회식 정보 및 식순 안내
4. 회사 임․직원이 관계 직원에게 금품, 향응 등을 제공하거나 담합 등 불공정 행위를 하지 않도록 하는 회사 윤리강령과 내부비리 제보자에 대해서도 일체의 불이익처분을 하지 않는 사규를 제정토록 노력하겠습니다.
위 청렴계약 서약은 상호 신뢰를 바탕으로 한 약속으로서 반드시 지킬 것이며, 낙찰자로 결정될시 본 서약 내용을 그대로 계약 특수조건으로 계약하여 이행하고, 입찰참가 자격제한, 계약해지 등 귀 조직위원회의 조치와 관련하여 당사자가 귀 조직위원회를 상대로 손해배상을 청구하거나 당사를 배제 하는 입찰에 관하여 민․형사상 이의를 제기하지 않을 것을 서약합니다.
2024. . .
서약자 : (인)
2025 구미아시아육상경기선수권대회 조직위원장 귀하
【 서식 9 】
가 격 제 안 서
입
찰
내
용
공고번호
제 호
입찰일자
2024. . .
사 업 명
2025 구미아시아육상경기선수권대회 종합정보시스템 및 홈페이지 등 구축 용역
제안금액
금 원(금 원)
※ 부가가치세 포함한 금액으로 기재.
제
안
자
상 호
또는 법인명칭
법인등록번호
주 소
전 화 번 호
대 표 자
생 년 월 일
본인은 지방자치단체를 당사자로 하는 계약에 관한 법률 시행령 제43조(협상에 의한 계약체결)에 의거 발주된 본 사업의 제안요청서 내용을 모두 숙지하고, 응찰에 임하며 귀 기관에서 정한 사업기간 내에 사업을 성실하게 수행할 것을 확약하며 본 가격제안입찰서를 제출합니다.
붙임 : 산출내역서 각 1부
2024. . .
제안업체 인(사용인감)
2025 구미아시아육상경기선수권대회 조직위원장 귀하
장대높이뛰기, 멀리뛰기, 세단뛰기, 포환던지기,
원반던지기, 해머던지기, 창던지기, 7종 경기
남·여
(1)
4×400m 믹스릴레이
요
2
과업개요
ㅇ (사 업 명) 2025 구미아시아육상경기선수권대회 종합정보시스템 및 홈페이지 등 구축 용역
ㅇ (사업기간) 계약일 ~ 2025. 6. 6.(금)
ㅇ (사업위치) 시민운동장 일원
ㅇ (사 업 비) 260,000,000원 ※ 부가가치세 포함
ㅇ (사업내용)
 아시아육상대회를 위한 종합정보시스템 구축 및 운영
 종합정보시스템 운영을 위한 대회 정보화 시설 구축
 경기 결과와 계측기, 전광판, 방송 장비, 인터넷 정보시스템의 연동
 종합정보시스템을 연계한 아시아육상대회 홈페이지 구축·관리
2. 추진배경 및 필요성
ㅇ 성공적인 대회 운영과 개최를 위하여 대회를 널리 알리고자 대회정보와 행사정보 등을 제공하는 등 대회 전반에 대해 홍보하고자 함
ㅇ 계측장비, 운동장 전광판(표출기기), 음향·방송기기 등과 연동하여 경기 일정, 결과, 기록, 인적자료(선수) 등의 각종 데이터를 신속하게 전달하고 데이터화 하기 위한 통합 시스템 필요
ㅇ IT 기술과 관련 소프트웨어 기술을 바탕으로 대회 종합정보시스템을 운영하여 대회 관계자와 선수, 관객에게 편의제공은 물론 대회 운영 및 관리에도 효율성을 높여 2025 구미아시아육상경기선수권대회의 성공개최를 도모하고자 함
입찰참가제한
2024. . .
대표자 : (인)
※ 해당사항이 없으면 “해당없음” 표기
※ 최근 3년간 회사(수급자)가 관계법령에 의해 입찰참가 제한 처분된 사실기재
※ 대표자 직인 반드시 날인하여야 함.
※ 사실과 다르게 기재한 경우로 인하여 발생하는 모든 문제는 업체가 모든 책임을 짐.
1 】
◦◦◦◦◦◦◦◦◦◦◦◦◦◦◦◦◦◦◦◦◦(가로 상철)◦◦◦◦◦◦◦◦◦◦◦◦◦◦◦◦◦◦◦◦◦
평가번호
(심사기호)
2025 구미아시아육상경기선수권대회 종합정보시스템 및 홈페이지 등 구축 용역
정성적 평가 제안서
(발 표 용 PPT 출 력 물)
업 체 명
주) 1. 표지 : A4, 표지의 지질은 백색아트지(무코팅), 내지(백상지) 사용


---

### openai answer
retriever A 점수: 3점
질문에 대한 정확한 개최 기간 “2025. 5. 27. ~ 5. 31. 5일간”이 명시되어 있어 답변에 필요한 정보가 모두 포함됨.

retriever B 점수: 3점
동일하게 “2025. 5. 27. ~ 5. 31. 5일간”이 명시되어 있어 정답 도출에 필요한 정보가 모두 포함됨. 

============================================================
### query:
구미 아시아육상경기선수권대회 홈페이지에서 제공되는 주요 콘텐츠 항목에는 무엇이 포함되는가?

### answer:
대회소개, 경기일정 및 결과, 메달순위, 참가선수 정보, 커뮤니티 등이 포함된다.

---

### retriever A:
-경기참가자(Start List), 경기결과(Official Result, Summary) PDF 파일 제공
<다음 정보를 홈페이지에 전송>
Level1
Level2
기능 설명
경기
일정
전체 경기 일정
대회기간 전체 일정 제공
일자별 경기 일정
일자별 세부 경기 일정 제공
세부 종목별 경기 일정
세부 종목별 경기 일정 제공
선수별 경기 일정
선수별 경기 일정 제공
메달
국가/선수별 메달 현황
국가/선수별 메달 현황 제공
종목별 메달 현황
종목별 메달현황 및 메달수상자 정보 제공
경기
정보
스타트 리스트
세부종목별 편성 조 및 출전선수 목록 제공
대진 정보
세부 종목별 대진정보 조회 및 출력
(경기진행 상태에 따른 결과 반영 포함)
경기 결과
각 조별, 종목별 경기 결과 조회 및 출력
선수
정보
국가별 선수 정보
국가별 참가선수 목록 및 정보 조회
세부 종목별 선수 정보
세부 종목별 참가선수 목록 및 정보 조회
순위 정보
세부종목별 참가선수 순위 정보 목록 조회
과업내용
명칭
대분류
대회 종합정보시스템 S/W 개발
소분류
보고서관리시스템 개발
과업내용
데이터 출력 기능
<PDF 및 Excel, 증서 페이지 등 출력>
-참가선수 신상정보
-국가별/세부 종목별 참가선수 리스트
-심판/임원 신상정보
-국가별/세부 종목별 심판/임원 리스트
-세부 종목별, 예결선 Start List 및 조 편성
-세부 종목별 경기 결과 (Live Result, Official Result, Summary)
-세부 종목별 메달 현황 및 신기록 현황
-참가증서
ː참가선수(종목별 1위~6위 증서 및 참가증서 필수)
 운영 매뉴얼 또는 지침서 제공 및 시스템 운영 관계자 교육
ㅇ (현장IT팀 상주 및 시스템 운영)
 사전 등록, 경기 진행 및 결과 출력 시스템 등 운영
 대회시작 4일전~대회 종료일까지 경기장 상주
 경기장/비경기장별 운영 인원 배치
ㅇ (홈페이지 구축)
 메뉴 구성도(안)
아시아육상경기선수권대회 메인
대회소개
소식지
경기일정/결과
메달순위
육상경기
커뮤니티
환영인사
공지사항
경기일정
국가별순위
경기종목
안내사항
대회개요
보도자료
행사일정
선수별순위
참가선수
아카이브
대회상징물
사진자료
시설안내
유실물센터
조직위안내
자주묻는질문
홍보영상
자원봉사자
찾아오시는 길
구미시 안내
메뉴 상세설명(안)
메뉴명
부메뉴
주요 내용
메인 화면
◦2025 구미 아시아육상경기선수권대회를 대표할 수 있는 디자인 (공식 상징물 활용 등)
◦활용빈도가 높은 메뉴 배치 및 팝업배너
대회 소개
환영인사
◦대회 개최에 대한 인사 말씀
대회개요
◦대회명칭, 대회기간, 대회장소, 참가규모, 참가국, 참가인원, 종목, 주최, 주관 등
대회상징물
◦대회마크, 마스코트, 포스터, 슬로건
조직위안내
◦조직위원회·아시아육상추진단 조직도
홍보영상
◦홍보영상, 유튜브 쇼츠 영상, 축하 메시지 등
찾아오시는 길
◦경기장 위치, 교통수단별 길 안내
소식지
공지사항
◦대회 관련 공지사항 등
보도자료
◦대회 관련기사 스크랩 자료
사진자료
◦개·폐회식 사진, 각 종목별 경기 이미지 게시
경기일정
/
결과
경기일정
◦경기장 및 개최 일자, 시간, 라운드, 진행상태
◦각 종목별 경기결과, 종합순위
행사일정
◦개·폐회식 정보 및 식순 안내
찾아오시는 길
◦경기장 위치, 교통수단별 길 안내
소식지
공지사항
◦대회 관련 공지사항 등
보도자료
◦대회 관련기사 스크랩 자료
사진자료
◦개·폐회식 사진, 각 종목별 경기 이미지 게시
경기일정
/
결과
경기일정
◦경기장 및 개최 일자, 시간, 라운드, 진행상태
◦각 종목별 경기결과, 종합순위
행사일정
◦개·폐회식 정보 및 식순 안내
◦행사장, 공연, 전시 등 안내
◦메달 수여식 안내
메달 순위
국가별 순위
◦국가별 메달 집계 및 랭킹 표시
선수별 순위
◦선수별 메달 집계 및 랭킹 표시
육상경기
경기종목
◦마스코트 이미지 또는 픽토그램으로 제시
◦경기 규칙, 유래 등 간단한 설명 자료 제공
◦종목별 주의사항, 해당 경기일정표 안내
참가선수
◦대회 참가 선수의 국가, 선수이름 안내
시설안내
◦종목별 경기장소, 선수촌, 훈련장 안내
커뮤니티
안내사항
◦대회 관련 안내사항 등
아카이브
◦대회 관련 자료 선별 보관
유실물 센터
◦대회 기간 중 유실물 현황
자주묻는질문
◦대회 관련 자주묻는질문 리스트
자원봉사자
◦자원봉사 소개, 신청안내
구미시 안내
◦구미시 문화관광 홈페이지 연동하여 관광 안내
◦지도 및 주요 고속도로, 국도 및 대중교통 안내
◦구미시 숙박시설 안내
◦구미시 모범음식점 등 안내
일정)
구분
작업일정
M
M+1
M+3
M+5
M+6
M+7
프로젝트 착수 및 보고
홈페이지
제작
업무 분석 및 설계
개발 및 디자인 작업
테스트 및 시범 운영
홈페이지 서비스
대회종합정보시스템
요구 분석
상세 설계
구축개발, 단위 테스트
초기데이터 구축 , 검증
구축완료, 통합테스트
유지보수
대회 종료시까지
종합정보시스템 및 홈페이지 등 구축 용역 제안요청서
1
대회개요
아시아의 꿈, 구미에서 세계로
 The dream of ASIA, From GUMI to the World 
비전
ㅇ (대 회 명) 2025 구미아시아육상경기선수권대회
ㅇ (기 간) 2025. 5. 27.(화) ~ 5. 31.(토) 5일간
ㅇ (장 소) 구미시민운동장 일원
ㅇ (규 모) 아시아 45개국 1,200여명 / 선수단 및 임원
ㅇ (주 최) 아시아육상연맹(Asian Athletics, AA)
ㅇ (주 관) 2025 구미아시아육상경기선수권대회 조직위원회
ㅇ (경기종목) 45개 종목
구 분
종 목 명
남자
(22)
100m, 200m, 400m, 800m, 1,500m, 5,000m,
10,000m, 110mH, 400mH, 3,000m장애물달리기,
20km경보, 4×100mR, 4×400mR, 높이뛰기,
장대높이뛰기, 멀리뛰기, 세단뛰기, 포환던지기,
원반던지기, 해머던지기, 창던지기, 10종 경기
여자
(22)
100m, 200m, 400m, 800m, 1,500m, 5,000m,
10,000m, 100mH, 400mH, 3,000m장애물달리기,
20km경보, 4×100mR, 4×400mR, 높이뛰기,
장대높이뛰기, 멀리뛰기, 세단뛰기, 포환던지기,
원반던지기, 해머던지기, 창던지기, 7종 경기
남·여
(1)
4×400m 믹스릴레이
요
2
과업개요
ㅇ (사 업 명) 2025 구미아시아육상경기선수권대회 종합정보시스템 및 홈페이지 등 구축 용역
ㅇ (사업기간) 계약일 ~ 2025. 6. 6.(금)
ㅇ (사업위치) 시민운동장 일원
장대높이뛰기, 멀리뛰기, 세단뛰기, 포환던지기,
원반던지기, 해머던지기, 창던지기, 7종 경기
남·여
(1)
4×400m 믹스릴레이
요
2
과업개요
ㅇ (사 업 명) 2025 구미아시아육상경기선수권대회 종합정보시스템 및 홈페이지 등 구축 용역
ㅇ (사업기간) 계약일 ~ 2025. 6. 6.(금)
ㅇ (사업위치) 시민운동장 일원
ㅇ (사 업 비) 260,000,000원 ※ 부가가치세 포함
ㅇ (사업내용)
 아시아육상대회를 위한 종합정보시스템 구축 및 운영
 종합정보시스템 운영을 위한 대회 정보화 시설 구축
 경기 결과와 계측기, 전광판, 방송 장비, 인터넷 정보시스템의 연동
 종합정보시스템을 연계한 아시아육상대회 홈페이지 구축·관리
2. 추진배경 및 필요성
ㅇ 성공적인 대회 운영과 개최를 위하여 대회를 널리 알리고자 대회정보와 행사정보 등을 제공하는 등 대회 전반에 대해 홍보하고자 함
ㅇ 계측장비, 운동장 전광판(표출기기), 음향·방송기기 등과 연동하여 경기 일정, 결과, 기록, 인적자료(선수) 등의 각종 데이터를 신속하게 전달하고 데이터화 하기 위한 통합 시스템 필요
ㅇ IT 기술과 관련 소프트웨어 기술을 바탕으로 대회 종합정보시스템을 운영하여 대회 관계자와 선수, 관객에게 편의제공은 물론 대회 운영 및 관리에도 효율성을 높여 2025 구미아시아육상경기선수권대회의 성공개최를 도모하고자 함
3. 사업내용
ㅇ (종합정보시스템 구축을 위한 요구분석 및 상세설계)
 종합정보시스템 업무에 대한 요구사항 분석
 대회 기본자료, 경기운영, 인터페이스, 보고서 관리 등 시스템별 상세설계
-공지 및 게시 사항 등록 및 조회
-시스템에서 사용하는 코드 관리
과업내용
명칭
대분류
대회 종합정보시스템 S/W 개발
소분류
경기운영시스템 개발
과업내용
종목별
조 편성
-관련 육상대회/경기 규칙에 따라 시스템 개발
-Start List 및 조편성 담당자 배치
-매일 경기 1~2일 전 송부된 참가신청서 (영문) 등록 및 Start List 생성
-Start List 기반하여 경기 조 자동 편성
-참가신청완료 선수 및 팀 상세정보 관리 및 제공
-종목별/조별 참가자 명단 생성 및 전송
-개발 시 가상 Start List 생성 시험을 통해 오류·오차 수정
-특수한 상황을 대비해 수동 조 편성 및 이동 편성 지원
경기 진행
-대회기간 전체의 종목별, 일자별, 경기장별 경기 일정 제공
-트랙경기 : 계측기 통한 종목별경기결과 자동 입력(실시간)
-필드경기: 기록지/기타 입력장치를 통한 종목별경기결과 수동 입력(실시간)
-종목별 별도의 입력 화면이 필요하고, 온라인으로 경기 결과 입력 기능
경기결과 처리 및 배포
-세부 종목별 기록 검증 및 순위 처리
-예선 종목은 다음 라운드 진출 인원에 의한 결선 진출 처리, 상위 대진 처리 (자동/수동)
-결선 출발 순서 배정 (자동/수동)
-세부 종목별 메달 확인 처리 및 집계 관리 후 인터페이스 관리시스템에 전송
-신기록 정보 관리
-처리한 경기 정보를 보고서 관리 시스템 및 내/외 배포
-트랙 경기의 경우 경기 결과 실시간으로 입력 후 전광판 및 방송 송출 화면 출력
과업내용
명칭
대분류
대회 종합정보시스템 S/W 개발
소분류
인터페이스관리시스템 개발
과업내용
1 】
◦◦◦◦◦◦◦◦◦◦◦◦◦◦◦◦◦◦◦◦◦(가로 상철)◦◦◦◦◦◦◦◦◦◦◦◦◦◦◦◦◦◦◦◦◦
평가번호
(심사기호)
2025 구미아시아육상경기선수권대회 종합정보시스템 및 홈페이지 등 구축 용역
정성적 평가 제안서
(발 표 용 PPT 출 력 물)
업 체 명
주) 1. 표지 : A4, 표지의 지질은 백색아트지(무코팅), 내지(백상지) 사용
2. 형식 : 단면인쇄, 가로 상철 스프링 제본
3. 제안회사명은 반드시 제안서 제출부수 10부중 1부만 기재하여 제출하여야 한다.
※ 1부를 제외한 제안서에는 업체를 인식할 수 있는 행사 실적 및 일체 표기(회사명, 로고, 마크, 대표자명 등)를 사용해서는 안된다.


---

### retriever B:
 운영 매뉴얼 또는 지침서 제공 및 시스템 운영 관계자 교육
ㅇ (현장IT팀 상주 및 시스템 운영)
 사전 등록, 경기 진행 및 결과 출력 시스템 등 운영
 대회시작 4일전~대회 종료일까지 경기장 상주
 경기장/비경기장별 운영 인원 배치
ㅇ (홈페이지 구축)
 메뉴 구성도(안)
아시아육상경기선수권대회 메인
대회소개
소식지
경기일정/결과
메달순위
육상경기
커뮤니티
환영인사
공지사항
경기일정
국가별순위
경기종목
안내사항
대회개요
보도자료
행사일정
선수별순위
참가선수
아카이브
대회상징물
사진자료
시설안내
유실물센터
조직위안내
자주묻는질문
홍보영상
자원봉사자
찾아오시는 길
구미시 안내
메뉴 상세설명(안)
메뉴명
부메뉴
주요 내용
메인 화면
◦2025 구미 아시아육상경기선수권대회를 대표할 수 있는 디자인 (공식 상징물 활용 등)
◦활용빈도가 높은 메뉴 배치 및 팝업배너
대회 소개
환영인사
◦대회 개최에 대한 인사 말씀
대회개요
◦대회명칭, 대회기간, 대회장소, 참가규모, 참가국, 참가인원, 종목, 주최, 주관 등
대회상징물
◦대회마크, 마스코트, 포스터, 슬로건
조직위안내
◦조직위원회·아시아육상추진단 조직도
홍보영상
◦홍보영상, 유튜브 쇼츠 영상, 축하 메시지 등
찾아오시는 길
◦경기장 위치, 교통수단별 길 안내
소식지
공지사항
◦대회 관련 공지사항 등
보도자료
◦대회 관련기사 스크랩 자료
사진자료
◦개·폐회식 사진, 각 종목별 경기 이미지 게시
경기일정
/
결과
경기일정
◦경기장 및 개최 일자, 시간, 라운드, 진행상태
◦각 종목별 경기결과, 종합순위
행사일정
◦개·폐회식 정보 및 식순 안내
종합정보시스템 및 홈페이지 등 구축 용역 제안요청서
1
대회개요
아시아의 꿈, 구미에서 세계로
 The dream of ASIA, From GUMI to the World 
비전
ㅇ (대 회 명) 2025 구미아시아육상경기선수권대회
ㅇ (기 간) 2025. 5. 27.(화) ~ 5. 31.(토) 5일간
ㅇ (장 소) 구미시민운동장 일원
ㅇ (규 모) 아시아 45개국 1,200여명 / 선수단 및 임원
ㅇ (주 최) 아시아육상연맹(Asian Athletics, AA)
ㅇ (주 관) 2025 구미아시아육상경기선수권대회 조직위원회
ㅇ (경기종목) 45개 종목
구 분
종 목 명
남자
(22)
100m, 200m, 400m, 800m, 1,500m, 5,000m,
10,000m, 110mH, 400mH, 3,000m장애물달리기,
20km경보, 4×100mR, 4×400mR, 높이뛰기,
장대높이뛰기, 멀리뛰기, 세단뛰기, 포환던지기,
원반던지기, 해머던지기, 창던지기, 10종 경기
여자
(22)
100m, 200m, 400m, 800m, 1,500m, 5,000m,
10,000m, 100mH, 400mH, 3,000m장애물달리기,
20km경보, 4×100mR, 4×400mR, 높이뛰기,
장대높이뛰기, 멀리뛰기, 세단뛰기, 포환던지기,
원반던지기, 해머던지기, 창던지기, 7종 경기
남·여
(1)
4×400m 믹스릴레이
요
2
과업개요
ㅇ (사 업 명) 2025 구미아시아육상경기선수권대회 종합정보시스템 및 홈페이지 등 구축 용역
ㅇ (사업기간) 계약일 ~ 2025. 6. 6.(금)
ㅇ (사업위치) 시민운동장 일원
찾아오시는 길
◦경기장 위치, 교통수단별 길 안내
소식지
공지사항
◦대회 관련 공지사항 등
보도자료
◦대회 관련기사 스크랩 자료
사진자료
◦개·폐회식 사진, 각 종목별 경기 이미지 게시
경기일정
/
결과
경기일정
◦경기장 및 개최 일자, 시간, 라운드, 진행상태
◦각 종목별 경기결과, 종합순위
행사일정
◦개·폐회식 정보 및 식순 안내
◦행사장, 공연, 전시 등 안내
◦메달 수여식 안내
메달 순위
국가별 순위
◦국가별 메달 집계 및 랭킹 표시
선수별 순위
◦선수별 메달 집계 및 랭킹 표시
육상경기
경기종목
◦마스코트 이미지 또는 픽토그램으로 제시
◦경기 규칙, 유래 등 간단한 설명 자료 제공
◦종목별 주의사항, 해당 경기일정표 안내
참가선수
◦대회 참가 선수의 국가, 선수이름 안내
시설안내
◦종목별 경기장소, 선수촌, 훈련장 안내
커뮤니티
안내사항
◦대회 관련 안내사항 등
아카이브
◦대회 관련 자료 선별 보관
유실물 센터
◦대회 기간 중 유실물 현황
자주묻는질문
◦대회 관련 자주묻는질문 리스트
자원봉사자
◦자원봉사 소개, 신청안내
구미시 안내
◦구미시 문화관광 홈페이지 연동하여 관광 안내
◦지도 및 주요 고속도로, 국도 및 대중교통 안내
◦구미시 숙박시설 안내
◦구미시 모범음식점 등 안내
일정
전체 경기 일정
대회기간 전체 일정 제공
일자별 경기 일정
일자별 세부 경기 일정 제공
세부 종목별 경기 일정
세부 종목별 경기 일정 제공
선수별 경기 일정
선수별 경기 일정 제공
메달
국가/선수별 메달 현황
국가/선수별 메달 현황 제공
종목별 메달 현황
종목별 메달현황 및 메달수상자 정보 제공
경기
정보
스타트 리스트
세부종목별 편성 조 및 출전선수 목록 제공
대진 정보
세부 종목별 대진정보 조회 및 출력
(경기진행 상태에 따른 결과 반영 포함)
경기 결과
각 조별, 종목별 경기 결과 조회 및 출력
선수
정보
국가별 선수 정보
국가별 참가선수 목록 및 정보 조회
세부 종목별 선수 정보
세부 종목별 참가선수 목록 및 정보 조회
순위 정보
세부종목별 참가선수 순위 정보 목록 조회
1. 기본정보
사업명
2025 구미아시아육상경기선수권대회 종합정보시스템 및 홈페이지 등 구축 용역
영향평가단계
□ 예산편성
□ 그 외 필요시
■ 사업발주
□ 재평가
주요 내용
대회 기본 자료 관리 및 대회 계측기 연동 등
사업기간
(또는 개발기간)
계약일 ~ 2025년 6월
구분
과업내용
데이터 출력 기능
<PDF 및 Excel, 증서 페이지 등 출력>
-참가선수 신상정보
-국가별/세부 종목별 참가선수 리스트
-심판/임원 신상정보
-국가별/세부 종목별 심판/임원 리스트
-세부 종목별, 예결선 Start List 및 조 편성
-세부 종목별 경기 결과 (Live Result, Official Result, Summary)
-세부 종목별 메달 현황 및 신기록 현황
-참가증서
ː참가선수(종목별 1위~6위 증서 및 참가증서 필수)
ː자원봉사자, 운영요원 등
-기타 대회 전반에 대한 정보 및 현황
요구사항
상세설명
정의
2025년 동계, 하계, 학생 전국장애인체육대회 정보제공 홈페이지
세부
내용
1) 요구사항 기능
- 메인화면
· 공지사항, 응원메시지, 오늘의 경기, 종합순위, 메달집계, 메달수상자,
다관왕, 포토갤러리 등 정보 표출
- 대회개요
· 대회 개요 및 일정 : 대회개요, 인사말, 개/폐회식 행사 안내
· 참가요강 : 일반참가요강, 장애별 참가가능 종목, 종목별 참가요강
· 참가현황 : 등록구분별, 종목/시도별, 장애유형별
· 선수/임원 검색 : 시도별, 종목별 참가선수 및 임원 검색
· 경기장 안내
- 경기일정 및 결과
· 종목별 경기일정 및 결과
· 전체 경기일정
- 득점 및 메달정보
· 인터넷을 통한 체육인을 등록 하며, 처리 결과에 대한 조회관리 기능
- 2025년 개최되는 동계, 하계, 학생 전국장애인체육대회 총 3개 대회 지원
2) 시스템간의 연계
- 체전운영시스템, 대진추첨시스템과 정보연계
(대진정보, 경기결과정보, 메달 및 점수 정보)
과업내용
종목별
조 편성
-관련 육상대회/경기 규칙에 따라 시스템 개발
-Start List 및 조편성 담당자 배치
-매일 경기 1~2일 전 송부된 참가신청서 (영문) 등록 및 Start List 생성
-Start List 기반하여 경기 조 자동 편성
-참가신청완료 선수 및 팀 상세정보 관리 및 제공
-종목별/조별 참가자 명단 생성 및 전송
-개발 시 가상 Start List 생성 시험을 통해 오류·오차 수정
-특수한 상황을 대비해 수동 조 편성 및 이동 편성 지원
경기 진행
-대회기간 전체의 종목별, 일자별, 경기장별 경기 일정 제공
-트랙경기 : 계측기 통한 종목별경기결과 자동 입력(실시간)
-필드경기: 기록지/기타 입력장치를 통한 종목별경기결과 수동 입력(실시간)
-종목별 별도의 입력 화면이 필요하고, 온라인으로 경기 결과 입력 기능
경기결과 처리 및 배포
-세부 종목별 기록 검증 및 순위 처리
-예선 종목은 다음 라운드 진출 인원에 의한 결선 진출 처리, 상위 대진 처리 (자동/수동)
-결선 출발 순서 배정 (자동/수동)
-세부 종목별 메달 확인 처리 및 집계 관리 후 인터페이스 관리시스템에 전송
-신기록 정보 관리
-처리한 경기 정보를 보고서 관리 시스템 및 내/외 배포
-트랙 경기의 경우 경기 결과 실시간으로 입력 후 전광판 및 방송 송출 화면 출력
3. 사업내용
ㅇ (종합정보시스템 구축을 위한 요구분석 및 상세설계)
 종합정보시스템 업무에 대한 요구사항 분석
 대회 기본자료, 경기운영, 인터페이스, 보고서 관리 등 시스템별 상세설계
 추진 일정에 부합하는 상세 추진 일정 작성 및 제출
ㅇ (종합정보시스템 S/W 개발)
 대회 기본자료 관리 시스템
- 선수 등록, 경기일정 등록, 심판/임원 등록 관리, 신기록 및 종목 관리, 시스템 관리 등
 경기 운영 시스템
- 조 편성 및 Start List 작성, 경기 결과 입력 및 처리, 메달 집계 처리
 인터페이스 관리 시스템
- 계측장비, 대회홈페이지, 전광판, 방송장비 등과 연동·정보 송수신
 보고서 관리 시스템
- 참가자, 심판, 임원 정보, Start List, 경기 결과, 메달 현황, 신기록 현황 등 보고서 출력
ㅇ (대회 정보화시설 구축)
 원활한 대회 운영에 필요한 정보화시설 구축
 경기장 등 시설 내 종합정보시스템에 필요한 시설 구축
ㅇ (종합정보시스템 시험 운영 및 관계자 교육)
 2025년 본 대회 전 리허설 육상대회 시험 운영
 내부/외부시스템(계측기, 전광판, 홈페이지 등)과 연동 시험
 돌발 상황(정보통신망 장애, 정전 등) 발생 시 대응 능력 시험
 운영 매뉴얼 또는 지침서 제공 및 시스템 운영 관계자 교육
ㅇ (현장IT팀 상주 및 시스템 운영)
 사전 등록, 경기 진행 및 결과 출력 시스템 등 운영
 대회시작 4일전~대회 종료일까지 경기장 상주
 경기장/비경기장별 운영 인원 배치
ㅇ (홈페이지 구축)
 메뉴 구성도(안)
장대높이뛰기, 멀리뛰기, 세단뛰기, 포환던지기,
원반던지기, 해머던지기, 창던지기, 7종 경기
남·여
(1)
4×400m 믹스릴레이
요
2
과업개요
ㅇ (사 업 명) 2025 구미아시아육상경기선수권대회 종합정보시스템 및 홈페이지 등 구축 용역
ㅇ (사업기간) 계약일 ~ 2025. 6. 6.(금)
ㅇ (사업위치) 시민운동장 일원
ㅇ (사 업 비) 260,000,000원 ※ 부가가치세 포함
ㅇ (사업내용)
 아시아육상대회를 위한 종합정보시스템 구축 및 운영
 종합정보시스템 운영을 위한 대회 정보화 시설 구축
 경기 결과와 계측기, 전광판, 방송 장비, 인터넷 정보시스템의 연동
 종합정보시스템을 연계한 아시아육상대회 홈페이지 구축·관리


---

### openai answer
retriever A 점수: 3점
문서에 홈페이지 메뉴로 대회소개, 경기일정/결과, 메달순위, 참가선수, 커뮤니티가 모두 명시되어 있어 answer의 항목이 전부 포함됨.

retriever B 점수: 3점
메뉴 구성도에 대회소개, 경기일정/결과, 메달순위, 참가선수, 커뮤니티가 모두 제시되어 있어 answer의 모든 항목을 충족함. 

============================================================
### query:
원자력 선량평가시스템의 성능 요구사항 중 질의응답 시간 기준은 어떻게 정의되어 있는가?

### answer:
일반 조회 요청은 3초 이내 처리되어야 하며, 10초 이상 소요되는 작업은 사용자에게 팝업 메시지를 제공해야 한다.

---

### retriever A:
질의·응답시간
요구사항
상세설명
정의
질의·응답시간
세부
내용
의는 사용자가 요청을 하는 시간으로부터 3000ms 내에 그 결과를 보여줘야 함
❍ 임의의 선택 기준이 허용되는 대량의 데이터에 대한 질의 및 고용량의 멀티미디어 파일을 가지고 있는 프로그램에서는 적절한 처리 상태를 표시하여야 함
산출정보
성능측정 결과 내역서
관련 요구사항
요구사항 고유번호
PER-002
요구사항 명
오류 응답시간
요구사항
상세설명
정의
오류 응답시간
세부
내용
시스템에 데이터 입력 등의 사용 중에 발생한 모든 오류는 원인이 발생한지 3000ms 이하로 사용자에게 오류 메시지를 보여줘야 함
산출정보
성능측정 결과 내역서
관련 요구사항
요구사항 고유번호
PER-003
요구사항 명
통계 처리 사전 경고
요구사항
상세설명
정의
통계 처리 사전 경고
세부
내용
리 시 5,000ms 이상 지연되는 경우 연산 작업을 시작하는 시점에 예상되는 결과에 대해 사용자에게 메시지로 알려야 함
산출정보
성능측정 결과 내역서
관련 요구사항
요구사항 고유번호
PER-004
요구사항 명
부하증가 대비 성능유지
요구사항
상세설명
정의
부하증가 대비 성능유지
세부
내용
시스템의 경우 동시 접속자(사용자)는 500명 이상 접속가능 하여야 하며, 이에 따른 처리속도 등 성능이 저하되지 않아야 함
❍ 포털 시스템의 경우 동시 접속자(사용자)는 3000명 이상 접속가능 하여야 하며, 이에 따른 처리속도 등 성능이 저하되지 않아야 함
산출정보
성능측정 결과 내역서
관련 요구사항
요구사항 고유번호
PER-005
요구사항 명
오프라인 배치성 업무 응답시간
요구사항
나. 성능 요구사항 (Performance Requirement)
요구사항분류
PER-001
요구사항명
응답시간
요구사항 내용
정의
개선 대상 웹사이트 및 콘텐츠 관리 시스템(CMS) 정보 조회에 대한 응답시간 성능 보장
세부 내용
○ 정보 처리량 및 확장성 등을 고려하여 웹페이지 디스플레이 시간 기능요청에 대한 응답시간이 최소화 되도록 함
- 단순 웹페이지의 경우, 사용자가 요청을 하는 시간으로부터 3초 내에 그 결과를 보여줘야 함
- 시스템이 만드는 각 페이지의 경우, 요청한 시간으로부터 5초 내에 디스플레이 되어야 함
- 이 요구사항은 시스템을 사용하는 동시 사용자가 최대 사용자 수의 90%를 초과하는 질의에는 적용되지 않음
○ 과도한 데이터베이스 접속을 자제하며 응답속도 개선(상영시간표 등)
- 이 요구사항은 임의의 선택 기준이 허용되는 대량의 데이터(데이터량 2GB 이상)에 대한 질의에는 적용되지 않음
- 이 요구사항은 임의의 선택 기준이 허용되는 대량의 데이터에 대한 요청 및 플래시가 있을 경우에는 적용되지 않으며, 한 개 이상의 큰 이미지(이미지 500KB 이상) 및 동영상을 가지고 있는 페이지에는 적용되지 않음
- 동영상의 경우 트래픽 관리를 위해 외부 플랫폼(예: 유튜브, 기타 CDN 서비스)과 연계하여 기능 구현
○ 기능 구현에 있어 10초 이상 걸릴 수 있는 작업에 대해서는 시간지연에 대한 정보를 사용자에게 팝업 등의 방법으로 사용자에게 알려야 함
요구사항분류
PER-002
요구사항명
데이터 형식 오류 응답 시간
요구사항 내용
정의
사용자 입력 오류 응답 시간 제시
세부 내용
요구사항 명칭 성능요건
정의 성능요건 일반사항
❍ 성능요건 일반
- 기존 인프라 장비, 솔루션과의 호환성과 성능 보장
- 기 구축되어 서비스 중인 모든 응용 프로그램의 안정성과 성능을 보장해야
함
- 시스템에 대한 성능 분석을 실시하고 결과 및 개선방안을 제시하여야 함
- 시스템의 안정적 운영을 위하여 서비스 대고객 오픈 전/후와 안정화 기간에
튜닝 등 시스템 환경을 최적화하고 운영 어플리케이션 및 각종 솔루션의
요구 업그레이드 및 시스템 환경을 최적화하고 업그레이드 및 장애이력을
사항 관리하여야 함
상세 세부
설명 내용 ❍ 개발시스템의 성능 관리
- 사업대상 시스템의 성능을 고려한 개발 방안을 제시
- 대상시스템에 대하여 안정적 운영지원 및 사용자 지원 방안을 제시
- 시스템 개발 중 로그 또는 툴(도구)을 이용하여 시스템 성능상태를
모니터링이 가능하도록 환경을 구성하여, 성능상 문제를 미리 파악하여
조치한 후 시스템을 오픈
- WAS 및 DBMS의 부하 감소, 성능 개선, 장애 요인 감소, 시스템 안정화를
위한 튜닝 실시
- 수강신청, 성적조회 등 트래픽이 집중되는 시기의 안정적인 서비스 방안을
제시하여야 함
산출정보 • 성능시험결과서
-175-
요구사항 분류 성능 요구사항
요구사항 번호 PER-002
요구사항 명칭 온라인성 업무응답시간
정의 온라인성 업무응답시간 목표
❍ 업무응답시간 요구사항
- 정보 요청에 따라 결과가 조회되어 화면에 표시되기까지의 소요 시간을
의미
- 모든 질의는 사용자가 요청을 하는 시간으로부터 3초 내에 그 결과를 처리
요구사항 분류
성능 요구사항
요구사항 고유번호
PER-003
요구사항 명칭
성능 요구수준 및 웹 페이지 디스플레이 시간
요구사항
상세설명
정의
통합접수시스템의 성능 요구수준과 웹 페이지 디스플레이 목표시간 정의
세부내용
○ 통합접수시스템은 사용자가 요청한 시점으로부터 3초 내에 완전히 디스플레이 되어야 함
○ 웹 페이지 용량 감축 등 다양한 방법을 고려하여 시스템 최적화 조치
- 통합접수시스템 메인페이지 등 주요 페이지의 용량은 3MB 이내로 적용(단, 발주기관과 협의하여 조정 가능)
- Http 요청 횟수 등을 최소화하여 응답속도 향상 방안 마련
○ 성능 예외사항
- 임의의 선택 기준이 허용되는 대량의 데이터에 대한 쿼리는 적용되지 않으며, 1개 이상의 큰 이미지(500kb 이상)가 포함된 웹페이지에는 적용하지 않음
산출 정보
성능개선계획, 성능분석보고서, 성능테스트 결과보고서 등
요구사항 분류
성능 요구사항
요구사항 고유번호
PER-004
요구사항 명칭
오류 응답시간
요구사항
상세설명
정의
사용자 오류에 대한 응답시간 및 지연 경고
세부내용
○ 오류 응답시간
- 모든 오류에 대한 메시지를 정보 입력 후 3초 인내에 제시하여야 함
- 오류 메시지는 사용자가 인지하여 즉시 조치할 수 있도록 작성하여야 함
○ 사용자가 입력한 데이터 형식의 모든 오류는 사용자가 시스템에 정보를 입력한 시점으로부터 2초 이내에 오류 메시지를 사용자에게 출력하여야 함.
○ 10초 이상 소요되는 작업은 시작하는 시점에 팝업 화면 등을 출력하여 응답 지연이 될 수 있다는 메시지를 안내해야 함
o 재무제표, 주석 기준 정보에 대한 정의, 매핑 내역 등은 컨설팅 완료보고서에 포함함
산출정보
회계 컨설팅 완료 보고서
11) 성능 요구사항(Performance Requirement)
요구사항 분류
성능 요구사항
요구사항 고유번호
PER-001
요구사항 명칭
성능 일반 요구사항
요구사항
상세설명
정의
성능 일반 요구사항
세부내용
o 사업 대상 시스템의 성능을 고려한 개발 방안을 제시하여야 함
o 대상 시스템에 대하여 안정적 운영지원 및 사용자 지원 방안을 제시하여야 함
o 시스템 개발 중 성능 진단 도구를 이용하여 시스템 성능 상태에 대해 모니터링 해야 하며, 성능상 문제를 미리 파악하여 조치하여야 함
o 기존 인프라 및 SW와의 완벽한 호환 및 안정성을 보장하여야 함
o 시스템에 대한 성능 상태를 모니터링하고, 성능상 문제를 미리 파악하여 조치한 다음 시스템을 오픈해야 함
산출정보
성능시험 결과서
요구사항 분류
성능 요구사항
요구사항 고유번호
PER-002
요구사항 명칭
성능 최적화
요구사항
상세설명
정의
성능 최적화 방안 제시
세부내용
o 서비스 응답속도를 보장하기 위해 서버, 응용프로그램, DB 쿼리문 등의 성능 점검을 실시하고 개선 작업을 수행해야 함
o 응답속도 및 DB 성능 최적화를 수행해야 함
산출정보
-
요구사항 분류
성능 요구사항
요구사항 고유번호
PER-003
요구사항 명칭
업무 응답시간
요구사항
상세설명
정의
업무 응답시간
세부내용
o 모든 질의는 사용자가 요청한 시간으로부터 3초 내에 그 결과가 표시되어야 함
o 응답시간 예외사항
산출정보
성능 시험 계획서/결과서
관련 요구사항
N/A
요구사항 분류
성능 요구사항
요구사항 ID
PER - 004
요구사항 명칭
웹 페이지 디스플레이 시간
요구
사항
상세
설명
정의
웹 페이지 디스플레이 시간 정의
세부
내용
□ 평균 응답시간은 일반적인 온라인 트랜잭션 업무를 기준으로 최대 3초 내에 완전히 디스플레이되어야 함
ㅇ 응답시간이란 사용자 요구 시각과 시스템 응답시각 사이의 시간 간격을 의미함
ㅇ 시스템 및 기타 여러 가지 원인으로 인한 순간적 왜곡으로 인한 오류 10%는 배제 가능함
ㅇ 임의의 선택 기준이 허용되는 대량의 데이터(기준 5,000건 이상)에 대한 질의가 있을 경우에는 적용되지 않으며, 한 개 이상의 큰 이미지(이미지 500KB 이상) 및 동영상을 가지고 있는 페이지에는 적용되지 않음
ㅇ 또한 시스템을 사용하는 사용자 숫자가 동시 사용자 용량의 80%를 초과하는 경우에는 적용되지 않음
ㅇ 적정 디스플레이 시간, 기준 데이터는 기관과 협의하여 조정 가능
□ 최종 사용자 응답시간은 네트워크 시간까지 포함해야 하며, 기관 내에 위치한 사용자의 네트워크 대역폭은 유선랜 100M bps를 기준으로 함
□ 서비스 속도가 중요한 주요 프로그램에 대해서는 쿼리 튜닝 및 최적 처리결과 시간 보장방안을 제시하여야함
산출정보
성능 시험 계획서/결과서
관련 요구사항
N/A
요구사항 분류
성능 요구사항
요구사항 ID
PER - 005
요구사항 명칭
오류 응답시간
요구
사항
상세
설명
정의
오류 응답시간 목표 정의
세부
내용
□ 오류 응답시간 목표 정의
산출정보
웹사이트 용량 및 응답속도 측정 결과서, 성능측정결과서
관련 요구사항
요구사항 분류
성능 요구사항
요구사항 고유번호
PER-002
요구사항 명칭
기능 수행 지연에 대한 사전 경고
요구사항
상세설명
정의
작업 수행시간 지연 시 사전 경고 표출
세부
내용
ㅇ 대용량 데이터 질의 등 수행 지연 건은 사용자에게 사전 알림 제공
- 작업 수행시간이 오래 걸리는 경우 사용자에게 이를 안내하거나 작업을 취소하는 기능 제공
- 팝업, 프로그레스바 등의 수단으로 사용자에게 작업의 시작과 끝을 예측할 수 있도록 안내
※ 기능 수행 지연 : 10초 이상 소요 예상 건
산출정보
SW개발에 따른 산출물
관련 요구사항
요구사항 분류
성능 요구사항
요구사항 고유번호
PER-003
요구사항 명칭
시스템 자원 효율성
요구사항
상세설명
정의
시스템 정보자원 구축 운영 방안
세부
내용
ㅇ 운영 장비 및 솔루션의 성능이 최대한 보장되도록 구축해야 함
ㅇ 업무시스템의 단위테스트, 통합테스트, 시스템 성능테스트 등 단계별 기능 테스트를 실시하여 시스템 구축 시 발견된 문제점들을 보완하여 기능 및 성능 향상
- 구축 초기부터 완료까지 지속적으로 성능 및 부하테스트를 실시해야 하며, 모니터링 결과 및 시험결과를 지속적으로 반영(에러 조치 확인 및 에러 외 불편사항과 개선사항까지 보완)하여 요구사항이 모두 충족될 수 있도록 시스템을 구축해야 함
ㅇ 시스템 자원(CPU, 메모리 등)의 평균사용률은 최대 80%를 초과하지 않도록 구현해야 함(시스템 본격 사용시기의 모의추정치 기준)
ㅇ 로그아웃된 사용자에 대한 세션은 종료 처리가 되어야 함


---

### retriever B:
성능 요구사항
상세
설명
정의
정보시스템 업무 응답시간 목표
세부
내용
□ 요청(질의) 응답시간
o 모든 질의는 사용자가 요청을 하는 시간으로부터 3초 내에 최초의 결과 값을 보여줘야 함
□ 이 요구사항은 임의의 선택 기준이 허용되는 대량의 데이터(2GB 이상)에 대한 질의에는 적용되지 않음
산출정보
o SW개발에 따른 산출물
성능 요구사항
상세
설명
정의
정보시스템 업무 응답시간 목표
세부
내용
□ 요청(질의) 응답시간
o 모든 질의는 사용자가 요청을 하는 시간으로부터 3초 내에 최초의 결과 값을 보여줘야 함
□ 이 요구사항은 임의의 선택 기준이 허용되는 대량의 데이터(2GB 이상)에 대한 질의에는 적용되지 않음
산출정보
o SW개발에 따른 산출물
성능 요구사항
응낙수준
필수
요구
사항
상세
설명
정의
사용자 요구에 대한 웹페이지 디스플레이 반응 시간
세부
내용
○ 시스템은 정상 상태에서 사용자의 질의 요청에 대한 결과 페이지를 화면에 출력할 때 3초 이내 보장
○ 평균 응답시간은 시스템을 사용하는 동시 사용자가 최대 사용자 수의 90%를 초과하는 경우에는 미적용
산출정보
관련요구사항
요구사항 내용
○ 모든 질의는 사용자가 요청하는 시간으로부터 3초 이내에 그 결과값을 보여줘야 함
○ 평균 응답 시간은 시스템을 사용하는 동시 사용자가 최대 사용자 수의 90%를 초과하는 경우 미적용함
○ 한계 인원 이상 접속 시 해결 방안 제시해야 함
② 성능 요구사항
PER-001
평균응답시간 및 성능 최적화
요구사항
상세설명
정의
업무응답시간에 목표 대한 기준
세부내용
○ 모든 질의는 평균 응답시간으로 사용자가 요청한 시간으로부터 3초 내에
그 결과를 보여줘야 함
○ 대량의 데이터 로딩(통계, 검색, 이미지 등), 사용자 숫자가 동시 사용자 용량의 90%를 초과하는 경우에는 적용되지 않음
산출 정보
개발 산출물, 테스트 결과서
관련 요구사항
성능 요구사항
상세
설명
정의
평균 응답시간
세부
내용
❍ 대상시스템은 정상 상태에서 사용자의 질의 요청에 대한 결과
페이지를 화면에 출력할 때 3초 이내에 완전히 출력되어야 함
요구사항
상세설명
정의
평균 응답시간 최적화
세부
내용
○ 시스템은 정상 상태에서 사용자의 질의요청에 대한 결과 페이지를 화면에 출력할 때 3초 이내 보장
○ 사용자의 정상적인 요청에 대한 응답시간 지연을 최소화하여야 함
○ 평균 응답시간은 임의의 선택기준이 허용되는 대량의 데이터에 대한 질의가 있을 경우에는 적용되지 않으며 시스템을 사용하는 동시 사용자가 최대 사용자 수의 90%를 초과하는 경우 미적용함
○ 요청 처리 과정에서 오류가 발생할 경우, 사용자에게 오류 메시지 표시
- 대량 데이터 처리 등 처리시간을 협의하여 결정할 수 있으나 반드시 사용자가 기대할 수 있는 진행상태 등을 인식하도록 구현
4) 인터페이스 요구사항
요구사항 명칭 온라인성 업무응답시간
정의 온라인성 업무응답시간 목표
❍ 업무응답시간 요구사항
- 정보 요청에 따라 결과가 조회되어 화면에 표시되기까지의 소요 시간을
의미
- 모든 질의는 사용자가 요청을 하는 시간으로부터 3초 내에 그 결과를 처리
요구 - 포탈/학사 외에도 레거시 시스템(일반행정, 연구포탈 등)의 업무응답시간도
사항 3초 내에 그 결과를 처리해야 함
상세 세부
설명 내용 ❍ 업무응답시간 예외사항
- 요구사항은 임의의 선택 기준이 허용되는 대량의 데이터(기준 5,000건
이상)에 대한 질의 한 개 이상의 대형 이미지(기준 500KB 이상), 영상이
포함된 페이지에는 미적용하나 대량데이터의 처리시간 기준을 제시하고
적용하여야 함
- 시스템을 사용하는 사용자 숫자가 동시 사용자 용량의 90%를 초과하는
경우에는 미적용
산출정보 • 성능시험결과서(모든 시스템 화면 포함)
요구사항 명칭
질의 응답시간
요구사항 명칭
시스템 응답시간
요구사항 명칭
평균 응답시간
요구
사항
상세
트 응답속도 향상을 보장하여야 하며, 시스템은 정상 상태에서 사용자의 모든 질의 요청에 대한 처리를 3초 이내에 화면에 보여주어야 함
○ 만약 회원관리시스템이 WEB 방식 또는 C/S방식으로 개발되는 경우 모두 회원 관리시스템의 응답속도는 3초 이내가 되어야 하며 연동 장비간(POS장비 등) 지연 또는 오류가 발생되지 않도록 반드시 주의해야 한다.
○ 대량의 데이터 로딩(통계, 검색, 이미지 등), 동시접속자가 사용자 용량의 90%를 초과하는 경우에는 적용되지 않음
○ 웹페이지 용량 최소화, 연결 최소화, 페이지 표시(랜더링) 순서 최적화 등 다양한 최적화 기법을 적용하여 웹사이트의 성능을 최적화해야 함
○ 서비스 개시 전(테스트 단계) 최적화 점검을 하고, 최적화 점검결과서에 따라 반드시 문제점 등을 보완해야 함
○ 사용자 정보 요청 시 또는 사용자가 입력한 데이터 형식의 모든 오류에 대하여 3초 이내의 적절한 응답 또는 오류 메시지를 제공해야 함
○ 대량 데이터 처리 등 10초 이상 걸리는 서비스는 사용자에게 팝업 메시지를 제공해야 함
○ 오류 메시지에 중요 정보(서버 설정 정보, 세션정보, SQL정보 등)가 노출되지 않도록 적절한 에러 처리 로직 및 에러 화면을 구성하여 제공해야 함
※ 단, 평균 응답시간을 향상시키기 위한 추가시스템(WAS, 웹가속시스템 등)이 필요한 경우 발주기관과 사전 협의해야 하고, 이에 소요되는 제반비용은 수주자가 부담하여야 한다.
요구사항 명칭
평균 응답시간
요구
사항
상세
설명
정의
평균 응답시간
세부
내용
◦ 전체 성능관점에서 사용자 응답속도, 데이터 처리속도를 일정 수준 이상으로 보장하는 방안을 제안하고, 상세설계 시 반영하여야 함
◦ 제안된 방안을 기반으로 분석/설계시 목표성능 기준을 수립해야 함
산출정보
테스트 결과서
요구사항 명
시스템 응답시간
상세
설명
정의
시스템 응답시간 목표 정의
세부
내용
❑ 모든 질의는 사용자가 요청하는 시간으로부터 3초 내에 그 결과를 보여줘야 함
※ 대량의 데이터(2GB 이상)에 대한 질의는 예외로 함
❑ 시스템 내 동시 사용자가 100명 이상 접속 시에도 지원 성능이 저하되지 않아야 함
❑ 대량 데이터 처리 등 10초 이상 걸리는 서비스는 작업에 대한 안내 화면을 제공해야 함
❑ 데이터 형식 오류 시 오류 메시지 제공
- 사용자가 입력한 데이터 형식의 모든 오류는 오류 메시지를 사용자에게 제시하여야 함
- 오류 메시지에 중요정보(서버 설정 정보, 세션 정보, SQL정보 등)가 노출되지 않도록 적당한 에러 처리 로직을 구성하여야 함
❑ 응답속도 향상을 위한 관리, 개선방안을 제시하여야 함
산출정보
2. 평균 응답 시간은 동시 접속자 수가 PER-002에서 요구하는 포털 동시
접속자 수의 90%를 초과 하는 경우 적용되지 않음
요구사항
상세설명
정의
평균 응답시간 목표 정의
세부
내용
○ 구축 시스템은 정상상태에서 평균 4초 이내에 사용자의 질의요청에 대한 결과가 화면에 출력되어야 한다.
○ 평균 처리시간은 시스템을 사용하는 동시 사용자가 전체 사용자 수의 90%를 초과하는 경우에는 적용되지 않는다.
산출정보
관련요구사항
요구사항 명칭
평균 응답 시간
응낙수준
필수
요구사항 명칭
평균 응답시간
요구
사항
상세
트 응답속도 향상을 보장하여야 하며, 시스템은 정상 상태에서 사용자의 모든 질의 요청에 대한 처리를 3초 이내에 화면에 보여주어야 함
○ 만약 고객정보통합시스템(회원관리시스템)이 WEB 방식 또는 C/S방식으로 개발되는 경우 모두 고객정보통합시스템(회원관리시스템)의 응답속도는 3초 이내가 되어야 하며 연동 장비간(POS장비 등) 지연 또는 오류가 발생되지 않도록 반드시 주의해야 한다.
○ 대량의 데이터 로딩(통계, 검색, 이미지 등), 동시접속자가 사용자 용량의 90%를 초과하는 경우에는 적용되지 않음
○ 웹페이지 용량 최소화, 연결 최소화, 페이지 표시(랜더링) 순서 최적화 등 다양한 최적화 기법을 적용하여 웹사이트의 성능을 최적화해야 함
○ 서비스 개시 전(테스트 단계) 최적화 점검을 하고, 최적화 점검결과서에 따라 반드시 문제점 등을 보완해야 함
○ 사용자 정보 요청 시 또는 사용자가 입력한 데이터 형식의 모든 오류에 대하여 3초 이내의 적절한 응답 또는 오류 메시지를 제공해야 함
○ 대량 데이터 처리 등 10초 이상 걸리는 서비스는 사용자에게 팝업 메시지를 제공해야 함
○ 오류 메시지에 중요 정보(서버 설정 정보, 세션정보, SQL정보 등)가 노출되지 않도록 적절한 에러 처리 로직 및 에러 화면을 구성하여 제공해야 함
※ 단, 평균 응답시간을 향상시키기 위한 추가시스템(WAS, 웹가속시스템 등)이 필요한 경우 발주기관과 사전 협의해야 하고, 이에 소요되는 제반비용은 사업수행자가 부담하여야 한다.
요구사항
세부내용
정의
산 질의응답 시간
세부
내용
개발시스템의 데이터 조회 및 선량평가 수행 시 처리속도 측면에서 최적화된 시스템을 구축해야 함
- 사용자의 건별 질의 요청에 대한 처리를 3초 이내에 화면에 보여주어야 함
- 계산수행 등 10초 이상 걸리는 서비스는 사용자에게 팝업 메시지를 제공해야 함
요구사항 명칭
오류 응답시간 목표 정의
요구사항
상세설명
정의
업무 응답시간
세부내용
o 모든 질의는 사용자가 요청한 시간으로부터 3초 내에 그 결과가 표시되어야 함
o 응답시간 예외사항
- 대량의 데이터 저장, 통계 분석 리포트, 파일 다운로드 등
o 대량 데이터 다운로드 시 다운로드 실패가 발생하지 않아야 함
- 대량의 데이터 다운로드 시 다운로드 실패 발생 원인 분석하여 개선해야 함
산출정보
-
요구사항
상세설명
정의
개발되어지는 모든 센터시스템 소프트웨어의 사용자 응답속도에 대한 성능
세부내용
○ 사용자와의 대화형 기능의 응답시간
▹ 대부분의 어플리케이션 기능이 3초 이내에 응답가능 하도록 개발
▹ 단, 데이터의 양이 많아, 로딩 시간이 길어질 경우, 운영자가 로딩중임을 인지할 수 있는 그래픽 상태바를 표시 하도록 함
○ 데이터 생성 등의 배치 작업 응답시간
▹ 통계 데이터 생성 및 가공 등 배치 작업에 따른 응답시간이 과하게 소요되는 통계 등의 모듈에 대하여는 정보 제공용 통계정보를 별도로 생성하여 응답시간을 최소화 하도록 함
○ 데이터 생성 등의 배치 작업 응답시간
▹ 통계 데이터 생성 및 가공 등 배치 작업의 응답시간에 대해 사전에 예상 소요시간을 제시
요구사항
상세설명
정의
질의응답 결과 시간과 예외 적용 제시
세부내용
◦시스템은 정상운영 상태에서 데이터 요청에 대하여 3초 이내 응답할 수 있도록 개발
- 예외사항
·대량의 데이터 질의 및 다운로드가 있을 경우
·시스템 장애 및 동시 사용자 성능을 초과하는 경우
산출정보
성능시험결과서
관련요구사항
3) 인터페이스 요구사항(System Interface Requirement)


---

### openai answer
retriever A 점수: 3점
- 모든 질의는 3초(3000ms) 내 결과 제공 명시
- 10초 이상 소요되는 작업은 팝업 등으로 지연 안내 명시

retriever B 점수: 3점
- 일반 질의 3초 내 결과 제공 반복 명시
- 10초 이상 소요 시 팝업/안내 화면 제공 명시
- 선량평가 수행 맥락에서도 “3초 내 처리, 10초 이상 팝업” 요건을 직접 포함 

============================================================
### query:
원자력 선량평가시스템의 포트란 모듈 개선 요구사항에서 추가로 고려해야 하는 평가 요소는 무엇인가?

### answer:
 ICRP60 기준에 따른 23개 장기를 고려한 평가장기 추가가 요구된다.

---

### retriever A:
가 방법
평가부문
평가항목
평가기준
배점
한도
정량
평가
(10)
경영상태
경영상태(신용상태등급) 평가
10
전략 및 방법론
(22)
사업
이해도
사업의 특성 및 목표에 부합하는 사업전략을 제시하고 있는지를 평가한다.
7
추진전략
개발업무 수행 시 일정 및 위험요소를 고려하여 얼마나 타당한 추진 전략을 수립하였는지 평가한다.
5
적용기술
사업에서 적용하고자 하는 기술이 향후 확장성을 고려하여 현실적으로 실현 가능하게 제시되어 있는지를 평가한다.
4
표준프레임 워크 적용
제안사는 사업에 네팔 정보통신기술부 프레임워크 기반의 공통 컴포넌트의 유형 및 사용계획을 기술하고 사용성 향상을 위한 추가 컴포넌트에 대해 적용방안을 제시하여야한다.
3
개발
방법론
사업에 적정한 개발 방법론이 제시되었는지 평가하고, 실제 적용 사례 및 경험을 바탕으로 단계별 산출물이 제시되었는지 여부를 평가한다.
3
기술 및 기능
(26)
시스템 장비 구성 요구사항
요구 규격을 충족하는 장비 제안 여부와 현 시스템과의 인터페이스 및 확장 가능성 여부를 평가한다. 또한 도입 장비의 설치 및 공급 계획, 유지관리에 대한 방안이 구체적으로 기술되어 있는가를 평가한다.
7
기능 요구사항
기능 요구사항·기대사항·제약사항 등을 파악한 후 구현 방안이 구체적으로 기술되어 있는지를 평가한다. 또한 제안한 방안 및 기술이 적용 가능한지를 평가한다.
7
데이터
요구사항
데이터 요구사항 분석을 바탕으로 데이터 전환계획 및 검증 방법, 데이터 오류 발생 시 처리 방안이 구체적으로 제시되어 있는지를 평가한다.
4
보안
요구사항
기능 보안
체적이고 적절한지 평가
(30점) 요구사항
- 구현방안이 설계단계부터 구현 및 검증단계까지 고려하고 있는지 평가
- 데이터 전환 계획 및 검증 방법, 에러 데이터 처리 방법에 대해 구체적
데이터
인 내용을 제시하고 있는지 평가 15
요구사항
- 데이터 전환을 위해 책임 조직이 투입되는지 평가
- 기능 및 품질 등 요구사항을 구현 시 관련 제약사항을 충족시키며 구현
제약사항
방안 및 테스트 방안을 구체적으로 기술하였는지 평가
-237-
평가
평가항목 평가기준 배점
부문
- 구현하고자 하는 기능에 요구 성능이 충족되도록 방법론 및 분석 도구
성능 를 통하여 내용을 구체적으로 분석하였는지, 구현 및 테스트 방안을 구
요구사항 체적으로 기술하였는지 평가
- 제안한 방안 및 기술을 통해 요구 성능을 충족시킬 수 있는지 평가
- 분석 도구 및 구현 방안, 테스트 방안을 구체적으로 기술하였는지, 분석
·설계 등 각 단계별 품질 요구사항의 점검 및 검토 방안을 구체적으로
품질
계획하였는지 평가
요구사항
- 각 단계마다 품질 요구사항을 점검하는 별도의 전문 인력이 투입되는지
성능 및 평가
품질 - (시스템 인터페이스) 타 시스템과의 연계 방안들에 대한 장·단점의 분석 10
(10점) 을 통해 가장 적합한 방안을 구체적으로 기술했는지, 구현 경험이 있는
인터페이
인터페이스 담당자가 투입되는지 평가
스
- (사용자 인터페이스) 사용자 편의성을 고려하여 요구사항을 제공하기 위
요구사항
한 분석 및 설계, 구현 방안과 검토 계획을 구체적으로 기술하였는지,
구현 경험이 있는 사용자 인터페이스 담당자가 투입되는지 평가
적용 기술
사업에서 적용하고자 하는 기술이 향후 확장성을 고려하였는지, 현실적으로 실현 가능한지 여부를 평가한다.
표준 프레임
워크 적용
표준 프레임워크 적용 여부와 적용 시 예상되는 문제점을 구체적으로 기술하고 실현 가능한 대응 방안을 제시하였는지 평가한다.
개발 방법론
사업에 적정한 개발 방법론이 제시되었는지 평가하고, 실제 적용 사례 및 경험을 바탕으로 단계별 산출물이 제시되었는지 여부를 평가한다.
기술 및 기능
시스템 요구
사항
요구 규격을 충족하는 장비 제안 여부와 현 시스템과의 인터페이스 및 확장 가능성 여부를 평가한다. 또한 도입 장비의 설치 및 공급 계획, 유지관리에 대한 방안이 구체적으로 기술되어 있는가를 평가한다.
25
기능 요구
사항
기능 요구사항ㆍ기대사항ㆍ제약사항 등을 파악한 후 구현 방안이 구체적으로 기술되어 있는지를 평가한다. 또한 제안한 방안 및 기술이 적용 가능한지를 평가한다.
보안 요구
사항
요구사항 및 시스템 분석을 바탕으로 보안 요구사항의 적용 방안이 구체적으로 기술되었는지 평가한다. 또한 제안한 방안과 기술의 설계단계 반영 여부 및 구현단계까지의 적용 가능성을 평가한다.
데이터 요구
사항
데이터 요구사항 분석을 바탕으로 데이터 전환 계획 및 검증 방법, 데이터 오류 발생 시 처리 방안이 구체적으로 제시되어 있는지를 평가한다.
시스템운영
요구 사항
시스템 운영 요구 사항에 맞는 운영 절차 및 방법을 제시하였는지 평가한다. 또한 운영 중 비상사태 발생 시 대응방안이 구체적으로 제시되고 있는지를 평가한다.
제약 사항
o 제안사는 기술평가 결과에 대해 일체의 이의를 제기할 수 없음
<기술제안서 기술평가 평가항목 및 평가배점>
평가부문
평가항목
평가기준
배점
전략 및 방법론
(20)
사업 이해도
사업수행계획이 사업의 목표 특성에 부합하며, 구체적이고 적절한지 평가
10
추진전략
사업수행 일정과 위험요소를 고려한 추진전략 수립의 타당성을 평가
5
사업추진
방법론
사업추진방법이 구체적이고 적절한지와 그 실제 적용 사례 및 경험에 따른 단계별 산출물을 제시하였는지 평가
5
기술 및 기능
(25)
기능 요구사항
기능 요구사항기대사항제약사항 등을 파악한 후 구현 방안이 구체적으로 기술되어 있는지 평가
10
보안 요구사항
요구사항 및 시스템 분석을 바탕으로 보안 요구사항의 적용 방안이 구체적으로 기술되었는지 평가
5
데이터
요구사항
데이터 요구사항 분석을 바탕으로 데이터 전환계획 및 검증방법, 데이터 오류 발생 시 처리방안이 구체적으로 제시되어 있는지 평가
5
시스템운영
요구사항
시스템 운영 요구 사항에 맞는 운영 절차 및 방법을 제시하였는지 평가
5
성능 및 품질
(15)
성능 요구사항
요구 성능 충족을 위한 구현 및 테스트 방안이 구체적으로 기술되어 있는지, 이를 위한 방법론 및 분석도구가 기술되어 있는지 평가
5
품질 요구사항
제공되는 개발 도구 및 구현 방안, 테스트 방안 등이 품질 요구사항에 부합되는지 평가하고, 분석설계구현테스트 등 각 단계별 품질 요구사항 점검 및 검토방안이 구체적으로 계획되어 있는가를 평가
5
인터
페이스
요구사항
[별표5] 기술제안서 정성평가 항목 및 평가기준
평가부문
평가항목
평가기준
전략 및 방법론
사업 이해도
사업의 특성 및 목표에 부합하는 사업전략을 제시하고 있는지를 평가한다.
추진 전략
개발업무 수행 시 일정 및 위험요소를 고려하여 얼마나 타당한 추진 전략을 수립하였는지 평가한다.
적용 기술
사업에서 적용하고자 하는 기술이 향후 확장성을 고려하였는지, 현실적으로 실현 가능한지 여부를 평가한다.
표준 프레임
워크 적용
표준 프레임워크 적용 여부와 적용 시 예상되는 문제점을 구체적으로 기술하고 실현 가능한 대응 방안을 제시하였는지 평가한다.
개발 방법론
사업에 적정한 개발 방법론이 제시되었는지 평가하고, 실제 적용 사례 및 경험을 바탕으로 단계별 산출물이 제시되었는지 여부를 평가한다.
기술 및 기능
시스템 요구
사항
요구 규격을 충족하는 장비 제안 여부와 현 시스템과의 인터페이스 및 확장 가능성 여부를 평가한다. 또한 도입 장비의 설치 및 공급 계획, 유지관리에 대한 방안이 구체적으로 기술되어 있는가를 평가한다.
기능 요구
사항
기능 요구사항ㆍ기대사항ㆍ제약사항 등을 파악한 후 구현 방안이 구체적으로 기술되어 있는지를 평가한다. 또한 제안한 방안 및 기술이 적용 가능한지를 평가한다.
보안 요구
사항
요구사항 및 시스템 분석을 바탕으로 보안 요구사항의 적용 방안이 구체적으로 기술되었는지 평가한다. 또한 제안한 방안과 기술의 설계단계 반영 여부 및 구현단계까지의 적용 가능성을 평가한다.
데이터 요구
사항
기술
및
기능
(30점)
보안 요구사항
5
관련 기능 등 타 요구사항 및 시스템과 관련되어 분석되고, 적용할 표준 및 구현 방안이 설계단계부터 반영되어 구체적으로 기술되어 있는가를 평가하고, 제안한 방안 및 기술이 적용 가능한지를 평가한다.
시스템
운영
요구사항
15
상시 근무인력의 기술 숙련 정도, 헬프데스크 운영인력의 적정성 등 시스템 위탁운영 시 필요한 사항에 대하여 경험을 바탕으로 제시하고, 고려 사항 및 유사 시 대응책을 제시하고 있는가를 평가한다.
제약사항
5
제약사항 충족도는 기능 및 품질 등 요구사항을 구현 시 관련 제약사항을 충족시키며 구현 방안 및 테스트 방안을 구체적으로 기술하였는가를 평가한다.
테스트
요구사항
5
개발된 시스템의 시험운영을 위해 제공 및 지원하는 각종 시험운영 방법 및 조직 등에 대해 평가한다.
사업 관리
(30점)
관리
방법론
5
사업위험, 사업진도, 사업수행시 보안을 관리하는 방법, 사업수행 성과물이나 산출물의 형상 및 문서를 관리하는 방법 등을 평가한다.
사업수행 조직
5
사업수행 조직 및 지원조직 구성 및 관리 방안이 적정하게 제시되었는가와 위험관리를 대비한 비상운영체계 수행방안이 적정한가를 평가한다.
사업수행
투입인력
10
투입인력의 보유기술(자격) 수준의 적정성 및 유사사업 수행 경험(경력)을 바탕으로 기술 등급별 투입인력의 규모가 적정한가를 평가한다.
이해도 및 준비성
10
③ 수치지도상에서 분기지점, 정수장 위치 및 도･송수관로의 주요 정점 등에 대한 표고를 산출하여 수리검토 등을 통하여 적정 동수압이 유지될 수 있도록 계획하여야 한다.
④ 기 시행된 시설계획을 고려하여 공급 지역별로 안정적인 용수를 공급받을 수 있는 가압시설 계획을 계획하여야 한다.
⑤ 부지 선정 시 현지답사를 통하여 해당지자체 및 주민의 의견을 수렴하고 이를 감독원과 협의하여 적극적으로 반영하여야 한다.
2.2.2.6 타당성 평가
계약상대자는 타당성 평가를 시행함에 있어, 건설기술진흥법 시행령 제81조(건설 공사의 타당성조사) 및 「K-water 건설기술관리규정 제17조」(타당성조사)에서 제시한 기준을 충족하도록 다음 사항을 충실히 검토하여야 한다. 용인 첨단 시스템반도체 국가산단 용수공급사업과 관련한 조사 및 기존 검토 내용을 충실히 반영하여 단계별 시행방안 등을 포함한 종합적인 검토가 될 수 있도록 하여야 한다.
(1) 각 안별 개략사업비 비교검토
① 상기에서 검토한 개발물량 및 시설물계획 등을 다각적으로 검토하고 각 방안별로 산정된 개략사업비를 비교 검토하여 최적의 방안을 제시한다.
② 각 안별 예상되는 운영 및 유지관리비를 산출하여야 한다.
③ 용수공급 계획 수립 시에 용수배분계획량, 주요시설물 설치계획 및 관로 설치한계 등에 대하여 관련 지자체와 협의가 필요한 사항에 대하여는 충분히 협의하여 그 결과를 정리․검토한다.
(2) 사업효과 분석
① 기 수립된 상위계획과 현지여건 등을 종합적으로 감안하여 사업의 필요성에 대해 검토한다.


---

### retriever B:
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항


---

### openai answer
retriever A 점수: 0점
질문 핵심(원자력 선량평가시스템, 포트란 모듈 개선, ICRP60, 23개 장기)에 대한 언급이 전혀 없고, 일반적인 기술제안서 평가기준만 포함되어 있어 연관성이 없음.

retriever B 점수: 0점
“기능 요구사항” 반복 외에 내용이 없으며, 원자력 선량평가, 포트란, ICRP60, 23개 장기 등 답변에 필요한 정보가 전혀 없음. 

## Query 2

## size 500

============================================================
### query:
본 과업지시서에서 '극저온시스템'은 무엇을 포함하는 것으로 정의하고 있습니까?

### answer:
'극저온시스템'은 SCL3/SCL2 극저온플랜트 및 부대설비, SCL3/SCL2 분배시스템 및 부대설비, SCL3/SCL2 초전도가속관 저온유지모듈, 기체헬륨 탱크, 액체질소 시스템 및 극저온시스템용 헬륨회수시스템을 모두 포함하는 것으로 정의됩니다.

---

### retriever A:
7. 안전관리 .............................................................................................................. 19
8. 붙임 자료 ........................................................................................................... 19
문서번호 : -
2025년도 중이온가속기용 극저온시스템
개정번호 : 0
운전 용역 과업지시서 발 행 일 : 2024. 10. 30
페 이 지 : 5/19
1. 서 론
1.1 목 적
기초과학연구원 중이온가속기연구소(이하 “연구원”)에서 운영중인 중이온가속기용 극저
온시스템(이하 “극저온시스템”)은 고도의 공정 시스템으로, 안전하고 안정적인 중이온가속
기의 운영을 위해 상시 모니터링이 필요하다. 이에 본 과업지시서는 극저온시스템의 운전
용역에 대한 상세 과업 내용을 기술한다.
1.2 용어 설명
본 과업지시서에서 사용하는 용어에 대한 상세 내용은 표 1.과 같다. 계약자는 반드시 해당
용어의 의미를 충분히 이해하고 숙지해야 하며, 용어 미숙지로 인한 책임은 계약자에게 있
다.
표 1. 본 과업지시서에 사용되는 용어 설명
용어 설명
연구원 중이온가속기연구소
계약자 본 과업을 수행할 계약상대자
SCL3/SCL2 극저온플랜트 및 부대설비,
SCL3/SCL2 분배시스템 및 부대설비,
SCL3/SCL2 초전도가속관 저온유지모듈,
극저온시스템
기체헬륨 탱크, 액체질소 시스템 및


---

### retriever B:
1. 서 론
1.1 목 적
기초과학연구원 중이온가속기연구소(이하 “연구원”)에서 운영중인 중이온가속기용 극저
온시스템(이하 “극저온시스템”)은 고도의 공정 시스템으로, 안전하고 안정적인 중이온가속
기의 운영을 위해 상시 모니터링이 필요하다. 이에 본 과업지시서는 극저온시스템의 운전
용역에 대한 상세 과업 내용을 기술한다.
1.2 용어 설명
본 과업지시서에서 사용하는 용어에 대한 상세 내용은 표 1.과 같다. 계약자는 반드시 해당
용어의 의미를 충분히 이해하고 숙지해야 하며, 용어 미숙지로 인한 책임은 계약자에게 있
다.
표 1. 본 과업지시서에 사용되는 용어 설명
용어 설명
연구원 중이온가속기연구소
계약자 본 과업을 수행할 계약상대자
SCL3/SCL2 극저온플랜트 및 부대설비,
SCL3/SCL2 분배시스템 및 부대설비,
SCL3/SCL2 초전도가속관 저온유지모듈,
극저온시스템
기체헬륨 탱크, 액체질소 시스템 및
극저온시스템용 헬륨회수시스템 포함
(“3.2절 운전대상”, 표 3. 참고)
중이온가속기 또는 극저온시스템이 냉각(냉각 착수
가동기간
이후) 또는 빔서비스 등을 위해 가동 중인 기간
중이온가속기 또는 극저온시스템이 유지보수 등을
휴지기간
위해 가동을 멈추고 휴지 중인 기간
중이온가속기용 극저온시스템이 가동기간 및 휴지
기간에 안전하고 안정적으로 운영될 수 있도록,
중이온가속기용 극저온시스템 i) 극저온시스템을 모니터링 및 제어하고,
운전 업무 ii) 극저온시스템의 주요 수치를 기록 및 분석하며,
iii) 중이온가속기용 극저온시스템의 운전 경험을
통합하여 정리(문서화)하는 업무
문서번호 : -


---

### openai answer
retriever A 점수: 2점
일부 포함. SCL3/SCL2 플랜트/분배시스템/저온유지모듈, 기체헬륨 탱크, 액체질소 시스템까지 보이나 문구가 절단되어 ‘극저온시스템용 헬륨회수시스템’이 확인되지 않음.

retriever B 점수: 3점
모두 포함. 위 항목들에 더해 ‘극저온시스템용 헬륨회수시스템 포함’이 명시되어 있어 정답의 모든 구성요소가 들어있음. 

============================================================
### query:
극저온시스템 과업 추진 일정에 따르면, 4분기에 SCL3 구간에 대해 예정된 주요 업무는 무엇이며, 이때의 업무 기간은 '가동기간'과 '휴지기간' 중 어느 것에 해당합니까?

### answer:
2025년 4분기에 SCL3 구간에 예정된 주요 업무는 '유지보수'이며, 이 기간은 '휴지기간'에 해당합니다.

---

### retriever A:
다. 계약자는 본 과업을 수행하기 위해 적절한 지식 또는 능력을 보유하고 있는 적정 수의 운
전 인력을 연구원에게 제공하여야 한다.
• 가동기간의 극저온시스템 운전 업무 (24시간 운전)
- 극저온시스템 모니터링
- 극저온시스템 주요 수치 기록 및 분석
- 극저온시스템 제어
- 기타 가동기간에 필요한 운전 업무
• 휴지기간의 극저온시스템 운전 업무 (주간 운전)
- 유지보수 등을 위한 극저온시스템 제어
- 유지보수 등을 위한 극저온시스템 모니터링
- 유지보수 등을 위한 극저온시스템 주요 수치 기록 및 분석
- 극저온시스템 운전 매뉴얼 작성 및 개정
- 극저온시스템 운전 교육
- 성과보고회 개최
- 기타 휴지기간에 필요한 운전 업무
• 본 과업 및 본 과업을 위해 투입된 인력에 대한 관리감독
본 과업의 기간은 2025년 1월 1일부터 2025년 12월 31일까지로 하며, 2025년 동안 가동기
간의 운전 업무를 수행하는 것을 기준으로 한다. 단, 표 2.와 같이 연구원의 운영 추진 상황
에 따라 휴지기간의 운전업무를 수행하는 것으로 변경될 수 있다. 가동기간과 휴지기간의
운전 업무는 “3. 과업 내용” 하위 절에 기술한 내용에 따른다.
표 2. 과업 추진 일정(안)
2025년
구분
1분기 2분기 3분기 4분기
SCL3 극저온플랜트
SCL3 중이온가속기 SCL3 구간 운전 SCL3 구간 유지보수
시운전 준비 및
구간 (가동기간) (휴지기간)
시운전 (가동기간)
SCL2 SCL2 극저온플랜트 중이온가속기 SCL2 구간 시운전 SCL2 구간 유지보수
구간 시운전 (가동기간) (가동기간) (휴지기간)
회수


---

### retriever B:
다. 계약자는 본 과업을 수행하기 위해 적절한 지식 또는 능력을 보유하고 있는 적정 수의 운
전 인력을 연구원에게 제공하여야 한다.
• 가동기간의 극저온시스템 운전 업무 (24시간 운전)
- 극저온시스템 모니터링
- 극저온시스템 주요 수치 기록 및 분석
- 극저온시스템 제어
- 기타 가동기간에 필요한 운전 업무
• 휴지기간의 극저온시스템 운전 업무 (주간 운전)
- 유지보수 등을 위한 극저온시스템 제어
- 유지보수 등을 위한 극저온시스템 모니터링
- 유지보수 등을 위한 극저온시스템 주요 수치 기록 및 분석
- 극저온시스템 운전 매뉴얼 작성 및 개정
- 극저온시스템 운전 교육
- 성과보고회 개최
- 기타 휴지기간에 필요한 운전 업무
• 본 과업 및 본 과업을 위해 투입된 인력에 대한 관리감독
본 과업의 기간은 2025년 1월 1일부터 2025년 12월 31일까지로 하며, 2025년 동안 가동기
간의 운전 업무를 수행하는 것을 기준으로 한다. 단, 표 2.와 같이 연구원의 운영 추진 상황
에 따라 휴지기간의 운전업무를 수행하는 것으로 변경될 수 있다. 가동기간과 휴지기간의
운전 업무는 “3. 과업 내용” 하위 절에 기술한 내용에 따른다.
표 2. 과업 추진 일정(안)
2025년
구분
1분기 2분기 3분기 4분기
SCL3 극저온플랜트
SCL3 중이온가속기 SCL3 구간 운전 SCL3 구간 유지보수
시운전 준비 및
구간 (가동기간) (휴지기간)
시운전 (가동기간)
SCL2 SCL2 극저온플랜트 중이온가속기 SCL2 구간 시운전 SCL2 구간 유지보수
구간 시운전 (가동기간) (가동기간) (휴지기간)
회수


---

### openai answer
retriever A 점수: 3점
표 2의 과업 추진 일정에 4분기 SCL3 구간이 ‘유지보수(휴지기간)’로 명시되어 있어, 질문의 두 요소(주요 업무와 기간 구분)를 모두 포함합니다.

retriever B 점수: 3점
동일하게 표 2에서 4분기 SCL3 구간이 ‘유지보수(휴지기간)’로 명시되어 있어, 필요한 정보가 모두 들어 있습니다. 

============================================================
### query:
극저온시스템 '가동기간' 동안 계약자가 24시간 수행해야 할 극저온시스템 모니터링 업무에는 어떤 것들이 있습니까? (3가지 이상)

### answer:
가동기간 동안 수행해야 할 모니터링 업무는 다음과 같습니다.
알람 메시지 확인
주요 센서(온도, 압력, 유량 등)의 트렌드 확인
정상 운전 상황의 주요 센서 지시값 파악 및 숙지
비정상 운전 상황 감지
제어로직 정상 작동 여부 확인

---

### retriever A:
다. 가동기간의 모니터링 업무는 극저온시스템이 안정적으로 가동되고 있는지 여부를 확인
하기 위해 극저온시스템을 지속적으로 관찰하는 업무를 의미하며, 표 4.에 가동기간의 모니
터링 업무에 해당되는 내용들을 기술하였다.
극저온시스템의 모니터링은 연구원에 구축되어있는 극저온시스템용 제어시스템(이하 “제
어시스템”)을 활용하여 또는 현장에 설치되어있는 장치를 통해 수행한다. 연구원은 구축되
어있는 제어시스템을 활용하기 위한 컴퓨터, 모니터, 키보드, 마우스, 책상 및 의자를 계약
자에게 제공한다. 현재 연구원에 구축되어있는 제어시스템은 표 5.과 같다.
연구원은 계약자에게 현재 구축되어있는 제어시스템에 대한 기본 사용 교육을 제공하며,
계약자는 연구원이 제공하는 교육을 이해하기 위한 배경지식을 보유해야 한다.
문서번호 : -
2025년도 중이온가속기용 극저온시스템
개정번호 : 0
운전 용역 과업지시서 발 행 일 : 2024. 10. 30
페 이 지 : 9/19
표 4. 가동기간의 극저온시스템 모니터링 업무 리스트
순번 극저온시스템 모니터링 업무 비고
1 알람 메시지 확인
2 주요 센서 트렌드 확인 온도, 압력, 유량, 속도 등
3 정상 운전 상황의 주요 센서 지시값 파악 및 숙지 온도, 압력, 유량, 속도 등
4 비정상 운전 상황 감지
연구원에서 현장 확인 필요상황 발생 시, 연구원이 지 무전 통신 활용
5
정하는 값 관찰, 확인 및 상황 전달 (연구원에서 무전기 제공)
6 제어로직 정상 작동 여부 확인
연구원 요청 시,
7 현장 상황 일시적 또는 지속적 관찰
비방사선구역 한정


---

### retriever B:
다. 가동기간의 모니터링 업무는 극저온시스템이 안정적으로 가동되고 있는지 여부를 확인
하기 위해 극저온시스템을 지속적으로 관찰하는 업무를 의미하며, 표 4.에 가동기간의 모니
터링 업무에 해당되는 내용들을 기술하였다.
극저온시스템의 모니터링은 연구원에 구축되어있는 극저온시스템용 제어시스템(이하 “제
어시스템”)을 활용하여 또는 현장에 설치되어있는 장치를 통해 수행한다. 연구원은 구축되
어있는 제어시스템을 활용하기 위한 컴퓨터, 모니터, 키보드, 마우스, 책상 및 의자를 계약
자에게 제공한다. 현재 연구원에 구축되어있는 제어시스템은 표 5.과 같다.
연구원은 계약자에게 현재 구축되어있는 제어시스템에 대한 기본 사용 교육을 제공하며,
계약자는 연구원이 제공하는 교육을 이해하기 위한 배경지식을 보유해야 한다.
문서번호 : -
2025년도 중이온가속기용 극저온시스템
개정번호 : 0
운전 용역 과업지시서 발 행 일 : 2024. 10. 30
페 이 지 : 9/19
표 4. 가동기간의 극저온시스템 모니터링 업무 리스트
순번 극저온시스템 모니터링 업무 비고
1 알람 메시지 확인
2 주요 센서 트렌드 확인 온도, 압력, 유량, 속도 등
3 정상 운전 상황의 주요 센서 지시값 파악 및 숙지 온도, 압력, 유량, 속도 등
4 비정상 운전 상황 감지
연구원에서 현장 확인 필요상황 발생 시, 연구원이 지 무전 통신 활용
5
정하는 값 관찰, 확인 및 상황 전달 (연구원에서 무전기 제공)
6 제어로직 정상 작동 여부 확인
연구원 요청 시,
7 현장 상황 일시적 또는 지속적 관찰
비방사선구역 한정


---

### openai answer
retriever A 점수: 3점
(표 4에 알람 메시지 확인, 주요 센서 트렌드 확인, 정상 지시값 파악·숙지, 비정상 상황 감지, 제어로직 정상 작동 여부 확인 등 답변의 항목이 모두 포함됨)

retriever B 점수: 3점
(표 4에 동일하게 알람 확인, 센서 트렌드, 정상 지시값 숙지, 비정상 감지, 제어로직 확인 등 답변의 모든 항목이 명시됨) 

============================================================
### query:
중이온가속기 극저온시스템의 '가동기간'과 '휴지기간'의 운전 업무 형태(근무 방식)는 어떻게 다릅니까?

### answer:
'가동기간'에는 시스템의 24시간 안정적인 운영을 위해 교대근무(24시간 운전)가 이루어집니다. 반면, '휴지기간'에는 주로 시스템 유지보수 등을 위해 주간 운전을 기본으로 합니다.

---

### retriever A:
중이온가속기 또는 극저온시스템 가동기간에는 24시간 운전 업무를 필요로 한다. 통상적으
로 가동기간에는 24시간 운전을 위한 교대근무가 이루어지며, i) 극저온시스템의 모니터링
및 제어, ii) 극저온시스템의 주요 수치를 기록 및 분석하는 업무가 주로 이루어진다.
다만, 통상적으로 이러한 교대근무로 인해 각 운전 인력마다 경험할 수 있는 운전 상황이
서로 다르게 되고, 서로 다른 경험을 했던 운전 인력 모두가 참여하여 각자의 iii) 운전 경험
을 통합/정리할 수 있는 여건이 형성될 수 없다는 한계가 존재한다. 이러한 운전 경험의 통
합 정리는 후술하는 휴지기간에의 운전업무에 집중적으로 수행 가능하다.
통상적으로 휴지기간에는 극저온시스템의 유지보수 등이 진행되며, 유지보수 등을 위한 제
어, 유지보수 등의 경과 및 결과 확인을 위한 극저온시스템 모니터링, 유지보수 등의 결과
확인을 위한 주요 수치 기록 및 분석 또한 필수적인 운전 업무이다. 휴지기간에는 통상적으
로 주간 근무로 운전이 이루어진다. 이와 같이 주간 근무가 이루어질 수 있는 휴지기간의
특성 상, 휴지기간은 가동기간에서 교대근무로 각 운전 인력 개인이 얻은 단편적인 운전 경
험을 통합하여 정리(문서화)할 수 있는 여건이 형성될 수 있는 기간이다.
결과적으로 휴지기간에는 i) 중이온가속기용 극저온시스템의 운전 경험을 통합하여 정리(문
서화)하는 업무가 집중적으로 이루어지며, ii) 유지보수 등을 위한 극저온시스템의 제어와 iii)
유지보수 등을 위한 극저온시스템의 주요 수치 기록 및 분석이 병행되어야 한다.
3.2 운전 대상 및 장소


---

### retriever B:
3. 과업 내용
3.1 중이온가속기용 극저온시스템 운전 업무의 정의 및 필요성
본 과업에서 사용하는 “중이온가속기용 극저온시스템 운전 업무”의 정의는, “중이온가
속기용 극저온시스템이 안전하고 안정적으로 운영(가동기간뿐만 아니라 휴지기간의 운영 포
함)될 수 있도록 i) 극저온시스템을 모니터링 및 제어하고, ii) 극저온시스템의 주요 수치를
기록 및 분석하며, iii) 중이온가속기용 극저온시스템의 운전 경험을 통합하여 정리(문서화)하
는 업무를 의미한다.
중이온가속기 또는 극저온시스템 가동기간에는 24시간 운전 업무를 필요로 한다. 통상적으
로 가동기간에는 24시간 운전을 위한 교대근무가 이루어지며, i) 극저온시스템의 모니터링
및 제어, ii) 극저온시스템의 주요 수치를 기록 및 분석하는 업무가 주로 이루어진다.
다만, 통상적으로 이러한 교대근무로 인해 각 운전 인력마다 경험할 수 있는 운전 상황이
서로 다르게 되고, 서로 다른 경험을 했던 운전 인력 모두가 참여하여 각자의 iii) 운전 경험
을 통합/정리할 수 있는 여건이 형성될 수 없다는 한계가 존재한다. 이러한 운전 경험의 통
합 정리는 후술하는 휴지기간에의 운전업무에 집중적으로 수행 가능하다.
통상적으로 휴지기간에는 극저온시스템의 유지보수 등이 진행되며, 유지보수 등을 위한 제
어, 유지보수 등의 경과 및 결과 확인을 위한 극저온시스템 모니터링, 유지보수 등의 결과
확인을 위한 주요 수치 기록 및 분석 또한 필수적인 운전 업무이다. 휴지기간에는 통상적으
로 주간 근무로 운전이 이루어진다. 이와 같이 주간 근무가 이루어질 수 있는 휴지기간의


---

### openai answer
retriever A 점수: 3점
가동기간의 24시간 교대근무와 휴지기간의 주간 근무(유지보수 중심)가 모두 명시되어 있어 답변에 필요한 정보가 완전히 포함됨.

retriever B 점수: 3점
가동기간의 24시간 교대근무와 휴지기간의 주간 근무(유지보수 중심)가 모두 명시되어 있어 답변에 필요한 정보가 완전히 포함됨. 

============================================================
### query:
극저온시스템 과업지시서에 따르면, 가동기간 중 교대근무 인력이 개별적으로 얻은 운전 경험을 통합하여 정리하기 어려운 이유는 무엇이며, 이 업무는 주로 언제 수행되어야 합니까?

### answer:
교대근무로 인해 각 운전 인력마다 경험하는 운전 상황이 서로 달라, 모든 인원이 함께 모여 운전 경험을 통합하고 정리할 여건이 형성되기 어렵기 때문입니다. 이러한 운전 경험 통합 및 정리(문서화) 업무는 주로 주간 근무가 이루어지는 '휴지기간'에 집중적으로 수행됩니다.

---

### retriever A:
중이온가속기 또는 극저온시스템 가동기간에는 24시간 운전 업무를 필요로 한다. 통상적으
로 가동기간에는 24시간 운전을 위한 교대근무가 이루어지며, i) 극저온시스템의 모니터링
및 제어, ii) 극저온시스템의 주요 수치를 기록 및 분석하는 업무가 주로 이루어진다.
다만, 통상적으로 이러한 교대근무로 인해 각 운전 인력마다 경험할 수 있는 운전 상황이
서로 다르게 되고, 서로 다른 경험을 했던 운전 인력 모두가 참여하여 각자의 iii) 운전 경험
을 통합/정리할 수 있는 여건이 형성될 수 없다는 한계가 존재한다. 이러한 운전 경험의 통
합 정리는 후술하는 휴지기간에의 운전업무에 집중적으로 수행 가능하다.
통상적으로 휴지기간에는 극저온시스템의 유지보수 등이 진행되며, 유지보수 등을 위한 제
어, 유지보수 등의 경과 및 결과 확인을 위한 극저온시스템 모니터링, 유지보수 등의 결과
확인을 위한 주요 수치 기록 및 분석 또한 필수적인 운전 업무이다. 휴지기간에는 통상적으
로 주간 근무로 운전이 이루어진다. 이와 같이 주간 근무가 이루어질 수 있는 휴지기간의
특성 상, 휴지기간은 가동기간에서 교대근무로 각 운전 인력 개인이 얻은 단편적인 운전 경
험을 통합하여 정리(문서화)할 수 있는 여건이 형성될 수 있는 기간이다.
결과적으로 휴지기간에는 i) 중이온가속기용 극저온시스템의 운전 경험을 통합하여 정리(문
서화)하는 업무가 집중적으로 이루어지며, ii) 유지보수 등을 위한 극저온시스템의 제어와 iii)
유지보수 등을 위한 극저온시스템의 주요 수치 기록 및 분석이 병행되어야 한다.
3.2 운전 대상 및 장소


---

### retriever B:
3. 과업 내용
3.1 중이온가속기용 극저온시스템 운전 업무의 정의 및 필요성
본 과업에서 사용하는 “중이온가속기용 극저온시스템 운전 업무”의 정의는, “중이온가
속기용 극저온시스템이 안전하고 안정적으로 운영(가동기간뿐만 아니라 휴지기간의 운영 포
함)될 수 있도록 i) 극저온시스템을 모니터링 및 제어하고, ii) 극저온시스템의 주요 수치를
기록 및 분석하며, iii) 중이온가속기용 극저온시스템의 운전 경험을 통합하여 정리(문서화)하
는 업무를 의미한다.
중이온가속기 또는 극저온시스템 가동기간에는 24시간 운전 업무를 필요로 한다. 통상적으
로 가동기간에는 24시간 운전을 위한 교대근무가 이루어지며, i) 극저온시스템의 모니터링
및 제어, ii) 극저온시스템의 주요 수치를 기록 및 분석하는 업무가 주로 이루어진다.
다만, 통상적으로 이러한 교대근무로 인해 각 운전 인력마다 경험할 수 있는 운전 상황이
서로 다르게 되고, 서로 다른 경험을 했던 운전 인력 모두가 참여하여 각자의 iii) 운전 경험
을 통합/정리할 수 있는 여건이 형성될 수 없다는 한계가 존재한다. 이러한 운전 경험의 통
합 정리는 후술하는 휴지기간에의 운전업무에 집중적으로 수행 가능하다.
통상적으로 휴지기간에는 극저온시스템의 유지보수 등이 진행되며, 유지보수 등을 위한 제
어, 유지보수 등의 경과 및 결과 확인을 위한 극저온시스템 모니터링, 유지보수 등의 결과
확인을 위한 주요 수치 기록 및 분석 또한 필수적인 운전 업무이다. 휴지기간에는 통상적으
로 주간 근무로 운전이 이루어진다. 이와 같이 주간 근무가 이루어질 수 있는 휴지기간의


---

### openai answer
retriever A 점수: 3점
교대근무로 각 인력의 경험이 달라 모두가 함께 통합·정리할 여건이 없다는 이유와, 통합·정리(문서화) 업무가 휴지기간(주간 근무)에 집중 수행되어야 함이 명시됨.

retriever B 점수: 3점
교대근무로 인한 경험 편차로 통합·정리가 어렵다는 이유와, 해당 업무를 휴지기간(주간 근무)에 집중 수행해야 한다는 내용이 모두 포함됨. 

============================================================
### query:
계약업체가 매월 운전 현황을 정리해서 연구원에 보고해야 하는 공식적인 문서는 무엇이며, 제출 시점은 언제입니까?

### answer:
매월 운전 상황을 요약하여 제출해야 하는 문서는 '월간보고서'이며, 매월 기성 비용을 신청할 때 제출해야 합니다.

---

### retriever A:
계약종료 및 인수인계에 관한 사항
세부내용
○ 수행업체는 본 사업 종료 1개월 전 시스템의 발전 및 안정성을 유지하기 위한 인수인계 계획을 제시하여야 함
- 본 사업수행 기간 동안 취득한 모든 정보 및 생상된 산출물 일체
- 인수인계의 주요 업무, 상세 일정 및 인수인계 대상 주요 산출물 제시
- 인수인계 기간 동안 발생하는 리스크에 대한 방지대책 제시
○ 수행업체는 계약기간 만료 시 차기 수행업체로 선정된 사업자가 본 사업의 업무를 차질없이 수행할 수 있도록 요청 시 통합접수시스템 운영 업무 수행에 필요한 중요사항 등을 인계하여야 하며, 시스템의 효율적인 운영을 위해 하자담보 책임기간 중에는 기술자문에 응하여야 함
- 시스템 구성도, 시스템 현황, DB권한 등
- 통합접수시스템 기동/종료 절차
- 타 시스템 연계 및 적재 절차
- 일반 사용자 및 관리자 매뉴얼 등
○ 인수 및 인계 기간은 차기 시스템 운영 사업 개시일로부터 30일 이내로 하며, 인수 및 인계기간 동안의 용역비용은 지급하지 아니함
산출 정보
인수인계 계획서(사업수행계획서 내 포함 가능)
관련 요구사항
요구사항 출처
요구사항 분류
프로젝트지원 요구사항
요구사항 고유번호
PSR-008
요구사항 명칭
정보화사업 주요점검 및 수행지원
요구사항
상세설명
정의
정보화사업 주요점검 및 수행지원
세부내용
○ 통합접수시스템 관련 점검사항 준수 및 현장점검 수행
○ 보안관리 실태 점검 대응
○ 공공데이터 품질관리(DB) 수준평가 대응 및 데이터 오류 정제 활동 실시
산출 정보
점검별 결과보고서
관련 요구사항
요구사항 출처
□ 품질관리 요구사항
요구사항 분류


---

### retriever B:
4. 제출문서
계약자는 표 13.와 같은 문서들을 제출일정에 맞춰 연구원에 제출해야 한다.
표 13. 제출문서 목록 및 관련 사항
연구원
제출문서 내용 비고 및 제출일
승인
투입 인력
- 투입 인력 이력 사항 X - 계약 시
이력서
- 최초: 계약 후 1개월 내
작업계획서 - 일별 투입 인력 - 이후: 매주 금요일까지 차주
X
(주간근무표) - 휴가상황 근무표 제출
- 문서 파일 제출
- IBS에서 제공한 양식 파일
극저온시스템 - 매 월간 기성 신청 시 제출
주요 수치 - 극저온시스템 주요 수치 X - 문서 파일 제출
기록문서 - 수치 기록을 위해 계약자가
준비한 PC에 상시 저장
일일근무일지 - 일일 운영 상황 기록 O - 매 월간 기성 신청 시 제출
- 문서 파일 제출
- 수치 기록을 위해 계약자가
인수인계 문서 - (가동기간) 인수인계 내용 X 준비한 PC에 상시 저장
- 수리 사유 발생일
- 매 월간 기성 신청 시 제출
- 수리 필요 대상
수리필요사항 - 문서 파일 제출
- 상세 내용 O
기록 문서 - 수치 기록을 위해 계약자가
- 발견자
준비한 PC에 상시 저장
- 기타 필요한 내용
- 발생한 경우, 기성 신청 시
특이사항, 위험상황 발생 시,
제출 (해당 월 발생 건)
- 상황 내용
- 문서 파일 제출
- 작성자
Lesson learn O - 운전 인력 전원 회람 후 확
- 금회 대처 방법
인 서명 필수
- 향후 대처 방법
- 수치 기록을 위해 계약자가
- 기타 필요한 내용
준비한 PC에 상시 저장
- 매 월간 기성 신청 시 제출
월간보고서 - 월간 운전 상황 요약 O
- 문서 파일 제출


---

### openai answer
retriever A 점수: 0점
질문과 무관한 인수인계·점검 내용만 포함되어 있으며, 월간보고서나 기성 신청 시 제출 관련 정보가 없음.

retriever B 점수: 3점
제출문서 목록에 ‘월간보고서(월간 운전 상황 요약)’가 명시되어 있고, 비고/제출일에 ‘매 월간 기성 신청 시 제출’로 제출 시점이 제시됨. 

## size 2000

============================================================
### query:
본 과업지시서에서 '극저온시스템'은 무엇을 포함하는 것으로 정의하고 있습니까?

### answer:
'극저온시스템'은 SCL3/SCL2 극저온플랜트 및 부대설비, SCL3/SCL2 분배시스템 및 부대설비, SCL3/SCL2 초전도가속관 저온유지모듈, 기체헬륨 탱크, 액체질소 시스템 및 극저온시스템용 헬륨회수시스템을 모두 포함하는 것으로 정의됩니다.

---

### retriever A:
7. 안전관리 .............................................................................................................. 19
8. 붙임 자료 ........................................................................................................... 19
문서번호 : -
2025년도 중이온가속기용 극저온시스템
개정번호 : 0
운전 용역 과업지시서 발 행 일 : 2024. 10. 30
페 이 지 : 5/19
1. 서 론
1.1 목 적
기초과학연구원 중이온가속기연구소(이하 “연구원”)에서 운영중인 중이온가속기용 극저
온시스템(이하 “극저온시스템”)은 고도의 공정 시스템으로, 안전하고 안정적인 중이온가속
기의 운영을 위해 상시 모니터링이 필요하다. 이에 본 과업지시서는 극저온시스템의 운전
용역에 대한 상세 과업 내용을 기술한다.
1.2 용어 설명
본 과업지시서에서 사용하는 용어에 대한 상세 내용은 표 1.과 같다. 계약자는 반드시 해당
용어의 의미를 충분히 이해하고 숙지해야 하며, 용어 미숙지로 인한 책임은 계약자에게 있
다.
표 1. 본 과업지시서에 사용되는 용어 설명
용어 설명
연구원 중이온가속기연구소
계약자 본 과업을 수행할 계약상대자
SCL3/SCL2 극저온플랜트 및 부대설비,
SCL3/SCL2 분배시스템 및 부대설비,
SCL3/SCL2 초전도가속관 저온유지모듈,
극저온시스템
기체헬륨 탱크, 액체질소 시스템 및
TBx 극저온설비동
밸브박스, 초전도가속관 저온유지
SCL3 터널/
극저온
SCL3 LINAC 모듈 및 밸브박스 진공단열용 진
ISOL 구간
분배
공펌프
시스템 밸브박스, 초전도가속관 저온유지
SCL2 터널/
SCL2 LINAC 모듈 및 밸브박스 진공단열용 진
IF 구간
공펌프
외부 정화기 외부
회수 압축시스템 헬륨압축기동
회수
가스백 헬륨압축기동
시스템
헬륨 탱크 외부 기체 헬륨량 관리 포함
액체 질소 시스템 외부 액체 질소량 관리 포함
3.3 중이온가속기 또는 극저온시스템 가동기간의 운전 업무
계약자는 중이온가속기 또는 극저온시스템 가동기간 동안 24시간 운전업무가 이루어질 수
있도록 운전 인력을 제공해야 한다. 24시간 운전 업무는 2인 1조를 기본으로 하며, 주간에
는 과업 및 운전 인력 감독자가 1인 이상 운전실에 존재해야 한다. 계약자가 가동기간에 수
행해야 하는 운전 업무 상세 내용은 하위 절에 기술된 내용을 따른다. 본 과업지시서에 기
술하지 않은 가동기간의 운전 업무는 연구원과 계약자가 협의하여 수행 여부를 결정한다.
3.3.1 극저온시스템 모니터링
계약자는 가동기간 동안 극저온시스템이 안정적으로 가동될 수 있도록 모니터링을 해야 한
다. 가동기간의 모니터링 업무는 극저온시스템이 안정적으로 가동되고 있는지 여부를 확인
하기 위해 극저온시스템을 지속적으로 관찰하는 업무를 의미하며, 표 4.에 가동기간의 모니
터링 업무에 해당되는 내용들을 기술하였다.
극저온시스템의 모니터링은 연구원에 구축되어있는 극저온시스템용 제어시스템(이하 “제
문서번호 : -
2025년도 중이온가속기용 극저온시스템
개정번호 : 0
운전 용역 과업지시서 발 행 일 : 2024. 10. 30
페 이 지 : 12/19
매뉴얼을 작성하고 개정하는 업무를 수행해야 한다. 또한, 과업 및 운전 인력 감독자가 1인
이상 운전실에 배치되어 상기 업무들을 감독해야 한다. 계약자가 휴지기간에 수행해야 하는
운전 업무 상세 내용은 하위 절에 기술된 내용을 따른다. 본 과업지시서에 기술하지 않은
휴지기간의 운전 업무는 연구원과 계약자가 협의하여 수행 여부를 결정한다.
3.4.1 유지보수 등을 위한 극저온시스템 제어
극저온시스템의 제어는 극저온시스템이 휴지지간에 안전하고 안정적으로 운영될 수 있도록
극저온시스템 또는 극저온시스템 관련 부품들을 제어시스템을 활용하여 또는 현장에서 손으
로 조작하는 행위를 의미한다. 계약자는 극저온시스템이 본 목적에 부합하도록 극저온시스
템의 제어를 수행해야 한다. 표 8.에 극저온시스템 제어 업무에 해당되는 사항들을 기술하
였다.
휴지기간의 극저온시스템 제어는 i) 극저온시스템에 대한 충분한 이해를 바탕으로 ii) 조작
후 유지보수 등 현장 활동에 미치는 영향을 충분히 검토한 뒤 이루어져야 한다. 더불어, iii)
조작 대상물이 위치하고 있는 현장에 인력이 존재하는지 여부를 확인하고, iv) 현장에 인력
이 존재할 경우, 조작 대상물을 조작했을 때 발생할 수 있는 상황에 대해 해당 인력의 안전
이 확보된 뒤에 조작을 수행해야 한다. 마지막으로 v) 조작 대상/조작 상태에 대한 혼동이
없도록 각별히 주의를 하여 제어가 진행되어야 한다.


---

### retriever B:
1. 서 론
1.1 목 적
기초과학연구원 중이온가속기연구소(이하 “연구원”)에서 운영중인 중이온가속기용 극저
온시스템(이하 “극저온시스템”)은 고도의 공정 시스템으로, 안전하고 안정적인 중이온가속
기의 운영을 위해 상시 모니터링이 필요하다. 이에 본 과업지시서는 극저온시스템의 운전
용역에 대한 상세 과업 내용을 기술한다.
1.2 용어 설명
본 과업지시서에서 사용하는 용어에 대한 상세 내용은 표 1.과 같다. 계약자는 반드시 해당
용어의 의미를 충분히 이해하고 숙지해야 하며, 용어 미숙지로 인한 책임은 계약자에게 있
다.
표 1. 본 과업지시서에 사용되는 용어 설명
용어 설명
연구원 중이온가속기연구소
계약자 본 과업을 수행할 계약상대자
SCL3/SCL2 극저온플랜트 및 부대설비,
SCL3/SCL2 분배시스템 및 부대설비,
SCL3/SCL2 초전도가속관 저온유지모듈,
극저온시스템
기체헬륨 탱크, 액체질소 시스템 및
극저온시스템용 헬륨회수시스템 포함
(“3.2절 운전대상”, 표 3. 참고)
중이온가속기 또는 극저온시스템이 냉각(냉각 착수
가동기간
이후) 또는 빔서비스 등을 위해 가동 중인 기간
중이온가속기 또는 극저온시스템이 유지보수 등을
휴지기간
위해 가동을 멈추고 휴지 중인 기간
중이온가속기용 극저온시스템이 가동기간 및 휴지
기간에 안전하고 안정적으로 운영될 수 있도록,
중이온가속기용 극저온시스템 i) 극저온시스템을 모니터링 및 제어하고,
운전 업무 ii) 극저온시스템의 주요 수치를 기록 및 분석하며,
iii) 중이온가속기용 극저온시스템의 운전 경험을
통합하여 정리(문서화)하는 업무
문서번호 : -
문서번호 : -
2025년도 중이온가속기용 극저온시스템
개정번호 : 0
운전 용역 과업지시서 발 행 일 : 2024. 10. 30
페 이 지 : 12/19
매뉴얼을 작성하고 개정하는 업무를 수행해야 한다. 또한, 과업 및 운전 인력 감독자가 1인
이상 운전실에 배치되어 상기 업무들을 감독해야 한다. 계약자가 휴지기간에 수행해야 하는
운전 업무 상세 내용은 하위 절에 기술된 내용을 따른다. 본 과업지시서에 기술하지 않은
휴지기간의 운전 업무는 연구원과 계약자가 협의하여 수행 여부를 결정한다.
3.4.1 유지보수 등을 위한 극저온시스템 제어
극저온시스템의 제어는 극저온시스템이 휴지지간에 안전하고 안정적으로 운영될 수 있도록
극저온시스템 또는 극저온시스템 관련 부품들을 제어시스템을 활용하여 또는 현장에서 손으
로 조작하는 행위를 의미한다. 계약자는 극저온시스템이 본 목적에 부합하도록 극저온시스
템의 제어를 수행해야 한다. 표 8.에 극저온시스템 제어 업무에 해당되는 사항들을 기술하
였다.
휴지기간의 극저온시스템 제어는 i) 극저온시스템에 대한 충분한 이해를 바탕으로 ii) 조작
후 유지보수 등 현장 활동에 미치는 영향을 충분히 검토한 뒤 이루어져야 한다. 더불어, iii)
조작 대상물이 위치하고 있는 현장에 인력이 존재하는지 여부를 확인하고, iv) 현장에 인력
이 존재할 경우, 조작 대상물을 조작했을 때 발생할 수 있는 상황에 대해 해당 인력의 안전
이 확보된 뒤에 조작을 수행해야 한다. 마지막으로 v) 조작 대상/조작 상태에 대한 혼동이
없도록 각별히 주의를 하여 제어가 진행되어야 한다.
3. 과업 내용
3.1 중이온가속기용 극저온시스템 운전 업무의 정의 및 필요성
본 과업에서 사용하는 “중이온가속기용 극저온시스템 운전 업무”의 정의는, “중이온가
속기용 극저온시스템이 안전하고 안정적으로 운영(가동기간뿐만 아니라 휴지기간의 운영 포
함)될 수 있도록 i) 극저온시스템을 모니터링 및 제어하고, ii) 극저온시스템의 주요 수치를
기록 및 분석하며, iii) 중이온가속기용 극저온시스템의 운전 경험을 통합하여 정리(문서화)하
는 업무를 의미한다.
중이온가속기 또는 극저온시스템 가동기간에는 24시간 운전 업무를 필요로 한다. 통상적으
로 가동기간에는 24시간 운전을 위한 교대근무가 이루어지며, i) 극저온시스템의 모니터링
및 제어, ii) 극저온시스템의 주요 수치를 기록 및 분석하는 업무가 주로 이루어진다.
다만, 통상적으로 이러한 교대근무로 인해 각 운전 인력마다 경험할 수 있는 운전 상황이
서로 다르게 되고, 서로 다른 경험을 했던 운전 인력 모두가 참여하여 각자의 iii) 운전 경험
을 통합/정리할 수 있는 여건이 형성될 수 없다는 한계가 존재한다. 이러한 운전 경험의 통
합 정리는 후술하는 휴지기간에의 운전업무에 집중적으로 수행 가능하다.
통상적으로 휴지기간에는 극저온시스템의 유지보수 등이 진행되며, 유지보수 등을 위한 제
어, 유지보수 등의 경과 및 결과 확인을 위한 극저온시스템 모니터링, 유지보수 등의 결과
확인을 위한 주요 수치 기록 및 분석 또한 필수적인 운전 업무이다. 휴지기간에는 통상적으
로 주간 근무로 운전이 이루어진다. 이와 같이 주간 근무가 이루어질 수 있는 휴지기간의


---

### openai answer
retriever A 점수: 2점
(표 1의 ‘극저온시스템’에 SCL3/SCL2 극저온플랜트·분배시스템·초전도가속관 저온유지모듈, 기체헬륨 탱크, 액체질소 시스템 등 핵심 구성요소가 포함되어 있으나, ‘극저온시스템용 헬륨회수시스템 포함’이 명시적으로 확인되지 않음)

retriever B 점수: 3점
(표 1의 ‘극저온시스템’ 정의에 SCL3/SCL2 극저온플랜트·분배시스템·초전도가속관 저온유지모듈, 기체헬륨 탱크, 액체질소 시스템 및 ‘극저온시스템용 헬륨회수시스템 포함’까지 모두 명시되어 있음) 

============================================================
### query:
극저온시스템 과업 추진 일정에 따르면, 4분기에 SCL3 구간에 대해 예정된 주요 업무는 무엇이며, 이때의 업무 기간은 '가동기간'과 '휴지기간' 중 어느 것에 해당합니까?

### answer:
2025년 4분기에 SCL3 구간에 예정된 주요 업무는 '유지보수'이며, 이 기간은 '휴지기간'에 해당합니다.

---

### retriever A:
다. 계약자는 본 과업을 수행하기 위해 적절한 지식 또는 능력을 보유하고 있는 적정 수의 운
전 인력을 연구원에게 제공하여야 한다.
• 가동기간의 극저온시스템 운전 업무 (24시간 운전)
- 극저온시스템 모니터링
- 극저온시스템 주요 수치 기록 및 분석
- 극저온시스템 제어
- 기타 가동기간에 필요한 운전 업무
• 휴지기간의 극저온시스템 운전 업무 (주간 운전)
- 유지보수 등을 위한 극저온시스템 제어
- 유지보수 등을 위한 극저온시스템 모니터링
- 유지보수 등을 위한 극저온시스템 주요 수치 기록 및 분석
- 극저온시스템 운전 매뉴얼 작성 및 개정
- 극저온시스템 운전 교육
- 성과보고회 개최
- 기타 휴지기간에 필요한 운전 업무
• 본 과업 및 본 과업을 위해 투입된 인력에 대한 관리감독
본 과업의 기간은 2025년 1월 1일부터 2025년 12월 31일까지로 하며, 2025년 동안 가동기
간의 운전 업무를 수행하는 것을 기준으로 한다. 단, 표 2.와 같이 연구원의 운영 추진 상황
에 따라 휴지기간의 운전업무를 수행하는 것으로 변경될 수 있다. 가동기간과 휴지기간의
운전 업무는 “3. 과업 내용” 하위 절에 기술한 내용에 따른다.
표 2. 과업 추진 일정(안)
2025년
구분
1분기 2분기 3분기 4분기
SCL3 극저온플랜트
SCL3 중이온가속기 SCL3 구간 운전 SCL3 구간 유지보수
시운전 준비 및
구간 (가동기간) (휴지기간)
시운전 (가동기간)
SCL2 SCL2 극저온플랜트 중이온가속기 SCL2 구간 시운전 SCL2 구간 유지보수
구간 시운전 (가동기간) (가동기간) (휴지기간)
회수
2025년
구분
1분기 2분기 3분기 4분기
SCL3 극저온플랜트
SCL3 중이온가속기 SCL3 구간 운전 SCL3 구간 유지보수
시운전 준비 및
구간 (가동기간) (휴지기간)
시운전 (가동기간)
SCL2 SCL2 극저온플랜트 중이온가속기 SCL2 구간 시운전 SCL2 구간 유지보수
구간 시운전 (가동기간) (가동기간) (휴지기간)
회수
필요시 간헐적 운전 (휴지기간에 준함)
시스템
문서번호 : -
2025년도 중이온가속기용 극저온시스템
개정번호 : 0
운전 용역 과업지시서 발 행 일 : 2024. 10. 30
페 이 지 : 7/19
3. 과업 내용
3.1 중이온가속기용 극저온시스템 운전 업무의 정의 및 필요성
본 과업에서 사용하는 “중이온가속기용 극저온시스템 운전 업무”의 정의는, “중이온가
속기용 극저온시스템이 안전하고 안정적으로 운영(가동기간뿐만 아니라 휴지기간의 운영 포
함)될 수 있도록 i) 극저온시스템을 모니터링 및 제어하고, ii) 극저온시스템의 주요 수치를
기록 및 분석하며, iii) 중이온가속기용 극저온시스템의 운전 경험을 통합하여 정리(문서화)하
는 업무를 의미한다.
중이온가속기 또는 극저온시스템 가동기간에는 24시간 운전 업무를 필요로 한다. 통상적으
로 가동기간에는 24시간 운전을 위한 교대근무가 이루어지며, i) 극저온시스템의 모니터링
및 제어, ii) 극저온시스템의 주요 수치를 기록 및 분석하는 업무가 주로 이루어진다.
다만, 통상적으로 이러한 교대근무로 인해 각 운전 인력마다 경험할 수 있는 운전 상황이
□ (업무 자동화) 미래기술 혁신 및 디지털 대전환 가속화를 지원하여 일하는 방식의 혁신을 통한 업무 환경 개선 추진
□ (중장기 비전 부합)「2023 디지털전환 실행과제」에서 도출된 시스템 설계를 반영하여 추후 구축되는 시스템과 호환성 확보
3. 추진 체계
□ 조직도
정보관리총괄
경영본부장
사업총괄
노사⦁경영지원실장
감리
프로젝트 관리자(PM)
협조부서
감리 사업자
디지털기획팀 팀장
경영지원팀,
안전기획팀 등
구축용역 사업자
추진부서
협력업체
시스템 구축
디지털기획팀
시스템 유지보수사업자
□ 인력구성 및 역할
구 분
주요 역할
경영본부장
- 정보화업무 총괄
노사⦁경영지원실장
- 사업 관리 총괄
디지털기획팀
- 사업 운영관리
사업 수행
추진부서
- 사업관리 실무 총괄 (계약관리, 인력 및 일정관리, 행정업무 등)
- 사업 진행사항 관리‧감독 및 중요 의사결정 사항 보고
- 사업추진 관련 유관부서 및 대외기관 협업 주관
구축용역 사업자
- 과업내용에 대한 사업 수행 (시스템 분석, 설계, 개발, 이행 등)
- 품질보증 활동 및 사업 산출물 제출
- 운영자‧사용자 교육 및 기술이전, 하자보수 등
- S/W 개발 및 H/W 환경 구축
사업 지원
협조부서
- 요구사항 도출 및 상세 업무요건 정의
- 업무 프로세스 설계 지원 및 검증
- 업무별 단위테스트, 통합테스트 등 수행 지원
감리 사업자
- 단계별 정보시스템 감리 수행
4. 추진일정
□ 사업기간 : 계약체결일로부터 6개월 (안정화 1개월 포함)
분석·설계
구현
오픈 및 안정화
구 분
M
1
M
2
M
3
M
4
M
5
M
6
구축
사업
시스템 구축


---

### retriever B:
다. 계약자는 본 과업을 수행하기 위해 적절한 지식 또는 능력을 보유하고 있는 적정 수의 운
전 인력을 연구원에게 제공하여야 한다.
• 가동기간의 극저온시스템 운전 업무 (24시간 운전)
- 극저온시스템 모니터링
- 극저온시스템 주요 수치 기록 및 분석
- 극저온시스템 제어
- 기타 가동기간에 필요한 운전 업무
• 휴지기간의 극저온시스템 운전 업무 (주간 운전)
- 유지보수 등을 위한 극저온시스템 제어
- 유지보수 등을 위한 극저온시스템 모니터링
- 유지보수 등을 위한 극저온시스템 주요 수치 기록 및 분석
- 극저온시스템 운전 매뉴얼 작성 및 개정
- 극저온시스템 운전 교육
- 성과보고회 개최
- 기타 휴지기간에 필요한 운전 업무
• 본 과업 및 본 과업을 위해 투입된 인력에 대한 관리감독
본 과업의 기간은 2025년 1월 1일부터 2025년 12월 31일까지로 하며, 2025년 동안 가동기
간의 운전 업무를 수행하는 것을 기준으로 한다. 단, 표 2.와 같이 연구원의 운영 추진 상황
에 따라 휴지기간의 운전업무를 수행하는 것으로 변경될 수 있다. 가동기간과 휴지기간의
운전 업무는 “3. 과업 내용” 하위 절에 기술한 내용에 따른다.
표 2. 과업 추진 일정(안)
2025년
구분
1분기 2분기 3분기 4분기
SCL3 극저온플랜트
SCL3 중이온가속기 SCL3 구간 운전 SCL3 구간 유지보수
시운전 준비 및
구간 (가동기간) (휴지기간)
시운전 (가동기간)
SCL2 SCL2 극저온플랜트 중이온가속기 SCL2 구간 시운전 SCL2 구간 유지보수
구간 시운전 (가동기간) (가동기간) (휴지기간)
회수
일정
구분
추진 일정
M
M+1
M+2
M+3
M+4
사업발주 및 사업자 선정
업무 협의
시스템
구축
기존 시스템 분석
업무(요구사항) 분석
설계 및 구축
시스템 구현 및 개발
시스템테스트 및 수정
시스템 가동 및 운영(안정화)
※ 위 추진일정은 사업 추진과정에서 조정 가능함
Ⅲ
다. 추진일정
구 분
2024년~2025년
M1
M2
M3
M4
M5
M6
◦ 시스템 개발
- 분석
- 설계/구현
- 단위/통합 시험
◦ 연계시스템 기능개선
◦ 운영개시 및 안정화
◦ 보고회
- 착수, 중간, 최종보고
★
★
★
* 추진과정에서 일정은 변경될 수 있음
Ⅱ
일반 현황
2. 추진체계
사업총괄기관
안양시
실무지원
사업주관
실무지원
도시공사(운영관리 부서)
(현장 운영지원)
안양시 체육과, 도시공사
도시공사(정보화 부서)
(ICT 기술지원)
설치,구축 수행
계약상대자
구분
주요역할 및 기능
구성
사업주관
○ 사업총괄 추진
○ 시스템 관련 요구사항 정의 및 조정
○ 사업수행 관리, 검수 및 결과평가
실무지원
○ 요구사항 정의 및 검토 지원
○ 업무관련 자료 제공
○ 관련 구축내용 검수 및 평가
○ 관련 업무 운영
·운영관리 부서
·정보화 부서
설치,구축수행
○ 기획/분석 ○ 시스템 구축/개발
○ 테스트 수행 및 하자보수 ○ 교육 및 기술 지원 등 기타 과제수행에 필요한 사항
·계약상대자
楴䵴3. 추진일정
 계약일로부터 120일간
추진내용
M+0
M+1
M+2
M+3
착수
프로젝트 착수
분석
4. 추진 일정
구 분
착수일로부터 5개월
M0
M1
M2
M3
M4
M5
사업자 선정 및
계약체결
시스템 분석 및 설계
통합정보시스템 구축(S/W 개발)
단위 / 통합테스트
시스템 오픈 및 운영지원
사업 관리
품질관리/지원
착수
보고
정례
보고
중간
보고
정례
보고
완료
보고
* 추진일정은 발주기관과 사업수행사 간 상호협의를 통해 변경 가능
4. 추진 일정
구 분
착수일로부터 5개월
M0
M1
M2
M3
M4
M5
사업자 선정 및
계약체결
시스템 분석 및 설계
통합정보시스템 구축(S/W 개발)
단위 / 통합테스트
시스템 오픈 및 운영지원
사업 관리
품질관리/지원
착수
보고
정례
보고
중간
보고
정례
보고
완료
보고
* 추진일정은 발주기관과 사업수행사 간 상호협의를 통해 변경 가능
4. 추진일정
□ 사업기간 : 계약체결일로부터 6개월 (안정화 1개월 포함)
분석·설계
구현
오픈 및 안정화
구 분
M
1
M
2
M
3
M
4
M
5
M
6
구축
사업
시스템 구축
▸ 그룹웨어 시스템 구축
▸ 기록물관리 시스템 구축
▸ 시스템 연계 및 커스터마이징
기록물 이관 및 마이그레이션
▸ 기록물이관 및 마이그레이션
사내SNS 구축
▸ 사내SNS(별별얘기) 시스템 구축
▸ 메신저(내⦁외부) 시스템 구축
▸ 개발지원 / 모니터링 / 운영
SW
상용SW 직접구매
▸ 직접구매 세부계획 수립
▸ 계약 체결 (조달청 나라장터)
▸ 상용SW 설치, 시험 및 안정화
감리 등
감리
▸ 감리(3단계)
▲요구 정의
▲설계
▲종료
보고
정기 보고(착수, 중간, 최종)
★
★
★
※ 상기 추진 일정은 사업추진 과정에서 협의·조정 가능


---

### openai answer
retriever A 점수: 3점
표 2 ‘과업 추진 일정(안)’에 2025년 4분기 SCL3 구간이 ‘유지보수(휴지기간)’로 명시되어 있어, 질문의 두 요소(주요 업무와 기간 구분)를 모두 포함합니다.

retriever B 점수: 3점
동일하게 표 2에 2025년 4분기 SCL3 구간 ‘유지보수(휴지기간)’가 명시되어 있어, 답을 완전히 근거할 수 있습니다. 

============================================================
### query:
극저온시스템 '가동기간' 동안 계약자가 24시간 수행해야 할 극저온시스템 모니터링 업무에는 어떤 것들이 있습니까? (3가지 이상)

### answer:
가동기간 동안 수행해야 할 모니터링 업무는 다음과 같습니다.
알람 메시지 확인
주요 센서(온도, 압력, 유량 등)의 트렌드 확인
정상 운전 상황의 주요 센서 지시값 파악 및 숙지
비정상 운전 상황 감지
제어로직 정상 작동 여부 확인

---

### retriever A:
다. 가동기간의 모니터링 업무는 극저온시스템이 안정적으로 가동되고 있는지 여부를 확인
하기 위해 극저온시스템을 지속적으로 관찰하는 업무를 의미하며, 표 4.에 가동기간의 모니
터링 업무에 해당되는 내용들을 기술하였다.
극저온시스템의 모니터링은 연구원에 구축되어있는 극저온시스템용 제어시스템(이하 “제
어시스템”)을 활용하여 또는 현장에 설치되어있는 장치를 통해 수행한다. 연구원은 구축되
어있는 제어시스템을 활용하기 위한 컴퓨터, 모니터, 키보드, 마우스, 책상 및 의자를 계약
자에게 제공한다. 현재 연구원에 구축되어있는 제어시스템은 표 5.과 같다.
연구원은 계약자에게 현재 구축되어있는 제어시스템에 대한 기본 사용 교육을 제공하며,
계약자는 연구원이 제공하는 교육을 이해하기 위한 배경지식을 보유해야 한다.
문서번호 : -
2025년도 중이온가속기용 극저온시스템
개정번호 : 0
운전 용역 과업지시서 발 행 일 : 2024. 10. 30
페 이 지 : 9/19
표 4. 가동기간의 극저온시스템 모니터링 업무 리스트
순번 극저온시스템 모니터링 업무 비고
1 알람 메시지 확인
2 주요 센서 트렌드 확인 온도, 압력, 유량, 속도 등
3 정상 운전 상황의 주요 센서 지시값 파악 및 숙지 온도, 압력, 유량, 속도 등
4 비정상 운전 상황 감지
연구원에서 현장 확인 필요상황 발생 시, 연구원이 지 무전 통신 활용
5
정하는 값 관찰, 확인 및 상황 전달 (연구원에서 무전기 제공)
6 제어로직 정상 작동 여부 확인
연구원 요청 시,
7 현장 상황 일시적 또는 지속적 관찰
비방사선구역 한정
이온가속기용 극저온시스템 운영에 중대한 영향이 생길 경우, 이는 계약자의 책임으로 한
다.
표 8. 휴지기간의 극저온시스템 제어 업무 리스트
순번 극저온시스템 제어 업무 비고
자동/수동(Auto/Manual) 제어 우선
1 계장류 제어 우선권(Priority) 변경
권 설정 변경
문서번호 : -
2025년도 중이온가속기용 극저온시스템
개정번호 : 0
운전 용역 과업지시서 발 행 일 : 2024. 10. 30
페 이 지 : 13/19
자동 밸브 – 제어시스템 활용
2 극저온시스템 자동/수동 밸브 조작
수동 밸브 – 현장에서 손으로 조작
압축기 emergency stop 버튼
3 현장에서 손으로 조작
활성화/비활성화
제어시스템 활용
4 진공펌프 가동/정지
(현장 조작 필요 시 연구원에 통지)
5 센서 오버라이드(Override) 설정 또는 해제 제어시스템 활용
6 계장류 제어 변수값(Ramp speed 등) 변경 제어시스템 활용
7 시퀀스 제어 변수값(Timer 등) 변경 제어시스템 활용
8 시퀀스 시작/종료 제어시스템 활용
현장 확인 및 최종 준비 상태 미진
9 극저온시스템 가동 전 최종 준비 상태 확인
할 경우, 상태 보완 포함
기타 극저온시스템의 안전하고 안정적인 운
10 제어시스템 활용 및 현장 조작 포함
전에 필요한 제어
3.4.2 유지보수 등을 위한 극저온시스템 모니터링
계약자는 휴지기간 동안 극저온시스템이 안정적으로 가동될 수 있도록 모니터링을 해야 한
다. 휴지기간의 모니터링 업무는 i) 유지보수 등을 위한 현장 활동 시 극저온시스템에 특이
TBx 극저온설비동
밸브박스, 초전도가속관 저온유지
SCL3 터널/
극저온
SCL3 LINAC 모듈 및 밸브박스 진공단열용 진
ISOL 구간
분배
공펌프
시스템 밸브박스, 초전도가속관 저온유지
SCL2 터널/
SCL2 LINAC 모듈 및 밸브박스 진공단열용 진
IF 구간
공펌프
외부 정화기 외부
회수 압축시스템 헬륨압축기동
회수
가스백 헬륨압축기동
시스템
헬륨 탱크 외부 기체 헬륨량 관리 포함
액체 질소 시스템 외부 액체 질소량 관리 포함
3.3 중이온가속기 또는 극저온시스템 가동기간의 운전 업무
계약자는 중이온가속기 또는 극저온시스템 가동기간 동안 24시간 운전업무가 이루어질 수
있도록 운전 인력을 제공해야 한다. 24시간 운전 업무는 2인 1조를 기본으로 하며, 주간에
는 과업 및 운전 인력 감독자가 1인 이상 운전실에 존재해야 한다. 계약자가 가동기간에 수
행해야 하는 운전 업무 상세 내용은 하위 절에 기술된 내용을 따른다. 본 과업지시서에 기
술하지 않은 가동기간의 운전 업무는 연구원과 계약자가 협의하여 수행 여부를 결정한다.
3.3.1 극저온시스템 모니터링
계약자는 가동기간 동안 극저온시스템이 안정적으로 가동될 수 있도록 모니터링을 해야 한
다. 가동기간의 모니터링 업무는 극저온시스템이 안정적으로 가동되고 있는지 여부를 확인
하기 위해 극저온시스템을 지속적으로 관찰하는 업무를 의미하며, 표 4.에 가동기간의 모니
터링 업무에 해당되는 내용들을 기술하였다.
극저온시스템의 모니터링은 연구원에 구축되어있는 극저온시스템용 제어시스템(이하 “제


---

### retriever B:
다. 가동기간의 모니터링 업무는 극저온시스템이 안정적으로 가동되고 있는지 여부를 확인
하기 위해 극저온시스템을 지속적으로 관찰하는 업무를 의미하며, 표 4.에 가동기간의 모니
터링 업무에 해당되는 내용들을 기술하였다.
극저온시스템의 모니터링은 연구원에 구축되어있는 극저온시스템용 제어시스템(이하 “제
어시스템”)을 활용하여 또는 현장에 설치되어있는 장치를 통해 수행한다. 연구원은 구축되
어있는 제어시스템을 활용하기 위한 컴퓨터, 모니터, 키보드, 마우스, 책상 및 의자를 계약
자에게 제공한다. 현재 연구원에 구축되어있는 제어시스템은 표 5.과 같다.
연구원은 계약자에게 현재 구축되어있는 제어시스템에 대한 기본 사용 교육을 제공하며,
계약자는 연구원이 제공하는 교육을 이해하기 위한 배경지식을 보유해야 한다.
문서번호 : -
2025년도 중이온가속기용 극저온시스템
개정번호 : 0
운전 용역 과업지시서 발 행 일 : 2024. 10. 30
페 이 지 : 9/19
표 4. 가동기간의 극저온시스템 모니터링 업무 리스트
순번 극저온시스템 모니터링 업무 비고
1 알람 메시지 확인
2 주요 센서 트렌드 확인 온도, 압력, 유량, 속도 등
3 정상 운전 상황의 주요 센서 지시값 파악 및 숙지 온도, 압력, 유량, 속도 등
4 비정상 운전 상황 감지
연구원에서 현장 확인 필요상황 발생 시, 연구원이 지 무전 통신 활용
5
정하는 값 관찰, 확인 및 상황 전달 (연구원에서 무전기 제공)
6 제어로직 정상 작동 여부 확인
연구원 요청 시,
7 현장 상황 일시적 또는 지속적 관찰
비방사선구역 한정
는 과업 및 운전 인력 감독자가 1인 이상 운전실에 존재해야 한다. 계약자가 가동기간에 수
행해야 하는 운전 업무 상세 내용은 하위 절에 기술된 내용을 따른다. 본 과업지시서에 기
술하지 않은 가동기간의 운전 업무는 연구원과 계약자가 협의하여 수행 여부를 결정한다.
3.3.1 극저온시스템 모니터링
계약자는 가동기간 동안 극저온시스템이 안정적으로 가동될 수 있도록 모니터링을 해야 한
다. 휴지기간의 모니터링 업무는 i) 유지보수 등을 위한 현장 활동 시 극저온시스템에 특이
사항이 발생하는지 여부를 확인하기 위해 또는 ii) 유지보수 등을 수행한 뒤 그 결과를 확인
하기 위해 극저온시스템을 지속적으로 관찰하는 업무를 의미한다. 표 9.에 휴지기간의 모니
터링 업무에 해당되는 내용들을 기술하였다.
극저온시스템의 모니터링은 연구원에 구축되어있는 극저온시스템용 제어시스템(이하 “제
어시스템”)을 활용하여 또는 현장에 설치되어있는 장치를 통해 수행한다. 연구원은 구축되
어있는 제어시스템을 활용하기 위한 컴퓨터, 모니터, 키보드, 마우스, 책상 및 의자를 계약
자에게 제공한다. 현재 연구원에 구축되어있는 제어시스템은 표 5.과 같다.
연구원은 계약자에게 현재 구축되어있는 제어시스템에 대한 기본 사용 교육을 제공하며,
계약자는 연구원이 제공하는 교육을 이해하기 위한 배경지식을 보유해야 한다.
표 9. 휴지기간의 극저온시스템 모니터링 업무 리스트
순번 극저온시스템 모니터링 업무 비고
1 알람 메시지 확인
2 주요 센서 트렌드 확인 온도, 압력, 유량, 속도 등
3 정상 상황의 주요 센서 지시값 파악 및 숙지 온도, 압력, 유량, 속도 등
4 비정상 상황 감지
연구원에서 현장 확인 필요상황 발생 시, 연구원이 지 무전 통신 활용
5
정하는 값 관찰, 확인 및 상황 전달 (연구원에서 무전기 제공)
6 제어로직 정상 작동 여부 확인
문서번호 : -
2025년도 중이온가속기용 극저온시스템
개정번호 : 0
운전 용역 과업지시서 발 행 일 : 2024. 10. 30
페 이 지 : 14/19
이온가속기용 극저온시스템 운영에 중대한 영향이 생길 경우, 이는 계약자의 책임으로 한
다.
표 8. 휴지기간의 극저온시스템 제어 업무 리스트
순번 극저온시스템 제어 업무 비고
자동/수동(Auto/Manual) 제어 우선
1 계장류 제어 우선권(Priority) 변경
권 설정 변경
문서번호 : -
2025년도 중이온가속기용 극저온시스템
개정번호 : 0
운전 용역 과업지시서 발 행 일 : 2024. 10. 30
페 이 지 : 13/19
자동 밸브 – 제어시스템 활용
2 극저온시스템 자동/수동 밸브 조작
수동 밸브 – 현장에서 손으로 조작
압축기 emergency stop 버튼
3 현장에서 손으로 조작
활성화/비활성화
제어시스템 활용
4 진공펌프 가동/정지
(현장 조작 필요 시 연구원에 통지)
5 센서 오버라이드(Override) 설정 또는 해제 제어시스템 활용
6 계장류 제어 변수값(Ramp speed 등) 변경 제어시스템 활용
7 시퀀스 제어 변수값(Timer 등) 변경 제어시스템 활용
8 시퀀스 시작/종료 제어시스템 활용
현장 확인 및 최종 준비 상태 미진
9 극저온시스템 가동 전 최종 준비 상태 확인
할 경우, 상태 보완 포함
기타 극저온시스템의 안전하고 안정적인 운
10 제어시스템 활용 및 현장 조작 포함
전에 필요한 제어
3.4.2 유지보수 등을 위한 극저온시스템 모니터링
계약자는 휴지기간 동안 극저온시스템이 안정적으로 가동될 수 있도록 모니터링을 해야 한


---

### openai answer
retriever A 점수: 3점
(가동기간 표 4의 모니터링 업무로 알람 메시지 확인, 주요 센서 트렌드 확인(온도·압력·유량 등), 정상 운전 센서 지시값 파악·숙지, 비정상 운전 감지, 제어로직 정상 작동 여부 확인 등이 그대로 포함되어 있어 답변의 항목이 모두 들어있음)

retriever B 점수: 3점
(가동기간 표 4의 동일 모니터링 리스트를 포함하고 있어 알람 확인, 센서 트렌드 확인, 정상 지시값 숙지, 비정상 감지, 제어로직 확인 등 답변의 전 항목이 완전히 포함됨) 

============================================================
### query:
중이온가속기 극저온시스템의 '가동기간'과 '휴지기간'의 운전 업무 형태(근무 방식)는 어떻게 다릅니까?

### answer:
'가동기간'에는 시스템의 24시간 안정적인 운영을 위해 교대근무(24시간 운전)가 이루어집니다. 반면, '휴지기간'에는 주로 시스템 유지보수 등을 위해 주간 운전을 기본으로 합니다.

---

### retriever A:
중이온가속기 또는 극저온시스템 가동기간에는 24시간 운전 업무를 필요로 한다. 통상적으
로 가동기간에는 24시간 운전을 위한 교대근무가 이루어지며, i) 극저온시스템의 모니터링
및 제어, ii) 극저온시스템의 주요 수치를 기록 및 분석하는 업무가 주로 이루어진다.
다만, 통상적으로 이러한 교대근무로 인해 각 운전 인력마다 경험할 수 있는 운전 상황이
서로 다르게 되고, 서로 다른 경험을 했던 운전 인력 모두가 참여하여 각자의 iii) 운전 경험
을 통합/정리할 수 있는 여건이 형성될 수 없다는 한계가 존재한다. 이러한 운전 경험의 통
합 정리는 후술하는 휴지기간에의 운전업무에 집중적으로 수행 가능하다.
통상적으로 휴지기간에는 극저온시스템의 유지보수 등이 진행되며, 유지보수 등을 위한 제
어, 유지보수 등의 경과 및 결과 확인을 위한 극저온시스템 모니터링, 유지보수 등의 결과
확인을 위한 주요 수치 기록 및 분석 또한 필수적인 운전 업무이다. 휴지기간에는 통상적으
로 주간 근무로 운전이 이루어진다. 이와 같이 주간 근무가 이루어질 수 있는 휴지기간의
특성 상, 휴지기간은 가동기간에서 교대근무로 각 운전 인력 개인이 얻은 단편적인 운전 경
험을 통합하여 정리(문서화)할 수 있는 여건이 형성될 수 있는 기간이다.
결과적으로 휴지기간에는 i) 중이온가속기용 극저온시스템의 운전 경험을 통합하여 정리(문
서화)하는 업무가 집중적으로 이루어지며, ii) 유지보수 등을 위한 극저온시스템의 제어와 iii)
유지보수 등을 위한 극저온시스템의 주요 수치 기록 및 분석이 병행되어야 한다.
3.2 운전 대상 및 장소
문서번호 : -
2025년도 중이온가속기용 극저온시스템
개정번호 : 0
운전 용역 과업지시서 발 행 일 : 2024. 10. 30
페 이 지 : 12/19
매뉴얼을 작성하고 개정하는 업무를 수행해야 한다. 또한, 과업 및 운전 인력 감독자가 1인
이상 운전실에 배치되어 상기 업무들을 감독해야 한다. 계약자가 휴지기간에 수행해야 하는
운전 업무 상세 내용은 하위 절에 기술된 내용을 따른다. 본 과업지시서에 기술하지 않은
휴지기간의 운전 업무는 연구원과 계약자가 협의하여 수행 여부를 결정한다.
3.4.1 유지보수 등을 위한 극저온시스템 제어
극저온시스템의 제어는 극저온시스템이 휴지지간에 안전하고 안정적으로 운영될 수 있도록
극저온시스템 또는 극저온시스템 관련 부품들을 제어시스템을 활용하여 또는 현장에서 손으
로 조작하는 행위를 의미한다. 계약자는 극저온시스템이 본 목적에 부합하도록 극저온시스
템의 제어를 수행해야 한다. 표 8.에 극저온시스템 제어 업무에 해당되는 사항들을 기술하
였다.
휴지기간의 극저온시스템 제어는 i) 극저온시스템에 대한 충분한 이해를 바탕으로 ii) 조작
후 유지보수 등 현장 활동에 미치는 영향을 충분히 검토한 뒤 이루어져야 한다. 더불어, iii)
조작 대상물이 위치하고 있는 현장에 인력이 존재하는지 여부를 확인하고, iv) 현장에 인력
이 존재할 경우, 조작 대상물을 조작했을 때 발생할 수 있는 상황에 대해 해당 인력의 안전
이 확보된 뒤에 조작을 수행해야 한다. 마지막으로 v) 조작 대상/조작 상태에 대한 혼동이
없도록 각별히 주의를 하여 제어가 진행되어야 한다.
이온가속기용 극저온시스템 운영에 중대한 영향이 생길 경우, 계약자의 책임으로 한다.
표 6. 가동기간의 극저온시스템 제어 업무 리스트
순번 극저온시스템 제어 업무 비고
자동/수동(Auto/Manual) 제어 우선
1 계장류 제어 우선권(Priority) 변경
권 설정 변경
자동 밸브 – 제어시스템 활용
2 극저온시스템 자동/수동 밸브 조작
수동 밸브 – 현장에서 손으로 조작
3 히터 조작 제어시스템 활용
4 액체 헬륨 레벨미터 모니터 측정 방식 변경 현장에서 설정 변경
제어시스템 활용
5 진공펌프 가동/정지
(현장 조작 필요 시 연구원에 통지)
6 센서 오버라이드(Override) 설정 또는 해제 제어시스템 활용
7 계장류 제어 변수값(Ramp speed 등) 변경 제어시스템 활용
8 시퀀스 제어 변수값(Timer 등) 변경 제어시스템 활용
9 시퀀스 시작/종료 제어시스템 활용
기타 극저온시스템의 안전하고 안정적인 운
10 제어시스템 활용 및 현장 조작 포함
전에 필요한 제어
문서번호 : -
2025년도 중이온가속기용 극저온시스템
개정번호 : 0
운전 용역 과업지시서 발 행 일 : 2024. 10. 30
페 이 지 : 11/19
3.3.4 기타 가동기간에 필요한 운전 업무
표 7.에 기술한 기타 가동기간에 필요한 운전 업무 또한 계약자의 업무 범위이다. 표 7.에
기술된 내용 중, 위험(긴급)상황 발생 시, 계약자가 i) 지체없이 연구원에게 상황을 전파하고,
ii) 성실히 연구원에서 전달한 긴급조치 사항을 수행했음에도 불구하고 발생한 문제들은 계


---

### retriever B:
3. 과업 내용
3.1 중이온가속기용 극저온시스템 운전 업무의 정의 및 필요성
본 과업에서 사용하는 “중이온가속기용 극저온시스템 운전 업무”의 정의는, “중이온가
속기용 극저온시스템이 안전하고 안정적으로 운영(가동기간뿐만 아니라 휴지기간의 운영 포
함)될 수 있도록 i) 극저온시스템을 모니터링 및 제어하고, ii) 극저온시스템의 주요 수치를
기록 및 분석하며, iii) 중이온가속기용 극저온시스템의 운전 경험을 통합하여 정리(문서화)하
는 업무를 의미한다.
중이온가속기 또는 극저온시스템 가동기간에는 24시간 운전 업무를 필요로 한다. 통상적으
로 가동기간에는 24시간 운전을 위한 교대근무가 이루어지며, i) 극저온시스템의 모니터링
및 제어, ii) 극저온시스템의 주요 수치를 기록 및 분석하는 업무가 주로 이루어진다.
다만, 통상적으로 이러한 교대근무로 인해 각 운전 인력마다 경험할 수 있는 운전 상황이
서로 다르게 되고, 서로 다른 경험을 했던 운전 인력 모두가 참여하여 각자의 iii) 운전 경험
을 통합/정리할 수 있는 여건이 형성될 수 없다는 한계가 존재한다. 이러한 운전 경험의 통
합 정리는 후술하는 휴지기간에의 운전업무에 집중적으로 수행 가능하다.
통상적으로 휴지기간에는 극저온시스템의 유지보수 등이 진행되며, 유지보수 등을 위한 제
어, 유지보수 등의 경과 및 결과 확인을 위한 극저온시스템 모니터링, 유지보수 등의 결과
확인을 위한 주요 수치 기록 및 분석 또한 필수적인 운전 업무이다. 휴지기간에는 통상적으
로 주간 근무로 운전이 이루어진다. 이와 같이 주간 근무가 이루어질 수 있는 휴지기간의
제외한다.
표 3. 2025년 중이온가속기용 극저온시스템 운전 용역 대상 장치 목록
구분 장치명 장소 비고
극저온 극저온설비동/ 압축기, 재생시스템, 2 K 시스템
SCL3 극저온플랜트
플랜트 헬륨압축기동 등 부대시설 일체
문서번호 : -
2025년도 중이온가속기용 극저온시스템
개정번호 : 0
운전 용역 과업지시서 발 행 일 : 2024. 10. 30
페 이 지 : 8/19
극저온설비동/ 압축기, 재생시스템, 2 K 시스템
SCL2 극저온플랜트
헬륨압축기동 등 부대시설 일체
TBx 극저온설비동
밸브박스, 초전도가속관 저온유지
SCL3 터널/
극저온
SCL3 LINAC 모듈 및 밸브박스 진공단열용 진
ISOL 구간
분배
공펌프
시스템 밸브박스, 초전도가속관 저온유지
SCL2 터널/
SCL2 LINAC 모듈 및 밸브박스 진공단열용 진
IF 구간
공펌프
외부 정화기 외부
회수 압축시스템 헬륨압축기동
회수
가스백 헬륨압축기동
시스템
헬륨 탱크 외부 기체 헬륨량 관리 포함
액체 질소 시스템 외부 액체 질소량 관리 포함
3.3 중이온가속기 또는 극저온시스템 가동기간의 운전 업무
계약자는 중이온가속기 또는 극저온시스템 가동기간 동안 24시간 운전업무가 이루어질 수
있도록 운전 인력을 제공해야 한다. 24시간 운전 업무는 2인 1조를 기본으로 하며, 주간에
는 과업 및 운전 인력 감독자가 1인 이상 운전실에 존재해야 한다. 계약자가 가동기간에 수
행해야 하는 운전 업무 상세 내용은 하위 절에 기술된 내용을 따른다. 본 과업지시서에 기
술하지 않은 가동기간의 운전 업무는 연구원과 계약자가 협의하여 수행 여부를 결정한다.
온시스템 가동 기간의 운전 업무”와 “3.4 중이온가속기 또는 극저온시스템 휴지 기간의
운전 업무”를 따른다.
- 본 과업 및 인력에 대한 관리감독은 계약자가 담당하며, 관리감독자(현장대리인)과 운전
인력은 구분하여 인력을 구성한다. 단, 유사 시 현장대리인이 현장대리인의 업무에 지장을
주지 않는 선에서 운전 업무를 병행할 수 있다.
- 계약자의 i) 인력관리 소홀 또는 조작 실수 등으로 ii) 중이온가속기 운영에 중대한 문제가
발생한 경우, 이는 계약자의 책임으로 한다.
- 단, 연구원에서 계약자가 투입한 운전 인력이 운전 업무를 불성실히 수행하는 상황을 발
견한 경우, 연구원은 계약자에게 시정조치를 요구할 수 있다.
- 계약자는 연구원이 시정조치를 요구할 경우, 7일 이내에 연구원에게 조치 결과를 회신해
야 한다.
- 만약 계약자가 i) 연구원의 시정조치에도 불구하고 ii) 시정 없이 투입 인력이 운전 업무를
불성실하게 수행하여 iii) 중이온가속기 운영에 중대한 문제가 발생한 경우, 계약자의 책임으
로 한다.
- 위험(긴급) 상황 발생 시, 계약자는 연구원에게 지체없이 통보해야 하며, 연구원의 협조
요청에 적극 대응하여야 한다.
- 본 과업은 2025년도에 국한된 일시 업무로 용역직의 정규직 전환과 무관하다.
(정규직 전환 대상 아님.)
- 본 용역은 연구원 운영 추진 상황에 따라 조기 종료 될 수 있다.
문서번호 : -
2025년도 중이온가속기용 극저온시스템
개정번호 : 0
운전 용역 과업지시서 발 행 일 : 2024. 10. 30
페 이 지 : 19/19


---

### openai answer
retriever A 점수: 3점
가동기간의 24시간 교대근무와 휴지기간의 주간 운전이 명확히 기술되어 있어, 정답의 핵심이 모두 포함되어 있습니다.

retriever B 점수: 3점
가동기간의 24시간 교대근무 및 휴지기간의 주간 운전이 모두 명시되어 있어, 정답에 필요한 정보가 완전하게 포함되어 있습니다. 

============================================================
### query:
극저온시스템 과업지시서에 따르면, 가동기간 중 교대근무 인력이 개별적으로 얻은 운전 경험을 통합하여 정리하기 어려운 이유는 무엇이며, 이 업무는 주로 언제 수행되어야 합니까?

### answer:
교대근무로 인해 각 운전 인력마다 경험하는 운전 상황이 서로 달라, 모든 인원이 함께 모여 운전 경험을 통합하고 정리할 여건이 형성되기 어렵기 때문입니다. 이러한 운전 경험 통합 및 정리(문서화) 업무는 주로 주간 근무가 이루어지는 '휴지기간'에 집중적으로 수행됩니다.

---

### retriever A:
중이온가속기 또는 극저온시스템 가동기간에는 24시간 운전 업무를 필요로 한다. 통상적으
로 가동기간에는 24시간 운전을 위한 교대근무가 이루어지며, i) 극저온시스템의 모니터링
및 제어, ii) 극저온시스템의 주요 수치를 기록 및 분석하는 업무가 주로 이루어진다.
다만, 통상적으로 이러한 교대근무로 인해 각 운전 인력마다 경험할 수 있는 운전 상황이
서로 다르게 되고, 서로 다른 경험을 했던 운전 인력 모두가 참여하여 각자의 iii) 운전 경험
을 통합/정리할 수 있는 여건이 형성될 수 없다는 한계가 존재한다. 이러한 운전 경험의 통
합 정리는 후술하는 휴지기간에의 운전업무에 집중적으로 수행 가능하다.
통상적으로 휴지기간에는 극저온시스템의 유지보수 등이 진행되며, 유지보수 등을 위한 제
어, 유지보수 등의 경과 및 결과 확인을 위한 극저온시스템 모니터링, 유지보수 등의 결과
확인을 위한 주요 수치 기록 및 분석 또한 필수적인 운전 업무이다. 휴지기간에는 통상적으
로 주간 근무로 운전이 이루어진다. 이와 같이 주간 근무가 이루어질 수 있는 휴지기간의
특성 상, 휴지기간은 가동기간에서 교대근무로 각 운전 인력 개인이 얻은 단편적인 운전 경
험을 통합하여 정리(문서화)할 수 있는 여건이 형성될 수 있는 기간이다.
결과적으로 휴지기간에는 i) 중이온가속기용 극저온시스템의 운전 경험을 통합하여 정리(문
서화)하는 업무가 집중적으로 이루어지며, ii) 유지보수 등을 위한 극저온시스템의 제어와 iii)
유지보수 등을 위한 극저온시스템의 주요 수치 기록 및 분석이 병행되어야 한다.
3.2 운전 대상 및 장소
다. 계약자는 본 과업을 수행하기 위해 적절한 지식 또는 능력을 보유하고 있는 적정 수의 운
전 인력을 연구원에게 제공하여야 한다.
• 가동기간의 극저온시스템 운전 업무 (24시간 운전)
- 극저온시스템 모니터링
- 극저온시스템 주요 수치 기록 및 분석
- 극저온시스템 제어
- 기타 가동기간에 필요한 운전 업무
• 휴지기간의 극저온시스템 운전 업무 (주간 운전)
- 유지보수 등을 위한 극저온시스템 제어
- 유지보수 등을 위한 극저온시스템 모니터링
- 유지보수 등을 위한 극저온시스템 주요 수치 기록 및 분석
- 극저온시스템 운전 매뉴얼 작성 및 개정
- 극저온시스템 운전 교육
- 성과보고회 개최
- 기타 휴지기간에 필요한 운전 업무
• 본 과업 및 본 과업을 위해 투입된 인력에 대한 관리감독
본 과업의 기간은 2025년 1월 1일부터 2025년 12월 31일까지로 하며, 2025년 동안 가동기
간의 운전 업무를 수행하는 것을 기준으로 한다. 단, 표 2.와 같이 연구원의 운영 추진 상황
에 따라 휴지기간의 운전업무를 수행하는 것으로 변경될 수 있다. 가동기간과 휴지기간의
운전 업무는 “3. 과업 내용” 하위 절에 기술한 내용에 따른다.
표 2. 과업 추진 일정(안)
2025년
구분
1분기 2분기 3분기 4분기
SCL3 극저온플랜트
SCL3 중이온가속기 SCL3 구간 운전 SCL3 구간 유지보수
시운전 준비 및
구간 (가동기간) (휴지기간)
시운전 (가동기간)
SCL2 SCL2 극저온플랜트 중이온가속기 SCL2 구간 시운전 SCL2 구간 유지보수
구간 시운전 (가동기간) (가동기간) (휴지기간)
회수
교대근무 인력들도 회람 필수
4 일일(Daily) 운영 상황 공유 회의 참석 유사 시 계약자가 진행 가능
일일근무일지 작성 당일 현장/제어 관련 발생 상
5
(연구원 양식 제공) 황 기록
i) 교대 변경 전 특이사항 또
교대근무자 변경 시 인수인계 및 인수인계서 작성
6 는 관찰 필요 사항 전달
(연구원 양식 제공)
ii) 성실교대근무 서약서 작성
운전 중 수리가 필요한 사항 발생 시 별도 기록
7
(연구원 양식 제공)
극저온시스템용 제어시스템(알람시스템 포함) 개선
8 연구원 요청 시 한정
및 보완을 위한 협조
기타 극저온시스템의 안전하고 안정적인 운영을 위 연구원과 계약자가 협의하여
9
해 필요한 업무 수행 여부 결정
3.4 중이온가속기 또는 극저온시스템 휴지기간의 운전 업무
계약자는 중이온가속기 또는 극저온시스템 휴지기간 동안 주간 운전업무가 이루어질 수 있
도록 운전 인력을 제공해야 한다. 주간 운전 업무는 운전인력 전원 운전을 기본으로 한다.
운전인력 중 1인 또는 2인은 유지보수 등을 위한 극저온시스템 모니터링 및 제어를 수행해
야 하며, 나머지 인력들은 각 운전 인력이 경험했던 상황들을 취합하여 극저온시스템 운전
문서번호 : -
2025년도 중이온가속기용 극저온시스템
개정번호 : 0
운전 용역 과업지시서 발 행 일 : 2024. 10. 30
페 이 지 : 12/19
매뉴얼을 작성하고 개정하는 업무를 수행해야 한다. 또한, 과업 및 운전 인력 감독자가 1인
이상 운전실에 배치되어 상기 업무들을 감독해야 한다. 계약자가 휴지기간에 수행해야 하는


---

### retriever B:
3. 과업 내용
3.1 중이온가속기용 극저온시스템 운전 업무의 정의 및 필요성
본 과업에서 사용하는 “중이온가속기용 극저온시스템 운전 업무”의 정의는, “중이온가
속기용 극저온시스템이 안전하고 안정적으로 운영(가동기간뿐만 아니라 휴지기간의 운영 포
함)될 수 있도록 i) 극저온시스템을 모니터링 및 제어하고, ii) 극저온시스템의 주요 수치를
기록 및 분석하며, iii) 중이온가속기용 극저온시스템의 운전 경험을 통합하여 정리(문서화)하
는 업무를 의미한다.
중이온가속기 또는 극저온시스템 가동기간에는 24시간 운전 업무를 필요로 한다. 통상적으
로 가동기간에는 24시간 운전을 위한 교대근무가 이루어지며, i) 극저온시스템의 모니터링
및 제어, ii) 극저온시스템의 주요 수치를 기록 및 분석하는 업무가 주로 이루어진다.
다만, 통상적으로 이러한 교대근무로 인해 각 운전 인력마다 경험할 수 있는 운전 상황이
서로 다르게 되고, 서로 다른 경험을 했던 운전 인력 모두가 참여하여 각자의 iii) 운전 경험
을 통합/정리할 수 있는 여건이 형성될 수 없다는 한계가 존재한다. 이러한 운전 경험의 통
합 정리는 후술하는 휴지기간에의 운전업무에 집중적으로 수행 가능하다.
통상적으로 휴지기간에는 극저온시스템의 유지보수 등이 진행되며, 유지보수 등을 위한 제
어, 유지보수 등의 경과 및 결과 확인을 위한 극저온시스템 모니터링, 유지보수 등의 결과
확인을 위한 주요 수치 기록 및 분석 또한 필수적인 운전 업무이다. 휴지기간에는 통상적으
로 주간 근무로 운전이 이루어진다. 이와 같이 주간 근무가 이루어질 수 있는 휴지기간의
는 과업 및 운전 인력 감독자가 1인 이상 운전실에 존재해야 한다. 계약자가 가동기간에 수
행해야 하는 운전 업무 상세 내용은 하위 절에 기술된 내용을 따른다. 본 과업지시서에 기
술하지 않은 가동기간의 운전 업무는 연구원과 계약자가 협의하여 수행 여부를 결정한다.
3.3.1 극저온시스템 모니터링
계약자는 가동기간 동안 극저온시스템이 안정적으로 가동될 수 있도록 모니터링을 해야 한
교대근무 인력들도 회람 필수
4 일일(Daily) 운영 상황 공유 회의 참석 유사 시 계약자가 진행 가능
일일근무일지 작성 당일 현장/제어 관련 발생 상
5
(연구원 양식 제공) 황 기록
i) 교대 변경 전 특이사항 또
교대근무자 변경 시 인수인계 및 인수인계서 작성
6 는 관찰 필요 사항 전달
(연구원 양식 제공)
ii) 성실교대근무 서약서 작성
운전 중 수리가 필요한 사항 발생 시 별도 기록
7
(연구원 양식 제공)
극저온시스템용 제어시스템(알람시스템 포함) 개선
8 연구원 요청 시 한정
및 보완을 위한 협조
기타 극저온시스템의 안전하고 안정적인 운영을 위 연구원과 계약자가 협의하여
9
해 필요한 업무 수행 여부 결정
3.4 중이온가속기 또는 극저온시스템 휴지기간의 운전 업무
계약자는 중이온가속기 또는 극저온시스템 휴지기간 동안 주간 운전업무가 이루어질 수 있
도록 운전 인력을 제공해야 한다. 주간 운전 업무는 운전인력 전원 운전을 기본으로 한다.
운전인력 중 1인 또는 2인은 유지보수 등을 위한 극저온시스템 모니터링 및 제어를 수행해
야 하며, 나머지 인력들은 각 운전 인력이 경험했던 상황들을 취합하여 극저온시스템 운전
문서번호 : -
2025년도 중이온가속기용 극저온시스템
개정번호 : 0
운전 용역 과업지시서 발 행 일 : 2024. 10. 30
페 이 지 : 12/19
매뉴얼을 작성하고 개정하는 업무를 수행해야 한다. 또한, 과업 및 운전 인력 감독자가 1인
이상 운전실에 배치되어 상기 업무들을 감독해야 한다. 계약자가 휴지기간에 수행해야 하는
다. 계약자는 본 과업을 수행하기 위해 적절한 지식 또는 능력을 보유하고 있는 적정 수의 운
전 인력을 연구원에게 제공하여야 한다.
• 가동기간의 극저온시스템 운전 업무 (24시간 운전)
- 극저온시스템 모니터링
- 극저온시스템 주요 수치 기록 및 분석
- 극저온시스템 제어
- 기타 가동기간에 필요한 운전 업무
• 휴지기간의 극저온시스템 운전 업무 (주간 운전)
- 유지보수 등을 위한 극저온시스템 제어
- 유지보수 등을 위한 극저온시스템 모니터링
- 유지보수 등을 위한 극저온시스템 주요 수치 기록 및 분석
- 극저온시스템 운전 매뉴얼 작성 및 개정
- 극저온시스템 운전 교육
- 성과보고회 개최
- 기타 휴지기간에 필요한 운전 업무
• 본 과업 및 본 과업을 위해 투입된 인력에 대한 관리감독
본 과업의 기간은 2025년 1월 1일부터 2025년 12월 31일까지로 하며, 2025년 동안 가동기
간의 운전 업무를 수행하는 것을 기준으로 한다. 단, 표 2.와 같이 연구원의 운영 추진 상황
에 따라 휴지기간의 운전업무를 수행하는 것으로 변경될 수 있다. 가동기간과 휴지기간의
운전 업무는 “3. 과업 내용” 하위 절에 기술한 내용에 따른다.
표 2. 과업 추진 일정(안)
2025년
구분
1분기 2분기 3분기 4분기
SCL3 극저온플랜트
SCL3 중이온가속기 SCL3 구간 운전 SCL3 구간 유지보수
시운전 준비 및
구간 (가동기간) (휴지기간)
시운전 (가동기간)
SCL2 SCL2 극저온플랜트 중이온가속기 SCL2 구간 시운전 SCL2 구간 유지보수
구간 시운전 (가동기간) (가동기간) (휴지기간)
회수


---

### openai answer
retriever A 점수: 3점
교대근무로 각 인력의 운전 상황이 달라 모두가 함께 참여해 경험을 통합/정리하기 어렵다는 이유와, 이 통합·정리(문서화) 업무가 휴지기간(주간 근무)에 집중 수행된다고 명시되어 있어 답의 근거가 모두 포함되어 있습니다.

retriever B 점수: 3점
교대근무로 인해 경험 통합/정리가 어려운 이유를 명시하고, 해당 업무가 휴지기간에 수행되며 휴지기간은 주간 근무로 운영된다는 내용까지 포함되어 있어 답을 전부 뒷받침합니다. 

============================================================
### query:
계약업체가 매월 운전 현황을 정리해서 연구원에 보고해야 하는 공식적인 문서는 무엇이며, 제출 시점은 언제입니까?

### answer:
매월 운전 상황을 요약하여 제출해야 하는 문서는 '월간보고서'이며, 매월 기성 비용을 신청할 때 제출해야 합니다.

---

### retriever A:
계약종료 및 인수인계에 관한 사항
세부내용
○ 수행업체는 본 사업 종료 1개월 전 시스템의 발전 및 안정성을 유지하기 위한 인수인계 계획을 제시하여야 함
- 본 사업수행 기간 동안 취득한 모든 정보 및 생상된 산출물 일체
- 인수인계의 주요 업무, 상세 일정 및 인수인계 대상 주요 산출물 제시
- 인수인계 기간 동안 발생하는 리스크에 대한 방지대책 제시
○ 수행업체는 계약기간 만료 시 차기 수행업체로 선정된 사업자가 본 사업의 업무를 차질없이 수행할 수 있도록 요청 시 통합접수시스템 운영 업무 수행에 필요한 중요사항 등을 인계하여야 하며, 시스템의 효율적인 운영을 위해 하자담보 책임기간 중에는 기술자문에 응하여야 함
- 시스템 구성도, 시스템 현황, DB권한 등
- 통합접수시스템 기동/종료 절차
- 타 시스템 연계 및 적재 절차
- 일반 사용자 및 관리자 매뉴얼 등
○ 인수 및 인계 기간은 차기 시스템 운영 사업 개시일로부터 30일 이내로 하며, 인수 및 인계기간 동안의 용역비용은 지급하지 아니함
산출 정보
인수인계 계획서(사업수행계획서 내 포함 가능)
관련 요구사항
요구사항 출처
요구사항 분류
프로젝트지원 요구사항
요구사항 고유번호
PSR-008
요구사항 명칭
정보화사업 주요점검 및 수행지원
요구사항
상세설명
정의
정보화사업 주요점검 및 수행지원
세부내용
○ 통합접수시스템 관련 점검사항 준수 및 현장점검 수행
○ 보안관리 실태 점검 대응
○ 공공데이터 품질관리(DB) 수준평가 대응 및 데이터 오류 정제 활동 실시
산출 정보
점검별 결과보고서
관련 요구사항
요구사항 출처
□ 품질관리 요구사항
요구사항 분류
· 월별 추진 실적 보고
· 관련부서 업무협의 내용
· 문제점 보고 및 특이사항
매월 말일
비정기
보고
인수인계
보고
· 신규 사업자 선정시 업무인계 결과보고
사업종료
1주전
기타
· 이슈사항 및 문제점 협의 등
이슈발생시
※ 보고형식 및 주기는 추후 협의하여 조정할 수 있음
ㅇ 계약 기간 만료일 전 과업 완료 보고서를 제출하여야 함
구분
보고명
보고내용
시기
사업
종료
완료
보고서
· 용역종료 시 전체 추진단계의 수행결과 보고로 인쇄물 2부, 외장하드 1본으로 제출
· 산출물을 종류별로 분류한 파일
용역종료일
2주일 전
- 용역 완료시 제출하는 최종 결과물(프로그램 소스 코드(주석, 설명서 포함), ERD, 테이블 정의서, 프로그램 정의서 등)은 발주자의 승인을 득한 후 원본 파일을 CD 또는 HDD에 수록하여 산출물과 같이 제출하여야 하고, 프로그램 소스에 대한 형상관리가 가능하도록 협조하여야 함
- 산출물은 기존 산출물을 기반으로 현행화 또는 추가 작성해야 하며, 상호 협의하여 조정 가능
ㅇ 제출 시기에 따라 종류별 산출물을 제출하되, 사업관리 및 검수를 위한 각종 업무 보고를 요청할 경우, 신속하게 작성하여 보고하는 수시보고 체계 마련 ※ 제출 전 주관기관 사전 검토 및 확인
산출정보
착수보고서, 완료보고서, 월간보고서
요구사항 고유번호
PMR-005
요구사항 명칭
인계 조건
요구사항
상세설명
정의
관한 사항
세부
내용
인계 요건
- 사업종료 1개월 전까지 업무인계 계획서를 작성해 진흥원의 승인을 받아야함
- 발주기관과 월 1회 이상 기술이전 등에 대한 협의를 하고, 그 결과를 문서화
- 금회 대처 방법
인 서명 필수
- 향후 대처 방법
- 수치 기록을 위해 계약자가
- 기타 필요한 내용
준비한 PC에 상시 저장
- 매 월간 기성 신청 시 제출
월간보고서 - 월간 운전 상황 요약 O
- 문서 파일 제출
- IBS에서 지정한 기한 내 제
기타 - IBS에서 요구한 결과물 O
출
문서번호 : -
2025년도 중이온가속기용 극저온시스템
개정번호 : 0
운전 용역 과업지시서 발 행 일 : 2024. 10. 30
페 이 지 : 18/19
5. 기타사항
계약자는 본 절에서 기술하는 사항들을 명확히 이해하고 숙지한 뒤, 본 과업을 착수해야
한다. 본 절의 내용에 대한 미숙지로 발생하는 불이익은 계약자의 책임으로 한다.
- 본 과업은 극저온시스템 운전에 대한 단독업무를 수행한다.
(단, 중이온가속기 운영 건전성과 관련된 중대한 운전 사항은 연구원과 협의하여 진행)
- 계약자는 연구원에게서 인수인계 받은 과업 일체 내용을 바탕으로 극저온시스템 운전에
필요한 인원, 인력구성 및 운전근무를 편성한다.
- 극저온시스템의 가동기간과 휴지기간은 중이온가속기 운영 상황에 따라 연구원이 결정하
며, 연구원은 가동기간/휴지기간 변경 최소 2 주 전에는 계약자에게 통지한다.
- 가동기간에는 24시간 운전이 이루어지며, 휴지기간에는 주간 운전이 이루어져야 한다. 휴
지기간에의 주간 근무는 가동기간동안 24시간 운전에 투입된 운전 인력 전원이 수행해야 하
며(휴가자 제외), 계약자는 연구원의 가동기간/휴지기간 선정에 대한 이의 및 각 기간의 운
전 형태(24시간 운전/주간 운전)에 대해 이의를 제기할 수 없다.


---

### retriever B:
4. 제출문서
계약자는 표 13.와 같은 문서들을 제출일정에 맞춰 연구원에 제출해야 한다.
표 13. 제출문서 목록 및 관련 사항
연구원
제출문서 내용 비고 및 제출일
승인
투입 인력
- 투입 인력 이력 사항 X - 계약 시
이력서
- 최초: 계약 후 1개월 내
작업계획서 - 일별 투입 인력 - 이후: 매주 금요일까지 차주
X
(주간근무표) - 휴가상황 근무표 제출
- 문서 파일 제출
- IBS에서 제공한 양식 파일
극저온시스템 - 매 월간 기성 신청 시 제출
주요 수치 - 극저온시스템 주요 수치 X - 문서 파일 제출
기록문서 - 수치 기록을 위해 계약자가
준비한 PC에 상시 저장
일일근무일지 - 일일 운영 상황 기록 O - 매 월간 기성 신청 시 제출
- 문서 파일 제출
- 수치 기록을 위해 계약자가
인수인계 문서 - (가동기간) 인수인계 내용 X 준비한 PC에 상시 저장
- 수리 사유 발생일
- 매 월간 기성 신청 시 제출
- 수리 필요 대상
수리필요사항 - 문서 파일 제출
- 상세 내용 O
기록 문서 - 수치 기록을 위해 계약자가
- 발견자
준비한 PC에 상시 저장
- 기타 필요한 내용
- 발생한 경우, 기성 신청 시
특이사항, 위험상황 발생 시,
제출 (해당 월 발생 건)
- 상황 내용
- 문서 파일 제출
- 작성자
Lesson learn O - 운전 인력 전원 회람 후 확
- 금회 대처 방법
인 서명 필수
- 향후 대처 방법
- 수치 기록을 위해 계약자가
- 기타 필요한 내용
준비한 PC에 상시 저장
- 매 월간 기성 신청 시 제출
월간보고서 - 월간 운전 상황 요약 O
- 문서 파일 제출
라. 계약 후 제출서류 : 사업수행계획서, 기술적용계획표
1) 사업수행자는 계약일로부터 14일 이내에 사업수행계획서와 기술적용계획표를 제출하고 사업 착수를 완료하여야 함
2) 사업수행계획서에는 제안요청서, 제안서, 협상에 의해 수정·보완·추가된 내용을 반영하여 제출하여야 하며 발주기관의 승인을 받아야 함
제출방법
1
착수계
A4
1
계약 후 7일 이내
서신문서
2
보안관련 문서
(보안서약서, 비밀유지서약서, 개인정보 위탁계약서, 개인정보 보호서약서)
A4
1
계약 후 7일 이내
3
사업수행계획서
A4
8
계약 후 10일 이내
4
착수보고서
A4
8
착수 후 10일 이내
5
완료보고서
A4
8
계약종료 2주 전 까지
6
최종산출물
(개발 산출물, 운영자 매뉴얼, 사용자 매뉴얼, 각종 분석/측정 결과보고서, 기술지원 확약서, 품질관리보고서, 기술적용계획/결과표, 기술이전계획서, 장애대응 매뉴얼, 테스트 수행 계획서/결과서 등)
USB
2
계약종료 2주 전 까지
부터 10일 이내
략, 사업추진내용, 일정, 참여인력, 예산 등
- 사업내용을 중심으로 세부 사업수행계획서를 계약일로부터 10일 이내에 제출하여야 함
※ 계약서, 제안요청서 등을 근거로 작성
발표자료
·사업수행계획서
보고회
간의 1/2 시점(협의)
기간의 1/2 시점에 사업 경과보고 제출 및 중간보고회 개최
※ 기한 및 날짜는 상호 협의하여 정함
발표자료
·중간보고서
보고회
료 전 10일 이내
만료일 15일 전까지 최종보고서 초안 제출
- 계약 만료 전 10일 이내에 최종보고회 개최
- 중간보고 이후의 보완․확정 내용 상호 점검
- 최종보고서 10부 준비 ※ 필요시 발표자료(PPT, PDF 파일 등) 준비
- 용역책임자가 사업 결과 종합 보고
발표자료
·최종보고서
·최종보고서
요약본
: 주 1회
회의) 주 1회, 수행내용 점검, 사업추진 현황관리
생 시
업의 추진상황에 대해 발주기관의 요구가 있을 때에는 수시로 용역 추진 상황 및 진도를 보고하여야 함
계획서
부터 1개월 이내
사 세부수행내용
- 데이터수집 및 자료조사 체크리스트
- 현지 네트워킹을 위한 기념품 제작·배포
※ 영문은 사업대상국 공유 사항으로, 국문의 간소화 버전으로 작성
(국/영문)
·체크리스트
보고서
후 5일 이내
장내용 및 회의록 등 포함
- 현지조사내용 및 현지방문 시 발견사항 등 포함
※ 영문은 사업대상국 공유 사항으로, 국문의 간소화 버전으로 작성
고서
(국/영문)
회의
진방향 및 진행 결과에 대한 논의
※ 회의준비, 회의참석, 회의리딩 및 결과정리
(국문/영문)
·회의록
료일 이전
고회 이후 수정 보완사항 반영하여 최종 산출물 제출


---

### openai answer
retriever A 점수: 3점
(‘월간보고서’가 월간 운전 상황 요약 문서임이 명시되어 있고, “매 월간 기성 신청 시 제출”이라는 제출 시점도 명확히 포함되어 있어 정답의 두 요소가 모두 있음)

retriever B 점수: 2점
(‘월간보고서’가 월간 운전 상황 요약 문서라는 점은 명시되어 있으나, 해당 문서의 제출 시점을 “매 월간 기성 신청 시”로 명확히 연결해 표기한 부분이 보이지 않아 일부 정보만 충족) 

## size 5000

============================================================
### query:
본 과업지시서에서 '극저온시스템'은 무엇을 포함하는 것으로 정의하고 있습니까?

### answer:
'극저온시스템'은 SCL3/SCL2 극저온플랜트 및 부대설비, SCL3/SCL2 분배시스템 및 부대설비, SCL3/SCL2 초전도가속관 저온유지모듈, 기체헬륨 탱크, 액체질소 시스템 및 극저온시스템용 헬륨회수시스템을 모두 포함하는 것으로 정의됩니다.

---

### retriever A:
7. 안전관리 .............................................................................................................. 19
8. 붙임 자료 ........................................................................................................... 19
문서번호 : -
2025년도 중이온가속기용 극저온시스템
개정번호 : 0
운전 용역 과업지시서 발 행 일 : 2024. 10. 30
페 이 지 : 5/19
1. 서 론
1.1 목 적
기초과학연구원 중이온가속기연구소(이하 “연구원”)에서 운영중인 중이온가속기용 극저
온시스템(이하 “극저온시스템”)은 고도의 공정 시스템으로, 안전하고 안정적인 중이온가속
기의 운영을 위해 상시 모니터링이 필요하다. 이에 본 과업지시서는 극저온시스템의 운전
용역에 대한 상세 과업 내용을 기술한다.
1.2 용어 설명
본 과업지시서에서 사용하는 용어에 대한 상세 내용은 표 1.과 같다. 계약자는 반드시 해당
용어의 의미를 충분히 이해하고 숙지해야 하며, 용어 미숙지로 인한 책임은 계약자에게 있
다.
표 1. 본 과업지시서에 사용되는 용어 설명
용어 설명
연구원 중이온가속기연구소
계약자 본 과업을 수행할 계약상대자
SCL3/SCL2 극저온플랜트 및 부대설비,
SCL3/SCL2 분배시스템 및 부대설비,
SCL3/SCL2 초전도가속관 저온유지모듈,
극저온시스템
기체헬륨 탱크, 액체질소 시스템 및
TBx 극저온설비동
밸브박스, 초전도가속관 저온유지
SCL3 터널/
극저온
SCL3 LINAC 모듈 및 밸브박스 진공단열용 진
ISOL 구간
분배
공펌프
시스템 밸브박스, 초전도가속관 저온유지
SCL2 터널/
SCL2 LINAC 모듈 및 밸브박스 진공단열용 진
IF 구간
공펌프
외부 정화기 외부
회수 압축시스템 헬륨압축기동
회수
가스백 헬륨압축기동
시스템
헬륨 탱크 외부 기체 헬륨량 관리 포함
액체 질소 시스템 외부 액체 질소량 관리 포함
3.3 중이온가속기 또는 극저온시스템 가동기간의 운전 업무
계약자는 중이온가속기 또는 극저온시스템 가동기간 동안 24시간 운전업무가 이루어질 수
있도록 운전 인력을 제공해야 한다. 24시간 운전 업무는 2인 1조를 기본으로 하며, 주간에
는 과업 및 운전 인력 감독자가 1인 이상 운전실에 존재해야 한다. 계약자가 가동기간에 수
행해야 하는 운전 업무 상세 내용은 하위 절에 기술된 내용을 따른다. 본 과업지시서에 기
술하지 않은 가동기간의 운전 업무는 연구원과 계약자가 협의하여 수행 여부를 결정한다.
3.3.1 극저온시스템 모니터링
계약자는 가동기간 동안 극저온시스템이 안정적으로 가동될 수 있도록 모니터링을 해야 한
다. 가동기간의 모니터링 업무는 극저온시스템이 안정적으로 가동되고 있는지 여부를 확인
하기 위해 극저온시스템을 지속적으로 관찰하는 업무를 의미하며, 표 4.에 가동기간의 모니
터링 업무에 해당되는 내용들을 기술하였다.
극저온시스템의 모니터링은 연구원에 구축되어있는 극저온시스템용 제어시스템(이하 “제
문서번호 : -
2025년도 중이온가속기용 극저온시스템
개정번호 : 0
운전 용역 과업지시서 발 행 일 : 2024. 10. 30
페 이 지 : 12/19
매뉴얼을 작성하고 개정하는 업무를 수행해야 한다. 또한, 과업 및 운전 인력 감독자가 1인
이상 운전실에 배치되어 상기 업무들을 감독해야 한다. 계약자가 휴지기간에 수행해야 하는
운전 업무 상세 내용은 하위 절에 기술된 내용을 따른다. 본 과업지시서에 기술하지 않은
휴지기간의 운전 업무는 연구원과 계약자가 협의하여 수행 여부를 결정한다.
3.4.1 유지보수 등을 위한 극저온시스템 제어
극저온시스템의 제어는 극저온시스템이 휴지지간에 안전하고 안정적으로 운영될 수 있도록
극저온시스템 또는 극저온시스템 관련 부품들을 제어시스템을 활용하여 또는 현장에서 손으
로 조작하는 행위를 의미한다. 계약자는 극저온시스템이 본 목적에 부합하도록 극저온시스
템의 제어를 수행해야 한다. 표 8.에 극저온시스템 제어 업무에 해당되는 사항들을 기술하
였다.
휴지기간의 극저온시스템 제어는 i) 극저온시스템에 대한 충분한 이해를 바탕으로 ii) 조작
후 유지보수 등 현장 활동에 미치는 영향을 충분히 검토한 뒤 이루어져야 한다. 더불어, iii)
조작 대상물이 위치하고 있는 현장에 인력이 존재하는지 여부를 확인하고, iv) 현장에 인력
이 존재할 경우, 조작 대상물을 조작했을 때 발생할 수 있는 상황에 대해 해당 인력의 안전
이 확보된 뒤에 조작을 수행해야 한다. 마지막으로 v) 조작 대상/조작 상태에 대한 혼동이
없도록 각별히 주의를 하여 제어가 진행되어야 한다.
기록이 필요한 주요 수치 항목 및 수치 기록 간격은 연구원이 정하는 바에 따른다. 본 수
치 항목과 기록 간격은 극저온시스템의 운영 상황에 따라 증감될 수 있다.
더불어, 연구원의 요청이 있는 경우, 계약자는 주요 수치들에 대한 분석을 수행해야 한다.
분석 대상, 분석 방법 및 분석 결과물의 형태 등은 연구원이 정하는 바에 따른다.
극저온시스템의 주요 수치/상황을 기록하고 분석하기 위해 필요한 노트북 컴퓨터와 키보드
및 마우스 등의 컴퓨터 악세서리류는 계약자가 준비해야 한다.
3.3.3 극저온시스템 제어
극저온시스템의 제어는 극저온시스템이 가동기간에 안전하고 안정적으로 운영될 수 있도록
극저온시스템 또는 극저온시스템 관련 부품들을 제어시스템을 활용하여 또는 현장에서 손으
로 조작하는 행위를 의미한다. 계약자는 극저온시스템이 본 목적에 부합하도록 극저온시스
문서번호 : -
2025년도 중이온가속기용 극저온시스템
개정번호 : 0
운전 용역 과업지시서 발 행 일 : 2024. 10. 30
페 이 지 : 10/19
템의 제어를 수행해야 한다. 표 6.에 극저온시스템 제어 업무에 해당되는 사항들을 기술하
였다.
가동기간에 극저온시스템의 제어는 i) 극저온시스템에 대한 충분한 이해를 바탕으로 ii) 조
작 후 공정에 미치는 영향을 충분히 검토한 뒤 이루어져야 한다. 더불어, iii) 조작 대상물이
위치하고 있는 현장에 인력이 존재하는지 여부를 확인하고, iv) 현장에 인력이 존재할 경우,
조작 대상물을 조작했을 때 발생할 수 있는 상황에 대해 해당 인력의 안전이 확보된 뒤에
다.
표 1. 본 과업지시서에 사용되는 용어 설명
용어 설명
연구원 중이온가속기연구소
계약자 본 과업을 수행할 계약상대자
SCL3/SCL2 극저온플랜트 및 부대설비,
SCL3/SCL2 분배시스템 및 부대설비,
SCL3/SCL2 초전도가속관 저온유지모듈,
극저온시스템
기체헬륨 탱크, 액체질소 시스템 및
극저온시스템용 헬륨회수시스템 포함
(“3.2절 운전대상”, 표 3. 참고)
중이온가속기 또는 극저온시스템이 냉각(냉각 착수
가동기간
이후) 또는 빔서비스 등을 위해 가동 중인 기간
중이온가속기 또는 극저온시스템이 유지보수 등을
휴지기간
위해 가동을 멈추고 휴지 중인 기간
중이온가속기용 극저온시스템이 가동기간 및 휴지
기간에 안전하고 안정적으로 운영될 수 있도록,
중이온가속기용 극저온시스템 i) 극저온시스템을 모니터링 및 제어하고,
운전 업무 ii) 극저온시스템의 주요 수치를 기록 및 분석하며,
iii) 중이온가속기용 극저온시스템의 운전 경험을
통합하여 정리(문서화)하는 업무
문서번호 : -
2025년도 중이온가속기용 극저온시스템
개정번호 : 0
운전 용역 과업지시서 발 행 일 : 2024. 10. 30
페 이 지 : 6/19
2. 과업범위 및 일정
본 과업의 범위는 아래와 같다. 해당 항목의 세부 업무내용은 “3. 과업 내용”에서 기술한
다. 계약자는 본 과업을 수행하기 위해 적절한 지식 또는 능력을 보유하고 있는 적정 수의 운
전 인력을 연구원에게 제공하여야 한다.
• 가동기간의 극저온시스템 운전 업무 (24시간 운전)
- 극저온시스템 모니터링
- 극저온시스템 주요 수치 기록 및 분석
- 극저온시스템 제어
다. 가동기간의 모니터링 업무는 극저온시스템이 안정적으로 가동되고 있는지 여부를 확인
하기 위해 극저온시스템을 지속적으로 관찰하는 업무를 의미하며, 표 4.에 가동기간의 모니
터링 업무에 해당되는 내용들을 기술하였다.
극저온시스템의 모니터링은 연구원에 구축되어있는 극저온시스템용 제어시스템(이하 “제
어시스템”)을 활용하여 또는 현장에 설치되어있는 장치를 통해 수행한다. 연구원은 구축되
어있는 제어시스템을 활용하기 위한 컴퓨터, 모니터, 키보드, 마우스, 책상 및 의자를 계약
자에게 제공한다. 현재 연구원에 구축되어있는 제어시스템은 표 5.과 같다.
연구원은 계약자에게 현재 구축되어있는 제어시스템에 대한 기본 사용 교육을 제공하며,
계약자는 연구원이 제공하는 교육을 이해하기 위한 배경지식을 보유해야 한다.
문서번호 : -
2025년도 중이온가속기용 극저온시스템
개정번호 : 0
운전 용역 과업지시서 발 행 일 : 2024. 10. 30
페 이 지 : 9/19
표 4. 가동기간의 극저온시스템 모니터링 업무 리스트
순번 극저온시스템 모니터링 업무 비고
1 알람 메시지 확인
2 주요 센서 트렌드 확인 온도, 압력, 유량, 속도 등
3 정상 운전 상황의 주요 센서 지시값 파악 및 숙지 온도, 압력, 유량, 속도 등
4 비정상 운전 상황 감지
연구원에서 현장 확인 필요상황 발생 시, 연구원이 지 무전 통신 활용
5
정하는 값 관찰, 확인 및 상황 전달 (연구원에서 무전기 제공)
6 제어로직 정상 작동 여부 확인
연구원 요청 시,
7 현장 상황 일시적 또는 지속적 관찰
비방사선구역 한정
2025년
구분
1분기 2분기 3분기 4분기
SCL3 극저온플랜트
SCL3 중이온가속기 SCL3 구간 운전 SCL3 구간 유지보수
시운전 준비 및
구간 (가동기간) (휴지기간)
시운전 (가동기간)
SCL2 SCL2 극저온플랜트 중이온가속기 SCL2 구간 시운전 SCL2 구간 유지보수
구간 시운전 (가동기간) (가동기간) (휴지기간)
회수
필요시 간헐적 운전 (휴지기간에 준함)
시스템
문서번호 : -
2025년도 중이온가속기용 극저온시스템
개정번호 : 0
운전 용역 과업지시서 발 행 일 : 2024. 10. 30
페 이 지 : 7/19
3. 과업 내용
3.1 중이온가속기용 극저온시스템 운전 업무의 정의 및 필요성
본 과업에서 사용하는 “중이온가속기용 극저온시스템 운전 업무”의 정의는, “중이온가
속기용 극저온시스템이 안전하고 안정적으로 운영(가동기간뿐만 아니라 휴지기간의 운영 포
함)될 수 있도록 i) 극저온시스템을 모니터링 및 제어하고, ii) 극저온시스템의 주요 수치를
기록 및 분석하며, iii) 중이온가속기용 극저온시스템의 운전 경험을 통합하여 정리(문서화)하
는 업무를 의미한다.
중이온가속기 또는 극저온시스템 가동기간에는 24시간 운전 업무를 필요로 한다. 통상적으
로 가동기간에는 24시간 운전을 위한 교대근무가 이루어지며, i) 극저온시스템의 모니터링
및 제어, ii) 극저온시스템의 주요 수치를 기록 및 분석하는 업무가 주로 이루어진다.
다만, 통상적으로 이러한 교대근무로 인해 각 운전 인력마다 경험할 수 있는 운전 상황이


---

### retriever B:
1. 서 론
1.1 목 적
기초과학연구원 중이온가속기연구소(이하 “연구원”)에서 운영중인 중이온가속기용 극저
온시스템(이하 “극저온시스템”)은 고도의 공정 시스템으로, 안전하고 안정적인 중이온가속
기의 운영을 위해 상시 모니터링이 필요하다. 이에 본 과업지시서는 극저온시스템의 운전
용역에 대한 상세 과업 내용을 기술한다.
1.2 용어 설명
본 과업지시서에서 사용하는 용어에 대한 상세 내용은 표 1.과 같다. 계약자는 반드시 해당
용어의 의미를 충분히 이해하고 숙지해야 하며, 용어 미숙지로 인한 책임은 계약자에게 있
다.
표 1. 본 과업지시서에 사용되는 용어 설명
용어 설명
연구원 중이온가속기연구소
계약자 본 과업을 수행할 계약상대자
SCL3/SCL2 극저온플랜트 및 부대설비,
SCL3/SCL2 분배시스템 및 부대설비,
SCL3/SCL2 초전도가속관 저온유지모듈,
극저온시스템
기체헬륨 탱크, 액체질소 시스템 및
극저온시스템용 헬륨회수시스템 포함
(“3.2절 운전대상”, 표 3. 참고)
중이온가속기 또는 극저온시스템이 냉각(냉각 착수
가동기간
이후) 또는 빔서비스 등을 위해 가동 중인 기간
중이온가속기 또는 극저온시스템이 유지보수 등을
휴지기간
위해 가동을 멈추고 휴지 중인 기간
중이온가속기용 극저온시스템이 가동기간 및 휴지
기간에 안전하고 안정적으로 운영될 수 있도록,
중이온가속기용 극저온시스템 i) 극저온시스템을 모니터링 및 제어하고,
운전 업무 ii) 극저온시스템의 주요 수치를 기록 및 분석하며,
iii) 중이온가속기용 극저온시스템의 운전 경험을
통합하여 정리(문서화)하는 업무
문서번호 : -
문서번호 : -
2025년도 중이온가속기용 극저온시스템
개정번호 : 0
운전 용역 과업지시서 발 행 일 : 2024. 10. 30
페 이 지 : 12/19
매뉴얼을 작성하고 개정하는 업무를 수행해야 한다. 또한, 과업 및 운전 인력 감독자가 1인
이상 운전실에 배치되어 상기 업무들을 감독해야 한다. 계약자가 휴지기간에 수행해야 하는
운전 업무 상세 내용은 하위 절에 기술된 내용을 따른다. 본 과업지시서에 기술하지 않은
휴지기간의 운전 업무는 연구원과 계약자가 협의하여 수행 여부를 결정한다.
3.4.1 유지보수 등을 위한 극저온시스템 제어
극저온시스템의 제어는 극저온시스템이 휴지지간에 안전하고 안정적으로 운영될 수 있도록
극저온시스템 또는 극저온시스템 관련 부품들을 제어시스템을 활용하여 또는 현장에서 손으
로 조작하는 행위를 의미한다. 계약자는 극저온시스템이 본 목적에 부합하도록 극저온시스
템의 제어를 수행해야 한다. 표 8.에 극저온시스템 제어 업무에 해당되는 사항들을 기술하
였다.
휴지기간의 극저온시스템 제어는 i) 극저온시스템에 대한 충분한 이해를 바탕으로 ii) 조작
후 유지보수 등 현장 활동에 미치는 영향을 충분히 검토한 뒤 이루어져야 한다. 더불어, iii)
조작 대상물이 위치하고 있는 현장에 인력이 존재하는지 여부를 확인하고, iv) 현장에 인력
이 존재할 경우, 조작 대상물을 조작했을 때 발생할 수 있는 상황에 대해 해당 인력의 안전
이 확보된 뒤에 조작을 수행해야 한다. 마지막으로 v) 조작 대상/조작 상태에 대한 혼동이
없도록 각별히 주의를 하여 제어가 진행되어야 한다.
3. 과업 내용
3.1 중이온가속기용 극저온시스템 운전 업무의 정의 및 필요성
본 과업에서 사용하는 “중이온가속기용 극저온시스템 운전 업무”의 정의는, “중이온가
속기용 극저온시스템이 안전하고 안정적으로 운영(가동기간뿐만 아니라 휴지기간의 운영 포
함)될 수 있도록 i) 극저온시스템을 모니터링 및 제어하고, ii) 극저온시스템의 주요 수치를
기록 및 분석하며, iii) 중이온가속기용 극저온시스템의 운전 경험을 통합하여 정리(문서화)하
는 업무를 의미한다.
중이온가속기 또는 극저온시스템 가동기간에는 24시간 운전 업무를 필요로 한다. 통상적으
로 가동기간에는 24시간 운전을 위한 교대근무가 이루어지며, i) 극저온시스템의 모니터링
및 제어, ii) 극저온시스템의 주요 수치를 기록 및 분석하는 업무가 주로 이루어진다.
다만, 통상적으로 이러한 교대근무로 인해 각 운전 인력마다 경험할 수 있는 운전 상황이
서로 다르게 되고, 서로 다른 경험을 했던 운전 인력 모두가 참여하여 각자의 iii) 운전 경험
을 통합/정리할 수 있는 여건이 형성될 수 없다는 한계가 존재한다. 이러한 운전 경험의 통
합 정리는 후술하는 휴지기간에의 운전업무에 집중적으로 수행 가능하다.
통상적으로 휴지기간에는 극저온시스템의 유지보수 등이 진행되며, 유지보수 등을 위한 제
어, 유지보수 등의 경과 및 결과 확인을 위한 극저온시스템 모니터링, 유지보수 등의 결과
확인을 위한 주요 수치 기록 및 분석 또한 필수적인 운전 업무이다. 휴지기간에는 통상적으
로 주간 근무로 운전이 이루어진다. 이와 같이 주간 근무가 이루어질 수 있는 휴지기간의
는 과업 및 운전 인력 감독자가 1인 이상 운전실에 존재해야 한다. 계약자가 가동기간에 수
행해야 하는 운전 업무 상세 내용은 하위 절에 기술된 내용을 따른다. 본 과업지시서에 기
술하지 않은 가동기간의 운전 업무는 연구원과 계약자가 협의하여 수행 여부를 결정한다.
3.3.1 극저온시스템 모니터링
계약자는 가동기간 동안 극저온시스템이 안정적으로 가동될 수 있도록 모니터링을 해야 한
기록이 필요한 주요 수치 항목 및 수치 기록 간격은 연구원이 정하는 바에 따른다. 본 수
치 항목과 기록 간격은 극저온시스템의 운영 상황에 따라 증감될 수 있다.
더불어, 연구원의 요청이 있는 경우, 계약자는 주요 수치들에 대한 분석을 수행해야 한다.
분석 대상, 분석 방법 및 분석 결과물의 형태 등은 연구원이 정하는 바에 따른다.
극저온시스템의 주요 수치/상황을 기록하고 분석하기 위해 필요한 노트북 컴퓨터와 키보드
및 마우스 등의 컴퓨터 악세서리류는 계약자가 준비해야 한다.
3.3.3 극저온시스템 제어
극저온시스템의 제어는 극저온시스템이 가동기간에 안전하고 안정적으로 운영될 수 있도록
극저온시스템 또는 극저온시스템 관련 부품들을 제어시스템을 활용하여 또는 현장에서 손으
로 조작하는 행위를 의미한다. 계약자는 극저온시스템이 본 목적에 부합하도록 극저온시스
문서번호 : -
2025년도 중이온가속기용 극저온시스템
개정번호 : 0
운전 용역 과업지시서 발 행 일 : 2024. 10. 30
페 이 지 : 10/19
템의 제어를 수행해야 한다. 표 6.에 극저온시스템 제어 업무에 해당되는 사항들을 기술하
였다.
가동기간에 극저온시스템의 제어는 i) 극저온시스템에 대한 충분한 이해를 바탕으로 ii) 조
작 후 공정에 미치는 영향을 충분히 검토한 뒤 이루어져야 한다. 더불어, iii) 조작 대상물이
위치하고 있는 현장에 인력이 존재하는지 여부를 확인하고, iv) 현장에 인력이 존재할 경우,
조작 대상물을 조작했을 때 발생할 수 있는 상황에 대해 해당 인력의 안전이 확보된 뒤에
다. 가동기간의 모니터링 업무는 극저온시스템이 안정적으로 가동되고 있는지 여부를 확인
하기 위해 극저온시스템을 지속적으로 관찰하는 업무를 의미하며, 표 4.에 가동기간의 모니
터링 업무에 해당되는 내용들을 기술하였다.
극저온시스템의 모니터링은 연구원에 구축되어있는 극저온시스템용 제어시스템(이하 “제
어시스템”)을 활용하여 또는 현장에 설치되어있는 장치를 통해 수행한다. 연구원은 구축되
어있는 제어시스템을 활용하기 위한 컴퓨터, 모니터, 키보드, 마우스, 책상 및 의자를 계약
자에게 제공한다. 현재 연구원에 구축되어있는 제어시스템은 표 5.과 같다.
연구원은 계약자에게 현재 구축되어있는 제어시스템에 대한 기본 사용 교육을 제공하며,
계약자는 연구원이 제공하는 교육을 이해하기 위한 배경지식을 보유해야 한다.
문서번호 : -
2025년도 중이온가속기용 극저온시스템
개정번호 : 0
운전 용역 과업지시서 발 행 일 : 2024. 10. 30
페 이 지 : 9/19
표 4. 가동기간의 극저온시스템 모니터링 업무 리스트
순번 극저온시스템 모니터링 업무 비고
1 알람 메시지 확인
2 주요 센서 트렌드 확인 온도, 압력, 유량, 속도 등
3 정상 운전 상황의 주요 센서 지시값 파악 및 숙지 온도, 압력, 유량, 속도 등
4 비정상 운전 상황 감지
연구원에서 현장 확인 필요상황 발생 시, 연구원이 지 무전 통신 활용
5
정하는 값 관찰, 확인 및 상황 전달 (연구원에서 무전기 제공)
6 제어로직 정상 작동 여부 확인
연구원 요청 시,
7 현장 상황 일시적 또는 지속적 관찰
비방사선구역 한정
요구사항 명칭
시스템 구축 사업 공통 적용 사항
요구사항 명칭
시스템 장비구성
요구사항 명칭
시스템 장비구성
요구사항 명칭
운영시스템 적용 및 운영
요구
사항
상세
설명
정의
운영시스템 적용 및 운영
세부
내용
- 시스템 가동을 위한 시험운영 방안을 제시하고 계획에 따라 차질 없이 시행하여야 함.
요구사항 명칭
운영시스템 적용 및 운영
요구
사항
상세
설명
정의
운영시스템 적용 및 운영
세부
내용
- 시스템 가동을 위한 시험운영 방안을 제시하고 계획에 따라 차질 없이 시행하여야 함.
나. 시스템 현황(홈페이지 및 내부시스템 운영)
업무시스템
연계시스템
<꿀등급판정시스템>
(시행업체, 규격검사기관)
<실험실관리시스템>
(내부직원용-시료접수자)
<품질평가시스템>
(내부직원용-품질평가사)
요구사항 명칭
시스템 연계 운영
요구사항 명칭
시스템 개발 장비
통상적으로 휴지기간에는 극저온시스템의 유지보수 등이 진행되며, 유지보수 등을 위한 제
어, 유지보수 등의 경과 및 결과 확인을 위한 극저온시스템 모니터링, 유지보수 등의 결과
확인을 위한 주요 수치 기록 및 분석 또한 필수적인 운전 업무이다. 휴지기간에는 통상적으
로 주간 근무로 운전이 이루어진다. 이와 같이 주간 근무가 이루어질 수 있는 휴지기간의
특성 상, 휴지기간은 가동기간에서 교대근무로 각 운전 인력 개인이 얻은 단편적인 운전 경
험을 통합하여 정리(문서화)할 수 있는 여건이 형성될 수 있는 기간이다.
결과적으로 휴지기간에는 i) 중이온가속기용 극저온시스템의 운전 경험을 통합하여 정리(문
서화)하는 업무가 집중적으로 이루어지며, ii) 유지보수 등을 위한 극저온시스템의 제어와 iii)
유지보수 등을 위한 극저온시스템의 주요 수치 기록 및 분석이 병행되어야 한다.
3.2 운전 대상 및 장소
본 과업의 운전 대상은 표 3.과 같다. 계약자는 표 3. 항목에 대한 운전을 수행해야 한다.
운전 장소는 극저온설비동 2층 제어실에 국한하지 않고, 현장 운전이 이루어지는 극저온설
비동 및 헬륨압축기동도 포함한다. 방사선구역인 SCL3/SCL2 LINAC 현장은 운전 장소에서
제외한다.
표 3. 2025년 중이온가속기용 극저온시스템 운전 용역 대상 장치 목록
구분 장치명 장소 비고
극저온 극저온설비동/ 압축기, 재생시스템, 2 K 시스템
SCL3 극저온플랜트
플랜트 헬륨압축기동 등 부대시설 일체
문서번호 : -
2025년도 중이온가속기용 극저온시스템
개정번호 : 0


---

### openai answer
retriever A 점수: 3점
표 1의 용어 설명에 ‘극저온시스템’이 SCL3/SCL2 극저온플랜트 및 부대설비, 분배시스템 및 부대설비, SCL3/SCL2 초전도가속관 저온유지모듈, 기체헬륨 탱크, 액체질소 시스템, 극저온시스템용 헬륨회수시스템을 포함한다고 명시되어 있어 정답의 모든 요소가 들어있습니다.

retriever B 점수: 3점
표 1의 용어 설명에 동일하게 ‘극저온시스템’ 구성(극저온플랜트/분배시스템 및 부대설비, 저온유지모듈, 기체헬륨 탱크, 액체질소 시스템, 헬륨회수시스템 포함)이 명시되어 있어 정답의 모든 요소가 들어있습니다. 

============================================================
### query:
극저온시스템 과업 추진 일정에 따르면, 4분기에 SCL3 구간에 대해 예정된 주요 업무는 무엇이며, 이때의 업무 기간은 '가동기간'과 '휴지기간' 중 어느 것에 해당합니까?

### answer:
2025년 4분기에 SCL3 구간에 예정된 주요 업무는 '유지보수'이며, 이 기간은 '휴지기간'에 해당합니다.

---

### retriever A:
다. 계약자는 본 과업을 수행하기 위해 적절한 지식 또는 능력을 보유하고 있는 적정 수의 운
전 인력을 연구원에게 제공하여야 한다.
• 가동기간의 극저온시스템 운전 업무 (24시간 운전)
- 극저온시스템 모니터링
- 극저온시스템 주요 수치 기록 및 분석
- 극저온시스템 제어
- 기타 가동기간에 필요한 운전 업무
• 휴지기간의 극저온시스템 운전 업무 (주간 운전)
- 유지보수 등을 위한 극저온시스템 제어
- 유지보수 등을 위한 극저온시스템 모니터링
- 유지보수 등을 위한 극저온시스템 주요 수치 기록 및 분석
- 극저온시스템 운전 매뉴얼 작성 및 개정
- 극저온시스템 운전 교육
- 성과보고회 개최
- 기타 휴지기간에 필요한 운전 업무
• 본 과업 및 본 과업을 위해 투입된 인력에 대한 관리감독
본 과업의 기간은 2025년 1월 1일부터 2025년 12월 31일까지로 하며, 2025년 동안 가동기
간의 운전 업무를 수행하는 것을 기준으로 한다. 단, 표 2.와 같이 연구원의 운영 추진 상황
에 따라 휴지기간의 운전업무를 수행하는 것으로 변경될 수 있다. 가동기간과 휴지기간의
운전 업무는 “3. 과업 내용” 하위 절에 기술한 내용에 따른다.
표 2. 과업 추진 일정(안)
2025년
구분
1분기 2분기 3분기 4분기
SCL3 극저온플랜트
SCL3 중이온가속기 SCL3 구간 운전 SCL3 구간 유지보수
시운전 준비 및
구간 (가동기간) (휴지기간)
시운전 (가동기간)
SCL2 SCL2 극저온플랜트 중이온가속기 SCL2 구간 시운전 SCL2 구간 유지보수
구간 시운전 (가동기간) (가동기간) (휴지기간)
회수
2025년
구분
1분기 2분기 3분기 4분기
SCL3 극저온플랜트
SCL3 중이온가속기 SCL3 구간 운전 SCL3 구간 유지보수
시운전 준비 및
구간 (가동기간) (휴지기간)
시운전 (가동기간)
SCL2 SCL2 극저온플랜트 중이온가속기 SCL2 구간 시운전 SCL2 구간 유지보수
구간 시운전 (가동기간) (가동기간) (휴지기간)
회수
필요시 간헐적 운전 (휴지기간에 준함)
시스템
문서번호 : -
2025년도 중이온가속기용 극저온시스템
개정번호 : 0
운전 용역 과업지시서 발 행 일 : 2024. 10. 30
페 이 지 : 7/19
3. 과업 내용
3.1 중이온가속기용 극저온시스템 운전 업무의 정의 및 필요성
본 과업에서 사용하는 “중이온가속기용 극저온시스템 운전 업무”의 정의는, “중이온가
속기용 극저온시스템이 안전하고 안정적으로 운영(가동기간뿐만 아니라 휴지기간의 운영 포
함)될 수 있도록 i) 극저온시스템을 모니터링 및 제어하고, ii) 극저온시스템의 주요 수치를
기록 및 분석하며, iii) 중이온가속기용 극저온시스템의 운전 경험을 통합하여 정리(문서화)하
는 업무를 의미한다.
중이온가속기 또는 극저온시스템 가동기간에는 24시간 운전 업무를 필요로 한다. 통상적으
로 가동기간에는 24시간 운전을 위한 교대근무가 이루어지며, i) 극저온시스템의 모니터링
및 제어, ii) 극저온시스템의 주요 수치를 기록 및 분석하는 업무가 주로 이루어진다.
다만, 통상적으로 이러한 교대근무로 인해 각 운전 인력마다 경험할 수 있는 운전 상황이
□ (업무 자동화) 미래기술 혁신 및 디지털 대전환 가속화를 지원하여 일하는 방식의 혁신을 통한 업무 환경 개선 추진
□ (중장기 비전 부합)「2023 디지털전환 실행과제」에서 도출된 시스템 설계를 반영하여 추후 구축되는 시스템과 호환성 확보
3. 추진 체계
□ 조직도
정보관리총괄
경영본부장
사업총괄
노사⦁경영지원실장
감리
프로젝트 관리자(PM)
협조부서
감리 사업자
디지털기획팀 팀장
경영지원팀,
안전기획팀 등
구축용역 사업자
추진부서
협력업체
시스템 구축
디지털기획팀
시스템 유지보수사업자
□ 인력구성 및 역할
구 분
주요 역할
경영본부장
- 정보화업무 총괄
노사⦁경영지원실장
- 사업 관리 총괄
디지털기획팀
- 사업 운영관리
사업 수행
추진부서
- 사업관리 실무 총괄 (계약관리, 인력 및 일정관리, 행정업무 등)
- 사업 진행사항 관리‧감독 및 중요 의사결정 사항 보고
- 사업추진 관련 유관부서 및 대외기관 협업 주관
구축용역 사업자
- 과업내용에 대한 사업 수행 (시스템 분석, 설계, 개발, 이행 등)
- 품질보증 활동 및 사업 산출물 제출
- 운영자‧사용자 교육 및 기술이전, 하자보수 등
- S/W 개발 및 H/W 환경 구축
사업 지원
협조부서
- 요구사항 도출 및 상세 업무요건 정의
- 업무 프로세스 설계 지원 및 검증
- 업무별 단위테스트, 통합테스트 등 수행 지원
감리 사업자
- 단계별 정보시스템 감리 수행
4. 추진일정
□ 사업기간 : 계약체결일로부터 6개월 (안정화 1개월 포함)
분석·설계
구현
오픈 및 안정화
구 분
M
1
M
2
M
3
M
4
M
5
M
6
구축
사업
시스템 구축
TBx 극저온설비동
밸브박스, 초전도가속관 저온유지
SCL3 터널/
극저온
SCL3 LINAC 모듈 및 밸브박스 진공단열용 진
ISOL 구간
분배
공펌프
시스템 밸브박스, 초전도가속관 저온유지
SCL2 터널/
SCL2 LINAC 모듈 및 밸브박스 진공단열용 진
IF 구간
공펌프
외부 정화기 외부
회수 압축시스템 헬륨압축기동
회수
가스백 헬륨압축기동
시스템
헬륨 탱크 외부 기체 헬륨량 관리 포함
액체 질소 시스템 외부 액체 질소량 관리 포함
3.3 중이온가속기 또는 극저온시스템 가동기간의 운전 업무
계약자는 중이온가속기 또는 극저온시스템 가동기간 동안 24시간 운전업무가 이루어질 수
있도록 운전 인력을 제공해야 한다. 24시간 운전 업무는 2인 1조를 기본으로 하며, 주간에
는 과업 및 운전 인력 감독자가 1인 이상 운전실에 존재해야 한다. 계약자가 가동기간에 수
행해야 하는 운전 업무 상세 내용은 하위 절에 기술된 내용을 따른다. 본 과업지시서에 기
술하지 않은 가동기간의 운전 업무는 연구원과 계약자가 협의하여 수행 여부를 결정한다.
3.3.1 극저온시스템 모니터링
계약자는 가동기간 동안 극저온시스템이 안정적으로 가동될 수 있도록 모니터링을 해야 한
다. 가동기간의 모니터링 업무는 극저온시스템이 안정적으로 가동되고 있는지 여부를 확인
하기 위해 극저온시스템을 지속적으로 관찰하는 업무를 의미하며, 표 4.에 가동기간의 모니
터링 업무에 해당되는 내용들을 기술하였다.
극저온시스템의 모니터링은 연구원에 구축되어있는 극저온시스템용 제어시스템(이하 “제
◦ 사업계획 및 운영계획 수립
◦ 시스템 관련 요구사항 정의 및 조정
◦ 제안요청서 작성 등 사업진행
◦ 사업 단계별 확인 및 산출물 검증
◦ 시스템 검수
연구관리팀장
실무지원
◦ 업무별 기능 개선 사항 요구
◦ 부서별 업무관련 자료 제공
◦ 현업 요구사항 정의 및 검토
◦ TF회의 및 사업 단계별 확인
◦ 시스템 테스트 및 검사
◦ 시스템 인수
현업담당
용역수행
◦ 실행계획 수립
◦ 시스템 구축/개발 및 설치
◦ 테스트 수행 및 유지보수
◦ 산출물 작성
◦ 교육 및 기술 지원 등
용역사업수행자
3. 추진일정
가. 사업기간 : 계약체결일로부터 3개월 이내
나. 추진일정표
구분
M
M+1
M+2
1
2
3
4
1
2
3
4
1
2
3
4
주요
계획
설계
개발
테스트
Open
안정화
※ 상기 일정은 내부 사정에 따라 변경될 수 있음
다. 고려사항
- 제안 업체는 제시한 추진 일정을 참고하여 최적의 구축용역이 수행될 수 있도록 추진 일정계획을 상세하게 제시
- 일정계획 수립 시에는 지연가능 요소를 미리 파악ㆍ분석하여 일정 내에 사업 수행
- 본 사업은 소프트웨어 개발사업 적정 사업기간 산정 기준에 따른 사업임
“[붙임3] 소프트웨어 개발사업 적정 사업기간 산정서” 참조
4. 추진방안
가. 대기업 참여 여부 : 대기업 참여제한 사업
❍ 소프트웨어 진흥법 제48조 (중소 소프트웨어사업자의 사업참여 지원)에 따른 “대기업인 소프트웨어 사업자가 참여 할 수 있는 사업금액의 하한”고시(과학기술정보통신부고시 제2024-9호, 2024.2.28.) 준수
나. 기술 대 가격 비율의 조정
문서번호 : -
2025년도 중이온가속기용 극저온시스템
개정번호 : 0
운전 용역 과업지시서 발 행 일 : 2024. 10. 30
페 이 지 : 12/19
매뉴얼을 작성하고 개정하는 업무를 수행해야 한다. 또한, 과업 및 운전 인력 감독자가 1인
이상 운전실에 배치되어 상기 업무들을 감독해야 한다. 계약자가 휴지기간에 수행해야 하는
운전 업무 상세 내용은 하위 절에 기술된 내용을 따른다. 본 과업지시서에 기술하지 않은
휴지기간의 운전 업무는 연구원과 계약자가 협의하여 수행 여부를 결정한다.
3.4.1 유지보수 등을 위한 극저온시스템 제어
극저온시스템의 제어는 극저온시스템이 휴지지간에 안전하고 안정적으로 운영될 수 있도록
극저온시스템 또는 극저온시스템 관련 부품들을 제어시스템을 활용하여 또는 현장에서 손으
로 조작하는 행위를 의미한다. 계약자는 극저온시스템이 본 목적에 부합하도록 극저온시스
템의 제어를 수행해야 한다. 표 8.에 극저온시스템 제어 업무에 해당되는 사항들을 기술하
였다.
휴지기간의 극저온시스템 제어는 i) 극저온시스템에 대한 충분한 이해를 바탕으로 ii) 조작
후 유지보수 등 현장 활동에 미치는 영향을 충분히 검토한 뒤 이루어져야 한다. 더불어, iii)
조작 대상물이 위치하고 있는 현장에 인력이 존재하는지 여부를 확인하고, iv) 현장에 인력
이 존재할 경우, 조작 대상물을 조작했을 때 발생할 수 있는 상황에 대해 해당 인력의 안전
이 확보된 뒤에 조작을 수행해야 한다. 마지막으로 v) 조작 대상/조작 상태에 대한 혼동이
없도록 각별히 주의를 하여 제어가 진행되어야 한다.
o 업무별 요구사항 확인 등 운영 관리 및 관리 감독
o 사업수행업체 진도관리 및 검수
o 품질 관리 및 의사 결정
o 사업관리자의 권한을 위임받아 사업통제 및 주요 이슈 해결
o 사업진행관리, 프로젝트 매니저 지원, 교육, 인력관리, 변경통제
o 단계별 품질검토 및 결과보고, 기술자문 수행
o 이해 당사자 간 의견 교환 및 조정
o 개인정보 영향평가
정보시스템 감리
o 프로젝트 진행과정 및 산출물 품질에 대한 감리
사업수행업체
o 사업 수행 계획 수립, 추진 및 결과 보고
o 장비 효율성과 서비스 확장성을 고려한 시스템 구축
o 개발인력 관리, 교육지원 등
o 업무 담당자와 커뮤니케이션 및 개발
3. 추진 일정
□ 사업 기간: 계약일로부터 19개월
□ 세부 일정
o 1단계 구축 기간: 2024.04. ~ 2024.10. (약 7개월)
※ 2024.10.25(금) 서비스 오픈 일정
※ 사업자 선정 이후, 우선협상과 병행하여 신속한 분석/설계 인력 투입 필요
o 2단계 구축 기간: 2024.11. ~ 2025.10. (약 12개월)
※ 2025.10.25(금) 서비스 오픈 일정
※※ 요양기관 대상별 점진적 오픈 고려
추진내용
구축 일정
1단계 구축
2단계 구축
M
M1
M2
M3
M4
M5
M6
M7
M8
M9
M10
M11
M12
M13
M14
M15
M16
M17
M18
○ 응용시스템
- 목표시스템 분석․설계
- 개발 및 단위 테스트
- 통합테스트
○ 기반환경 및 시스템구축
- 계획 수립 및 분석
- 개발 환경 구축
- 운영 시스템 구축
- DR 구축
- 시스템 시험
○ 서비스 오픈 및 안정화


---

### retriever B:
다. 계약자는 본 과업을 수행하기 위해 적절한 지식 또는 능력을 보유하고 있는 적정 수의 운
전 인력을 연구원에게 제공하여야 한다.
• 가동기간의 극저온시스템 운전 업무 (24시간 운전)
- 극저온시스템 모니터링
- 극저온시스템 주요 수치 기록 및 분석
- 극저온시스템 제어
- 기타 가동기간에 필요한 운전 업무
• 휴지기간의 극저온시스템 운전 업무 (주간 운전)
- 유지보수 등을 위한 극저온시스템 제어
- 유지보수 등을 위한 극저온시스템 모니터링
- 유지보수 등을 위한 극저온시스템 주요 수치 기록 및 분석
- 극저온시스템 운전 매뉴얼 작성 및 개정
- 극저온시스템 운전 교육
- 성과보고회 개최
- 기타 휴지기간에 필요한 운전 업무
• 본 과업 및 본 과업을 위해 투입된 인력에 대한 관리감독
본 과업의 기간은 2025년 1월 1일부터 2025년 12월 31일까지로 하며, 2025년 동안 가동기
간의 운전 업무를 수행하는 것을 기준으로 한다. 단, 표 2.와 같이 연구원의 운영 추진 상황
에 따라 휴지기간의 운전업무를 수행하는 것으로 변경될 수 있다. 가동기간과 휴지기간의
운전 업무는 “3. 과업 내용” 하위 절에 기술한 내용에 따른다.
표 2. 과업 추진 일정(안)
2025년
구분
1분기 2분기 3분기 4분기
SCL3 극저온플랜트
SCL3 중이온가속기 SCL3 구간 운전 SCL3 구간 유지보수
시운전 준비 및
구간 (가동기간) (휴지기간)
시운전 (가동기간)
SCL2 SCL2 극저온플랜트 중이온가속기 SCL2 구간 시운전 SCL2 구간 유지보수
구간 시운전 (가동기간) (가동기간) (휴지기간)
회수
일정
구분
추진 일정
M
M+1
M+2
M+3
M+4
사업발주 및 사업자 선정
업무 협의
시스템
구축
기존 시스템 분석
업무(요구사항) 분석
설계 및 구축
시스템 구현 및 개발
시스템테스트 및 수정
시스템 가동 및 운영(안정화)
※ 위 추진일정은 사업 추진과정에서 조정 가능함
Ⅲ
다. 추진일정
구 분
2024년~2025년
M1
M2
M3
M4
M5
M6
◦ 시스템 개발
- 분석
- 설계/구현
- 단위/통합 시험
◦ 연계시스템 기능개선
◦ 운영개시 및 안정화
◦ 보고회
- 착수, 중간, 최종보고
★
★
★
* 추진과정에서 일정은 변경될 수 있음
Ⅱ
일반 현황
2. 추진체계
사업총괄기관
안양시
실무지원
사업주관
실무지원
도시공사(운영관리 부서)
(현장 운영지원)
안양시 체육과, 도시공사
도시공사(정보화 부서)
(ICT 기술지원)
설치,구축 수행
계약상대자
구분
주요역할 및 기능
구성
사업주관
○ 사업총괄 추진
○ 시스템 관련 요구사항 정의 및 조정
○ 사업수행 관리, 검수 및 결과평가
실무지원
○ 요구사항 정의 및 검토 지원
○ 업무관련 자료 제공
○ 관련 구축내용 검수 및 평가
○ 관련 업무 운영
·운영관리 부서
·정보화 부서
설치,구축수행
○ 기획/분석 ○ 시스템 구축/개발
○ 테스트 수행 및 하자보수 ○ 교육 및 기술 지원 등 기타 과제수행에 필요한 사항
·계약상대자
楴䵴3. 추진일정
 계약일로부터 120일간
추진내용
M+0
M+1
M+2
M+3
착수
프로젝트 착수
분석
4. 추진 일정
구 분
착수일로부터 5개월
M0
M1
M2
M3
M4
M5
사업자 선정 및
계약체결
시스템 분석 및 설계
통합정보시스템 구축(S/W 개발)
단위 / 통합테스트
시스템 오픈 및 운영지원
사업 관리
품질관리/지원
착수
보고
정례
보고
중간
보고
정례
보고
완료
보고
* 추진일정은 발주기관과 사업수행사 간 상호협의를 통해 변경 가능
4. 추진 일정
구 분
착수일로부터 5개월
M0
M1
M2
M3
M4
M5
사업자 선정 및
계약체결
시스템 분석 및 설계
통합정보시스템 구축(S/W 개발)
단위 / 통합테스트
시스템 오픈 및 운영지원
사업 관리
품질관리/지원
착수
보고
정례
보고
중간
보고
정례
보고
완료
보고
* 추진일정은 발주기관과 사업수행사 간 상호협의를 통해 변경 가능
4. 추진일정
□ 사업기간 : 계약체결일로부터 6개월 (안정화 1개월 포함)
분석·설계
구현
오픈 및 안정화
구 분
M
1
M
2
M
3
M
4
M
5
M
6
구축
사업
시스템 구축
▸ 그룹웨어 시스템 구축
▸ 기록물관리 시스템 구축
▸ 시스템 연계 및 커스터마이징
기록물 이관 및 마이그레이션
▸ 기록물이관 및 마이그레이션
사내SNS 구축
▸ 사내SNS(별별얘기) 시스템 구축
▸ 메신저(내⦁외부) 시스템 구축
▸ 개발지원 / 모니터링 / 운영
SW
상용SW 직접구매
▸ 직접구매 세부계획 수립
▸ 계약 체결 (조달청 나라장터)
▸ 상용SW 설치, 시험 및 안정화
감리 등
감리
▸ 감리(3단계)
▲요구 정의
▲설계
▲종료
보고
정기 보고(착수, 중간, 최종)
★
★
★
※ 상기 추진 일정은 사업추진 과정에서 협의·조정 가능
3. 과업 내용
3.1 중이온가속기용 극저온시스템 운전 업무의 정의 및 필요성
본 과업에서 사용하는 “중이온가속기용 극저온시스템 운전 업무”의 정의는, “중이온가
속기용 극저온시스템이 안전하고 안정적으로 운영(가동기간뿐만 아니라 휴지기간의 운영 포
함)될 수 있도록 i) 극저온시스템을 모니터링 및 제어하고, ii) 극저온시스템의 주요 수치를
기록 및 분석하며, iii) 중이온가속기용 극저온시스템의 운전 경험을 통합하여 정리(문서화)하
는 업무를 의미한다.
중이온가속기 또는 극저온시스템 가동기간에는 24시간 운전 업무를 필요로 한다. 통상적으
로 가동기간에는 24시간 운전을 위한 교대근무가 이루어지며, i) 극저온시스템의 모니터링
및 제어, ii) 극저온시스템의 주요 수치를 기록 및 분석하는 업무가 주로 이루어진다.
다만, 통상적으로 이러한 교대근무로 인해 각 운전 인력마다 경험할 수 있는 운전 상황이
서로 다르게 되고, 서로 다른 경험을 했던 운전 인력 모두가 참여하여 각자의 iii) 운전 경험
을 통합/정리할 수 있는 여건이 형성될 수 없다는 한계가 존재한다. 이러한 운전 경험의 통
합 정리는 후술하는 휴지기간에의 운전업무에 집중적으로 수행 가능하다.
통상적으로 휴지기간에는 극저온시스템의 유지보수 등이 진행되며, 유지보수 등을 위한 제
어, 유지보수 등의 경과 및 결과 확인을 위한 극저온시스템 모니터링, 유지보수 등의 결과
확인을 위한 주요 수치 기록 및 분석 또한 필수적인 운전 업무이다. 휴지기간에는 통상적으
로 주간 근무로 운전이 이루어진다. 이와 같이 주간 근무가 이루어질 수 있는 휴지기간의
통상적으로 휴지기간에는 극저온시스템의 유지보수 등이 진행되며, 유지보수 등을 위한 제
어, 유지보수 등의 경과 및 결과 확인을 위한 극저온시스템 모니터링, 유지보수 등의 결과
확인을 위한 주요 수치 기록 및 분석 또한 필수적인 운전 업무이다. 휴지기간에는 통상적으
로 주간 근무로 운전이 이루어진다. 이와 같이 주간 근무가 이루어질 수 있는 휴지기간의
특성 상, 휴지기간은 가동기간에서 교대근무로 각 운전 인력 개인이 얻은 단편적인 운전 경
험을 통합하여 정리(문서화)할 수 있는 여건이 형성될 수 있는 기간이다.
결과적으로 휴지기간에는 i) 중이온가속기용 극저온시스템의 운전 경험을 통합하여 정리(문
서화)하는 업무가 집중적으로 이루어지며, ii) 유지보수 등을 위한 극저온시스템의 제어와 iii)
유지보수 등을 위한 극저온시스템의 주요 수치 기록 및 분석이 병행되어야 한다.
3.2 운전 대상 및 장소
본 과업의 운전 대상은 표 3.과 같다. 계약자는 표 3. 항목에 대한 운전을 수행해야 한다.
운전 장소는 극저온설비동 2층 제어실에 국한하지 않고, 현장 운전이 이루어지는 극저온설
비동 및 헬륨압축기동도 포함한다. 방사선구역인 SCL3/SCL2 LINAC 현장은 운전 장소에서
제외한다.
표 3. 2025년 중이온가속기용 극저온시스템 운전 용역 대상 장치 목록
구분 장치명 장소 비고
극저온 극저온설비동/ 압축기, 재생시스템, 2 K 시스템
SCL3 극저온플랜트
플랜트 헬륨압축기동 등 부대시설 일체
문서번호 : -
2025년도 중이온가속기용 극저온시스템
개정번호 : 0
4. 추진 전략
□ 사업 목표
○ 쾌적하고 스마트한 수산전문 온라인 쇼핑 환경 구축
○ 시장환경과 소비자 요구에 신속한 대응이 가능한 업무프로세스 수립
○ 중앙회·회원조합 및 계통조직 간 온·오프라인 상생 플랫폼 구축
○ B2C, B2B 사업의 사업성 분석에 따른 온라인 사업영역 재정립
□ 추진 절차
○ (1단계) ISMP(정보시스템마스터플랜) 수립, 데이터 표준화 및 정비
- 명확하고 객관적인 구축계획 수립 및 비용산출을 통한 성공적인 시스템 개발 수행
* 구축 제안요청서(RFP)의 기술적 상세 요구사항 산출
- 네트워크 환경 및 기존 인프라 분석을 통한 최적의 구축 환경 제시
* 구축 환경(온프레미스, 클라우드 등)에 따른 5년간의 손익모델 제시
- 현행 Workflow를 분석하여 To-be 시스템에 적합한 업무프로세스의 확립(PI*)
- 현행 DB구조 분석 및 개선방안 도출을 통한 원활한 데이터 이관 준비
○ (2단계) 시스템 구축 및 개발
- ISMP에 의해 도출된 구축 계획에 따른 신 시스템 개발
- 데이터 이관 방안에 따른 데이터 이관
- 통합 테스트 및 안정화
□ 추진 일정
(단위 : 백만원)
단계
사업기간
추진과제
사업비
계
14개월
4,800 내외
1단계
‘24. 6. ~ ’24.10
(4개월)
ㅇ ISMP(정보시스템마스터플랜) 수립
- To-be시스템의 구조 및 상세기능 설계
- 구축비용 산정, 구축제안서(RFP)작성 및 구축업체 선정 지원
ㅇ 데이터 표준화 및 정비
- 데이터 구조분석·검증 및 목표 모델 설계 제시
300
내외
2단계
‘24.12 ~ ’25.10
(10개월)
가. 사업추진 일정(총 사업 수행기간: 8개월)
일 정
내 용
M1
M2
M3
M4
M5
M6
M7
M8
M9
M10
분석
설계
개발/테스트/완료
오픈 및
안정화
시스템 구축
시스템 운영
품질 관리
보고 관리
착수
보고
중간
보고
완료
보고
데이터 이관
운영 교육
※ 사업추진 일정은 사업수행업체의 개발계획에 따라 대학과 협의하여 일정 조정 진행
Ⅳ
3. 사업 추진 일정
ㅇ 사업 추진 일정
추진 내용
사업기간
M
M+1
M+2
M+3
체육특기자 경기기록 관리시스템 개발
분석
설계
구현
사전 검수
통합테스트
시범 운영
※ 추진일정은 사업일정에 따라 변경될 수 있으며, 변경 시 사전 협의 필수


---

### openai answer
retriever A 점수: 3점
표 2의 과업 추진 일정에 4분기 SCL3 구간이 “유지보수(휴지기간)”로 명시되어 있어, 정답의 내용(주요 업무=유지보수, 기간=휴지기간)을 모두 포함합니다.

retriever B 점수: 3점
표 2의 과업 추진 일정에 4분기 SCL3 구간이 “유지보수(휴지기간)”로 명시되어 있어, 정답의 내용(주요 업무=유지보수, 기간=휴지기간)을 모두 포함합니다. 

============================================================
### query:
극저온시스템 '가동기간' 동안 계약자가 24시간 수행해야 할 극저온시스템 모니터링 업무에는 어떤 것들이 있습니까? (3가지 이상)

### answer:
가동기간 동안 수행해야 할 모니터링 업무는 다음과 같습니다.
알람 메시지 확인
주요 센서(온도, 압력, 유량 등)의 트렌드 확인
정상 운전 상황의 주요 센서 지시값 파악 및 숙지
비정상 운전 상황 감지
제어로직 정상 작동 여부 확인

---

### retriever A:
다. 가동기간의 모니터링 업무는 극저온시스템이 안정적으로 가동되고 있는지 여부를 확인
하기 위해 극저온시스템을 지속적으로 관찰하는 업무를 의미하며, 표 4.에 가동기간의 모니
터링 업무에 해당되는 내용들을 기술하였다.
극저온시스템의 모니터링은 연구원에 구축되어있는 극저온시스템용 제어시스템(이하 “제
어시스템”)을 활용하여 또는 현장에 설치되어있는 장치를 통해 수행한다. 연구원은 구축되
어있는 제어시스템을 활용하기 위한 컴퓨터, 모니터, 키보드, 마우스, 책상 및 의자를 계약
자에게 제공한다. 현재 연구원에 구축되어있는 제어시스템은 표 5.과 같다.
연구원은 계약자에게 현재 구축되어있는 제어시스템에 대한 기본 사용 교육을 제공하며,
계약자는 연구원이 제공하는 교육을 이해하기 위한 배경지식을 보유해야 한다.
문서번호 : -
2025년도 중이온가속기용 극저온시스템
개정번호 : 0
운전 용역 과업지시서 발 행 일 : 2024. 10. 30
페 이 지 : 9/19
표 4. 가동기간의 극저온시스템 모니터링 업무 리스트
순번 극저온시스템 모니터링 업무 비고
1 알람 메시지 확인
2 주요 센서 트렌드 확인 온도, 압력, 유량, 속도 등
3 정상 운전 상황의 주요 센서 지시값 파악 및 숙지 온도, 압력, 유량, 속도 등
4 비정상 운전 상황 감지
연구원에서 현장 확인 필요상황 발생 시, 연구원이 지 무전 통신 활용
5
정하는 값 관찰, 확인 및 상황 전달 (연구원에서 무전기 제공)
6 제어로직 정상 작동 여부 확인
연구원 요청 시,
7 현장 상황 일시적 또는 지속적 관찰
비방사선구역 한정
이온가속기용 극저온시스템 운영에 중대한 영향이 생길 경우, 이는 계약자의 책임으로 한
다.
표 8. 휴지기간의 극저온시스템 제어 업무 리스트
순번 극저온시스템 제어 업무 비고
자동/수동(Auto/Manual) 제어 우선
1 계장류 제어 우선권(Priority) 변경
권 설정 변경
문서번호 : -
2025년도 중이온가속기용 극저온시스템
개정번호 : 0
운전 용역 과업지시서 발 행 일 : 2024. 10. 30
페 이 지 : 13/19
자동 밸브 – 제어시스템 활용
2 극저온시스템 자동/수동 밸브 조작
수동 밸브 – 현장에서 손으로 조작
압축기 emergency stop 버튼
3 현장에서 손으로 조작
활성화/비활성화
제어시스템 활용
4 진공펌프 가동/정지
(현장 조작 필요 시 연구원에 통지)
5 센서 오버라이드(Override) 설정 또는 해제 제어시스템 활용
6 계장류 제어 변수값(Ramp speed 등) 변경 제어시스템 활용
7 시퀀스 제어 변수값(Timer 등) 변경 제어시스템 활용
8 시퀀스 시작/종료 제어시스템 활용
현장 확인 및 최종 준비 상태 미진
9 극저온시스템 가동 전 최종 준비 상태 확인
할 경우, 상태 보완 포함
기타 극저온시스템의 안전하고 안정적인 운
10 제어시스템 활용 및 현장 조작 포함
전에 필요한 제어
3.4.2 유지보수 등을 위한 극저온시스템 모니터링
계약자는 휴지기간 동안 극저온시스템이 안정적으로 가동될 수 있도록 모니터링을 해야 한
다. 휴지기간의 모니터링 업무는 i) 유지보수 등을 위한 현장 활동 시 극저온시스템에 특이
TBx 극저온설비동
밸브박스, 초전도가속관 저온유지
SCL3 터널/
극저온
SCL3 LINAC 모듈 및 밸브박스 진공단열용 진
ISOL 구간
분배
공펌프
시스템 밸브박스, 초전도가속관 저온유지
SCL2 터널/
SCL2 LINAC 모듈 및 밸브박스 진공단열용 진
IF 구간
공펌프
외부 정화기 외부
회수 압축시스템 헬륨압축기동
회수
가스백 헬륨압축기동
시스템
헬륨 탱크 외부 기체 헬륨량 관리 포함
액체 질소 시스템 외부 액체 질소량 관리 포함
3.3 중이온가속기 또는 극저온시스템 가동기간의 운전 업무
계약자는 중이온가속기 또는 극저온시스템 가동기간 동안 24시간 운전업무가 이루어질 수
있도록 운전 인력을 제공해야 한다. 24시간 운전 업무는 2인 1조를 기본으로 하며, 주간에
는 과업 및 운전 인력 감독자가 1인 이상 운전실에 존재해야 한다. 계약자가 가동기간에 수
행해야 하는 운전 업무 상세 내용은 하위 절에 기술된 내용을 따른다. 본 과업지시서에 기
술하지 않은 가동기간의 운전 업무는 연구원과 계약자가 협의하여 수행 여부를 결정한다.
3.3.1 극저온시스템 모니터링
계약자는 가동기간 동안 극저온시스템이 안정적으로 가동될 수 있도록 모니터링을 해야 한
다. 가동기간의 모니터링 업무는 극저온시스템이 안정적으로 가동되고 있는지 여부를 확인
하기 위해 극저온시스템을 지속적으로 관찰하는 업무를 의미하며, 표 4.에 가동기간의 모니
터링 업무에 해당되는 내용들을 기술하였다.
극저온시스템의 모니터링은 연구원에 구축되어있는 극저온시스템용 제어시스템(이하 “제
할 경우, 상태 보완 포함
기타 극저온시스템의 안전하고 안정적인 운
10 제어시스템 활용 및 현장 조작 포함
전에 필요한 제어
3.4.2 유지보수 등을 위한 극저온시스템 모니터링
계약자는 휴지기간 동안 극저온시스템이 안정적으로 가동될 수 있도록 모니터링을 해야 한
다. 휴지기간의 모니터링 업무는 i) 유지보수 등을 위한 현장 활동 시 극저온시스템에 특이
사항이 발생하는지 여부를 확인하기 위해 또는 ii) 유지보수 등을 수행한 뒤 그 결과를 확인
하기 위해 극저온시스템을 지속적으로 관찰하는 업무를 의미한다. 표 9.에 휴지기간의 모니
터링 업무에 해당되는 내용들을 기술하였다.
극저온시스템의 모니터링은 연구원에 구축되어있는 극저온시스템용 제어시스템(이하 “제
어시스템”)을 활용하여 또는 현장에 설치되어있는 장치를 통해 수행한다. 연구원은 구축되
어있는 제어시스템을 활용하기 위한 컴퓨터, 모니터, 키보드, 마우스, 책상 및 의자를 계약
자에게 제공한다. 현재 연구원에 구축되어있는 제어시스템은 표 5.과 같다.
연구원은 계약자에게 현재 구축되어있는 제어시스템에 대한 기본 사용 교육을 제공하며,
계약자는 연구원이 제공하는 교육을 이해하기 위한 배경지식을 보유해야 한다.
표 9. 휴지기간의 극저온시스템 모니터링 업무 리스트
순번 극저온시스템 모니터링 업무 비고
1 알람 메시지 확인
2 주요 센서 트렌드 확인 온도, 압력, 유량, 속도 등
3 정상 상황의 주요 센서 지시값 파악 및 숙지 온도, 압력, 유량, 속도 등
4 비정상 상황 감지
연구원에서 현장 확인 필요상황 발생 시, 연구원이 지 무전 통신 활용
5
중이온가속기 또는 극저온시스템 가동기간에는 24시간 운전 업무를 필요로 한다. 통상적으
로 가동기간에는 24시간 운전을 위한 교대근무가 이루어지며, i) 극저온시스템의 모니터링
및 제어, ii) 극저온시스템의 주요 수치를 기록 및 분석하는 업무가 주로 이루어진다.
다만, 통상적으로 이러한 교대근무로 인해 각 운전 인력마다 경험할 수 있는 운전 상황이
서로 다르게 되고, 서로 다른 경험을 했던 운전 인력 모두가 참여하여 각자의 iii) 운전 경험
을 통합/정리할 수 있는 여건이 형성될 수 없다는 한계가 존재한다. 이러한 운전 경험의 통
합 정리는 후술하는 휴지기간에의 운전업무에 집중적으로 수행 가능하다.
통상적으로 휴지기간에는 극저온시스템의 유지보수 등이 진행되며, 유지보수 등을 위한 제
어, 유지보수 등의 경과 및 결과 확인을 위한 극저온시스템 모니터링, 유지보수 등의 결과
확인을 위한 주요 수치 기록 및 분석 또한 필수적인 운전 업무이다. 휴지기간에는 통상적으
로 주간 근무로 운전이 이루어진다. 이와 같이 주간 근무가 이루어질 수 있는 휴지기간의
특성 상, 휴지기간은 가동기간에서 교대근무로 각 운전 인력 개인이 얻은 단편적인 운전 경
험을 통합하여 정리(문서화)할 수 있는 여건이 형성될 수 있는 기간이다.
결과적으로 휴지기간에는 i) 중이온가속기용 극저온시스템의 운전 경험을 통합하여 정리(문
서화)하는 업무가 집중적으로 이루어지며, ii) 유지보수 등을 위한 극저온시스템의 제어와 iii)
유지보수 등을 위한 극저온시스템의 주요 수치 기록 및 분석이 병행되어야 한다.
3.2 운전 대상 및 장소
교대근무 인력들도 회람 필수
4 일일(Daily) 운영 상황 공유 회의 참석 유사 시 계약자가 진행 가능
일일근무일지 작성 당일 현장/제어 관련 발생 상
5
(연구원 양식 제공) 황 기록
i) 교대 변경 전 특이사항 또
교대근무자 변경 시 인수인계 및 인수인계서 작성
6 는 관찰 필요 사항 전달
(연구원 양식 제공)
ii) 성실교대근무 서약서 작성
운전 중 수리가 필요한 사항 발생 시 별도 기록
7
(연구원 양식 제공)
극저온시스템용 제어시스템(알람시스템 포함) 개선
8 연구원 요청 시 한정
및 보완을 위한 협조
기타 극저온시스템의 안전하고 안정적인 운영을 위 연구원과 계약자가 협의하여
9
해 필요한 업무 수행 여부 결정
3.4 중이온가속기 또는 극저온시스템 휴지기간의 운전 업무
계약자는 중이온가속기 또는 극저온시스템 휴지기간 동안 주간 운전업무가 이루어질 수 있
도록 운전 인력을 제공해야 한다. 주간 운전 업무는 운전인력 전원 운전을 기본으로 한다.
운전인력 중 1인 또는 2인은 유지보수 등을 위한 극저온시스템 모니터링 및 제어를 수행해
야 하며, 나머지 인력들은 각 운전 인력이 경험했던 상황들을 취합하여 극저온시스템 운전
문서번호 : -
2025년도 중이온가속기용 극저온시스템
개정번호 : 0
운전 용역 과업지시서 발 행 일 : 2024. 10. 30
페 이 지 : 12/19
매뉴얼을 작성하고 개정하는 업무를 수행해야 한다. 또한, 과업 및 운전 인력 감독자가 1인
이상 운전실에 배치되어 상기 업무들을 감독해야 한다. 계약자가 휴지기간에 수행해야 하는
표 9. 휴지기간의 극저온시스템 모니터링 업무 리스트
순번 극저온시스템 모니터링 업무 비고
1 알람 메시지 확인
2 주요 센서 트렌드 확인 온도, 압력, 유량, 속도 등
3 정상 상황의 주요 센서 지시값 파악 및 숙지 온도, 압력, 유량, 속도 등
4 비정상 상황 감지
연구원에서 현장 확인 필요상황 발생 시, 연구원이 지 무전 통신 활용
5
정하는 값 관찰, 확인 및 상황 전달 (연구원에서 무전기 제공)
6 제어로직 정상 작동 여부 확인
문서번호 : -
2025년도 중이온가속기용 극저온시스템
개정번호 : 0
운전 용역 과업지시서 발 행 일 : 2024. 10. 30
페 이 지 : 14/19
현장 상황 일시적 또는 지속적 관찰 연구원 요청 시,
7
(유지보수 활동 내용 파악 및 진행 상황 파악 포함) 비방사선구역 한정
8 기타 극저온시스템 운전 상황 확인에 필요한 모니터링 주기적인 현장 순찰 포함
3.4.3 유지보수 등을 위한 극저온시스템 주요 수치/상황 기록 및 분석
계약자는 극저온시스템의 유지보수 상황을 파악하여 기록하고, 제어 화면 및 현장 센서에
서 지시하는 주요 수치들을 기록해야 한다. 유지보수 상황은 상황 변동 시마다 기록하며,
주요 수치는 일정 시간 간격마다 주기적으로 기록한다. 기록 형식은 전자파일로 한다.
기록이 필요한 주요 수치 항목 및 수치 기록 간격은 연구원이 정하는 바에 따른다. 본 수
치 항목과 기록 간격은 극저온시스템의 운영 상황에 따라 증감될 수 있다.
더불어, 연구원의 요청이 있는 경우, 계약자는 주요 수치들에 대한 분석을 수행해야 한다.


---

### retriever B:
다. 가동기간의 모니터링 업무는 극저온시스템이 안정적으로 가동되고 있는지 여부를 확인
하기 위해 극저온시스템을 지속적으로 관찰하는 업무를 의미하며, 표 4.에 가동기간의 모니
터링 업무에 해당되는 내용들을 기술하였다.
극저온시스템의 모니터링은 연구원에 구축되어있는 극저온시스템용 제어시스템(이하 “제
어시스템”)을 활용하여 또는 현장에 설치되어있는 장치를 통해 수행한다. 연구원은 구축되
어있는 제어시스템을 활용하기 위한 컴퓨터, 모니터, 키보드, 마우스, 책상 및 의자를 계약
자에게 제공한다. 현재 연구원에 구축되어있는 제어시스템은 표 5.과 같다.
연구원은 계약자에게 현재 구축되어있는 제어시스템에 대한 기본 사용 교육을 제공하며,
계약자는 연구원이 제공하는 교육을 이해하기 위한 배경지식을 보유해야 한다.
문서번호 : -
2025년도 중이온가속기용 극저온시스템
개정번호 : 0
운전 용역 과업지시서 발 행 일 : 2024. 10. 30
페 이 지 : 9/19
표 4. 가동기간의 극저온시스템 모니터링 업무 리스트
순번 극저온시스템 모니터링 업무 비고
1 알람 메시지 확인
2 주요 센서 트렌드 확인 온도, 압력, 유량, 속도 등
3 정상 운전 상황의 주요 센서 지시값 파악 및 숙지 온도, 압력, 유량, 속도 등
4 비정상 운전 상황 감지
연구원에서 현장 확인 필요상황 발생 시, 연구원이 지 무전 통신 활용
5
정하는 값 관찰, 확인 및 상황 전달 (연구원에서 무전기 제공)
6 제어로직 정상 작동 여부 확인
연구원 요청 시,
7 현장 상황 일시적 또는 지속적 관찰
비방사선구역 한정
는 과업 및 운전 인력 감독자가 1인 이상 운전실에 존재해야 한다. 계약자가 가동기간에 수
행해야 하는 운전 업무 상세 내용은 하위 절에 기술된 내용을 따른다. 본 과업지시서에 기
술하지 않은 가동기간의 운전 업무는 연구원과 계약자가 협의하여 수행 여부를 결정한다.
3.3.1 극저온시스템 모니터링
계약자는 가동기간 동안 극저온시스템이 안정적으로 가동될 수 있도록 모니터링을 해야 한
다. 휴지기간의 모니터링 업무는 i) 유지보수 등을 위한 현장 활동 시 극저온시스템에 특이
사항이 발생하는지 여부를 확인하기 위해 또는 ii) 유지보수 등을 수행한 뒤 그 결과를 확인
하기 위해 극저온시스템을 지속적으로 관찰하는 업무를 의미한다. 표 9.에 휴지기간의 모니
터링 업무에 해당되는 내용들을 기술하였다.
극저온시스템의 모니터링은 연구원에 구축되어있는 극저온시스템용 제어시스템(이하 “제
어시스템”)을 활용하여 또는 현장에 설치되어있는 장치를 통해 수행한다. 연구원은 구축되
어있는 제어시스템을 활용하기 위한 컴퓨터, 모니터, 키보드, 마우스, 책상 및 의자를 계약
자에게 제공한다. 현재 연구원에 구축되어있는 제어시스템은 표 5.과 같다.
연구원은 계약자에게 현재 구축되어있는 제어시스템에 대한 기본 사용 교육을 제공하며,
계약자는 연구원이 제공하는 교육을 이해하기 위한 배경지식을 보유해야 한다.
표 9. 휴지기간의 극저온시스템 모니터링 업무 리스트
순번 극저온시스템 모니터링 업무 비고
1 알람 메시지 확인
2 주요 센서 트렌드 확인 온도, 압력, 유량, 속도 등
3 정상 상황의 주요 센서 지시값 파악 및 숙지 온도, 압력, 유량, 속도 등
4 비정상 상황 감지
연구원에서 현장 확인 필요상황 발생 시, 연구원이 지 무전 통신 활용
5
정하는 값 관찰, 확인 및 상황 전달 (연구원에서 무전기 제공)
6 제어로직 정상 작동 여부 확인
문서번호 : -
2025년도 중이온가속기용 극저온시스템
개정번호 : 0
운전 용역 과업지시서 발 행 일 : 2024. 10. 30
페 이 지 : 14/19
이온가속기용 극저온시스템 운영에 중대한 영향이 생길 경우, 이는 계약자의 책임으로 한
다.
표 8. 휴지기간의 극저온시스템 제어 업무 리스트
순번 극저온시스템 제어 업무 비고
자동/수동(Auto/Manual) 제어 우선
1 계장류 제어 우선권(Priority) 변경
권 설정 변경
문서번호 : -
2025년도 중이온가속기용 극저온시스템
개정번호 : 0
운전 용역 과업지시서 발 행 일 : 2024. 10. 30
페 이 지 : 13/19
자동 밸브 – 제어시스템 활용
2 극저온시스템 자동/수동 밸브 조작
수동 밸브 – 현장에서 손으로 조작
압축기 emergency stop 버튼
3 현장에서 손으로 조작
활성화/비활성화
제어시스템 활용
4 진공펌프 가동/정지
(현장 조작 필요 시 연구원에 통지)
5 센서 오버라이드(Override) 설정 또는 해제 제어시스템 활용
6 계장류 제어 변수값(Ramp speed 등) 변경 제어시스템 활용
7 시퀀스 제어 변수값(Timer 등) 변경 제어시스템 활용
8 시퀀스 시작/종료 제어시스템 활용
현장 확인 및 최종 준비 상태 미진
9 극저온시스템 가동 전 최종 준비 상태 확인
할 경우, 상태 보완 포함
기타 극저온시스템의 안전하고 안정적인 운
10 제어시스템 활용 및 현장 조작 포함
전에 필요한 제어
3.4.2 유지보수 등을 위한 극저온시스템 모니터링
계약자는 휴지기간 동안 극저온시스템이 안정적으로 가동될 수 있도록 모니터링을 해야 한
교대근무 인력들도 회람 필수
4 일일(Daily) 운영 상황 공유 회의 참석 유사 시 계약자가 진행 가능
일일근무일지 작성 당일 현장/제어 관련 발생 상
5
(연구원 양식 제공) 황 기록
i) 교대 변경 전 특이사항 또
교대근무자 변경 시 인수인계 및 인수인계서 작성
6 는 관찰 필요 사항 전달
(연구원 양식 제공)
ii) 성실교대근무 서약서 작성
운전 중 수리가 필요한 사항 발생 시 별도 기록
7
(연구원 양식 제공)
극저온시스템용 제어시스템(알람시스템 포함) 개선
8 연구원 요청 시 한정
및 보완을 위한 협조
기타 극저온시스템의 안전하고 안정적인 운영을 위 연구원과 계약자가 협의하여
9
해 필요한 업무 수행 여부 결정
3.4 중이온가속기 또는 극저온시스템 휴지기간의 운전 업무
계약자는 중이온가속기 또는 극저온시스템 휴지기간 동안 주간 운전업무가 이루어질 수 있
도록 운전 인력을 제공해야 한다. 주간 운전 업무는 운전인력 전원 운전을 기본으로 한다.
운전인력 중 1인 또는 2인은 유지보수 등을 위한 극저온시스템 모니터링 및 제어를 수행해
야 하며, 나머지 인력들은 각 운전 인력이 경험했던 상황들을 취합하여 극저온시스템 운전
문서번호 : -
2025년도 중이온가속기용 극저온시스템
개정번호 : 0
운전 용역 과업지시서 발 행 일 : 2024. 10. 30
페 이 지 : 12/19
매뉴얼을 작성하고 개정하는 업무를 수행해야 한다. 또한, 과업 및 운전 인력 감독자가 1인
이상 운전실에 배치되어 상기 업무들을 감독해야 한다. 계약자가 휴지기간에 수행해야 하는
이온가속기용 극저온시스템 운영에 중대한 영향이 생길 경우, 계약자의 책임으로 한다.
표 6. 가동기간의 극저온시스템 제어 업무 리스트
순번 극저온시스템 제어 업무 비고
자동/수동(Auto/Manual) 제어 우선
1 계장류 제어 우선권(Priority) 변경
권 설정 변경
자동 밸브 – 제어시스템 활용
2 극저온시스템 자동/수동 밸브 조작
수동 밸브 – 현장에서 손으로 조작
3 히터 조작 제어시스템 활용
4 액체 헬륨 레벨미터 모니터 측정 방식 변경 현장에서 설정 변경
제어시스템 활용
5 진공펌프 가동/정지
(현장 조작 필요 시 연구원에 통지)
6 센서 오버라이드(Override) 설정 또는 해제 제어시스템 활용
7 계장류 제어 변수값(Ramp speed 등) 변경 제어시스템 활용
8 시퀀스 제어 변수값(Timer 등) 변경 제어시스템 활용
9 시퀀스 시작/종료 제어시스템 활용
기타 극저온시스템의 안전하고 안정적인 운
10 제어시스템 활용 및 현장 조작 포함
전에 필요한 제어
문서번호 : -
2025년도 중이온가속기용 극저온시스템
개정번호 : 0
운전 용역 과업지시서 발 행 일 : 2024. 10. 30
페 이 지 : 11/19
3.3.4 기타 가동기간에 필요한 운전 업무
표 7.에 기술한 기타 가동기간에 필요한 운전 업무 또한 계약자의 업무 범위이다. 표 7.에
기술된 내용 중, 위험(긴급)상황 발생 시, 계약자가 i) 지체없이 연구원에게 상황을 전파하고,
ii) 성실히 연구원에서 전달한 긴급조치 사항을 수행했음에도 불구하고 발생한 문제들은 계
제외한다.
표 3. 2025년 중이온가속기용 극저온시스템 운전 용역 대상 장치 목록
구분 장치명 장소 비고
극저온 극저온설비동/ 압축기, 재생시스템, 2 K 시스템
SCL3 극저온플랜트
플랜트 헬륨압축기동 등 부대시설 일체
문서번호 : -
2025년도 중이온가속기용 극저온시스템
개정번호 : 0
운전 용역 과업지시서 발 행 일 : 2024. 10. 30
페 이 지 : 8/19
극저온설비동/ 압축기, 재생시스템, 2 K 시스템
SCL2 극저온플랜트
헬륨압축기동 등 부대시설 일체
TBx 극저온설비동
밸브박스, 초전도가속관 저온유지
SCL3 터널/
극저온
SCL3 LINAC 모듈 및 밸브박스 진공단열용 진
ISOL 구간
분배
공펌프
시스템 밸브박스, 초전도가속관 저온유지
SCL2 터널/
SCL2 LINAC 모듈 및 밸브박스 진공단열용 진
IF 구간
공펌프
외부 정화기 외부
회수 압축시스템 헬륨압축기동
회수
가스백 헬륨압축기동
시스템
헬륨 탱크 외부 기체 헬륨량 관리 포함
액체 질소 시스템 외부 액체 질소량 관리 포함
3.3 중이온가속기 또는 극저온시스템 가동기간의 운전 업무
계약자는 중이온가속기 또는 극저온시스템 가동기간 동안 24시간 운전업무가 이루어질 수
있도록 운전 인력을 제공해야 한다. 24시간 운전 업무는 2인 1조를 기본으로 하며, 주간에
는 과업 및 운전 인력 감독자가 1인 이상 운전실에 존재해야 한다. 계약자가 가동기간에 수
행해야 하는 운전 업무 상세 내용은 하위 절에 기술된 내용을 따른다. 본 과업지시서에 기
술하지 않은 가동기간의 운전 업무는 연구원과 계약자가 협의하여 수행 여부를 결정한다.
3. 과업 내용
3.1 중이온가속기용 극저온시스템 운전 업무의 정의 및 필요성
본 과업에서 사용하는 “중이온가속기용 극저온시스템 운전 업무”의 정의는, “중이온가
속기용 극저온시스템이 안전하고 안정적으로 운영(가동기간뿐만 아니라 휴지기간의 운영 포
함)될 수 있도록 i) 극저온시스템을 모니터링 및 제어하고, ii) 극저온시스템의 주요 수치를
기록 및 분석하며, iii) 중이온가속기용 극저온시스템의 운전 경험을 통합하여 정리(문서화)하
는 업무를 의미한다.
중이온가속기 또는 극저온시스템 가동기간에는 24시간 운전 업무를 필요로 한다. 통상적으
로 가동기간에는 24시간 운전을 위한 교대근무가 이루어지며, i) 극저온시스템의 모니터링
및 제어, ii) 극저온시스템의 주요 수치를 기록 및 분석하는 업무가 주로 이루어진다.
다만, 통상적으로 이러한 교대근무로 인해 각 운전 인력마다 경험할 수 있는 운전 상황이
서로 다르게 되고, 서로 다른 경험을 했던 운전 인력 모두가 참여하여 각자의 iii) 운전 경험
을 통합/정리할 수 있는 여건이 형성될 수 없다는 한계가 존재한다. 이러한 운전 경험의 통
합 정리는 후술하는 휴지기간에의 운전업무에 집중적으로 수행 가능하다.
통상적으로 휴지기간에는 극저온시스템의 유지보수 등이 진행되며, 유지보수 등을 위한 제
어, 유지보수 등의 경과 및 결과 확인을 위한 극저온시스템 모니터링, 유지보수 등의 결과
확인을 위한 주요 수치 기록 및 분석 또한 필수적인 운전 업무이다. 휴지기간에는 통상적으
로 주간 근무로 운전이 이루어진다. 이와 같이 주간 근무가 이루어질 수 있는 휴지기간의


---

### openai answer
retriever A 점수: 3점
(표 4에 가동기간 모니터링 업무로 알람 메시지 확인, 주요 센서 트렌드 확인(온도·압력·유량 등), 정상 운전 센서 지시값 파악·숙지, 비정상 운전 상황 감지, 제어로직 정상 작동 여부 확인이 모두 명시되어 있어 answer 전부를 충족)

retriever B 점수: 3점
(가동기간 모니터링 업무 리스트에 알람 메시지 확인, 주요 센서 트렌드 확인, 정상 운전 센서 지시값 파악·숙지, 비정상 운전 상황 감지, 제어로직 정상 작동 여부 확인이 모두 포함되어 있어 answer 전부를 충족) 

============================================================
### query:
중이온가속기 극저온시스템의 '가동기간'과 '휴지기간'의 운전 업무 형태(근무 방식)는 어떻게 다릅니까?

### answer:
'가동기간'에는 시스템의 24시간 안정적인 운영을 위해 교대근무(24시간 운전)가 이루어집니다. 반면, '휴지기간'에는 주로 시스템 유지보수 등을 위해 주간 운전을 기본으로 합니다.

---

### retriever A:
중이온가속기 또는 극저온시스템 가동기간에는 24시간 운전 업무를 필요로 한다. 통상적으
로 가동기간에는 24시간 운전을 위한 교대근무가 이루어지며, i) 극저온시스템의 모니터링
및 제어, ii) 극저온시스템의 주요 수치를 기록 및 분석하는 업무가 주로 이루어진다.
다만, 통상적으로 이러한 교대근무로 인해 각 운전 인력마다 경험할 수 있는 운전 상황이
서로 다르게 되고, 서로 다른 경험을 했던 운전 인력 모두가 참여하여 각자의 iii) 운전 경험
을 통합/정리할 수 있는 여건이 형성될 수 없다는 한계가 존재한다. 이러한 운전 경험의 통
합 정리는 후술하는 휴지기간에의 운전업무에 집중적으로 수행 가능하다.
통상적으로 휴지기간에는 극저온시스템의 유지보수 등이 진행되며, 유지보수 등을 위한 제
어, 유지보수 등의 경과 및 결과 확인을 위한 극저온시스템 모니터링, 유지보수 등의 결과
확인을 위한 주요 수치 기록 및 분석 또한 필수적인 운전 업무이다. 휴지기간에는 통상적으
로 주간 근무로 운전이 이루어진다. 이와 같이 주간 근무가 이루어질 수 있는 휴지기간의
특성 상, 휴지기간은 가동기간에서 교대근무로 각 운전 인력 개인이 얻은 단편적인 운전 경
험을 통합하여 정리(문서화)할 수 있는 여건이 형성될 수 있는 기간이다.
결과적으로 휴지기간에는 i) 중이온가속기용 극저온시스템의 운전 경험을 통합하여 정리(문
서화)하는 업무가 집중적으로 이루어지며, ii) 유지보수 등을 위한 극저온시스템의 제어와 iii)
유지보수 등을 위한 극저온시스템의 주요 수치 기록 및 분석이 병행되어야 한다.
3.2 운전 대상 및 장소
문서번호 : -
2025년도 중이온가속기용 극저온시스템
개정번호 : 0
운전 용역 과업지시서 발 행 일 : 2024. 10. 30
페 이 지 : 12/19
매뉴얼을 작성하고 개정하는 업무를 수행해야 한다. 또한, 과업 및 운전 인력 감독자가 1인
이상 운전실에 배치되어 상기 업무들을 감독해야 한다. 계약자가 휴지기간에 수행해야 하는
운전 업무 상세 내용은 하위 절에 기술된 내용을 따른다. 본 과업지시서에 기술하지 않은
휴지기간의 운전 업무는 연구원과 계약자가 협의하여 수행 여부를 결정한다.
3.4.1 유지보수 등을 위한 극저온시스템 제어
극저온시스템의 제어는 극저온시스템이 휴지지간에 안전하고 안정적으로 운영될 수 있도록
극저온시스템 또는 극저온시스템 관련 부품들을 제어시스템을 활용하여 또는 현장에서 손으
로 조작하는 행위를 의미한다. 계약자는 극저온시스템이 본 목적에 부합하도록 극저온시스
템의 제어를 수행해야 한다. 표 8.에 극저온시스템 제어 업무에 해당되는 사항들을 기술하
였다.
휴지기간의 극저온시스템 제어는 i) 극저온시스템에 대한 충분한 이해를 바탕으로 ii) 조작
후 유지보수 등 현장 활동에 미치는 영향을 충분히 검토한 뒤 이루어져야 한다. 더불어, iii)
조작 대상물이 위치하고 있는 현장에 인력이 존재하는지 여부를 확인하고, iv) 현장에 인력
이 존재할 경우, 조작 대상물을 조작했을 때 발생할 수 있는 상황에 대해 해당 인력의 안전
이 확보된 뒤에 조작을 수행해야 한다. 마지막으로 v) 조작 대상/조작 상태에 대한 혼동이
없도록 각별히 주의를 하여 제어가 진행되어야 한다.
이온가속기용 극저온시스템 운영에 중대한 영향이 생길 경우, 계약자의 책임으로 한다.
표 6. 가동기간의 극저온시스템 제어 업무 리스트
순번 극저온시스템 제어 업무 비고
자동/수동(Auto/Manual) 제어 우선
1 계장류 제어 우선권(Priority) 변경
권 설정 변경
자동 밸브 – 제어시스템 활용
2 극저온시스템 자동/수동 밸브 조작
수동 밸브 – 현장에서 손으로 조작
3 히터 조작 제어시스템 활용
4 액체 헬륨 레벨미터 모니터 측정 방식 변경 현장에서 설정 변경
제어시스템 활용
5 진공펌프 가동/정지
(현장 조작 필요 시 연구원에 통지)
6 센서 오버라이드(Override) 설정 또는 해제 제어시스템 활용
7 계장류 제어 변수값(Ramp speed 등) 변경 제어시스템 활용
8 시퀀스 제어 변수값(Timer 등) 변경 제어시스템 활용
9 시퀀스 시작/종료 제어시스템 활용
기타 극저온시스템의 안전하고 안정적인 운
10 제어시스템 활용 및 현장 조작 포함
전에 필요한 제어
문서번호 : -
2025년도 중이온가속기용 극저온시스템
개정번호 : 0
운전 용역 과업지시서 발 행 일 : 2024. 10. 30
페 이 지 : 11/19
3.3.4 기타 가동기간에 필요한 운전 업무
표 7.에 기술한 기타 가동기간에 필요한 운전 업무 또한 계약자의 업무 범위이다. 표 7.에
기술된 내용 중, 위험(긴급)상황 발생 시, 계약자가 i) 지체없이 연구원에게 상황을 전파하고,
ii) 성실히 연구원에서 전달한 긴급조치 사항을 수행했음에도 불구하고 발생한 문제들은 계
다. 계약자는 본 과업을 수행하기 위해 적절한 지식 또는 능력을 보유하고 있는 적정 수의 운
전 인력을 연구원에게 제공하여야 한다.
• 가동기간의 극저온시스템 운전 업무 (24시간 운전)
- 극저온시스템 모니터링
- 극저온시스템 주요 수치 기록 및 분석
- 극저온시스템 제어
- 기타 가동기간에 필요한 운전 업무
• 휴지기간의 극저온시스템 운전 업무 (주간 운전)
- 유지보수 등을 위한 극저온시스템 제어
- 유지보수 등을 위한 극저온시스템 모니터링
- 유지보수 등을 위한 극저온시스템 주요 수치 기록 및 분석
- 극저온시스템 운전 매뉴얼 작성 및 개정
- 극저온시스템 운전 교육
- 성과보고회 개최
- 기타 휴지기간에 필요한 운전 업무
• 본 과업 및 본 과업을 위해 투입된 인력에 대한 관리감독
본 과업의 기간은 2025년 1월 1일부터 2025년 12월 31일까지로 하며, 2025년 동안 가동기
간의 운전 업무를 수행하는 것을 기준으로 한다. 단, 표 2.와 같이 연구원의 운영 추진 상황
에 따라 휴지기간의 운전업무를 수행하는 것으로 변경될 수 있다. 가동기간과 휴지기간의
운전 업무는 “3. 과업 내용” 하위 절에 기술한 내용에 따른다.
표 2. 과업 추진 일정(안)
2025년
구분
1분기 2분기 3분기 4분기
SCL3 극저온플랜트
SCL3 중이온가속기 SCL3 구간 운전 SCL3 구간 유지보수
시운전 준비 및
구간 (가동기간) (휴지기간)
시운전 (가동기간)
SCL2 SCL2 극저온플랜트 중이온가속기 SCL2 구간 시운전 SCL2 구간 유지보수
구간 시운전 (가동기간) (가동기간) (휴지기간)
회수
교대근무 인력들도 회람 필수
4 일일(Daily) 운영 상황 공유 회의 참석 유사 시 계약자가 진행 가능
일일근무일지 작성 당일 현장/제어 관련 발생 상
5
(연구원 양식 제공) 황 기록
i) 교대 변경 전 특이사항 또
교대근무자 변경 시 인수인계 및 인수인계서 작성
6 는 관찰 필요 사항 전달
(연구원 양식 제공)
ii) 성실교대근무 서약서 작성
운전 중 수리가 필요한 사항 발생 시 별도 기록
7
(연구원 양식 제공)
극저온시스템용 제어시스템(알람시스템 포함) 개선
8 연구원 요청 시 한정
및 보완을 위한 협조
기타 극저온시스템의 안전하고 안정적인 운영을 위 연구원과 계약자가 협의하여
9
해 필요한 업무 수행 여부 결정
3.4 중이온가속기 또는 극저온시스템 휴지기간의 운전 업무
계약자는 중이온가속기 또는 극저온시스템 휴지기간 동안 주간 운전업무가 이루어질 수 있
도록 운전 인력을 제공해야 한다. 주간 운전 업무는 운전인력 전원 운전을 기본으로 한다.
운전인력 중 1인 또는 2인은 유지보수 등을 위한 극저온시스템 모니터링 및 제어를 수행해
야 하며, 나머지 인력들은 각 운전 인력이 경험했던 상황들을 취합하여 극저온시스템 운전
문서번호 : -
2025년도 중이온가속기용 극저온시스템
개정번호 : 0
운전 용역 과업지시서 발 행 일 : 2024. 10. 30
페 이 지 : 12/19
매뉴얼을 작성하고 개정하는 업무를 수행해야 한다. 또한, 과업 및 운전 인력 감독자가 1인
이상 운전실에 배치되어 상기 업무들을 감독해야 한다. 계약자가 휴지기간에 수행해야 하는
이온가속기용 극저온시스템 운영에 중대한 영향이 생길 경우, 이는 계약자의 책임으로 한
다.
표 8. 휴지기간의 극저온시스템 제어 업무 리스트
순번 극저온시스템 제어 업무 비고
자동/수동(Auto/Manual) 제어 우선
1 계장류 제어 우선권(Priority) 변경
권 설정 변경
문서번호 : -
2025년도 중이온가속기용 극저온시스템
개정번호 : 0
운전 용역 과업지시서 발 행 일 : 2024. 10. 30
페 이 지 : 13/19
자동 밸브 – 제어시스템 활용
2 극저온시스템 자동/수동 밸브 조작
수동 밸브 – 현장에서 손으로 조작
압축기 emergency stop 버튼
3 현장에서 손으로 조작
활성화/비활성화
제어시스템 활용
4 진공펌프 가동/정지
(현장 조작 필요 시 연구원에 통지)
5 센서 오버라이드(Override) 설정 또는 해제 제어시스템 활용
6 계장류 제어 변수값(Ramp speed 등) 변경 제어시스템 활용
7 시퀀스 제어 변수값(Timer 등) 변경 제어시스템 활용
8 시퀀스 시작/종료 제어시스템 활용
현장 확인 및 최종 준비 상태 미진
9 극저온시스템 가동 전 최종 준비 상태 확인
할 경우, 상태 보완 포함
기타 극저온시스템의 안전하고 안정적인 운
10 제어시스템 활용 및 현장 조작 포함
전에 필요한 제어
3.4.2 유지보수 등을 위한 극저온시스템 모니터링
계약자는 휴지기간 동안 극저온시스템이 안정적으로 가동될 수 있도록 모니터링을 해야 한
다. 휴지기간의 모니터링 업무는 i) 유지보수 등을 위한 현장 활동 시 극저온시스템에 특이
2025년
구분
1분기 2분기 3분기 4분기
SCL3 극저온플랜트
SCL3 중이온가속기 SCL3 구간 운전 SCL3 구간 유지보수
시운전 준비 및
구간 (가동기간) (휴지기간)
시운전 (가동기간)
SCL2 SCL2 극저온플랜트 중이온가속기 SCL2 구간 시운전 SCL2 구간 유지보수
구간 시운전 (가동기간) (가동기간) (휴지기간)
회수
필요시 간헐적 운전 (휴지기간에 준함)
시스템
문서번호 : -
2025년도 중이온가속기용 극저온시스템
개정번호 : 0
운전 용역 과업지시서 발 행 일 : 2024. 10. 30
페 이 지 : 7/19
3. 과업 내용
3.1 중이온가속기용 극저온시스템 운전 업무의 정의 및 필요성
본 과업에서 사용하는 “중이온가속기용 극저온시스템 운전 업무”의 정의는, “중이온가
속기용 극저온시스템이 안전하고 안정적으로 운영(가동기간뿐만 아니라 휴지기간의 운영 포
함)될 수 있도록 i) 극저온시스템을 모니터링 및 제어하고, ii) 극저온시스템의 주요 수치를
기록 및 분석하며, iii) 중이온가속기용 극저온시스템의 운전 경험을 통합하여 정리(문서화)하
는 업무를 의미한다.
중이온가속기 또는 극저온시스템 가동기간에는 24시간 운전 업무를 필요로 한다. 통상적으
로 가동기간에는 24시간 운전을 위한 교대근무가 이루어지며, i) 극저온시스템의 모니터링
및 제어, ii) 극저온시스템의 주요 수치를 기록 및 분석하는 업무가 주로 이루어진다.
다만, 통상적으로 이러한 교대근무로 인해 각 운전 인력마다 경험할 수 있는 운전 상황이


---

### retriever B:
3. 과업 내용
3.1 중이온가속기용 극저온시스템 운전 업무의 정의 및 필요성
본 과업에서 사용하는 “중이온가속기용 극저온시스템 운전 업무”의 정의는, “중이온가
속기용 극저온시스템이 안전하고 안정적으로 운영(가동기간뿐만 아니라 휴지기간의 운영 포
함)될 수 있도록 i) 극저온시스템을 모니터링 및 제어하고, ii) 극저온시스템의 주요 수치를
기록 및 분석하며, iii) 중이온가속기용 극저온시스템의 운전 경험을 통합하여 정리(문서화)하
는 업무를 의미한다.
중이온가속기 또는 극저온시스템 가동기간에는 24시간 운전 업무를 필요로 한다. 통상적으
로 가동기간에는 24시간 운전을 위한 교대근무가 이루어지며, i) 극저온시스템의 모니터링
및 제어, ii) 극저온시스템의 주요 수치를 기록 및 분석하는 업무가 주로 이루어진다.
다만, 통상적으로 이러한 교대근무로 인해 각 운전 인력마다 경험할 수 있는 운전 상황이
서로 다르게 되고, 서로 다른 경험을 했던 운전 인력 모두가 참여하여 각자의 iii) 운전 경험
을 통합/정리할 수 있는 여건이 형성될 수 없다는 한계가 존재한다. 이러한 운전 경험의 통
합 정리는 후술하는 휴지기간에의 운전업무에 집중적으로 수행 가능하다.
통상적으로 휴지기간에는 극저온시스템의 유지보수 등이 진행되며, 유지보수 등을 위한 제
어, 유지보수 등의 경과 및 결과 확인을 위한 극저온시스템 모니터링, 유지보수 등의 결과
확인을 위한 주요 수치 기록 및 분석 또한 필수적인 운전 업무이다. 휴지기간에는 통상적으
로 주간 근무로 운전이 이루어진다. 이와 같이 주간 근무가 이루어질 수 있는 휴지기간의
제외한다.
표 3. 2025년 중이온가속기용 극저온시스템 운전 용역 대상 장치 목록
구분 장치명 장소 비고
극저온 극저온설비동/ 압축기, 재생시스템, 2 K 시스템
SCL3 극저온플랜트
플랜트 헬륨압축기동 등 부대시설 일체
문서번호 : -
2025년도 중이온가속기용 극저온시스템
개정번호 : 0
운전 용역 과업지시서 발 행 일 : 2024. 10. 30
페 이 지 : 8/19
극저온설비동/ 압축기, 재생시스템, 2 K 시스템
SCL2 극저온플랜트
헬륨압축기동 등 부대시설 일체
TBx 극저온설비동
밸브박스, 초전도가속관 저온유지
SCL3 터널/
극저온
SCL3 LINAC 모듈 및 밸브박스 진공단열용 진
ISOL 구간
분배
공펌프
시스템 밸브박스, 초전도가속관 저온유지
SCL2 터널/
SCL2 LINAC 모듈 및 밸브박스 진공단열용 진
IF 구간
공펌프
외부 정화기 외부
회수 압축시스템 헬륨압축기동
회수
가스백 헬륨압축기동
시스템
헬륨 탱크 외부 기체 헬륨량 관리 포함
액체 질소 시스템 외부 액체 질소량 관리 포함
3.3 중이온가속기 또는 극저온시스템 가동기간의 운전 업무
계약자는 중이온가속기 또는 극저온시스템 가동기간 동안 24시간 운전업무가 이루어질 수
있도록 운전 인력을 제공해야 한다. 24시간 운전 업무는 2인 1조를 기본으로 하며, 주간에
는 과업 및 운전 인력 감독자가 1인 이상 운전실에 존재해야 한다. 계약자가 가동기간에 수
행해야 하는 운전 업무 상세 내용은 하위 절에 기술된 내용을 따른다. 본 과업지시서에 기
술하지 않은 가동기간의 운전 업무는 연구원과 계약자가 협의하여 수행 여부를 결정한다.
온시스템 가동 기간의 운전 업무”와 “3.4 중이온가속기 또는 극저온시스템 휴지 기간의
운전 업무”를 따른다.
- 본 과업 및 인력에 대한 관리감독은 계약자가 담당하며, 관리감독자(현장대리인)과 운전
인력은 구분하여 인력을 구성한다. 단, 유사 시 현장대리인이 현장대리인의 업무에 지장을
주지 않는 선에서 운전 업무를 병행할 수 있다.
- 계약자의 i) 인력관리 소홀 또는 조작 실수 등으로 ii) 중이온가속기 운영에 중대한 문제가
발생한 경우, 이는 계약자의 책임으로 한다.
- 단, 연구원에서 계약자가 투입한 운전 인력이 운전 업무를 불성실히 수행하는 상황을 발
견한 경우, 연구원은 계약자에게 시정조치를 요구할 수 있다.
- 계약자는 연구원이 시정조치를 요구할 경우, 7일 이내에 연구원에게 조치 결과를 회신해
야 한다.
- 만약 계약자가 i) 연구원의 시정조치에도 불구하고 ii) 시정 없이 투입 인력이 운전 업무를
불성실하게 수행하여 iii) 중이온가속기 운영에 중대한 문제가 발생한 경우, 계약자의 책임으
로 한다.
- 위험(긴급) 상황 발생 시, 계약자는 연구원에게 지체없이 통보해야 하며, 연구원의 협조
요청에 적극 대응하여야 한다.
- 본 과업은 2025년도에 국한된 일시 업무로 용역직의 정규직 전환과 무관하다.
(정규직 전환 대상 아님.)
- 본 용역은 연구원 운영 추진 상황에 따라 조기 종료 될 수 있다.
문서번호 : -
2025년도 중이온가속기용 극저온시스템
개정번호 : 0
운전 용역 과업지시서 발 행 일 : 2024. 10. 30
페 이 지 : 19/19
중이온가속기용 극저온시스템이 가동기간 및 휴지
기간에 안전하고 안정적으로 운영될 수 있도록,
중이온가속기용 극저온시스템 i) 극저온시스템을 모니터링 및 제어하고,
운전 업무 ii) 극저온시스템의 주요 수치를 기록 및 분석하며,
iii) 중이온가속기용 극저온시스템의 운전 경험을
통합하여 정리(문서화)하는 업무
문서번호 : -
2025년도 중이온가속기용 극저온시스템
개정번호 : 0
운전 용역 과업지시서 발 행 일 : 2024. 10. 30
페 이 지 : 6/19
문서번호 : -
2025년도 중이온가속기용 극저온시스템
개정번호 : 0
운전 용역 과업지시서 발 행 일 : 2024. 10. 30
페 이 지 : 12/19
매뉴얼을 작성하고 개정하는 업무를 수행해야 한다. 또한, 과업 및 운전 인력 감독자가 1인
이상 운전실에 배치되어 상기 업무들을 감독해야 한다. 계약자가 휴지기간에 수행해야 하는
운전 업무 상세 내용은 하위 절에 기술된 내용을 따른다. 본 과업지시서에 기술하지 않은
휴지기간의 운전 업무는 연구원과 계약자가 협의하여 수행 여부를 결정한다.
3.4.1 유지보수 등을 위한 극저온시스템 제어
극저온시스템의 제어는 극저온시스템이 휴지지간에 안전하고 안정적으로 운영될 수 있도록
극저온시스템 또는 극저온시스템 관련 부품들을 제어시스템을 활용하여 또는 현장에서 손으
로 조작하는 행위를 의미한다. 계약자는 극저온시스템이 본 목적에 부합하도록 극저온시스
템의 제어를 수행해야 한다. 표 8.에 극저온시스템 제어 업무에 해당되는 사항들을 기술하
였다.
휴지기간의 극저온시스템 제어는 i) 극저온시스템에 대한 충분한 이해를 바탕으로 ii) 조작
후 유지보수 등 현장 활동에 미치는 영향을 충분히 검토한 뒤 이루어져야 한다. 더불어, iii)
조작 대상물이 위치하고 있는 현장에 인력이 존재하는지 여부를 확인하고, iv) 현장에 인력
이 존재할 경우, 조작 대상물을 조작했을 때 발생할 수 있는 상황에 대해 해당 인력의 안전
이 확보된 뒤에 조작을 수행해야 한다. 마지막으로 v) 조작 대상/조작 상태에 대한 혼동이
없도록 각별히 주의를 하여 제어가 진행되어야 한다.
이온가속기용 극저온시스템 운영에 중대한 영향이 생길 경우, 계약자의 책임으로 한다.
표 6. 가동기간의 극저온시스템 제어 업무 리스트
순번 극저온시스템 제어 업무 비고
자동/수동(Auto/Manual) 제어 우선
1 계장류 제어 우선권(Priority) 변경
권 설정 변경
자동 밸브 – 제어시스템 활용
2 극저온시스템 자동/수동 밸브 조작
수동 밸브 – 현장에서 손으로 조작
3 히터 조작 제어시스템 활용
4 액체 헬륨 레벨미터 모니터 측정 방식 변경 현장에서 설정 변경
제어시스템 활용
5 진공펌프 가동/정지
(현장 조작 필요 시 연구원에 통지)
6 센서 오버라이드(Override) 설정 또는 해제 제어시스템 활용
7 계장류 제어 변수값(Ramp speed 등) 변경 제어시스템 활용
8 시퀀스 제어 변수값(Timer 등) 변경 제어시스템 활용
9 시퀀스 시작/종료 제어시스템 활용
기타 극저온시스템의 안전하고 안정적인 운
10 제어시스템 활용 및 현장 조작 포함
전에 필요한 제어
문서번호 : -
2025년도 중이온가속기용 극저온시스템
개정번호 : 0
운전 용역 과업지시서 발 행 일 : 2024. 10. 30
페 이 지 : 11/19
3.3.4 기타 가동기간에 필요한 운전 업무
표 7.에 기술한 기타 가동기간에 필요한 운전 업무 또한 계약자의 업무 범위이다. 표 7.에
기술된 내용 중, 위험(긴급)상황 발생 시, 계약자가 i) 지체없이 연구원에게 상황을 전파하고,
ii) 성실히 연구원에서 전달한 긴급조치 사항을 수행했음에도 불구하고 발생한 문제들은 계
다. 계약자는 본 과업을 수행하기 위해 적절한 지식 또는 능력을 보유하고 있는 적정 수의 운
전 인력을 연구원에게 제공하여야 한다.
• 가동기간의 극저온시스템 운전 업무 (24시간 운전)
- 극저온시스템 모니터링
- 극저온시스템 주요 수치 기록 및 분석
- 극저온시스템 제어
- 기타 가동기간에 필요한 운전 업무
• 휴지기간의 극저온시스템 운전 업무 (주간 운전)
- 유지보수 등을 위한 극저온시스템 제어
- 유지보수 등을 위한 극저온시스템 모니터링
- 유지보수 등을 위한 극저온시스템 주요 수치 기록 및 분석
- 극저온시스템 운전 매뉴얼 작성 및 개정
- 극저온시스템 운전 교육
- 성과보고회 개최
- 기타 휴지기간에 필요한 운전 업무
• 본 과업 및 본 과업을 위해 투입된 인력에 대한 관리감독
본 과업의 기간은 2025년 1월 1일부터 2025년 12월 31일까지로 하며, 2025년 동안 가동기
간의 운전 업무를 수행하는 것을 기준으로 한다. 단, 표 2.와 같이 연구원의 운영 추진 상황
에 따라 휴지기간의 운전업무를 수행하는 것으로 변경될 수 있다. 가동기간과 휴지기간의
운전 업무는 “3. 과업 내용” 하위 절에 기술한 내용에 따른다.
표 2. 과업 추진 일정(안)
2025년
구분
1분기 2분기 3분기 4분기
SCL3 극저온플랜트
SCL3 중이온가속기 SCL3 구간 운전 SCL3 구간 유지보수
시운전 준비 및
구간 (가동기간) (휴지기간)
시운전 (가동기간)
SCL2 SCL2 극저온플랜트 중이온가속기 SCL2 구간 시운전 SCL2 구간 유지보수
구간 시운전 (가동기간) (가동기간) (휴지기간)
회수
교대근무 인력들도 회람 필수
4 일일(Daily) 운영 상황 공유 회의 참석 유사 시 계약자가 진행 가능
일일근무일지 작성 당일 현장/제어 관련 발생 상
5
(연구원 양식 제공) 황 기록
i) 교대 변경 전 특이사항 또
교대근무자 변경 시 인수인계 및 인수인계서 작성
6 는 관찰 필요 사항 전달
(연구원 양식 제공)
ii) 성실교대근무 서약서 작성
운전 중 수리가 필요한 사항 발생 시 별도 기록
7
(연구원 양식 제공)
극저온시스템용 제어시스템(알람시스템 포함) 개선
8 연구원 요청 시 한정
및 보완을 위한 협조
기타 극저온시스템의 안전하고 안정적인 운영을 위 연구원과 계약자가 협의하여
9
해 필요한 업무 수행 여부 결정
3.4 중이온가속기 또는 극저온시스템 휴지기간의 운전 업무
계약자는 중이온가속기 또는 극저온시스템 휴지기간 동안 주간 운전업무가 이루어질 수 있
도록 운전 인력을 제공해야 한다. 주간 운전 업무는 운전인력 전원 운전을 기본으로 한다.
운전인력 중 1인 또는 2인은 유지보수 등을 위한 극저온시스템 모니터링 및 제어를 수행해
야 하며, 나머지 인력들은 각 운전 인력이 경험했던 상황들을 취합하여 극저온시스템 운전
문서번호 : -
2025년도 중이온가속기용 극저온시스템
개정번호 : 0
운전 용역 과업지시서 발 행 일 : 2024. 10. 30
페 이 지 : 12/19
매뉴얼을 작성하고 개정하는 업무를 수행해야 한다. 또한, 과업 및 운전 인력 감독자가 1인
이상 운전실에 배치되어 상기 업무들을 감독해야 한다. 계약자가 휴지기간에 수행해야 하는


---

### openai answer
retriever A 점수: 3점
가동기간에 24시간 교대근무가 이루어짐과 휴지기간에 주간 운전(유지보수 중심)으로 운영됨을 명시하여, 답변의 핵심이 모두 포함되어 있습니다.

retriever B 점수: 3점
가동기간 24시간 교대근무와 휴지기간 주간 운전을 명시하고 있어, 질문에 대한 답을 완전히 뒷받침합니다. 

============================================================
### query:
극저온시스템 과업지시서에 따르면, 가동기간 중 교대근무 인력이 개별적으로 얻은 운전 경험을 통합하여 정리하기 어려운 이유는 무엇이며, 이 업무는 주로 언제 수행되어야 합니까?

### answer:
교대근무로 인해 각 운전 인력마다 경험하는 운전 상황이 서로 달라, 모든 인원이 함께 모여 운전 경험을 통합하고 정리할 여건이 형성되기 어렵기 때문입니다. 이러한 운전 경험 통합 및 정리(문서화) 업무는 주로 주간 근무가 이루어지는 '휴지기간'에 집중적으로 수행됩니다.

---

### retriever A:
중이온가속기 또는 극저온시스템 가동기간에는 24시간 운전 업무를 필요로 한다. 통상적으
로 가동기간에는 24시간 운전을 위한 교대근무가 이루어지며, i) 극저온시스템의 모니터링
및 제어, ii) 극저온시스템의 주요 수치를 기록 및 분석하는 업무가 주로 이루어진다.
다만, 통상적으로 이러한 교대근무로 인해 각 운전 인력마다 경험할 수 있는 운전 상황이
서로 다르게 되고, 서로 다른 경험을 했던 운전 인력 모두가 참여하여 각자의 iii) 운전 경험
을 통합/정리할 수 있는 여건이 형성될 수 없다는 한계가 존재한다. 이러한 운전 경험의 통
합 정리는 후술하는 휴지기간에의 운전업무에 집중적으로 수행 가능하다.
통상적으로 휴지기간에는 극저온시스템의 유지보수 등이 진행되며, 유지보수 등을 위한 제
어, 유지보수 등의 경과 및 결과 확인을 위한 극저온시스템 모니터링, 유지보수 등의 결과
확인을 위한 주요 수치 기록 및 분석 또한 필수적인 운전 업무이다. 휴지기간에는 통상적으
로 주간 근무로 운전이 이루어진다. 이와 같이 주간 근무가 이루어질 수 있는 휴지기간의
특성 상, 휴지기간은 가동기간에서 교대근무로 각 운전 인력 개인이 얻은 단편적인 운전 경
험을 통합하여 정리(문서화)할 수 있는 여건이 형성될 수 있는 기간이다.
결과적으로 휴지기간에는 i) 중이온가속기용 극저온시스템의 운전 경험을 통합하여 정리(문
서화)하는 업무가 집중적으로 이루어지며, ii) 유지보수 등을 위한 극저온시스템의 제어와 iii)
유지보수 등을 위한 극저온시스템의 주요 수치 기록 및 분석이 병행되어야 한다.
3.2 운전 대상 및 장소
다. 계약자는 본 과업을 수행하기 위해 적절한 지식 또는 능력을 보유하고 있는 적정 수의 운
전 인력을 연구원에게 제공하여야 한다.
• 가동기간의 극저온시스템 운전 업무 (24시간 운전)
- 극저온시스템 모니터링
- 극저온시스템 주요 수치 기록 및 분석
- 극저온시스템 제어
- 기타 가동기간에 필요한 운전 업무
• 휴지기간의 극저온시스템 운전 업무 (주간 운전)
- 유지보수 등을 위한 극저온시스템 제어
- 유지보수 등을 위한 극저온시스템 모니터링
- 유지보수 등을 위한 극저온시스템 주요 수치 기록 및 분석
- 극저온시스템 운전 매뉴얼 작성 및 개정
- 극저온시스템 운전 교육
- 성과보고회 개최
- 기타 휴지기간에 필요한 운전 업무
• 본 과업 및 본 과업을 위해 투입된 인력에 대한 관리감독
본 과업의 기간은 2025년 1월 1일부터 2025년 12월 31일까지로 하며, 2025년 동안 가동기
간의 운전 업무를 수행하는 것을 기준으로 한다. 단, 표 2.와 같이 연구원의 운영 추진 상황
에 따라 휴지기간의 운전업무를 수행하는 것으로 변경될 수 있다. 가동기간과 휴지기간의
운전 업무는 “3. 과업 내용” 하위 절에 기술한 내용에 따른다.
표 2. 과업 추진 일정(안)
2025년
구분
1분기 2분기 3분기 4분기
SCL3 극저온플랜트
SCL3 중이온가속기 SCL3 구간 운전 SCL3 구간 유지보수
시운전 준비 및
구간 (가동기간) (휴지기간)
시운전 (가동기간)
SCL2 SCL2 극저온플랜트 중이온가속기 SCL2 구간 시운전 SCL2 구간 유지보수
구간 시운전 (가동기간) (가동기간) (휴지기간)
회수
교대근무 인력들도 회람 필수
4 일일(Daily) 운영 상황 공유 회의 참석 유사 시 계약자가 진행 가능
일일근무일지 작성 당일 현장/제어 관련 발생 상
5
(연구원 양식 제공) 황 기록
i) 교대 변경 전 특이사항 또
교대근무자 변경 시 인수인계 및 인수인계서 작성
6 는 관찰 필요 사항 전달
(연구원 양식 제공)
ii) 성실교대근무 서약서 작성
운전 중 수리가 필요한 사항 발생 시 별도 기록
7
(연구원 양식 제공)
극저온시스템용 제어시스템(알람시스템 포함) 개선
8 연구원 요청 시 한정
및 보완을 위한 협조
기타 극저온시스템의 안전하고 안정적인 운영을 위 연구원과 계약자가 협의하여
9
해 필요한 업무 수행 여부 결정
3.4 중이온가속기 또는 극저온시스템 휴지기간의 운전 업무
계약자는 중이온가속기 또는 극저온시스템 휴지기간 동안 주간 운전업무가 이루어질 수 있
도록 운전 인력을 제공해야 한다. 주간 운전 업무는 운전인력 전원 운전을 기본으로 한다.
운전인력 중 1인 또는 2인은 유지보수 등을 위한 극저온시스템 모니터링 및 제어를 수행해
야 하며, 나머지 인력들은 각 운전 인력이 경험했던 상황들을 취합하여 극저온시스템 운전
문서번호 : -
2025년도 중이온가속기용 극저온시스템
개정번호 : 0
운전 용역 과업지시서 발 행 일 : 2024. 10. 30
페 이 지 : 12/19
매뉴얼을 작성하고 개정하는 업무를 수행해야 한다. 또한, 과업 및 운전 인력 감독자가 1인
이상 운전실에 배치되어 상기 업무들을 감독해야 한다. 계약자가 휴지기간에 수행해야 하는
다. 가동기간의 모니터링 업무는 극저온시스템이 안정적으로 가동되고 있는지 여부를 확인
하기 위해 극저온시스템을 지속적으로 관찰하는 업무를 의미하며, 표 4.에 가동기간의 모니
터링 업무에 해당되는 내용들을 기술하였다.
극저온시스템의 모니터링은 연구원에 구축되어있는 극저온시스템용 제어시스템(이하 “제
어시스템”)을 활용하여 또는 현장에 설치되어있는 장치를 통해 수행한다. 연구원은 구축되
어있는 제어시스템을 활용하기 위한 컴퓨터, 모니터, 키보드, 마우스, 책상 및 의자를 계약
자에게 제공한다. 현재 연구원에 구축되어있는 제어시스템은 표 5.과 같다.
연구원은 계약자에게 현재 구축되어있는 제어시스템에 대한 기본 사용 교육을 제공하며,
계약자는 연구원이 제공하는 교육을 이해하기 위한 배경지식을 보유해야 한다.
문서번호 : -
2025년도 중이온가속기용 극저온시스템
개정번호 : 0
운전 용역 과업지시서 발 행 일 : 2024. 10. 30
페 이 지 : 9/19
표 4. 가동기간의 극저온시스템 모니터링 업무 리스트
순번 극저온시스템 모니터링 업무 비고
1 알람 메시지 확인
2 주요 센서 트렌드 확인 온도, 압력, 유량, 속도 등
3 정상 운전 상황의 주요 센서 지시값 파악 및 숙지 온도, 압력, 유량, 속도 등
4 비정상 운전 상황 감지
연구원에서 현장 확인 필요상황 발생 시, 연구원이 지 무전 통신 활용
5
정하는 값 관찰, 확인 및 상황 전달 (연구원에서 무전기 제공)
6 제어로직 정상 작동 여부 확인
연구원 요청 시,
7 현장 상황 일시적 또는 지속적 관찰
비방사선구역 한정
문서번호 : -
2025년도 중이온가속기용 극저온시스템
개정번호 : 0
운전 용역 과업지시서 발 행 일 : 2024. 10. 30
페 이 지 : 12/19
매뉴얼을 작성하고 개정하는 업무를 수행해야 한다. 또한, 과업 및 운전 인력 감독자가 1인
이상 운전실에 배치되어 상기 업무들을 감독해야 한다. 계약자가 휴지기간에 수행해야 하는
운전 업무 상세 내용은 하위 절에 기술된 내용을 따른다. 본 과업지시서에 기술하지 않은
휴지기간의 운전 업무는 연구원과 계약자가 협의하여 수행 여부를 결정한다.
3.4.1 유지보수 등을 위한 극저온시스템 제어
극저온시스템의 제어는 극저온시스템이 휴지지간에 안전하고 안정적으로 운영될 수 있도록
극저온시스템 또는 극저온시스템 관련 부품들을 제어시스템을 활용하여 또는 현장에서 손으
로 조작하는 행위를 의미한다. 계약자는 극저온시스템이 본 목적에 부합하도록 극저온시스
템의 제어를 수행해야 한다. 표 8.에 극저온시스템 제어 업무에 해당되는 사항들을 기술하
였다.
휴지기간의 극저온시스템 제어는 i) 극저온시스템에 대한 충분한 이해를 바탕으로 ii) 조작
후 유지보수 등 현장 활동에 미치는 영향을 충분히 검토한 뒤 이루어져야 한다. 더불어, iii)
조작 대상물이 위치하고 있는 현장에 인력이 존재하는지 여부를 확인하고, iv) 현장에 인력
이 존재할 경우, 조작 대상물을 조작했을 때 발생할 수 있는 상황에 대해 해당 인력의 안전
이 확보된 뒤에 조작을 수행해야 한다. 마지막으로 v) 조작 대상/조작 상태에 대한 혼동이
없도록 각별히 주의를 하여 제어가 진행되어야 한다.
이온가속기용 극저온시스템 운영에 중대한 영향이 생길 경우, 계약자의 책임으로 한다.
표 6. 가동기간의 극저온시스템 제어 업무 리스트
순번 극저온시스템 제어 업무 비고
자동/수동(Auto/Manual) 제어 우선
1 계장류 제어 우선권(Priority) 변경
권 설정 변경
자동 밸브 – 제어시스템 활용
2 극저온시스템 자동/수동 밸브 조작
수동 밸브 – 현장에서 손으로 조작
3 히터 조작 제어시스템 활용
4 액체 헬륨 레벨미터 모니터 측정 방식 변경 현장에서 설정 변경
제어시스템 활용
5 진공펌프 가동/정지
(현장 조작 필요 시 연구원에 통지)
6 센서 오버라이드(Override) 설정 또는 해제 제어시스템 활용
7 계장류 제어 변수값(Ramp speed 등) 변경 제어시스템 활용
8 시퀀스 제어 변수값(Timer 등) 변경 제어시스템 활용
9 시퀀스 시작/종료 제어시스템 활용
기타 극저온시스템의 안전하고 안정적인 운
10 제어시스템 활용 및 현장 조작 포함
전에 필요한 제어
문서번호 : -
2025년도 중이온가속기용 극저온시스템
개정번호 : 0
운전 용역 과업지시서 발 행 일 : 2024. 10. 30
페 이 지 : 11/19
3.3.4 기타 가동기간에 필요한 운전 업무
표 7.에 기술한 기타 가동기간에 필요한 운전 업무 또한 계약자의 업무 범위이다. 표 7.에
기술된 내용 중, 위험(긴급)상황 발생 시, 계약자가 i) 지체없이 연구원에게 상황을 전파하고,
ii) 성실히 연구원에서 전달한 긴급조치 사항을 수행했음에도 불구하고 발생한 문제들은 계
- 금회 대처 방법
인 서명 필수
- 향후 대처 방법
- 수치 기록을 위해 계약자가
- 기타 필요한 내용
준비한 PC에 상시 저장
- 매 월간 기성 신청 시 제출
월간보고서 - 월간 운전 상황 요약 O
- 문서 파일 제출
- IBS에서 지정한 기한 내 제
기타 - IBS에서 요구한 결과물 O
출
문서번호 : -
2025년도 중이온가속기용 극저온시스템
개정번호 : 0
운전 용역 과업지시서 발 행 일 : 2024. 10. 30
페 이 지 : 18/19
5. 기타사항
계약자는 본 절에서 기술하는 사항들을 명확히 이해하고 숙지한 뒤, 본 과업을 착수해야
한다. 본 절의 내용에 대한 미숙지로 발생하는 불이익은 계약자의 책임으로 한다.
- 본 과업은 극저온시스템 운전에 대한 단독업무를 수행한다.
(단, 중이온가속기 운영 건전성과 관련된 중대한 운전 사항은 연구원과 협의하여 진행)
- 계약자는 연구원에게서 인수인계 받은 과업 일체 내용을 바탕으로 극저온시스템 운전에
필요한 인원, 인력구성 및 운전근무를 편성한다.
- 극저온시스템의 가동기간과 휴지기간은 중이온가속기 운영 상황에 따라 연구원이 결정하
며, 연구원은 가동기간/휴지기간 변경 최소 2 주 전에는 계약자에게 통지한다.
- 가동기간에는 24시간 운전이 이루어지며, 휴지기간에는 주간 운전이 이루어져야 한다. 휴
지기간에의 주간 근무는 가동기간동안 24시간 운전에 투입된 운전 인력 전원이 수행해야 하
며(휴가자 제외), 계약자는 연구원의 가동기간/휴지기간 선정에 대한 이의 및 각 기간의 운
전 형태(24시간 운전/주간 운전)에 대해 이의를 제기할 수 없다.


---

### retriever B:
3. 과업 내용
3.1 중이온가속기용 극저온시스템 운전 업무의 정의 및 필요성
본 과업에서 사용하는 “중이온가속기용 극저온시스템 운전 업무”의 정의는, “중이온가
속기용 극저온시스템이 안전하고 안정적으로 운영(가동기간뿐만 아니라 휴지기간의 운영 포
함)될 수 있도록 i) 극저온시스템을 모니터링 및 제어하고, ii) 극저온시스템의 주요 수치를
기록 및 분석하며, iii) 중이온가속기용 극저온시스템의 운전 경험을 통합하여 정리(문서화)하
는 업무를 의미한다.
중이온가속기 또는 극저온시스템 가동기간에는 24시간 운전 업무를 필요로 한다. 통상적으
로 가동기간에는 24시간 운전을 위한 교대근무가 이루어지며, i) 극저온시스템의 모니터링
및 제어, ii) 극저온시스템의 주요 수치를 기록 및 분석하는 업무가 주로 이루어진다.
다만, 통상적으로 이러한 교대근무로 인해 각 운전 인력마다 경험할 수 있는 운전 상황이
서로 다르게 되고, 서로 다른 경험을 했던 운전 인력 모두가 참여하여 각자의 iii) 운전 경험
을 통합/정리할 수 있는 여건이 형성될 수 없다는 한계가 존재한다. 이러한 운전 경험의 통
합 정리는 후술하는 휴지기간에의 운전업무에 집중적으로 수행 가능하다.
통상적으로 휴지기간에는 극저온시스템의 유지보수 등이 진행되며, 유지보수 등을 위한 제
어, 유지보수 등의 경과 및 결과 확인을 위한 극저온시스템 모니터링, 유지보수 등의 결과
확인을 위한 주요 수치 기록 및 분석 또한 필수적인 운전 업무이다. 휴지기간에는 통상적으
로 주간 근무로 운전이 이루어진다. 이와 같이 주간 근무가 이루어질 수 있는 휴지기간의
는 과업 및 운전 인력 감독자가 1인 이상 운전실에 존재해야 한다. 계약자가 가동기간에 수
행해야 하는 운전 업무 상세 내용은 하위 절에 기술된 내용을 따른다. 본 과업지시서에 기
술하지 않은 가동기간의 운전 업무는 연구원과 계약자가 협의하여 수행 여부를 결정한다.
3.3.1 극저온시스템 모니터링
계약자는 가동기간 동안 극저온시스템이 안정적으로 가동될 수 있도록 모니터링을 해야 한
교대근무 인력들도 회람 필수
4 일일(Daily) 운영 상황 공유 회의 참석 유사 시 계약자가 진행 가능
일일근무일지 작성 당일 현장/제어 관련 발생 상
5
(연구원 양식 제공) 황 기록
i) 교대 변경 전 특이사항 또
교대근무자 변경 시 인수인계 및 인수인계서 작성
6 는 관찰 필요 사항 전달
(연구원 양식 제공)
ii) 성실교대근무 서약서 작성
운전 중 수리가 필요한 사항 발생 시 별도 기록
7
(연구원 양식 제공)
극저온시스템용 제어시스템(알람시스템 포함) 개선
8 연구원 요청 시 한정
및 보완을 위한 협조
기타 극저온시스템의 안전하고 안정적인 운영을 위 연구원과 계약자가 협의하여
9
해 필요한 업무 수행 여부 결정
3.4 중이온가속기 또는 극저온시스템 휴지기간의 운전 업무
계약자는 중이온가속기 또는 극저온시스템 휴지기간 동안 주간 운전업무가 이루어질 수 있
도록 운전 인력을 제공해야 한다. 주간 운전 업무는 운전인력 전원 운전을 기본으로 한다.
운전인력 중 1인 또는 2인은 유지보수 등을 위한 극저온시스템 모니터링 및 제어를 수행해
야 하며, 나머지 인력들은 각 운전 인력이 경험했던 상황들을 취합하여 극저온시스템 운전
문서번호 : -
2025년도 중이온가속기용 극저온시스템
개정번호 : 0
운전 용역 과업지시서 발 행 일 : 2024. 10. 30
페 이 지 : 12/19
매뉴얼을 작성하고 개정하는 업무를 수행해야 한다. 또한, 과업 및 운전 인력 감독자가 1인
이상 운전실에 배치되어 상기 업무들을 감독해야 한다. 계약자가 휴지기간에 수행해야 하는
다. 계약자는 본 과업을 수행하기 위해 적절한 지식 또는 능력을 보유하고 있는 적정 수의 운
전 인력을 연구원에게 제공하여야 한다.
• 가동기간의 극저온시스템 운전 업무 (24시간 운전)
- 극저온시스템 모니터링
- 극저온시스템 주요 수치 기록 및 분석
- 극저온시스템 제어
- 기타 가동기간에 필요한 운전 업무
• 휴지기간의 극저온시스템 운전 업무 (주간 운전)
- 유지보수 등을 위한 극저온시스템 제어
- 유지보수 등을 위한 극저온시스템 모니터링
- 유지보수 등을 위한 극저온시스템 주요 수치 기록 및 분석
- 극저온시스템 운전 매뉴얼 작성 및 개정
- 극저온시스템 운전 교육
- 성과보고회 개최
- 기타 휴지기간에 필요한 운전 업무
• 본 과업 및 본 과업을 위해 투입된 인력에 대한 관리감독
본 과업의 기간은 2025년 1월 1일부터 2025년 12월 31일까지로 하며, 2025년 동안 가동기
간의 운전 업무를 수행하는 것을 기준으로 한다. 단, 표 2.와 같이 연구원의 운영 추진 상황
에 따라 휴지기간의 운전업무를 수행하는 것으로 변경될 수 있다. 가동기간과 휴지기간의
운전 업무는 “3. 과업 내용” 하위 절에 기술한 내용에 따른다.
표 2. 과업 추진 일정(안)
2025년
구분
1분기 2분기 3분기 4분기
SCL3 극저온플랜트
SCL3 중이온가속기 SCL3 구간 운전 SCL3 구간 유지보수
시운전 준비 및
구간 (가동기간) (휴지기간)
시운전 (가동기간)
SCL2 SCL2 극저온플랜트 중이온가속기 SCL2 구간 시운전 SCL2 구간 유지보수
구간 시운전 (가동기간) (가동기간) (휴지기간)
회수
다. 가동기간의 모니터링 업무는 극저온시스템이 안정적으로 가동되고 있는지 여부를 확인
하기 위해 극저온시스템을 지속적으로 관찰하는 업무를 의미하며, 표 4.에 가동기간의 모니
터링 업무에 해당되는 내용들을 기술하였다.
극저온시스템의 모니터링은 연구원에 구축되어있는 극저온시스템용 제어시스템(이하 “제
어시스템”)을 활용하여 또는 현장에 설치되어있는 장치를 통해 수행한다. 연구원은 구축되
어있는 제어시스템을 활용하기 위한 컴퓨터, 모니터, 키보드, 마우스, 책상 및 의자를 계약
자에게 제공한다. 현재 연구원에 구축되어있는 제어시스템은 표 5.과 같다.
연구원은 계약자에게 현재 구축되어있는 제어시스템에 대한 기본 사용 교육을 제공하며,
계약자는 연구원이 제공하는 교육을 이해하기 위한 배경지식을 보유해야 한다.
문서번호 : -
2025년도 중이온가속기용 극저온시스템
개정번호 : 0
운전 용역 과업지시서 발 행 일 : 2024. 10. 30
페 이 지 : 9/19
표 4. 가동기간의 극저온시스템 모니터링 업무 리스트
순번 극저온시스템 모니터링 업무 비고
1 알람 메시지 확인
2 주요 센서 트렌드 확인 온도, 압력, 유량, 속도 등
3 정상 운전 상황의 주요 센서 지시값 파악 및 숙지 온도, 압력, 유량, 속도 등
4 비정상 운전 상황 감지
연구원에서 현장 확인 필요상황 발생 시, 연구원이 지 무전 통신 활용
5
정하는 값 관찰, 확인 및 상황 전달 (연구원에서 무전기 제공)
6 제어로직 정상 작동 여부 확인
연구원 요청 시,
7 현장 상황 일시적 또는 지속적 관찰
비방사선구역 한정
제외한다.
표 3. 2025년 중이온가속기용 극저온시스템 운전 용역 대상 장치 목록
구분 장치명 장소 비고
극저온 극저온설비동/ 압축기, 재생시스템, 2 K 시스템
SCL3 극저온플랜트
플랜트 헬륨압축기동 등 부대시설 일체
문서번호 : -
2025년도 중이온가속기용 극저온시스템
개정번호 : 0
운전 용역 과업지시서 발 행 일 : 2024. 10. 30
페 이 지 : 8/19
극저온설비동/ 압축기, 재생시스템, 2 K 시스템
SCL2 극저온플랜트
헬륨압축기동 등 부대시설 일체
TBx 극저온설비동
밸브박스, 초전도가속관 저온유지
SCL3 터널/
극저온
SCL3 LINAC 모듈 및 밸브박스 진공단열용 진
ISOL 구간
분배
공펌프
시스템 밸브박스, 초전도가속관 저온유지
SCL2 터널/
SCL2 LINAC 모듈 및 밸브박스 진공단열용 진
IF 구간
공펌프
외부 정화기 외부
회수 압축시스템 헬륨압축기동
회수
가스백 헬륨압축기동
시스템
헬륨 탱크 외부 기체 헬륨량 관리 포함
액체 질소 시스템 외부 액체 질소량 관리 포함
3.3 중이온가속기 또는 극저온시스템 가동기간의 운전 업무
계약자는 중이온가속기 또는 극저온시스템 가동기간 동안 24시간 운전업무가 이루어질 수
있도록 운전 인력을 제공해야 한다. 24시간 운전 업무는 2인 1조를 기본으로 하며, 주간에
는 과업 및 운전 인력 감독자가 1인 이상 운전실에 존재해야 한다. 계약자가 가동기간에 수
행해야 하는 운전 업무 상세 내용은 하위 절에 기술된 내용을 따른다. 본 과업지시서에 기
술하지 않은 가동기간의 운전 업무는 연구원과 계약자가 협의하여 수행 여부를 결정한다.
문서번호 : -
2025년도 중이온가속기용 극저온시스템
개정번호 : 0
운전 용역 과업지시서 발 행 일 : 2024. 10. 30
페 이 지 : 12/19
매뉴얼을 작성하고 개정하는 업무를 수행해야 한다. 또한, 과업 및 운전 인력 감독자가 1인
이상 운전실에 배치되어 상기 업무들을 감독해야 한다. 계약자가 휴지기간에 수행해야 하는
운전 업무 상세 내용은 하위 절에 기술된 내용을 따른다. 본 과업지시서에 기술하지 않은
휴지기간의 운전 업무는 연구원과 계약자가 협의하여 수행 여부를 결정한다.
3.4.1 유지보수 등을 위한 극저온시스템 제어
극저온시스템의 제어는 극저온시스템이 휴지지간에 안전하고 안정적으로 운영될 수 있도록
극저온시스템 또는 극저온시스템 관련 부품들을 제어시스템을 활용하여 또는 현장에서 손으
로 조작하는 행위를 의미한다. 계약자는 극저온시스템이 본 목적에 부합하도록 극저온시스
템의 제어를 수행해야 한다. 표 8.에 극저온시스템 제어 업무에 해당되는 사항들을 기술하
였다.
휴지기간의 극저온시스템 제어는 i) 극저온시스템에 대한 충분한 이해를 바탕으로 ii) 조작
후 유지보수 등 현장 활동에 미치는 영향을 충분히 검토한 뒤 이루어져야 한다. 더불어, iii)
조작 대상물이 위치하고 있는 현장에 인력이 존재하는지 여부를 확인하고, iv) 현장에 인력
이 존재할 경우, 조작 대상물을 조작했을 때 발생할 수 있는 상황에 대해 해당 인력의 안전
이 확보된 뒤에 조작을 수행해야 한다. 마지막으로 v) 조작 대상/조작 상태에 대한 혼동이
없도록 각별히 주의를 하여 제어가 진행되어야 한다.
이온가속기용 극저온시스템 운영에 중대한 영향이 생길 경우, 계약자의 책임으로 한다.
표 6. 가동기간의 극저온시스템 제어 업무 리스트
순번 극저온시스템 제어 업무 비고
자동/수동(Auto/Manual) 제어 우선
1 계장류 제어 우선권(Priority) 변경
권 설정 변경
자동 밸브 – 제어시스템 활용
2 극저온시스템 자동/수동 밸브 조작
수동 밸브 – 현장에서 손으로 조작
3 히터 조작 제어시스템 활용
4 액체 헬륨 레벨미터 모니터 측정 방식 변경 현장에서 설정 변경
제어시스템 활용
5 진공펌프 가동/정지
(현장 조작 필요 시 연구원에 통지)
6 센서 오버라이드(Override) 설정 또는 해제 제어시스템 활용
7 계장류 제어 변수값(Ramp speed 등) 변경 제어시스템 활용
8 시퀀스 제어 변수값(Timer 등) 변경 제어시스템 활용
9 시퀀스 시작/종료 제어시스템 활용
기타 극저온시스템의 안전하고 안정적인 운
10 제어시스템 활용 및 현장 조작 포함
전에 필요한 제어
문서번호 : -
2025년도 중이온가속기용 극저온시스템
개정번호 : 0
운전 용역 과업지시서 발 행 일 : 2024. 10. 30
페 이 지 : 11/19
3.3.4 기타 가동기간에 필요한 운전 업무
표 7.에 기술한 기타 가동기간에 필요한 운전 업무 또한 계약자의 업무 범위이다. 표 7.에
기술된 내용 중, 위험(긴급)상황 발생 시, 계약자가 i) 지체없이 연구원에게 상황을 전파하고,
ii) 성실히 연구원에서 전달한 긴급조치 사항을 수행했음에도 불구하고 발생한 문제들은 계


---

### openai answer
retriever A 점수: 3점
교대근무로 인해 경험이 분산되고 전원이 모여 통합/정리하기 어렵다는 이유와, 운전 경험 통합·정리 업무가 휴지기간(주간 근무)에 집중 수행된다고 명시되어 있음.

retriever B 점수: 3점
교대근무로 경험 통합 여건이 부족하다는 이유와, 해당 업무를 휴지기간(주간 근무)에 집중 수행한다고 명시되어 있어 답의 모든 요소가 포함됨. 

============================================================
### query:
계약업체가 매월 운전 현황을 정리해서 연구원에 보고해야 하는 공식적인 문서는 무엇이며, 제출 시점은 언제입니까?

### answer:
매월 운전 상황을 요약하여 제출해야 하는 문서는 '월간보고서'이며, 매월 기성 비용을 신청할 때 제출해야 합니다.

---

### retriever A:
계약종료 및 인수인계에 관한 사항
세부내용
○ 수행업체는 본 사업 종료 1개월 전 시스템의 발전 및 안정성을 유지하기 위한 인수인계 계획을 제시하여야 함
- 본 사업수행 기간 동안 취득한 모든 정보 및 생상된 산출물 일체
- 인수인계의 주요 업무, 상세 일정 및 인수인계 대상 주요 산출물 제시
- 인수인계 기간 동안 발생하는 리스크에 대한 방지대책 제시
○ 수행업체는 계약기간 만료 시 차기 수행업체로 선정된 사업자가 본 사업의 업무를 차질없이 수행할 수 있도록 요청 시 통합접수시스템 운영 업무 수행에 필요한 중요사항 등을 인계하여야 하며, 시스템의 효율적인 운영을 위해 하자담보 책임기간 중에는 기술자문에 응하여야 함
- 시스템 구성도, 시스템 현황, DB권한 등
- 통합접수시스템 기동/종료 절차
- 타 시스템 연계 및 적재 절차
- 일반 사용자 및 관리자 매뉴얼 등
○ 인수 및 인계 기간은 차기 시스템 운영 사업 개시일로부터 30일 이내로 하며, 인수 및 인계기간 동안의 용역비용은 지급하지 아니함
산출 정보
인수인계 계획서(사업수행계획서 내 포함 가능)
관련 요구사항
요구사항 출처
요구사항 분류
프로젝트지원 요구사항
요구사항 고유번호
PSR-008
요구사항 명칭
정보화사업 주요점검 및 수행지원
요구사항
상세설명
정의
정보화사업 주요점검 및 수행지원
세부내용
○ 통합접수시스템 관련 점검사항 준수 및 현장점검 수행
○ 보안관리 실태 점검 대응
○ 공공데이터 품질관리(DB) 수준평가 대응 및 데이터 오류 정제 활동 실시
산출 정보
점검별 결과보고서
관련 요구사항
요구사항 출처
□ 품질관리 요구사항
요구사항 분류
· 월별 추진 실적 보고
· 관련부서 업무협의 내용
· 문제점 보고 및 특이사항
매월 말일
비정기
보고
인수인계
보고
· 신규 사업자 선정시 업무인계 결과보고
사업종료
1주전
기타
· 이슈사항 및 문제점 협의 등
이슈발생시
※ 보고형식 및 주기는 추후 협의하여 조정할 수 있음
ㅇ 계약 기간 만료일 전 과업 완료 보고서를 제출하여야 함
구분
보고명
보고내용
시기
사업
종료
완료
보고서
· 용역종료 시 전체 추진단계의 수행결과 보고로 인쇄물 2부, 외장하드 1본으로 제출
· 산출물을 종류별로 분류한 파일
용역종료일
2주일 전
- 용역 완료시 제출하는 최종 결과물(프로그램 소스 코드(주석, 설명서 포함), ERD, 테이블 정의서, 프로그램 정의서 등)은 발주자의 승인을 득한 후 원본 파일을 CD 또는 HDD에 수록하여 산출물과 같이 제출하여야 하고, 프로그램 소스에 대한 형상관리가 가능하도록 협조하여야 함
- 산출물은 기존 산출물을 기반으로 현행화 또는 추가 작성해야 하며, 상호 협의하여 조정 가능
ㅇ 제출 시기에 따라 종류별 산출물을 제출하되, 사업관리 및 검수를 위한 각종 업무 보고를 요청할 경우, 신속하게 작성하여 보고하는 수시보고 체계 마련 ※ 제출 전 주관기관 사전 검토 및 확인
산출정보
착수보고서, 완료보고서, 월간보고서
요구사항 고유번호
PMR-005
요구사항 명칭
인계 조건
요구사항
상세설명
정의
관한 사항
세부
내용
인계 요건
- 사업종료 1개월 전까지 업무인계 계획서를 작성해 진흥원의 승인을 받아야함
- 발주기관과 월 1회 이상 기술이전 등에 대한 협의를 하고, 그 결과를 문서화
- 금회 대처 방법
인 서명 필수
- 향후 대처 방법
- 수치 기록을 위해 계약자가
- 기타 필요한 내용
준비한 PC에 상시 저장
- 매 월간 기성 신청 시 제출
월간보고서 - 월간 운전 상황 요약 O
- 문서 파일 제출
- IBS에서 지정한 기한 내 제
기타 - IBS에서 요구한 결과물 O
출
문서번호 : -
2025년도 중이온가속기용 극저온시스템
개정번호 : 0
운전 용역 과업지시서 발 행 일 : 2024. 10. 30
페 이 지 : 18/19
5. 기타사항
계약자는 본 절에서 기술하는 사항들을 명확히 이해하고 숙지한 뒤, 본 과업을 착수해야
한다. 본 절의 내용에 대한 미숙지로 발생하는 불이익은 계약자의 책임으로 한다.
- 본 과업은 극저온시스템 운전에 대한 단독업무를 수행한다.
(단, 중이온가속기 운영 건전성과 관련된 중대한 운전 사항은 연구원과 협의하여 진행)
- 계약자는 연구원에게서 인수인계 받은 과업 일체 내용을 바탕으로 극저온시스템 운전에
필요한 인원, 인력구성 및 운전근무를 편성한다.
- 극저온시스템의 가동기간과 휴지기간은 중이온가속기 운영 상황에 따라 연구원이 결정하
며, 연구원은 가동기간/휴지기간 변경 최소 2 주 전에는 계약자에게 통지한다.
- 가동기간에는 24시간 운전이 이루어지며, 휴지기간에는 주간 운전이 이루어져야 한다. 휴
지기간에의 주간 근무는 가동기간동안 24시간 운전에 투입된 운전 인력 전원이 수행해야 하
며(휴가자 제외), 계약자는 연구원의 가동기간/휴지기간 선정에 대한 이의 및 각 기간의 운
전 형태(24시간 운전/주간 운전)에 대해 이의를 제기할 수 없다.
문서파일
착수계 제출
(계약일로부터
7일 이내)
중간보고
ㅇ 중간 운영결과보고
문서파일
2024년 8월 중
월간보고
ㅇ 월별 운영결과보고
문서파일
익월 10일 이내
기획안 및
수시보고
ㅇ 기획회의 자료, 홍보안 등 기획안 일체
ㅇ 이슈상황 조치 계획서·결과보고서
문서파일
해당 사항
발생시
최종보고
결과보고(세부추진내용)
- 운영 실적 및 성과
- 통계분석자료, 프로모션 이벤트 결과보고
- 향후 사업개선방향 등 제언
ㅇ 과업 제작결과물 원본 데이터
- 사진, 영상, 홍보제작물 이미지, 제작원고 등 제작물 일체
USB 1개
(문서파일 및 제작결과물 저장)
과업 완수후
10일 이내
ㅇ 원활한 과업추진을 위해 필요시 비정기적인 보고를 요청할 수 있음
ㅇ 사업자는 과업이 완료되기 15일 전에 보고서의 초안을 발주기관에 제출하여 심의를 받고, 수정사항에 대해서는 보완토록 함
※ 제안사(계약상대자)는 보고서 작성 중 항상 발주기관과 긴밀한 협의를 하여야 함
※ 사업 수행과정에서 취득 또는 작성하는 성과품 및 산출물에 지식재산권은 본원과 사업자가 공동으로 소유함을 원칙으로 하며 세부사항은 협의하여 정함
요구사항 고유번호
PMR-003
요구사항 명칭
보안서약 및 책임
요구사항 분류
프로젝트 관리 요구사항
요구사항
상세설명
정의
보안서약 및 책임에 관한 사항
세부
내용
ㅇ 사업에 참여하는 모든 인원은 별도 양식에 의하여 상기 내용을 포함한 보안서약서를 반드시 제출하여야 함
- 계약일로부터 10일 이내 착수보고서 작성 및 착수보고 실시
- 검수 완료 후 완료보고서 작성 및 완료보고 실시
- 의사결정 필요시 및 프로젝트 위험상황 발생 시
산출 정보
주/월간 보고
요구사항 고유번호
PMR-004
요구사항 분류
프로젝트 관리 요구사항
요구사항 명칭
일정계획 및 산출물관리
요구사항
상세설명
정의
일정계획 및 산출물관리
세부내용
❍ 사업수행에 필요한 활동을 도출하여 정확한 활동 기간, 자원, 조직 등을 제시
❍ 제안업체는 본 사업을 기간 내에 완수하기 위해 단계별 추진 일정 및 세부 활동 내용 등이 포함된 일정계획을 제시하여야 함
❍ 제안업체는 일정 계획에 따른 관리방안 및 체계를 제시하여야 함
❍ 제안업체는 본 사업에 대하여 감리를 실시함에 따라 감리에 소요되는 기간을 감안하여 일정 계획을 수립하여야 함
❍ 선정된 사업자는 계약일로부터 14일 이내에 계약서, 제안요청서, 제안서, 과업지시서 등을 근거로, 본 사업을 수행하기 위한 프로젝트 세부 사업추진계획서를 작성·제출하여야 함
산출물 종류
제출시기
수량
비고
사업수행계획서
계약일로부터 10일 이내
1부
중간보고서
(분석서 포함)
분석완료 후 7일 이내
1부
최종보고서
사업종료시
3부
❍ 선정된 사업자는 사업추진과정에서 생산되는 제반 작업 단위별 산출물에 대하여 작업 일정 계획 및 품질보증계획과 연계하여 산출물의 종류, 주요 내용 작성 및 제출 시기 등 제시
산출 정보
N/A
요구사항 고유번호
PMR-005
요구사항 분류
프로젝트 관리 요구사항
요구사항 명칭
개발장비
요구사항
상세설명
정의
개발장비
세부내용
매주 1회
월간보고서
매월 1회
보안점검표
교육훈련 결과보고서
완료 시
업무인수인계서
자료인수인계대장
대표자 보안확약서
보안점검 결과서
(백신설치여부, CMOS 및 윈도우 패스워드 설정여부, 용역업체 PC 완전삭제 등)
전산장비 반출입대장
개인정보 처리 위탁 계약서
해당 시
개인정보 파기 확약서
기타 (각종 보고자료, 요구서류)
주관기관 요구 시
※ 산출물 종류, 제출일정은 주관기관과의 협의 및 내부사정에 따라 변경 가능
Ⅳ. 제안서 작성요령
1
제안서의 효력
□ 제안서에 제시된 내용과 발주자 요구에 따라 수정, 보완, 변경된 제안내용은 계약서에 명시하지 아니하더라도 계약서와 같은 효력이 있음. 다만 계약서에 명시된 경우는 계약서가 우선함
2
제안서 작성 유의사항
□ 제안서 작성 시 유의사항은 다음과 같음
○ 제안사는 제안요청서에 명시된 사항 이외의 본 과업에 필요한 사항을 추가로 제안하거나 변경·제안 가능하며, 이 경우 타당한 사유를 명시하여야 함
○ 제안서는 제시된 제안서 목차 및 제안서 세부 작성지침을 준용하여 각각 세분하여 누락 없이 작성하고, 제안요청서의 요구항목들이 제안서의 어느 부분에 기술되었는지 참조표를 제시
○ 제안서(요약서 포함)는 협상에 의한 계약체결기준(기획재정부 계약예규 2023.6.16) 제 6조 개정·시행에 따라 전자적(온라인)으로 제출하여야 함
○ 제안서는 한글 작성이 원칙이며, 사용된 영문약어에 대해서는 약어표를 제공하고 A4 종 방향 작성을 원칙으로 함
매주 1회
월간보고서
매월 1회
보안점검표
교육훈련 결과보고서
완료 시
업무인수인계서
자료인수인계대장
대표자 보안확약서
보안점검 결과서
(백신설치여부, CMOS 및 윈도우 패스워드 설정여부, 용역업체 PC 완전삭제 등)
전산장비 반출입대장
개인정보 처리 위탁 계약서
해당 시
개인정보 파기 확약서
기타 (각종 보고자료, 요구서류)
주관기관 요구 시
※ 산출물 종류, 제출일정은 주관기관과의 협의 및 내부사정에 따라 변경 가능
Ⅳ. 제안서 작성요령
1
제안서의 효력
□ 제안서에 제시된 내용과 발주자 요구에 따라 수정, 보완, 변경된 제안내용은 계약서에 명시하지 아니하더라도 계약서와 같은 효력이 있음. 다만 계약서에 명시된 경우는 계약서가 우선함
2
제안서 작성 유의사항
□ 제안서 작성 시 유의사항은 다음과 같음
○ 제안사는 제안요청서에 명시된 사항 이외의 본 과업에 필요한 사항을 추가로 제안하거나 변경·제안 가능하며, 이 경우 타당한 사유를 명시하여야 함
○ 제안서는 제시된 제안서 목차 및 제안서 세부 작성지침을 준용하여 각각 세분하여 누락 없이 작성하고, 제안요청서의 요구항목들이 제안서의 어느 부분에 기술되었는지 참조표를 제시
○ 제안서(요약서 포함)는 협상에 의한 계약체결기준(기획재정부 계약예규 2023.6.16) 제 6조 개정·시행에 따라 전자적(온라인)으로 제출하여야 함
○ 제안서는 한글 작성이 원칙이며, 사용된 영문약어에 대해서는 약어표를 제공하고 A4 종 방향 작성을 원칙으로 함


---

### retriever B:
4. 제출문서
계약자는 표 13.와 같은 문서들을 제출일정에 맞춰 연구원에 제출해야 한다.
표 13. 제출문서 목록 및 관련 사항
연구원
제출문서 내용 비고 및 제출일
승인
투입 인력
- 투입 인력 이력 사항 X - 계약 시
이력서
- 최초: 계약 후 1개월 내
작업계획서 - 일별 투입 인력 - 이후: 매주 금요일까지 차주
X
(주간근무표) - 휴가상황 근무표 제출
- 문서 파일 제출
- IBS에서 제공한 양식 파일
극저온시스템 - 매 월간 기성 신청 시 제출
주요 수치 - 극저온시스템 주요 수치 X - 문서 파일 제출
기록문서 - 수치 기록을 위해 계약자가
준비한 PC에 상시 저장
일일근무일지 - 일일 운영 상황 기록 O - 매 월간 기성 신청 시 제출
- 문서 파일 제출
- 수치 기록을 위해 계약자가
인수인계 문서 - (가동기간) 인수인계 내용 X 준비한 PC에 상시 저장
- 수리 사유 발생일
- 매 월간 기성 신청 시 제출
- 수리 필요 대상
수리필요사항 - 문서 파일 제출
- 상세 내용 O
기록 문서 - 수치 기록을 위해 계약자가
- 발견자
준비한 PC에 상시 저장
- 기타 필요한 내용
- 발생한 경우, 기성 신청 시
특이사항, 위험상황 발생 시,
제출 (해당 월 발생 건)
- 상황 내용
- 문서 파일 제출
- 작성자
Lesson learn O - 운전 인력 전원 회람 후 확
- 금회 대처 방법
인 서명 필수
- 향후 대처 방법
- 수치 기록을 위해 계약자가
- 기타 필요한 내용
준비한 PC에 상시 저장
- 매 월간 기성 신청 시 제출
월간보고서 - 월간 운전 상황 요약 O
- 문서 파일 제출
라. 계약 후 제출서류 : 사업수행계획서, 기술적용계획표
1) 사업수행자는 계약일로부터 14일 이내에 사업수행계획서와 기술적용계획표를 제출하고 사업 착수를 완료하여야 함
2) 사업수행계획서에는 제안요청서, 제안서, 협상에 의해 수정·보완·추가된 내용을 반영하여 제출하여야 하며 발주기관의 승인을 받아야 함
제출방법
1
착수계
A4
1
계약 후 7일 이내
서신문서
2
보안관련 문서
(보안서약서, 비밀유지서약서, 개인정보 위탁계약서, 개인정보 보호서약서)
A4
1
계약 후 7일 이내
3
사업수행계획서
A4
8
계약 후 10일 이내
4
착수보고서
A4
8
착수 후 10일 이내
5
완료보고서
A4
8
계약종료 2주 전 까지
6
최종산출물
(개발 산출물, 운영자 매뉴얼, 사용자 매뉴얼, 각종 분석/측정 결과보고서, 기술지원 확약서, 품질관리보고서, 기술적용계획/결과표, 기술이전계획서, 장애대응 매뉴얼, 테스트 수행 계획서/결과서 등)
USB
2
계약종료 2주 전 까지
부터 10일 이내
략, 사업추진내용, 일정, 참여인력, 예산 등
- 사업내용을 중심으로 세부 사업수행계획서를 계약일로부터 10일 이내에 제출하여야 함
※ 계약서, 제안요청서 등을 근거로 작성
발표자료
·사업수행계획서
보고회
간의 1/2 시점(협의)
기간의 1/2 시점에 사업 경과보고 제출 및 중간보고회 개최
※ 기한 및 날짜는 상호 협의하여 정함
발표자료
·중간보고서
보고회
료 전 10일 이내
만료일 15일 전까지 최종보고서 초안 제출
- 계약 만료 전 10일 이내에 최종보고회 개최
- 중간보고 이후의 보완․확정 내용 상호 점검
- 최종보고서 10부 준비 ※ 필요시 발표자료(PPT, PDF 파일 등) 준비
- 용역책임자가 사업 결과 종합 보고
발표자료
·최종보고서
·최종보고서
요약본
: 주 1회
회의) 주 1회, 수행내용 점검, 사업추진 현황관리
생 시
업의 추진상황에 대해 발주기관의 요구가 있을 때에는 수시로 용역 추진 상황 및 진도를 보고하여야 함
계획서
부터 1개월 이내
사 세부수행내용
- 데이터수집 및 자료조사 체크리스트
- 현지 네트워킹을 위한 기념품 제작·배포
※ 영문은 사업대상국 공유 사항으로, 국문의 간소화 버전으로 작성
(국/영문)
·체크리스트
보고서
후 5일 이내
장내용 및 회의록 등 포함
- 현지조사내용 및 현지방문 시 발견사항 등 포함
※ 영문은 사업대상국 공유 사항으로, 국문의 간소화 버전으로 작성
고서
(국/영문)
회의
진방향 및 진행 결과에 대한 논의
※ 회의준비, 회의참석, 회의리딩 및 결과정리
(국문/영문)
·회의록
료일 이전
고회 이후 수정 보완사항 반영하여 최종 산출물 제출
일정관리에 대한 정의
세부내용
◦계약일로부터 사업 완료일까지 매월 사업수행계획서에 제시된 사업수행에 따른 현재의 구체적인 공정 및 진행사항, 업무 추진 상 문제점 및 대안방안 등에 대한 진도 보고서를 작성·제출
산출정보
관련요구사항
일정 계획 관련 요구사항
세부
내용
◦ 사업을 기간 내에 완수하기 위한 개발 단계별 추진일정 및 세부 활동 내용 등이 포함된 개발 일정계획을 제시하여야 함
◦ 사업수행에 필요한 활동을 도출하여 정확한 활동 기간의 산정과 도출된 활동 간의 배열이 합리적인지, 중간목표가 적정하게 제시하며, 각 활동에 적합한 자원을 적절히 할당해야 함
◦ 정기보고 제출
- 사업진행에 대한 업무내용, 진척사항, 기타 특기사항을 기록한 업무일지를 작성하여 주간 단위로 작성․제출하여야 함
◦ 수시보고 방안 제시
- 원활한 과업 추진을 위해 필요시 비정기적인 보고 요청에 대응함
구분
내용
시기
정기
보고
주간보고
- 진도관리(계획대비 실적)
- 문제점 보고 및 특이사항
- 차주계획
협의
월간보고
- 진도관리(계획대비 실적)
- 문제점 보고 및 특이사항
- 향후계획
협의
착수보고
- 사업 추진전략 및 세부 진행계획 보고
- 협조사항 요청
계약 후 14일 이내
중간보고
개발현황 및 현안 보고
협의
완료보고
- 개발결과 보고
사업 완료 시
수시 보고
- 이슈사항 및 문제점 협의 등
협의
◦ 기타 산출물 제출
구분
내용
시기
신원진술서
보안서약서
- 신원진술서 및 보안서약서
계약 후
7일 이내
보안측정
- 보안측정 검증 제출자료
협의
보안확약서
(대표자용/참여자용)
- 비밀 누설 금지 및 산출물 폐기 서약
종결보고 후 7일 이내
◦ 사업이 완료하기 전에 발주자가 시행 및, 운영을 위한 활동을 제시해야 함
◦ 결함에 대한 조치로 인하여 계획된 일정에 영향을 주지 않아야 함
- 테스트 기간에 발견된 결함에 대한 조치율은 100% 조치하여야 하며,
일정
산 출 물
수량
계약체결 후
10일 이내
착수계
- 사업책임자계, 사업수행계획서, 청렴계약서약서, 보안서약서, 참여인력 신원조사 관련 서류, 저작권 및 기타 권리 양도 허락 확인서(붙임 14)
- 조정사항 이행 확약서(붙임 15), 책임 및 역할 분담표(붙임 17)
2부
사업
준공 시
시스템 및 서비스 구성도, 메뉴구조도, 프로그램 목록
개발 프로그램 소스 파일
- Document 및 Comment를 포함한 소스파일 및 실행파일
교육계획서/교육결과보고서, 교육자료(사용자/운영자 매뉴얼 등)
기술이전계획서, 하자보수계획서, 기술적용결과표, 보안확약서(대표 명의)
2부
보고회 시
• 사업 착수, 중간, 완료 보고서(PT자료)
각2부
주1회/
월1회
• 주간/월간 업무보고서(필요시)
2부
필요 시
• 변경요청서(회의록 포함)
2부
• 각종 보고 자료 및 요구 서류 등
요구
수량
라. 산출물 작성 및 제출시기
구분
결과물
제출시기
부수
언어
형태
시설구축
- 설계사 입찰 결과서
계약 체결시
1부
국문
공문접수
- 설계 완료 보고서
(설계 도서, 도면 등 포함)
설계 완료시
3부
국문,영문
공문접수
(제본)
- 시공사 선정 계획 보고
(시공사 입찰 계획서)
현지 시공사 공모 전
1부
국문
공문접수
- 시공사 선정 결과 보고
선정 이후 2주 이내
1부
국문
공문접수
전문가
파견
- 전문가 파견계획서
파견 1주전
1부
국문
공문접수
- 전문가 파견 결과 보고서
파견 종료 후 3주내
1부
국문
공문접수
사업관리
- 착수보고서
계약 후 1개월 이내
1부
국문
공문접수
- 월별 공정 보고
매월 5일 이내
1부
국문
공문접수
- 연도별 실적보고서
연차계약 종료 1개월 이내
1부
국문
공문접수
(제본)
- 연차계약 준공 정산서
연차계약 종료 이전
1부
국문
공문접수
(제본)
- 최종 보고서
사업종료후 1개월 이내
5부
국문,영문
공문접수
(제본)
※ 기타 이행시기 및 주요 산출물 목록, 부수, 양식, 사양 등은 기술협상 단계에서 사무국과 협의하여 결정
Ⅲ
입찰 안내사항
② 공급자는 모든 과업에 대한 월간진도보고서(협력업체 내용 포함)를 매월 말 다음 각
호의 정보현황을 포함하여 연구원에 제출하여야 한다.
6. 제안서 제출 일정 및 방법
1) 제출기한 및 제출방법 : ‘입찰공고문’ 참조
2) 문의처 : ‘입찰공고문’ 참조
3) 계약 후 제출 서류
- 사업수행자는 계약일로부터 7일 이내에 사업수행계획서를 제출하고 사업착수를 완료하여야 함
※ 사업수행계획서에는 제안요청서, 제안서, 협상에 의해 수정·보완·추가된 내용을 반영하고 산출 내역서를
포함하여 제출하여야 하며 발주기관의 승인을 받아야 함
산출물 작성 및 제출일정
세부
내용
○ 사업 단계별 산출물은 제출일정에 맞추어 제출
구분
주 요 내 용
수량
제출일정
착수
보고서
○ 제안요청서, 제안서, 계약서, 과업내용서 등을 근거로, 본 사업을 수행하기 위한 프로젝트 세부 추진계획서를 작성하여 제출
1부
계약
착수일
14일
이내
월간
보고서
○ 계약일로부터 사업완료일까지 매월 말 기준 세부 추진계획서에 제시된 사업수행 절차에 따른 현재의 구체적인 공정 및 진행사항, 업무 추진 상 문제점 및 대한 등에 대한 진도 보고서 제출
- 추진계획 대비 실적 분석
- 주요 추진내용 및 단계별 산출내역
- 차주·월 계획 사항 및 계획 변경 사항
- 주요 의사결정 및 협조 및 기타 사항
1부
매월
중간
보고서
○ 추진실적에 대한 분석결과와 향후 추진세부계획 점검 제출
- 추진현황 분석결과 내용(현황과 문제점, 개선요구 사항 및 과제)
- 사용자 요구사항 분석서, 설계서 등
- 개발 현황 및 향후 추진일정 등
1부
상세
설계
완료 후
14일
이내
완료
보고서
○ 완료보고 시 제안요청서, 제안서, 계약서, 과업지시서 등 용역설계서의 업무 범위에 포함된 최종 산출물에 대한 완료보고서 제출
○ 사업 종료 후 1개월 이내에 Source, DB 백업자료 등 업무 범위에 포함된 모든 사항에 대한 최종산출물 제출 (인쇄본 및 CD 각 2매 및 관련 원본파일)
- 원본 소스코드, 콘텐츠 제작자료 일체
- 데이터베이스 스키마 및 개체관계도 등
- 개발보고서(프로그램 개발명세서, 메뉴구성도 등)
- 보안확약서(대표자)
- S/W라이센스 인증서
1부
사업
종료
14일
이전


---

### openai answer
retriever A 점수: 3점
월간보고서 명시 및 “월간 운전 상황 요약”, “매 월간 기성 신청 시 제출”이 그대로 포함되어 있어 문서명과 제출 시점이 모두 확인됨.

retriever B 점수: 3점
제출문서 목록에서 “월간보고서 - 월간 운전 상황 요약”이 있고, 같은 목록에 “매 월간 기성 신청 시 제출” 문구가 제시되어 문서명과 제출 시점이 모두 충족됨. 

# search algorithm results

## Query 1

## sim_results

============================================================
### query:
아이돌봄 인적성 검사 시스템 운영 사업의 주요 목적은 무엇인가?

### answer:
인적성 검사 수행을 지원하고, 돌봄활동과 연계한 사후관리 및 인력 검증을 강화하는 것이다.

---

### retriever A:
◦원할한 인적성 검사 수행을 위한 정보시스템 지원 필요
◦돌봄활동과 연계한 사후관리 기반을 강화하여 이용자 만족도 향상
◦아이돌봄인력 인·적성에 기초한 돌봄인력 관리 강화 필요
◦정보시스템 개인정보보호법 준수를 위한 보호조치
2
사업 개요
□ 사 업 명: 아이돌봄인력 인적성 검사 정보시스템 운영
- 2024. 정보시스템 구축에 따른 유지보수 사업
□ 사업기간: 계약일 ~ 2025. 12. 31.
□ 소요예산: 172,700,000원(부가세 포함)
□ 추진방식: 제한경쟁입찰(협상에 의한 계약)
3
사업 범위
□ 인·적성 프로그램 기능수정 및 안정화 기술지원
◦인·적성 시스템의 안정적인 정보화 지원을 위한 상시 유지관리
체계 지원
◦인·적성 검사 변경에 따른 프로그램 수정 및 결과 반영
◦웹 접근성, 기술적 보안 등 전자정부 가이드라인 준수
□ 개인정보보호 및 정보보안 강화
◦개인정보보호 및 정보보호 강화 조치(방안)
◦정보시스템 운영을 위한 물리적, 기술적 보안 대책 제시
□ 안정적인 클라우드 환경 운영 및 제공
◦클라우드 무중단 환경 제공
◦클라우드 환경 운영을 위한 기술적 유지관리 지원
◦시스템 용을 위한 SW 및 외부 서비스 연계
4
기대 효과
□ (안전한 돌봄 환경 조성) 아이돌봄인력 선별 및 검증 강화 통한 안심하고 맡길 수 있는 돌봄서비스 제공
□ (효율성 제고) 아이돌봄인력 대상 인적성 검사를 편리하게 실시하여 돌봄인력 검증 절차 효율성 제고
□ (정책지원 및 제도개선 지원) 체계적인 검사 결과를 축적하고 정책 수립지원을 위한 기초자료를 제공, 지속 가능한 아이돌봄서비스 제도개선에 활용
□ (안전한 돌봄 환경 조성) 아이돌봄인력 선별 및 검증 강화 통한 안심하고 맡길 수 있는 돌봄서비스 제공
□ (효율성 제고) 아이돌봄인력 대상 인적성 검사를 편리하게 실시하여 돌봄인력 검증 절차 효율성 제고
□ (정책지원 및 제도개선 지원) 체계적인 검사 결과를 축적하고 정책 수립지원을 위한 기초자료를 제공, 지속 가능한 아이돌봄서비스 제도개선에 활용
□ (돌봄인력 관리) 데이터를 기반으로 하는 아이돌봄인력에 대한 효율적 관리·감독 가능
Ⅱ. 사업 현황
1
업무현황
□ 인·적성검사 실시 정보시스템 지원
ㅇ 인·적성검사 및 실시를 위한 정보 시스템 제공
ㅇ 정보시스템 안정화 및 운영지원
□ 인·적성검사 운영
ㅇ 인적성 검사 실시 및 수행 방법 안내
ㅇ 인적성 검사 수행 모니터링 및 결과 분석
ㅇ 검사 실시 운영자 권한 부여 및 인증 관리
2
정보화 현황
□ 시스템 구성도
□ 정보시스템 구성도 요청 시 별도 공개
※ 행정기관 및 공공기관 정보시스템 구축 및 운영 지침(행정안전부 고시 제 2024-66호, 2024.8.28.) 제 17조(제안요청서 보안사항 등)에 따라, 시스템 구성도 및 상세자료 등은 공개하지 않으며, 입찰에 참가하고자 하는 사업자의 요청이 있는 경우 보안서약서를 제출 후 담당자 입회하에 제한적으로 열람 가능
□ 시스템(장비)·소프트웨어 운영현황
AS-IS
TO-BE (2025년)
ㅇ인·적성 검사 관리기능 및 사용자화면 개발
※ 클라우드 환경 기반 개발
ㅇ인·적성 검사 기능 최적화 및 운영 기술지원
-인·적성 검사 고도화에 따른 수정 반영
(검사도구 4종)
※사업 운영에 따라 변경될수 있음


---

### retriever B:
기대 효과
□ (안전한 돌봄 환경 조성) 아이돌봄인력 선별 및 검증 강화 통한 안심하고 맡길 수 있는 돌봄서비스 제공
□ (효율성 제고) 아이돌봄인력 대상 인적성 검사를 편리하게 실시하여 돌봄인력 검증 절차 효율성 제고
□ (정책지원 및 제도개선 지원) 체계적인 검사 결과를 축적하고 정책 수립지원을 위한 기초자료를 제공, 지속 가능한 아이돌봄서비스 제도개선에 활용
□ (돌봄인력 관리) 데이터를 기반으로 하는 아이돌봄인력에 대한 효율적 관리·감독 가능
Ⅱ. 사업 현황
1
업무현황
□ 인·적성검사 실시 정보시스템 지원
ㅇ 인·적성검사 및 실시를 위한 정보 시스템 제공
ㅇ 정보시스템 안정화 및 운영지원
□ 인·적성검사 운영
ㅇ 인적성 검사 실시 및 수행 방법 안내
ㅇ 인적성 검사 수행 모니터링 및 결과 분석
ㅇ 검사 실시 운영자 권한 부여 및 인증 관리
2
정보화 현황
□ 시스템 구성도
□ 정보시스템 구성도 요청 시 별도 공개
※ 행정기관 및 공공기관 정보시스템 구축 및 운영 지침(행정안전부 고시 제 2024-66호, 2024.8.28.) 제 17조(제안요청서 보안사항 등)에 따라, 시스템 구성도 및 상세자료 등은 공개하지 않으며, 입찰에 참가하고자 하는 사업자의 요청이 있는 경우 보안서약서를 제출 후 담당자 입회하에 제한적으로 열람 가능
□ 시스템(장비)·소프트웨어 운영현황
AS-IS
TO-BE (2025년)
ㅇ인·적성 검사 관리기능 및 사용자화면 개발
※ 클라우드 환경 기반 개발
ㅇ인·적성 검사 기능 최적화 및 운영 기술지원
-인·적성 검사 고도화에 따른 수정 반영
(검사도구 4종)
□ 시스템(장비)·소프트웨어 운영현황
AS-IS
TO-BE (2025년)
ㅇ인·적성 검사 관리기능 및 사용자화면 개발
※ 클라우드 환경 기반 개발
ㅇ인·적성 검사 기능 최적화 및 운영 기술지원
-인·적성 검사 고도화에 따른 수정 반영
(검사도구 4종)
※사업 운영에 따라 변경될수 있음
ㅇ무중단 클라우드 기반 제공
ㅇ클라우드 안정화 운영기반 및 기술지원
SW 도입 현황
SW 명칭
용도
비고
리포팅 툴
OZ Report
인·적성 결과지 출력
발주처 제공
□ 시스템 장비 현황 요청 시 별도 공개
※ 행정기관 및 공공기관 정보시스템 구축 및 운영 지침(행정안전부 고시 제 2024-66호, 2024.8.28.) 제 17조(제안요청서 보안사항 등)에 따라, 시스템 구성도 및 상세자료 등은 공개하지 않으며, 입찰에 참가하고자 하는 사업자의 요청이 있는 경우 보안서약서를 제출 후 담당자 입회하에 제한적으로 열람 가능
Ⅲ. 사업 추진방안
1
추진 목표
□ `24년 개발된 인·적성 검사 운영을 지원하고, 데이터 수집 및 분석을 위한 정보체계 관리
□ 인·적성 검사 정보시스템 안정적 기반 지원 및 보안 강화
2
추진체계
□ 추진체계도
발주기관
<한국건강가정진흥원>
협조부서
<경영지원부/
소통협력실>
주관부서
<돌봄지원부>
관련부처
<여성가족부>
주관사업자
<사업수행업체>
□ 추진 주체별 역할
구 분
주요 업무 내용
주관부서
ㅇ 사업총괄 관리 및 시행
ㅇ 사업계획서 수립 및 제안요청서 작성
ㅇ 개발내역 업무정의 및 과업내용 제시
ㅇ 개발관련 자료제공 및 시스템 인수
ㅇ 사업수행 관리 및 검사, 결과 평가
협조부서


---

### openai answer
retriever A 점수: 3점
- 인적성 검사 수행 지원, 돌봄활동과 연계한 사후관리 강화, 아이돌봄인력 검증/관리 강화가 모두 명시되어 있어 답변의 핵심이 모두 포함됨.

retriever B 점수: 2점
- 인적성 검사 운영 지원 및 인력 선별·검증 강화는 포함되나, 돌봄활동과 연계한 사후관리 강화를 명시적으로 담고 있지 않아 부분 충족임. 

============================================================
### query:
아이돌봄 시스템에서 개인정보 보호를 위해 요구되는 조치는 무엇인가?

### answer:
접속기록 관리, 내부관리계획 수립 등 기술적·관리적 보호조치를 수행해야 한다.

---

### retriever A:
- 개인정보의 기술절·관리적 보호조치 기준(개인정보위)
- 표준 개인정보 보호 지침(개인정보위)
가명정보의 결합 및 반출 등에 관한 고시(개인정보위)
개인정보 영향평가에 관한 고시(개인정보위)
※ 고시에 따라 아래 기준 이상의 개인정보 처리 시 발주처에서 개인정보 영향평가를 발주하며 영향평가결과를 반영하여 시스템을 구축해야 함
내 용
1
구축‧운용 또는 변경하려는 개인정보파일로서 5만명 이상의 정보주체에 관한 민감정보 또는 고유식별정보의 처리가 수반되는 개인정보파일
2
구축‧운용하고 있는 개인정보파일을 해당 기관 내부 또는 외부에서 구축‧운용하고 있는 다른 개인정보파일과 연계하려는 경우로서 연계결과 50만명 이상의 정보주체에 관한 개인정보가 포함되는 개인정보파일
3
구축‧운용 또는 변경하려는 개인정보파일로서 100만명 이상의 정보주체에 관한 개인정보파일
4
개인정보 영향평가를 받은 후에 개인정보 검색체계 등 개인정보파일의 운용체계를 변경하려는 경우 그 개인정보파일
○ 개인정보 안전성 확보조치 기준 이행·반영
- 개인정보 열람 시 접속기록(최소 2년 이상) 보관·관리
- 개인정보 접근권한 부여·변경·말소 이력(최소 3년 이상) 보관·관리
- 고유식별번호 및 비밀번호 생체인식정보는 안전한 암호 알고리즘을 사용 하여 저장
○ 데이터의 보안 강화를 위해 SSL서비스 도입
○ 개인정보보호를 위한 DB 암호화하여 자료 유출방지
- 인증서 기반(GPKI) 로그인 방식 적용 후 ID/PW 로그인
- 정보보안 강화를 위한 시스템에 각종 패치, 환경설정 지원 등
◦ 보안 요구사항
요구사항 고유번호
SER-001
요구사항 명칭
개인정보 처리
요구사항 분류
보안 요구사항
요구사항
상세설명
정의
개인정보 처리 및 암호화
세부
내용
ㅇ 개인정보를 보유할 경우에는 관련 개인정보를 암호화하여야 함
ㅇ 서비스의 보안성 강화를 위하여 SSL 보안 통신을 실시하여야 함
ㅇ 사업의 암호화 대상을 식별하고 암호화 도구를 활용하여 암복호화 방안을 마련해야 함
- 고유식별정보 : 주민번호, 여권번호, 운전면허정보 등
ㅇ개인을 식별할 수 있는 정보(사용자 인증 정보, 패스워드 등)를 운영DB 또는 개발DB에 저장할 경우, 암호화하여 저장하며, 소스코드에 직접 하드코딩하지 않음(패스워드는 단방향 암호화 처리, SHA-256 이상 암호화 알고리즘 사용)
ㅇ 개인정보 취급자가 개인정보처리 시스템에 접속하는 경우 접속 로그를 시스템에 최소 1년 이상 기록 관리해야 함(5만명 이상의 정보주체에 관하여 개인정보를 처리하거나 고유식별정보 또는 민감정보를 처리하는 개인정보처리시스템의 경우에는 2년 이상 보관)
◈ 필수 관리 항목
• 개인정보취급자 식별정보 : ID ୴Ā • 날짜 및 시간 : 접속일시
• 접속자 IP 주소 : 접속자 정보
• 처리한 정보주체 정보 : 정보주체 식별정보(계정, 회원번호 등)
• 수행업무 : 열람, 수정, 삭제, 인쇄, 입력, 다운로드 등
ㅇ 개인정보 취급자가 개인정보를 처리할 경우 해당 접근 로그를 시스템에 기록할 것
ㅇ 개인정보처리방침에 따라 개인정보를 처리/파기해야 하며 백업데이터 생성 금지


---

### retriever B:
6. 그 밖에 개인정보의 보호를 위하여 필요한 사항
-295-
1. 개인정보의 안전성 확보조치 철저(「개인정보보호법」제29조)
❍ 개인정보처리자는 개인정보가 분실·도난·유출·변조 또는 훼손되지 않도록 다음 각 호의 안전성 확보조치를 하여야한다.
구 분
주 요 내 용
내부관리계획(제3조)
◦내부관리계획 포함사항 명시 ※개인정보 보호책임자 지정, 개인정보취급자 교육 등
◦개인정보 처리 업무 위탁 시 관리·감독 사항
접근권한 관리(제4조)
◦개인정보처리시스템의 접근권한 부여․변경․말소 기록 최소 3년간 보관
◦안전한 비밀번호 작성규칙 수립․적용
접근통제(제5조)
◦접근통제 시스템 설치․운영 의무화 ※접근통제 시스템 : 침입차단(Firewall), 침입탐지(IDS) 기능 포함
◦주민번호 처리 시 추가 정보 확인, 모바일 기기 보호조치 조항, 홈페이지에서 고유식별정보를 처리할 경우 년 1회 취약점 점검 조치
개인정보 암호화
(제6조)
◦암호화 대상 : 고유식별정보(주민등록번호, 여권번호, 운전면허번호, 외국인등록번호), 비밀번호, 바이오정보
◦암호화 기준
구 분
암 호 화 기 준
송․수신시
모든 정보 암호화 전송
저장시
비밀번호,
바이오정보
비밀번호는 단방향 암호화,
바이오정보는 양방향 암호화
고유식별정보
ⓛ 인터넷 구간,공통(DMZ) 구간 저장시 : 의무적 암호화
보안 요구사항
상세
설명
정의
개인정보의 안전성 확보를 위한 기술적 조치사항
세부
내용
 개인정보를 처리함에 있어 『개인정보 보호법』, 『개인정보의 안전성 확보조치 기준』 등 관련 법규 및 규정을 준수하여야 함
 접근 권한 관리
- 개인정보처리시스템에 접속할 수 있는 사용자계정을 발급하는 경우, 개인정보취급자 별로 사용자계정을 발급하여야 하며, 취급자의 역할에 따라 차등부여할 수 있어야 함
- 개인정보 처리내역에 대한 책임 추적성을 확보하기 위하여 접근권한 부여, 변경, 말소에 대한 내역을 이력 관리 또는 조회가 가능하여야 하며, 해당 기록을 최소 3년간 보관할 수 있도록 구현해야 함
예) 신청자 정보, 신청일시, 승인자 및 발급자 정보, 신청 및 발급 사유 등
- 개인정보처리시스템에 접근하여 개인정보를 생성수정삭제열람 등을 하고자 하는 경우 다음 사항을 포함하는 정보이용 내역을 기록하고 남길 수 있도록 시스템을 구현하여야 함.


---

### openai answer
retriever A 점수: 2점
- 접속기록 관리와 암호화 등 기술적·관리적 보호조치는 명확히 포함되어 있으나, 내부관리계획 수립에 대한 명시적 근거가 보이지 않음.

retriever B 점수: 3점
- 내부관리계획(제3조) 명시, 접근권한 기록·정보이용내역 기록 등 접속기록 관리 포함으로, 기술적·관리적 보호조치 전반이 답변 내용과 모두 일치함. 

============================================================
### query:
2025 구미 아시아육상경기선수권대회의 개최 기간은 언제인가?

### answer:
2025년 5월 27일부터 5월 31일까지 총 5일간 개최된다.

---

### retriever A:
종합정보시스템 및 홈페이지 등 구축 용역 제안요청서
1
대회개요
아시아의 꿈, 구미에서 세계로
 The dream of ASIA, From GUMI to the World 
비전
ㅇ (대 회 명) 2025 구미아시아육상경기선수권대회
ㅇ (기 간) 2025. 5. 27.(화) ~ 5. 31.(토) 5일간
ㅇ (장 소) 구미시민운동장 일원
ㅇ (규 모) 아시아 45개국 1,200여명 / 선수단 및 임원
ㅇ (주 최) 아시아육상연맹(Asian Athletics, AA)
ㅇ (주 관) 2025 구미아시아육상경기선수권대회 조직위원회
ㅇ (경기종목) 45개 종목
구 분
종 목 명
남자
(22)
100m, 200m, 400m, 800m, 1,500m, 5,000m,
10,000m, 110mH, 400mH, 3,000m장애물달리기,
20km경보, 4×100mR, 4×400mR, 높이뛰기,
장대높이뛰기, 멀리뛰기, 세단뛰기, 포환던지기,
원반던지기, 해머던지기, 창던지기, 10종 경기
여자
(22)
100m, 200m, 400m, 800m, 1,500m, 5,000m,
10,000m, 100mH, 400mH, 3,000m장애물달리기,
20km경보, 4×100mR, 4×400mR, 높이뛰기,
장대높이뛰기, 멀리뛰기, 세단뛰기, 포환던지기,
원반던지기, 해머던지기, 창던지기, 7종 경기
남·여
(1)
4×400m 믹스릴레이
요
2
과업개요
ㅇ (사 업 명) 2025 구미아시아육상경기선수권대회 종합정보시스템 및 홈페이지 등 구축 용역
ㅇ (사업기간) 계약일 ~ 2025. 6. 6.(금)
ㅇ (사업위치) 시민운동장 일원
2024. . .
대표자 : (인)
대리인 : (인)
2025 구미아시아육상경기선수권대회 조직위원장 귀하
※ 첨부서류 : 재직증명서, 신분증
※ 위임장에 사용하는 도장은 대표자 인감과 같아야 함.
※ 대리 접수로 인하여 발생하는 모든 문제는 대표자 본인이 책임짐
【 서식 3 】
재 직 증 명 서
성 명
생 년 월 일
부서/직위
소 속
주 소
재직기간
위와 같이 당사에 재직 중임을 증명합니다.
2024. . .
ྠĀ ྠĀ ྠĀ ྠĀ ྠĀ ྠĀ 업 체 명 :
ྠĀ ྠĀ ྠĀ ྠĀ ྠĀ ྠĀ 주 소 :
ྠĀ ྠĀ ྠĀ ྠĀ ྠĀ ྠĀ 대 표 자 : (인)
2025 구미아시아육상경기선수권대회 조직위원장 귀하
【 서식 4 】
사 용 인 감 계
사용인감
는(은) 2025 구미아시아육상경기선수권대회 조직위원회와 거래함에 있어 입찰, 계약 및 이에 수반되는 모든 행위에 대하여 위의 인감을 사용하고자 하며, 동 인감을 사용함으로써 민․형사상의 문제가 발생할 경우 이(가) 책임을 질 것을 확인합니다.
붙임 : 인감(법인)증명서 1부. 끝.
2024년 월 일
ྠĀ ྠĀ ྠĀ ྠĀ ྠĀ ྠĀ 주 소 :
ྠĀ ྠĀ ྠĀ ྠĀ ྠĀ ྠĀ 상 호 :
ྠĀ ྠĀ ྠĀ ྠĀ ྠĀ ྠĀ 대표자 : (인)
2025 구미아시아육상경기선수권대회 조직위원장 귀하
【 서식 5 】
서 약 서
당사는 2025 구미아시아육상경기선수권대회 종합정보시스템 및 홈페이지 등 구축 용역 참가신청과 관련하여 모든 서류와 증빙자료를 신의와 성실의 원칙에 입각하여 작성 제출하며, 아래 내용을 준수할 것을 확약하고 이에 서약서를 제출합니다.


---

### retriever B:
종합정보시스템 및 홈페이지 등 구축 용역 제안요청서
1
대회개요
아시아의 꿈, 구미에서 세계로
 The dream of ASIA, From GUMI to the World 
비전
ㅇ (대 회 명) 2025 구미아시아육상경기선수권대회
ㅇ (기 간) 2025. 5. 27.(화) ~ 5. 31.(토) 5일간
ㅇ (장 소) 구미시민운동장 일원
ㅇ (규 모) 아시아 45개국 1,200여명 / 선수단 및 임원
ㅇ (주 최) 아시아육상연맹(Asian Athletics, AA)
ㅇ (주 관) 2025 구미아시아육상경기선수권대회 조직위원회
ㅇ (경기종목) 45개 종목
구 분
종 목 명
남자
(22)
100m, 200m, 400m, 800m, 1,500m, 5,000m,
10,000m, 110mH, 400mH, 3,000m장애물달리기,
20km경보, 4×100mR, 4×400mR, 높이뛰기,
장대높이뛰기, 멀리뛰기, 세단뛰기, 포환던지기,
원반던지기, 해머던지기, 창던지기, 10종 경기
여자
(22)
100m, 200m, 400m, 800m, 1,500m, 5,000m,
10,000m, 100mH, 400mH, 3,000m장애물달리기,
20km경보, 4×100mR, 4×400mR, 높이뛰기,
장대높이뛰기, 멀리뛰기, 세단뛰기, 포환던지기,
원반던지기, 해머던지기, 창던지기, 7종 경기
남·여
(1)
4×400m 믹스릴레이
요
2
과업개요
ㅇ (사 업 명) 2025 구미아시아육상경기선수권대회 종합정보시스템 및 홈페이지 등 구축 용역
ㅇ (사업기간) 계약일 ~ 2025. 6. 6.(금)
ㅇ (사업위치) 시민운동장 일원
1. 기본정보
사업명
2025 구미아시아육상경기선수권대회 종합정보시스템 및 홈페이지 등 구축 용역
영향평가단계
□ 예산편성
□ 그 외 필요시
■ 사업발주
□ 재평가
주요 내용
대회 기본 자료 관리 및 대회 계측기 연동 등
사업기간
(또는 개발기간)
계약일 ~ 2025년 6월
구분
붙임 : 산출내역서 각 1부
2024. . .
제안업체 인(사용인감)
2025 구미아시아육상경기선수권대회 조직위원장 귀하
※ 제안금액은 반드시 한글로 띄어쓰기 없이 붙여 쓰고 아라비아 숫자는 괄호 안에 기재할 것.
【 서식 9-1 】
산 출 내 역 서
구 분
항 목
금 액
구 성 비
비 고
순
용
역
원
가
소 계
소 계
소 계
소 계
일반관리비 ( )%
이 윤 ( )%
총 원 가
부 가 가 치 세
총 용 역 비
※ 작성내용에 따라 줄 수 증감가능
【 서식 10 】
접수번호
2025 구미아시아육상경기선수권대회 종합정보시스템 및 홈페이지 등 구축 용역
정량적 평가 제안서
2024년 월 일
업 체 명
※ 주의사항


---

### openai answer
retriever A 점수: 3점
대회 기간이 “2025. 5. 27.(화) ~ 5. 31.(토) 5일간”으로 명시되어 있어 정답(날짜와 총 5일)과 완전히 일치함.

retriever B 점수: 3점
대회 기간이 “2025. 5. 27.(화) ~ 5. 31.(토) 5일간”으로 명시되어 있어 정답(날짜와 총 5일)과 완전히 일치함. 

============================================================
### query:
구미 아시아육상경기선수권대회 홈페이지에서 제공되는 주요 콘텐츠 항목에는 무엇이 포함되는가?

### answer:
대회소개, 경기일정 및 결과, 메달순위, 참가선수 정보, 커뮤니티 등이 포함된다.

---

### retriever A:
-경기참가자(Start List), 경기결과(Official Result, Summary) PDF 파일 제공
<다음 정보를 홈페이지에 전송>
Level1
Level2
기능 설명
경기
일정
전체 경기 일정
대회기간 전체 일정 제공
일자별 경기 일정
일자별 세부 경기 일정 제공
세부 종목별 경기 일정
세부 종목별 경기 일정 제공
선수별 경기 일정
선수별 경기 일정 제공
메달
국가/선수별 메달 현황
국가/선수별 메달 현황 제공
종목별 메달 현황
종목별 메달현황 및 메달수상자 정보 제공
경기
정보
스타트 리스트
세부종목별 편성 조 및 출전선수 목록 제공
대진 정보
세부 종목별 대진정보 조회 및 출력
(경기진행 상태에 따른 결과 반영 포함)
경기 결과
각 조별, 종목별 경기 결과 조회 및 출력
선수
정보
국가별 선수 정보
국가별 참가선수 목록 및 정보 조회
세부 종목별 선수 정보
세부 종목별 참가선수 목록 및 정보 조회
순위 정보
세부종목별 참가선수 순위 정보 목록 조회
과업내용
명칭
대분류
대회 종합정보시스템 S/W 개발
소분류
보고서관리시스템 개발
과업내용
데이터 출력 기능
<PDF 및 Excel, 증서 페이지 등 출력>
-참가선수 신상정보
-국가별/세부 종목별 참가선수 리스트
-심판/임원 신상정보
-국가별/세부 종목별 심판/임원 리스트
-세부 종목별, 예결선 Start List 및 조 편성
-세부 종목별 경기 결과 (Live Result, Official Result, Summary)
-세부 종목별 메달 현황 및 신기록 현황
-참가증서
ː참가선수(종목별 1위~6위 증서 및 참가증서 필수)
 운영 매뉴얼 또는 지침서 제공 및 시스템 운영 관계자 교육
ㅇ (현장IT팀 상주 및 시스템 운영)
 사전 등록, 경기 진행 및 결과 출력 시스템 등 운영
 대회시작 4일전~대회 종료일까지 경기장 상주
 경기장/비경기장별 운영 인원 배치
ㅇ (홈페이지 구축)
 메뉴 구성도(안)
아시아육상경기선수권대회 메인
대회소개
소식지
경기일정/결과
메달순위
육상경기
커뮤니티
환영인사
공지사항
경기일정
국가별순위
경기종목
안내사항
대회개요
보도자료
행사일정
선수별순위
참가선수
아카이브
대회상징물
사진자료
시설안내
유실물센터
조직위안내
자주묻는질문
홍보영상
자원봉사자
찾아오시는 길
구미시 안내
메뉴 상세설명(안)
메뉴명
부메뉴
주요 내용
메인 화면
◦2025 구미 아시아육상경기선수권대회를 대표할 수 있는 디자인 (공식 상징물 활용 등)
◦활용빈도가 높은 메뉴 배치 및 팝업배너
대회 소개
환영인사
◦대회 개최에 대한 인사 말씀
대회개요
◦대회명칭, 대회기간, 대회장소, 참가규모, 참가국, 참가인원, 종목, 주최, 주관 등
대회상징물
◦대회마크, 마스코트, 포스터, 슬로건
조직위안내
◦조직위원회·아시아육상추진단 조직도
홍보영상
◦홍보영상, 유튜브 쇼츠 영상, 축하 메시지 등
찾아오시는 길
◦경기장 위치, 교통수단별 길 안내
소식지
공지사항
◦대회 관련 공지사항 등
보도자료
◦대회 관련기사 스크랩 자료
사진자료
◦개·폐회식 사진, 각 종목별 경기 이미지 게시
경기일정
/
결과
경기일정
◦경기장 및 개최 일자, 시간, 라운드, 진행상태
◦각 종목별 경기결과, 종합순위
행사일정
◦개·폐회식 정보 및 식순 안내


---

### retriever B:
 운영 매뉴얼 또는 지침서 제공 및 시스템 운영 관계자 교육
ㅇ (현장IT팀 상주 및 시스템 운영)
 사전 등록, 경기 진행 및 결과 출력 시스템 등 운영
 대회시작 4일전~대회 종료일까지 경기장 상주
 경기장/비경기장별 운영 인원 배치
ㅇ (홈페이지 구축)
 메뉴 구성도(안)
아시아육상경기선수권대회 메인
대회소개
소식지
경기일정/결과
메달순위
육상경기
커뮤니티
환영인사
공지사항
경기일정
국가별순위
경기종목
안내사항
대회개요
보도자료
행사일정
선수별순위
참가선수
아카이브
대회상징물
사진자료
시설안내
유실물센터
조직위안내
자주묻는질문
홍보영상
자원봉사자
찾아오시는 길
구미시 안내
메뉴 상세설명(안)
메뉴명
부메뉴
주요 내용
메인 화면
◦2025 구미 아시아육상경기선수권대회를 대표할 수 있는 디자인 (공식 상징물 활용 등)
◦활용빈도가 높은 메뉴 배치 및 팝업배너
대회 소개
환영인사
◦대회 개최에 대한 인사 말씀
대회개요
◦대회명칭, 대회기간, 대회장소, 참가규모, 참가국, 참가인원, 종목, 주최, 주관 등
대회상징물
◦대회마크, 마스코트, 포스터, 슬로건
조직위안내
◦조직위원회·아시아육상추진단 조직도
홍보영상
◦홍보영상, 유튜브 쇼츠 영상, 축하 메시지 등
찾아오시는 길
◦경기장 위치, 교통수단별 길 안내
소식지
공지사항
◦대회 관련 공지사항 등
보도자료
◦대회 관련기사 스크랩 자료
사진자료
◦개·폐회식 사진, 각 종목별 경기 이미지 게시
경기일정
/
결과
경기일정
◦경기장 및 개최 일자, 시간, 라운드, 진행상태
◦각 종목별 경기결과, 종합순위
행사일정
◦개·폐회식 정보 및 식순 안내
종합정보시스템 및 홈페이지 등 구축 용역 제안요청서
1
대회개요
아시아의 꿈, 구미에서 세계로
 The dream of ASIA, From GUMI to the World 
비전
ㅇ (대 회 명) 2025 구미아시아육상경기선수권대회
ㅇ (기 간) 2025. 5. 27.(화) ~ 5. 31.(토) 5일간
ㅇ (장 소) 구미시민운동장 일원
ㅇ (규 모) 아시아 45개국 1,200여명 / 선수단 및 임원
ㅇ (주 최) 아시아육상연맹(Asian Athletics, AA)
ㅇ (주 관) 2025 구미아시아육상경기선수권대회 조직위원회
ㅇ (경기종목) 45개 종목
구 분
종 목 명
남자
(22)
100m, 200m, 400m, 800m, 1,500m, 5,000m,
10,000m, 110mH, 400mH, 3,000m장애물달리기,
20km경보, 4×100mR, 4×400mR, 높이뛰기,
장대높이뛰기, 멀리뛰기, 세단뛰기, 포환던지기,
원반던지기, 해머던지기, 창던지기, 10종 경기
여자
(22)
100m, 200m, 400m, 800m, 1,500m, 5,000m,
10,000m, 100mH, 400mH, 3,000m장애물달리기,
20km경보, 4×100mR, 4×400mR, 높이뛰기,
장대높이뛰기, 멀리뛰기, 세단뛰기, 포환던지기,
원반던지기, 해머던지기, 창던지기, 7종 경기
남·여
(1)
4×400m 믹스릴레이
요
2
과업개요
ㅇ (사 업 명) 2025 구미아시아육상경기선수권대회 종합정보시스템 및 홈페이지 등 구축 용역
ㅇ (사업기간) 계약일 ~ 2025. 6. 6.(금)
ㅇ (사업위치) 시민운동장 일원


---

### openai answer
retriever A 점수: 3점
홈페이지 메뉴에 ‘대회소개’, ‘경기일정/결과’, ‘메달순위’, ‘참가선수’, ‘커뮤니티’가 명시되어 있어 정답의 모든 항목이 포함됨.

retriever B 점수: 3점
메뉴 구성에 ‘대회소개’, ‘경기일정/결과’, ‘메달순위’, ‘참가선수’, ‘커뮤니티’가 포함되어 있어 정답과 완전히 일치함. 

============================================================
### query:
원자력 선량평가시스템의 성능 요구사항 중 질의응답 시간 기준은 어떻게 정의되어 있는가?

### answer:
일반 조회 요청은 3초 이내 처리되어야 하며, 10초 이상 소요되는 작업은 사용자에게 팝업 메시지를 제공해야 한다.

---

### retriever A:
질의·응답시간
요구사항
상세설명
정의
질의·응답시간
세부
내용
의는 사용자가 요청을 하는 시간으로부터 3000ms 내에 그 결과를 보여줘야 함
❍ 임의의 선택 기준이 허용되는 대량의 데이터에 대한 질의 및 고용량의 멀티미디어 파일을 가지고 있는 프로그램에서는 적절한 처리 상태를 표시하여야 함
산출정보
성능측정 결과 내역서
관련 요구사항
요구사항 고유번호
PER-002
요구사항 명
오류 응답시간
요구사항
상세설명
정의
오류 응답시간
세부
내용
시스템에 데이터 입력 등의 사용 중에 발생한 모든 오류는 원인이 발생한지 3000ms 이하로 사용자에게 오류 메시지를 보여줘야 함
산출정보
성능측정 결과 내역서
관련 요구사항
요구사항 고유번호
PER-003
요구사항 명
통계 처리 사전 경고
요구사항
상세설명
정의
통계 처리 사전 경고
세부
내용
리 시 5,000ms 이상 지연되는 경우 연산 작업을 시작하는 시점에 예상되는 결과에 대해 사용자에게 메시지로 알려야 함
산출정보
성능측정 결과 내역서
관련 요구사항
요구사항 고유번호
PER-004
요구사항 명
부하증가 대비 성능유지
요구사항
상세설명
정의
부하증가 대비 성능유지
세부
내용
시스템의 경우 동시 접속자(사용자)는 500명 이상 접속가능 하여야 하며, 이에 따른 처리속도 등 성능이 저하되지 않아야 함
❍ 포털 시스템의 경우 동시 접속자(사용자)는 3000명 이상 접속가능 하여야 하며, 이에 따른 처리속도 등 성능이 저하되지 않아야 함
산출정보
성능측정 결과 내역서
관련 요구사항
요구사항 고유번호
PER-005
요구사항 명
오프라인 배치성 업무 응답시간
요구사항
나. 성능 요구사항 (Performance Requirement)
요구사항분류
PER-001
요구사항명
응답시간
요구사항 내용
정의
개선 대상 웹사이트 및 콘텐츠 관리 시스템(CMS) 정보 조회에 대한 응답시간 성능 보장
세부 내용
○ 정보 처리량 및 확장성 등을 고려하여 웹페이지 디스플레이 시간 기능요청에 대한 응답시간이 최소화 되도록 함
- 단순 웹페이지의 경우, 사용자가 요청을 하는 시간으로부터 3초 내에 그 결과를 보여줘야 함
- 시스템이 만드는 각 페이지의 경우, 요청한 시간으로부터 5초 내에 디스플레이 되어야 함
- 이 요구사항은 시스템을 사용하는 동시 사용자가 최대 사용자 수의 90%를 초과하는 질의에는 적용되지 않음
○ 과도한 데이터베이스 접속을 자제하며 응답속도 개선(상영시간표 등)
- 이 요구사항은 임의의 선택 기준이 허용되는 대량의 데이터(데이터량 2GB 이상)에 대한 질의에는 적용되지 않음
- 이 요구사항은 임의의 선택 기준이 허용되는 대량의 데이터에 대한 요청 및 플래시가 있을 경우에는 적용되지 않으며, 한 개 이상의 큰 이미지(이미지 500KB 이상) 및 동영상을 가지고 있는 페이지에는 적용되지 않음
- 동영상의 경우 트래픽 관리를 위해 외부 플랫폼(예: 유튜브, 기타 CDN 서비스)과 연계하여 기능 구현
○ 기능 구현에 있어 10초 이상 걸릴 수 있는 작업에 대해서는 시간지연에 대한 정보를 사용자에게 팝업 등의 방법으로 사용자에게 알려야 함
요구사항분류
PER-002
요구사항명
데이터 형식 오류 응답 시간
요구사항 내용
정의
사용자 입력 오류 응답 시간 제시
세부 내용


---

### retriever B:
성능 요구사항
상세
설명
정의
정보시스템 업무 응답시간 목표
세부
내용
□ 요청(질의) 응답시간
o 모든 질의는 사용자가 요청을 하는 시간으로부터 3초 내에 최초의 결과 값을 보여줘야 함
□ 이 요구사항은 임의의 선택 기준이 허용되는 대량의 데이터(2GB 이상)에 대한 질의에는 적용되지 않음
산출정보
o SW개발에 따른 산출물
성능 요구사항
상세
설명
정의
정보시스템 업무 응답시간 목표
세부
내용
□ 요청(질의) 응답시간
o 모든 질의는 사용자가 요청을 하는 시간으로부터 3초 내에 최초의 결과 값을 보여줘야 함
□ 이 요구사항은 임의의 선택 기준이 허용되는 대량의 데이터(2GB 이상)에 대한 질의에는 적용되지 않음
산출정보
o SW개발에 따른 산출물
성능 요구사항
응낙수준
필수
요구
사항
상세
설명
정의
사용자 요구에 대한 웹페이지 디스플레이 반응 시간
세부
내용
○ 시스템은 정상 상태에서 사용자의 질의 요청에 대한 결과 페이지를 화면에 출력할 때 3초 이내 보장
○ 평균 응답시간은 시스템을 사용하는 동시 사용자가 최대 사용자 수의 90%를 초과하는 경우에는 미적용
산출정보
관련요구사항
요구사항 내용
○ 모든 질의는 사용자가 요청하는 시간으로부터 3초 이내에 그 결과값을 보여줘야 함
○ 평균 응답 시간은 시스템을 사용하는 동시 사용자가 최대 사용자 수의 90%를 초과하는 경우 미적용함
○ 한계 인원 이상 접속 시 해결 방안 제시해야 함
② 성능 요구사항
PER-001
평균응답시간 및 성능 최적화
요구사항
상세설명
정의
업무응답시간에 목표 대한 기준
세부내용
○ 모든 질의는 평균 응답시간으로 사용자가 요청한 시간으로부터 3초 내에
그 결과를 보여줘야 함
○ 대량의 데이터 로딩(통계, 검색, 이미지 등), 사용자 숫자가 동시 사용자 용량의 90%를 초과하는 경우에는 적용되지 않음
산출 정보
개발 산출물, 테스트 결과서
관련 요구사항
성능 요구사항
상세
설명
정의
평균 응답시간
세부
내용
❍ 대상시스템은 정상 상태에서 사용자의 질의 요청에 대한 결과
페이지를 화면에 출력할 때 3초 이내에 완전히 출력되어야 함


---

### openai answer
retriever A 점수: 3점
- 일반 질의 3초 이내 처리(“3000ms 내에 결과”)와 10초 이상 소요 작업 시 팝업 안내(“10초 이상 걸릴 수 있는 작업… 팝업… 알려야 함”)가 모두 명시되어 있어 정답 내용을 완전히 포함함.

retriever B 점수: 2점
- 일반 질의 3초 이내 처리 기준은 포함되어 있으나, 10초 이상 소요 시 팝업 안내 기준은 없음. 정답의 일부만 충족. 

============================================================
### query:
원자력 선량평가시스템의 포트란 모듈 개선 요구사항에서 추가로 고려해야 하는 평가 요소는 무엇인가?

### answer:
 ICRP60 기준에 따른 23개 장기를 고려한 평가장기 추가가 요구된다.

---

### retriever A:
가 방법
평가부문
평가항목
평가기준
배점
한도
정량
평가
(10)
경영상태
경영상태(신용상태등급) 평가
10
전략 및 방법론
(22)
사업
이해도
사업의 특성 및 목표에 부합하는 사업전략을 제시하고 있는지를 평가한다.
7
추진전략
개발업무 수행 시 일정 및 위험요소를 고려하여 얼마나 타당한 추진 전략을 수립하였는지 평가한다.
5
적용기술
사업에서 적용하고자 하는 기술이 향후 확장성을 고려하여 현실적으로 실현 가능하게 제시되어 있는지를 평가한다.
4
표준프레임 워크 적용
제안사는 사업에 네팔 정보통신기술부 프레임워크 기반의 공통 컴포넌트의 유형 및 사용계획을 기술하고 사용성 향상을 위한 추가 컴포넌트에 대해 적용방안을 제시하여야한다.
3
개발
방법론
사업에 적정한 개발 방법론이 제시되었는지 평가하고, 실제 적용 사례 및 경험을 바탕으로 단계별 산출물이 제시되었는지 여부를 평가한다.
3
기술 및 기능
(26)
시스템 장비 구성 요구사항
요구 규격을 충족하는 장비 제안 여부와 현 시스템과의 인터페이스 및 확장 가능성 여부를 평가한다. 또한 도입 장비의 설치 및 공급 계획, 유지관리에 대한 방안이 구체적으로 기술되어 있는가를 평가한다.
7
기능 요구사항
기능 요구사항·기대사항·제약사항 등을 파악한 후 구현 방안이 구체적으로 기술되어 있는지를 평가한다. 또한 제안한 방안 및 기술이 적용 가능한지를 평가한다.
7
데이터
요구사항
데이터 요구사항 분석을 바탕으로 데이터 전환계획 및 검증 방법, 데이터 오류 발생 시 처리 방안이 구체적으로 제시되어 있는지를 평가한다.
4
보안
요구사항
기능 보안
체적이고 적절한지 평가
(30점) 요구사항
- 구현방안이 설계단계부터 구현 및 검증단계까지 고려하고 있는지 평가
- 데이터 전환 계획 및 검증 방법, 에러 데이터 처리 방법에 대해 구체적
데이터
인 내용을 제시하고 있는지 평가 15
요구사항
- 데이터 전환을 위해 책임 조직이 투입되는지 평가
- 기능 및 품질 등 요구사항을 구현 시 관련 제약사항을 충족시키며 구현
제약사항
방안 및 테스트 방안을 구체적으로 기술하였는지 평가
-237-
평가
평가항목 평가기준 배점
부문
- 구현하고자 하는 기능에 요구 성능이 충족되도록 방법론 및 분석 도구
성능 를 통하여 내용을 구체적으로 분석하였는지, 구현 및 테스트 방안을 구
요구사항 체적으로 기술하였는지 평가
- 제안한 방안 및 기술을 통해 요구 성능을 충족시킬 수 있는지 평가
- 분석 도구 및 구현 방안, 테스트 방안을 구체적으로 기술하였는지, 분석
·설계 등 각 단계별 품질 요구사항의 점검 및 검토 방안을 구체적으로
품질
계획하였는지 평가
요구사항
- 각 단계마다 품질 요구사항을 점검하는 별도의 전문 인력이 투입되는지
성능 및 평가
품질 - (시스템 인터페이스) 타 시스템과의 연계 방안들에 대한 장·단점의 분석 10
(10점) 을 통해 가장 적합한 방안을 구체적으로 기술했는지, 구현 경험이 있는
인터페이
인터페이스 담당자가 투입되는지 평가
스
- (사용자 인터페이스) 사용자 편의성을 고려하여 요구사항을 제공하기 위
요구사항
한 분석 및 설계, 구현 방안과 검토 계획을 구체적으로 기술하였는지,
구현 경험이 있는 사용자 인터페이스 담당자가 투입되는지 평가


---

### retriever B:
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항


---

### openai answer
retriever A 점수: 0점
질문·정답의 핵심 키워드(원자력/선량평가/포트란/ICRP60/23개 장기)가 전혀 없고, 일반 사업 평가·요구사항만 다룸.

retriever B 점수: 0점
“기능 요구사항” 반복 외에 질의·정답과 관련된 핵심 키워드나 내용이 없음. 

## mmr_results

============================================================
### query:
아이돌봄 인적성 검사 시스템 운영 사업의 주요 목적은 무엇인가?

### answer:
인적성 검사 수행을 지원하고, 돌봄활동과 연계한 사후관리 및 인력 검증을 강화하는 것이다.

---

### retriever A:
◦원할한 인적성 검사 수행을 위한 정보시스템 지원 필요
◦돌봄활동과 연계한 사후관리 기반을 강화하여 이용자 만족도 향상
◦아이돌봄인력 인·적성에 기초한 돌봄인력 관리 강화 필요
◦정보시스템 개인정보보호법 준수를 위한 보호조치
2
사업 개요
□ 사 업 명: 아이돌봄인력 인적성 검사 정보시스템 운영
- 2024. 정보시스템 구축에 따른 유지보수 사업
□ 사업기간: 계약일 ~ 2025. 12. 31.
□ 소요예산: 172,700,000원(부가세 포함)
□ 추진방식: 제한경쟁입찰(협상에 의한 계약)
3
사업 범위
□ 인·적성 프로그램 기능수정 및 안정화 기술지원
◦인·적성 시스템의 안정적인 정보화 지원을 위한 상시 유지관리
체계 지원
◦인·적성 검사 변경에 따른 프로그램 수정 및 결과 반영
◦웹 접근성, 기술적 보안 등 전자정부 가이드라인 준수
□ 개인정보보호 및 정보보안 강화
◦개인정보보호 및 정보보호 강화 조치(방안)
◦정보시스템 운영을 위한 물리적, 기술적 보안 대책 제시
□ 안정적인 클라우드 환경 운영 및 제공
◦클라우드 무중단 환경 제공
◦클라우드 환경 운영을 위한 기술적 유지관리 지원
◦시스템 용을 위한 SW 및 외부 서비스 연계
4
기대 효과
□ (안전한 돌봄 환경 조성) 아이돌봄인력 선별 및 검증 강화 통한 안심하고 맡길 수 있는 돌봄서비스 제공
□ (효율성 제고) 아이돌봄인력 대상 인적성 검사를 편리하게 실시하여 돌봄인력 검증 절차 효율성 제고
□ (정책지원 및 제도개선 지원) 체계적인 검사 결과를 축적하고 정책 수립지원을 위한 기초자료를 제공, 지속 가능한 아이돌봄서비스 제도개선에 활용
□ (안전한 돌봄 환경 조성) 아이돌봄인력 선별 및 검증 강화 통한 안심하고 맡길 수 있는 돌봄서비스 제공
□ (효율성 제고) 아이돌봄인력 대상 인적성 검사를 편리하게 실시하여 돌봄인력 검증 절차 효율성 제고
□ (정책지원 및 제도개선 지원) 체계적인 검사 결과를 축적하고 정책 수립지원을 위한 기초자료를 제공, 지속 가능한 아이돌봄서비스 제도개선에 활용
□ (돌봄인력 관리) 데이터를 기반으로 하는 아이돌봄인력에 대한 효율적 관리·감독 가능
Ⅱ. 사업 현황
1
업무현황
□ 인·적성검사 실시 정보시스템 지원
ㅇ 인·적성검사 및 실시를 위한 정보 시스템 제공
ㅇ 정보시스템 안정화 및 운영지원
□ 인·적성검사 운영
ㅇ 인적성 검사 실시 및 수행 방법 안내
ㅇ 인적성 검사 수행 모니터링 및 결과 분석
ㅇ 검사 실시 운영자 권한 부여 및 인증 관리
2
정보화 현황
□ 시스템 구성도
□ 정보시스템 구성도 요청 시 별도 공개
※ 행정기관 및 공공기관 정보시스템 구축 및 운영 지침(행정안전부 고시 제 2024-66호, 2024.8.28.) 제 17조(제안요청서 보안사항 등)에 따라, 시스템 구성도 및 상세자료 등은 공개하지 않으며, 입찰에 참가하고자 하는 사업자의 요청이 있는 경우 보안서약서를 제출 후 담당자 입회하에 제한적으로 열람 가능
□ 시스템(장비)·소프트웨어 운영현황
AS-IS
TO-BE (2025년)
ㅇ인·적성 검사 관리기능 및 사용자화면 개발
※ 클라우드 환경 기반 개발
ㅇ인·적성 검사 기능 최적화 및 운영 기술지원
-인·적성 검사 고도화에 따른 수정 반영
(검사도구 4종)
※사업 운영에 따라 변경될수 있음


---

### retriever B:
기대 효과
□ (안전한 돌봄 환경 조성) 아이돌봄인력 선별 및 검증 강화 통한 안심하고 맡길 수 있는 돌봄서비스 제공
□ (효율성 제고) 아이돌봄인력 대상 인적성 검사를 편리하게 실시하여 돌봄인력 검증 절차 효율성 제고
□ (정책지원 및 제도개선 지원) 체계적인 검사 결과를 축적하고 정책 수립지원을 위한 기초자료를 제공, 지속 가능한 아이돌봄서비스 제도개선에 활용
□ (돌봄인력 관리) 데이터를 기반으로 하는 아이돌봄인력에 대한 효율적 관리·감독 가능
Ⅱ. 사업 현황
1
업무현황
□ 인·적성검사 실시 정보시스템 지원
ㅇ 인·적성검사 및 실시를 위한 정보 시스템 제공
ㅇ 정보시스템 안정화 및 운영지원
□ 인·적성검사 운영
ㅇ 인적성 검사 실시 및 수행 방법 안내
ㅇ 인적성 검사 수행 모니터링 및 결과 분석
ㅇ 검사 실시 운영자 권한 부여 및 인증 관리
2
정보화 현황
□ 시스템 구성도
□ 정보시스템 구성도 요청 시 별도 공개
※ 행정기관 및 공공기관 정보시스템 구축 및 운영 지침(행정안전부 고시 제 2024-66호, 2024.8.28.) 제 17조(제안요청서 보안사항 등)에 따라, 시스템 구성도 및 상세자료 등은 공개하지 않으며, 입찰에 참가하고자 하는 사업자의 요청이 있는 경우 보안서약서를 제출 후 담당자 입회하에 제한적으로 열람 가능
□ 시스템(장비)·소프트웨어 운영현황
AS-IS
TO-BE (2025년)
ㅇ인·적성 검사 관리기능 및 사용자화면 개발
※ 클라우드 환경 기반 개발
ㅇ인·적성 검사 기능 최적화 및 운영 기술지원
-인·적성 검사 고도화에 따른 수정 반영
(검사도구 4종)
□ 시스템(장비)·소프트웨어 운영현황
AS-IS
TO-BE (2025년)
ㅇ인·적성 검사 관리기능 및 사용자화면 개발
※ 클라우드 환경 기반 개발
ㅇ인·적성 검사 기능 최적화 및 운영 기술지원
-인·적성 검사 고도화에 따른 수정 반영
(검사도구 4종)
※사업 운영에 따라 변경될수 있음
ㅇ무중단 클라우드 기반 제공
ㅇ클라우드 안정화 운영기반 및 기술지원
SW 도입 현황
SW 명칭
용도
비고
리포팅 툴
OZ Report
인·적성 결과지 출력
발주처 제공
□ 시스템 장비 현황 요청 시 별도 공개
※ 행정기관 및 공공기관 정보시스템 구축 및 운영 지침(행정안전부 고시 제 2024-66호, 2024.8.28.) 제 17조(제안요청서 보안사항 등)에 따라, 시스템 구성도 및 상세자료 등은 공개하지 않으며, 입찰에 참가하고자 하는 사업자의 요청이 있는 경우 보안서약서를 제출 후 담당자 입회하에 제한적으로 열람 가능
Ⅲ. 사업 추진방안
1
추진 목표
□ `24년 개발된 인·적성 검사 운영을 지원하고, 데이터 수집 및 분석을 위한 정보체계 관리
□ 인·적성 검사 정보시스템 안정적 기반 지원 및 보안 강화
2
추진체계
□ 추진체계도
발주기관
<한국건강가정진흥원>
협조부서
<경영지원부/
소통협력실>
주관부서
<돌봄지원부>
관련부처
<여성가족부>
주관사업자
<사업수행업체>
□ 추진 주체별 역할
구 분
주요 업무 내용
주관부서
ㅇ 사업총괄 관리 및 시행
ㅇ 사업계획서 수립 및 제안요청서 작성
ㅇ 개발내역 업무정의 및 과업내용 제시
ㅇ 개발관련 자료제공 및 시스템 인수
ㅇ 사업수행 관리 및 검사, 결과 평가
협조부서


---

### openai answer
retriever A 점수: 3점
- “원활한 인적성 검사 수행 지원”, “돌봄활동과 연계한 사후관리 기반 강화”, “아이돌봄인력 선별 및 검증 강화”가 모두 포함되어 있어 정답의 요소를 완전히 충족함.

retriever B 점수: 2점
- “인·적성검사 실시 지원”과 “선별 및 검증 강화”는 있으나, “돌봄활동과 연계한 사후관리”에 대한 명시가 없어 일부만 포함됨. 

============================================================
### query:
아이돌봄 시스템에서 개인정보 보호를 위해 요구되는 조치는 무엇인가?

### answer:
접속기록 관리, 내부관리계획 수립 등 기술적·관리적 보호조치를 수행해야 한다.

---

### retriever A:
- 개인정보의 기술절·관리적 보호조치 기준(개인정보위)
- 표준 개인정보 보호 지침(개인정보위)
가명정보의 결합 및 반출 등에 관한 고시(개인정보위)
개인정보 영향평가에 관한 고시(개인정보위)
※ 고시에 따라 아래 기준 이상의 개인정보 처리 시 발주처에서 개인정보 영향평가를 발주하며 영향평가결과를 반영하여 시스템을 구축해야 함
내 용
1
구축‧운용 또는 변경하려는 개인정보파일로서 5만명 이상의 정보주체에 관한 민감정보 또는 고유식별정보의 처리가 수반되는 개인정보파일
2
구축‧운용하고 있는 개인정보파일을 해당 기관 내부 또는 외부에서 구축‧운용하고 있는 다른 개인정보파일과 연계하려는 경우로서 연계결과 50만명 이상의 정보주체에 관한 개인정보가 포함되는 개인정보파일
3
구축‧운용 또는 변경하려는 개인정보파일로서 100만명 이상의 정보주체에 관한 개인정보파일
4
개인정보 영향평가를 받은 후에 개인정보 검색체계 등 개인정보파일의 운용체계를 변경하려는 경우 그 개인정보파일
○ 개인정보 안전성 확보조치 기준 이행·반영
- 개인정보 열람 시 접속기록(최소 2년 이상) 보관·관리
- 개인정보 접근권한 부여·변경·말소 이력(최소 3년 이상) 보관·관리
- 고유식별번호 및 비밀번호 생체인식정보는 안전한 암호 알고리즘을 사용 하여 저장
○ 데이터의 보안 강화를 위해 SSL서비스 도입
○ 개인정보보호를 위한 DB 암호화하여 자료 유출방지
- 인증서 기반(GPKI) 로그인 방식 적용 후 ID/PW 로그인
- 정보보안 강화를 위한 시스템에 각종 패치, 환경설정 지원 등
◦ 보안 요구사항
요구사항 고유번호
SER-001
요구사항 명칭
개인정보 처리
요구사항 분류
보안 요구사항
요구사항
상세설명
정의
개인정보 처리 및 암호화
세부
내용
ㅇ 개인정보를 보유할 경우에는 관련 개인정보를 암호화하여야 함
ㅇ 서비스의 보안성 강화를 위하여 SSL 보안 통신을 실시하여야 함
ㅇ 사업의 암호화 대상을 식별하고 암호화 도구를 활용하여 암복호화 방안을 마련해야 함
- 고유식별정보 : 주민번호, 여권번호, 운전면허정보 등
ㅇ개인을 식별할 수 있는 정보(사용자 인증 정보, 패스워드 등)를 운영DB 또는 개발DB에 저장할 경우, 암호화하여 저장하며, 소스코드에 직접 하드코딩하지 않음(패스워드는 단방향 암호화 처리, SHA-256 이상 암호화 알고리즘 사용)
ㅇ 개인정보 취급자가 개인정보처리 시스템에 접속하는 경우 접속 로그를 시스템에 최소 1년 이상 기록 관리해야 함(5만명 이상의 정보주체에 관하여 개인정보를 처리하거나 고유식별정보 또는 민감정보를 처리하는 개인정보처리시스템의 경우에는 2년 이상 보관)
◈ 필수 관리 항목
• 개인정보취급자 식별정보 : ID ୴Ā • 날짜 및 시간 : 접속일시
• 접속자 IP 주소 : 접속자 정보
• 처리한 정보주체 정보 : 정보주체 식별정보(계정, 회원번호 등)
• 수행업무 : 열람, 수정, 삭제, 인쇄, 입력, 다운로드 등
ㅇ 개인정보 취급자가 개인정보를 처리할 경우 해당 접근 로그를 시스템에 기록할 것
ㅇ 개인정보처리방침에 따라 개인정보를 처리/파기해야 하며 백업데이터 생성 금지


---

### retriever B:
6. 그 밖에 개인정보의 보호를 위하여 필요한 사항
-295-
1. 개인정보의 안전성 확보조치 철저(「개인정보보호법」제29조)
❍ 개인정보처리자는 개인정보가 분실·도난·유출·변조 또는 훼손되지 않도록 다음 각 호의 안전성 확보조치를 하여야한다.
구 분
주 요 내 용
내부관리계획(제3조)
◦내부관리계획 포함사항 명시 ※개인정보 보호책임자 지정, 개인정보취급자 교육 등
◦개인정보 처리 업무 위탁 시 관리·감독 사항
접근권한 관리(제4조)
◦개인정보처리시스템의 접근권한 부여․변경․말소 기록 최소 3년간 보관
◦안전한 비밀번호 작성규칙 수립․적용
접근통제(제5조)
◦접근통제 시스템 설치․운영 의무화 ※접근통제 시스템 : 침입차단(Firewall), 침입탐지(IDS) 기능 포함
◦주민번호 처리 시 추가 정보 확인, 모바일 기기 보호조치 조항, 홈페이지에서 고유식별정보를 처리할 경우 년 1회 취약점 점검 조치
개인정보 암호화
(제6조)
◦암호화 대상 : 고유식별정보(주민등록번호, 여권번호, 운전면허번호, 외국인등록번호), 비밀번호, 바이오정보
◦암호화 기준
구 분
암 호 화 기 준
송․수신시
모든 정보 암호화 전송
저장시
비밀번호,
바이오정보
비밀번호는 단방향 암호화,
바이오정보는 양방향 암호화
고유식별정보
ⓛ 인터넷 구간,공통(DMZ) 구간 저장시 : 의무적 암호화
보안 요구사항
상세
설명
정의
개인정보의 안전성 확보를 위한 기술적 조치사항
세부
내용
 개인정보를 처리함에 있어 『개인정보 보호법』, 『개인정보의 안전성 확보조치 기준』 등 관련 법규 및 규정을 준수하여야 함
 접근 권한 관리
- 개인정보처리시스템에 접속할 수 있는 사용자계정을 발급하는 경우, 개인정보취급자 별로 사용자계정을 발급하여야 하며, 취급자의 역할에 따라 차등부여할 수 있어야 함
- 개인정보 처리내역에 대한 책임 추적성을 확보하기 위하여 접근권한 부여, 변경, 말소에 대한 내역을 이력 관리 또는 조회가 가능하여야 하며, 해당 기록을 최소 3년간 보관할 수 있도록 구현해야 함
예) 신청자 정보, 신청일시, 승인자 및 발급자 정보, 신청 및 발급 사유 등
- 개인정보처리시스템에 접근하여 개인정보를 생성수정삭제열람 등을 하고자 하는 경우 다음 사항을 포함하는 정보이용 내역을 기록하고 남길 수 있도록 시스템을 구현하여야 함.


---

### openai answer
retriever A 점수: 2점
접속기록 관리, 접근권한 이력 보관, 암호화 등 기술·관리적 보호조치는 명시되어 있으나 ‘내부관리계획 수립’이 명확히 언급되지 않아 답의 일부만 포함합니다.

retriever B 점수: 3점
‘내부관리계획(제3조)’이 명시되어 있고, 접근권한 기록·정보이용 내역 기록 등 접속기록 관리와 기술·관리적 보호조치 전반이 포함되어 답을 모두 충족합니다. 

============================================================
### query:
2025 구미 아시아육상경기선수권대회의 개최 기간은 언제인가?

### answer:
2025년 5월 27일부터 5월 31일까지 총 5일간 개최된다.

---

### retriever A:
종합정보시스템 및 홈페이지 등 구축 용역 제안요청서
1
대회개요
아시아의 꿈, 구미에서 세계로
 The dream of ASIA, From GUMI to the World 
비전
ㅇ (대 회 명) 2025 구미아시아육상경기선수권대회
ㅇ (기 간) 2025. 5. 27.(화) ~ 5. 31.(토) 5일간
ㅇ (장 소) 구미시민운동장 일원
ㅇ (규 모) 아시아 45개국 1,200여명 / 선수단 및 임원
ㅇ (주 최) 아시아육상연맹(Asian Athletics, AA)
ㅇ (주 관) 2025 구미아시아육상경기선수권대회 조직위원회
ㅇ (경기종목) 45개 종목
구 분
종 목 명
남자
(22)
100m, 200m, 400m, 800m, 1,500m, 5,000m,
10,000m, 110mH, 400mH, 3,000m장애물달리기,
20km경보, 4×100mR, 4×400mR, 높이뛰기,
장대높이뛰기, 멀리뛰기, 세단뛰기, 포환던지기,
원반던지기, 해머던지기, 창던지기, 10종 경기
여자
(22)
100m, 200m, 400m, 800m, 1,500m, 5,000m,
10,000m, 100mH, 400mH, 3,000m장애물달리기,
20km경보, 4×100mR, 4×400mR, 높이뛰기,
장대높이뛰기, 멀리뛰기, 세단뛰기, 포환던지기,
원반던지기, 해머던지기, 창던지기, 7종 경기
남·여
(1)
4×400m 믹스릴레이
요
2
과업개요
ㅇ (사 업 명) 2025 구미아시아육상경기선수권대회 종합정보시스템 및 홈페이지 등 구축 용역
ㅇ (사업기간) 계약일 ~ 2025. 6. 6.(금)
ㅇ (사업위치) 시민운동장 일원
2024. . .
대표자 : (인)
대리인 : (인)
2025 구미아시아육상경기선수권대회 조직위원장 귀하
※ 첨부서류 : 재직증명서, 신분증
※ 위임장에 사용하는 도장은 대표자 인감과 같아야 함.
※ 대리 접수로 인하여 발생하는 모든 문제는 대표자 본인이 책임짐
【 서식 3 】
재 직 증 명 서
성 명
생 년 월 일
부서/직위
소 속
주 소
재직기간
위와 같이 당사에 재직 중임을 증명합니다.
2024. . .
ྠĀ ྠĀ ྠĀ ྠĀ ྠĀ ྠĀ 업 체 명 :
ྠĀ ྠĀ ྠĀ ྠĀ ྠĀ ྠĀ 주 소 :
ྠĀ ྠĀ ྠĀ ྠĀ ྠĀ ྠĀ 대 표 자 : (인)
2025 구미아시아육상경기선수권대회 조직위원장 귀하
【 서식 4 】
사 용 인 감 계
사용인감
는(은) 2025 구미아시아육상경기선수권대회 조직위원회와 거래함에 있어 입찰, 계약 및 이에 수반되는 모든 행위에 대하여 위의 인감을 사용하고자 하며, 동 인감을 사용함으로써 민․형사상의 문제가 발생할 경우 이(가) 책임을 질 것을 확인합니다.
붙임 : 인감(법인)증명서 1부. 끝.
2024년 월 일
ྠĀ ྠĀ ྠĀ ྠĀ ྠĀ ྠĀ 주 소 :
ྠĀ ྠĀ ྠĀ ྠĀ ྠĀ ྠĀ 상 호 :
ྠĀ ྠĀ ྠĀ ྠĀ ྠĀ ྠĀ 대표자 : (인)
2025 구미아시아육상경기선수권대회 조직위원장 귀하
【 서식 5 】
서 약 서
당사는 2025 구미아시아육상경기선수권대회 종합정보시스템 및 홈페이지 등 구축 용역 참가신청과 관련하여 모든 서류와 증빙자료를 신의와 성실의 원칙에 입각하여 작성 제출하며, 아래 내용을 준수할 것을 확약하고 이에 서약서를 제출합니다.


---

### retriever B:
종합정보시스템 및 홈페이지 등 구축 용역 제안요청서
1
대회개요
아시아의 꿈, 구미에서 세계로
 The dream of ASIA, From GUMI to the World 
비전
ㅇ (대 회 명) 2025 구미아시아육상경기선수권대회
ㅇ (기 간) 2025. 5. 27.(화) ~ 5. 31.(토) 5일간
ㅇ (장 소) 구미시민운동장 일원
ㅇ (규 모) 아시아 45개국 1,200여명 / 선수단 및 임원
ㅇ (주 최) 아시아육상연맹(Asian Athletics, AA)
ㅇ (주 관) 2025 구미아시아육상경기선수권대회 조직위원회
ㅇ (경기종목) 45개 종목
구 분
종 목 명
남자
(22)
100m, 200m, 400m, 800m, 1,500m, 5,000m,
10,000m, 110mH, 400mH, 3,000m장애물달리기,
20km경보, 4×100mR, 4×400mR, 높이뛰기,
장대높이뛰기, 멀리뛰기, 세단뛰기, 포환던지기,
원반던지기, 해머던지기, 창던지기, 10종 경기
여자
(22)
100m, 200m, 400m, 800m, 1,500m, 5,000m,
10,000m, 100mH, 400mH, 3,000m장애물달리기,
20km경보, 4×100mR, 4×400mR, 높이뛰기,
장대높이뛰기, 멀리뛰기, 세단뛰기, 포환던지기,
원반던지기, 해머던지기, 창던지기, 7종 경기
남·여
(1)
4×400m 믹스릴레이
요
2
과업개요
ㅇ (사 업 명) 2025 구미아시아육상경기선수권대회 종합정보시스템 및 홈페이지 등 구축 용역
ㅇ (사업기간) 계약일 ~ 2025. 6. 6.(금)
ㅇ (사업위치) 시민운동장 일원
1. 기본정보
사업명
2025 구미아시아육상경기선수권대회 종합정보시스템 및 홈페이지 등 구축 용역
영향평가단계
□ 예산편성
□ 그 외 필요시
■ 사업발주
□ 재평가
주요 내용
대회 기본 자료 관리 및 대회 계측기 연동 등
사업기간
(또는 개발기간)
계약일 ~ 2025년 6월
구분
붙임 : 산출내역서 각 1부
2024. . .
제안업체 인(사용인감)
2025 구미아시아육상경기선수권대회 조직위원장 귀하
※ 제안금액은 반드시 한글로 띄어쓰기 없이 붙여 쓰고 아라비아 숫자는 괄호 안에 기재할 것.
【 서식 9-1 】
산 출 내 역 서
구 분
항 목
금 액
구 성 비
비 고
순
용
역
원
가
소 계
소 계
소 계
소 계
일반관리비 ( )%
이 윤 ( )%
총 원 가
부 가 가 치 세
총 용 역 비
※ 작성내용에 따라 줄 수 증감가능
【 서식 10 】
접수번호
2025 구미아시아육상경기선수권대회 종합정보시스템 및 홈페이지 등 구축 용역
정량적 평가 제안서
2024년 월 일
업 체 명
※ 주의사항


---

### openai answer
retriever A 점수: 3점
(‘2025. 5. 27.(화) ~ 5. 31.(토) 5일간’이 명시되어 있어 정답이 모두 포함됨)

retriever B 점수: 3점
(동일하게 개최 기간 ‘2025. 5. 27. ~ 5. 31. 5일간’이 명시되어 있어 정답이 모두 포함됨) 

============================================================
### query:
구미 아시아육상경기선수권대회 홈페이지에서 제공되는 주요 콘텐츠 항목에는 무엇이 포함되는가?

### answer:
대회소개, 경기일정 및 결과, 메달순위, 참가선수 정보, 커뮤니티 등이 포함된다.

---

### retriever A:
-경기참가자(Start List), 경기결과(Official Result, Summary) PDF 파일 제공
<다음 정보를 홈페이지에 전송>
Level1
Level2
기능 설명
경기
일정
전체 경기 일정
대회기간 전체 일정 제공
일자별 경기 일정
일자별 세부 경기 일정 제공
세부 종목별 경기 일정
세부 종목별 경기 일정 제공
선수별 경기 일정
선수별 경기 일정 제공
메달
국가/선수별 메달 현황
국가/선수별 메달 현황 제공
종목별 메달 현황
종목별 메달현황 및 메달수상자 정보 제공
경기
정보
스타트 리스트
세부종목별 편성 조 및 출전선수 목록 제공
대진 정보
세부 종목별 대진정보 조회 및 출력
(경기진행 상태에 따른 결과 반영 포함)
경기 결과
각 조별, 종목별 경기 결과 조회 및 출력
선수
정보
국가별 선수 정보
국가별 참가선수 목록 및 정보 조회
세부 종목별 선수 정보
세부 종목별 참가선수 목록 및 정보 조회
순위 정보
세부종목별 참가선수 순위 정보 목록 조회
과업내용
명칭
대분류
대회 종합정보시스템 S/W 개발
소분류
보고서관리시스템 개발
과업내용
데이터 출력 기능
<PDF 및 Excel, 증서 페이지 등 출력>
-참가선수 신상정보
-국가별/세부 종목별 참가선수 리스트
-심판/임원 신상정보
-국가별/세부 종목별 심판/임원 리스트
-세부 종목별, 예결선 Start List 및 조 편성
-세부 종목별 경기 결과 (Live Result, Official Result, Summary)
-세부 종목별 메달 현황 및 신기록 현황
-참가증서
ː참가선수(종목별 1위~6위 증서 및 참가증서 필수)
 운영 매뉴얼 또는 지침서 제공 및 시스템 운영 관계자 교육
ㅇ (현장IT팀 상주 및 시스템 운영)
 사전 등록, 경기 진행 및 결과 출력 시스템 등 운영
 대회시작 4일전~대회 종료일까지 경기장 상주
 경기장/비경기장별 운영 인원 배치
ㅇ (홈페이지 구축)
 메뉴 구성도(안)
아시아육상경기선수권대회 메인
대회소개
소식지
경기일정/결과
메달순위
육상경기
커뮤니티
환영인사
공지사항
경기일정
국가별순위
경기종목
안내사항
대회개요
보도자료
행사일정
선수별순위
참가선수
아카이브
대회상징물
사진자료
시설안내
유실물센터
조직위안내
자주묻는질문
홍보영상
자원봉사자
찾아오시는 길
구미시 안내
메뉴 상세설명(안)
메뉴명
부메뉴
주요 내용
메인 화면
◦2025 구미 아시아육상경기선수권대회를 대표할 수 있는 디자인 (공식 상징물 활용 등)
◦활용빈도가 높은 메뉴 배치 및 팝업배너
대회 소개
환영인사
◦대회 개최에 대한 인사 말씀
대회개요
◦대회명칭, 대회기간, 대회장소, 참가규모, 참가국, 참가인원, 종목, 주최, 주관 등
대회상징물
◦대회마크, 마스코트, 포스터, 슬로건
조직위안내
◦조직위원회·아시아육상추진단 조직도
홍보영상
◦홍보영상, 유튜브 쇼츠 영상, 축하 메시지 등
찾아오시는 길
◦경기장 위치, 교통수단별 길 안내
소식지
공지사항
◦대회 관련 공지사항 등
보도자료
◦대회 관련기사 스크랩 자료
사진자료
◦개·폐회식 사진, 각 종목별 경기 이미지 게시
경기일정
/
결과
경기일정
◦경기장 및 개최 일자, 시간, 라운드, 진행상태
◦각 종목별 경기결과, 종합순위
행사일정
◦개·폐회식 정보 및 식순 안내


---

### retriever B:
 운영 매뉴얼 또는 지침서 제공 및 시스템 운영 관계자 교육
ㅇ (현장IT팀 상주 및 시스템 운영)
 사전 등록, 경기 진행 및 결과 출력 시스템 등 운영
 대회시작 4일전~대회 종료일까지 경기장 상주
 경기장/비경기장별 운영 인원 배치
ㅇ (홈페이지 구축)
 메뉴 구성도(안)
아시아육상경기선수권대회 메인
대회소개
소식지
경기일정/결과
메달순위
육상경기
커뮤니티
환영인사
공지사항
경기일정
국가별순위
경기종목
안내사항
대회개요
보도자료
행사일정
선수별순위
참가선수
아카이브
대회상징물
사진자료
시설안내
유실물센터
조직위안내
자주묻는질문
홍보영상
자원봉사자
찾아오시는 길
구미시 안내
메뉴 상세설명(안)
메뉴명
부메뉴
주요 내용
메인 화면
◦2025 구미 아시아육상경기선수권대회를 대표할 수 있는 디자인 (공식 상징물 활용 등)
◦활용빈도가 높은 메뉴 배치 및 팝업배너
대회 소개
환영인사
◦대회 개최에 대한 인사 말씀
대회개요
◦대회명칭, 대회기간, 대회장소, 참가규모, 참가국, 참가인원, 종목, 주최, 주관 등
대회상징물
◦대회마크, 마스코트, 포스터, 슬로건
조직위안내
◦조직위원회·아시아육상추진단 조직도
홍보영상
◦홍보영상, 유튜브 쇼츠 영상, 축하 메시지 등
찾아오시는 길
◦경기장 위치, 교통수단별 길 안내
소식지
공지사항
◦대회 관련 공지사항 등
보도자료
◦대회 관련기사 스크랩 자료
사진자료
◦개·폐회식 사진, 각 종목별 경기 이미지 게시
경기일정
/
결과
경기일정
◦경기장 및 개최 일자, 시간, 라운드, 진행상태
◦각 종목별 경기결과, 종합순위
행사일정
◦개·폐회식 정보 및 식순 안내
종합정보시스템 및 홈페이지 등 구축 용역 제안요청서
1
대회개요
아시아의 꿈, 구미에서 세계로
 The dream of ASIA, From GUMI to the World 
비전
ㅇ (대 회 명) 2025 구미아시아육상경기선수권대회
ㅇ (기 간) 2025. 5. 27.(화) ~ 5. 31.(토) 5일간
ㅇ (장 소) 구미시민운동장 일원
ㅇ (규 모) 아시아 45개국 1,200여명 / 선수단 및 임원
ㅇ (주 최) 아시아육상연맹(Asian Athletics, AA)
ㅇ (주 관) 2025 구미아시아육상경기선수권대회 조직위원회
ㅇ (경기종목) 45개 종목
구 분
종 목 명
남자
(22)
100m, 200m, 400m, 800m, 1,500m, 5,000m,
10,000m, 110mH, 400mH, 3,000m장애물달리기,
20km경보, 4×100mR, 4×400mR, 높이뛰기,
장대높이뛰기, 멀리뛰기, 세단뛰기, 포환던지기,
원반던지기, 해머던지기, 창던지기, 10종 경기
여자
(22)
100m, 200m, 400m, 800m, 1,500m, 5,000m,
10,000m, 100mH, 400mH, 3,000m장애물달리기,
20km경보, 4×100mR, 4×400mR, 높이뛰기,
장대높이뛰기, 멀리뛰기, 세단뛰기, 포환던지기,
원반던지기, 해머던지기, 창던지기, 7종 경기
남·여
(1)
4×400m 믹스릴레이
요
2
과업개요
ㅇ (사 업 명) 2025 구미아시아육상경기선수권대회 종합정보시스템 및 홈페이지 등 구축 용역
ㅇ (사업기간) 계약일 ~ 2025. 6. 6.(금)
ㅇ (사업위치) 시민운동장 일원


---

### openai answer
retriever A 점수: 3점
- 홈페이지 메뉴 구성에 대회소개, 경기일정/결과, 메달순위, 참가선수, 커뮤니티가 모두 명시되어 있어 정답의 모든 항목을 포함함.

retriever B 점수: 3점
- 홈페이지 메뉴 구성에 대회소개, 경기일정/결과, 메달순위, 참가선수, 커뮤니티가 모두 명시되어 있어 정답의 모든 항목을 포함함. 

============================================================
### query:
원자력 선량평가시스템의 성능 요구사항 중 질의응답 시간 기준은 어떻게 정의되어 있는가?

### answer:
일반 조회 요청은 3초 이내 처리되어야 하며, 10초 이상 소요되는 작업은 사용자에게 팝업 메시지를 제공해야 한다.

---

### retriever A:
질의·응답시간
요구사항
상세설명
정의
질의·응답시간
세부
내용
의는 사용자가 요청을 하는 시간으로부터 3000ms 내에 그 결과를 보여줘야 함
❍ 임의의 선택 기준이 허용되는 대량의 데이터에 대한 질의 및 고용량의 멀티미디어 파일을 가지고 있는 프로그램에서는 적절한 처리 상태를 표시하여야 함
산출정보
성능측정 결과 내역서
관련 요구사항
요구사항 고유번호
PER-002
요구사항 명
오류 응답시간
요구사항
상세설명
정의
오류 응답시간
세부
내용
시스템에 데이터 입력 등의 사용 중에 발생한 모든 오류는 원인이 발생한지 3000ms 이하로 사용자에게 오류 메시지를 보여줘야 함
산출정보
성능측정 결과 내역서
관련 요구사항
요구사항 고유번호
PER-003
요구사항 명
통계 처리 사전 경고
요구사항
상세설명
정의
통계 처리 사전 경고
세부
내용
리 시 5,000ms 이상 지연되는 경우 연산 작업을 시작하는 시점에 예상되는 결과에 대해 사용자에게 메시지로 알려야 함
산출정보
성능측정 결과 내역서
관련 요구사항
요구사항 고유번호
PER-004
요구사항 명
부하증가 대비 성능유지
요구사항
상세설명
정의
부하증가 대비 성능유지
세부
내용
시스템의 경우 동시 접속자(사용자)는 500명 이상 접속가능 하여야 하며, 이에 따른 처리속도 등 성능이 저하되지 않아야 함
❍ 포털 시스템의 경우 동시 접속자(사용자)는 3000명 이상 접속가능 하여야 하며, 이에 따른 처리속도 등 성능이 저하되지 않아야 함
산출정보
성능측정 결과 내역서
관련 요구사항
요구사항 고유번호
PER-005
요구사항 명
오프라인 배치성 업무 응답시간
요구사항
나. 성능 요구사항 (Performance Requirement)
요구사항분류
PER-001
요구사항명
응답시간
요구사항 내용
정의
개선 대상 웹사이트 및 콘텐츠 관리 시스템(CMS) 정보 조회에 대한 응답시간 성능 보장
세부 내용
○ 정보 처리량 및 확장성 등을 고려하여 웹페이지 디스플레이 시간 기능요청에 대한 응답시간이 최소화 되도록 함
- 단순 웹페이지의 경우, 사용자가 요청을 하는 시간으로부터 3초 내에 그 결과를 보여줘야 함
- 시스템이 만드는 각 페이지의 경우, 요청한 시간으로부터 5초 내에 디스플레이 되어야 함
- 이 요구사항은 시스템을 사용하는 동시 사용자가 최대 사용자 수의 90%를 초과하는 질의에는 적용되지 않음
○ 과도한 데이터베이스 접속을 자제하며 응답속도 개선(상영시간표 등)
- 이 요구사항은 임의의 선택 기준이 허용되는 대량의 데이터(데이터량 2GB 이상)에 대한 질의에는 적용되지 않음
- 이 요구사항은 임의의 선택 기준이 허용되는 대량의 데이터에 대한 요청 및 플래시가 있을 경우에는 적용되지 않으며, 한 개 이상의 큰 이미지(이미지 500KB 이상) 및 동영상을 가지고 있는 페이지에는 적용되지 않음
- 동영상의 경우 트래픽 관리를 위해 외부 플랫폼(예: 유튜브, 기타 CDN 서비스)과 연계하여 기능 구현
○ 기능 구현에 있어 10초 이상 걸릴 수 있는 작업에 대해서는 시간지연에 대한 정보를 사용자에게 팝업 등의 방법으로 사용자에게 알려야 함
요구사항분류
PER-002
요구사항명
데이터 형식 오류 응답 시간
요구사항 내용
정의
사용자 입력 오류 응답 시간 제시
세부 내용


---

### retriever B:
성능 요구사항
상세
설명
정의
정보시스템 업무 응답시간 목표
세부
내용
□ 요청(질의) 응답시간
o 모든 질의는 사용자가 요청을 하는 시간으로부터 3초 내에 최초의 결과 값을 보여줘야 함
□ 이 요구사항은 임의의 선택 기준이 허용되는 대량의 데이터(2GB 이상)에 대한 질의에는 적용되지 않음
산출정보
o SW개발에 따른 산출물
성능 요구사항
상세
설명
정의
정보시스템 업무 응답시간 목표
세부
내용
□ 요청(질의) 응답시간
o 모든 질의는 사용자가 요청을 하는 시간으로부터 3초 내에 최초의 결과 값을 보여줘야 함
□ 이 요구사항은 임의의 선택 기준이 허용되는 대량의 데이터(2GB 이상)에 대한 질의에는 적용되지 않음
산출정보
o SW개발에 따른 산출물
성능 요구사항
응낙수준
필수
요구
사항
상세
설명
정의
사용자 요구에 대한 웹페이지 디스플레이 반응 시간
세부
내용
○ 시스템은 정상 상태에서 사용자의 질의 요청에 대한 결과 페이지를 화면에 출력할 때 3초 이내 보장
○ 평균 응답시간은 시스템을 사용하는 동시 사용자가 최대 사용자 수의 90%를 초과하는 경우에는 미적용
산출정보
관련요구사항
요구사항 내용
○ 모든 질의는 사용자가 요청하는 시간으로부터 3초 이내에 그 결과값을 보여줘야 함
○ 평균 응답 시간은 시스템을 사용하는 동시 사용자가 최대 사용자 수의 90%를 초과하는 경우 미적용함
○ 한계 인원 이상 접속 시 해결 방안 제시해야 함
② 성능 요구사항
PER-001
평균응답시간 및 성능 최적화
요구사항
상세설명
정의
업무응답시간에 목표 대한 기준
세부내용
○ 모든 질의는 평균 응답시간으로 사용자가 요청한 시간으로부터 3초 내에
그 결과를 보여줘야 함
○ 대량의 데이터 로딩(통계, 검색, 이미지 등), 사용자 숫자가 동시 사용자 용량의 90%를 초과하는 경우에는 적용되지 않음
산출 정보
개발 산출물, 테스트 결과서
관련 요구사항
성능 요구사항
상세
설명
정의
평균 응답시간
세부
내용
❍ 대상시스템은 정상 상태에서 사용자의 질의 요청에 대한 결과
페이지를 화면에 출력할 때 3초 이내에 완전히 출력되어야 함


---

### openai answer
retriever A 점수: 3점
- 3초 이내 응답(“3000ms 내”)과 10초 이상 소요 시 팝업 안내 요구 모두 포함되어 있어 정답 내용을 완전히 충족함.

retriever B 점수: 2점
- 3초 이내 응답 기준은 명확히 포함되어 있으나, 10초 이상 소요 작업에 대한 팝업 안내 요구가 없어 정답의 일부만 충족함. 

============================================================
### query:
원자력 선량평가시스템의 포트란 모듈 개선 요구사항에서 추가로 고려해야 하는 평가 요소는 무엇인가?

### answer:
 ICRP60 기준에 따른 23개 장기를 고려한 평가장기 추가가 요구된다.

---

### retriever A:
가 방법
평가부문
평가항목
평가기준
배점
한도
정량
평가
(10)
경영상태
경영상태(신용상태등급) 평가
10
전략 및 방법론
(22)
사업
이해도
사업의 특성 및 목표에 부합하는 사업전략을 제시하고 있는지를 평가한다.
7
추진전략
개발업무 수행 시 일정 및 위험요소를 고려하여 얼마나 타당한 추진 전략을 수립하였는지 평가한다.
5
적용기술
사업에서 적용하고자 하는 기술이 향후 확장성을 고려하여 현실적으로 실현 가능하게 제시되어 있는지를 평가한다.
4
표준프레임 워크 적용
제안사는 사업에 네팔 정보통신기술부 프레임워크 기반의 공통 컴포넌트의 유형 및 사용계획을 기술하고 사용성 향상을 위한 추가 컴포넌트에 대해 적용방안을 제시하여야한다.
3
개발
방법론
사업에 적정한 개발 방법론이 제시되었는지 평가하고, 실제 적용 사례 및 경험을 바탕으로 단계별 산출물이 제시되었는지 여부를 평가한다.
3
기술 및 기능
(26)
시스템 장비 구성 요구사항
요구 규격을 충족하는 장비 제안 여부와 현 시스템과의 인터페이스 및 확장 가능성 여부를 평가한다. 또한 도입 장비의 설치 및 공급 계획, 유지관리에 대한 방안이 구체적으로 기술되어 있는가를 평가한다.
7
기능 요구사항
기능 요구사항·기대사항·제약사항 등을 파악한 후 구현 방안이 구체적으로 기술되어 있는지를 평가한다. 또한 제안한 방안 및 기술이 적용 가능한지를 평가한다.
7
데이터
요구사항
데이터 요구사항 분석을 바탕으로 데이터 전환계획 및 검증 방법, 데이터 오류 발생 시 처리 방안이 구체적으로 제시되어 있는지를 평가한다.
4
보안
요구사항
기능 보안
체적이고 적절한지 평가
(30점) 요구사항
- 구현방안이 설계단계부터 구현 및 검증단계까지 고려하고 있는지 평가
- 데이터 전환 계획 및 검증 방법, 에러 데이터 처리 방법에 대해 구체적
데이터
인 내용을 제시하고 있는지 평가 15
요구사항
- 데이터 전환을 위해 책임 조직이 투입되는지 평가
- 기능 및 품질 등 요구사항을 구현 시 관련 제약사항을 충족시키며 구현
제약사항
방안 및 테스트 방안을 구체적으로 기술하였는지 평가
-237-
평가
평가항목 평가기준 배점
부문
- 구현하고자 하는 기능에 요구 성능이 충족되도록 방법론 및 분석 도구
성능 를 통하여 내용을 구체적으로 분석하였는지, 구현 및 테스트 방안을 구
요구사항 체적으로 기술하였는지 평가
- 제안한 방안 및 기술을 통해 요구 성능을 충족시킬 수 있는지 평가
- 분석 도구 및 구현 방안, 테스트 방안을 구체적으로 기술하였는지, 분석
·설계 등 각 단계별 품질 요구사항의 점검 및 검토 방안을 구체적으로
품질
계획하였는지 평가
요구사항
- 각 단계마다 품질 요구사항을 점검하는 별도의 전문 인력이 투입되는지
성능 및 평가
품질 - (시스템 인터페이스) 타 시스템과의 연계 방안들에 대한 장·단점의 분석 10
(10점) 을 통해 가장 적합한 방안을 구체적으로 기술했는지, 구현 경험이 있는
인터페이
인터페이스 담당자가 투입되는지 평가
스
- (사용자 인터페이스) 사용자 편의성을 고려하여 요구사항을 제공하기 위
요구사항
한 분석 및 설계, 구현 방안과 검토 계획을 구체적으로 기술하였는지,
구현 경험이 있는 사용자 인터페이스 담당자가 투입되는지 평가


---

### retriever B:
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항


---

### openai answer
retriever A 점수: 0점
질문/정답의 핵심(원자력 선량평가시스템, 포트란 모듈, ICRP60, 23개 장기 추가)과 관련 정보가 전혀 없음. 일반적 평가기준·요구사항만 있음.

retriever B 점수: 0점
“기능 요구사항” 반복뿐이며, ICRP60, 장기 수, 원자력 선량평가, 포트란 등 핵심 키워드나 정답 관련 내용이 전혀 없음. 

## hybrid_results

============================================================
### query:
아이돌봄 인적성 검사 시스템 운영 사업의 주요 목적은 무엇인가?

### answer:
인적성 검사 수행을 지원하고, 돌봄활동과 연계한 사후관리 및 인력 검증을 강화하는 것이다.

---

### retriever A:
◦원할한 인적성 검사 수행을 위한 정보시스템 지원 필요
◦돌봄활동과 연계한 사후관리 기반을 강화하여 이용자 만족도 향상
◦아이돌봄인력 인·적성에 기초한 돌봄인력 관리 강화 필요
◦정보시스템 개인정보보호법 준수를 위한 보호조치
2
사업 개요
□ 사 업 명: 아이돌봄인력 인적성 검사 정보시스템 운영
- 2024. 정보시스템 구축에 따른 유지보수 사업
□ 사업기간: 계약일 ~ 2025. 12. 31.
□ 소요예산: 172,700,000원(부가세 포함)
□ 추진방식: 제한경쟁입찰(협상에 의한 계약)
3
사업 범위
□ 인·적성 프로그램 기능수정 및 안정화 기술지원
◦인·적성 시스템의 안정적인 정보화 지원을 위한 상시 유지관리
체계 지원
◦인·적성 검사 변경에 따른 프로그램 수정 및 결과 반영
◦웹 접근성, 기술적 보안 등 전자정부 가이드라인 준수
□ 개인정보보호 및 정보보안 강화
◦개인정보보호 및 정보보호 강화 조치(방안)
◦정보시스템 운영을 위한 물리적, 기술적 보안 대책 제시
□ 안정적인 클라우드 환경 운영 및 제공
◦클라우드 무중단 환경 제공
◦클라우드 환경 운영을 위한 기술적 유지관리 지원
◦시스템 용을 위한 SW 및 외부 서비스 연계
4
기대 효과
□ (안전한 돌봄 환경 조성) 아이돌봄인력 선별 및 검증 강화 통한 안심하고 맡길 수 있는 돌봄서비스 제공
□ (효율성 제고) 아이돌봄인력 대상 인적성 검사를 편리하게 실시하여 돌봄인력 검증 절차 효율성 제고
□ (정책지원 및 제도개선 지원) 체계적인 검사 결과를 축적하고 정책 수립지원을 위한 기초자료를 제공, 지속 가능한 아이돌봄서비스 제도개선에 활용
□ (안전한 돌봄 환경 조성) 아이돌봄인력 선별 및 검증 강화 통한 안심하고 맡길 수 있는 돌봄서비스 제공
□ (효율성 제고) 아이돌봄인력 대상 인적성 검사를 편리하게 실시하여 돌봄인력 검증 절차 효율성 제고
□ (정책지원 및 제도개선 지원) 체계적인 검사 결과를 축적하고 정책 수립지원을 위한 기초자료를 제공, 지속 가능한 아이돌봄서비스 제도개선에 활용
□ (돌봄인력 관리) 데이터를 기반으로 하는 아이돌봄인력에 대한 효율적 관리·감독 가능
Ⅱ. 사업 현황
1
업무현황
□ 인·적성검사 실시 정보시스템 지원
ㅇ 인·적성검사 및 실시를 위한 정보 시스템 제공
ㅇ 정보시스템 안정화 및 운영지원
□ 인·적성검사 운영
ㅇ 인적성 검사 실시 및 수행 방법 안내
ㅇ 인적성 검사 수행 모니터링 및 결과 분석
ㅇ 검사 실시 운영자 권한 부여 및 인증 관리
2
정보화 현황
□ 시스템 구성도
□ 정보시스템 구성도 요청 시 별도 공개
※ 행정기관 및 공공기관 정보시스템 구축 및 운영 지침(행정안전부 고시 제 2024-66호, 2024.8.28.) 제 17조(제안요청서 보안사항 등)에 따라, 시스템 구성도 및 상세자료 등은 공개하지 않으며, 입찰에 참가하고자 하는 사업자의 요청이 있는 경우 보안서약서를 제출 후 담당자 입회하에 제한적으로 열람 가능
□ 시스템(장비)·소프트웨어 운영현황
AS-IS
TO-BE (2025년)
ㅇ인·적성 검사 관리기능 및 사용자화면 개발
※ 클라우드 환경 기반 개발
ㅇ인·적성 검사 기능 최적화 및 운영 기술지원
-인·적성 검사 고도화에 따른 수정 반영
(검사도구 4종)
※사업 운영에 따라 변경될수 있음


---

### retriever B:
기대 효과
□ (안전한 돌봄 환경 조성) 아이돌봄인력 선별 및 검증 강화 통한 안심하고 맡길 수 있는 돌봄서비스 제공
□ (효율성 제고) 아이돌봄인력 대상 인적성 검사를 편리하게 실시하여 돌봄인력 검증 절차 효율성 제고
□ (정책지원 및 제도개선 지원) 체계적인 검사 결과를 축적하고 정책 수립지원을 위한 기초자료를 제공, 지속 가능한 아이돌봄서비스 제도개선에 활용
□ (돌봄인력 관리) 데이터를 기반으로 하는 아이돌봄인력에 대한 효율적 관리·감독 가능
Ⅱ. 사업 현황
1
업무현황
□ 인·적성검사 실시 정보시스템 지원
ㅇ 인·적성검사 및 실시를 위한 정보 시스템 제공
ㅇ 정보시스템 안정화 및 운영지원
□ 인·적성검사 운영
ㅇ 인적성 검사 실시 및 수행 방법 안내
ㅇ 인적성 검사 수행 모니터링 및 결과 분석
ㅇ 검사 실시 운영자 권한 부여 및 인증 관리
2
정보화 현황
□ 시스템 구성도
□ 정보시스템 구성도 요청 시 별도 공개
※ 행정기관 및 공공기관 정보시스템 구축 및 운영 지침(행정안전부 고시 제 2024-66호, 2024.8.28.) 제 17조(제안요청서 보안사항 등)에 따라, 시스템 구성도 및 상세자료 등은 공개하지 않으며, 입찰에 참가하고자 하는 사업자의 요청이 있는 경우 보안서약서를 제출 후 담당자 입회하에 제한적으로 열람 가능
□ 시스템(장비)·소프트웨어 운영현황
AS-IS
TO-BE (2025년)
ㅇ인·적성 검사 관리기능 및 사용자화면 개발
※ 클라우드 환경 기반 개발
ㅇ인·적성 검사 기능 최적화 및 운영 기술지원
-인·적성 검사 고도화에 따른 수정 반영
(검사도구 4종)
□ 시스템(장비)·소프트웨어 운영현황
AS-IS
TO-BE (2025년)
ㅇ인·적성 검사 관리기능 및 사용자화면 개발
※ 클라우드 환경 기반 개발
ㅇ인·적성 검사 기능 최적화 및 운영 기술지원
-인·적성 검사 고도화에 따른 수정 반영
(검사도구 4종)
※사업 운영에 따라 변경될수 있음
ㅇ무중단 클라우드 기반 제공
ㅇ클라우드 안정화 운영기반 및 기술지원
SW 도입 현황
SW 명칭
용도
비고
리포팅 툴
OZ Report
인·적성 결과지 출력
발주처 제공
□ 시스템 장비 현황 요청 시 별도 공개
※ 행정기관 및 공공기관 정보시스템 구축 및 운영 지침(행정안전부 고시 제 2024-66호, 2024.8.28.) 제 17조(제안요청서 보안사항 등)에 따라, 시스템 구성도 및 상세자료 등은 공개하지 않으며, 입찰에 참가하고자 하는 사업자의 요청이 있는 경우 보안서약서를 제출 후 담당자 입회하에 제한적으로 열람 가능
Ⅲ. 사업 추진방안
1
추진 목표
□ `24년 개발된 인·적성 검사 운영을 지원하고, 데이터 수집 및 분석을 위한 정보체계 관리
□ 인·적성 검사 정보시스템 안정적 기반 지원 및 보안 강화
2
추진체계
□ 추진체계도
발주기관
<한국건강가정진흥원>
협조부서
<경영지원부/
소통협력실>
주관부서
<돌봄지원부>
관련부처
<여성가족부>
주관사업자
<사업수행업체>
□ 추진 주체별 역할
구 분
주요 업무 내용
주관부서
ㅇ 사업총괄 관리 및 시행
ㅇ 사업계획서 수립 및 제안요청서 작성
ㅇ 개발내역 업무정의 및 과업내용 제시
ㅇ 개발관련 자료제공 및 시스템 인수
ㅇ 사업수행 관리 및 검사, 결과 평가
협조부서


---

### openai answer
retriever A 점수: 3점
(검사 수행 지원, 돌봄활동 연계 사후관리 강화, 인력 검증 강화가 모두 명시되어 있어 답의 모든 요소를 포함)

retriever B 점수: 2점
(검사 수행 지원과 인력 검증 강화는 있으나, ‘돌봄활동과 연계한 사후관리’가 명시적으로 없음) 

============================================================
### query:
아이돌봄 시스템에서 개인정보 보호를 위해 요구되는 조치는 무엇인가?

### answer:
접속기록 관리, 내부관리계획 수립 등 기술적·관리적 보호조치를 수행해야 한다.

---

### retriever A:
- 개인정보의 기술절·관리적 보호조치 기준(개인정보위)
- 표준 개인정보 보호 지침(개인정보위)
가명정보의 결합 및 반출 등에 관한 고시(개인정보위)
개인정보 영향평가에 관한 고시(개인정보위)
※ 고시에 따라 아래 기준 이상의 개인정보 처리 시 발주처에서 개인정보 영향평가를 발주하며 영향평가결과를 반영하여 시스템을 구축해야 함
내 용
1
구축‧운용 또는 변경하려는 개인정보파일로서 5만명 이상의 정보주체에 관한 민감정보 또는 고유식별정보의 처리가 수반되는 개인정보파일
2
구축‧운용하고 있는 개인정보파일을 해당 기관 내부 또는 외부에서 구축‧운용하고 있는 다른 개인정보파일과 연계하려는 경우로서 연계결과 50만명 이상의 정보주체에 관한 개인정보가 포함되는 개인정보파일
3
구축‧운용 또는 변경하려는 개인정보파일로서 100만명 이상의 정보주체에 관한 개인정보파일
4
개인정보 영향평가를 받은 후에 개인정보 검색체계 등 개인정보파일의 운용체계를 변경하려는 경우 그 개인정보파일
○ 개인정보 안전성 확보조치 기준 이행·반영
- 개인정보 열람 시 접속기록(최소 2년 이상) 보관·관리
- 개인정보 접근권한 부여·변경·말소 이력(최소 3년 이상) 보관·관리
- 고유식별번호 및 비밀번호 생체인식정보는 안전한 암호 알고리즘을 사용 하여 저장
○ 데이터의 보안 강화를 위해 SSL서비스 도입
○ 개인정보보호를 위한 DB 암호화하여 자료 유출방지
- 인증서 기반(GPKI) 로그인 방식 적용 후 ID/PW 로그인
- 정보보안 강화를 위한 시스템에 각종 패치, 환경설정 지원 등
◦ 보안 요구사항
요구사항 고유번호
SER-001
요구사항 명칭
개인정보 처리
요구사항 분류
보안 요구사항
요구사항
상세설명
정의
개인정보 처리 및 암호화
세부
내용
ㅇ 개인정보를 보유할 경우에는 관련 개인정보를 암호화하여야 함
ㅇ 서비스의 보안성 강화를 위하여 SSL 보안 통신을 실시하여야 함
ㅇ 사업의 암호화 대상을 식별하고 암호화 도구를 활용하여 암복호화 방안을 마련해야 함
- 고유식별정보 : 주민번호, 여권번호, 운전면허정보 등
ㅇ개인을 식별할 수 있는 정보(사용자 인증 정보, 패스워드 등)를 운영DB 또는 개발DB에 저장할 경우, 암호화하여 저장하며, 소스코드에 직접 하드코딩하지 않음(패스워드는 단방향 암호화 처리, SHA-256 이상 암호화 알고리즘 사용)
ㅇ 개인정보 취급자가 개인정보처리 시스템에 접속하는 경우 접속 로그를 시스템에 최소 1년 이상 기록 관리해야 함(5만명 이상의 정보주체에 관하여 개인정보를 처리하거나 고유식별정보 또는 민감정보를 처리하는 개인정보처리시스템의 경우에는 2년 이상 보관)
◈ 필수 관리 항목
• 개인정보취급자 식별정보 : ID ୴Ā • 날짜 및 시간 : 접속일시
• 접속자 IP 주소 : 접속자 정보
• 처리한 정보주체 정보 : 정보주체 식별정보(계정, 회원번호 등)
• 수행업무 : 열람, 수정, 삭제, 인쇄, 입력, 다운로드 등
ㅇ 개인정보 취급자가 개인정보를 처리할 경우 해당 접근 로그를 시스템에 기록할 것
ㅇ 개인정보처리방침에 따라 개인정보를 처리/파기해야 하며 백업데이터 생성 금지


---

### retriever B:
6. 그 밖에 개인정보의 보호를 위하여 필요한 사항
-295-
1. 개인정보의 안전성 확보조치 철저(「개인정보보호법」제29조)
❍ 개인정보처리자는 개인정보가 분실·도난·유출·변조 또는 훼손되지 않도록 다음 각 호의 안전성 확보조치를 하여야한다.
구 분
주 요 내 용
내부관리계획(제3조)
◦내부관리계획 포함사항 명시 ※개인정보 보호책임자 지정, 개인정보취급자 교육 등
◦개인정보 처리 업무 위탁 시 관리·감독 사항
접근권한 관리(제4조)
◦개인정보처리시스템의 접근권한 부여․변경․말소 기록 최소 3년간 보관
◦안전한 비밀번호 작성규칙 수립․적용
접근통제(제5조)
◦접근통제 시스템 설치․운영 의무화 ※접근통제 시스템 : 침입차단(Firewall), 침입탐지(IDS) 기능 포함
◦주민번호 처리 시 추가 정보 확인, 모바일 기기 보호조치 조항, 홈페이지에서 고유식별정보를 처리할 경우 년 1회 취약점 점검 조치
개인정보 암호화
(제6조)
◦암호화 대상 : 고유식별정보(주민등록번호, 여권번호, 운전면허번호, 외국인등록번호), 비밀번호, 바이오정보
◦암호화 기준
구 분
암 호 화 기 준
송․수신시
모든 정보 암호화 전송
저장시
비밀번호,
바이오정보
비밀번호는 단방향 암호화,
바이오정보는 양방향 암호화
고유식별정보
ⓛ 인터넷 구간,공통(DMZ) 구간 저장시 : 의무적 암호화
보안 요구사항
상세
설명
정의
개인정보의 안전성 확보를 위한 기술적 조치사항
세부
내용
 개인정보를 처리함에 있어 『개인정보 보호법』, 『개인정보의 안전성 확보조치 기준』 등 관련 법규 및 규정을 준수하여야 함
 접근 권한 관리
- 개인정보처리시스템에 접속할 수 있는 사용자계정을 발급하는 경우, 개인정보취급자 별로 사용자계정을 발급하여야 하며, 취급자의 역할에 따라 차등부여할 수 있어야 함
- 개인정보 처리내역에 대한 책임 추적성을 확보하기 위하여 접근권한 부여, 변경, 말소에 대한 내역을 이력 관리 또는 조회가 가능하여야 하며, 해당 기록을 최소 3년간 보관할 수 있도록 구현해야 함
예) 신청자 정보, 신청일시, 승인자 및 발급자 정보, 신청 및 발급 사유 등
- 개인정보처리시스템에 접근하여 개인정보를 생성수정삭제열람 등을 하고자 하는 경우 다음 사항을 포함하는 정보이용 내역을 기록하고 남길 수 있도록 시스템을 구현하여야 함.


---

### openai answer
retriever A 점수: 2점
접속기록 관리와 암호화 등 기술·관리적 보호조치는 제시되나, 내부관리계획 수립이 명시되어 있지 않아 부분 충족입니다.

retriever B 점수: 3점
내부관리계획(제3조)과 접속·이용내역 기록 등 접속기록 관리가 모두 명시되어 있어 답변 요소를 모두 포함합니다. 

============================================================
### query:
2025 구미 아시아육상경기선수권대회의 개최 기간은 언제인가?

### answer:
2025년 5월 27일부터 5월 31일까지 총 5일간 개최된다.

---

### retriever A:
종합정보시스템 및 홈페이지 등 구축 용역 제안요청서
1
대회개요
아시아의 꿈, 구미에서 세계로
 The dream of ASIA, From GUMI to the World 
비전
ㅇ (대 회 명) 2025 구미아시아육상경기선수권대회
ㅇ (기 간) 2025. 5. 27.(화) ~ 5. 31.(토) 5일간
ㅇ (장 소) 구미시민운동장 일원
ㅇ (규 모) 아시아 45개국 1,200여명 / 선수단 및 임원
ㅇ (주 최) 아시아육상연맹(Asian Athletics, AA)
ㅇ (주 관) 2025 구미아시아육상경기선수권대회 조직위원회
ㅇ (경기종목) 45개 종목
구 분
종 목 명
남자
(22)
100m, 200m, 400m, 800m, 1,500m, 5,000m,
10,000m, 110mH, 400mH, 3,000m장애물달리기,
20km경보, 4×100mR, 4×400mR, 높이뛰기,
장대높이뛰기, 멀리뛰기, 세단뛰기, 포환던지기,
원반던지기, 해머던지기, 창던지기, 10종 경기
여자
(22)
100m, 200m, 400m, 800m, 1,500m, 5,000m,
10,000m, 100mH, 400mH, 3,000m장애물달리기,
20km경보, 4×100mR, 4×400mR, 높이뛰기,
장대높이뛰기, 멀리뛰기, 세단뛰기, 포환던지기,
원반던지기, 해머던지기, 창던지기, 7종 경기
남·여
(1)
4×400m 믹스릴레이
요
2
과업개요
ㅇ (사 업 명) 2025 구미아시아육상경기선수권대회 종합정보시스템 및 홈페이지 등 구축 용역
ㅇ (사업기간) 계약일 ~ 2025. 6. 6.(금)
ㅇ (사업위치) 시민운동장 일원
2024. . .
대표자 : (인)
대리인 : (인)
2025 구미아시아육상경기선수권대회 조직위원장 귀하
※ 첨부서류 : 재직증명서, 신분증
※ 위임장에 사용하는 도장은 대표자 인감과 같아야 함.
※ 대리 접수로 인하여 발생하는 모든 문제는 대표자 본인이 책임짐
【 서식 3 】
재 직 증 명 서
성 명
생 년 월 일
부서/직위
소 속
주 소
재직기간
위와 같이 당사에 재직 중임을 증명합니다.
2024. . .
ྠĀ ྠĀ ྠĀ ྠĀ ྠĀ ྠĀ 업 체 명 :
ྠĀ ྠĀ ྠĀ ྠĀ ྠĀ ྠĀ 주 소 :
ྠĀ ྠĀ ྠĀ ྠĀ ྠĀ ྠĀ 대 표 자 : (인)
2025 구미아시아육상경기선수권대회 조직위원장 귀하
【 서식 4 】
사 용 인 감 계
사용인감
는(은) 2025 구미아시아육상경기선수권대회 조직위원회와 거래함에 있어 입찰, 계약 및 이에 수반되는 모든 행위에 대하여 위의 인감을 사용하고자 하며, 동 인감을 사용함으로써 민․형사상의 문제가 발생할 경우 이(가) 책임을 질 것을 확인합니다.
붙임 : 인감(법인)증명서 1부. 끝.
2024년 월 일
ྠĀ ྠĀ ྠĀ ྠĀ ྠĀ ྠĀ 주 소 :
ྠĀ ྠĀ ྠĀ ྠĀ ྠĀ ྠĀ 상 호 :
ྠĀ ྠĀ ྠĀ ྠĀ ྠĀ ྠĀ 대표자 : (인)
2025 구미아시아육상경기선수권대회 조직위원장 귀하
【 서식 5 】
서 약 서
당사는 2025 구미아시아육상경기선수권대회 종합정보시스템 및 홈페이지 등 구축 용역 참가신청과 관련하여 모든 서류와 증빙자료를 신의와 성실의 원칙에 입각하여 작성 제출하며, 아래 내용을 준수할 것을 확약하고 이에 서약서를 제출합니다.


---

### retriever B:
종합정보시스템 및 홈페이지 등 구축 용역 제안요청서
1
대회개요
아시아의 꿈, 구미에서 세계로
 The dream of ASIA, From GUMI to the World 
비전
ㅇ (대 회 명) 2025 구미아시아육상경기선수권대회
ㅇ (기 간) 2025. 5. 27.(화) ~ 5. 31.(토) 5일간
ㅇ (장 소) 구미시민운동장 일원
ㅇ (규 모) 아시아 45개국 1,200여명 / 선수단 및 임원
ㅇ (주 최) 아시아육상연맹(Asian Athletics, AA)
ㅇ (주 관) 2025 구미아시아육상경기선수권대회 조직위원회
ㅇ (경기종목) 45개 종목
구 분
종 목 명
남자
(22)
100m, 200m, 400m, 800m, 1,500m, 5,000m,
10,000m, 110mH, 400mH, 3,000m장애물달리기,
20km경보, 4×100mR, 4×400mR, 높이뛰기,
장대높이뛰기, 멀리뛰기, 세단뛰기, 포환던지기,
원반던지기, 해머던지기, 창던지기, 10종 경기
여자
(22)
100m, 200m, 400m, 800m, 1,500m, 5,000m,
10,000m, 100mH, 400mH, 3,000m장애물달리기,
20km경보, 4×100mR, 4×400mR, 높이뛰기,
장대높이뛰기, 멀리뛰기, 세단뛰기, 포환던지기,
원반던지기, 해머던지기, 창던지기, 7종 경기
남·여
(1)
4×400m 믹스릴레이
요
2
과업개요
ㅇ (사 업 명) 2025 구미아시아육상경기선수권대회 종합정보시스템 및 홈페이지 등 구축 용역
ㅇ (사업기간) 계약일 ~ 2025. 6. 6.(금)
ㅇ (사업위치) 시민운동장 일원
1. 기본정보
사업명
2025 구미아시아육상경기선수권대회 종합정보시스템 및 홈페이지 등 구축 용역
영향평가단계
□ 예산편성
□ 그 외 필요시
■ 사업발주
□ 재평가
주요 내용
대회 기본 자료 관리 및 대회 계측기 연동 등
사업기간
(또는 개발기간)
계약일 ~ 2025년 6월
구분
붙임 : 산출내역서 각 1부
2024. . .
제안업체 인(사용인감)
2025 구미아시아육상경기선수권대회 조직위원장 귀하
※ 제안금액은 반드시 한글로 띄어쓰기 없이 붙여 쓰고 아라비아 숫자는 괄호 안에 기재할 것.
【 서식 9-1 】
산 출 내 역 서
구 분
항 목
금 액
구 성 비
비 고
순
용
역
원
가
소 계
소 계
소 계
소 계
일반관리비 ( )%
이 윤 ( )%
총 원 가
부 가 가 치 세
총 용 역 비
※ 작성내용에 따라 줄 수 증감가능
【 서식 10 】
접수번호
2025 구미아시아육상경기선수권대회 종합정보시스템 및 홈페이지 등 구축 용역
정량적 평가 제안서
2024년 월 일
업 체 명
※ 주의사항


---

### openai answer
retriever A 점수: 3점
대회 개요에 “(기 간) 2025. 5. 27.(화) ~ 5. 31.(토) 5일간”이 명시되어 정답의 시작일·종료일·총일수가 모두 포함됨.

retriever B 점수: 3점
동일하게 “(기 간) 2025. 5. 27.(화) ~ 5. 31.(토) 5일간”이 명시되어 정답 정보를 모두 포함함. 

============================================================
### query:
구미 아시아육상경기선수권대회 홈페이지에서 제공되는 주요 콘텐츠 항목에는 무엇이 포함되는가?

### answer:
대회소개, 경기일정 및 결과, 메달순위, 참가선수 정보, 커뮤니티 등이 포함된다.

---

### retriever A:
-경기참가자(Start List), 경기결과(Official Result, Summary) PDF 파일 제공
<다음 정보를 홈페이지에 전송>
Level1
Level2
기능 설명
경기
일정
전체 경기 일정
대회기간 전체 일정 제공
일자별 경기 일정
일자별 세부 경기 일정 제공
세부 종목별 경기 일정
세부 종목별 경기 일정 제공
선수별 경기 일정
선수별 경기 일정 제공
메달
국가/선수별 메달 현황
국가/선수별 메달 현황 제공
종목별 메달 현황
종목별 메달현황 및 메달수상자 정보 제공
경기
정보
스타트 리스트
세부종목별 편성 조 및 출전선수 목록 제공
대진 정보
세부 종목별 대진정보 조회 및 출력
(경기진행 상태에 따른 결과 반영 포함)
경기 결과
각 조별, 종목별 경기 결과 조회 및 출력
선수
정보
국가별 선수 정보
국가별 참가선수 목록 및 정보 조회
세부 종목별 선수 정보
세부 종목별 참가선수 목록 및 정보 조회
순위 정보
세부종목별 참가선수 순위 정보 목록 조회
과업내용
명칭
대분류
대회 종합정보시스템 S/W 개발
소분류
보고서관리시스템 개발
과업내용
데이터 출력 기능
<PDF 및 Excel, 증서 페이지 등 출력>
-참가선수 신상정보
-국가별/세부 종목별 참가선수 리스트
-심판/임원 신상정보
-국가별/세부 종목별 심판/임원 리스트
-세부 종목별, 예결선 Start List 및 조 편성
-세부 종목별 경기 결과 (Live Result, Official Result, Summary)
-세부 종목별 메달 현황 및 신기록 현황
-참가증서
ː참가선수(종목별 1위~6위 증서 및 참가증서 필수)
 운영 매뉴얼 또는 지침서 제공 및 시스템 운영 관계자 교육
ㅇ (현장IT팀 상주 및 시스템 운영)
 사전 등록, 경기 진행 및 결과 출력 시스템 등 운영
 대회시작 4일전~대회 종료일까지 경기장 상주
 경기장/비경기장별 운영 인원 배치
ㅇ (홈페이지 구축)
 메뉴 구성도(안)
아시아육상경기선수권대회 메인
대회소개
소식지
경기일정/결과
메달순위
육상경기
커뮤니티
환영인사
공지사항
경기일정
국가별순위
경기종목
안내사항
대회개요
보도자료
행사일정
선수별순위
참가선수
아카이브
대회상징물
사진자료
시설안내
유실물센터
조직위안내
자주묻는질문
홍보영상
자원봉사자
찾아오시는 길
구미시 안내
메뉴 상세설명(안)
메뉴명
부메뉴
주요 내용
메인 화면
◦2025 구미 아시아육상경기선수권대회를 대표할 수 있는 디자인 (공식 상징물 활용 등)
◦활용빈도가 높은 메뉴 배치 및 팝업배너
대회 소개
환영인사
◦대회 개최에 대한 인사 말씀
대회개요
◦대회명칭, 대회기간, 대회장소, 참가규모, 참가국, 참가인원, 종목, 주최, 주관 등
대회상징물
◦대회마크, 마스코트, 포스터, 슬로건
조직위안내
◦조직위원회·아시아육상추진단 조직도
홍보영상
◦홍보영상, 유튜브 쇼츠 영상, 축하 메시지 등
찾아오시는 길
◦경기장 위치, 교통수단별 길 안내
소식지
공지사항
◦대회 관련 공지사항 등
보도자료
◦대회 관련기사 스크랩 자료
사진자료
◦개·폐회식 사진, 각 종목별 경기 이미지 게시
경기일정
/
결과
경기일정
◦경기장 및 개최 일자, 시간, 라운드, 진행상태
◦각 종목별 경기결과, 종합순위
행사일정
◦개·폐회식 정보 및 식순 안내


---

### retriever B:
 운영 매뉴얼 또는 지침서 제공 및 시스템 운영 관계자 교육
ㅇ (현장IT팀 상주 및 시스템 운영)
 사전 등록, 경기 진행 및 결과 출력 시스템 등 운영
 대회시작 4일전~대회 종료일까지 경기장 상주
 경기장/비경기장별 운영 인원 배치
ㅇ (홈페이지 구축)
 메뉴 구성도(안)
아시아육상경기선수권대회 메인
대회소개
소식지
경기일정/결과
메달순위
육상경기
커뮤니티
환영인사
공지사항
경기일정
국가별순위
경기종목
안내사항
대회개요
보도자료
행사일정
선수별순위
참가선수
아카이브
대회상징물
사진자료
시설안내
유실물센터
조직위안내
자주묻는질문
홍보영상
자원봉사자
찾아오시는 길
구미시 안내
메뉴 상세설명(안)
메뉴명
부메뉴
주요 내용
메인 화면
◦2025 구미 아시아육상경기선수권대회를 대표할 수 있는 디자인 (공식 상징물 활용 등)
◦활용빈도가 높은 메뉴 배치 및 팝업배너
대회 소개
환영인사
◦대회 개최에 대한 인사 말씀
대회개요
◦대회명칭, 대회기간, 대회장소, 참가규모, 참가국, 참가인원, 종목, 주최, 주관 등
대회상징물
◦대회마크, 마스코트, 포스터, 슬로건
조직위안내
◦조직위원회·아시아육상추진단 조직도
홍보영상
◦홍보영상, 유튜브 쇼츠 영상, 축하 메시지 등
찾아오시는 길
◦경기장 위치, 교통수단별 길 안내
소식지
공지사항
◦대회 관련 공지사항 등
보도자료
◦대회 관련기사 스크랩 자료
사진자료
◦개·폐회식 사진, 각 종목별 경기 이미지 게시
경기일정
/
결과
경기일정
◦경기장 및 개최 일자, 시간, 라운드, 진행상태
◦각 종목별 경기결과, 종합순위
행사일정
◦개·폐회식 정보 및 식순 안내
종합정보시스템 및 홈페이지 등 구축 용역 제안요청서
1
대회개요
아시아의 꿈, 구미에서 세계로
 The dream of ASIA, From GUMI to the World 
비전
ㅇ (대 회 명) 2025 구미아시아육상경기선수권대회
ㅇ (기 간) 2025. 5. 27.(화) ~ 5. 31.(토) 5일간
ㅇ (장 소) 구미시민운동장 일원
ㅇ (규 모) 아시아 45개국 1,200여명 / 선수단 및 임원
ㅇ (주 최) 아시아육상연맹(Asian Athletics, AA)
ㅇ (주 관) 2025 구미아시아육상경기선수권대회 조직위원회
ㅇ (경기종목) 45개 종목
구 분
종 목 명
남자
(22)
100m, 200m, 400m, 800m, 1,500m, 5,000m,
10,000m, 110mH, 400mH, 3,000m장애물달리기,
20km경보, 4×100mR, 4×400mR, 높이뛰기,
장대높이뛰기, 멀리뛰기, 세단뛰기, 포환던지기,
원반던지기, 해머던지기, 창던지기, 10종 경기
여자
(22)
100m, 200m, 400m, 800m, 1,500m, 5,000m,
10,000m, 100mH, 400mH, 3,000m장애물달리기,
20km경보, 4×100mR, 4×400mR, 높이뛰기,
장대높이뛰기, 멀리뛰기, 세단뛰기, 포환던지기,
원반던지기, 해머던지기, 창던지기, 7종 경기
남·여
(1)
4×400m 믹스릴레이
요
2
과업개요
ㅇ (사 업 명) 2025 구미아시아육상경기선수권대회 종합정보시스템 및 홈페이지 등 구축 용역
ㅇ (사업기간) 계약일 ~ 2025. 6. 6.(금)
ㅇ (사업위치) 시민운동장 일원


---

### openai answer
retriever A 점수: 3점
(이유: 메뉴 구성에 ‘대회소개’, ‘경기일정/결과’, ‘메달순위’, ‘참가선수’, ‘커뮤니티’가 명시되어 있어 정답의 항목이 모두 포함됨)

retriever B 점수: 3점
(이유: 메뉴 구성에 ‘대회소개’, ‘경기일정/결과’, ‘메달순위’, ‘참가선수’, ‘커뮤니티’가 제시되어 정답의 항목을 모두 충족함) 

============================================================
### query:
원자력 선량평가시스템의 성능 요구사항 중 질의응답 시간 기준은 어떻게 정의되어 있는가?

### answer:
일반 조회 요청은 3초 이내 처리되어야 하며, 10초 이상 소요되는 작업은 사용자에게 팝업 메시지를 제공해야 한다.

---

### retriever A:
질의·응답시간
요구사항
상세설명
정의
질의·응답시간
세부
내용
의는 사용자가 요청을 하는 시간으로부터 3000ms 내에 그 결과를 보여줘야 함
❍ 임의의 선택 기준이 허용되는 대량의 데이터에 대한 질의 및 고용량의 멀티미디어 파일을 가지고 있는 프로그램에서는 적절한 처리 상태를 표시하여야 함
산출정보
성능측정 결과 내역서
관련 요구사항
요구사항 고유번호
PER-002
요구사항 명
오류 응답시간
요구사항
상세설명
정의
오류 응답시간
세부
내용
시스템에 데이터 입력 등의 사용 중에 발생한 모든 오류는 원인이 발생한지 3000ms 이하로 사용자에게 오류 메시지를 보여줘야 함
산출정보
성능측정 결과 내역서
관련 요구사항
요구사항 고유번호
PER-003
요구사항 명
통계 처리 사전 경고
요구사항
상세설명
정의
통계 처리 사전 경고
세부
내용
리 시 5,000ms 이상 지연되는 경우 연산 작업을 시작하는 시점에 예상되는 결과에 대해 사용자에게 메시지로 알려야 함
산출정보
성능측정 결과 내역서
관련 요구사항
요구사항 고유번호
PER-004
요구사항 명
부하증가 대비 성능유지
요구사항
상세설명
정의
부하증가 대비 성능유지
세부
내용
시스템의 경우 동시 접속자(사용자)는 500명 이상 접속가능 하여야 하며, 이에 따른 처리속도 등 성능이 저하되지 않아야 함
❍ 포털 시스템의 경우 동시 접속자(사용자)는 3000명 이상 접속가능 하여야 하며, 이에 따른 처리속도 등 성능이 저하되지 않아야 함
산출정보
성능측정 결과 내역서
관련 요구사항
요구사항 고유번호
PER-005
요구사항 명
오프라인 배치성 업무 응답시간
요구사항
나. 성능 요구사항 (Performance Requirement)
요구사항분류
PER-001
요구사항명
응답시간
요구사항 내용
정의
개선 대상 웹사이트 및 콘텐츠 관리 시스템(CMS) 정보 조회에 대한 응답시간 성능 보장
세부 내용
○ 정보 처리량 및 확장성 등을 고려하여 웹페이지 디스플레이 시간 기능요청에 대한 응답시간이 최소화 되도록 함
- 단순 웹페이지의 경우, 사용자가 요청을 하는 시간으로부터 3초 내에 그 결과를 보여줘야 함
- 시스템이 만드는 각 페이지의 경우, 요청한 시간으로부터 5초 내에 디스플레이 되어야 함
- 이 요구사항은 시스템을 사용하는 동시 사용자가 최대 사용자 수의 90%를 초과하는 질의에는 적용되지 않음
○ 과도한 데이터베이스 접속을 자제하며 응답속도 개선(상영시간표 등)
- 이 요구사항은 임의의 선택 기준이 허용되는 대량의 데이터(데이터량 2GB 이상)에 대한 질의에는 적용되지 않음
- 이 요구사항은 임의의 선택 기준이 허용되는 대량의 데이터에 대한 요청 및 플래시가 있을 경우에는 적용되지 않으며, 한 개 이상의 큰 이미지(이미지 500KB 이상) 및 동영상을 가지고 있는 페이지에는 적용되지 않음
- 동영상의 경우 트래픽 관리를 위해 외부 플랫폼(예: 유튜브, 기타 CDN 서비스)과 연계하여 기능 구현
○ 기능 구현에 있어 10초 이상 걸릴 수 있는 작업에 대해서는 시간지연에 대한 정보를 사용자에게 팝업 등의 방법으로 사용자에게 알려야 함
요구사항분류
PER-002
요구사항명
데이터 형식 오류 응답 시간
요구사항 내용
정의
사용자 입력 오류 응답 시간 제시
세부 내용


---

### retriever B:
성능 요구사항
상세
설명
정의
정보시스템 업무 응답시간 목표
세부
내용
□ 요청(질의) 응답시간
o 모든 질의는 사용자가 요청을 하는 시간으로부터 3초 내에 최초의 결과 값을 보여줘야 함
□ 이 요구사항은 임의의 선택 기준이 허용되는 대량의 데이터(2GB 이상)에 대한 질의에는 적용되지 않음
산출정보
o SW개발에 따른 산출물
성능 요구사항
상세
설명
정의
정보시스템 업무 응답시간 목표
세부
내용
□ 요청(질의) 응답시간
o 모든 질의는 사용자가 요청을 하는 시간으로부터 3초 내에 최초의 결과 값을 보여줘야 함
□ 이 요구사항은 임의의 선택 기준이 허용되는 대량의 데이터(2GB 이상)에 대한 질의에는 적용되지 않음
산출정보
o SW개발에 따른 산출물
성능 요구사항
응낙수준
필수
요구
사항
상세
설명
정의
사용자 요구에 대한 웹페이지 디스플레이 반응 시간
세부
내용
○ 시스템은 정상 상태에서 사용자의 질의 요청에 대한 결과 페이지를 화면에 출력할 때 3초 이내 보장
○ 평균 응답시간은 시스템을 사용하는 동시 사용자가 최대 사용자 수의 90%를 초과하는 경우에는 미적용
산출정보
관련요구사항
요구사항 내용
○ 모든 질의는 사용자가 요청하는 시간으로부터 3초 이내에 그 결과값을 보여줘야 함
○ 평균 응답 시간은 시스템을 사용하는 동시 사용자가 최대 사용자 수의 90%를 초과하는 경우 미적용함
○ 한계 인원 이상 접속 시 해결 방안 제시해야 함
② 성능 요구사항
PER-001
평균응답시간 및 성능 최적화
요구사항
상세설명
정의
업무응답시간에 목표 대한 기준
세부내용
○ 모든 질의는 평균 응답시간으로 사용자가 요청한 시간으로부터 3초 내에
그 결과를 보여줘야 함
○ 대량의 데이터 로딩(통계, 검색, 이미지 등), 사용자 숫자가 동시 사용자 용량의 90%를 초과하는 경우에는 적용되지 않음
산출 정보
개발 산출물, 테스트 결과서
관련 요구사항
성능 요구사항
상세
설명
정의
평균 응답시간
세부
내용
❍ 대상시스템은 정상 상태에서 사용자의 질의 요청에 대한 결과
페이지를 화면에 출력할 때 3초 이내에 완전히 출력되어야 함


---

### openai answer
retriever A 점수: 3점
(일반 질의 3초 이내 처리와 10초 이상 소요 작업 시 팝업 안내 요구사항이 모두 명시되어 있음)

retriever B 점수: 2점
(일반 질의 3초 이내 응답 기준은 있으나 10초 이상 소요 시 팝업 안내에 대한 내용이 없음) 

============================================================
### query:
원자력 선량평가시스템의 포트란 모듈 개선 요구사항에서 추가로 고려해야 하는 평가 요소는 무엇인가?

### answer:
 ICRP60 기준에 따른 23개 장기를 고려한 평가장기 추가가 요구된다.

---

### retriever A:
가 방법
평가부문
평가항목
평가기준
배점
한도
정량
평가
(10)
경영상태
경영상태(신용상태등급) 평가
10
전략 및 방법론
(22)
사업
이해도
사업의 특성 및 목표에 부합하는 사업전략을 제시하고 있는지를 평가한다.
7
추진전략
개발업무 수행 시 일정 및 위험요소를 고려하여 얼마나 타당한 추진 전략을 수립하였는지 평가한다.
5
적용기술
사업에서 적용하고자 하는 기술이 향후 확장성을 고려하여 현실적으로 실현 가능하게 제시되어 있는지를 평가한다.
4
표준프레임 워크 적용
제안사는 사업에 네팔 정보통신기술부 프레임워크 기반의 공통 컴포넌트의 유형 및 사용계획을 기술하고 사용성 향상을 위한 추가 컴포넌트에 대해 적용방안을 제시하여야한다.
3
개발
방법론
사업에 적정한 개발 방법론이 제시되었는지 평가하고, 실제 적용 사례 및 경험을 바탕으로 단계별 산출물이 제시되었는지 여부를 평가한다.
3
기술 및 기능
(26)
시스템 장비 구성 요구사항
요구 규격을 충족하는 장비 제안 여부와 현 시스템과의 인터페이스 및 확장 가능성 여부를 평가한다. 또한 도입 장비의 설치 및 공급 계획, 유지관리에 대한 방안이 구체적으로 기술되어 있는가를 평가한다.
7
기능 요구사항
기능 요구사항·기대사항·제약사항 등을 파악한 후 구현 방안이 구체적으로 기술되어 있는지를 평가한다. 또한 제안한 방안 및 기술이 적용 가능한지를 평가한다.
7
데이터
요구사항
데이터 요구사항 분석을 바탕으로 데이터 전환계획 및 검증 방법, 데이터 오류 발생 시 처리 방안이 구체적으로 제시되어 있는지를 평가한다.
4
보안
요구사항
기능 보안
체적이고 적절한지 평가
(30점) 요구사항
- 구현방안이 설계단계부터 구현 및 검증단계까지 고려하고 있는지 평가
- 데이터 전환 계획 및 검증 방법, 에러 데이터 처리 방법에 대해 구체적
데이터
인 내용을 제시하고 있는지 평가 15
요구사항
- 데이터 전환을 위해 책임 조직이 투입되는지 평가
- 기능 및 품질 등 요구사항을 구현 시 관련 제약사항을 충족시키며 구현
제약사항
방안 및 테스트 방안을 구체적으로 기술하였는지 평가
-237-
평가
평가항목 평가기준 배점
부문
- 구현하고자 하는 기능에 요구 성능이 충족되도록 방법론 및 분석 도구
성능 를 통하여 내용을 구체적으로 분석하였는지, 구현 및 테스트 방안을 구
요구사항 체적으로 기술하였는지 평가
- 제안한 방안 및 기술을 통해 요구 성능을 충족시킬 수 있는지 평가
- 분석 도구 및 구현 방안, 테스트 방안을 구체적으로 기술하였는지, 분석
·설계 등 각 단계별 품질 요구사항의 점검 및 검토 방안을 구체적으로
품질
계획하였는지 평가
요구사항
- 각 단계마다 품질 요구사항을 점검하는 별도의 전문 인력이 투입되는지
성능 및 평가
품질 - (시스템 인터페이스) 타 시스템과의 연계 방안들에 대한 장·단점의 분석 10
(10점) 을 통해 가장 적합한 방안을 구체적으로 기술했는지, 구현 경험이 있는
인터페이
인터페이스 담당자가 투입되는지 평가
스
- (사용자 인터페이스) 사용자 편의성을 고려하여 요구사항을 제공하기 위
요구사항
한 분석 및 설계, 구현 방안과 검토 계획을 구체적으로 기술하였는지,
구현 경험이 있는 사용자 인터페이스 담당자가 투입되는지 평가


---

### retriever B:
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항


---

### openai answer
retriever A 점수: 0점
원자력/선량/포트란/ICRP60/23개 장기 등 핵심 키워드나 답변 내용이 전혀 없고, 일반 평가항목 설명만 포함됨.

retriever B 점수: 0점
'기능 요구사항' 문구 반복뿐이며, 질문·답변과 관련된 핵심 정보(원자력, 선량, ICRP60, 23개 장기 등)가 없음. 


## multi_query_search

============================================================
### query:
아이돌봄 인적성 검사 시스템 운영 사업의 주요 목적은 무엇인가?

### answer:
인적성 검사 수행을 지원하고, 돌봄활동과 연계한 사후관리 및 인력 검증을 강화하는 것이다.

---

### retriever A:
◦원할한 인적성 검사 수행을 위한 정보시스템 지원 필요
◦돌봄활동과 연계한 사후관리 기반을 강화하여 이용자 만족도 향상
◦아이돌봄인력 인·적성에 기초한 돌봄인력 관리 강화 필요
◦정보시스템 개인정보보호법 준수를 위한 보호조치
2
사업 개요
□ 사 업 명: 아이돌봄인력 인적성 검사 정보시스템 운영
- 2024. 정보시스템 구축에 따른 유지보수 사업
□ 사업기간: 계약일 ~ 2025. 12. 31.
□ 소요예산: 172,700,000원(부가세 포함)
□ 추진방식: 제한경쟁입찰(협상에 의한 계약)
3
사업 범위
□ 인·적성 프로그램 기능수정 및 안정화 기술지원
◦인·적성 시스템의 안정적인 정보화 지원을 위한 상시 유지관리
체계 지원
◦인·적성 검사 변경에 따른 프로그램 수정 및 결과 반영
◦웹 접근성, 기술적 보안 등 전자정부 가이드라인 준수
□ 개인정보보호 및 정보보안 강화
◦개인정보보호 및 정보보호 강화 조치(방안)
◦정보시스템 운영을 위한 물리적, 기술적 보안 대책 제시
□ 안정적인 클라우드 환경 운영 및 제공
◦클라우드 무중단 환경 제공
◦클라우드 환경 운영을 위한 기술적 유지관리 지원
◦시스템 용을 위한 SW 및 외부 서비스 연계
4
기대 효과
□ (안전한 돌봄 환경 조성) 아이돌봄인력 선별 및 검증 강화 통한 안심하고 맡길 수 있는 돌봄서비스 제공
□ (효율성 제고) 아이돌봄인력 대상 인적성 검사를 편리하게 실시하여 돌봄인력 검증 절차 효율성 제고
□ (정책지원 및 제도개선 지원) 체계적인 검사 결과를 축적하고 정책 수립지원을 위한 기초자료를 제공, 지속 가능한 아이돌봄서비스 제도개선에 활용
□ (안전한 돌봄 환경 조성) 아이돌봄인력 선별 및 검증 강화 통한 안심하고 맡길 수 있는 돌봄서비스 제공
□ (효율성 제고) 아이돌봄인력 대상 인적성 검사를 편리하게 실시하여 돌봄인력 검증 절차 효율성 제고
□ (정책지원 및 제도개선 지원) 체계적인 검사 결과를 축적하고 정책 수립지원을 위한 기초자료를 제공, 지속 가능한 아이돌봄서비스 제도개선에 활용
□ (돌봄인력 관리) 데이터를 기반으로 하는 아이돌봄인력에 대한 효율적 관리·감독 가능
Ⅱ. 사업 현황
1
업무현황
□ 인·적성검사 실시 정보시스템 지원
ㅇ 인·적성검사 및 실시를 위한 정보 시스템 제공
ㅇ 정보시스템 안정화 및 운영지원
□ 인·적성검사 운영
ㅇ 인적성 검사 실시 및 수행 방법 안내
ㅇ 인적성 검사 수행 모니터링 및 결과 분석
ㅇ 검사 실시 운영자 권한 부여 및 인증 관리
2
정보화 현황
□ 시스템 구성도
□ 정보시스템 구성도 요청 시 별도 공개
※ 행정기관 및 공공기관 정보시스템 구축 및 운영 지침(행정안전부 고시 제 2024-66호, 2024.8.28.) 제 17조(제안요청서 보안사항 등)에 따라, 시스템 구성도 및 상세자료 등은 공개하지 않으며, 입찰에 참가하고자 하는 사업자의 요청이 있는 경우 보안서약서를 제출 후 담당자 입회하에 제한적으로 열람 가능
□ 시스템(장비)·소프트웨어 운영현황
AS-IS
TO-BE (2025년)
ㅇ인·적성 검사 관리기능 및 사용자화면 개발
※ 클라우드 환경 기반 개발
ㅇ인·적성 검사 기능 최적화 및 운영 기술지원
-인·적성 검사 고도화에 따른 수정 반영
(검사도구 4종)
※사업 운영에 따라 변경될수 있음


---

### retriever B:
기대 효과
□ (안전한 돌봄 환경 조성) 아이돌봄인력 선별 및 검증 강화 통한 안심하고 맡길 수 있는 돌봄서비스 제공
□ (효율성 제고) 아이돌봄인력 대상 인적성 검사를 편리하게 실시하여 돌봄인력 검증 절차 효율성 제고
□ (정책지원 및 제도개선 지원) 체계적인 검사 결과를 축적하고 정책 수립지원을 위한 기초자료를 제공, 지속 가능한 아이돌봄서비스 제도개선에 활용
□ (돌봄인력 관리) 데이터를 기반으로 하는 아이돌봄인력에 대한 효율적 관리·감독 가능
Ⅱ. 사업 현황
1
업무현황
□ 인·적성검사 실시 정보시스템 지원
ㅇ 인·적성검사 및 실시를 위한 정보 시스템 제공
ㅇ 정보시스템 안정화 및 운영지원
□ 인·적성검사 운영
ㅇ 인적성 검사 실시 및 수행 방법 안내
ㅇ 인적성 검사 수행 모니터링 및 결과 분석
ㅇ 검사 실시 운영자 권한 부여 및 인증 관리
2
정보화 현황
□ 시스템 구성도
□ 정보시스템 구성도 요청 시 별도 공개
※ 행정기관 및 공공기관 정보시스템 구축 및 운영 지침(행정안전부 고시 제 2024-66호, 2024.8.28.) 제 17조(제안요청서 보안사항 등)에 따라, 시스템 구성도 및 상세자료 등은 공개하지 않으며, 입찰에 참가하고자 하는 사업자의 요청이 있는 경우 보안서약서를 제출 후 담당자 입회하에 제한적으로 열람 가능
□ 시스템(장비)·소프트웨어 운영현황
AS-IS
TO-BE (2025년)
ㅇ인·적성 검사 관리기능 및 사용자화면 개발
※ 클라우드 환경 기반 개발
ㅇ인·적성 검사 기능 최적화 및 운영 기술지원
-인·적성 검사 고도화에 따른 수정 반영
(검사도구 4종)
□ 시스템(장비)·소프트웨어 운영현황
AS-IS
TO-BE (2025년)
ㅇ인·적성 검사 관리기능 및 사용자화면 개발
※ 클라우드 환경 기반 개발
ㅇ인·적성 검사 기능 최적화 및 운영 기술지원
-인·적성 검사 고도화에 따른 수정 반영
(검사도구 4종)
※사업 운영에 따라 변경될수 있음
ㅇ무중단 클라우드 기반 제공
ㅇ클라우드 안정화 운영기반 및 기술지원
SW 도입 현황
SW 명칭
용도
비고
리포팅 툴
OZ Report
인·적성 결과지 출력
발주처 제공
□ 시스템 장비 현황 요청 시 별도 공개
※ 행정기관 및 공공기관 정보시스템 구축 및 운영 지침(행정안전부 고시 제 2024-66호, 2024.8.28.) 제 17조(제안요청서 보안사항 등)에 따라, 시스템 구성도 및 상세자료 등은 공개하지 않으며, 입찰에 참가하고자 하는 사업자의 요청이 있는 경우 보안서약서를 제출 후 담당자 입회하에 제한적으로 열람 가능
Ⅲ. 사업 추진방안
1
추진 목표
□ `24년 개발된 인·적성 검사 운영을 지원하고, 데이터 수집 및 분석을 위한 정보체계 관리
□ 인·적성 검사 정보시스템 안정적 기반 지원 및 보안 강화
2
추진체계
□ 추진체계도
발주기관
<한국건강가정진흥원>
협조부서
<경영지원부/
소통협력실>
주관부서
<돌봄지원부>
관련부처
<여성가족부>
주관사업자
<사업수행업체>
□ 추진 주체별 역할
구 분
주요 업무 내용
주관부서
ㅇ 사업총괄 관리 및 시행
ㅇ 사업계획서 수립 및 제안요청서 작성
ㅇ 개발내역 업무정의 및 과업내용 제시
ㅇ 개발관련 자료제공 및 시스템 인수
ㅇ 사업수행 관리 및 검사, 결과 평가
협조부서


---

### openai answer
retriever A 점수: 3점
- “원활한 인적성 검사 수행 지원”, “돌봄활동과 연계한 사후관리 기반 강화”, “인력 선별·검증 강화”가 모두 명시되어 있어 정답의 요소를 전부 포함함.

retriever B 점수: 2점
- “인·적성 검사 운영 지원”, “선별·검증 강화”는 포함되나, “돌봄활동과 연계한 사후관리”에 대한 명시가 없어 일부만 충족함. 

============================================================
### query:
아이돌봄 시스템에서 개인정보 보호를 위해 요구되는 조치는 무엇인가?

### answer:
접속기록 관리, 내부관리계획 수립 등 기술적·관리적 보호조치를 수행해야 한다.

---

### retriever A:
- 개인정보의 기술절·관리적 보호조치 기준(개인정보위)
- 표준 개인정보 보호 지침(개인정보위)
가명정보의 결합 및 반출 등에 관한 고시(개인정보위)
개인정보 영향평가에 관한 고시(개인정보위)
※ 고시에 따라 아래 기준 이상의 개인정보 처리 시 발주처에서 개인정보 영향평가를 발주하며 영향평가결과를 반영하여 시스템을 구축해야 함
내 용
1
구축‧운용 또는 변경하려는 개인정보파일로서 5만명 이상의 정보주체에 관한 민감정보 또는 고유식별정보의 처리가 수반되는 개인정보파일
2
구축‧운용하고 있는 개인정보파일을 해당 기관 내부 또는 외부에서 구축‧운용하고 있는 다른 개인정보파일과 연계하려는 경우로서 연계결과 50만명 이상의 정보주체에 관한 개인정보가 포함되는 개인정보파일
3
구축‧운용 또는 변경하려는 개인정보파일로서 100만명 이상의 정보주체에 관한 개인정보파일
4
개인정보 영향평가를 받은 후에 개인정보 검색체계 등 개인정보파일의 운용체계를 변경하려는 경우 그 개인정보파일
○ 개인정보 안전성 확보조치 기준 이행·반영
- 개인정보 열람 시 접속기록(최소 2년 이상) 보관·관리
- 개인정보 접근권한 부여·변경·말소 이력(최소 3년 이상) 보관·관리
- 고유식별번호 및 비밀번호 생체인식정보는 안전한 암호 알고리즘을 사용 하여 저장
○ 데이터의 보안 강화를 위해 SSL서비스 도입
○ 개인정보보호를 위한 DB 암호화하여 자료 유출방지
- 인증서 기반(GPKI) 로그인 방식 적용 후 ID/PW 로그인
- 정보보안 강화를 위한 시스템에 각종 패치, 환경설정 지원 등
◦ 보안 요구사항
요구사항 고유번호
SER-001
요구사항 명칭
개인정보 처리
요구사항 분류
보안 요구사항
요구사항
상세설명
정의
개인정보 처리 및 암호화
세부
내용
ㅇ 개인정보를 보유할 경우에는 관련 개인정보를 암호화하여야 함
ㅇ 서비스의 보안성 강화를 위하여 SSL 보안 통신을 실시하여야 함
ㅇ 사업의 암호화 대상을 식별하고 암호화 도구를 활용하여 암복호화 방안을 마련해야 함
- 고유식별정보 : 주민번호, 여권번호, 운전면허정보 등
ㅇ개인을 식별할 수 있는 정보(사용자 인증 정보, 패스워드 등)를 운영DB 또는 개발DB에 저장할 경우, 암호화하여 저장하며, 소스코드에 직접 하드코딩하지 않음(패스워드는 단방향 암호화 처리, SHA-256 이상 암호화 알고리즘 사용)
ㅇ 개인정보 취급자가 개인정보처리 시스템에 접속하는 경우 접속 로그를 시스템에 최소 1년 이상 기록 관리해야 함(5만명 이상의 정보주체에 관하여 개인정보를 처리하거나 고유식별정보 또는 민감정보를 처리하는 개인정보처리시스템의 경우에는 2년 이상 보관)
◈ 필수 관리 항목
• 개인정보취급자 식별정보 : ID ୴Ā • 날짜 및 시간 : 접속일시
• 접속자 IP 주소 : 접속자 정보
• 처리한 정보주체 정보 : 정보주체 식별정보(계정, 회원번호 등)
• 수행업무 : 열람, 수정, 삭제, 인쇄, 입력, 다운로드 등
ㅇ 개인정보 취급자가 개인정보를 처리할 경우 해당 접근 로그를 시스템에 기록할 것
ㅇ 개인정보처리방침에 따라 개인정보를 처리/파기해야 하며 백업데이터 생성 금지


---

### retriever B:
6. 그 밖에 개인정보의 보호를 위하여 필요한 사항
-295-
1. 개인정보의 안전성 확보조치 철저(「개인정보보호법」제29조)
❍ 개인정보처리자는 개인정보가 분실·도난·유출·변조 또는 훼손되지 않도록 다음 각 호의 안전성 확보조치를 하여야한다.
구 분
주 요 내 용
내부관리계획(제3조)
◦내부관리계획 포함사항 명시 ※개인정보 보호책임자 지정, 개인정보취급자 교육 등
◦개인정보 처리 업무 위탁 시 관리·감독 사항
접근권한 관리(제4조)
◦개인정보처리시스템의 접근권한 부여․변경․말소 기록 최소 3년간 보관
◦안전한 비밀번호 작성규칙 수립․적용
접근통제(제5조)
◦접근통제 시스템 설치․운영 의무화 ※접근통제 시스템 : 침입차단(Firewall), 침입탐지(IDS) 기능 포함
◦주민번호 처리 시 추가 정보 확인, 모바일 기기 보호조치 조항, 홈페이지에서 고유식별정보를 처리할 경우 년 1회 취약점 점검 조치
개인정보 암호화
(제6조)
◦암호화 대상 : 고유식별정보(주민등록번호, 여권번호, 운전면허번호, 외국인등록번호), 비밀번호, 바이오정보
◦암호화 기준
구 분
암 호 화 기 준
송․수신시
모든 정보 암호화 전송
저장시
비밀번호,
바이오정보
비밀번호는 단방향 암호화,
바이오정보는 양방향 암호화
고유식별정보
ⓛ 인터넷 구간,공통(DMZ) 구간 저장시 : 의무적 암호화
보안 요구사항
상세
설명
정의
개인정보의 안전성 확보를 위한 기술적 조치사항
세부
내용
 개인정보를 처리함에 있어 『개인정보 보호법』, 『개인정보의 안전성 확보조치 기준』 등 관련 법규 및 규정을 준수하여야 함
 접근 권한 관리
- 개인정보처리시스템에 접속할 수 있는 사용자계정을 발급하는 경우, 개인정보취급자 별로 사용자계정을 발급하여야 하며, 취급자의 역할에 따라 차등부여할 수 있어야 함
- 개인정보 처리내역에 대한 책임 추적성을 확보하기 위하여 접근권한 부여, 변경, 말소에 대한 내역을 이력 관리 또는 조회가 가능하여야 하며, 해당 기록을 최소 3년간 보관할 수 있도록 구현해야 함
예) 신청자 정보, 신청일시, 승인자 및 발급자 정보, 신청 및 발급 사유 등
- 개인정보처리시스템에 접근하여 개인정보를 생성수정삭제열람 등을 하고자 하는 경우 다음 사항을 포함하는 정보이용 내역을 기록하고 남길 수 있도록 시스템을 구현하여야 함.


---

### openai answer
retriever A 점수: 2점
(접속기록 관리, 접근권한 이력 보관, 암호화 등 기술·관리적 보호조치 내용은 포함하나 ‘내부관리계획 수립’이 명시적으로 제시되지 않아 일부만 충족)

retriever B 점수: 3점
(‘내부관리계획(제3조)’이 명시되고, 접근권한 기록 및 정보이용 내역 등 접속기록 관리가 포함되어 있어 요구되는 기술적·관리적 보호조치가 모두 제시됨) 

============================================================
### query:
2025 구미 아시아육상경기선수권대회의 개최 기간은 언제인가?

### answer:
2025년 5월 27일부터 5월 31일까지 총 5일간 개최된다.

---

### retriever A:
종합정보시스템 및 홈페이지 등 구축 용역 제안요청서
1
대회개요
아시아의 꿈, 구미에서 세계로
 The dream of ASIA, From GUMI to the World 
비전
ㅇ (대 회 명) 2025 구미아시아육상경기선수권대회
ㅇ (기 간) 2025. 5. 27.(화) ~ 5. 31.(토) 5일간
ㅇ (장 소) 구미시민운동장 일원
ㅇ (규 모) 아시아 45개국 1,200여명 / 선수단 및 임원
ㅇ (주 최) 아시아육상연맹(Asian Athletics, AA)
ㅇ (주 관) 2025 구미아시아육상경기선수권대회 조직위원회
ㅇ (경기종목) 45개 종목
구 분
종 목 명
남자
(22)
100m, 200m, 400m, 800m, 1,500m, 5,000m,
10,000m, 110mH, 400mH, 3,000m장애물달리기,
20km경보, 4×100mR, 4×400mR, 높이뛰기,
장대높이뛰기, 멀리뛰기, 세단뛰기, 포환던지기,
원반던지기, 해머던지기, 창던지기, 10종 경기
여자
(22)
100m, 200m, 400m, 800m, 1,500m, 5,000m,
10,000m, 100mH, 400mH, 3,000m장애물달리기,
20km경보, 4×100mR, 4×400mR, 높이뛰기,
장대높이뛰기, 멀리뛰기, 세단뛰기, 포환던지기,
원반던지기, 해머던지기, 창던지기, 7종 경기
남·여
(1)
4×400m 믹스릴레이
요
2
과업개요
ㅇ (사 업 명) 2025 구미아시아육상경기선수권대회 종합정보시스템 및 홈페이지 등 구축 용역
ㅇ (사업기간) 계약일 ~ 2025. 6. 6.(금)
ㅇ (사업위치) 시민운동장 일원
2024. . .
대표자 : (인)
대리인 : (인)
2025 구미아시아육상경기선수권대회 조직위원장 귀하
※ 첨부서류 : 재직증명서, 신분증
※ 위임장에 사용하는 도장은 대표자 인감과 같아야 함.
※ 대리 접수로 인하여 발생하는 모든 문제는 대표자 본인이 책임짐
【 서식 3 】
재 직 증 명 서
성 명
생 년 월 일
부서/직위
소 속
주 소
재직기간
위와 같이 당사에 재직 중임을 증명합니다.
2024. . .
ྠĀ ྠĀ ྠĀ ྠĀ ྠĀ ྠĀ 업 체 명 :
ྠĀ ྠĀ ྠĀ ྠĀ ྠĀ ྠĀ 주 소 :
ྠĀ ྠĀ ྠĀ ྠĀ ྠĀ ྠĀ 대 표 자 : (인)
2025 구미아시아육상경기선수권대회 조직위원장 귀하
【 서식 4 】
사 용 인 감 계
사용인감
는(은) 2025 구미아시아육상경기선수권대회 조직위원회와 거래함에 있어 입찰, 계약 및 이에 수반되는 모든 행위에 대하여 위의 인감을 사용하고자 하며, 동 인감을 사용함으로써 민․형사상의 문제가 발생할 경우 이(가) 책임을 질 것을 확인합니다.
붙임 : 인감(법인)증명서 1부. 끝.
2024년 월 일
ྠĀ ྠĀ ྠĀ ྠĀ ྠĀ ྠĀ 주 소 :
ྠĀ ྠĀ ྠĀ ྠĀ ྠĀ ྠĀ 상 호 :
ྠĀ ྠĀ ྠĀ ྠĀ ྠĀ ྠĀ 대표자 : (인)
2025 구미아시아육상경기선수권대회 조직위원장 귀하
【 서식 5 】
서 약 서
당사는 2025 구미아시아육상경기선수권대회 종합정보시스템 및 홈페이지 등 구축 용역 참가신청과 관련하여 모든 서류와 증빙자료를 신의와 성실의 원칙에 입각하여 작성 제출하며, 아래 내용을 준수할 것을 확약하고 이에 서약서를 제출합니다.


---

### retriever B:
종합정보시스템 및 홈페이지 등 구축 용역 제안요청서
1
대회개요
아시아의 꿈, 구미에서 세계로
 The dream of ASIA, From GUMI to the World 
비전
ㅇ (대 회 명) 2025 구미아시아육상경기선수권대회
ㅇ (기 간) 2025. 5. 27.(화) ~ 5. 31.(토) 5일간
ㅇ (장 소) 구미시민운동장 일원
ㅇ (규 모) 아시아 45개국 1,200여명 / 선수단 및 임원
ㅇ (주 최) 아시아육상연맹(Asian Athletics, AA)
ㅇ (주 관) 2025 구미아시아육상경기선수권대회 조직위원회
ㅇ (경기종목) 45개 종목
구 분
종 목 명
남자
(22)
100m, 200m, 400m, 800m, 1,500m, 5,000m,
10,000m, 110mH, 400mH, 3,000m장애물달리기,
20km경보, 4×100mR, 4×400mR, 높이뛰기,
장대높이뛰기, 멀리뛰기, 세단뛰기, 포환던지기,
원반던지기, 해머던지기, 창던지기, 10종 경기
여자
(22)
100m, 200m, 400m, 800m, 1,500m, 5,000m,
10,000m, 100mH, 400mH, 3,000m장애물달리기,
20km경보, 4×100mR, 4×400mR, 높이뛰기,
장대높이뛰기, 멀리뛰기, 세단뛰기, 포환던지기,
원반던지기, 해머던지기, 창던지기, 7종 경기
남·여
(1)
4×400m 믹스릴레이
요
2
과업개요
ㅇ (사 업 명) 2025 구미아시아육상경기선수권대회 종합정보시스템 및 홈페이지 등 구축 용역
ㅇ (사업기간) 계약일 ~ 2025. 6. 6.(금)
ㅇ (사업위치) 시민운동장 일원
1. 기본정보
사업명
2025 구미아시아육상경기선수권대회 종합정보시스템 및 홈페이지 등 구축 용역
영향평가단계
□ 예산편성
□ 그 외 필요시
■ 사업발주
□ 재평가
주요 내용
대회 기본 자료 관리 및 대회 계측기 연동 등
사업기간
(또는 개발기간)
계약일 ~ 2025년 6월
구분
붙임 : 산출내역서 각 1부
2024. . .
제안업체 인(사용인감)
2025 구미아시아육상경기선수권대회 조직위원장 귀하
※ 제안금액은 반드시 한글로 띄어쓰기 없이 붙여 쓰고 아라비아 숫자는 괄호 안에 기재할 것.
【 서식 9-1 】
산 출 내 역 서
구 분
항 목
금 액
구 성 비
비 고
순
용
역
원
가
소 계
소 계
소 계
소 계
일반관리비 ( )%
이 윤 ( )%
총 원 가
부 가 가 치 세
총 용 역 비
※ 작성내용에 따라 줄 수 증감가능
【 서식 10 】
접수번호
2025 구미아시아육상경기선수권대회 종합정보시스템 및 홈페이지 등 구축 용역
정량적 평가 제안서
2024년 월 일
업 체 명
※ 주의사항


---

### openai answer
retriever A 점수: 3점
질문한 개최 기간이 “2025. 5. 27.(화) ~ 5. 31.(토) 5일간”으로 정확히 포함되어 있어 정답을 모두 담고 있음.

retriever B 점수: 3점
동일하게 개최 기간 “2025. 5. 27.(화) ~ 5. 31.(토) 5일간”이 명시되어 있어 정답을 모두 포함함. 

============================================================
### query:
구미 아시아육상경기선수권대회 홈페이지에서 제공되는 주요 콘텐츠 항목에는 무엇이 포함되는가?

### answer:
대회소개, 경기일정 및 결과, 메달순위, 참가선수 정보, 커뮤니티 등이 포함된다.

---

### retriever A:
-경기참가자(Start List), 경기결과(Official Result, Summary) PDF 파일 제공
<다음 정보를 홈페이지에 전송>
Level1
Level2
기능 설명
경기
일정
전체 경기 일정
대회기간 전체 일정 제공
일자별 경기 일정
일자별 세부 경기 일정 제공
세부 종목별 경기 일정
세부 종목별 경기 일정 제공
선수별 경기 일정
선수별 경기 일정 제공
메달
국가/선수별 메달 현황
국가/선수별 메달 현황 제공
종목별 메달 현황
종목별 메달현황 및 메달수상자 정보 제공
경기
정보
스타트 리스트
세부종목별 편성 조 및 출전선수 목록 제공
대진 정보
세부 종목별 대진정보 조회 및 출력
(경기진행 상태에 따른 결과 반영 포함)
경기 결과
각 조별, 종목별 경기 결과 조회 및 출력
선수
정보
국가별 선수 정보
국가별 참가선수 목록 및 정보 조회
세부 종목별 선수 정보
세부 종목별 참가선수 목록 및 정보 조회
순위 정보
세부종목별 참가선수 순위 정보 목록 조회
과업내용
명칭
대분류
대회 종합정보시스템 S/W 개발
소분류
보고서관리시스템 개발
과업내용
데이터 출력 기능
<PDF 및 Excel, 증서 페이지 등 출력>
-참가선수 신상정보
-국가별/세부 종목별 참가선수 리스트
-심판/임원 신상정보
-국가별/세부 종목별 심판/임원 리스트
-세부 종목별, 예결선 Start List 및 조 편성
-세부 종목별 경기 결과 (Live Result, Official Result, Summary)
-세부 종목별 메달 현황 및 신기록 현황
-참가증서
ː참가선수(종목별 1위~6위 증서 및 참가증서 필수)
 운영 매뉴얼 또는 지침서 제공 및 시스템 운영 관계자 교육
ㅇ (현장IT팀 상주 및 시스템 운영)
 사전 등록, 경기 진행 및 결과 출력 시스템 등 운영
 대회시작 4일전~대회 종료일까지 경기장 상주
 경기장/비경기장별 운영 인원 배치
ㅇ (홈페이지 구축)
 메뉴 구성도(안)
아시아육상경기선수권대회 메인
대회소개
소식지
경기일정/결과
메달순위
육상경기
커뮤니티
환영인사
공지사항
경기일정
국가별순위
경기종목
안내사항
대회개요
보도자료
행사일정
선수별순위
참가선수
아카이브
대회상징물
사진자료
시설안내
유실물센터
조직위안내
자주묻는질문
홍보영상
자원봉사자
찾아오시는 길
구미시 안내
메뉴 상세설명(안)
메뉴명
부메뉴
주요 내용
메인 화면
◦2025 구미 아시아육상경기선수권대회를 대표할 수 있는 디자인 (공식 상징물 활용 등)
◦활용빈도가 높은 메뉴 배치 및 팝업배너
대회 소개
환영인사
◦대회 개최에 대한 인사 말씀
대회개요
◦대회명칭, 대회기간, 대회장소, 참가규모, 참가국, 참가인원, 종목, 주최, 주관 등
대회상징물
◦대회마크, 마스코트, 포스터, 슬로건
조직위안내
◦조직위원회·아시아육상추진단 조직도
홍보영상
◦홍보영상, 유튜브 쇼츠 영상, 축하 메시지 등
찾아오시는 길
◦경기장 위치, 교통수단별 길 안내
소식지
공지사항
◦대회 관련 공지사항 등
보도자료
◦대회 관련기사 스크랩 자료
사진자료
◦개·폐회식 사진, 각 종목별 경기 이미지 게시
경기일정
/
결과
경기일정
◦경기장 및 개최 일자, 시간, 라운드, 진행상태
◦각 종목별 경기결과, 종합순위
행사일정
◦개·폐회식 정보 및 식순 안내


---

### retriever B:
 운영 매뉴얼 또는 지침서 제공 및 시스템 운영 관계자 교육
ㅇ (현장IT팀 상주 및 시스템 운영)
 사전 등록, 경기 진행 및 결과 출력 시스템 등 운영
 대회시작 4일전~대회 종료일까지 경기장 상주
 경기장/비경기장별 운영 인원 배치
ㅇ (홈페이지 구축)
 메뉴 구성도(안)
아시아육상경기선수권대회 메인
대회소개
소식지
경기일정/결과
메달순위
육상경기
커뮤니티
환영인사
공지사항
경기일정
국가별순위
경기종목
안내사항
대회개요
보도자료
행사일정
선수별순위
참가선수
아카이브
대회상징물
사진자료
시설안내
유실물센터
조직위안내
자주묻는질문
홍보영상
자원봉사자
찾아오시는 길
구미시 안내
메뉴 상세설명(안)
메뉴명
부메뉴
주요 내용
메인 화면
◦2025 구미 아시아육상경기선수권대회를 대표할 수 있는 디자인 (공식 상징물 활용 등)
◦활용빈도가 높은 메뉴 배치 및 팝업배너
대회 소개
환영인사
◦대회 개최에 대한 인사 말씀
대회개요
◦대회명칭, 대회기간, 대회장소, 참가규모, 참가국, 참가인원, 종목, 주최, 주관 등
대회상징물
◦대회마크, 마스코트, 포스터, 슬로건
조직위안내
◦조직위원회·아시아육상추진단 조직도
홍보영상
◦홍보영상, 유튜브 쇼츠 영상, 축하 메시지 등
찾아오시는 길
◦경기장 위치, 교통수단별 길 안내
소식지
공지사항
◦대회 관련 공지사항 등
보도자료
◦대회 관련기사 스크랩 자료
사진자료
◦개·폐회식 사진, 각 종목별 경기 이미지 게시
경기일정
/
결과
경기일정
◦경기장 및 개최 일자, 시간, 라운드, 진행상태
◦각 종목별 경기결과, 종합순위
행사일정
◦개·폐회식 정보 및 식순 안내
종합정보시스템 및 홈페이지 등 구축 용역 제안요청서
1
대회개요
아시아의 꿈, 구미에서 세계로
 The dream of ASIA, From GUMI to the World 
비전
ㅇ (대 회 명) 2025 구미아시아육상경기선수권대회
ㅇ (기 간) 2025. 5. 27.(화) ~ 5. 31.(토) 5일간
ㅇ (장 소) 구미시민운동장 일원
ㅇ (규 모) 아시아 45개국 1,200여명 / 선수단 및 임원
ㅇ (주 최) 아시아육상연맹(Asian Athletics, AA)
ㅇ (주 관) 2025 구미아시아육상경기선수권대회 조직위원회
ㅇ (경기종목) 45개 종목
구 분
종 목 명
남자
(22)
100m, 200m, 400m, 800m, 1,500m, 5,000m,
10,000m, 110mH, 400mH, 3,000m장애물달리기,
20km경보, 4×100mR, 4×400mR, 높이뛰기,
장대높이뛰기, 멀리뛰기, 세단뛰기, 포환던지기,
원반던지기, 해머던지기, 창던지기, 10종 경기
여자
(22)
100m, 200m, 400m, 800m, 1,500m, 5,000m,
10,000m, 100mH, 400mH, 3,000m장애물달리기,
20km경보, 4×100mR, 4×400mR, 높이뛰기,
장대높이뛰기, 멀리뛰기, 세단뛰기, 포환던지기,
원반던지기, 해머던지기, 창던지기, 7종 경기
남·여
(1)
4×400m 믹스릴레이
요
2
과업개요
ㅇ (사 업 명) 2025 구미아시아육상경기선수권대회 종합정보시스템 및 홈페이지 등 구축 용역
ㅇ (사업기간) 계약일 ~ 2025. 6. 6.(금)
ㅇ (사업위치) 시민운동장 일원


---

### openai answer
retriever A 점수: 3점
(메뉴 구성도에 대회소개, 경기일정/결과, 메달순위, 커뮤니티, 참가선수(선수 정보) 등이 명시되어 있어 answer의 모든 항목을 포함)

retriever B 점수: 3점
(메뉴 구성도에 대회소개, 경기일정/결과, 메달순위, 커뮤니티, 참가선수 등이 포함되어 있어 answer의 모든 항목을 충족) 

============================================================
### query:
원자력 선량평가시스템의 성능 요구사항 중 질의응답 시간 기준은 어떻게 정의되어 있는가?

### answer:
일반 조회 요청은 3초 이내 처리되어야 하며, 10초 이상 소요되는 작업은 사용자에게 팝업 메시지를 제공해야 한다.

---

### retriever A:
질의·응답시간
요구사항
상세설명
정의
질의·응답시간
세부
내용
의는 사용자가 요청을 하는 시간으로부터 3000ms 내에 그 결과를 보여줘야 함
❍ 임의의 선택 기준이 허용되는 대량의 데이터에 대한 질의 및 고용량의 멀티미디어 파일을 가지고 있는 프로그램에서는 적절한 처리 상태를 표시하여야 함
산출정보
성능측정 결과 내역서
관련 요구사항
요구사항 고유번호
PER-002
요구사항 명
오류 응답시간
요구사항
상세설명
정의
오류 응답시간
세부
내용
시스템에 데이터 입력 등의 사용 중에 발생한 모든 오류는 원인이 발생한지 3000ms 이하로 사용자에게 오류 메시지를 보여줘야 함
산출정보
성능측정 결과 내역서
관련 요구사항
요구사항 고유번호
PER-003
요구사항 명
통계 처리 사전 경고
요구사항
상세설명
정의
통계 처리 사전 경고
세부
내용
리 시 5,000ms 이상 지연되는 경우 연산 작업을 시작하는 시점에 예상되는 결과에 대해 사용자에게 메시지로 알려야 함
산출정보
성능측정 결과 내역서
관련 요구사항
요구사항 고유번호
PER-004
요구사항 명
부하증가 대비 성능유지
요구사항
상세설명
정의
부하증가 대비 성능유지
세부
내용
시스템의 경우 동시 접속자(사용자)는 500명 이상 접속가능 하여야 하며, 이에 따른 처리속도 등 성능이 저하되지 않아야 함
❍ 포털 시스템의 경우 동시 접속자(사용자)는 3000명 이상 접속가능 하여야 하며, 이에 따른 처리속도 등 성능이 저하되지 않아야 함
산출정보
성능측정 결과 내역서
관련 요구사항
요구사항 고유번호
PER-005
요구사항 명
오프라인 배치성 업무 응답시간
요구사항
나. 성능 요구사항 (Performance Requirement)
요구사항분류
PER-001
요구사항명
응답시간
요구사항 내용
정의
개선 대상 웹사이트 및 콘텐츠 관리 시스템(CMS) 정보 조회에 대한 응답시간 성능 보장
세부 내용
○ 정보 처리량 및 확장성 등을 고려하여 웹페이지 디스플레이 시간 기능요청에 대한 응답시간이 최소화 되도록 함
- 단순 웹페이지의 경우, 사용자가 요청을 하는 시간으로부터 3초 내에 그 결과를 보여줘야 함
- 시스템이 만드는 각 페이지의 경우, 요청한 시간으로부터 5초 내에 디스플레이 되어야 함
- 이 요구사항은 시스템을 사용하는 동시 사용자가 최대 사용자 수의 90%를 초과하는 질의에는 적용되지 않음
○ 과도한 데이터베이스 접속을 자제하며 응답속도 개선(상영시간표 등)
- 이 요구사항은 임의의 선택 기준이 허용되는 대량의 데이터(데이터량 2GB 이상)에 대한 질의에는 적용되지 않음
- 이 요구사항은 임의의 선택 기준이 허용되는 대량의 데이터에 대한 요청 및 플래시가 있을 경우에는 적용되지 않으며, 한 개 이상의 큰 이미지(이미지 500KB 이상) 및 동영상을 가지고 있는 페이지에는 적용되지 않음
- 동영상의 경우 트래픽 관리를 위해 외부 플랫폼(예: 유튜브, 기타 CDN 서비스)과 연계하여 기능 구현
○ 기능 구현에 있어 10초 이상 걸릴 수 있는 작업에 대해서는 시간지연에 대한 정보를 사용자에게 팝업 등의 방법으로 사용자에게 알려야 함
요구사항분류
PER-002
요구사항명
데이터 형식 오류 응답 시간
요구사항 내용
정의
사용자 입력 오류 응답 시간 제시
세부 내용


---

### retriever B:
성능 요구사항
상세
설명
정의
정보시스템 업무 응답시간 목표
세부
내용
□ 요청(질의) 응답시간
o 모든 질의는 사용자가 요청을 하는 시간으로부터 3초 내에 최초의 결과 값을 보여줘야 함
□ 이 요구사항은 임의의 선택 기준이 허용되는 대량의 데이터(2GB 이상)에 대한 질의에는 적용되지 않음
산출정보
o SW개발에 따른 산출물
성능 요구사항
상세
설명
정의
정보시스템 업무 응답시간 목표
세부
내용
□ 요청(질의) 응답시간
o 모든 질의는 사용자가 요청을 하는 시간으로부터 3초 내에 최초의 결과 값을 보여줘야 함
□ 이 요구사항은 임의의 선택 기준이 허용되는 대량의 데이터(2GB 이상)에 대한 질의에는 적용되지 않음
산출정보
o SW개발에 따른 산출물
성능 요구사항
응낙수준
필수
요구
사항
상세
설명
정의
사용자 요구에 대한 웹페이지 디스플레이 반응 시간
세부
내용
○ 시스템은 정상 상태에서 사용자의 질의 요청에 대한 결과 페이지를 화면에 출력할 때 3초 이내 보장
○ 평균 응답시간은 시스템을 사용하는 동시 사용자가 최대 사용자 수의 90%를 초과하는 경우에는 미적용
산출정보
관련요구사항
요구사항 내용
○ 모든 질의는 사용자가 요청하는 시간으로부터 3초 이내에 그 결과값을 보여줘야 함
○ 평균 응답 시간은 시스템을 사용하는 동시 사용자가 최대 사용자 수의 90%를 초과하는 경우 미적용함
○ 한계 인원 이상 접속 시 해결 방안 제시해야 함
② 성능 요구사항
PER-001
평균응답시간 및 성능 최적화
요구사항
상세설명
정의
업무응답시간에 목표 대한 기준
세부내용
○ 모든 질의는 평균 응답시간으로 사용자가 요청한 시간으로부터 3초 내에
그 결과를 보여줘야 함
○ 대량의 데이터 로딩(통계, 검색, 이미지 등), 사용자 숫자가 동시 사용자 용량의 90%를 초과하는 경우에는 적용되지 않음
산출 정보
개발 산출물, 테스트 결과서
관련 요구사항
성능 요구사항
상세
설명
정의
평균 응답시간
세부
내용
❍ 대상시스템은 정상 상태에서 사용자의 질의 요청에 대한 결과
페이지를 화면에 출력할 때 3초 이내에 완전히 출력되어야 함


---

### openai answer
retriever A 점수: 3점
- 3초 이내 응답 기준(3000ms)과 “10초 이상 소요 작업은 팝업 등으로 지연 안내” 요구사항이 모두 명시되어 있음.

retriever B 점수: 2점
- 3초 이내 응답 기준은 포함되어 있으나, 10초 이상 소요 시 팝업 안내 요구사항은 없음. 

============================================================
### query:
원자력 선량평가시스템의 포트란 모듈 개선 요구사항에서 추가로 고려해야 하는 평가 요소는 무엇인가?

### answer:
 ICRP60 기준에 따른 23개 장기를 고려한 평가장기 추가가 요구된다.

---

### retriever A:
가 방법
평가부문
평가항목
평가기준
배점
한도
정량
평가
(10)
경영상태
경영상태(신용상태등급) 평가
10
전략 및 방법론
(22)
사업
이해도
사업의 특성 및 목표에 부합하는 사업전략을 제시하고 있는지를 평가한다.
7
추진전략
개발업무 수행 시 일정 및 위험요소를 고려하여 얼마나 타당한 추진 전략을 수립하였는지 평가한다.
5
적용기술
사업에서 적용하고자 하는 기술이 향후 확장성을 고려하여 현실적으로 실현 가능하게 제시되어 있는지를 평가한다.
4
표준프레임 워크 적용
제안사는 사업에 네팔 정보통신기술부 프레임워크 기반의 공통 컴포넌트의 유형 및 사용계획을 기술하고 사용성 향상을 위한 추가 컴포넌트에 대해 적용방안을 제시하여야한다.
3
개발
방법론
사업에 적정한 개발 방법론이 제시되었는지 평가하고, 실제 적용 사례 및 경험을 바탕으로 단계별 산출물이 제시되었는지 여부를 평가한다.
3
기술 및 기능
(26)
시스템 장비 구성 요구사항
요구 규격을 충족하는 장비 제안 여부와 현 시스템과의 인터페이스 및 확장 가능성 여부를 평가한다. 또한 도입 장비의 설치 및 공급 계획, 유지관리에 대한 방안이 구체적으로 기술되어 있는가를 평가한다.
7
기능 요구사항
기능 요구사항·기대사항·제약사항 등을 파악한 후 구현 방안이 구체적으로 기술되어 있는지를 평가한다. 또한 제안한 방안 및 기술이 적용 가능한지를 평가한다.
7
데이터
요구사항
데이터 요구사항 분석을 바탕으로 데이터 전환계획 및 검증 방법, 데이터 오류 발생 시 처리 방안이 구체적으로 제시되어 있는지를 평가한다.
4
보안
요구사항
기능 보안
체적이고 적절한지 평가
(30점) 요구사항
- 구현방안이 설계단계부터 구현 및 검증단계까지 고려하고 있는지 평가
- 데이터 전환 계획 및 검증 방법, 에러 데이터 처리 방법에 대해 구체적
데이터
인 내용을 제시하고 있는지 평가 15
요구사항
- 데이터 전환을 위해 책임 조직이 투입되는지 평가
- 기능 및 품질 등 요구사항을 구현 시 관련 제약사항을 충족시키며 구현
제약사항
방안 및 테스트 방안을 구체적으로 기술하였는지 평가
-237-
평가
평가항목 평가기준 배점
부문
- 구현하고자 하는 기능에 요구 성능이 충족되도록 방법론 및 분석 도구
성능 를 통하여 내용을 구체적으로 분석하였는지, 구현 및 테스트 방안을 구
요구사항 체적으로 기술하였는지 평가
- 제안한 방안 및 기술을 통해 요구 성능을 충족시킬 수 있는지 평가
- 분석 도구 및 구현 방안, 테스트 방안을 구체적으로 기술하였는지, 분석
·설계 등 각 단계별 품질 요구사항의 점검 및 검토 방안을 구체적으로
품질
계획하였는지 평가
요구사항
- 각 단계마다 품질 요구사항을 점검하는 별도의 전문 인력이 투입되는지
성능 및 평가
품질 - (시스템 인터페이스) 타 시스템과의 연계 방안들에 대한 장·단점의 분석 10
(10점) 을 통해 가장 적합한 방안을 구체적으로 기술했는지, 구현 경험이 있는
인터페이
인터페이스 담당자가 투입되는지 평가
스
- (사용자 인터페이스) 사용자 편의성을 고려하여 요구사항을 제공하기 위
요구사항
한 분석 및 설계, 구현 방안과 검토 계획을 구체적으로 기술하였는지,
구현 경험이 있는 사용자 인터페이스 담당자가 투입되는지 평가


---

### retriever B:
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항
기능 요구사항


---

### openai answer
retriever A 점수: 0점
- 원자력 선량평가, 포트란 모듈, ICRP60, 23개 장기 등 핵심 키워드나 내용이 전혀 없고, 일반적 평가기준만 포함됨.

retriever B 점수: 0점
- “기능 요구사항” 반복 외에 내용이 없으며, ICRP60 기준이나 23개 장기 등 정답과 관련된 정보가 전혀 없음. 


## Query 2

## sim_results

============================================================
### query:
본 과업지시서에서 '극저온시스템'은 무엇을 포함하는 것으로 정의하고 있습니까?

### answer:
'극저온시스템'은 SCL3/SCL2 극저온플랜트 및 부대설비, SCL3/SCL2 분배시스템 및 부대설비, SCL3/SCL2 초전도가속관 저온유지모듈, 기체헬륨 탱크, 액체질소 시스템 및 극저온시스템용 헬륨회수시스템을 모두 포함하는 것으로 정의됩니다.

---

### retriever A:
7. 안전관리 .............................................................................................................. 19
8. 붙임 자료 ........................................................................................................... 19
문서번호 : -
2025년도 중이온가속기용 극저온시스템
개정번호 : 0
운전 용역 과업지시서 발 행 일 : 2024. 10. 30
페 이 지 : 5/19
1. 서 론
1.1 목 적
기초과학연구원 중이온가속기연구소(이하 “연구원”)에서 운영중인 중이온가속기용 극저
온시스템(이하 “극저온시스템”)은 고도의 공정 시스템으로, 안전하고 안정적인 중이온가속
기의 운영을 위해 상시 모니터링이 필요하다. 이에 본 과업지시서는 극저온시스템의 운전
용역에 대한 상세 과업 내용을 기술한다.
1.2 용어 설명
본 과업지시서에서 사용하는 용어에 대한 상세 내용은 표 1.과 같다. 계약자는 반드시 해당
용어의 의미를 충분히 이해하고 숙지해야 하며, 용어 미숙지로 인한 책임은 계약자에게 있
다.
표 1. 본 과업지시서에 사용되는 용어 설명
용어 설명
연구원 중이온가속기연구소
계약자 본 과업을 수행할 계약상대자
SCL3/SCL2 극저온플랜트 및 부대설비,
SCL3/SCL2 분배시스템 및 부대설비,
SCL3/SCL2 초전도가속관 저온유지모듈,
극저온시스템
기체헬륨 탱크, 액체질소 시스템 및
TBx 극저온설비동
밸브박스, 초전도가속관 저온유지
SCL3 터널/
극저온
SCL3 LINAC 모듈 및 밸브박스 진공단열용 진
ISOL 구간
분배
공펌프
시스템 밸브박스, 초전도가속관 저온유지
SCL2 터널/
SCL2 LINAC 모듈 및 밸브박스 진공단열용 진
IF 구간
공펌프
외부 정화기 외부
회수 압축시스템 헬륨압축기동
회수
가스백 헬륨압축기동
시스템
헬륨 탱크 외부 기체 헬륨량 관리 포함
액체 질소 시스템 외부 액체 질소량 관리 포함
3.3 중이온가속기 또는 극저온시스템 가동기간의 운전 업무
계약자는 중이온가속기 또는 극저온시스템 가동기간 동안 24시간 운전업무가 이루어질 수
있도록 운전 인력을 제공해야 한다. 24시간 운전 업무는 2인 1조를 기본으로 하며, 주간에
는 과업 및 운전 인력 감독자가 1인 이상 운전실에 존재해야 한다. 계약자가 가동기간에 수
행해야 하는 운전 업무 상세 내용은 하위 절에 기술된 내용을 따른다. 본 과업지시서에 기
술하지 않은 가동기간의 운전 업무는 연구원과 계약자가 협의하여 수행 여부를 결정한다.
3.3.1 극저온시스템 모니터링
계약자는 가동기간 동안 극저온시스템이 안정적으로 가동될 수 있도록 모니터링을 해야 한
다. 가동기간의 모니터링 업무는 극저온시스템이 안정적으로 가동되고 있는지 여부를 확인
하기 위해 극저온시스템을 지속적으로 관찰하는 업무를 의미하며, 표 4.에 가동기간의 모니
터링 업무에 해당되는 내용들을 기술하였다.
극저온시스템의 모니터링은 연구원에 구축되어있는 극저온시스템용 제어시스템(이하 “제


---

### retriever B:
1. 서 론
1.1 목 적
기초과학연구원 중이온가속기연구소(이하 “연구원”)에서 운영중인 중이온가속기용 극저
온시스템(이하 “극저온시스템”)은 고도의 공정 시스템으로, 안전하고 안정적인 중이온가속
기의 운영을 위해 상시 모니터링이 필요하다. 이에 본 과업지시서는 극저온시스템의 운전
용역에 대한 상세 과업 내용을 기술한다.
1.2 용어 설명
본 과업지시서에서 사용하는 용어에 대한 상세 내용은 표 1.과 같다. 계약자는 반드시 해당
용어의 의미를 충분히 이해하고 숙지해야 하며, 용어 미숙지로 인한 책임은 계약자에게 있
다.
표 1. 본 과업지시서에 사용되는 용어 설명
용어 설명
연구원 중이온가속기연구소
계약자 본 과업을 수행할 계약상대자
SCL3/SCL2 극저온플랜트 및 부대설비,
SCL3/SCL2 분배시스템 및 부대설비,
SCL3/SCL2 초전도가속관 저온유지모듈,
극저온시스템
기체헬륨 탱크, 액체질소 시스템 및
극저온시스템용 헬륨회수시스템 포함
(“3.2절 운전대상”, 표 3. 참고)
중이온가속기 또는 극저온시스템이 냉각(냉각 착수
가동기간
이후) 또는 빔서비스 등을 위해 가동 중인 기간
중이온가속기 또는 극저온시스템이 유지보수 등을
휴지기간
위해 가동을 멈추고 휴지 중인 기간
중이온가속기용 극저온시스템이 가동기간 및 휴지
기간에 안전하고 안정적으로 운영될 수 있도록,
중이온가속기용 극저온시스템 i) 극저온시스템을 모니터링 및 제어하고,
운전 업무 ii) 극저온시스템의 주요 수치를 기록 및 분석하며,
iii) 중이온가속기용 극저온시스템의 운전 경험을
통합하여 정리(문서화)하는 업무
문서번호 : -
문서번호 : -
2025년도 중이온가속기용 극저온시스템
개정번호 : 0
운전 용역 과업지시서 발 행 일 : 2024. 10. 30
페 이 지 : 12/19
매뉴얼을 작성하고 개정하는 업무를 수행해야 한다. 또한, 과업 및 운전 인력 감독자가 1인
이상 운전실에 배치되어 상기 업무들을 감독해야 한다. 계약자가 휴지기간에 수행해야 하는
운전 업무 상세 내용은 하위 절에 기술된 내용을 따른다. 본 과업지시서에 기술하지 않은
휴지기간의 운전 업무는 연구원과 계약자가 협의하여 수행 여부를 결정한다.
3.4.1 유지보수 등을 위한 극저온시스템 제어
극저온시스템의 제어는 극저온시스템이 휴지지간에 안전하고 안정적으로 운영될 수 있도록
극저온시스템 또는 극저온시스템 관련 부품들을 제어시스템을 활용하여 또는 현장에서 손으
로 조작하는 행위를 의미한다. 계약자는 극저온시스템이 본 목적에 부합하도록 극저온시스
템의 제어를 수행해야 한다. 표 8.에 극저온시스템 제어 업무에 해당되는 사항들을 기술하
였다.
휴지기간의 극저온시스템 제어는 i) 극저온시스템에 대한 충분한 이해를 바탕으로 ii) 조작
후 유지보수 등 현장 활동에 미치는 영향을 충분히 검토한 뒤 이루어져야 한다. 더불어, iii)
조작 대상물이 위치하고 있는 현장에 인력이 존재하는지 여부를 확인하고, iv) 현장에 인력
이 존재할 경우, 조작 대상물을 조작했을 때 발생할 수 있는 상황에 대해 해당 인력의 안전
이 확보된 뒤에 조작을 수행해야 한다. 마지막으로 v) 조작 대상/조작 상태에 대한 혼동이
없도록 각별히 주의를 하여 제어가 진행되어야 한다.


---

### openai answer
retriever A 점수: 2점
일부 정의 항목(SCL3/SCL2 극저온플랜트·분배시스템·저온유지모듈, 기체헬륨 탱크, 액체질소 시스템)은 포함하나, ‘극저온시스템용 헬륨회수시스템’이 정의 문맥에서 명시되지 않아 전체 답을 모두 충족하지 못함.

retriever B 점수: 3점
표 1의 정의에 ‘SCL3/SCL2 극저온플랜트 및 부대설비, 분배시스템 및 부대설비, 초전도가속관 저온유지모듈, 기체헬륨 탱크, 액체질소 시스템 및 극저온시스템용 헬륨회수시스템 포함’을 명시적으로 모두 제공함. 

============================================================
### query:
극저온시스템 과업 추진 일정에 따르면, 4분기에 SCL3 구간에 대해 예정된 주요 업무는 무엇이며, 이때의 업무 기간은 '가동기간'과 '휴지기간' 중 어느 것에 해당합니까?

### answer:
2025년 4분기에 SCL3 구간에 예정된 주요 업무는 '유지보수'이며, 이 기간은 '휴지기간'에 해당합니다.

---

### retriever A:
다. 계약자는 본 과업을 수행하기 위해 적절한 지식 또는 능력을 보유하고 있는 적정 수의 운
전 인력을 연구원에게 제공하여야 한다.
• 가동기간의 극저온시스템 운전 업무 (24시간 운전)
- 극저온시스템 모니터링
- 극저온시스템 주요 수치 기록 및 분석
- 극저온시스템 제어
- 기타 가동기간에 필요한 운전 업무
• 휴지기간의 극저온시스템 운전 업무 (주간 운전)
- 유지보수 등을 위한 극저온시스템 제어
- 유지보수 등을 위한 극저온시스템 모니터링
- 유지보수 등을 위한 극저온시스템 주요 수치 기록 및 분석
- 극저온시스템 운전 매뉴얼 작성 및 개정
- 극저온시스템 운전 교육
- 성과보고회 개최
- 기타 휴지기간에 필요한 운전 업무
• 본 과업 및 본 과업을 위해 투입된 인력에 대한 관리감독
본 과업의 기간은 2025년 1월 1일부터 2025년 12월 31일까지로 하며, 2025년 동안 가동기
간의 운전 업무를 수행하는 것을 기준으로 한다. 단, 표 2.와 같이 연구원의 운영 추진 상황
에 따라 휴지기간의 운전업무를 수행하는 것으로 변경될 수 있다. 가동기간과 휴지기간의
운전 업무는 “3. 과업 내용” 하위 절에 기술한 내용에 따른다.
표 2. 과업 추진 일정(안)
2025년
구분
1분기 2분기 3분기 4분기
SCL3 극저온플랜트
SCL3 중이온가속기 SCL3 구간 운전 SCL3 구간 유지보수
시운전 준비 및
구간 (가동기간) (휴지기간)
시운전 (가동기간)
SCL2 SCL2 극저온플랜트 중이온가속기 SCL2 구간 시운전 SCL2 구간 유지보수
구간 시운전 (가동기간) (가동기간) (휴지기간)
회수
2025년
구분
1분기 2분기 3분기 4분기
SCL3 극저온플랜트
SCL3 중이온가속기 SCL3 구간 운전 SCL3 구간 유지보수
시운전 준비 및
구간 (가동기간) (휴지기간)
시운전 (가동기간)
SCL2 SCL2 극저온플랜트 중이온가속기 SCL2 구간 시운전 SCL2 구간 유지보수
구간 시운전 (가동기간) (가동기간) (휴지기간)
회수
필요시 간헐적 운전 (휴지기간에 준함)
시스템
문서번호 : -
2025년도 중이온가속기용 극저온시스템
개정번호 : 0
운전 용역 과업지시서 발 행 일 : 2024. 10. 30
페 이 지 : 7/19
3. 과업 내용
3.1 중이온가속기용 극저온시스템 운전 업무의 정의 및 필요성
본 과업에서 사용하는 “중이온가속기용 극저온시스템 운전 업무”의 정의는, “중이온가
속기용 극저온시스템이 안전하고 안정적으로 운영(가동기간뿐만 아니라 휴지기간의 운영 포
함)될 수 있도록 i) 극저온시스템을 모니터링 및 제어하고, ii) 극저온시스템의 주요 수치를
기록 및 분석하며, iii) 중이온가속기용 극저온시스템의 운전 경험을 통합하여 정리(문서화)하
는 업무를 의미한다.
중이온가속기 또는 극저온시스템 가동기간에는 24시간 운전 업무를 필요로 한다. 통상적으
로 가동기간에는 24시간 운전을 위한 교대근무가 이루어지며, i) 극저온시스템의 모니터링
및 제어, ii) 극저온시스템의 주요 수치를 기록 및 분석하는 업무가 주로 이루어진다.
다만, 통상적으로 이러한 교대근무로 인해 각 운전 인력마다 경험할 수 있는 운전 상황이


---

### retriever B:
다. 계약자는 본 과업을 수행하기 위해 적절한 지식 또는 능력을 보유하고 있는 적정 수의 운
전 인력을 연구원에게 제공하여야 한다.
• 가동기간의 극저온시스템 운전 업무 (24시간 운전)
- 극저온시스템 모니터링
- 극저온시스템 주요 수치 기록 및 분석
- 극저온시스템 제어
- 기타 가동기간에 필요한 운전 업무
• 휴지기간의 극저온시스템 운전 업무 (주간 운전)
- 유지보수 등을 위한 극저온시스템 제어
- 유지보수 등을 위한 극저온시스템 모니터링
- 유지보수 등을 위한 극저온시스템 주요 수치 기록 및 분석
- 극저온시스템 운전 매뉴얼 작성 및 개정
- 극저온시스템 운전 교육
- 성과보고회 개최
- 기타 휴지기간에 필요한 운전 업무
• 본 과업 및 본 과업을 위해 투입된 인력에 대한 관리감독
본 과업의 기간은 2025년 1월 1일부터 2025년 12월 31일까지로 하며, 2025년 동안 가동기
간의 운전 업무를 수행하는 것을 기준으로 한다. 단, 표 2.와 같이 연구원의 운영 추진 상황
에 따라 휴지기간의 운전업무를 수행하는 것으로 변경될 수 있다. 가동기간과 휴지기간의
운전 업무는 “3. 과업 내용” 하위 절에 기술한 내용에 따른다.
표 2. 과업 추진 일정(안)
2025년
구분
1분기 2분기 3분기 4분기
SCL3 극저온플랜트
SCL3 중이온가속기 SCL3 구간 운전 SCL3 구간 유지보수
시운전 준비 및
구간 (가동기간) (휴지기간)
시운전 (가동기간)
SCL2 SCL2 극저온플랜트 중이온가속기 SCL2 구간 시운전 SCL2 구간 유지보수
구간 시운전 (가동기간) (가동기간) (휴지기간)
회수
일정
구분
추진 일정
M
M+1
M+2
M+3
M+4
사업발주 및 사업자 선정
업무 협의
시스템
구축
기존 시스템 분석
업무(요구사항) 분석
설계 및 구축
시스템 구현 및 개발
시스템테스트 및 수정
시스템 가동 및 운영(안정화)
※ 위 추진일정은 사업 추진과정에서 조정 가능함
Ⅲ
다. 추진일정
구 분
2024년~2025년
M1
M2
M3
M4
M5
M6
◦ 시스템 개발
- 분석
- 설계/구현
- 단위/통합 시험
◦ 연계시스템 기능개선
◦ 운영개시 및 안정화
◦ 보고회
- 착수, 중간, 최종보고
★
★
★
* 추진과정에서 일정은 변경될 수 있음
Ⅱ
일반 현황


---

### openai answer
retriever A 점수: 3점
- 표 2의 과업 추진 일정에 2025년 4분기 SCL3 구간 ‘유지보수(휴지기간)’가 명시되어 있어, 답의 두 요소(유지보수, 휴지기간)가 모두 포함됨.

retriever B 점수: 3점
- 동일하게 표 2에 2025년 4분기 SCL3 구간 ‘유지보수(휴지기간)’가 명시되어 있어, 질문의 답을 완전히 뒷받침함. 

============================================================
### query:
극저온시스템 '가동기간' 동안 계약자가 24시간 수행해야 할 극저온시스템 모니터링 업무에는 어떤 것들이 있습니까? (3가지 이상)

### answer:
가동기간 동안 수행해야 할 모니터링 업무는 다음과 같습니다.
알람 메시지 확인
주요 센서(온도, 압력, 유량 등)의 트렌드 확인
정상 운전 상황의 주요 센서 지시값 파악 및 숙지
비정상 운전 상황 감지
제어로직 정상 작동 여부 확인

---

### retriever A:
다. 가동기간의 모니터링 업무는 극저온시스템이 안정적으로 가동되고 있는지 여부를 확인
하기 위해 극저온시스템을 지속적으로 관찰하는 업무를 의미하며, 표 4.에 가동기간의 모니
터링 업무에 해당되는 내용들을 기술하였다.
극저온시스템의 모니터링은 연구원에 구축되어있는 극저온시스템용 제어시스템(이하 “제
어시스템”)을 활용하여 또는 현장에 설치되어있는 장치를 통해 수행한다. 연구원은 구축되
어있는 제어시스템을 활용하기 위한 컴퓨터, 모니터, 키보드, 마우스, 책상 및 의자를 계약
자에게 제공한다. 현재 연구원에 구축되어있는 제어시스템은 표 5.과 같다.
연구원은 계약자에게 현재 구축되어있는 제어시스템에 대한 기본 사용 교육을 제공하며,
계약자는 연구원이 제공하는 교육을 이해하기 위한 배경지식을 보유해야 한다.
문서번호 : -
2025년도 중이온가속기용 극저온시스템
개정번호 : 0
운전 용역 과업지시서 발 행 일 : 2024. 10. 30
페 이 지 : 9/19
표 4. 가동기간의 극저온시스템 모니터링 업무 리스트
순번 극저온시스템 모니터링 업무 비고
1 알람 메시지 확인
2 주요 센서 트렌드 확인 온도, 압력, 유량, 속도 등
3 정상 운전 상황의 주요 센서 지시값 파악 및 숙지 온도, 압력, 유량, 속도 등
4 비정상 운전 상황 감지
연구원에서 현장 확인 필요상황 발생 시, 연구원이 지 무전 통신 활용
5
정하는 값 관찰, 확인 및 상황 전달 (연구원에서 무전기 제공)
6 제어로직 정상 작동 여부 확인
연구원 요청 시,
7 현장 상황 일시적 또는 지속적 관찰
비방사선구역 한정
이온가속기용 극저온시스템 운영에 중대한 영향이 생길 경우, 이는 계약자의 책임으로 한
다.
표 8. 휴지기간의 극저온시스템 제어 업무 리스트
순번 극저온시스템 제어 업무 비고
자동/수동(Auto/Manual) 제어 우선
1 계장류 제어 우선권(Priority) 변경
권 설정 변경
문서번호 : -
2025년도 중이온가속기용 극저온시스템
개정번호 : 0
운전 용역 과업지시서 발 행 일 : 2024. 10. 30
페 이 지 : 13/19
자동 밸브 – 제어시스템 활용
2 극저온시스템 자동/수동 밸브 조작
수동 밸브 – 현장에서 손으로 조작
압축기 emergency stop 버튼
3 현장에서 손으로 조작
활성화/비활성화
제어시스템 활용
4 진공펌프 가동/정지
(현장 조작 필요 시 연구원에 통지)
5 센서 오버라이드(Override) 설정 또는 해제 제어시스템 활용
6 계장류 제어 변수값(Ramp speed 등) 변경 제어시스템 활용
7 시퀀스 제어 변수값(Timer 등) 변경 제어시스템 활용
8 시퀀스 시작/종료 제어시스템 활용
현장 확인 및 최종 준비 상태 미진
9 극저온시스템 가동 전 최종 준비 상태 확인
할 경우, 상태 보완 포함
기타 극저온시스템의 안전하고 안정적인 운
10 제어시스템 활용 및 현장 조작 포함
전에 필요한 제어
3.4.2 유지보수 등을 위한 극저온시스템 모니터링
계약자는 휴지기간 동안 극저온시스템이 안정적으로 가동될 수 있도록 모니터링을 해야 한
다. 휴지기간의 모니터링 업무는 i) 유지보수 등을 위한 현장 활동 시 극저온시스템에 특이


---

### retriever B:
다. 가동기간의 모니터링 업무는 극저온시스템이 안정적으로 가동되고 있는지 여부를 확인
하기 위해 극저온시스템을 지속적으로 관찰하는 업무를 의미하며, 표 4.에 가동기간의 모니
터링 업무에 해당되는 내용들을 기술하였다.
극저온시스템의 모니터링은 연구원에 구축되어있는 극저온시스템용 제어시스템(이하 “제
어시스템”)을 활용하여 또는 현장에 설치되어있는 장치를 통해 수행한다. 연구원은 구축되
어있는 제어시스템을 활용하기 위한 컴퓨터, 모니터, 키보드, 마우스, 책상 및 의자를 계약
자에게 제공한다. 현재 연구원에 구축되어있는 제어시스템은 표 5.과 같다.
연구원은 계약자에게 현재 구축되어있는 제어시스템에 대한 기본 사용 교육을 제공하며,
계약자는 연구원이 제공하는 교육을 이해하기 위한 배경지식을 보유해야 한다.
문서번호 : -
2025년도 중이온가속기용 극저온시스템
개정번호 : 0
운전 용역 과업지시서 발 행 일 : 2024. 10. 30
페 이 지 : 9/19
표 4. 가동기간의 극저온시스템 모니터링 업무 리스트
순번 극저온시스템 모니터링 업무 비고
1 알람 메시지 확인
2 주요 센서 트렌드 확인 온도, 압력, 유량, 속도 등
3 정상 운전 상황의 주요 센서 지시값 파악 및 숙지 온도, 압력, 유량, 속도 등
4 비정상 운전 상황 감지
연구원에서 현장 확인 필요상황 발생 시, 연구원이 지 무전 통신 활용
5
정하는 값 관찰, 확인 및 상황 전달 (연구원에서 무전기 제공)
6 제어로직 정상 작동 여부 확인
연구원 요청 시,
7 현장 상황 일시적 또는 지속적 관찰
비방사선구역 한정
는 과업 및 운전 인력 감독자가 1인 이상 운전실에 존재해야 한다. 계약자가 가동기간에 수
행해야 하는 운전 업무 상세 내용은 하위 절에 기술된 내용을 따른다. 본 과업지시서에 기
술하지 않은 가동기간의 운전 업무는 연구원과 계약자가 협의하여 수행 여부를 결정한다.
3.3.1 극저온시스템 모니터링
계약자는 가동기간 동안 극저온시스템이 안정적으로 가동될 수 있도록 모니터링을 해야 한
다. 휴지기간의 모니터링 업무는 i) 유지보수 등을 위한 현장 활동 시 극저온시스템에 특이
사항이 발생하는지 여부를 확인하기 위해 또는 ii) 유지보수 등을 수행한 뒤 그 결과를 확인
하기 위해 극저온시스템을 지속적으로 관찰하는 업무를 의미한다. 표 9.에 휴지기간의 모니
터링 업무에 해당되는 내용들을 기술하였다.
극저온시스템의 모니터링은 연구원에 구축되어있는 극저온시스템용 제어시스템(이하 “제
어시스템”)을 활용하여 또는 현장에 설치되어있는 장치를 통해 수행한다. 연구원은 구축되
어있는 제어시스템을 활용하기 위한 컴퓨터, 모니터, 키보드, 마우스, 책상 및 의자를 계약
자에게 제공한다. 현재 연구원에 구축되어있는 제어시스템은 표 5.과 같다.
연구원은 계약자에게 현재 구축되어있는 제어시스템에 대한 기본 사용 교육을 제공하며,
계약자는 연구원이 제공하는 교육을 이해하기 위한 배경지식을 보유해야 한다.
표 9. 휴지기간의 극저온시스템 모니터링 업무 리스트
순번 극저온시스템 모니터링 업무 비고
1 알람 메시지 확인
2 주요 센서 트렌드 확인 온도, 압력, 유량, 속도 등
3 정상 상황의 주요 센서 지시값 파악 및 숙지 온도, 압력, 유량, 속도 등
4 비정상 상황 감지
연구원에서 현장 확인 필요상황 발생 시, 연구원이 지 무전 통신 활용
5
정하는 값 관찰, 확인 및 상황 전달 (연구원에서 무전기 제공)
6 제어로직 정상 작동 여부 확인
문서번호 : -
2025년도 중이온가속기용 극저온시스템
개정번호 : 0
운전 용역 과업지시서 발 행 일 : 2024. 10. 30
페 이 지 : 14/19


---

### openai answer
retriever A 점수: 3점
- 표 4의 가동기간 모니터링 업무에 ‘알람 메시지 확인’, ‘주요 센서 트렌드 확인’, ‘정상 운전 상황의 주요 센서 지시값 파악 및 숙지’, ‘비정상 운전 상황 감지’, ‘제어로직 정상 작동 여부 확인’이 모두 명시되어 있어 답의 모든 항목을 포함합니다.

retriever B 점수: 3점
- 마찬가지로 표 4의 가동기간 모니터링 업무 목록에 위 다섯 항목이 모두 포함되어 있어 제시된 답을 완전히 충족합니다. 

============================================================
### query:
중이온가속기 극저온시스템의 '가동기간'과 '휴지기간'의 운전 업무 형태(근무 방식)는 어떻게 다릅니까?

### answer:
'가동기간'에는 시스템의 24시간 안정적인 운영을 위해 교대근무(24시간 운전)가 이루어집니다. 반면, '휴지기간'에는 주로 시스템 유지보수 등을 위해 주간 운전을 기본으로 합니다.

---

### retriever A:
중이온가속기 또는 극저온시스템 가동기간에는 24시간 운전 업무를 필요로 한다. 통상적으
로 가동기간에는 24시간 운전을 위한 교대근무가 이루어지며, i) 극저온시스템의 모니터링
및 제어, ii) 극저온시스템의 주요 수치를 기록 및 분석하는 업무가 주로 이루어진다.
다만, 통상적으로 이러한 교대근무로 인해 각 운전 인력마다 경험할 수 있는 운전 상황이
서로 다르게 되고, 서로 다른 경험을 했던 운전 인력 모두가 참여하여 각자의 iii) 운전 경험
을 통합/정리할 수 있는 여건이 형성될 수 없다는 한계가 존재한다. 이러한 운전 경험의 통
합 정리는 후술하는 휴지기간에의 운전업무에 집중적으로 수행 가능하다.
통상적으로 휴지기간에는 극저온시스템의 유지보수 등이 진행되며, 유지보수 등을 위한 제
어, 유지보수 등의 경과 및 결과 확인을 위한 극저온시스템 모니터링, 유지보수 등의 결과
확인을 위한 주요 수치 기록 및 분석 또한 필수적인 운전 업무이다. 휴지기간에는 통상적으
로 주간 근무로 운전이 이루어진다. 이와 같이 주간 근무가 이루어질 수 있는 휴지기간의
특성 상, 휴지기간은 가동기간에서 교대근무로 각 운전 인력 개인이 얻은 단편적인 운전 경
험을 통합하여 정리(문서화)할 수 있는 여건이 형성될 수 있는 기간이다.
결과적으로 휴지기간에는 i) 중이온가속기용 극저온시스템의 운전 경험을 통합하여 정리(문
서화)하는 업무가 집중적으로 이루어지며, ii) 유지보수 등을 위한 극저온시스템의 제어와 iii)
유지보수 등을 위한 극저온시스템의 주요 수치 기록 및 분석이 병행되어야 한다.
3.2 운전 대상 및 장소
문서번호 : -
2025년도 중이온가속기용 극저온시스템
개정번호 : 0
운전 용역 과업지시서 발 행 일 : 2024. 10. 30
페 이 지 : 12/19
매뉴얼을 작성하고 개정하는 업무를 수행해야 한다. 또한, 과업 및 운전 인력 감독자가 1인
이상 운전실에 배치되어 상기 업무들을 감독해야 한다. 계약자가 휴지기간에 수행해야 하는
운전 업무 상세 내용은 하위 절에 기술된 내용을 따른다. 본 과업지시서에 기술하지 않은
휴지기간의 운전 업무는 연구원과 계약자가 협의하여 수행 여부를 결정한다.
3.4.1 유지보수 등을 위한 극저온시스템 제어
극저온시스템의 제어는 극저온시스템이 휴지지간에 안전하고 안정적으로 운영될 수 있도록
극저온시스템 또는 극저온시스템 관련 부품들을 제어시스템을 활용하여 또는 현장에서 손으
로 조작하는 행위를 의미한다. 계약자는 극저온시스템이 본 목적에 부합하도록 극저온시스
템의 제어를 수행해야 한다. 표 8.에 극저온시스템 제어 업무에 해당되는 사항들을 기술하
였다.
휴지기간의 극저온시스템 제어는 i) 극저온시스템에 대한 충분한 이해를 바탕으로 ii) 조작
후 유지보수 등 현장 활동에 미치는 영향을 충분히 검토한 뒤 이루어져야 한다. 더불어, iii)
조작 대상물이 위치하고 있는 현장에 인력이 존재하는지 여부를 확인하고, iv) 현장에 인력
이 존재할 경우, 조작 대상물을 조작했을 때 발생할 수 있는 상황에 대해 해당 인력의 안전
이 확보된 뒤에 조작을 수행해야 한다. 마지막으로 v) 조작 대상/조작 상태에 대한 혼동이
없도록 각별히 주의를 하여 제어가 진행되어야 한다.


---

### retriever B:
3. 과업 내용
3.1 중이온가속기용 극저온시스템 운전 업무의 정의 및 필요성
본 과업에서 사용하는 “중이온가속기용 극저온시스템 운전 업무”의 정의는, “중이온가
속기용 극저온시스템이 안전하고 안정적으로 운영(가동기간뿐만 아니라 휴지기간의 운영 포
함)될 수 있도록 i) 극저온시스템을 모니터링 및 제어하고, ii) 극저온시스템의 주요 수치를
기록 및 분석하며, iii) 중이온가속기용 극저온시스템의 운전 경험을 통합하여 정리(문서화)하
는 업무를 의미한다.
중이온가속기 또는 극저온시스템 가동기간에는 24시간 운전 업무를 필요로 한다. 통상적으
로 가동기간에는 24시간 운전을 위한 교대근무가 이루어지며, i) 극저온시스템의 모니터링
및 제어, ii) 극저온시스템의 주요 수치를 기록 및 분석하는 업무가 주로 이루어진다.
다만, 통상적으로 이러한 교대근무로 인해 각 운전 인력마다 경험할 수 있는 운전 상황이
서로 다르게 되고, 서로 다른 경험을 했던 운전 인력 모두가 참여하여 각자의 iii) 운전 경험
을 통합/정리할 수 있는 여건이 형성될 수 없다는 한계가 존재한다. 이러한 운전 경험의 통
합 정리는 후술하는 휴지기간에의 운전업무에 집중적으로 수행 가능하다.
통상적으로 휴지기간에는 극저온시스템의 유지보수 등이 진행되며, 유지보수 등을 위한 제
어, 유지보수 등의 경과 및 결과 확인을 위한 극저온시스템 모니터링, 유지보수 등의 결과
확인을 위한 주요 수치 기록 및 분석 또한 필수적인 운전 업무이다. 휴지기간에는 통상적으
로 주간 근무로 운전이 이루어진다. 이와 같이 주간 근무가 이루어질 수 있는 휴지기간의
제외한다.
표 3. 2025년 중이온가속기용 극저온시스템 운전 용역 대상 장치 목록
구분 장치명 장소 비고
극저온 극저온설비동/ 압축기, 재생시스템, 2 K 시스템
SCL3 극저온플랜트
플랜트 헬륨압축기동 등 부대시설 일체
문서번호 : -
2025년도 중이온가속기용 극저온시스템
개정번호 : 0
운전 용역 과업지시서 발 행 일 : 2024. 10. 30
페 이 지 : 8/19
극저온설비동/ 압축기, 재생시스템, 2 K 시스템
SCL2 극저온플랜트
헬륨압축기동 등 부대시설 일체
TBx 극저온설비동
밸브박스, 초전도가속관 저온유지
SCL3 터널/
극저온
SCL3 LINAC 모듈 및 밸브박스 진공단열용 진
ISOL 구간
분배
공펌프
시스템 밸브박스, 초전도가속관 저온유지
SCL2 터널/
SCL2 LINAC 모듈 및 밸브박스 진공단열용 진
IF 구간
공펌프
외부 정화기 외부
회수 압축시스템 헬륨압축기동
회수
가스백 헬륨압축기동
시스템
헬륨 탱크 외부 기체 헬륨량 관리 포함
액체 질소 시스템 외부 액체 질소량 관리 포함
3.3 중이온가속기 또는 극저온시스템 가동기간의 운전 업무
계약자는 중이온가속기 또는 극저온시스템 가동기간 동안 24시간 운전업무가 이루어질 수
있도록 운전 인력을 제공해야 한다. 24시간 운전 업무는 2인 1조를 기본으로 하며, 주간에
는 과업 및 운전 인력 감독자가 1인 이상 운전실에 존재해야 한다. 계약자가 가동기간에 수
행해야 하는 운전 업무 상세 내용은 하위 절에 기술된 내용을 따른다. 본 과업지시서에 기
술하지 않은 가동기간의 운전 업무는 연구원과 계약자가 협의하여 수행 여부를 결정한다.


---

### openai answer
retriever A 점수: 3점
가동기간의 24시간 교대근무와 휴지기간의 주간 근무(유지보수 중심)가 모두 명확히 포함되어 있음.

retriever B 점수: 3점
가동기간의 24시간 교대근무와 휴지기간의 주간 근무(유지보수 진행) 정보가 모두 포함되어 있음. 

============================================================
### query:
극저온시스템 과업지시서에 따르면, 가동기간 중 교대근무 인력이 개별적으로 얻은 운전 경험을 통합하여 정리하기 어려운 이유는 무엇이며, 이 업무는 주로 언제 수행되어야 합니까?

### answer:
교대근무로 인해 각 운전 인력마다 경험하는 운전 상황이 서로 달라, 모든 인원이 함께 모여 운전 경험을 통합하고 정리할 여건이 형성되기 어렵기 때문입니다. 이러한 운전 경험 통합 및 정리(문서화) 업무는 주로 주간 근무가 이루어지는 '휴지기간'에 집중적으로 수행됩니다.

---

### retriever A:
중이온가속기 또는 극저온시스템 가동기간에는 24시간 운전 업무를 필요로 한다. 통상적으
로 가동기간에는 24시간 운전을 위한 교대근무가 이루어지며, i) 극저온시스템의 모니터링
및 제어, ii) 극저온시스템의 주요 수치를 기록 및 분석하는 업무가 주로 이루어진다.
다만, 통상적으로 이러한 교대근무로 인해 각 운전 인력마다 경험할 수 있는 운전 상황이
서로 다르게 되고, 서로 다른 경험을 했던 운전 인력 모두가 참여하여 각자의 iii) 운전 경험
을 통합/정리할 수 있는 여건이 형성될 수 없다는 한계가 존재한다. 이러한 운전 경험의 통
합 정리는 후술하는 휴지기간에의 운전업무에 집중적으로 수행 가능하다.
통상적으로 휴지기간에는 극저온시스템의 유지보수 등이 진행되며, 유지보수 등을 위한 제
어, 유지보수 등의 경과 및 결과 확인을 위한 극저온시스템 모니터링, 유지보수 등의 결과
확인을 위한 주요 수치 기록 및 분석 또한 필수적인 운전 업무이다. 휴지기간에는 통상적으
로 주간 근무로 운전이 이루어진다. 이와 같이 주간 근무가 이루어질 수 있는 휴지기간의
특성 상, 휴지기간은 가동기간에서 교대근무로 각 운전 인력 개인이 얻은 단편적인 운전 경
험을 통합하여 정리(문서화)할 수 있는 여건이 형성될 수 있는 기간이다.
결과적으로 휴지기간에는 i) 중이온가속기용 극저온시스템의 운전 경험을 통합하여 정리(문
서화)하는 업무가 집중적으로 이루어지며, ii) 유지보수 등을 위한 극저온시스템의 제어와 iii)
유지보수 등을 위한 극저온시스템의 주요 수치 기록 및 분석이 병행되어야 한다.
3.2 운전 대상 및 장소
다. 계약자는 본 과업을 수행하기 위해 적절한 지식 또는 능력을 보유하고 있는 적정 수의 운
전 인력을 연구원에게 제공하여야 한다.
• 가동기간의 극저온시스템 운전 업무 (24시간 운전)
- 극저온시스템 모니터링
- 극저온시스템 주요 수치 기록 및 분석
- 극저온시스템 제어
- 기타 가동기간에 필요한 운전 업무
• 휴지기간의 극저온시스템 운전 업무 (주간 운전)
- 유지보수 등을 위한 극저온시스템 제어
- 유지보수 등을 위한 극저온시스템 모니터링
- 유지보수 등을 위한 극저온시스템 주요 수치 기록 및 분석
- 극저온시스템 운전 매뉴얼 작성 및 개정
- 극저온시스템 운전 교육
- 성과보고회 개최
- 기타 휴지기간에 필요한 운전 업무
• 본 과업 및 본 과업을 위해 투입된 인력에 대한 관리감독
본 과업의 기간은 2025년 1월 1일부터 2025년 12월 31일까지로 하며, 2025년 동안 가동기
간의 운전 업무를 수행하는 것을 기준으로 한다. 단, 표 2.와 같이 연구원의 운영 추진 상황
에 따라 휴지기간의 운전업무를 수행하는 것으로 변경될 수 있다. 가동기간과 휴지기간의
운전 업무는 “3. 과업 내용” 하위 절에 기술한 내용에 따른다.
표 2. 과업 추진 일정(안)
2025년
구분
1분기 2분기 3분기 4분기
SCL3 극저온플랜트
SCL3 중이온가속기 SCL3 구간 운전 SCL3 구간 유지보수
시운전 준비 및
구간 (가동기간) (휴지기간)
시운전 (가동기간)
SCL2 SCL2 극저온플랜트 중이온가속기 SCL2 구간 시운전 SCL2 구간 유지보수
구간 시운전 (가동기간) (가동기간) (휴지기간)
회수


---

### retriever B:
3. 과업 내용
3.1 중이온가속기용 극저온시스템 운전 업무의 정의 및 필요성
본 과업에서 사용하는 “중이온가속기용 극저온시스템 운전 업무”의 정의는, “중이온가
속기용 극저온시스템이 안전하고 안정적으로 운영(가동기간뿐만 아니라 휴지기간의 운영 포
함)될 수 있도록 i) 극저온시스템을 모니터링 및 제어하고, ii) 극저온시스템의 주요 수치를
기록 및 분석하며, iii) 중이온가속기용 극저온시스템의 운전 경험을 통합하여 정리(문서화)하
는 업무를 의미한다.
중이온가속기 또는 극저온시스템 가동기간에는 24시간 운전 업무를 필요로 한다. 통상적으
로 가동기간에는 24시간 운전을 위한 교대근무가 이루어지며, i) 극저온시스템의 모니터링
및 제어, ii) 극저온시스템의 주요 수치를 기록 및 분석하는 업무가 주로 이루어진다.
다만, 통상적으로 이러한 교대근무로 인해 각 운전 인력마다 경험할 수 있는 운전 상황이
서로 다르게 되고, 서로 다른 경험을 했던 운전 인력 모두가 참여하여 각자의 iii) 운전 경험
을 통합/정리할 수 있는 여건이 형성될 수 없다는 한계가 존재한다. 이러한 운전 경험의 통
합 정리는 후술하는 휴지기간에의 운전업무에 집중적으로 수행 가능하다.
통상적으로 휴지기간에는 극저온시스템의 유지보수 등이 진행되며, 유지보수 등을 위한 제
어, 유지보수 등의 경과 및 결과 확인을 위한 극저온시스템 모니터링, 유지보수 등의 결과
확인을 위한 주요 수치 기록 및 분석 또한 필수적인 운전 업무이다. 휴지기간에는 통상적으
로 주간 근무로 운전이 이루어진다. 이와 같이 주간 근무가 이루어질 수 있는 휴지기간의
는 과업 및 운전 인력 감독자가 1인 이상 운전실에 존재해야 한다. 계약자가 가동기간에 수
행해야 하는 운전 업무 상세 내용은 하위 절에 기술된 내용을 따른다. 본 과업지시서에 기
술하지 않은 가동기간의 운전 업무는 연구원과 계약자가 협의하여 수행 여부를 결정한다.
3.3.1 극저온시스템 모니터링
계약자는 가동기간 동안 극저온시스템이 안정적으로 가동될 수 있도록 모니터링을 해야 한


---

### openai answer
retriever A 점수: 3점
교대근무로 인한 경험 분산과 통합·정리 여건 부재의 이유를 명시하고, 해당 업무가 주간 근무가 가능한 ‘휴지기간’에 집중 수행된다고 구체적으로 제시함.

retriever B 점수: 3점
교대근무로 경험 통합·정리 여건이 형성되지 않는다는 이유와, 통합 정리가 ‘휴지기간’(주간 근무)에 집중 수행 가능하다는 시점을 모두 포함함. 

============================================================
### query:
계약업체가 매월 운전 현황을 정리해서 연구원에 보고해야 하는 공식적인 문서는 무엇이며, 제출 시점은 언제입니까?

### answer:
매월 운전 상황을 요약하여 제출해야 하는 문서는 '월간보고서'이며, 매월 기성 비용을 신청할 때 제출해야 합니다.

---

### retriever A:
계약종료 및 인수인계에 관한 사항
세부내용
○ 수행업체는 본 사업 종료 1개월 전 시스템의 발전 및 안정성을 유지하기 위한 인수인계 계획을 제시하여야 함
- 본 사업수행 기간 동안 취득한 모든 정보 및 생상된 산출물 일체
- 인수인계의 주요 업무, 상세 일정 및 인수인계 대상 주요 산출물 제시
- 인수인계 기간 동안 발생하는 리스크에 대한 방지대책 제시
○ 수행업체는 계약기간 만료 시 차기 수행업체로 선정된 사업자가 본 사업의 업무를 차질없이 수행할 수 있도록 요청 시 통합접수시스템 운영 업무 수행에 필요한 중요사항 등을 인계하여야 하며, 시스템의 효율적인 운영을 위해 하자담보 책임기간 중에는 기술자문에 응하여야 함
- 시스템 구성도, 시스템 현황, DB권한 등
- 통합접수시스템 기동/종료 절차
- 타 시스템 연계 및 적재 절차
- 일반 사용자 및 관리자 매뉴얼 등
○ 인수 및 인계 기간은 차기 시스템 운영 사업 개시일로부터 30일 이내로 하며, 인수 및 인계기간 동안의 용역비용은 지급하지 아니함
산출 정보
인수인계 계획서(사업수행계획서 내 포함 가능)
관련 요구사항
요구사항 출처
요구사항 분류
프로젝트지원 요구사항
요구사항 고유번호
PSR-008
요구사항 명칭
정보화사업 주요점검 및 수행지원
요구사항
상세설명
정의
정보화사업 주요점검 및 수행지원
세부내용
○ 통합접수시스템 관련 점검사항 준수 및 현장점검 수행
○ 보안관리 실태 점검 대응
○ 공공데이터 품질관리(DB) 수준평가 대응 및 데이터 오류 정제 활동 실시
산출 정보
점검별 결과보고서
관련 요구사항
요구사항 출처
□ 품질관리 요구사항
요구사항 분류
· 월별 추진 실적 보고
· 관련부서 업무협의 내용
· 문제점 보고 및 특이사항
매월 말일
비정기
보고
인수인계
보고
· 신규 사업자 선정시 업무인계 결과보고
사업종료
1주전
기타
· 이슈사항 및 문제점 협의 등
이슈발생시
※ 보고형식 및 주기는 추후 협의하여 조정할 수 있음
ㅇ 계약 기간 만료일 전 과업 완료 보고서를 제출하여야 함
구분
보고명
보고내용
시기
사업
종료
완료
보고서
· 용역종료 시 전체 추진단계의 수행결과 보고로 인쇄물 2부, 외장하드 1본으로 제출
· 산출물을 종류별로 분류한 파일
용역종료일
2주일 전
- 용역 완료시 제출하는 최종 결과물(프로그램 소스 코드(주석, 설명서 포함), ERD, 테이블 정의서, 프로그램 정의서 등)은 발주자의 승인을 득한 후 원본 파일을 CD 또는 HDD에 수록하여 산출물과 같이 제출하여야 하고, 프로그램 소스에 대한 형상관리가 가능하도록 협조하여야 함
- 산출물은 기존 산출물을 기반으로 현행화 또는 추가 작성해야 하며, 상호 협의하여 조정 가능
ㅇ 제출 시기에 따라 종류별 산출물을 제출하되, 사업관리 및 검수를 위한 각종 업무 보고를 요청할 경우, 신속하게 작성하여 보고하는 수시보고 체계 마련 ※ 제출 전 주관기관 사전 검토 및 확인
산출정보
착수보고서, 완료보고서, 월간보고서
요구사항 고유번호
PMR-005
요구사항 명칭
인계 조건
요구사항
상세설명
정의
관한 사항
세부
내용
인계 요건
- 사업종료 1개월 전까지 업무인계 계획서를 작성해 진흥원의 승인을 받아야함
- 발주기관과 월 1회 이상 기술이전 등에 대한 협의를 하고, 그 결과를 문서화


---

### retriever B:
4. 제출문서
계약자는 표 13.와 같은 문서들을 제출일정에 맞춰 연구원에 제출해야 한다.
표 13. 제출문서 목록 및 관련 사항
연구원
제출문서 내용 비고 및 제출일
승인
투입 인력
- 투입 인력 이력 사항 X - 계약 시
이력서
- 최초: 계약 후 1개월 내
작업계획서 - 일별 투입 인력 - 이후: 매주 금요일까지 차주
X
(주간근무표) - 휴가상황 근무표 제출
- 문서 파일 제출
- IBS에서 제공한 양식 파일
극저온시스템 - 매 월간 기성 신청 시 제출
주요 수치 - 극저온시스템 주요 수치 X - 문서 파일 제출
기록문서 - 수치 기록을 위해 계약자가
준비한 PC에 상시 저장
일일근무일지 - 일일 운영 상황 기록 O - 매 월간 기성 신청 시 제출
- 문서 파일 제출
- 수치 기록을 위해 계약자가
인수인계 문서 - (가동기간) 인수인계 내용 X 준비한 PC에 상시 저장
- 수리 사유 발생일
- 매 월간 기성 신청 시 제출
- 수리 필요 대상
수리필요사항 - 문서 파일 제출
- 상세 내용 O
기록 문서 - 수치 기록을 위해 계약자가
- 발견자
준비한 PC에 상시 저장
- 기타 필요한 내용
- 발생한 경우, 기성 신청 시
특이사항, 위험상황 발생 시,
제출 (해당 월 발생 건)
- 상황 내용
- 문서 파일 제출
- 작성자
Lesson learn O - 운전 인력 전원 회람 후 확
- 금회 대처 방법
인 서명 필수
- 향후 대처 방법
- 수치 기록을 위해 계약자가
- 기타 필요한 내용
준비한 PC에 상시 저장
- 매 월간 기성 신청 시 제출
월간보고서 - 월간 운전 상황 요약 O
- 문서 파일 제출
라. 계약 후 제출서류 : 사업수행계획서, 기술적용계획표
1) 사업수행자는 계약일로부터 14일 이내에 사업수행계획서와 기술적용계획표를 제출하고 사업 착수를 완료하여야 함
2) 사업수행계획서에는 제안요청서, 제안서, 협상에 의해 수정·보완·추가된 내용을 반영하여 제출하여야 하며 발주기관의 승인을 받아야 함
제출방법
1
착수계
A4
1
계약 후 7일 이내
서신문서
2
보안관련 문서
(보안서약서, 비밀유지서약서, 개인정보 위탁계약서, 개인정보 보호서약서)
A4
1
계약 후 7일 이내
3
사업수행계획서
A4
8
계약 후 10일 이내
4
착수보고서
A4
8
착수 후 10일 이내
5
완료보고서
A4
8
계약종료 2주 전 까지
6
최종산출물
(개발 산출물, 운영자 매뉴얼, 사용자 매뉴얼, 각종 분석/측정 결과보고서, 기술지원 확약서, 품질관리보고서, 기술적용계획/결과표, 기술이전계획서, 장애대응 매뉴얼, 테스트 수행 계획서/결과서 등)
USB
2
계약종료 2주 전 까지


---

### openai answer
retriever A 점수: 2점
(‘월간보고서’ 산출물 존재는 명시되어 있으나, 제출 시점이 ‘매 월간 기성 신청 시’라는 정보는 없고 별도로 ‘매월 말일’ 등 다른 주기만 제시되어 있어 답의 일부만 포함됨)

retriever B 점수: 3점
(‘월간보고서(월간 운전 상황 요약)’와 제출 시점 ‘매 월간 기성 신청 시 제출’이 명확히 기재되어 있어 정답을 모두 포함함) 


## mmr_results

============================================================
### query:
본 과업지시서에서 '극저온시스템'은 무엇을 포함하는 것으로 정의하고 있습니까?

### answer:
'극저온시스템'은 SCL3/SCL2 극저온플랜트 및 부대설비, SCL3/SCL2 분배시스템 및 부대설비, SCL3/SCL2 초전도가속관 저온유지모듈, 기체헬륨 탱크, 액체질소 시스템 및 극저온시스템용 헬륨회수시스템을 모두 포함하는 것으로 정의됩니다.

---

### retriever A:
7. 안전관리 .............................................................................................................. 19
8. 붙임 자료 ........................................................................................................... 19
문서번호 : -
2025년도 중이온가속기용 극저온시스템
개정번호 : 0
운전 용역 과업지시서 발 행 일 : 2024. 10. 30
페 이 지 : 5/19
1. 서 론
1.1 목 적
기초과학연구원 중이온가속기연구소(이하 “연구원”)에서 운영중인 중이온가속기용 극저
온시스템(이하 “극저온시스템”)은 고도의 공정 시스템으로, 안전하고 안정적인 중이온가속
기의 운영을 위해 상시 모니터링이 필요하다. 이에 본 과업지시서는 극저온시스템의 운전
용역에 대한 상세 과업 내용을 기술한다.
1.2 용어 설명
본 과업지시서에서 사용하는 용어에 대한 상세 내용은 표 1.과 같다. 계약자는 반드시 해당
용어의 의미를 충분히 이해하고 숙지해야 하며, 용어 미숙지로 인한 책임은 계약자에게 있
다.
표 1. 본 과업지시서에 사용되는 용어 설명
용어 설명
연구원 중이온가속기연구소
계약자 본 과업을 수행할 계약상대자
SCL3/SCL2 극저온플랜트 및 부대설비,
SCL3/SCL2 분배시스템 및 부대설비,
SCL3/SCL2 초전도가속관 저온유지모듈,
극저온시스템
기체헬륨 탱크, 액체질소 시스템 및
TBx 극저온설비동
밸브박스, 초전도가속관 저온유지
SCL3 터널/
극저온
SCL3 LINAC 모듈 및 밸브박스 진공단열용 진
ISOL 구간
분배
공펌프
시스템 밸브박스, 초전도가속관 저온유지
SCL2 터널/
SCL2 LINAC 모듈 및 밸브박스 진공단열용 진
IF 구간
공펌프
외부 정화기 외부
회수 압축시스템 헬륨압축기동
회수
가스백 헬륨압축기동
시스템
헬륨 탱크 외부 기체 헬륨량 관리 포함
액체 질소 시스템 외부 액체 질소량 관리 포함
3.3 중이온가속기 또는 극저온시스템 가동기간의 운전 업무
계약자는 중이온가속기 또는 극저온시스템 가동기간 동안 24시간 운전업무가 이루어질 수
있도록 운전 인력을 제공해야 한다. 24시간 운전 업무는 2인 1조를 기본으로 하며, 주간에
는 과업 및 운전 인력 감독자가 1인 이상 운전실에 존재해야 한다. 계약자가 가동기간에 수
행해야 하는 운전 업무 상세 내용은 하위 절에 기술된 내용을 따른다. 본 과업지시서에 기
술하지 않은 가동기간의 운전 업무는 연구원과 계약자가 협의하여 수행 여부를 결정한다.
3.3.1 극저온시스템 모니터링
계약자는 가동기간 동안 극저온시스템이 안정적으로 가동될 수 있도록 모니터링을 해야 한
다. 가동기간의 모니터링 업무는 극저온시스템이 안정적으로 가동되고 있는지 여부를 확인
하기 위해 극저온시스템을 지속적으로 관찰하는 업무를 의미하며, 표 4.에 가동기간의 모니
터링 업무에 해당되는 내용들을 기술하였다.
극저온시스템의 모니터링은 연구원에 구축되어있는 극저온시스템용 제어시스템(이하 “제


---

### retriever B:
1. 서 론
1.1 목 적
기초과학연구원 중이온가속기연구소(이하 “연구원”)에서 운영중인 중이온가속기용 극저
온시스템(이하 “극저온시스템”)은 고도의 공정 시스템으로, 안전하고 안정적인 중이온가속
기의 운영을 위해 상시 모니터링이 필요하다. 이에 본 과업지시서는 극저온시스템의 운전
용역에 대한 상세 과업 내용을 기술한다.
1.2 용어 설명
본 과업지시서에서 사용하는 용어에 대한 상세 내용은 표 1.과 같다. 계약자는 반드시 해당
용어의 의미를 충분히 이해하고 숙지해야 하며, 용어 미숙지로 인한 책임은 계약자에게 있
다.
표 1. 본 과업지시서에 사용되는 용어 설명
용어 설명
연구원 중이온가속기연구소
계약자 본 과업을 수행할 계약상대자
SCL3/SCL2 극저온플랜트 및 부대설비,
SCL3/SCL2 분배시스템 및 부대설비,
SCL3/SCL2 초전도가속관 저온유지모듈,
극저온시스템
기체헬륨 탱크, 액체질소 시스템 및
극저온시스템용 헬륨회수시스템 포함
(“3.2절 운전대상”, 표 3. 참고)
중이온가속기 또는 극저온시스템이 냉각(냉각 착수
가동기간
이후) 또는 빔서비스 등을 위해 가동 중인 기간
중이온가속기 또는 극저온시스템이 유지보수 등을
휴지기간
위해 가동을 멈추고 휴지 중인 기간
중이온가속기용 극저온시스템이 가동기간 및 휴지
기간에 안전하고 안정적으로 운영될 수 있도록,
중이온가속기용 극저온시스템 i) 극저온시스템을 모니터링 및 제어하고,
운전 업무 ii) 극저온시스템의 주요 수치를 기록 및 분석하며,
iii) 중이온가속기용 극저온시스템의 운전 경험을
통합하여 정리(문서화)하는 업무
문서번호 : -
문서번호 : -
2025년도 중이온가속기용 극저온시스템
개정번호 : 0
운전 용역 과업지시서 발 행 일 : 2024. 10. 30
페 이 지 : 12/19
매뉴얼을 작성하고 개정하는 업무를 수행해야 한다. 또한, 과업 및 운전 인력 감독자가 1인
이상 운전실에 배치되어 상기 업무들을 감독해야 한다. 계약자가 휴지기간에 수행해야 하는
운전 업무 상세 내용은 하위 절에 기술된 내용을 따른다. 본 과업지시서에 기술하지 않은
휴지기간의 운전 업무는 연구원과 계약자가 협의하여 수행 여부를 결정한다.
3.4.1 유지보수 등을 위한 극저온시스템 제어
극저온시스템의 제어는 극저온시스템이 휴지지간에 안전하고 안정적으로 운영될 수 있도록
극저온시스템 또는 극저온시스템 관련 부품들을 제어시스템을 활용하여 또는 현장에서 손으
로 조작하는 행위를 의미한다. 계약자는 극저온시스템이 본 목적에 부합하도록 극저온시스
템의 제어를 수행해야 한다. 표 8.에 극저온시스템 제어 업무에 해당되는 사항들을 기술하
였다.
휴지기간의 극저온시스템 제어는 i) 극저온시스템에 대한 충분한 이해를 바탕으로 ii) 조작
후 유지보수 등 현장 활동에 미치는 영향을 충분히 검토한 뒤 이루어져야 한다. 더불어, iii)
조작 대상물이 위치하고 있는 현장에 인력이 존재하는지 여부를 확인하고, iv) 현장에 인력
이 존재할 경우, 조작 대상물을 조작했을 때 발생할 수 있는 상황에 대해 해당 인력의 안전
이 확보된 뒤에 조작을 수행해야 한다. 마지막으로 v) 조작 대상/조작 상태에 대한 혼동이
없도록 각별히 주의를 하여 제어가 진행되어야 한다.


---

### openai answer
retriever A 점수: 2점
표 1의 ‘극저온시스템’ 정의에 SCL3/SCL2 극저온플랜트·분배시스템·저온유지모듈, 기체헬륨 탱크, 액체질소 시스템까지는 보이나, ‘극저온시스템용 헬륨회수시스템 포함’이 명시적으로 확인되지 않아 정답의 일부만 포함됨.

retriever B 점수: 3점
표 1에서 ‘극저온시스템’을 SCL3/SCL2 극저온플랜트·분배시스템·저온유지모듈, 기체헬륨 탱크, 액체질소 시스템 및 ‘극저온시스템용 헬륨회수시스템 포함’으로 명시하여 정답의 모든 요소를 포함함. 

============================================================
### query:
극저온시스템 과업 추진 일정에 따르면, 4분기에 SCL3 구간에 대해 예정된 주요 업무는 무엇이며, 이때의 업무 기간은 '가동기간'과 '휴지기간' 중 어느 것에 해당합니까?

### answer:
2025년 4분기에 SCL3 구간에 예정된 주요 업무는 '유지보수'이며, 이 기간은 '휴지기간'에 해당합니다.

---

### retriever A:
다. 계약자는 본 과업을 수행하기 위해 적절한 지식 또는 능력을 보유하고 있는 적정 수의 운
전 인력을 연구원에게 제공하여야 한다.
• 가동기간의 극저온시스템 운전 업무 (24시간 운전)
- 극저온시스템 모니터링
- 극저온시스템 주요 수치 기록 및 분석
- 극저온시스템 제어
- 기타 가동기간에 필요한 운전 업무
• 휴지기간의 극저온시스템 운전 업무 (주간 운전)
- 유지보수 등을 위한 극저온시스템 제어
- 유지보수 등을 위한 극저온시스템 모니터링
- 유지보수 등을 위한 극저온시스템 주요 수치 기록 및 분석
- 극저온시스템 운전 매뉴얼 작성 및 개정
- 극저온시스템 운전 교육
- 성과보고회 개최
- 기타 휴지기간에 필요한 운전 업무
• 본 과업 및 본 과업을 위해 투입된 인력에 대한 관리감독
본 과업의 기간은 2025년 1월 1일부터 2025년 12월 31일까지로 하며, 2025년 동안 가동기
간의 운전 업무를 수행하는 것을 기준으로 한다. 단, 표 2.와 같이 연구원의 운영 추진 상황
에 따라 휴지기간의 운전업무를 수행하는 것으로 변경될 수 있다. 가동기간과 휴지기간의
운전 업무는 “3. 과업 내용” 하위 절에 기술한 내용에 따른다.
표 2. 과업 추진 일정(안)
2025년
구분
1분기 2분기 3분기 4분기
SCL3 극저온플랜트
SCL3 중이온가속기 SCL3 구간 운전 SCL3 구간 유지보수
시운전 준비 및
구간 (가동기간) (휴지기간)
시운전 (가동기간)
SCL2 SCL2 극저온플랜트 중이온가속기 SCL2 구간 시운전 SCL2 구간 유지보수
구간 시운전 (가동기간) (가동기간) (휴지기간)
회수
2025년
구분
1분기 2분기 3분기 4분기
SCL3 극저온플랜트
SCL3 중이온가속기 SCL3 구간 운전 SCL3 구간 유지보수
시운전 준비 및
구간 (가동기간) (휴지기간)
시운전 (가동기간)
SCL2 SCL2 극저온플랜트 중이온가속기 SCL2 구간 시운전 SCL2 구간 유지보수
구간 시운전 (가동기간) (가동기간) (휴지기간)
회수
필요시 간헐적 운전 (휴지기간에 준함)
시스템
문서번호 : -
2025년도 중이온가속기용 극저온시스템
개정번호 : 0
운전 용역 과업지시서 발 행 일 : 2024. 10. 30
페 이 지 : 7/19
3. 과업 내용
3.1 중이온가속기용 극저온시스템 운전 업무의 정의 및 필요성
본 과업에서 사용하는 “중이온가속기용 극저온시스템 운전 업무”의 정의는, “중이온가
속기용 극저온시스템이 안전하고 안정적으로 운영(가동기간뿐만 아니라 휴지기간의 운영 포
함)될 수 있도록 i) 극저온시스템을 모니터링 및 제어하고, ii) 극저온시스템의 주요 수치를
기록 및 분석하며, iii) 중이온가속기용 극저온시스템의 운전 경험을 통합하여 정리(문서화)하
는 업무를 의미한다.
중이온가속기 또는 극저온시스템 가동기간에는 24시간 운전 업무를 필요로 한다. 통상적으
로 가동기간에는 24시간 운전을 위한 교대근무가 이루어지며, i) 극저온시스템의 모니터링
및 제어, ii) 극저온시스템의 주요 수치를 기록 및 분석하는 업무가 주로 이루어진다.
다만, 통상적으로 이러한 교대근무로 인해 각 운전 인력마다 경험할 수 있는 운전 상황이


---

### retriever B:
다. 계약자는 본 과업을 수행하기 위해 적절한 지식 또는 능력을 보유하고 있는 적정 수의 운
전 인력을 연구원에게 제공하여야 한다.
• 가동기간의 극저온시스템 운전 업무 (24시간 운전)
- 극저온시스템 모니터링
- 극저온시스템 주요 수치 기록 및 분석
- 극저온시스템 제어
- 기타 가동기간에 필요한 운전 업무
• 휴지기간의 극저온시스템 운전 업무 (주간 운전)
- 유지보수 등을 위한 극저온시스템 제어
- 유지보수 등을 위한 극저온시스템 모니터링
- 유지보수 등을 위한 극저온시스템 주요 수치 기록 및 분석
- 극저온시스템 운전 매뉴얼 작성 및 개정
- 극저온시스템 운전 교육
- 성과보고회 개최
- 기타 휴지기간에 필요한 운전 업무
• 본 과업 및 본 과업을 위해 투입된 인력에 대한 관리감독
본 과업의 기간은 2025년 1월 1일부터 2025년 12월 31일까지로 하며, 2025년 동안 가동기
간의 운전 업무를 수행하는 것을 기준으로 한다. 단, 표 2.와 같이 연구원의 운영 추진 상황
에 따라 휴지기간의 운전업무를 수행하는 것으로 변경될 수 있다. 가동기간과 휴지기간의
운전 업무는 “3. 과업 내용” 하위 절에 기술한 내용에 따른다.
표 2. 과업 추진 일정(안)
2025년
구분
1분기 2분기 3분기 4분기
SCL3 극저온플랜트
SCL3 중이온가속기 SCL3 구간 운전 SCL3 구간 유지보수
시운전 준비 및
구간 (가동기간) (휴지기간)
시운전 (가동기간)
SCL2 SCL2 극저온플랜트 중이온가속기 SCL2 구간 시운전 SCL2 구간 유지보수
구간 시운전 (가동기간) (가동기간) (휴지기간)
회수
일정
구분
추진 일정
M
M+1
M+2
M+3
M+4
사업발주 및 사업자 선정
업무 협의
시스템
구축
기존 시스템 분석
업무(요구사항) 분석
설계 및 구축
시스템 구현 및 개발
시스템테스트 및 수정
시스템 가동 및 운영(안정화)
※ 위 추진일정은 사업 추진과정에서 조정 가능함
Ⅲ
다. 추진일정
구 분
2024년~2025년
M1
M2
M3
M4
M5
M6
◦ 시스템 개발
- 분석
- 설계/구현
- 단위/통합 시험
◦ 연계시스템 기능개선
◦ 운영개시 및 안정화
◦ 보고회
- 착수, 중간, 최종보고
★
★
★
* 추진과정에서 일정은 변경될 수 있음
Ⅱ
일반 현황


---

### openai answer
retriever A 점수: 3점
(표 2 과업 추진 일정에 2025년 4분기 SCL3 구간이 ‘유지보수(휴지기간)’로 명시되어 있어, 주요 업무와 기간 유형이 모두 포함됨)

retriever B 점수: 3점
(동일하게 표 2에 4분기 SCL3 구간 ‘유지보수(휴지기간)’로 기재되어 있어, 정답의 모든 정보가 포함됨) 

============================================================
### query:
극저온시스템 '가동기간' 동안 계약자가 24시간 수행해야 할 극저온시스템 모니터링 업무에는 어떤 것들이 있습니까? (3가지 이상)

### answer:
가동기간 동안 수행해야 할 모니터링 업무는 다음과 같습니다.
알람 메시지 확인
주요 센서(온도, 압력, 유량 등)의 트렌드 확인
정상 운전 상황의 주요 센서 지시값 파악 및 숙지
비정상 운전 상황 감지
제어로직 정상 작동 여부 확인

---

### retriever A:
다. 가동기간의 모니터링 업무는 극저온시스템이 안정적으로 가동되고 있는지 여부를 확인
하기 위해 극저온시스템을 지속적으로 관찰하는 업무를 의미하며, 표 4.에 가동기간의 모니
터링 업무에 해당되는 내용들을 기술하였다.
극저온시스템의 모니터링은 연구원에 구축되어있는 극저온시스템용 제어시스템(이하 “제
어시스템”)을 활용하여 또는 현장에 설치되어있는 장치를 통해 수행한다. 연구원은 구축되
어있는 제어시스템을 활용하기 위한 컴퓨터, 모니터, 키보드, 마우스, 책상 및 의자를 계약
자에게 제공한다. 현재 연구원에 구축되어있는 제어시스템은 표 5.과 같다.
연구원은 계약자에게 현재 구축되어있는 제어시스템에 대한 기본 사용 교육을 제공하며,
계약자는 연구원이 제공하는 교육을 이해하기 위한 배경지식을 보유해야 한다.
문서번호 : -
2025년도 중이온가속기용 극저온시스템
개정번호 : 0
운전 용역 과업지시서 발 행 일 : 2024. 10. 30
페 이 지 : 9/19
표 4. 가동기간의 극저온시스템 모니터링 업무 리스트
순번 극저온시스템 모니터링 업무 비고
1 알람 메시지 확인
2 주요 센서 트렌드 확인 온도, 압력, 유량, 속도 등
3 정상 운전 상황의 주요 센서 지시값 파악 및 숙지 온도, 압력, 유량, 속도 등
4 비정상 운전 상황 감지
연구원에서 현장 확인 필요상황 발생 시, 연구원이 지 무전 통신 활용
5
정하는 값 관찰, 확인 및 상황 전달 (연구원에서 무전기 제공)
6 제어로직 정상 작동 여부 확인
연구원 요청 시,
7 현장 상황 일시적 또는 지속적 관찰
비방사선구역 한정
이온가속기용 극저온시스템 운영에 중대한 영향이 생길 경우, 이는 계약자의 책임으로 한
다.
표 8. 휴지기간의 극저온시스템 제어 업무 리스트
순번 극저온시스템 제어 업무 비고
자동/수동(Auto/Manual) 제어 우선
1 계장류 제어 우선권(Priority) 변경
권 설정 변경
문서번호 : -
2025년도 중이온가속기용 극저온시스템
개정번호 : 0
운전 용역 과업지시서 발 행 일 : 2024. 10. 30
페 이 지 : 13/19
자동 밸브 – 제어시스템 활용
2 극저온시스템 자동/수동 밸브 조작
수동 밸브 – 현장에서 손으로 조작
압축기 emergency stop 버튼
3 현장에서 손으로 조작
활성화/비활성화
제어시스템 활용
4 진공펌프 가동/정지
(현장 조작 필요 시 연구원에 통지)
5 센서 오버라이드(Override) 설정 또는 해제 제어시스템 활용
6 계장류 제어 변수값(Ramp speed 등) 변경 제어시스템 활용
7 시퀀스 제어 변수값(Timer 등) 변경 제어시스템 활용
8 시퀀스 시작/종료 제어시스템 활용
현장 확인 및 최종 준비 상태 미진
9 극저온시스템 가동 전 최종 준비 상태 확인
할 경우, 상태 보완 포함
기타 극저온시스템의 안전하고 안정적인 운
10 제어시스템 활용 및 현장 조작 포함
전에 필요한 제어
3.4.2 유지보수 등을 위한 극저온시스템 모니터링
계약자는 휴지기간 동안 극저온시스템이 안정적으로 가동될 수 있도록 모니터링을 해야 한
다. 휴지기간의 모니터링 업무는 i) 유지보수 등을 위한 현장 활동 시 극저온시스템에 특이


---

### retriever B:
다. 가동기간의 모니터링 업무는 극저온시스템이 안정적으로 가동되고 있는지 여부를 확인
하기 위해 극저온시스템을 지속적으로 관찰하는 업무를 의미하며, 표 4.에 가동기간의 모니
터링 업무에 해당되는 내용들을 기술하였다.
극저온시스템의 모니터링은 연구원에 구축되어있는 극저온시스템용 제어시스템(이하 “제
어시스템”)을 활용하여 또는 현장에 설치되어있는 장치를 통해 수행한다. 연구원은 구축되
어있는 제어시스템을 활용하기 위한 컴퓨터, 모니터, 키보드, 마우스, 책상 및 의자를 계약
자에게 제공한다. 현재 연구원에 구축되어있는 제어시스템은 표 5.과 같다.
연구원은 계약자에게 현재 구축되어있는 제어시스템에 대한 기본 사용 교육을 제공하며,
계약자는 연구원이 제공하는 교육을 이해하기 위한 배경지식을 보유해야 한다.
문서번호 : -
2025년도 중이온가속기용 극저온시스템
개정번호 : 0
운전 용역 과업지시서 발 행 일 : 2024. 10. 30
페 이 지 : 9/19
표 4. 가동기간의 극저온시스템 모니터링 업무 리스트
순번 극저온시스템 모니터링 업무 비고
1 알람 메시지 확인
2 주요 센서 트렌드 확인 온도, 압력, 유량, 속도 등
3 정상 운전 상황의 주요 센서 지시값 파악 및 숙지 온도, 압력, 유량, 속도 등
4 비정상 운전 상황 감지
연구원에서 현장 확인 필요상황 발생 시, 연구원이 지 무전 통신 활용
5
정하는 값 관찰, 확인 및 상황 전달 (연구원에서 무전기 제공)
6 제어로직 정상 작동 여부 확인
연구원 요청 시,
7 현장 상황 일시적 또는 지속적 관찰
비방사선구역 한정
는 과업 및 운전 인력 감독자가 1인 이상 운전실에 존재해야 한다. 계약자가 가동기간에 수
행해야 하는 운전 업무 상세 내용은 하위 절에 기술된 내용을 따른다. 본 과업지시서에 기
술하지 않은 가동기간의 운전 업무는 연구원과 계약자가 협의하여 수행 여부를 결정한다.
3.3.1 극저온시스템 모니터링
계약자는 가동기간 동안 극저온시스템이 안정적으로 가동될 수 있도록 모니터링을 해야 한
다. 휴지기간의 모니터링 업무는 i) 유지보수 등을 위한 현장 활동 시 극저온시스템에 특이
사항이 발생하는지 여부를 확인하기 위해 또는 ii) 유지보수 등을 수행한 뒤 그 결과를 확인
하기 위해 극저온시스템을 지속적으로 관찰하는 업무를 의미한다. 표 9.에 휴지기간의 모니
터링 업무에 해당되는 내용들을 기술하였다.
극저온시스템의 모니터링은 연구원에 구축되어있는 극저온시스템용 제어시스템(이하 “제
어시스템”)을 활용하여 또는 현장에 설치되어있는 장치를 통해 수행한다. 연구원은 구축되
어있는 제어시스템을 활용하기 위한 컴퓨터, 모니터, 키보드, 마우스, 책상 및 의자를 계약
자에게 제공한다. 현재 연구원에 구축되어있는 제어시스템은 표 5.과 같다.
연구원은 계약자에게 현재 구축되어있는 제어시스템에 대한 기본 사용 교육을 제공하며,
계약자는 연구원이 제공하는 교육을 이해하기 위한 배경지식을 보유해야 한다.
표 9. 휴지기간의 극저온시스템 모니터링 업무 리스트
순번 극저온시스템 모니터링 업무 비고
1 알람 메시지 확인
2 주요 센서 트렌드 확인 온도, 압력, 유량, 속도 등
3 정상 상황의 주요 센서 지시값 파악 및 숙지 온도, 압력, 유량, 속도 등
4 비정상 상황 감지
연구원에서 현장 확인 필요상황 발생 시, 연구원이 지 무전 통신 활용
5
정하는 값 관찰, 확인 및 상황 전달 (연구원에서 무전기 제공)
6 제어로직 정상 작동 여부 확인
문서번호 : -
2025년도 중이온가속기용 극저온시스템
개정번호 : 0
운전 용역 과업지시서 발 행 일 : 2024. 10. 30
페 이 지 : 14/19


---

### openai answer
retriever A 점수: 3점
답변의 항목(알람 확인, 주요 센서 트렌드, 정상 지시값 숙지, 비정상 감지, 제어로직 정상 작동 확인 등)이 표 4의 가동기간 모니터링 리스트에 모두 포함되어 있음.

retriever B 점수: 3점
가동기간 표 4에 동일 항목들이 모두 명시되어 있어 답변 전부를 충족함(추가로 휴지기간 내용도 있으나 질의에 필요한 정보는 완비). 

============================================================
### query:
중이온가속기 극저온시스템의 '가동기간'과 '휴지기간'의 운전 업무 형태(근무 방식)는 어떻게 다릅니까?

### answer:
'가동기간'에는 시스템의 24시간 안정적인 운영을 위해 교대근무(24시간 운전)가 이루어집니다. 반면, '휴지기간'에는 주로 시스템 유지보수 등을 위해 주간 운전을 기본으로 합니다.

---

### retriever A:
중이온가속기 또는 극저온시스템 가동기간에는 24시간 운전 업무를 필요로 한다. 통상적으
로 가동기간에는 24시간 운전을 위한 교대근무가 이루어지며, i) 극저온시스템의 모니터링
및 제어, ii) 극저온시스템의 주요 수치를 기록 및 분석하는 업무가 주로 이루어진다.
다만, 통상적으로 이러한 교대근무로 인해 각 운전 인력마다 경험할 수 있는 운전 상황이
서로 다르게 되고, 서로 다른 경험을 했던 운전 인력 모두가 참여하여 각자의 iii) 운전 경험
을 통합/정리할 수 있는 여건이 형성될 수 없다는 한계가 존재한다. 이러한 운전 경험의 통
합 정리는 후술하는 휴지기간에의 운전업무에 집중적으로 수행 가능하다.
통상적으로 휴지기간에는 극저온시스템의 유지보수 등이 진행되며, 유지보수 등을 위한 제
어, 유지보수 등의 경과 및 결과 확인을 위한 극저온시스템 모니터링, 유지보수 등의 결과
확인을 위한 주요 수치 기록 및 분석 또한 필수적인 운전 업무이다. 휴지기간에는 통상적으
로 주간 근무로 운전이 이루어진다. 이와 같이 주간 근무가 이루어질 수 있는 휴지기간의
특성 상, 휴지기간은 가동기간에서 교대근무로 각 운전 인력 개인이 얻은 단편적인 운전 경
험을 통합하여 정리(문서화)할 수 있는 여건이 형성될 수 있는 기간이다.
결과적으로 휴지기간에는 i) 중이온가속기용 극저온시스템의 운전 경험을 통합하여 정리(문
서화)하는 업무가 집중적으로 이루어지며, ii) 유지보수 등을 위한 극저온시스템의 제어와 iii)
유지보수 등을 위한 극저온시스템의 주요 수치 기록 및 분석이 병행되어야 한다.
3.2 운전 대상 및 장소
문서번호 : -
2025년도 중이온가속기용 극저온시스템
개정번호 : 0
운전 용역 과업지시서 발 행 일 : 2024. 10. 30
페 이 지 : 12/19
매뉴얼을 작성하고 개정하는 업무를 수행해야 한다. 또한, 과업 및 운전 인력 감독자가 1인
이상 운전실에 배치되어 상기 업무들을 감독해야 한다. 계약자가 휴지기간에 수행해야 하는
운전 업무 상세 내용은 하위 절에 기술된 내용을 따른다. 본 과업지시서에 기술하지 않은
휴지기간의 운전 업무는 연구원과 계약자가 협의하여 수행 여부를 결정한다.
3.4.1 유지보수 등을 위한 극저온시스템 제어
극저온시스템의 제어는 극저온시스템이 휴지지간에 안전하고 안정적으로 운영될 수 있도록
극저온시스템 또는 극저온시스템 관련 부품들을 제어시스템을 활용하여 또는 현장에서 손으
로 조작하는 행위를 의미한다. 계약자는 극저온시스템이 본 목적에 부합하도록 극저온시스
템의 제어를 수행해야 한다. 표 8.에 극저온시스템 제어 업무에 해당되는 사항들을 기술하
였다.
휴지기간의 극저온시스템 제어는 i) 극저온시스템에 대한 충분한 이해를 바탕으로 ii) 조작
후 유지보수 등 현장 활동에 미치는 영향을 충분히 검토한 뒤 이루어져야 한다. 더불어, iii)
조작 대상물이 위치하고 있는 현장에 인력이 존재하는지 여부를 확인하고, iv) 현장에 인력
이 존재할 경우, 조작 대상물을 조작했을 때 발생할 수 있는 상황에 대해 해당 인력의 안전
이 확보된 뒤에 조작을 수행해야 한다. 마지막으로 v) 조작 대상/조작 상태에 대한 혼동이
없도록 각별히 주의를 하여 제어가 진행되어야 한다.


---

### retriever B:
3. 과업 내용
3.1 중이온가속기용 극저온시스템 운전 업무의 정의 및 필요성
본 과업에서 사용하는 “중이온가속기용 극저온시스템 운전 업무”의 정의는, “중이온가
속기용 극저온시스템이 안전하고 안정적으로 운영(가동기간뿐만 아니라 휴지기간의 운영 포
함)될 수 있도록 i) 극저온시스템을 모니터링 및 제어하고, ii) 극저온시스템의 주요 수치를
기록 및 분석하며, iii) 중이온가속기용 극저온시스템의 운전 경험을 통합하여 정리(문서화)하
는 업무를 의미한다.
중이온가속기 또는 극저온시스템 가동기간에는 24시간 운전 업무를 필요로 한다. 통상적으
로 가동기간에는 24시간 운전을 위한 교대근무가 이루어지며, i) 극저온시스템의 모니터링
및 제어, ii) 극저온시스템의 주요 수치를 기록 및 분석하는 업무가 주로 이루어진다.
다만, 통상적으로 이러한 교대근무로 인해 각 운전 인력마다 경험할 수 있는 운전 상황이
서로 다르게 되고, 서로 다른 경험을 했던 운전 인력 모두가 참여하여 각자의 iii) 운전 경험
을 통합/정리할 수 있는 여건이 형성될 수 없다는 한계가 존재한다. 이러한 운전 경험의 통
합 정리는 후술하는 휴지기간에의 운전업무에 집중적으로 수행 가능하다.
통상적으로 휴지기간에는 극저온시스템의 유지보수 등이 진행되며, 유지보수 등을 위한 제
어, 유지보수 등의 경과 및 결과 확인을 위한 극저온시스템 모니터링, 유지보수 등의 결과
확인을 위한 주요 수치 기록 및 분석 또한 필수적인 운전 업무이다. 휴지기간에는 통상적으
로 주간 근무로 운전이 이루어진다. 이와 같이 주간 근무가 이루어질 수 있는 휴지기간의
제외한다.
표 3. 2025년 중이온가속기용 극저온시스템 운전 용역 대상 장치 목록
구분 장치명 장소 비고
극저온 극저온설비동/ 압축기, 재생시스템, 2 K 시스템
SCL3 극저온플랜트
플랜트 헬륨압축기동 등 부대시설 일체
문서번호 : -
2025년도 중이온가속기용 극저온시스템
개정번호 : 0
운전 용역 과업지시서 발 행 일 : 2024. 10. 30
페 이 지 : 8/19
극저온설비동/ 압축기, 재생시스템, 2 K 시스템
SCL2 극저온플랜트
헬륨압축기동 등 부대시설 일체
TBx 극저온설비동
밸브박스, 초전도가속관 저온유지
SCL3 터널/
극저온
SCL3 LINAC 모듈 및 밸브박스 진공단열용 진
ISOL 구간
분배
공펌프
시스템 밸브박스, 초전도가속관 저온유지
SCL2 터널/
SCL2 LINAC 모듈 및 밸브박스 진공단열용 진
IF 구간
공펌프
외부 정화기 외부
회수 압축시스템 헬륨압축기동
회수
가스백 헬륨압축기동
시스템
헬륨 탱크 외부 기체 헬륨량 관리 포함
액체 질소 시스템 외부 액체 질소량 관리 포함
3.3 중이온가속기 또는 극저온시스템 가동기간의 운전 업무
계약자는 중이온가속기 또는 극저온시스템 가동기간 동안 24시간 운전업무가 이루어질 수
있도록 운전 인력을 제공해야 한다. 24시간 운전 업무는 2인 1조를 기본으로 하며, 주간에
는 과업 및 운전 인력 감독자가 1인 이상 운전실에 존재해야 한다. 계약자가 가동기간에 수
행해야 하는 운전 업무 상세 내용은 하위 절에 기술된 내용을 따른다. 본 과업지시서에 기
술하지 않은 가동기간의 운전 업무는 연구원과 계약자가 협의하여 수행 여부를 결정한다.


---

### openai answer
retriever A 점수: 3점
(가동기간의 24시간 교대근무와 휴지기간의 주간 운전(유지보수 중심)을 모두 명시)

retriever B 점수: 3점
(가동기간 24시간 교대근무와 휴지기간 주간 운전을 명확히 포함) 

============================================================
### query:
극저온시스템 과업지시서에 따르면, 가동기간 중 교대근무 인력이 개별적으로 얻은 운전 경험을 통합하여 정리하기 어려운 이유는 무엇이며, 이 업무는 주로 언제 수행되어야 합니까?

### answer:
교대근무로 인해 각 운전 인력마다 경험하는 운전 상황이 서로 달라, 모든 인원이 함께 모여 운전 경험을 통합하고 정리할 여건이 형성되기 어렵기 때문입니다. 이러한 운전 경험 통합 및 정리(문서화) 업무는 주로 주간 근무가 이루어지는 '휴지기간'에 집중적으로 수행됩니다.

---

### retriever A:
중이온가속기 또는 극저온시스템 가동기간에는 24시간 운전 업무를 필요로 한다. 통상적으
로 가동기간에는 24시간 운전을 위한 교대근무가 이루어지며, i) 극저온시스템의 모니터링
및 제어, ii) 극저온시스템의 주요 수치를 기록 및 분석하는 업무가 주로 이루어진다.
다만, 통상적으로 이러한 교대근무로 인해 각 운전 인력마다 경험할 수 있는 운전 상황이
서로 다르게 되고, 서로 다른 경험을 했던 운전 인력 모두가 참여하여 각자의 iii) 운전 경험
을 통합/정리할 수 있는 여건이 형성될 수 없다는 한계가 존재한다. 이러한 운전 경험의 통
합 정리는 후술하는 휴지기간에의 운전업무에 집중적으로 수행 가능하다.
통상적으로 휴지기간에는 극저온시스템의 유지보수 등이 진행되며, 유지보수 등을 위한 제
어, 유지보수 등의 경과 및 결과 확인을 위한 극저온시스템 모니터링, 유지보수 등의 결과
확인을 위한 주요 수치 기록 및 분석 또한 필수적인 운전 업무이다. 휴지기간에는 통상적으
로 주간 근무로 운전이 이루어진다. 이와 같이 주간 근무가 이루어질 수 있는 휴지기간의
특성 상, 휴지기간은 가동기간에서 교대근무로 각 운전 인력 개인이 얻은 단편적인 운전 경
험을 통합하여 정리(문서화)할 수 있는 여건이 형성될 수 있는 기간이다.
결과적으로 휴지기간에는 i) 중이온가속기용 극저온시스템의 운전 경험을 통합하여 정리(문
서화)하는 업무가 집중적으로 이루어지며, ii) 유지보수 등을 위한 극저온시스템의 제어와 iii)
유지보수 등을 위한 극저온시스템의 주요 수치 기록 및 분석이 병행되어야 한다.
3.2 운전 대상 및 장소
다. 계약자는 본 과업을 수행하기 위해 적절한 지식 또는 능력을 보유하고 있는 적정 수의 운
전 인력을 연구원에게 제공하여야 한다.
• 가동기간의 극저온시스템 운전 업무 (24시간 운전)
- 극저온시스템 모니터링
- 극저온시스템 주요 수치 기록 및 분석
- 극저온시스템 제어
- 기타 가동기간에 필요한 운전 업무
• 휴지기간의 극저온시스템 운전 업무 (주간 운전)
- 유지보수 등을 위한 극저온시스템 제어
- 유지보수 등을 위한 극저온시스템 모니터링
- 유지보수 등을 위한 극저온시스템 주요 수치 기록 및 분석
- 극저온시스템 운전 매뉴얼 작성 및 개정
- 극저온시스템 운전 교육
- 성과보고회 개최
- 기타 휴지기간에 필요한 운전 업무
• 본 과업 및 본 과업을 위해 투입된 인력에 대한 관리감독
본 과업의 기간은 2025년 1월 1일부터 2025년 12월 31일까지로 하며, 2025년 동안 가동기
간의 운전 업무를 수행하는 것을 기준으로 한다. 단, 표 2.와 같이 연구원의 운영 추진 상황
에 따라 휴지기간의 운전업무를 수행하는 것으로 변경될 수 있다. 가동기간과 휴지기간의
운전 업무는 “3. 과업 내용” 하위 절에 기술한 내용에 따른다.
표 2. 과업 추진 일정(안)
2025년
구분
1분기 2분기 3분기 4분기
SCL3 극저온플랜트
SCL3 중이온가속기 SCL3 구간 운전 SCL3 구간 유지보수
시운전 준비 및
구간 (가동기간) (휴지기간)
시운전 (가동기간)
SCL2 SCL2 극저온플랜트 중이온가속기 SCL2 구간 시운전 SCL2 구간 유지보수
구간 시운전 (가동기간) (가동기간) (휴지기간)
회수


---

### retriever B:
3. 과업 내용
3.1 중이온가속기용 극저온시스템 운전 업무의 정의 및 필요성
본 과업에서 사용하는 “중이온가속기용 극저온시스템 운전 업무”의 정의는, “중이온가
속기용 극저온시스템이 안전하고 안정적으로 운영(가동기간뿐만 아니라 휴지기간의 운영 포
함)될 수 있도록 i) 극저온시스템을 모니터링 및 제어하고, ii) 극저온시스템의 주요 수치를
기록 및 분석하며, iii) 중이온가속기용 극저온시스템의 운전 경험을 통합하여 정리(문서화)하
는 업무를 의미한다.
중이온가속기 또는 극저온시스템 가동기간에는 24시간 운전 업무를 필요로 한다. 통상적으
로 가동기간에는 24시간 운전을 위한 교대근무가 이루어지며, i) 극저온시스템의 모니터링
및 제어, ii) 극저온시스템의 주요 수치를 기록 및 분석하는 업무가 주로 이루어진다.
다만, 통상적으로 이러한 교대근무로 인해 각 운전 인력마다 경험할 수 있는 운전 상황이
서로 다르게 되고, 서로 다른 경험을 했던 운전 인력 모두가 참여하여 각자의 iii) 운전 경험
을 통합/정리할 수 있는 여건이 형성될 수 없다는 한계가 존재한다. 이러한 운전 경험의 통
합 정리는 후술하는 휴지기간에의 운전업무에 집중적으로 수행 가능하다.
통상적으로 휴지기간에는 극저온시스템의 유지보수 등이 진행되며, 유지보수 등을 위한 제
어, 유지보수 등의 경과 및 결과 확인을 위한 극저온시스템 모니터링, 유지보수 등의 결과
확인을 위한 주요 수치 기록 및 분석 또한 필수적인 운전 업무이다. 휴지기간에는 통상적으
로 주간 근무로 운전이 이루어진다. 이와 같이 주간 근무가 이루어질 수 있는 휴지기간의
는 과업 및 운전 인력 감독자가 1인 이상 운전실에 존재해야 한다. 계약자가 가동기간에 수
행해야 하는 운전 업무 상세 내용은 하위 절에 기술된 내용을 따른다. 본 과업지시서에 기
술하지 않은 가동기간의 운전 업무는 연구원과 계약자가 협의하여 수행 여부를 결정한다.
3.3.1 극저온시스템 모니터링
계약자는 가동기간 동안 극저온시스템이 안정적으로 가동될 수 있도록 모니터링을 해야 한


---

### openai answer
retriever A 점수: 3점
(교대근무로 경험이 제각각이고 전원이 함께 모여 통합/정리할 여건이 없다는 이유와, 운전 경험 통합·정리는 휴지기간(주간 근무)에 집중 수행돼야 함이 명시되어 있어 정답의 근거가 모두 포함됨)

retriever B 점수: 3점
(교대근무로 인해 경험 통합 여건이 형성되기 어렵다는 이유와, 해당 업무가 휴지기간에 주로 수행되며 휴지기간은 주간 근무로 운영된다는 내용이 포함되어 정답 근거가 모두 있음) 

============================================================
### query:
계약업체가 매월 운전 현황을 정리해서 연구원에 보고해야 하는 공식적인 문서는 무엇이며, 제출 시점은 언제입니까?

### answer:
매월 운전 상황을 요약하여 제출해야 하는 문서는 '월간보고서'이며, 매월 기성 비용을 신청할 때 제출해야 합니다.

---

### retriever A:
계약종료 및 인수인계에 관한 사항
세부내용
○ 수행업체는 본 사업 종료 1개월 전 시스템의 발전 및 안정성을 유지하기 위한 인수인계 계획을 제시하여야 함
- 본 사업수행 기간 동안 취득한 모든 정보 및 생상된 산출물 일체
- 인수인계의 주요 업무, 상세 일정 및 인수인계 대상 주요 산출물 제시
- 인수인계 기간 동안 발생하는 리스크에 대한 방지대책 제시
○ 수행업체는 계약기간 만료 시 차기 수행업체로 선정된 사업자가 본 사업의 업무를 차질없이 수행할 수 있도록 요청 시 통합접수시스템 운영 업무 수행에 필요한 중요사항 등을 인계하여야 하며, 시스템의 효율적인 운영을 위해 하자담보 책임기간 중에는 기술자문에 응하여야 함
- 시스템 구성도, 시스템 현황, DB권한 등
- 통합접수시스템 기동/종료 절차
- 타 시스템 연계 및 적재 절차
- 일반 사용자 및 관리자 매뉴얼 등
○ 인수 및 인계 기간은 차기 시스템 운영 사업 개시일로부터 30일 이내로 하며, 인수 및 인계기간 동안의 용역비용은 지급하지 아니함
산출 정보
인수인계 계획서(사업수행계획서 내 포함 가능)
관련 요구사항
요구사항 출처
요구사항 분류
프로젝트지원 요구사항
요구사항 고유번호
PSR-008
요구사항 명칭
정보화사업 주요점검 및 수행지원
요구사항
상세설명
정의
정보화사업 주요점검 및 수행지원
세부내용
○ 통합접수시스템 관련 점검사항 준수 및 현장점검 수행
○ 보안관리 실태 점검 대응
○ 공공데이터 품질관리(DB) 수준평가 대응 및 데이터 오류 정제 활동 실시
산출 정보
점검별 결과보고서
관련 요구사항
요구사항 출처
□ 품질관리 요구사항
요구사항 분류
· 월별 추진 실적 보고
· 관련부서 업무협의 내용
· 문제점 보고 및 특이사항
매월 말일
비정기
보고
인수인계
보고
· 신규 사업자 선정시 업무인계 결과보고
사업종료
1주전
기타
· 이슈사항 및 문제점 협의 등
이슈발생시
※ 보고형식 및 주기는 추후 협의하여 조정할 수 있음
ㅇ 계약 기간 만료일 전 과업 완료 보고서를 제출하여야 함
구분
보고명
보고내용
시기
사업
종료
완료
보고서
· 용역종료 시 전체 추진단계의 수행결과 보고로 인쇄물 2부, 외장하드 1본으로 제출
· 산출물을 종류별로 분류한 파일
용역종료일
2주일 전
- 용역 완료시 제출하는 최종 결과물(프로그램 소스 코드(주석, 설명서 포함), ERD, 테이블 정의서, 프로그램 정의서 등)은 발주자의 승인을 득한 후 원본 파일을 CD 또는 HDD에 수록하여 산출물과 같이 제출하여야 하고, 프로그램 소스에 대한 형상관리가 가능하도록 협조하여야 함
- 산출물은 기존 산출물을 기반으로 현행화 또는 추가 작성해야 하며, 상호 협의하여 조정 가능
ㅇ 제출 시기에 따라 종류별 산출물을 제출하되, 사업관리 및 검수를 위한 각종 업무 보고를 요청할 경우, 신속하게 작성하여 보고하는 수시보고 체계 마련 ※ 제출 전 주관기관 사전 검토 및 확인
산출정보
착수보고서, 완료보고서, 월간보고서
요구사항 고유번호
PMR-005
요구사항 명칭
인계 조건
요구사항
상세설명
정의
관한 사항
세부
내용
인계 요건
- 사업종료 1개월 전까지 업무인계 계획서를 작성해 진흥원의 승인을 받아야함
- 발주기관과 월 1회 이상 기술이전 등에 대한 협의를 하고, 그 결과를 문서화


---

### retriever B:
4. 제출문서
계약자는 표 13.와 같은 문서들을 제출일정에 맞춰 연구원에 제출해야 한다.
표 13. 제출문서 목록 및 관련 사항
연구원
제출문서 내용 비고 및 제출일
승인
투입 인력
- 투입 인력 이력 사항 X - 계약 시
이력서
- 최초: 계약 후 1개월 내
작업계획서 - 일별 투입 인력 - 이후: 매주 금요일까지 차주
X
(주간근무표) - 휴가상황 근무표 제출
- 문서 파일 제출
- IBS에서 제공한 양식 파일
극저온시스템 - 매 월간 기성 신청 시 제출
주요 수치 - 극저온시스템 주요 수치 X - 문서 파일 제출
기록문서 - 수치 기록을 위해 계약자가
준비한 PC에 상시 저장
일일근무일지 - 일일 운영 상황 기록 O - 매 월간 기성 신청 시 제출
- 문서 파일 제출
- 수치 기록을 위해 계약자가
인수인계 문서 - (가동기간) 인수인계 내용 X 준비한 PC에 상시 저장
- 수리 사유 발생일
- 매 월간 기성 신청 시 제출
- 수리 필요 대상
수리필요사항 - 문서 파일 제출
- 상세 내용 O
기록 문서 - 수치 기록을 위해 계약자가
- 발견자
준비한 PC에 상시 저장
- 기타 필요한 내용
- 발생한 경우, 기성 신청 시
특이사항, 위험상황 발생 시,
제출 (해당 월 발생 건)
- 상황 내용
- 문서 파일 제출
- 작성자
Lesson learn O - 운전 인력 전원 회람 후 확
- 금회 대처 방법
인 서명 필수
- 향후 대처 방법
- 수치 기록을 위해 계약자가
- 기타 필요한 내용
준비한 PC에 상시 저장
- 매 월간 기성 신청 시 제출
월간보고서 - 월간 운전 상황 요약 O
- 문서 파일 제출
라. 계약 후 제출서류 : 사업수행계획서, 기술적용계획표
1) 사업수행자는 계약일로부터 14일 이내에 사업수행계획서와 기술적용계획표를 제출하고 사업 착수를 완료하여야 함
2) 사업수행계획서에는 제안요청서, 제안서, 협상에 의해 수정·보완·추가된 내용을 반영하여 제출하여야 하며 발주기관의 승인을 받아야 함
제출방법
1
착수계
A4
1
계약 후 7일 이내
서신문서
2
보안관련 문서
(보안서약서, 비밀유지서약서, 개인정보 위탁계약서, 개인정보 보호서약서)
A4
1
계약 후 7일 이내
3
사업수행계획서
A4
8
계약 후 10일 이내
4
착수보고서
A4
8
착수 후 10일 이내
5
완료보고서
A4
8
계약종료 2주 전 까지
6
최종산출물
(개발 산출물, 운영자 매뉴얼, 사용자 매뉴얼, 각종 분석/측정 결과보고서, 기술지원 확약서, 품질관리보고서, 기술적용계획/결과표, 기술이전계획서, 장애대응 매뉴얼, 테스트 수행 계획서/결과서 등)
USB
2
계약종료 2주 전 까지


---

### openai answer
retriever A 점수: 2점
- ‘월간보고서’와 월별 보고가 산출정보로 언급되어 문서 종류는 파악 가능하나, ‘운전 현황’ 및 제출 시점 ‘매 월간 기성 신청 시’ 정보가 없어 답의 일부만 포함됨

retriever B 점수: 3점
- ‘월간보고서’가 공식 문서로 명시되고, ‘월간 운전 상황 요약’ 내용과 제출 시점 ‘매 월간 기성 신청 시 제출’이 정확히 제시되어 답을 완전히 포함함 

## hybrid_results

============================================================
### query:
본 과업지시서에서 '극저온시스템'은 무엇을 포함하는 것으로 정의하고 있습니까?

### answer:
'극저온시스템'은 SCL3/SCL2 극저온플랜트 및 부대설비, SCL3/SCL2 분배시스템 및 부대설비, SCL3/SCL2 초전도가속관 저온유지모듈, 기체헬륨 탱크, 액체질소 시스템 및 극저온시스템용 헬륨회수시스템을 모두 포함하는 것으로 정의됩니다.

---

### retriever A:
7. 안전관리 .............................................................................................................. 19
8. 붙임 자료 ........................................................................................................... 19
문서번호 : -
2025년도 중이온가속기용 극저온시스템
개정번호 : 0
운전 용역 과업지시서 발 행 일 : 2024. 10. 30
페 이 지 : 5/19
1. 서 론
1.1 목 적
기초과학연구원 중이온가속기연구소(이하 “연구원”)에서 운영중인 중이온가속기용 극저
온시스템(이하 “극저온시스템”)은 고도의 공정 시스템으로, 안전하고 안정적인 중이온가속
기의 운영을 위해 상시 모니터링이 필요하다. 이에 본 과업지시서는 극저온시스템의 운전
용역에 대한 상세 과업 내용을 기술한다.
1.2 용어 설명
본 과업지시서에서 사용하는 용어에 대한 상세 내용은 표 1.과 같다. 계약자는 반드시 해당
용어의 의미를 충분히 이해하고 숙지해야 하며, 용어 미숙지로 인한 책임은 계약자에게 있
다.
표 1. 본 과업지시서에 사용되는 용어 설명
용어 설명
연구원 중이온가속기연구소
계약자 본 과업을 수행할 계약상대자
SCL3/SCL2 극저온플랜트 및 부대설비,
SCL3/SCL2 분배시스템 및 부대설비,
SCL3/SCL2 초전도가속관 저온유지모듈,
극저온시스템
기체헬륨 탱크, 액체질소 시스템 및
TBx 극저온설비동
밸브박스, 초전도가속관 저온유지
SCL3 터널/
극저온
SCL3 LINAC 모듈 및 밸브박스 진공단열용 진
ISOL 구간
분배
공펌프
시스템 밸브박스, 초전도가속관 저온유지
SCL2 터널/
SCL2 LINAC 모듈 및 밸브박스 진공단열용 진
IF 구간
공펌프
외부 정화기 외부
회수 압축시스템 헬륨압축기동
회수
가스백 헬륨압축기동
시스템
헬륨 탱크 외부 기체 헬륨량 관리 포함
액체 질소 시스템 외부 액체 질소량 관리 포함
3.3 중이온가속기 또는 극저온시스템 가동기간의 운전 업무
계약자는 중이온가속기 또는 극저온시스템 가동기간 동안 24시간 운전업무가 이루어질 수
있도록 운전 인력을 제공해야 한다. 24시간 운전 업무는 2인 1조를 기본으로 하며, 주간에
는 과업 및 운전 인력 감독자가 1인 이상 운전실에 존재해야 한다. 계약자가 가동기간에 수
행해야 하는 운전 업무 상세 내용은 하위 절에 기술된 내용을 따른다. 본 과업지시서에 기
술하지 않은 가동기간의 운전 업무는 연구원과 계약자가 협의하여 수행 여부를 결정한다.
3.3.1 극저온시스템 모니터링
계약자는 가동기간 동안 극저온시스템이 안정적으로 가동될 수 있도록 모니터링을 해야 한
다. 가동기간의 모니터링 업무는 극저온시스템이 안정적으로 가동되고 있는지 여부를 확인
하기 위해 극저온시스템을 지속적으로 관찰하는 업무를 의미하며, 표 4.에 가동기간의 모니
터링 업무에 해당되는 내용들을 기술하였다.
극저온시스템의 모니터링은 연구원에 구축되어있는 극저온시스템용 제어시스템(이하 “제


---

### retriever B:
1. 서 론
1.1 목 적
기초과학연구원 중이온가속기연구소(이하 “연구원”)에서 운영중인 중이온가속기용 극저
온시스템(이하 “극저온시스템”)은 고도의 공정 시스템으로, 안전하고 안정적인 중이온가속
기의 운영을 위해 상시 모니터링이 필요하다. 이에 본 과업지시서는 극저온시스템의 운전
용역에 대한 상세 과업 내용을 기술한다.
1.2 용어 설명
본 과업지시서에서 사용하는 용어에 대한 상세 내용은 표 1.과 같다. 계약자는 반드시 해당
용어의 의미를 충분히 이해하고 숙지해야 하며, 용어 미숙지로 인한 책임은 계약자에게 있
다.
표 1. 본 과업지시서에 사용되는 용어 설명
용어 설명
연구원 중이온가속기연구소
계약자 본 과업을 수행할 계약상대자
SCL3/SCL2 극저온플랜트 및 부대설비,
SCL3/SCL2 분배시스템 및 부대설비,
SCL3/SCL2 초전도가속관 저온유지모듈,
극저온시스템
기체헬륨 탱크, 액체질소 시스템 및
극저온시스템용 헬륨회수시스템 포함
(“3.2절 운전대상”, 표 3. 참고)
중이온가속기 또는 극저온시스템이 냉각(냉각 착수
가동기간
이후) 또는 빔서비스 등을 위해 가동 중인 기간
중이온가속기 또는 극저온시스템이 유지보수 등을
휴지기간
위해 가동을 멈추고 휴지 중인 기간
중이온가속기용 극저온시스템이 가동기간 및 휴지
기간에 안전하고 안정적으로 운영될 수 있도록,
중이온가속기용 극저온시스템 i) 극저온시스템을 모니터링 및 제어하고,
운전 업무 ii) 극저온시스템의 주요 수치를 기록 및 분석하며,
iii) 중이온가속기용 극저온시스템의 운전 경험을
통합하여 정리(문서화)하는 업무
문서번호 : -
문서번호 : -
2025년도 중이온가속기용 극저온시스템
개정번호 : 0
운전 용역 과업지시서 발 행 일 : 2024. 10. 30
페 이 지 : 12/19
매뉴얼을 작성하고 개정하는 업무를 수행해야 한다. 또한, 과업 및 운전 인력 감독자가 1인
이상 운전실에 배치되어 상기 업무들을 감독해야 한다. 계약자가 휴지기간에 수행해야 하는
운전 업무 상세 내용은 하위 절에 기술된 내용을 따른다. 본 과업지시서에 기술하지 않은
휴지기간의 운전 업무는 연구원과 계약자가 협의하여 수행 여부를 결정한다.
3.4.1 유지보수 등을 위한 극저온시스템 제어
극저온시스템의 제어는 극저온시스템이 휴지지간에 안전하고 안정적으로 운영될 수 있도록
극저온시스템 또는 극저온시스템 관련 부품들을 제어시스템을 활용하여 또는 현장에서 손으
로 조작하는 행위를 의미한다. 계약자는 극저온시스템이 본 목적에 부합하도록 극저온시스
템의 제어를 수행해야 한다. 표 8.에 극저온시스템 제어 업무에 해당되는 사항들을 기술하
였다.
휴지기간의 극저온시스템 제어는 i) 극저온시스템에 대한 충분한 이해를 바탕으로 ii) 조작
후 유지보수 등 현장 활동에 미치는 영향을 충분히 검토한 뒤 이루어져야 한다. 더불어, iii)
조작 대상물이 위치하고 있는 현장에 인력이 존재하는지 여부를 확인하고, iv) 현장에 인력
이 존재할 경우, 조작 대상물을 조작했을 때 발생할 수 있는 상황에 대해 해당 인력의 안전
이 확보된 뒤에 조작을 수행해야 한다. 마지막으로 v) 조작 대상/조작 상태에 대한 혼동이
없도록 각별히 주의를 하여 제어가 진행되어야 한다.


---

### openai answer
retriever A 점수: 2점
(표 1에서 SCL3/SCL2 극저온플랜트·분배시스템·저온유지모듈, 기체헬륨 탱크, 액체질소 시스템까지는 명시되나 ‘극저온시스템용 헬륨회수시스템’ 포함 문구가 확인되지 않음)

retriever B 점수: 3점
(표 1에서 SCL3/SCL2 극저온플랜트·분배시스템·저온유지모듈, 기체헬륨 탱크, 액체질소 시스템 및 ‘극저온시스템용 헬륨회수시스템 포함’까지 모두 명시) 

============================================================
### query:
극저온시스템 과업 추진 일정에 따르면, 4분기에 SCL3 구간에 대해 예정된 주요 업무는 무엇이며, 이때의 업무 기간은 '가동기간'과 '휴지기간' 중 어느 것에 해당합니까?

### answer:
2025년 4분기에 SCL3 구간에 예정된 주요 업무는 '유지보수'이며, 이 기간은 '휴지기간'에 해당합니다.

---

### retriever A:
다. 계약자는 본 과업을 수행하기 위해 적절한 지식 또는 능력을 보유하고 있는 적정 수의 운
전 인력을 연구원에게 제공하여야 한다.
• 가동기간의 극저온시스템 운전 업무 (24시간 운전)
- 극저온시스템 모니터링
- 극저온시스템 주요 수치 기록 및 분석
- 극저온시스템 제어
- 기타 가동기간에 필요한 운전 업무
• 휴지기간의 극저온시스템 운전 업무 (주간 운전)
- 유지보수 등을 위한 극저온시스템 제어
- 유지보수 등을 위한 극저온시스템 모니터링
- 유지보수 등을 위한 극저온시스템 주요 수치 기록 및 분석
- 극저온시스템 운전 매뉴얼 작성 및 개정
- 극저온시스템 운전 교육
- 성과보고회 개최
- 기타 휴지기간에 필요한 운전 업무
• 본 과업 및 본 과업을 위해 투입된 인력에 대한 관리감독
본 과업의 기간은 2025년 1월 1일부터 2025년 12월 31일까지로 하며, 2025년 동안 가동기
간의 운전 업무를 수행하는 것을 기준으로 한다. 단, 표 2.와 같이 연구원의 운영 추진 상황
에 따라 휴지기간의 운전업무를 수행하는 것으로 변경될 수 있다. 가동기간과 휴지기간의
운전 업무는 “3. 과업 내용” 하위 절에 기술한 내용에 따른다.
표 2. 과업 추진 일정(안)
2025년
구분
1분기 2분기 3분기 4분기
SCL3 극저온플랜트
SCL3 중이온가속기 SCL3 구간 운전 SCL3 구간 유지보수
시운전 준비 및
구간 (가동기간) (휴지기간)
시운전 (가동기간)
SCL2 SCL2 극저온플랜트 중이온가속기 SCL2 구간 시운전 SCL2 구간 유지보수
구간 시운전 (가동기간) (가동기간) (휴지기간)
회수
2025년
구분
1분기 2분기 3분기 4분기
SCL3 극저온플랜트
SCL3 중이온가속기 SCL3 구간 운전 SCL3 구간 유지보수
시운전 준비 및
구간 (가동기간) (휴지기간)
시운전 (가동기간)
SCL2 SCL2 극저온플랜트 중이온가속기 SCL2 구간 시운전 SCL2 구간 유지보수
구간 시운전 (가동기간) (가동기간) (휴지기간)
회수
필요시 간헐적 운전 (휴지기간에 준함)
시스템
문서번호 : -
2025년도 중이온가속기용 극저온시스템
개정번호 : 0
운전 용역 과업지시서 발 행 일 : 2024. 10. 30
페 이 지 : 7/19
3. 과업 내용
3.1 중이온가속기용 극저온시스템 운전 업무의 정의 및 필요성
본 과업에서 사용하는 “중이온가속기용 극저온시스템 운전 업무”의 정의는, “중이온가
속기용 극저온시스템이 안전하고 안정적으로 운영(가동기간뿐만 아니라 휴지기간의 운영 포
함)될 수 있도록 i) 극저온시스템을 모니터링 및 제어하고, ii) 극저온시스템의 주요 수치를
기록 및 분석하며, iii) 중이온가속기용 극저온시스템의 운전 경험을 통합하여 정리(문서화)하
는 업무를 의미한다.
중이온가속기 또는 극저온시스템 가동기간에는 24시간 운전 업무를 필요로 한다. 통상적으
로 가동기간에는 24시간 운전을 위한 교대근무가 이루어지며, i) 극저온시스템의 모니터링
및 제어, ii) 극저온시스템의 주요 수치를 기록 및 분석하는 업무가 주로 이루어진다.
다만, 통상적으로 이러한 교대근무로 인해 각 운전 인력마다 경험할 수 있는 운전 상황이


---

### retriever B:
다. 계약자는 본 과업을 수행하기 위해 적절한 지식 또는 능력을 보유하고 있는 적정 수의 운
전 인력을 연구원에게 제공하여야 한다.
• 가동기간의 극저온시스템 운전 업무 (24시간 운전)
- 극저온시스템 모니터링
- 극저온시스템 주요 수치 기록 및 분석
- 극저온시스템 제어
- 기타 가동기간에 필요한 운전 업무
• 휴지기간의 극저온시스템 운전 업무 (주간 운전)
- 유지보수 등을 위한 극저온시스템 제어
- 유지보수 등을 위한 극저온시스템 모니터링
- 유지보수 등을 위한 극저온시스템 주요 수치 기록 및 분석
- 극저온시스템 운전 매뉴얼 작성 및 개정
- 극저온시스템 운전 교육
- 성과보고회 개최
- 기타 휴지기간에 필요한 운전 업무
• 본 과업 및 본 과업을 위해 투입된 인력에 대한 관리감독
본 과업의 기간은 2025년 1월 1일부터 2025년 12월 31일까지로 하며, 2025년 동안 가동기
간의 운전 업무를 수행하는 것을 기준으로 한다. 단, 표 2.와 같이 연구원의 운영 추진 상황
에 따라 휴지기간의 운전업무를 수행하는 것으로 변경될 수 있다. 가동기간과 휴지기간의
운전 업무는 “3. 과업 내용” 하위 절에 기술한 내용에 따른다.
표 2. 과업 추진 일정(안)
2025년
구분
1분기 2분기 3분기 4분기
SCL3 극저온플랜트
SCL3 중이온가속기 SCL3 구간 운전 SCL3 구간 유지보수
시운전 준비 및
구간 (가동기간) (휴지기간)
시운전 (가동기간)
SCL2 SCL2 극저온플랜트 중이온가속기 SCL2 구간 시운전 SCL2 구간 유지보수
구간 시운전 (가동기간) (가동기간) (휴지기간)
회수
일정
구분
추진 일정
M
M+1
M+2
M+3
M+4
사업발주 및 사업자 선정
업무 협의
시스템
구축
기존 시스템 분석
업무(요구사항) 분석
설계 및 구축
시스템 구현 및 개발
시스템테스트 및 수정
시스템 가동 및 운영(안정화)
※ 위 추진일정은 사업 추진과정에서 조정 가능함
Ⅲ
다. 추진일정
구 분
2024년~2025년
M1
M2
M3
M4
M5
M6
◦ 시스템 개발
- 분석
- 설계/구현
- 단위/통합 시험
◦ 연계시스템 기능개선
◦ 운영개시 및 안정화
◦ 보고회
- 착수, 중간, 최종보고
★
★
★
* 추진과정에서 일정은 변경될 수 있음
Ⅱ
일반 현황


---

### openai answer
retriever A 점수: 3점
- 표 2. 과업 추진 일정(안)에 2025년 4분기 SCL3 구간이 ‘유지보수(휴지기간)’로 명시되어 있어, 질문의 답(업무: 유지보수, 기간: 휴지기간)이 모두 포함됨.

retriever B 점수: 3점
- 표 2. 과업 추진 일정(안)에 동일하게 2025년 4분기 SCL3 구간 ‘유지보수(휴지기간)’가 명시되어 있어, 필요한 정보가 모두 포함됨. 

============================================================
### query:
극저온시스템 '가동기간' 동안 계약자가 24시간 수행해야 할 극저온시스템 모니터링 업무에는 어떤 것들이 있습니까? (3가지 이상)

### answer:
가동기간 동안 수행해야 할 모니터링 업무는 다음과 같습니다.
알람 메시지 확인
주요 센서(온도, 압력, 유량 등)의 트렌드 확인
정상 운전 상황의 주요 센서 지시값 파악 및 숙지
비정상 운전 상황 감지
제어로직 정상 작동 여부 확인

---

### retriever A:
다. 가동기간의 모니터링 업무는 극저온시스템이 안정적으로 가동되고 있는지 여부를 확인
하기 위해 극저온시스템을 지속적으로 관찰하는 업무를 의미하며, 표 4.에 가동기간의 모니
터링 업무에 해당되는 내용들을 기술하였다.
극저온시스템의 모니터링은 연구원에 구축되어있는 극저온시스템용 제어시스템(이하 “제
어시스템”)을 활용하여 또는 현장에 설치되어있는 장치를 통해 수행한다. 연구원은 구축되
어있는 제어시스템을 활용하기 위한 컴퓨터, 모니터, 키보드, 마우스, 책상 및 의자를 계약
자에게 제공한다. 현재 연구원에 구축되어있는 제어시스템은 표 5.과 같다.
연구원은 계약자에게 현재 구축되어있는 제어시스템에 대한 기본 사용 교육을 제공하며,
계약자는 연구원이 제공하는 교육을 이해하기 위한 배경지식을 보유해야 한다.
문서번호 : -
2025년도 중이온가속기용 극저온시스템
개정번호 : 0
운전 용역 과업지시서 발 행 일 : 2024. 10. 30
페 이 지 : 9/19
표 4. 가동기간의 극저온시스템 모니터링 업무 리스트
순번 극저온시스템 모니터링 업무 비고
1 알람 메시지 확인
2 주요 센서 트렌드 확인 온도, 압력, 유량, 속도 등
3 정상 운전 상황의 주요 센서 지시값 파악 및 숙지 온도, 압력, 유량, 속도 등
4 비정상 운전 상황 감지
연구원에서 현장 확인 필요상황 발생 시, 연구원이 지 무전 통신 활용
5
정하는 값 관찰, 확인 및 상황 전달 (연구원에서 무전기 제공)
6 제어로직 정상 작동 여부 확인
연구원 요청 시,
7 현장 상황 일시적 또는 지속적 관찰
비방사선구역 한정
이온가속기용 극저온시스템 운영에 중대한 영향이 생길 경우, 이는 계약자의 책임으로 한
다.
표 8. 휴지기간의 극저온시스템 제어 업무 리스트
순번 극저온시스템 제어 업무 비고
자동/수동(Auto/Manual) 제어 우선
1 계장류 제어 우선권(Priority) 변경
권 설정 변경
문서번호 : -
2025년도 중이온가속기용 극저온시스템
개정번호 : 0
운전 용역 과업지시서 발 행 일 : 2024. 10. 30
페 이 지 : 13/19
자동 밸브 – 제어시스템 활용
2 극저온시스템 자동/수동 밸브 조작
수동 밸브 – 현장에서 손으로 조작
압축기 emergency stop 버튼
3 현장에서 손으로 조작
활성화/비활성화
제어시스템 활용
4 진공펌프 가동/정지
(현장 조작 필요 시 연구원에 통지)
5 센서 오버라이드(Override) 설정 또는 해제 제어시스템 활용
6 계장류 제어 변수값(Ramp speed 등) 변경 제어시스템 활용
7 시퀀스 제어 변수값(Timer 등) 변경 제어시스템 활용
8 시퀀스 시작/종료 제어시스템 활용
현장 확인 및 최종 준비 상태 미진
9 극저온시스템 가동 전 최종 준비 상태 확인
할 경우, 상태 보완 포함
기타 극저온시스템의 안전하고 안정적인 운
10 제어시스템 활용 및 현장 조작 포함
전에 필요한 제어
3.4.2 유지보수 등을 위한 극저온시스템 모니터링
계약자는 휴지기간 동안 극저온시스템이 안정적으로 가동될 수 있도록 모니터링을 해야 한
다. 휴지기간의 모니터링 업무는 i) 유지보수 등을 위한 현장 활동 시 극저온시스템에 특이


---

### retriever B:
다. 가동기간의 모니터링 업무는 극저온시스템이 안정적으로 가동되고 있는지 여부를 확인
하기 위해 극저온시스템을 지속적으로 관찰하는 업무를 의미하며, 표 4.에 가동기간의 모니
터링 업무에 해당되는 내용들을 기술하였다.
극저온시스템의 모니터링은 연구원에 구축되어있는 극저온시스템용 제어시스템(이하 “제
어시스템”)을 활용하여 또는 현장에 설치되어있는 장치를 통해 수행한다. 연구원은 구축되
어있는 제어시스템을 활용하기 위한 컴퓨터, 모니터, 키보드, 마우스, 책상 및 의자를 계약
자에게 제공한다. 현재 연구원에 구축되어있는 제어시스템은 표 5.과 같다.
연구원은 계약자에게 현재 구축되어있는 제어시스템에 대한 기본 사용 교육을 제공하며,
계약자는 연구원이 제공하는 교육을 이해하기 위한 배경지식을 보유해야 한다.
문서번호 : -
2025년도 중이온가속기용 극저온시스템
개정번호 : 0
운전 용역 과업지시서 발 행 일 : 2024. 10. 30
페 이 지 : 9/19
표 4. 가동기간의 극저온시스템 모니터링 업무 리스트
순번 극저온시스템 모니터링 업무 비고
1 알람 메시지 확인
2 주요 센서 트렌드 확인 온도, 압력, 유량, 속도 등
3 정상 운전 상황의 주요 센서 지시값 파악 및 숙지 온도, 압력, 유량, 속도 등
4 비정상 운전 상황 감지
연구원에서 현장 확인 필요상황 발생 시, 연구원이 지 무전 통신 활용
5
정하는 값 관찰, 확인 및 상황 전달 (연구원에서 무전기 제공)
6 제어로직 정상 작동 여부 확인
연구원 요청 시,
7 현장 상황 일시적 또는 지속적 관찰
비방사선구역 한정
는 과업 및 운전 인력 감독자가 1인 이상 운전실에 존재해야 한다. 계약자가 가동기간에 수
행해야 하는 운전 업무 상세 내용은 하위 절에 기술된 내용을 따른다. 본 과업지시서에 기
술하지 않은 가동기간의 운전 업무는 연구원과 계약자가 협의하여 수행 여부를 결정한다.
3.3.1 극저온시스템 모니터링
계약자는 가동기간 동안 극저온시스템이 안정적으로 가동될 수 있도록 모니터링을 해야 한
다. 휴지기간의 모니터링 업무는 i) 유지보수 등을 위한 현장 활동 시 극저온시스템에 특이
사항이 발생하는지 여부를 확인하기 위해 또는 ii) 유지보수 등을 수행한 뒤 그 결과를 확인
하기 위해 극저온시스템을 지속적으로 관찰하는 업무를 의미한다. 표 9.에 휴지기간의 모니
터링 업무에 해당되는 내용들을 기술하였다.
극저온시스템의 모니터링은 연구원에 구축되어있는 극저온시스템용 제어시스템(이하 “제
어시스템”)을 활용하여 또는 현장에 설치되어있는 장치를 통해 수행한다. 연구원은 구축되
어있는 제어시스템을 활용하기 위한 컴퓨터, 모니터, 키보드, 마우스, 책상 및 의자를 계약
자에게 제공한다. 현재 연구원에 구축되어있는 제어시스템은 표 5.과 같다.
연구원은 계약자에게 현재 구축되어있는 제어시스템에 대한 기본 사용 교육을 제공하며,
계약자는 연구원이 제공하는 교육을 이해하기 위한 배경지식을 보유해야 한다.
표 9. 휴지기간의 극저온시스템 모니터링 업무 리스트
순번 극저온시스템 모니터링 업무 비고
1 알람 메시지 확인
2 주요 센서 트렌드 확인 온도, 압력, 유량, 속도 등
3 정상 상황의 주요 센서 지시값 파악 및 숙지 온도, 압력, 유량, 속도 등
4 비정상 상황 감지
연구원에서 현장 확인 필요상황 발생 시, 연구원이 지 무전 통신 활용
5
정하는 값 관찰, 확인 및 상황 전달 (연구원에서 무전기 제공)
6 제어로직 정상 작동 여부 확인
문서번호 : -
2025년도 중이온가속기용 극저온시스템
개정번호 : 0
운전 용역 과업지시서 발 행 일 : 2024. 10. 30
페 이 지 : 14/19


---

### openai answer
retriever A 점수: 3점
(가동기간 모니터링 업무 표에 알람 메시지 확인, 주요 센서 트렌드 확인, 정상 운전 센서 지시값 숙지, 비정상 운전 상황 감지, 제어로직 정상 작동 여부 등 answer의 항목이 모두 포함됨)

retriever B 점수: 3점
(가동기간 모니터링 업무 표에 answer의 항목들이 동일하게 모두 포함되어 있음: 알람 확인, 센서 트렌드, 정상 지시값 숙지, 비정상 감지, 제어로직 확인) 

============================================================
### query:
중이온가속기 극저온시스템의 '가동기간'과 '휴지기간'의 운전 업무 형태(근무 방식)는 어떻게 다릅니까?

### answer:
'가동기간'에는 시스템의 24시간 안정적인 운영을 위해 교대근무(24시간 운전)가 이루어집니다. 반면, '휴지기간'에는 주로 시스템 유지보수 등을 위해 주간 운전을 기본으로 합니다.

---

### retriever A:
중이온가속기 또는 극저온시스템 가동기간에는 24시간 운전 업무를 필요로 한다. 통상적으
로 가동기간에는 24시간 운전을 위한 교대근무가 이루어지며, i) 극저온시스템의 모니터링
및 제어, ii) 극저온시스템의 주요 수치를 기록 및 분석하는 업무가 주로 이루어진다.
다만, 통상적으로 이러한 교대근무로 인해 각 운전 인력마다 경험할 수 있는 운전 상황이
서로 다르게 되고, 서로 다른 경험을 했던 운전 인력 모두가 참여하여 각자의 iii) 운전 경험
을 통합/정리할 수 있는 여건이 형성될 수 없다는 한계가 존재한다. 이러한 운전 경험의 통
합 정리는 후술하는 휴지기간에의 운전업무에 집중적으로 수행 가능하다.
통상적으로 휴지기간에는 극저온시스템의 유지보수 등이 진행되며, 유지보수 등을 위한 제
어, 유지보수 등의 경과 및 결과 확인을 위한 극저온시스템 모니터링, 유지보수 등의 결과
확인을 위한 주요 수치 기록 및 분석 또한 필수적인 운전 업무이다. 휴지기간에는 통상적으
로 주간 근무로 운전이 이루어진다. 이와 같이 주간 근무가 이루어질 수 있는 휴지기간의
특성 상, 휴지기간은 가동기간에서 교대근무로 각 운전 인력 개인이 얻은 단편적인 운전 경
험을 통합하여 정리(문서화)할 수 있는 여건이 형성될 수 있는 기간이다.
결과적으로 휴지기간에는 i) 중이온가속기용 극저온시스템의 운전 경험을 통합하여 정리(문
서화)하는 업무가 집중적으로 이루어지며, ii) 유지보수 등을 위한 극저온시스템의 제어와 iii)
유지보수 등을 위한 극저온시스템의 주요 수치 기록 및 분석이 병행되어야 한다.
3.2 운전 대상 및 장소
문서번호 : -
2025년도 중이온가속기용 극저온시스템
개정번호 : 0
운전 용역 과업지시서 발 행 일 : 2024. 10. 30
페 이 지 : 12/19
매뉴얼을 작성하고 개정하는 업무를 수행해야 한다. 또한, 과업 및 운전 인력 감독자가 1인
이상 운전실에 배치되어 상기 업무들을 감독해야 한다. 계약자가 휴지기간에 수행해야 하는
운전 업무 상세 내용은 하위 절에 기술된 내용을 따른다. 본 과업지시서에 기술하지 않은
휴지기간의 운전 업무는 연구원과 계약자가 협의하여 수행 여부를 결정한다.
3.4.1 유지보수 등을 위한 극저온시스템 제어
극저온시스템의 제어는 극저온시스템이 휴지지간에 안전하고 안정적으로 운영될 수 있도록
극저온시스템 또는 극저온시스템 관련 부품들을 제어시스템을 활용하여 또는 현장에서 손으
로 조작하는 행위를 의미한다. 계약자는 극저온시스템이 본 목적에 부합하도록 극저온시스
템의 제어를 수행해야 한다. 표 8.에 극저온시스템 제어 업무에 해당되는 사항들을 기술하
였다.
휴지기간의 극저온시스템 제어는 i) 극저온시스템에 대한 충분한 이해를 바탕으로 ii) 조작
후 유지보수 등 현장 활동에 미치는 영향을 충분히 검토한 뒤 이루어져야 한다. 더불어, iii)
조작 대상물이 위치하고 있는 현장에 인력이 존재하는지 여부를 확인하고, iv) 현장에 인력
이 존재할 경우, 조작 대상물을 조작했을 때 발생할 수 있는 상황에 대해 해당 인력의 안전
이 확보된 뒤에 조작을 수행해야 한다. 마지막으로 v) 조작 대상/조작 상태에 대한 혼동이
없도록 각별히 주의를 하여 제어가 진행되어야 한다.


---

### retriever B:
3. 과업 내용
3.1 중이온가속기용 극저온시스템 운전 업무의 정의 및 필요성
본 과업에서 사용하는 “중이온가속기용 극저온시스템 운전 업무”의 정의는, “중이온가
속기용 극저온시스템이 안전하고 안정적으로 운영(가동기간뿐만 아니라 휴지기간의 운영 포
함)될 수 있도록 i) 극저온시스템을 모니터링 및 제어하고, ii) 극저온시스템의 주요 수치를
기록 및 분석하며, iii) 중이온가속기용 극저온시스템의 운전 경험을 통합하여 정리(문서화)하
는 업무를 의미한다.
중이온가속기 또는 극저온시스템 가동기간에는 24시간 운전 업무를 필요로 한다. 통상적으
로 가동기간에는 24시간 운전을 위한 교대근무가 이루어지며, i) 극저온시스템의 모니터링
및 제어, ii) 극저온시스템의 주요 수치를 기록 및 분석하는 업무가 주로 이루어진다.
다만, 통상적으로 이러한 교대근무로 인해 각 운전 인력마다 경험할 수 있는 운전 상황이
서로 다르게 되고, 서로 다른 경험을 했던 운전 인력 모두가 참여하여 각자의 iii) 운전 경험
을 통합/정리할 수 있는 여건이 형성될 수 없다는 한계가 존재한다. 이러한 운전 경험의 통
합 정리는 후술하는 휴지기간에의 운전업무에 집중적으로 수행 가능하다.
통상적으로 휴지기간에는 극저온시스템의 유지보수 등이 진행되며, 유지보수 등을 위한 제
어, 유지보수 등의 경과 및 결과 확인을 위한 극저온시스템 모니터링, 유지보수 등의 결과
확인을 위한 주요 수치 기록 및 분석 또한 필수적인 운전 업무이다. 휴지기간에는 통상적으
로 주간 근무로 운전이 이루어진다. 이와 같이 주간 근무가 이루어질 수 있는 휴지기간의
제외한다.
표 3. 2025년 중이온가속기용 극저온시스템 운전 용역 대상 장치 목록
구분 장치명 장소 비고
극저온 극저온설비동/ 압축기, 재생시스템, 2 K 시스템
SCL3 극저온플랜트
플랜트 헬륨압축기동 등 부대시설 일체
문서번호 : -
2025년도 중이온가속기용 극저온시스템
개정번호 : 0
운전 용역 과업지시서 발 행 일 : 2024. 10. 30
페 이 지 : 8/19
극저온설비동/ 압축기, 재생시스템, 2 K 시스템
SCL2 극저온플랜트
헬륨압축기동 등 부대시설 일체
TBx 극저온설비동
밸브박스, 초전도가속관 저온유지
SCL3 터널/
극저온
SCL3 LINAC 모듈 및 밸브박스 진공단열용 진
ISOL 구간
분배
공펌프
시스템 밸브박스, 초전도가속관 저온유지
SCL2 터널/
SCL2 LINAC 모듈 및 밸브박스 진공단열용 진
IF 구간
공펌프
외부 정화기 외부
회수 압축시스템 헬륨압축기동
회수
가스백 헬륨압축기동
시스템
헬륨 탱크 외부 기체 헬륨량 관리 포함
액체 질소 시스템 외부 액체 질소량 관리 포함
3.3 중이온가속기 또는 극저온시스템 가동기간의 운전 업무
계약자는 중이온가속기 또는 극저온시스템 가동기간 동안 24시간 운전업무가 이루어질 수
있도록 운전 인력을 제공해야 한다. 24시간 운전 업무는 2인 1조를 기본으로 하며, 주간에
는 과업 및 운전 인력 감독자가 1인 이상 운전실에 존재해야 한다. 계약자가 가동기간에 수
행해야 하는 운전 업무 상세 내용은 하위 절에 기술된 내용을 따른다. 본 과업지시서에 기
술하지 않은 가동기간의 운전 업무는 연구원과 계약자가 협의하여 수행 여부를 결정한다.


---

### openai answer
retriever A 점수: 3점
가동기간의 24시간 교대근무와 휴지기간의 주간 운전이 모두 명시되어 있어 정답의 핵심 정보를 완전히 포함합니다.

retriever B 점수: 3점
가동기간의 24시간 교대근무와 휴지기간의 주간 운전이 모두 명시되어 있어 정답의 핵심 정보를 완전히 포함합니다. 

============================================================
### query:
극저온시스템 과업지시서에 따르면, 가동기간 중 교대근무 인력이 개별적으로 얻은 운전 경험을 통합하여 정리하기 어려운 이유는 무엇이며, 이 업무는 주로 언제 수행되어야 합니까?

### answer:
교대근무로 인해 각 운전 인력마다 경험하는 운전 상황이 서로 달라, 모든 인원이 함께 모여 운전 경험을 통합하고 정리할 여건이 형성되기 어렵기 때문입니다. 이러한 운전 경험 통합 및 정리(문서화) 업무는 주로 주간 근무가 이루어지는 '휴지기간'에 집중적으로 수행됩니다.

---

### retriever A:
중이온가속기 또는 극저온시스템 가동기간에는 24시간 운전 업무를 필요로 한다. 통상적으
로 가동기간에는 24시간 운전을 위한 교대근무가 이루어지며, i) 극저온시스템의 모니터링
및 제어, ii) 극저온시스템의 주요 수치를 기록 및 분석하는 업무가 주로 이루어진다.
다만, 통상적으로 이러한 교대근무로 인해 각 운전 인력마다 경험할 수 있는 운전 상황이
서로 다르게 되고, 서로 다른 경험을 했던 운전 인력 모두가 참여하여 각자의 iii) 운전 경험
을 통합/정리할 수 있는 여건이 형성될 수 없다는 한계가 존재한다. 이러한 운전 경험의 통
합 정리는 후술하는 휴지기간에의 운전업무에 집중적으로 수행 가능하다.
통상적으로 휴지기간에는 극저온시스템의 유지보수 등이 진행되며, 유지보수 등을 위한 제
어, 유지보수 등의 경과 및 결과 확인을 위한 극저온시스템 모니터링, 유지보수 등의 결과
확인을 위한 주요 수치 기록 및 분석 또한 필수적인 운전 업무이다. 휴지기간에는 통상적으
로 주간 근무로 운전이 이루어진다. 이와 같이 주간 근무가 이루어질 수 있는 휴지기간의
특성 상, 휴지기간은 가동기간에서 교대근무로 각 운전 인력 개인이 얻은 단편적인 운전 경
험을 통합하여 정리(문서화)할 수 있는 여건이 형성될 수 있는 기간이다.
결과적으로 휴지기간에는 i) 중이온가속기용 극저온시스템의 운전 경험을 통합하여 정리(문
서화)하는 업무가 집중적으로 이루어지며, ii) 유지보수 등을 위한 극저온시스템의 제어와 iii)
유지보수 등을 위한 극저온시스템의 주요 수치 기록 및 분석이 병행되어야 한다.
3.2 운전 대상 및 장소
다. 계약자는 본 과업을 수행하기 위해 적절한 지식 또는 능력을 보유하고 있는 적정 수의 운
전 인력을 연구원에게 제공하여야 한다.
• 가동기간의 극저온시스템 운전 업무 (24시간 운전)
- 극저온시스템 모니터링
- 극저온시스템 주요 수치 기록 및 분석
- 극저온시스템 제어
- 기타 가동기간에 필요한 운전 업무
• 휴지기간의 극저온시스템 운전 업무 (주간 운전)
- 유지보수 등을 위한 극저온시스템 제어
- 유지보수 등을 위한 극저온시스템 모니터링
- 유지보수 등을 위한 극저온시스템 주요 수치 기록 및 분석
- 극저온시스템 운전 매뉴얼 작성 및 개정
- 극저온시스템 운전 교육
- 성과보고회 개최
- 기타 휴지기간에 필요한 운전 업무
• 본 과업 및 본 과업을 위해 투입된 인력에 대한 관리감독
본 과업의 기간은 2025년 1월 1일부터 2025년 12월 31일까지로 하며, 2025년 동안 가동기
간의 운전 업무를 수행하는 것을 기준으로 한다. 단, 표 2.와 같이 연구원의 운영 추진 상황
에 따라 휴지기간의 운전업무를 수행하는 것으로 변경될 수 있다. 가동기간과 휴지기간의
운전 업무는 “3. 과업 내용” 하위 절에 기술한 내용에 따른다.
표 2. 과업 추진 일정(안)
2025년
구분
1분기 2분기 3분기 4분기
SCL3 극저온플랜트
SCL3 중이온가속기 SCL3 구간 운전 SCL3 구간 유지보수
시운전 준비 및
구간 (가동기간) (휴지기간)
시운전 (가동기간)
SCL2 SCL2 극저온플랜트 중이온가속기 SCL2 구간 시운전 SCL2 구간 유지보수
구간 시운전 (가동기간) (가동기간) (휴지기간)
회수


---

### retriever B:
3. 과업 내용
3.1 중이온가속기용 극저온시스템 운전 업무의 정의 및 필요성
본 과업에서 사용하는 “중이온가속기용 극저온시스템 운전 업무”의 정의는, “중이온가
속기용 극저온시스템이 안전하고 안정적으로 운영(가동기간뿐만 아니라 휴지기간의 운영 포
함)될 수 있도록 i) 극저온시스템을 모니터링 및 제어하고, ii) 극저온시스템의 주요 수치를
기록 및 분석하며, iii) 중이온가속기용 극저온시스템의 운전 경험을 통합하여 정리(문서화)하
는 업무를 의미한다.
중이온가속기 또는 극저온시스템 가동기간에는 24시간 운전 업무를 필요로 한다. 통상적으
로 가동기간에는 24시간 운전을 위한 교대근무가 이루어지며, i) 극저온시스템의 모니터링
및 제어, ii) 극저온시스템의 주요 수치를 기록 및 분석하는 업무가 주로 이루어진다.
다만, 통상적으로 이러한 교대근무로 인해 각 운전 인력마다 경험할 수 있는 운전 상황이
서로 다르게 되고, 서로 다른 경험을 했던 운전 인력 모두가 참여하여 각자의 iii) 운전 경험
을 통합/정리할 수 있는 여건이 형성될 수 없다는 한계가 존재한다. 이러한 운전 경험의 통
합 정리는 후술하는 휴지기간에의 운전업무에 집중적으로 수행 가능하다.
통상적으로 휴지기간에는 극저온시스템의 유지보수 등이 진행되며, 유지보수 등을 위한 제
어, 유지보수 등의 경과 및 결과 확인을 위한 극저온시스템 모니터링, 유지보수 등의 결과
확인을 위한 주요 수치 기록 및 분석 또한 필수적인 운전 업무이다. 휴지기간에는 통상적으
로 주간 근무로 운전이 이루어진다. 이와 같이 주간 근무가 이루어질 수 있는 휴지기간의
는 과업 및 운전 인력 감독자가 1인 이상 운전실에 존재해야 한다. 계약자가 가동기간에 수
행해야 하는 운전 업무 상세 내용은 하위 절에 기술된 내용을 따른다. 본 과업지시서에 기
술하지 않은 가동기간의 운전 업무는 연구원과 계약자가 협의하여 수행 여부를 결정한다.
3.3.1 극저온시스템 모니터링
계약자는 가동기간 동안 극저온시스템이 안정적으로 가동될 수 있도록 모니터링을 해야 한


---

### openai answer
retriever A 점수: 3점
(교대근무로 각 인력의 운전 상황이 달라 모두 모여 통합/정리하기 어렵다는 이유와, 이 업무가 휴지기간(주간 근무)에 집중 수행되어야 함이 명시되어 있음.)

retriever B 점수: 3점
(교대근무로 인한 경험 분산과 통합 어려움이 언급되고, 운전 경험 통합·정리가 휴지기간에 집중 수행 가능하며 휴지기간은 통상 주간 근무라는 점이 포함되어 있음.) 

============================================================
### query:
계약업체가 매월 운전 현황을 정리해서 연구원에 보고해야 하는 공식적인 문서는 무엇이며, 제출 시점은 언제입니까?

### answer:
매월 운전 상황을 요약하여 제출해야 하는 문서는 '월간보고서'이며, 매월 기성 비용을 신청할 때 제출해야 합니다.

---

### retriever A:
계약종료 및 인수인계에 관한 사항
세부내용
○ 수행업체는 본 사업 종료 1개월 전 시스템의 발전 및 안정성을 유지하기 위한 인수인계 계획을 제시하여야 함
- 본 사업수행 기간 동안 취득한 모든 정보 및 생상된 산출물 일체
- 인수인계의 주요 업무, 상세 일정 및 인수인계 대상 주요 산출물 제시
- 인수인계 기간 동안 발생하는 리스크에 대한 방지대책 제시
○ 수행업체는 계약기간 만료 시 차기 수행업체로 선정된 사업자가 본 사업의 업무를 차질없이 수행할 수 있도록 요청 시 통합접수시스템 운영 업무 수행에 필요한 중요사항 등을 인계하여야 하며, 시스템의 효율적인 운영을 위해 하자담보 책임기간 중에는 기술자문에 응하여야 함
- 시스템 구성도, 시스템 현황, DB권한 등
- 통합접수시스템 기동/종료 절차
- 타 시스템 연계 및 적재 절차
- 일반 사용자 및 관리자 매뉴얼 등
○ 인수 및 인계 기간은 차기 시스템 운영 사업 개시일로부터 30일 이내로 하며, 인수 및 인계기간 동안의 용역비용은 지급하지 아니함
산출 정보
인수인계 계획서(사업수행계획서 내 포함 가능)
관련 요구사항
요구사항 출처
요구사항 분류
프로젝트지원 요구사항
요구사항 고유번호
PSR-008
요구사항 명칭
정보화사업 주요점검 및 수행지원
요구사항
상세설명
정의
정보화사업 주요점검 및 수행지원
세부내용
○ 통합접수시스템 관련 점검사항 준수 및 현장점검 수행
○ 보안관리 실태 점검 대응
○ 공공데이터 품질관리(DB) 수준평가 대응 및 데이터 오류 정제 활동 실시
산출 정보
점검별 결과보고서
관련 요구사항
요구사항 출처
□ 품질관리 요구사항
요구사항 분류
· 월별 추진 실적 보고
· 관련부서 업무협의 내용
· 문제점 보고 및 특이사항
매월 말일
비정기
보고
인수인계
보고
· 신규 사업자 선정시 업무인계 결과보고
사업종료
1주전
기타
· 이슈사항 및 문제점 협의 등
이슈발생시
※ 보고형식 및 주기는 추후 협의하여 조정할 수 있음
ㅇ 계약 기간 만료일 전 과업 완료 보고서를 제출하여야 함
구분
보고명
보고내용
시기
사업
종료
완료
보고서
· 용역종료 시 전체 추진단계의 수행결과 보고로 인쇄물 2부, 외장하드 1본으로 제출
· 산출물을 종류별로 분류한 파일
용역종료일
2주일 전
- 용역 완료시 제출하는 최종 결과물(프로그램 소스 코드(주석, 설명서 포함), ERD, 테이블 정의서, 프로그램 정의서 등)은 발주자의 승인을 득한 후 원본 파일을 CD 또는 HDD에 수록하여 산출물과 같이 제출하여야 하고, 프로그램 소스에 대한 형상관리가 가능하도록 협조하여야 함
- 산출물은 기존 산출물을 기반으로 현행화 또는 추가 작성해야 하며, 상호 협의하여 조정 가능
ㅇ 제출 시기에 따라 종류별 산출물을 제출하되, 사업관리 및 검수를 위한 각종 업무 보고를 요청할 경우, 신속하게 작성하여 보고하는 수시보고 체계 마련 ※ 제출 전 주관기관 사전 검토 및 확인
산출정보
착수보고서, 완료보고서, 월간보고서
요구사항 고유번호
PMR-005
요구사항 명칭
인계 조건
요구사항
상세설명
정의
관한 사항
세부
내용
인계 요건
- 사업종료 1개월 전까지 업무인계 계획서를 작성해 진흥원의 승인을 받아야함
- 발주기관과 월 1회 이상 기술이전 등에 대한 협의를 하고, 그 결과를 문서화


---

### retriever B:
4. 제출문서
계약자는 표 13.와 같은 문서들을 제출일정에 맞춰 연구원에 제출해야 한다.
표 13. 제출문서 목록 및 관련 사항
연구원
제출문서 내용 비고 및 제출일
승인
투입 인력
- 투입 인력 이력 사항 X - 계약 시
이력서
- 최초: 계약 후 1개월 내
작업계획서 - 일별 투입 인력 - 이후: 매주 금요일까지 차주
X
(주간근무표) - 휴가상황 근무표 제출
- 문서 파일 제출
- IBS에서 제공한 양식 파일
극저온시스템 - 매 월간 기성 신청 시 제출
주요 수치 - 극저온시스템 주요 수치 X - 문서 파일 제출
기록문서 - 수치 기록을 위해 계약자가
준비한 PC에 상시 저장
일일근무일지 - 일일 운영 상황 기록 O - 매 월간 기성 신청 시 제출
- 문서 파일 제출
- 수치 기록을 위해 계약자가
인수인계 문서 - (가동기간) 인수인계 내용 X 준비한 PC에 상시 저장
- 수리 사유 발생일
- 매 월간 기성 신청 시 제출
- 수리 필요 대상
수리필요사항 - 문서 파일 제출
- 상세 내용 O
기록 문서 - 수치 기록을 위해 계약자가
- 발견자
준비한 PC에 상시 저장
- 기타 필요한 내용
- 발생한 경우, 기성 신청 시
특이사항, 위험상황 발생 시,
제출 (해당 월 발생 건)
- 상황 내용
- 문서 파일 제출
- 작성자
Lesson learn O - 운전 인력 전원 회람 후 확
- 금회 대처 방법
인 서명 필수
- 향후 대처 방법
- 수치 기록을 위해 계약자가
- 기타 필요한 내용
준비한 PC에 상시 저장
- 매 월간 기성 신청 시 제출
월간보고서 - 월간 운전 상황 요약 O
- 문서 파일 제출
라. 계약 후 제출서류 : 사업수행계획서, 기술적용계획표
1) 사업수행자는 계약일로부터 14일 이내에 사업수행계획서와 기술적용계획표를 제출하고 사업 착수를 완료하여야 함
2) 사업수행계획서에는 제안요청서, 제안서, 협상에 의해 수정·보완·추가된 내용을 반영하여 제출하여야 하며 발주기관의 승인을 받아야 함
제출방법
1
착수계
A4
1
계약 후 7일 이내
서신문서
2
보안관련 문서
(보안서약서, 비밀유지서약서, 개인정보 위탁계약서, 개인정보 보호서약서)
A4
1
계약 후 7일 이내
3
사업수행계획서
A4
8
계약 후 10일 이내
4
착수보고서
A4
8
착수 후 10일 이내
5
완료보고서
A4
8
계약종료 2주 전 까지
6
최종산출물
(개발 산출물, 운영자 매뉴얼, 사용자 매뉴얼, 각종 분석/측정 결과보고서, 기술지원 확약서, 품질관리보고서, 기술적용계획/결과표, 기술이전계획서, 장애대응 매뉴얼, 테스트 수행 계획서/결과서 등)
USB
2
계약종료 2주 전 까지


---

### openai answer
retriever A 점수: 2점
- ‘월간보고서’ 산출물 존재는 명시되어 있으나, 제출 시점(매 월간 기성 신청 시 제출)이 명확히 제시되지 않음. 또한 ‘운전 현황’ 요약 문서로의 성격이 구체화되어 있지 않음.

retriever B 점수: 3점
- ‘월간보고서(월간 운전 상황 요약)’가 명시되어 있고, 제출 시점이 ‘매 월간 기성 신청 시 제출’로 정확히 기재되어 있어 정답의 모든 요소를 포함함. 


## multi_query_search

============================================================
### query:
본 과업지시서에서 '극저온시스템'은 무엇을 포함하는 것으로 정의하고 있습니까?

### answer:
'극저온시스템'은 SCL3/SCL2 극저온플랜트 및 부대설비, SCL3/SCL2 분배시스템 및 부대설비, SCL3/SCL2 초전도가속관 저온유지모듈, 기체헬륨 탱크, 액체질소 시스템 및 극저온시스템용 헬륨회수시스템을 모두 포함하는 것으로 정의됩니다.

---

### retriever A:
7. 안전관리 .............................................................................................................. 19
8. 붙임 자료 ........................................................................................................... 19
문서번호 : -
2025년도 중이온가속기용 극저온시스템
개정번호 : 0
운전 용역 과업지시서 발 행 일 : 2024. 10. 30
페 이 지 : 5/19
1. 서 론
1.1 목 적
기초과학연구원 중이온가속기연구소(이하 “연구원”)에서 운영중인 중이온가속기용 극저
온시스템(이하 “극저온시스템”)은 고도의 공정 시스템으로, 안전하고 안정적인 중이온가속
기의 운영을 위해 상시 모니터링이 필요하다. 이에 본 과업지시서는 극저온시스템의 운전
용역에 대한 상세 과업 내용을 기술한다.
1.2 용어 설명
본 과업지시서에서 사용하는 용어에 대한 상세 내용은 표 1.과 같다. 계약자는 반드시 해당
용어의 의미를 충분히 이해하고 숙지해야 하며, 용어 미숙지로 인한 책임은 계약자에게 있
다.
표 1. 본 과업지시서에 사용되는 용어 설명
용어 설명
연구원 중이온가속기연구소
계약자 본 과업을 수행할 계약상대자
SCL3/SCL2 극저온플랜트 및 부대설비,
SCL3/SCL2 분배시스템 및 부대설비,
SCL3/SCL2 초전도가속관 저온유지모듈,
극저온시스템
기체헬륨 탱크, 액체질소 시스템 및
TBx 극저온설비동
밸브박스, 초전도가속관 저온유지
SCL3 터널/
극저온
SCL3 LINAC 모듈 및 밸브박스 진공단열용 진
ISOL 구간
분배
공펌프
시스템 밸브박스, 초전도가속관 저온유지
SCL2 터널/
SCL2 LINAC 모듈 및 밸브박스 진공단열용 진
IF 구간
공펌프
외부 정화기 외부
회수 압축시스템 헬륨압축기동
회수
가스백 헬륨압축기동
시스템
헬륨 탱크 외부 기체 헬륨량 관리 포함
액체 질소 시스템 외부 액체 질소량 관리 포함
3.3 중이온가속기 또는 극저온시스템 가동기간의 운전 업무
계약자는 중이온가속기 또는 극저온시스템 가동기간 동안 24시간 운전업무가 이루어질 수
있도록 운전 인력을 제공해야 한다. 24시간 운전 업무는 2인 1조를 기본으로 하며, 주간에
는 과업 및 운전 인력 감독자가 1인 이상 운전실에 존재해야 한다. 계약자가 가동기간에 수
행해야 하는 운전 업무 상세 내용은 하위 절에 기술된 내용을 따른다. 본 과업지시서에 기
술하지 않은 가동기간의 운전 업무는 연구원과 계약자가 협의하여 수행 여부를 결정한다.
3.3.1 극저온시스템 모니터링
계약자는 가동기간 동안 극저온시스템이 안정적으로 가동될 수 있도록 모니터링을 해야 한
다. 가동기간의 모니터링 업무는 극저온시스템이 안정적으로 가동되고 있는지 여부를 확인
하기 위해 극저온시스템을 지속적으로 관찰하는 업무를 의미하며, 표 4.에 가동기간의 모니
터링 업무에 해당되는 내용들을 기술하였다.
극저온시스템의 모니터링은 연구원에 구축되어있는 극저온시스템용 제어시스템(이하 “제


---

### retriever B:
1. 서 론
1.1 목 적
기초과학연구원 중이온가속기연구소(이하 “연구원”)에서 운영중인 중이온가속기용 극저
온시스템(이하 “극저온시스템”)은 고도의 공정 시스템으로, 안전하고 안정적인 중이온가속
기의 운영을 위해 상시 모니터링이 필요하다. 이에 본 과업지시서는 극저온시스템의 운전
용역에 대한 상세 과업 내용을 기술한다.
1.2 용어 설명
본 과업지시서에서 사용하는 용어에 대한 상세 내용은 표 1.과 같다. 계약자는 반드시 해당
용어의 의미를 충분히 이해하고 숙지해야 하며, 용어 미숙지로 인한 책임은 계약자에게 있
다.
표 1. 본 과업지시서에 사용되는 용어 설명
용어 설명
연구원 중이온가속기연구소
계약자 본 과업을 수행할 계약상대자
SCL3/SCL2 극저온플랜트 및 부대설비,
SCL3/SCL2 분배시스템 및 부대설비,
SCL3/SCL2 초전도가속관 저온유지모듈,
극저온시스템
기체헬륨 탱크, 액체질소 시스템 및
극저온시스템용 헬륨회수시스템 포함
(“3.2절 운전대상”, 표 3. 참고)
중이온가속기 또는 극저온시스템이 냉각(냉각 착수
가동기간
이후) 또는 빔서비스 등을 위해 가동 중인 기간
중이온가속기 또는 극저온시스템이 유지보수 등을
휴지기간
위해 가동을 멈추고 휴지 중인 기간
중이온가속기용 극저온시스템이 가동기간 및 휴지
기간에 안전하고 안정적으로 운영될 수 있도록,
중이온가속기용 극저온시스템 i) 극저온시스템을 모니터링 및 제어하고,
운전 업무 ii) 극저온시스템의 주요 수치를 기록 및 분석하며,
iii) 중이온가속기용 극저온시스템의 운전 경험을
통합하여 정리(문서화)하는 업무
문서번호 : -
문서번호 : -
2025년도 중이온가속기용 극저온시스템
개정번호 : 0
운전 용역 과업지시서 발 행 일 : 2024. 10. 30
페 이 지 : 12/19
매뉴얼을 작성하고 개정하는 업무를 수행해야 한다. 또한, 과업 및 운전 인력 감독자가 1인
이상 운전실에 배치되어 상기 업무들을 감독해야 한다. 계약자가 휴지기간에 수행해야 하는
운전 업무 상세 내용은 하위 절에 기술된 내용을 따른다. 본 과업지시서에 기술하지 않은
휴지기간의 운전 업무는 연구원과 계약자가 협의하여 수행 여부를 결정한다.
3.4.1 유지보수 등을 위한 극저온시스템 제어
극저온시스템의 제어는 극저온시스템이 휴지지간에 안전하고 안정적으로 운영될 수 있도록
극저온시스템 또는 극저온시스템 관련 부품들을 제어시스템을 활용하여 또는 현장에서 손으
로 조작하는 행위를 의미한다. 계약자는 극저온시스템이 본 목적에 부합하도록 극저온시스
템의 제어를 수행해야 한다. 표 8.에 극저온시스템 제어 업무에 해당되는 사항들을 기술하
였다.
휴지기간의 극저온시스템 제어는 i) 극저온시스템에 대한 충분한 이해를 바탕으로 ii) 조작
후 유지보수 등 현장 활동에 미치는 영향을 충분히 검토한 뒤 이루어져야 한다. 더불어, iii)
조작 대상물이 위치하고 있는 현장에 인력이 존재하는지 여부를 확인하고, iv) 현장에 인력
이 존재할 경우, 조작 대상물을 조작했을 때 발생할 수 있는 상황에 대해 해당 인력의 안전
이 확보된 뒤에 조작을 수행해야 한다. 마지막으로 v) 조작 대상/조작 상태에 대한 혼동이
없도록 각별히 주의를 하여 제어가 진행되어야 한다.


---

### openai answer
retriever A 점수: 2점
일부 구성(SCL3/SCL2 극저온플랜트·분배시스템·초전도가속관 저온유지모듈, 기체헬륨 탱크, 액체질소 시스템)은 명시되나, “극저온시스템용 헬륨회수시스템 포함”이 정의 문구로 명확히 확인되지 않음.

retriever B 점수: 3점
표 1 용어 설명에 ‘극저온시스템’이 SCL3/SCL2 극저온플랜트·분배시스템·초전도가속관 저온유지모듈, 기체헬륨 탱크, 액체질소 시스템 및 극저온시스템용 헬륨회수시스템을 포함한다고 명시되어 있어, 답변의 모든 요소가 포함됨. 

============================================================
### query:
극저온시스템 과업 추진 일정에 따르면, 4분기에 SCL3 구간에 대해 예정된 주요 업무는 무엇이며, 이때의 업무 기간은 '가동기간'과 '휴지기간' 중 어느 것에 해당합니까?

### answer:
2025년 4분기에 SCL3 구간에 예정된 주요 업무는 '유지보수'이며, 이 기간은 '휴지기간'에 해당합니다.

---

### retriever A:
다. 계약자는 본 과업을 수행하기 위해 적절한 지식 또는 능력을 보유하고 있는 적정 수의 운
전 인력을 연구원에게 제공하여야 한다.
• 가동기간의 극저온시스템 운전 업무 (24시간 운전)
- 극저온시스템 모니터링
- 극저온시스템 주요 수치 기록 및 분석
- 극저온시스템 제어
- 기타 가동기간에 필요한 운전 업무
• 휴지기간의 극저온시스템 운전 업무 (주간 운전)
- 유지보수 등을 위한 극저온시스템 제어
- 유지보수 등을 위한 극저온시스템 모니터링
- 유지보수 등을 위한 극저온시스템 주요 수치 기록 및 분석
- 극저온시스템 운전 매뉴얼 작성 및 개정
- 극저온시스템 운전 교육
- 성과보고회 개최
- 기타 휴지기간에 필요한 운전 업무
• 본 과업 및 본 과업을 위해 투입된 인력에 대한 관리감독
본 과업의 기간은 2025년 1월 1일부터 2025년 12월 31일까지로 하며, 2025년 동안 가동기
간의 운전 업무를 수행하는 것을 기준으로 한다. 단, 표 2.와 같이 연구원의 운영 추진 상황
에 따라 휴지기간의 운전업무를 수행하는 것으로 변경될 수 있다. 가동기간과 휴지기간의
운전 업무는 “3. 과업 내용” 하위 절에 기술한 내용에 따른다.
표 2. 과업 추진 일정(안)
2025년
구분
1분기 2분기 3분기 4분기
SCL3 극저온플랜트
SCL3 중이온가속기 SCL3 구간 운전 SCL3 구간 유지보수
시운전 준비 및
구간 (가동기간) (휴지기간)
시운전 (가동기간)
SCL2 SCL2 극저온플랜트 중이온가속기 SCL2 구간 시운전 SCL2 구간 유지보수
구간 시운전 (가동기간) (가동기간) (휴지기간)
회수
2025년
구분
1분기 2분기 3분기 4분기
SCL3 극저온플랜트
SCL3 중이온가속기 SCL3 구간 운전 SCL3 구간 유지보수
시운전 준비 및
구간 (가동기간) (휴지기간)
시운전 (가동기간)
SCL2 SCL2 극저온플랜트 중이온가속기 SCL2 구간 시운전 SCL2 구간 유지보수
구간 시운전 (가동기간) (가동기간) (휴지기간)
회수
필요시 간헐적 운전 (휴지기간에 준함)
시스템
문서번호 : -
2025년도 중이온가속기용 극저온시스템
개정번호 : 0
운전 용역 과업지시서 발 행 일 : 2024. 10. 30
페 이 지 : 7/19
3. 과업 내용
3.1 중이온가속기용 극저온시스템 운전 업무의 정의 및 필요성
본 과업에서 사용하는 “중이온가속기용 극저온시스템 운전 업무”의 정의는, “중이온가
속기용 극저온시스템이 안전하고 안정적으로 운영(가동기간뿐만 아니라 휴지기간의 운영 포
함)될 수 있도록 i) 극저온시스템을 모니터링 및 제어하고, ii) 극저온시스템의 주요 수치를
기록 및 분석하며, iii) 중이온가속기용 극저온시스템의 운전 경험을 통합하여 정리(문서화)하
는 업무를 의미한다.
중이온가속기 또는 극저온시스템 가동기간에는 24시간 운전 업무를 필요로 한다. 통상적으
로 가동기간에는 24시간 운전을 위한 교대근무가 이루어지며, i) 극저온시스템의 모니터링
및 제어, ii) 극저온시스템의 주요 수치를 기록 및 분석하는 업무가 주로 이루어진다.
다만, 통상적으로 이러한 교대근무로 인해 각 운전 인력마다 경험할 수 있는 운전 상황이


---

### retriever B:
다. 계약자는 본 과업을 수행하기 위해 적절한 지식 또는 능력을 보유하고 있는 적정 수의 운
전 인력을 연구원에게 제공하여야 한다.
• 가동기간의 극저온시스템 운전 업무 (24시간 운전)
- 극저온시스템 모니터링
- 극저온시스템 주요 수치 기록 및 분석
- 극저온시스템 제어
- 기타 가동기간에 필요한 운전 업무
• 휴지기간의 극저온시스템 운전 업무 (주간 운전)
- 유지보수 등을 위한 극저온시스템 제어
- 유지보수 등을 위한 극저온시스템 모니터링
- 유지보수 등을 위한 극저온시스템 주요 수치 기록 및 분석
- 극저온시스템 운전 매뉴얼 작성 및 개정
- 극저온시스템 운전 교육
- 성과보고회 개최
- 기타 휴지기간에 필요한 운전 업무
• 본 과업 및 본 과업을 위해 투입된 인력에 대한 관리감독
본 과업의 기간은 2025년 1월 1일부터 2025년 12월 31일까지로 하며, 2025년 동안 가동기
간의 운전 업무를 수행하는 것을 기준으로 한다. 단, 표 2.와 같이 연구원의 운영 추진 상황
에 따라 휴지기간의 운전업무를 수행하는 것으로 변경될 수 있다. 가동기간과 휴지기간의
운전 업무는 “3. 과업 내용” 하위 절에 기술한 내용에 따른다.
표 2. 과업 추진 일정(안)
2025년
구분
1분기 2분기 3분기 4분기
SCL3 극저온플랜트
SCL3 중이온가속기 SCL3 구간 운전 SCL3 구간 유지보수
시운전 준비 및
구간 (가동기간) (휴지기간)
시운전 (가동기간)
SCL2 SCL2 극저온플랜트 중이온가속기 SCL2 구간 시운전 SCL2 구간 유지보수
구간 시운전 (가동기간) (가동기간) (휴지기간)
회수
일정
구분
추진 일정
M
M+1
M+2
M+3
M+4
사업발주 및 사업자 선정
업무 협의
시스템
구축
기존 시스템 분석
업무(요구사항) 분석
설계 및 구축
시스템 구현 및 개발
시스템테스트 및 수정
시스템 가동 및 운영(안정화)
※ 위 추진일정은 사업 추진과정에서 조정 가능함
Ⅲ
다. 추진일정
구 분
2024년~2025년
M1
M2
M3
M4
M5
M6
◦ 시스템 개발
- 분석
- 설계/구현
- 단위/통합 시험
◦ 연계시스템 기능개선
◦ 운영개시 및 안정화
◦ 보고회
- 착수, 중간, 최종보고
★
★
★
* 추진과정에서 일정은 변경될 수 있음
Ⅱ
일반 현황


---

### openai answer
retriever A 점수: 3점
표 2의 과업 추진 일정에 4분기 SCL3 구간이 ‘유지보수(휴지기간)’로 명시되어 있어, 질문의 Answer를 모두 포함합니다.

retriever B 점수: 3점
표 2의 과업 추진 일정에 4분기 SCL3 구간이 ‘유지보수(휴지기간)’로 명시되어 있어, 질문의 Answer를 모두 포함합니다. 

============================================================
### query:
극저온시스템 '가동기간' 동안 계약자가 24시간 수행해야 할 극저온시스템 모니터링 업무에는 어떤 것들이 있습니까? (3가지 이상)

### answer:
가동기간 동안 수행해야 할 모니터링 업무는 다음과 같습니다.
알람 메시지 확인
주요 센서(온도, 압력, 유량 등)의 트렌드 확인
정상 운전 상황의 주요 센서 지시값 파악 및 숙지
비정상 운전 상황 감지
제어로직 정상 작동 여부 확인

---

### retriever A:
다. 가동기간의 모니터링 업무는 극저온시스템이 안정적으로 가동되고 있는지 여부를 확인
하기 위해 극저온시스템을 지속적으로 관찰하는 업무를 의미하며, 표 4.에 가동기간의 모니
터링 업무에 해당되는 내용들을 기술하였다.
극저온시스템의 모니터링은 연구원에 구축되어있는 극저온시스템용 제어시스템(이하 “제
어시스템”)을 활용하여 또는 현장에 설치되어있는 장치를 통해 수행한다. 연구원은 구축되
어있는 제어시스템을 활용하기 위한 컴퓨터, 모니터, 키보드, 마우스, 책상 및 의자를 계약
자에게 제공한다. 현재 연구원에 구축되어있는 제어시스템은 표 5.과 같다.
연구원은 계약자에게 현재 구축되어있는 제어시스템에 대한 기본 사용 교육을 제공하며,
계약자는 연구원이 제공하는 교육을 이해하기 위한 배경지식을 보유해야 한다.
문서번호 : -
2025년도 중이온가속기용 극저온시스템
개정번호 : 0
운전 용역 과업지시서 발 행 일 : 2024. 10. 30
페 이 지 : 9/19
표 4. 가동기간의 극저온시스템 모니터링 업무 리스트
순번 극저온시스템 모니터링 업무 비고
1 알람 메시지 확인
2 주요 센서 트렌드 확인 온도, 압력, 유량, 속도 등
3 정상 운전 상황의 주요 센서 지시값 파악 및 숙지 온도, 압력, 유량, 속도 등
4 비정상 운전 상황 감지
연구원에서 현장 확인 필요상황 발생 시, 연구원이 지 무전 통신 활용
5
정하는 값 관찰, 확인 및 상황 전달 (연구원에서 무전기 제공)
6 제어로직 정상 작동 여부 확인
연구원 요청 시,
7 현장 상황 일시적 또는 지속적 관찰
비방사선구역 한정
이온가속기용 극저온시스템 운영에 중대한 영향이 생길 경우, 이는 계약자의 책임으로 한
다.
표 8. 휴지기간의 극저온시스템 제어 업무 리스트
순번 극저온시스템 제어 업무 비고
자동/수동(Auto/Manual) 제어 우선
1 계장류 제어 우선권(Priority) 변경
권 설정 변경
문서번호 : -
2025년도 중이온가속기용 극저온시스템
개정번호 : 0
운전 용역 과업지시서 발 행 일 : 2024. 10. 30
페 이 지 : 13/19
자동 밸브 – 제어시스템 활용
2 극저온시스템 자동/수동 밸브 조작
수동 밸브 – 현장에서 손으로 조작
압축기 emergency stop 버튼
3 현장에서 손으로 조작
활성화/비활성화
제어시스템 활용
4 진공펌프 가동/정지
(현장 조작 필요 시 연구원에 통지)
5 센서 오버라이드(Override) 설정 또는 해제 제어시스템 활용
6 계장류 제어 변수값(Ramp speed 등) 변경 제어시스템 활용
7 시퀀스 제어 변수값(Timer 등) 변경 제어시스템 활용
8 시퀀스 시작/종료 제어시스템 활용
현장 확인 및 최종 준비 상태 미진
9 극저온시스템 가동 전 최종 준비 상태 확인
할 경우, 상태 보완 포함
기타 극저온시스템의 안전하고 안정적인 운
10 제어시스템 활용 및 현장 조작 포함
전에 필요한 제어
3.4.2 유지보수 등을 위한 극저온시스템 모니터링
계약자는 휴지기간 동안 극저온시스템이 안정적으로 가동될 수 있도록 모니터링을 해야 한
다. 휴지기간의 모니터링 업무는 i) 유지보수 등을 위한 현장 활동 시 극저온시스템에 특이


---

### retriever B:
다. 가동기간의 모니터링 업무는 극저온시스템이 안정적으로 가동되고 있는지 여부를 확인
하기 위해 극저온시스템을 지속적으로 관찰하는 업무를 의미하며, 표 4.에 가동기간의 모니
터링 업무에 해당되는 내용들을 기술하였다.
극저온시스템의 모니터링은 연구원에 구축되어있는 극저온시스템용 제어시스템(이하 “제
어시스템”)을 활용하여 또는 현장에 설치되어있는 장치를 통해 수행한다. 연구원은 구축되
어있는 제어시스템을 활용하기 위한 컴퓨터, 모니터, 키보드, 마우스, 책상 및 의자를 계약
자에게 제공한다. 현재 연구원에 구축되어있는 제어시스템은 표 5.과 같다.
연구원은 계약자에게 현재 구축되어있는 제어시스템에 대한 기본 사용 교육을 제공하며,
계약자는 연구원이 제공하는 교육을 이해하기 위한 배경지식을 보유해야 한다.
문서번호 : -
2025년도 중이온가속기용 극저온시스템
개정번호 : 0
운전 용역 과업지시서 발 행 일 : 2024. 10. 30
페 이 지 : 9/19
표 4. 가동기간의 극저온시스템 모니터링 업무 리스트
순번 극저온시스템 모니터링 업무 비고
1 알람 메시지 확인
2 주요 센서 트렌드 확인 온도, 압력, 유량, 속도 등
3 정상 운전 상황의 주요 센서 지시값 파악 및 숙지 온도, 압력, 유량, 속도 등
4 비정상 운전 상황 감지
연구원에서 현장 확인 필요상황 발생 시, 연구원이 지 무전 통신 활용
5
정하는 값 관찰, 확인 및 상황 전달 (연구원에서 무전기 제공)
6 제어로직 정상 작동 여부 확인
연구원 요청 시,
7 현장 상황 일시적 또는 지속적 관찰
비방사선구역 한정
는 과업 및 운전 인력 감독자가 1인 이상 운전실에 존재해야 한다. 계약자가 가동기간에 수
행해야 하는 운전 업무 상세 내용은 하위 절에 기술된 내용을 따른다. 본 과업지시서에 기
술하지 않은 가동기간의 운전 업무는 연구원과 계약자가 협의하여 수행 여부를 결정한다.
3.3.1 극저온시스템 모니터링
계약자는 가동기간 동안 극저온시스템이 안정적으로 가동될 수 있도록 모니터링을 해야 한
다. 휴지기간의 모니터링 업무는 i) 유지보수 등을 위한 현장 활동 시 극저온시스템에 특이
사항이 발생하는지 여부를 확인하기 위해 또는 ii) 유지보수 등을 수행한 뒤 그 결과를 확인
하기 위해 극저온시스템을 지속적으로 관찰하는 업무를 의미한다. 표 9.에 휴지기간의 모니
터링 업무에 해당되는 내용들을 기술하였다.
극저온시스템의 모니터링은 연구원에 구축되어있는 극저온시스템용 제어시스템(이하 “제
어시스템”)을 활용하여 또는 현장에 설치되어있는 장치를 통해 수행한다. 연구원은 구축되
어있는 제어시스템을 활용하기 위한 컴퓨터, 모니터, 키보드, 마우스, 책상 및 의자를 계약
자에게 제공한다. 현재 연구원에 구축되어있는 제어시스템은 표 5.과 같다.
연구원은 계약자에게 현재 구축되어있는 제어시스템에 대한 기본 사용 교육을 제공하며,
계약자는 연구원이 제공하는 교육을 이해하기 위한 배경지식을 보유해야 한다.
표 9. 휴지기간의 극저온시스템 모니터링 업무 리스트
순번 극저온시스템 모니터링 업무 비고
1 알람 메시지 확인
2 주요 센서 트렌드 확인 온도, 압력, 유량, 속도 등
3 정상 상황의 주요 센서 지시값 파악 및 숙지 온도, 압력, 유량, 속도 등
4 비정상 상황 감지
연구원에서 현장 확인 필요상황 발생 시, 연구원이 지 무전 통신 활용
5
정하는 값 관찰, 확인 및 상황 전달 (연구원에서 무전기 제공)
6 제어로직 정상 작동 여부 확인
문서번호 : -
2025년도 중이온가속기용 극저온시스템
개정번호 : 0
운전 용역 과업지시서 발 행 일 : 2024. 10. 30
페 이 지 : 14/19


---

### openai answer
retriever A 점수: 3점
- 가동기간 모니터링 업무 표(표 4)에 알람 메시지 확인, 주요 센서 트렌드 확인, 정상 운전 상황의 주요 센서 지시값 파악·숙지, 비정상 운전 상황 감지, 제어로직 정상 작동 여부 확인 등 답변의 항목이 모두 포함됨.

retriever B 점수: 3점
- 가동기간 모니터링 업무 표(표 4)가 그대로 포함되어 있어 알람 메시지 확인, 주요 센서 트렌드 확인, 정상 운전 상황의 주요 센서 지시값 파악·숙지, 비정상 운전 상황 감지, 제어로직 정상 작동 여부 확인 등 답변의 항목이 모두 들어있음. 

============================================================
### query:
중이온가속기 극저온시스템의 '가동기간'과 '휴지기간'의 운전 업무 형태(근무 방식)는 어떻게 다릅니까?

### answer:
'가동기간'에는 시스템의 24시간 안정적인 운영을 위해 교대근무(24시간 운전)가 이루어집니다. 반면, '휴지기간'에는 주로 시스템 유지보수 등을 위해 주간 운전을 기본으로 합니다.

---

### retriever A:
중이온가속기 또는 극저온시스템 가동기간에는 24시간 운전 업무를 필요로 한다. 통상적으
로 가동기간에는 24시간 운전을 위한 교대근무가 이루어지며, i) 극저온시스템의 모니터링
및 제어, ii) 극저온시스템의 주요 수치를 기록 및 분석하는 업무가 주로 이루어진다.
다만, 통상적으로 이러한 교대근무로 인해 각 운전 인력마다 경험할 수 있는 운전 상황이
서로 다르게 되고, 서로 다른 경험을 했던 운전 인력 모두가 참여하여 각자의 iii) 운전 경험
을 통합/정리할 수 있는 여건이 형성될 수 없다는 한계가 존재한다. 이러한 운전 경험의 통
합 정리는 후술하는 휴지기간에의 운전업무에 집중적으로 수행 가능하다.
통상적으로 휴지기간에는 극저온시스템의 유지보수 등이 진행되며, 유지보수 등을 위한 제
어, 유지보수 등의 경과 및 결과 확인을 위한 극저온시스템 모니터링, 유지보수 등의 결과
확인을 위한 주요 수치 기록 및 분석 또한 필수적인 운전 업무이다. 휴지기간에는 통상적으
로 주간 근무로 운전이 이루어진다. 이와 같이 주간 근무가 이루어질 수 있는 휴지기간의
특성 상, 휴지기간은 가동기간에서 교대근무로 각 운전 인력 개인이 얻은 단편적인 운전 경
험을 통합하여 정리(문서화)할 수 있는 여건이 형성될 수 있는 기간이다.
결과적으로 휴지기간에는 i) 중이온가속기용 극저온시스템의 운전 경험을 통합하여 정리(문
서화)하는 업무가 집중적으로 이루어지며, ii) 유지보수 등을 위한 극저온시스템의 제어와 iii)
유지보수 등을 위한 극저온시스템의 주요 수치 기록 및 분석이 병행되어야 한다.
3.2 운전 대상 및 장소
문서번호 : -
2025년도 중이온가속기용 극저온시스템
개정번호 : 0
운전 용역 과업지시서 발 행 일 : 2024. 10. 30
페 이 지 : 12/19
매뉴얼을 작성하고 개정하는 업무를 수행해야 한다. 또한, 과업 및 운전 인력 감독자가 1인
이상 운전실에 배치되어 상기 업무들을 감독해야 한다. 계약자가 휴지기간에 수행해야 하는
운전 업무 상세 내용은 하위 절에 기술된 내용을 따른다. 본 과업지시서에 기술하지 않은
휴지기간의 운전 업무는 연구원과 계약자가 협의하여 수행 여부를 결정한다.
3.4.1 유지보수 등을 위한 극저온시스템 제어
극저온시스템의 제어는 극저온시스템이 휴지지간에 안전하고 안정적으로 운영될 수 있도록
극저온시스템 또는 극저온시스템 관련 부품들을 제어시스템을 활용하여 또는 현장에서 손으
로 조작하는 행위를 의미한다. 계약자는 극저온시스템이 본 목적에 부합하도록 극저온시스
템의 제어를 수행해야 한다. 표 8.에 극저온시스템 제어 업무에 해당되는 사항들을 기술하
였다.
휴지기간의 극저온시스템 제어는 i) 극저온시스템에 대한 충분한 이해를 바탕으로 ii) 조작
후 유지보수 등 현장 활동에 미치는 영향을 충분히 검토한 뒤 이루어져야 한다. 더불어, iii)
조작 대상물이 위치하고 있는 현장에 인력이 존재하는지 여부를 확인하고, iv) 현장에 인력
이 존재할 경우, 조작 대상물을 조작했을 때 발생할 수 있는 상황에 대해 해당 인력의 안전
이 확보된 뒤에 조작을 수행해야 한다. 마지막으로 v) 조작 대상/조작 상태에 대한 혼동이
없도록 각별히 주의를 하여 제어가 진행되어야 한다.


---

### retriever B:
3. 과업 내용
3.1 중이온가속기용 극저온시스템 운전 업무의 정의 및 필요성
본 과업에서 사용하는 “중이온가속기용 극저온시스템 운전 업무”의 정의는, “중이온가
속기용 극저온시스템이 안전하고 안정적으로 운영(가동기간뿐만 아니라 휴지기간의 운영 포
함)될 수 있도록 i) 극저온시스템을 모니터링 및 제어하고, ii) 극저온시스템의 주요 수치를
기록 및 분석하며, iii) 중이온가속기용 극저온시스템의 운전 경험을 통합하여 정리(문서화)하
는 업무를 의미한다.
중이온가속기 또는 극저온시스템 가동기간에는 24시간 운전 업무를 필요로 한다. 통상적으
로 가동기간에는 24시간 운전을 위한 교대근무가 이루어지며, i) 극저온시스템의 모니터링
및 제어, ii) 극저온시스템의 주요 수치를 기록 및 분석하는 업무가 주로 이루어진다.
다만, 통상적으로 이러한 교대근무로 인해 각 운전 인력마다 경험할 수 있는 운전 상황이
서로 다르게 되고, 서로 다른 경험을 했던 운전 인력 모두가 참여하여 각자의 iii) 운전 경험
을 통합/정리할 수 있는 여건이 형성될 수 없다는 한계가 존재한다. 이러한 운전 경험의 통
합 정리는 후술하는 휴지기간에의 운전업무에 집중적으로 수행 가능하다.
통상적으로 휴지기간에는 극저온시스템의 유지보수 등이 진행되며, 유지보수 등을 위한 제
어, 유지보수 등의 경과 및 결과 확인을 위한 극저온시스템 모니터링, 유지보수 등의 결과
확인을 위한 주요 수치 기록 및 분석 또한 필수적인 운전 업무이다. 휴지기간에는 통상적으
로 주간 근무로 운전이 이루어진다. 이와 같이 주간 근무가 이루어질 수 있는 휴지기간의
제외한다.
표 3. 2025년 중이온가속기용 극저온시스템 운전 용역 대상 장치 목록
구분 장치명 장소 비고
극저온 극저온설비동/ 압축기, 재생시스템, 2 K 시스템
SCL3 극저온플랜트
플랜트 헬륨압축기동 등 부대시설 일체
문서번호 : -
2025년도 중이온가속기용 극저온시스템
개정번호 : 0
운전 용역 과업지시서 발 행 일 : 2024. 10. 30
페 이 지 : 8/19
극저온설비동/ 압축기, 재생시스템, 2 K 시스템
SCL2 극저온플랜트
헬륨압축기동 등 부대시설 일체
TBx 극저온설비동
밸브박스, 초전도가속관 저온유지
SCL3 터널/
극저온
SCL3 LINAC 모듈 및 밸브박스 진공단열용 진
ISOL 구간
분배
공펌프
시스템 밸브박스, 초전도가속관 저온유지
SCL2 터널/
SCL2 LINAC 모듈 및 밸브박스 진공단열용 진
IF 구간
공펌프
외부 정화기 외부
회수 압축시스템 헬륨압축기동
회수
가스백 헬륨압축기동
시스템
헬륨 탱크 외부 기체 헬륨량 관리 포함
액체 질소 시스템 외부 액체 질소량 관리 포함
3.3 중이온가속기 또는 극저온시스템 가동기간의 운전 업무
계약자는 중이온가속기 또는 극저온시스템 가동기간 동안 24시간 운전업무가 이루어질 수
있도록 운전 인력을 제공해야 한다. 24시간 운전 업무는 2인 1조를 기본으로 하며, 주간에
는 과업 및 운전 인력 감독자가 1인 이상 운전실에 존재해야 한다. 계약자가 가동기간에 수
행해야 하는 운전 업무 상세 내용은 하위 절에 기술된 내용을 따른다. 본 과업지시서에 기
술하지 않은 가동기간의 운전 업무는 연구원과 계약자가 협의하여 수행 여부를 결정한다.


---

### openai answer
retriever A 점수: 3점
가동기간의 24시간 교대근무 및 휴지기간의 주간 운전(유지보수 중심) 모두 명시되어 있어 정답을 완전히 뒷받침합니다.

retriever B 점수: 3점
가동기간의 24시간 교대근무와 휴지기간의 주간 운전(유지보수 중심)이 모두 포함되어 있어 정답을 완전히 뒷받침합니다. 

============================================================
### query:
극저온시스템 과업지시서에 따르면, 가동기간 중 교대근무 인력이 개별적으로 얻은 운전 경험을 통합하여 정리하기 어려운 이유는 무엇이며, 이 업무는 주로 언제 수행되어야 합니까?

### answer:
교대근무로 인해 각 운전 인력마다 경험하는 운전 상황이 서로 달라, 모든 인원이 함께 모여 운전 경험을 통합하고 정리할 여건이 형성되기 어렵기 때문입니다. 이러한 운전 경험 통합 및 정리(문서화) 업무는 주로 주간 근무가 이루어지는 '휴지기간'에 집중적으로 수행됩니다.

---

### retriever A:
중이온가속기 또는 극저온시스템 가동기간에는 24시간 운전 업무를 필요로 한다. 통상적으
로 가동기간에는 24시간 운전을 위한 교대근무가 이루어지며, i) 극저온시스템의 모니터링
및 제어, ii) 극저온시스템의 주요 수치를 기록 및 분석하는 업무가 주로 이루어진다.
다만, 통상적으로 이러한 교대근무로 인해 각 운전 인력마다 경험할 수 있는 운전 상황이
서로 다르게 되고, 서로 다른 경험을 했던 운전 인력 모두가 참여하여 각자의 iii) 운전 경험
을 통합/정리할 수 있는 여건이 형성될 수 없다는 한계가 존재한다. 이러한 운전 경험의 통
합 정리는 후술하는 휴지기간에의 운전업무에 집중적으로 수행 가능하다.
통상적으로 휴지기간에는 극저온시스템의 유지보수 등이 진행되며, 유지보수 등을 위한 제
어, 유지보수 등의 경과 및 결과 확인을 위한 극저온시스템 모니터링, 유지보수 등의 결과
확인을 위한 주요 수치 기록 및 분석 또한 필수적인 운전 업무이다. 휴지기간에는 통상적으
로 주간 근무로 운전이 이루어진다. 이와 같이 주간 근무가 이루어질 수 있는 휴지기간의
특성 상, 휴지기간은 가동기간에서 교대근무로 각 운전 인력 개인이 얻은 단편적인 운전 경
험을 통합하여 정리(문서화)할 수 있는 여건이 형성될 수 있는 기간이다.
결과적으로 휴지기간에는 i) 중이온가속기용 극저온시스템의 운전 경험을 통합하여 정리(문
서화)하는 업무가 집중적으로 이루어지며, ii) 유지보수 등을 위한 극저온시스템의 제어와 iii)
유지보수 등을 위한 극저온시스템의 주요 수치 기록 및 분석이 병행되어야 한다.
3.2 운전 대상 및 장소
다. 계약자는 본 과업을 수행하기 위해 적절한 지식 또는 능력을 보유하고 있는 적정 수의 운
전 인력을 연구원에게 제공하여야 한다.
• 가동기간의 극저온시스템 운전 업무 (24시간 운전)
- 극저온시스템 모니터링
- 극저온시스템 주요 수치 기록 및 분석
- 극저온시스템 제어
- 기타 가동기간에 필요한 운전 업무
• 휴지기간의 극저온시스템 운전 업무 (주간 운전)
- 유지보수 등을 위한 극저온시스템 제어
- 유지보수 등을 위한 극저온시스템 모니터링
- 유지보수 등을 위한 극저온시스템 주요 수치 기록 및 분석
- 극저온시스템 운전 매뉴얼 작성 및 개정
- 극저온시스템 운전 교육
- 성과보고회 개최
- 기타 휴지기간에 필요한 운전 업무
• 본 과업 및 본 과업을 위해 투입된 인력에 대한 관리감독
본 과업의 기간은 2025년 1월 1일부터 2025년 12월 31일까지로 하며, 2025년 동안 가동기
간의 운전 업무를 수행하는 것을 기준으로 한다. 단, 표 2.와 같이 연구원의 운영 추진 상황
에 따라 휴지기간의 운전업무를 수행하는 것으로 변경될 수 있다. 가동기간과 휴지기간의
운전 업무는 “3. 과업 내용” 하위 절에 기술한 내용에 따른다.
표 2. 과업 추진 일정(안)
2025년
구분
1분기 2분기 3분기 4분기
SCL3 극저온플랜트
SCL3 중이온가속기 SCL3 구간 운전 SCL3 구간 유지보수
시운전 준비 및
구간 (가동기간) (휴지기간)
시운전 (가동기간)
SCL2 SCL2 극저온플랜트 중이온가속기 SCL2 구간 시운전 SCL2 구간 유지보수
구간 시운전 (가동기간) (가동기간) (휴지기간)
회수


---

### retriever B:
3. 과업 내용
3.1 중이온가속기용 극저온시스템 운전 업무의 정의 및 필요성
본 과업에서 사용하는 “중이온가속기용 극저온시스템 운전 업무”의 정의는, “중이온가
속기용 극저온시스템이 안전하고 안정적으로 운영(가동기간뿐만 아니라 휴지기간의 운영 포
함)될 수 있도록 i) 극저온시스템을 모니터링 및 제어하고, ii) 극저온시스템의 주요 수치를
기록 및 분석하며, iii) 중이온가속기용 극저온시스템의 운전 경험을 통합하여 정리(문서화)하
는 업무를 의미한다.
중이온가속기 또는 극저온시스템 가동기간에는 24시간 운전 업무를 필요로 한다. 통상적으
로 가동기간에는 24시간 운전을 위한 교대근무가 이루어지며, i) 극저온시스템의 모니터링
및 제어, ii) 극저온시스템의 주요 수치를 기록 및 분석하는 업무가 주로 이루어진다.
다만, 통상적으로 이러한 교대근무로 인해 각 운전 인력마다 경험할 수 있는 운전 상황이
서로 다르게 되고, 서로 다른 경험을 했던 운전 인력 모두가 참여하여 각자의 iii) 운전 경험
을 통합/정리할 수 있는 여건이 형성될 수 없다는 한계가 존재한다. 이러한 운전 경험의 통
합 정리는 후술하는 휴지기간에의 운전업무에 집중적으로 수행 가능하다.
통상적으로 휴지기간에는 극저온시스템의 유지보수 등이 진행되며, 유지보수 등을 위한 제
어, 유지보수 등의 경과 및 결과 확인을 위한 극저온시스템 모니터링, 유지보수 등의 결과
확인을 위한 주요 수치 기록 및 분석 또한 필수적인 운전 업무이다. 휴지기간에는 통상적으
로 주간 근무로 운전이 이루어진다. 이와 같이 주간 근무가 이루어질 수 있는 휴지기간의
는 과업 및 운전 인력 감독자가 1인 이상 운전실에 존재해야 한다. 계약자가 가동기간에 수
행해야 하는 운전 업무 상세 내용은 하위 절에 기술된 내용을 따른다. 본 과업지시서에 기
술하지 않은 가동기간의 운전 업무는 연구원과 계약자가 협의하여 수행 여부를 결정한다.
3.3.1 극저온시스템 모니터링
계약자는 가동기간 동안 극저온시스템이 안정적으로 가동될 수 있도록 모니터링을 해야 한


---

### openai answer
retriever A 점수: 3점
교대근무로 각자 다른 운전 경험을 모두가 함께 통합·정리하기 어렵다는 이유와, 이 업무가 휴지기간(주간 근무)에 집중 수행된다는 내용이 명시되어 있음.

retriever B 점수: 3점
교대근무로 인해 통합·정리가 어렵다는 사유와, 휴지기간에 집중적으로 수행(주간 근무)한다는 시점이 포함되어 있어 답을 모두 충족함. 

============================================================
### query:
계약업체가 매월 운전 현황을 정리해서 연구원에 보고해야 하는 공식적인 문서는 무엇이며, 제출 시점은 언제입니까?

### answer:
매월 운전 상황을 요약하여 제출해야 하는 문서는 '월간보고서'이며, 매월 기성 비용을 신청할 때 제출해야 합니다.

---

### retriever A:
계약종료 및 인수인계에 관한 사항
세부내용
○ 수행업체는 본 사업 종료 1개월 전 시스템의 발전 및 안정성을 유지하기 위한 인수인계 계획을 제시하여야 함
- 본 사업수행 기간 동안 취득한 모든 정보 및 생상된 산출물 일체
- 인수인계의 주요 업무, 상세 일정 및 인수인계 대상 주요 산출물 제시
- 인수인계 기간 동안 발생하는 리스크에 대한 방지대책 제시
○ 수행업체는 계약기간 만료 시 차기 수행업체로 선정된 사업자가 본 사업의 업무를 차질없이 수행할 수 있도록 요청 시 통합접수시스템 운영 업무 수행에 필요한 중요사항 등을 인계하여야 하며, 시스템의 효율적인 운영을 위해 하자담보 책임기간 중에는 기술자문에 응하여야 함
- 시스템 구성도, 시스템 현황, DB권한 등
- 통합접수시스템 기동/종료 절차
- 타 시스템 연계 및 적재 절차
- 일반 사용자 및 관리자 매뉴얼 등
○ 인수 및 인계 기간은 차기 시스템 운영 사업 개시일로부터 30일 이내로 하며, 인수 및 인계기간 동안의 용역비용은 지급하지 아니함
산출 정보
인수인계 계획서(사업수행계획서 내 포함 가능)
관련 요구사항
요구사항 출처
요구사항 분류
프로젝트지원 요구사항
요구사항 고유번호
PSR-008
요구사항 명칭
정보화사업 주요점검 및 수행지원
요구사항
상세설명
정의
정보화사업 주요점검 및 수행지원
세부내용
○ 통합접수시스템 관련 점검사항 준수 및 현장점검 수행
○ 보안관리 실태 점검 대응
○ 공공데이터 품질관리(DB) 수준평가 대응 및 데이터 오류 정제 활동 실시
산출 정보
점검별 결과보고서
관련 요구사항
요구사항 출처
□ 품질관리 요구사항
요구사항 분류
· 월별 추진 실적 보고
· 관련부서 업무협의 내용
· 문제점 보고 및 특이사항
매월 말일
비정기
보고
인수인계
보고
· 신규 사업자 선정시 업무인계 결과보고
사업종료
1주전
기타
· 이슈사항 및 문제점 협의 등
이슈발생시
※ 보고형식 및 주기는 추후 협의하여 조정할 수 있음
ㅇ 계약 기간 만료일 전 과업 완료 보고서를 제출하여야 함
구분
보고명
보고내용
시기
사업
종료
완료
보고서
· 용역종료 시 전체 추진단계의 수행결과 보고로 인쇄물 2부, 외장하드 1본으로 제출
· 산출물을 종류별로 분류한 파일
용역종료일
2주일 전
- 용역 완료시 제출하는 최종 결과물(프로그램 소스 코드(주석, 설명서 포함), ERD, 테이블 정의서, 프로그램 정의서 등)은 발주자의 승인을 득한 후 원본 파일을 CD 또는 HDD에 수록하여 산출물과 같이 제출하여야 하고, 프로그램 소스에 대한 형상관리가 가능하도록 협조하여야 함
- 산출물은 기존 산출물을 기반으로 현행화 또는 추가 작성해야 하며, 상호 협의하여 조정 가능
ㅇ 제출 시기에 따라 종류별 산출물을 제출하되, 사업관리 및 검수를 위한 각종 업무 보고를 요청할 경우, 신속하게 작성하여 보고하는 수시보고 체계 마련 ※ 제출 전 주관기관 사전 검토 및 확인
산출정보
착수보고서, 완료보고서, 월간보고서
요구사항 고유번호
PMR-005
요구사항 명칭
인계 조건
요구사항
상세설명
정의
관한 사항
세부
내용
인계 요건
- 사업종료 1개월 전까지 업무인계 계획서를 작성해 진흥원의 승인을 받아야함
- 발주기관과 월 1회 이상 기술이전 등에 대한 협의를 하고, 그 결과를 문서화


---

### retriever B:
4. 제출문서
계약자는 표 13.와 같은 문서들을 제출일정에 맞춰 연구원에 제출해야 한다.
표 13. 제출문서 목록 및 관련 사항
연구원
제출문서 내용 비고 및 제출일
승인
투입 인력
- 투입 인력 이력 사항 X - 계약 시
이력서
- 최초: 계약 후 1개월 내
작업계획서 - 일별 투입 인력 - 이후: 매주 금요일까지 차주
X
(주간근무표) - 휴가상황 근무표 제출
- 문서 파일 제출
- IBS에서 제공한 양식 파일
극저온시스템 - 매 월간 기성 신청 시 제출
주요 수치 - 극저온시스템 주요 수치 X - 문서 파일 제출
기록문서 - 수치 기록을 위해 계약자가
준비한 PC에 상시 저장
일일근무일지 - 일일 운영 상황 기록 O - 매 월간 기성 신청 시 제출
- 문서 파일 제출
- 수치 기록을 위해 계약자가
인수인계 문서 - (가동기간) 인수인계 내용 X 준비한 PC에 상시 저장
- 수리 사유 발생일
- 매 월간 기성 신청 시 제출
- 수리 필요 대상
수리필요사항 - 문서 파일 제출
- 상세 내용 O
기록 문서 - 수치 기록을 위해 계약자가
- 발견자
준비한 PC에 상시 저장
- 기타 필요한 내용
- 발생한 경우, 기성 신청 시
특이사항, 위험상황 발생 시,
제출 (해당 월 발생 건)
- 상황 내용
- 문서 파일 제출
- 작성자
Lesson learn O - 운전 인력 전원 회람 후 확
- 금회 대처 방법
인 서명 필수
- 향후 대처 방법
- 수치 기록을 위해 계약자가
- 기타 필요한 내용
준비한 PC에 상시 저장
- 매 월간 기성 신청 시 제출
월간보고서 - 월간 운전 상황 요약 O
- 문서 파일 제출
라. 계약 후 제출서류 : 사업수행계획서, 기술적용계획표
1) 사업수행자는 계약일로부터 14일 이내에 사업수행계획서와 기술적용계획표를 제출하고 사업 착수를 완료하여야 함
2) 사업수행계획서에는 제안요청서, 제안서, 협상에 의해 수정·보완·추가된 내용을 반영하여 제출하여야 하며 발주기관의 승인을 받아야 함
제출방법
1
착수계
A4
1
계약 후 7일 이내
서신문서
2
보안관련 문서
(보안서약서, 비밀유지서약서, 개인정보 위탁계약서, 개인정보 보호서약서)
A4
1
계약 후 7일 이내
3
사업수행계획서
A4
8
계약 후 10일 이내
4
착수보고서
A4
8
착수 후 10일 이내
5
완료보고서
A4
8
계약종료 2주 전 까지
6
최종산출물
(개발 산출물, 운영자 매뉴얼, 사용자 매뉴얼, 각종 분석/측정 결과보고서, 기술지원 확약서, 품질관리보고서, 기술적용계획/결과표, 기술이전계획서, 장애대응 매뉴얼, 테스트 수행 계획서/결과서 등)
USB
2
계약종료 2주 전 까지


---

### openai answer
retriever A 점수: 2점
- '월간보고서' 산출정보가 명시되어 문서명은 확인 가능하나, '운전 현황' 및 '매 월간 기성 신청 시 제출'이라는 제출 시점은 명확히 제시되지 않음(매월 말일 보고 등 일반 월별 보고만 언급).

retriever B 점수: 3점
- '월간보고서(월간 운전 상황 요약)'와 '매 월간 기성 신청 시 제출'이 표기되어 문서명과 제출 시점을 모두 정확히 포함. 